# Renderello - AI Product Photography Studio
One self-contained notebook: multi-image product + reference inputs, 12 researched shoot presets, very detailed prompts, and three engines (procedural CPU, SDXL-Lightning on this GPU, Magic Hour API).

**How to use:** Runtime -> Run all, then follow the sections in order. On a fresh runtime the dependency check cell may restart the kernel once (a 'session crashed' notice); if that happens, run Runtime -> Run all again - at most one restart per runtime VM; Colab sometimes assigns a fresh VM after the notice, so repeat Run all if needed (installs persist on the VM and the check then passes). Interactive use only - Colab's free runtime does not allow serving content-generation web UIs, so everything runs through this notebook UI.

## 0. Install dependencies

In [ ]:
!pip -q install fastapi uvicorn python-multipart 'pillow>=12.3,<13' numpy rembg onnxruntime kornia diffusers transformers accelerate safetensors huggingface_hub
print('installs done')

In [ ]:
# SETUP FIX (2026-11-21): Pillow install-vs-import check + one-time guarded
# kernel restart. Colab's kernel pre-imports PIL before user code runs, so a
# pip-upgraded Pillow only takes effect after a kernel restart. Hard setup
# failure (no fallback-continue) when the install or the restart cannot deliver
# the pinned Pillow. Marker bounds this to ONE restart per runtime VM.
import os
import importlib.metadata as md
import PIL

PIN = 'pillow>=12.3,<13'

def _pillow_ok(v):
    try:
        p = v.split('.')
        return (int(p[0]), int(p[1])) >= (12, 3) and int(p[0]) < 13
    except Exception:
        return False

try:
    installed = md.version('pillow')
except md.PackageNotFoundError:
    raise RuntimeError(
        'RENDERELLO SETUP FAILURE: no Pillow distribution installed after the '
        'pip step above. Check the pip output for network/resolution errors, '
        'fix the install, then Runtime -> Run all again.'
    )
print('Pillow INSTALLED:', installed)
print('Pillow IMPORTED:', PIL.__version__, PIL.__file__)

if _pillow_ok(PIL.__version__):
    print('Pillow OK for birefnet.')
elif not _pillow_ok(installed):
    raise RuntimeError(
        'RENDERELLO SETUP FAILURE: installed Pillow %s does not satisfy %s, so '
        'a kernel restart cannot fix the import. Inspect the pip output above, '
        'repair the install (manual: !pip install \'%s\'), then Runtime -> '
        'Run all again.' % (installed, PIN, PIN)
    )
elif not os.path.exists('/content/.renderello_pillow_restart'):
    open('/content/.renderello_pillow_restart', 'w').write('restarted once')
    print('Installed Pillow', installed, 'satisfies the pin but this kernel still')
    print('imports', PIL.__version__, '(pre-imported at kernel start). Restarting')
    print('the kernel ONCE. ACTION NEEDED: Runtime -> Run all again afterwards;')
    print('the marker prevents a second restart.')
    os._exit(0)
else:
    raise RuntimeError(
        'RENDERELLO SETUP FAILURE: kernel still imports Pillow %s after one '
        'restart although %s is installed. Fix: Runtime -> Disconnect and '
        'delete runtime, then Runtime -> Run all (fresh VM). If it persists, '
        'birefnet cannot run on this Colab image; report INSTALLED/IMPORTED '
        'lines above.' % (PIL.__version__, installed)
    )


## 1. The full Renderello pipeline, embedded (same code as the website)
Every module of the project is written out below by one visible cell each - nothing is downloaded as project source. Only pip packages and (for the learned cutout) the public BiRefNet model weights download at runtime.

In [ ]:
import pathlib, sys
base = pathlib.Path('/content/renderello-src')
base.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(base))
print('writing', 171, 'embedded modules into', base)

### embedded source: `renderello/`

In [ ]:
SRC = '"""Renderello / Shotelier - AI product photography prototype pipeline.\n\nQuality-first template-driven product photography: preserve the product exactly,\nchange only scene/lighting/context. See README.md for routes and status.\n"""\n\n__version__ = "0.4.77"\n'
# ---- embedded module: renderello/__init__.py ----
path = base / 'renderello/__init__.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Explicit alpha-aware byte-RGB resampling. No alpha or material recovery.\n\nFilters premultiplied sRGB code values, not calibrated linear scene radiance.\nOpaque-only inputs retain OpenCV\'s exact established byte resampler. Fully\ntransparent contributors cannot carry hidden RGB into a visible product edge.\n"""\nimport cv2\nimport numpy as np\n\n\ndef resize_rgba(rgba, size, *, interpolation):\n    if not isinstance(rgba,np.ndarray) or rgba.dtype!=np.uint8 or rgba.ndim!=3 or rgba.shape[2]!=4:\n        raise ValueError(\'uint8 RGBA required\')\n    if min(rgba.shape[:2])<1 or rgba.shape[0]*rgba.shape[1]>4_000_000:\n        raise ValueError(\'nonempty bounded4MP source required\')\n    if not isinstance(size,(tuple,list)) or len(size)!=2 or any(type(v)is not int or v<1 for v in size) or size[0]*size[1]>4_000_000:\n        raise ValueError(\'integer positive4MP destination required\')\n    if type(interpolation)is not int or interpolation not in (cv2.INTER_AREA,cv2.INTER_LANCZOS4):\n        raise ValueError(\'explicit area or Lanczos4 resampling required\')\n    if size[0]==rgba.shape[1] and size[1]==rgba.shape[0]:\n        return rgba.copy()\n    if np.all(rgba[...,3]==255):\n        return cv2.resize(rgba,tuple(size),interpolation=interpolation)\n    a=rgba[...,3].astype(np.float32)/255\n    p=rgba[...,:3].astype(np.float32)*a[...,None]\n    af=cv2.resize(a,tuple(size),interpolation=interpolation)\n    pf=cv2.resize(p,tuple(size),interpolation=interpolation)\n    # Lanczos has negative lobes. Bound coverage before division and suppress\n    # invisible RGB, including coverage that rounds to zero in the stored alpha.\n    # Divide by the unbounded filtered coverage to preserve constant straight\n    # colour when a negative-lobe filter overshoots coverage above one.\n    coverage=np.clip(af,0,1)\n    alpha=np.rint(coverage*255).astype(np.uint8)\n    straight=np.zeros_like(pf)\n    np.divide(pf,af[...,None],out=straight,where=af[...,None]>1e-8)\n    rgb=np.clip(np.rint(straight),0,255).astype(np.uint8)\n    rgb[alpha==0]=0\n    return np.dstack((rgb,alpha))\n'
# ---- embedded module: renderello/alpha_resample.py ----
path = base / 'renderello/alpha_resample.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Source-bound command-line export of explicit review-only alpha annotations.\n\nRun: python -m renderello.alpha_review_cli --source native.png --alpha alpha.png\n     --request annotation.json --output new-review-directory\n\nThe request binds decoded RGB/alpha hashes and native dimensions. It cannot\napprove production use. Reviewers must inspect both source and final composites.\n"""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\n\nfrom PIL import Image\n\nfrom .explicit_alpha_patch import export_explicit_alpha_review\n\n\ndef export_bound_review(source_path, alpha_path, request_path, output):\n    """Refuse a mismatched source/baseline before creating any output files."""\n    output = Path(output)\n    if output.exists() or output.is_symlink():\n        raise ValueError(\'preserve previous output\')\n    request = json.loads(Path(request_path).read_text(encoding=\'utf-8\'))\n    required = {\'schema_version\', \'native_size\', \'source_rgb_sha256\',\n                \'baseline_alpha_sha256\', \'protected_regions\', \'patches\'}\n    if not isinstance(request, dict) or set(request) != required:\n        raise ValueError(\'exact source-bound request fields required\')\n    if type(request[\'schema_version\']) is not int or request[\'schema_version\'] != 1:\n        raise ValueError(\'unsupported request schema\')\n    size = request[\'native_size\']\n    if (not isinstance(size, list) or len(size) != 2\n            or any(type(n) is not int or n <= 0 for n in size)\n            or size[0] * size[1] > 4_000_000):\n        raise ValueError(\'invalid native dimensions\')\n    for field in [\'source_rgb_sha256\', \'baseline_alpha_sha256\']:\n        value = request[field]\n        if (not isinstance(value, str) or len(value) != 64\n                or any(c not in \'0123456789abcdef\' for c in value)):\n            raise ValueError(\'lowercase SHA-256 binding required\')\n    with Image.open(source_path) as image:\n        if image.mode != \'RGB\' or image.size != tuple(size):\n            raise ValueError(\'native RGB mode and dimensions must match request\')\n        source = image.copy()\n    with Image.open(alpha_path) as image:\n        if image.mode != \'L\' or image.size != tuple(size):\n            raise ValueError(\'native L alpha and dimensions must match request\')\n        alpha = image.copy()\n    if hashlib.sha256(source.tobytes()).hexdigest() != request[\'source_rgb_sha256\']:\n        raise ValueError(\'source RGB hash mismatch; inspect and re-annotate the intended source\')\n    if hashlib.sha256(alpha.tobytes()).hexdigest() != request[\'baseline_alpha_sha256\']:\n        raise ValueError(\'baseline alpha hash mismatch; inspect the new baseline before replay\')\n    return export_explicit_alpha_review(source, alpha, request[\'protected_regions\'],\n                                        request[\'patches\'], output)\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser(description=__doc__)\n    for name in [\'source\', \'alpha\', \'request\', \'output\']:\n        parser.add_argument(\'--\' + name, required=True)\n    args = parser.parse_args(argv)\n    try:\n        manifest = export_bound_review(args.source, args.alpha, args.request, args.output)\n    except (ValueError, OSError, TypeError, KeyError) as exc:\n        parser.exit(2, f\'Review export refused: {exc}\\n\')\n    print(json.dumps({\'output\': str(Path(args.output).resolve()),\n                      \'alpha_changed_pixels\': manifest[\'alpha_changed_pixels\'],\n                      \'kind\': manifest[\'kind\'], \'quality_accepted\': False,\n                      \'production_review_created\': False}))\n    return 0\n\n\nif __name__ == \'__main__\':\n    raise SystemExit(main())\n'
# ---- embedded module: renderello/alpha_review_cli.py ----
path = base / 'renderello/alpha_review_cli.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Resolve archived run artifacts without searching unrelated directories."""\nfrom pathlib import Path, PurePosixPath\n\n\ndef run_artifact(run: Path, recorded: str) -> Path | None:\n    """Use only a path contained in this exact run. Old machine prefixes are data.\n\n    Accept relative paths or an absolute Windows/POSIX path anchored by the\n    exact unique run folder name. No basename-only guessing or parent traversal.\n    """\n    if not recorded:\n        return None\n    run = run.resolve()\n    parts = PurePosixPath(recorded.replace("\\\\", "/")).parts\n    if ".." in parts:\n        return None\n    if run.name in parts:\n        # Repeated run-name anchors are ambiguous, never pick one silently.\n        if parts.count(run.name) != 1:\n            return None\n        tail = parts[parts.index(run.name) + 1:]\n    elif not PurePosixPath(recorded.replace("\\\\", "/")).is_absolute() and not (len(parts) and ":" in parts[0]):\n        tail = parts\n    else:\n        return None\n    if not tail:\n        return None\n    candidate = run.joinpath(*tail).resolve()\n    if run not in candidate.parents or not candidate.is_file():\n        return None\n    return candidate\n'
# ---- embedded module: renderello/artifact_paths.py ----
path = base / 'renderello/artifact_paths.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Explicit authored platform planes and clipped 2D contact, not recovered 3D.\n\nOpt-in native translation only. Supports two designed styles. A typed source\nreview supplies camera/material/contact spans; this module checks attachment\nagainst the actual rasterized top-plane mask, never against a guessed floor.\n"""\nfrom dataclasses import replace\nimport numpy as np\nfrom PIL import Image,ImageDraw\nfrom .native_studio import background\nfrom .candidate_diagnostics import native_rgb,native_alpha,digest\nfrom .contact_spans import eligible,ContactSpan,contact_map\nfrom .candidate_harmonization import linear_over\nfrom .checked_research.contact_camera import linear,srgb\n\n\ndef platform_scene(style,*,seed=0):\n    if style not in (\'clay_plinth\',\'blush_steps\'):raise ValueError(\'explicit clay/blush authored platform required\')\n    if type(seed) is not int or not 0<=seed<=2**32-1:raise ValueError(\'integer seed0..2**32-1 required\')\n    bg=background(style,seed=seed)\n    # Clay uses its original visible top face. Blush gains an explicit top\n    # trapezoid, replacing the ambiguous thin stripe only in this opt-in route.\n    vertices=([(230,860),(760,860),(790,885),(205,885)] if style==\'clay_plinth\' else\n              [(180,860),(844,860),(874,900),(150,900)])\n    if style==\'blush_steps\':ImageDraw.Draw(bg).polygon(vertices,fill=(242,216,206))\n    plane=Image.new(\'L\',bg.size);ImageDraw.Draw(plane).polygon(vertices,fill=255)\n    rgb=np.array(bg);mask=np.array(plane)>0\n    return rgb,mask,dict(style=style,seed=seed,vertices=[list(v) for v in vertices],\n        background_sha256=digest(rgb),surface_sha256=digest(mask),\n        authored_surface=True,recovered_geometry=False,accepted=False)\n\n\ndef compose_platform_contact(source,alpha,style,spans,policy,*,translation,\n        native_floor_row,support_tolerance=2,seed=0):\n    native_rgb(source);native_alpha(alpha,source.shape[:2]);eligible(source,alpha,policy)\n    if policy.camera!=\'front\' or policy.source_pose!=\'front\':raise ValueError(\'reviewed front camera/pose only; no viewpoint change\')\n    if not isinstance(translation,(list,tuple)) or len(translation)!=2 or any(type(v) is not int or v<0 for v in translation):\n        raise ValueError(\'nonnegative integer translation required\')\n    if type(native_floor_row) is not int or not 0<=native_floor_row<alpha.shape[0]-1:\n        raise ValueError(\'native support floor row required\')\n    if type(support_tolerance) is not int or not 0<=support_tolerance<=4:raise ValueError(\'integer0..4support tolerance required\')\n    if not isinstance(spans,(list,tuple)) or not spans or any(not isinstance(s,ContactSpan) for s in spans):\n        raise ValueError(\'explicit typed native contact spans required\')\n    bg,plane,design=platform_scene(style,seed=seed);dx,dy=translation;h,w=alpha.shape\n    if dx+w>1024 or dy+h>1024:raise ValueError(\'entire native source must fit; no crop/resize\')\n    # Validate native bounds before translating, so blank canvas cannot excuse\n    # a declaration that lies outside the original image.\n    if any(s.x1>w or s.bottom_max>=h for s in spans):raise ValueError(\'native span outside source\')\n    rgb=np.zeros_like(bg);a=np.zeros((1024,1024),np.uint8)\n    rgb[dy:dy+h,dx:dx+w]=source;a[dy:dy+h,dx:dx+w]=alpha\n    canvas_policy=replace(policy,source_sha256=digest(rgb),alpha_sha256=digest(a))\n    canvas_spans=[ContactSpan(s.name,s.x0+dx,s.x1+dx,s.bottom_min+dy,s.bottom_max+dy) for s in spans]\n    shade,contact=contact_map(rgb,a,canvas_spans,canvas_policy)\n    hard=a>=128;lower=np.where(hard.any(0),1023-np.argmax(hard[::-1],axis=0),-1);attached=[]\n    for span in canvas_spans:\n        cols=np.arange(span.x0,span.x1);rows=lower[cols]\n        if np.any(np.abs(rows-(native_floor_row+dy))>support_tolerance):\n            raise ValueError(\'declared floor contradicts observed native contact contour\')\n        if not np.all(plane[rows,cols]&plane[rows+1,cols]):\n            raise ValueError(\'actual contact footprint lies outside visible top plane\')\n        attached.append(dict(name=span.name,columns=len(cols),all_first_rows_on_surface=True))\n    clipped=shade.copy();clipped[~plane]=0\n    for span in canvas_spans:\n        cols=np.arange(span.x0,span.x1)\n        if not np.all(clipped[lower[cols]+1,cols]>0):raise ValueError(\'clipped shadow detached from contact\')\n    out=linear_over(rgb,a,srgb(linear(bg)*(1-clipped[...,None])))\n    if np.any(clipped[hard]) or not np.array_equal(out[a==255],rgb[a==255]):raise RuntimeError(\'source/shadow invariant failed\')\n    record=dict(schema=\'authored-platform-contact.v1\',design=design,contact=contact,\n        source_sha256=digest(source),alpha_sha256=digest(alpha),\n        source_to_canvas_translation=list(translation),native_floor_row=native_floor_row,\n        support_tolerance=support_tolerance,attachments=attached,\n        shadow_pixels_clipped_off_surface=int(((shade>0)&~plane).sum()),\n        shadow_pixels_outside_surface=int(((clipped>0)&~plane).sum()),\n        result_sha256=digest(out),shadow_sha256=digest(clipped),source_cropped=False,\n        source_resampled=False,source_rgb_repainted=False,opaque_source_pixels_exact=True,\n        accepted=False,independent_review=False,recovered_geometry=False,physical_relighting=False,\n        limitations=[\'Visible platform plane is authored 2D design, not recovered geometry.\',\n          \'Reviewed camera, material, completeness and support spans are not independently verified.\',\n          \'Source lighting, edge contamination and old reflections remain unchanged.\',\n          \'Raster attachment and source preservation do not imply plausible lighting or publication quality.\'])\n    return out,clipped,bg,plane,record\n'
# ---- embedded module: renderello/authored_platform_contact.py ----
path = base / 'renderello/authored_platform_contact.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


### embedded source: `renderello/backends/`

In [ ]:
SRC = '"""Editor backends."""\nfrom .base import EditorBackend, StubEditor\n\n__all__ = ["EditorBackend", "StubEditor"]\n'
# ---- embedded module: renderello/backends/__init__.py ----
path = base / 'renderello/backends/__init__.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Editor backend interface. A backend takes (source image, prompt, negative\nprompt, seed, aspect) and returns an edited PIL image. Implementations:\nqwen_local (GPU, Apache-2.0), magichour (API, mock by default), stub (tests)."""\nfrom __future__ import annotations\n\nfrom typing import Protocol\n\nfrom PIL import Image\n\nfrom ..schemas import Aspect\n\n\nclass EditorBackend(Protocol):\n    name: str\n\n    def edit(self, source: Image.Image, prompt: str, negative_prompt: str,\n             seed: int, aspect: Aspect, reference: Image.Image | None = None) -> Image.Image:\n        ...\n\n\nclass StubEditor:\n    """Deterministic test double: applies a fixed, seed-dependent color grade so\n    pipeline/QA behavior is testable without GPU or network. NOT a quality\n    signal - it exists only to exercise orchestration code."""\n    name = "stub"\n\n    def edit(self, source: Image.Image, prompt: str, negative_prompt: str,\n             seed: int, aspect: Aspect, reference: Image.Image | None = None) -> Image.Image:\n        import numpy as np\n        arr = np.array(source.convert("RGB")).astype(np.float32)\n        shift = ((seed % 7) - 3) * 2.0\n        arr[..., 0] = np.clip(arr[..., 0] + shift, 0, 255)\n        return Image.fromarray(arr.astype(np.uint8)).resize(\n            aspect.size, Image.LANCZOS)\n'
# ---- embedded module: renderello/backends/base.py ----
path = base / 'renderello/backends/base.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Opt-in FLUX.2[klein]4B source editor. Never selected as the default.\n\nOwning model card: https://huggingface.co/black-forest-labs/FLUX.2-klein-4B\nApache2 per card, pinned revision. This is NOT the MagicHour API configuration.\nQuality and device fit need an actual run. Model edits can change labels/geometry.\nNo exact-product, physical-relight or publication-acceptance guarantee.\n"""\nfrom __future__ import annotations\nimport threading\nfrom PIL import Image\nfrom ..schemas import Aspect\n\nMODEL_ID=\'black-forest-labs/FLUX.2-klein-4B\'\nMODEL_REVISION=\'e7b7dc27f91deacad38e78976d1f2b499d76a294\'\n\ndef install_vae_decode_dtype_guard(vae):\n    """Match decode sample to VAE weights; preserve original decode/hooks.\n\n    Flux2KleinPipeline casts decode latents to device but not VAE dtype.\n    T4 fp16 denoising with a float32 VAE therefore needs this boundary cast.\n    This does not alter prompts, seeds, denoising or model weights.\n    """\n    if getattr(vae, \'_renderello_decode_dtype_guard\', False):\n        return\n    original_decode = vae.decode\n    def decode_with_matching_dtype(sample, *args, **kwargs):\n        return original_decode(sample.to(dtype=vae.dtype), *args, **kwargs)\n    vae.decode = decode_with_matching_dtype\n    vae._renderello_decode_dtype_guard = True\n\nclass KleinEditBackend:\n    name=\'flux2-klein-4b-local-experimental\'\n    _lock=threading.RLock()\n\n    def __init__(self, *, cpu_offload: bool=True, max_sequence_length: int=512):\n        if type(cpu_offload) is not bool:raise ValueError(\'cpu_offload must be boolean\')\n        if type(max_sequence_length) is not int or not 32<=max_sequence_length<=512:\n            raise ValueError(\'max_sequence_length must be integer32..512\')\n        self.cpu_offload=cpu_offload\n        self.max_sequence_length=max_sequence_length\n        self._pipe=None\n        self.last_diagnostics={}\n\n    def _load(self):\n        import torch\n        from diffusers import Flux2KleinPipeline\n        if not torch.cuda.is_available():raise RuntimeError(\'Klein editor requires CUDA here; no CPU fallback silently selected\')\n        if self._pipe is None:\n            major,_=torch.cuda.get_device_capability()\n            dtype=torch.bfloat16 if major>=8 else torch.float16\n            pipe=Flux2KleinPipeline.from_pretrained(MODEL_ID,revision=MODEL_REVISION,torch_dtype=dtype)\n            # VAE\'s owning config force_upcast=True. T4 lacks native bf16.\n            pipe.vae.to(dtype=torch.float32)\n            install_vae_decode_dtype_guard(pipe.vae)\n            if self.cpu_offload:pipe.enable_model_cpu_offload()\n            else:pipe.to(\'cuda\')\n            pipe.vae.enable_slicing()\n            self._pipe=pipe\n        return self._pipe\n\n    def edit(self, source: Image.Image, prompt: str, negative_prompt: str,\n             seed: int, aspect: Aspect, reference: Image.Image|None=None)->Image.Image:\n        if not isinstance(source,Image.Image):raise ValueError(\'PIL source required\')\n        if not isinstance(prompt,str) or not prompt.strip():raise ValueError(\'nonempty prompt required\')\n        if not isinstance(negative_prompt,str):raise ValueError(\'negative_prompt must be string\')\n        if type(seed) is not int or not 0<=seed<2**63:raise ValueError(\'seed must be integer0..2**63-1\')\n        if not isinstance(aspect,Aspect):raise ValueError(\'known Aspect required\')\n        if reference is not None and not isinstance(reference,Image.Image):raise ValueError(\'PIL reference required\')\n        # Step-distilled Klein has no separate negative-prompt channel. Do not\n        # silently drop exclusions: preserve them as explicit narrative text.\n        text=prompt if not negative_prompt.strip() else prompt+\'\\nAvoid the following: \'+negative_prompt\n        with self._lock:\n            pipe=self._load()\n            rendered=pipe.tokenizer.apply_chat_template([{\'role\':\'user\',\'content\':text}],tokenize=False,add_generation_prompt=True,enable_thinking=False)\n            ids=pipe.tokenizer(rendered,truncation=False)[\'input_ids\']\n            if len(ids)>self.max_sequence_length:\n                raise ValueError(f\'Full prompt requires{len(ids)}tokens; limit{self.max_sequence_length}. No truncation performed.\')\n            images=source.convert(\'RGB\') if reference is None else [source.convert(\'RGB\'),reference.convert(\'RGB\')]\n            import torch\n            w,h=aspect.size\n            self.last_diagnostics=dict(model=MODEL_ID,revision=MODEL_REVISION,source_conditioned=True,reference_count=1 if reference is None else 2,full_prompt=text,prompt_tokens=len(ids),prompt_truncated=False,steps=4,negative_channel=\'narrative_append\' if negative_prompt.strip() else \'none\',magic_hour_equivalent=False,quality_accepted=False)\n            with torch.inference_mode():\n                out=pipe(image=images,prompt=text,width=w,height=h,num_inference_steps=4,guidance_scale=1.,max_sequence_length=self.max_sequence_length,generator=torch.Generator(device=\'cuda\').manual_seed(seed)).images[0]\n            if not isinstance(out,Image.Image) or out.size!=(w,h):raise RuntimeError(\'Editor returned wrong image type/dimensions\')\n            if out.mode!=\'RGB\':out=out.convert(\'RGB\')\n            if all(high-low<=2 for low,high in out.getextrema()):raise RuntimeError(\'Editor returned blank image\')\n            return out\n\n    def close(self):\n        with self._lock:\n            if self._pipe is not None:\n                self._pipe=None\n                import torch,gc\n                gc.collect();torch.cuda.empty_cache()\n'
# ---- embedded module: renderello/backends/klein_local.py ----
path = base / 'renderello/backends/klein_local.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Magic Hour AI Image Editor backend. Two hard rules baked in:\n\n1. MOCK BY DEFAULT. Without live=True every call is served by an offline mock\n   (deterministic gradient echo) and records zero credits. Live calls require\n   live=True AND MAGIC_HOUR_API_KEY in the environment. The key is read from\n   the environment per call and never written to disk or logs by this repo.\n2. READ-ONLY ACCOUNT ENDPOINTS do not generate images or spend credits.\n   Permission to access an account must be established outside this code.\n   Generation requires live=True; a CLI flag is not owner authorization.\n\nModel/credit table from docs.magichour.ai/api-reference/models\n(refreshed 2026-11-20, edit-capable models); free-tier models are 640px-only, NB2.1 is creator-tier. NB2 requests run on 2.1 at 2.1 rates per docs.\nEditor API has NO mask input (schema-verified in the plan), so Route 3-style\nconstrained edits are not available through this backend.\n"""\nfrom __future__ import annotations\n\nimport io\nimport json\nimport time\nfrom pathlib import Path\n\nfrom PIL import Image\n\nfrom ..config import MAGICHOUR_API_BASE\nfrom ..schemas import Aspect\n\n\n# model key -> (credits per image, tier, max long side). Source: plan Part B.\nMODEL_TABLE: dict[str, dict] = {\n    # Edit-capable models only, from docs.magichour.ai/api-reference/models\n    # (fetched 2026-11-14). credits = "From" single-image lowest-res rate;\n    # actual charge varies with resolution and image count (docs warn the\n    # batch rate is not always count x single rate, e.g. nano-banana-2.1 grid).\n    # max_side 640 = free-plan cap; larger resolutions need a paid plan.\n    # max_inputs = "Max additional input images (editor)" column.\n    "flux-2-klein":    {"credits": 5,   "tier": "free",    "max_side": 640,  "max_inputs": 5},\n    "qwen-edit":       {"credits": 10,  "tier": "free",    "max_side": 640,  "max_inputs": 2},\n    "krea-2":          {"credits": 10,  "tier": "free",    "max_side": 640,  "max_inputs": 1},\n    "qwen-image-2.1":  {"credits": 10,  "tier": "free",    "max_side": 640,  "max_inputs": 2},\n    "seedream-v4":     {"credits": 40,  "tier": "creator", "max_side": 4096, "max_inputs": 9},\n    "seedream-v4.5":   {"credits": 50,  "tier": "creator", "max_side": 4096, "max_inputs": 9},\n    "seedream-v5-flash": {"credits": 30, "tier": "creator", "max_side": 2048, "max_inputs": 9},\n    "nano-banana":     {"credits": 50,  "tier": "creator", "max_side": 1024, "max_inputs": 9},\n    "nano-banana-2-lite": {"credits": 50, "tier": "creator", "max_side": 1024, "max_inputs": 9},\n    "nano-banana-2.1": {"credits": 50,  "tier": "creator", "max_side": 4096, "max_inputs": 9},\n    "seedream-v5-pro": {"credits": 75,  "tier": "creator", "max_side": 2048, "max_inputs": 9},\n    "nano-banana-2":   {"credits": 50, "tier": "creator", "max_side": 4096, "max_inputs": 9},\n    "gpt-image-2": {"credits": 50, "tier": "creator", "max_side": 4096, "max_inputs": 9},\n    "gpt-image-2.5-flare": {"credits": 100, "tier": "creator", "max_side": 4096, "max_inputs": 9},\n    # Source-verified 13 Nov 2026 receipts (pre-dating the 20 Nov 2026\n    # 1000-credit round; 3 receipts = 600 total, earlier baseline scope):\n    # 200 credits per image at res=1k, 1:1, single input. Source-code\n    # option only - NOT on the live allowlist; a separate pricing/cap\n    # analysis is required before any spend on this entry.\n    "gpt-image-2.5-flare-1k": {"credits": 200, "tier": "creator", "max_side": 1024, "max_inputs": 1,\n                               "api_model": "gpt-image-2.5-flare", "resolution": "1k"},\n    "nano-banana-pro": {"credits": 150, "tier": "creator", "max_side": 4096, "max_inputs": 9},\n}\n\n_ASPECT_TO_API = {Aspect.SQUARE: "1:1", Aspect.PORTRAIT: "4:5",\n                  Aspect.LANDSCAPE: "3:2"}\n\n\nclass MagicHourError(RuntimeError):\n    pass\n\n\nclass MagicHourBackend:\n    """Editor + read-only account client."""\n\n    def __init__(self, model: str = "qwen-edit", live: bool = False,\n                 base_url: str = MAGICHOUR_API_BASE,\n                 cost_log: Path | None = None, timeout: float = 120.0,\n                 poll_timeout: float = 300.0, poll_interval: float = 3.0,\n                 receipt_path: Path | None = None):\n        if model not in MODEL_TABLE:\n            raise KeyError(f"unknown Magic Hour model \'{model}\'. "\n                           f"known: {\', \'.join(sorted(MODEL_TABLE))}")\n        self.model = model\n        self.live = live\n        self.base_url = base_url.rstrip("/")\n        self.cost_log = cost_log\n        self.timeout = timeout\n        self.poll_timeout = poll_timeout\n        self.poll_interval = poll_interval\n        self.receipt_path = receipt_path or Path("runs/mh-projects.jsonl")\n        self.last_result = None\n\n    @property\n    def name(self):\n        return \'magichour:\'+self.model+(\':live\' if self.live else \':mock\')\n\n    def attempt_evidence(self):\n        """Sanitized per-call facts, not provider/account authentication or price.\n\n        Only a current terminal detail supplies settled recorded charge. Submission\n        and pending charges are separate observations, never a zero-cost fallback.\n        """\n        import math\n        def charge(row):\n            value = row.get(\'credits_charged\') if isinstance(row,dict) else None\n            return value if type(value) in (int,float) and math.isfinite(value) and value>=0 else None\n        state = self.last_result if isinstance(self.last_result,dict) else {}\n        detail = state.get(\'final\') or state.get(\'latest\') or {}\n        status = detail.get(\'status\')\n        terminal = status in (\'complete\',\'error\',\'canceled\')\n        project = state.get(\'id\')\n        if not isinstance(project,str) or not project or len(project)>240:project=None\n        return dict(schema=\'provider-attempt.v1\',provider=\'magichour\',model=self.model,\n            attempt_id=getattr(self,\'_attempt_id\',None),mock=not self.live,\n            stage=getattr(self,\'_attempt_stage\',\'not_started\'),project_id=project,\n            provider_status=status if status in (\'complete\',\'error\',\'canceled\',\'draft\',\'queued\',\'rendering\') else None,\n            terminal_detail_observed=terminal,\n            credits_charged=charge(detail) if terminal else None,\n            submission_credits_observed=charge(state.get(\'submission\')),\n            latest_credits_observed=charge(detail),\n            charge_basis=\'terminal_detail\' if terminal else \'UNKNOWN_NOT_SETTLED\',\n            receipt_path=str(self.receipt_path),cost_log_path=None if self.cost_log is None else str(self.cost_log),\n            account_credit_system=\'NOT_BOUND_REQUIRES_CALLER_ACCOUNT_SCOPE\',\n            semantics=\'recorded_backend_response_not_provider_authentication; no_price_or_profit_inferred\')\n\n    # ---- read-only account ----\n    def balance(self) -> dict:\n        """GET /v1/account - read-only, no generation. Account-access authority\n        must be checked separately. Returns the provider JSON."""\n        import os\n        import requests\n        key = os.environ.get("MAGIC_HOUR_API_KEY")\n        if not key:\n            raise MagicHourError("MAGIC_HOUR_API_KEY not set; balance is read-only "\n                                 "but still needs the key in the environment")\n        r = requests.get(f"{self.base_url}/v1/account",\n                         headers={"Authorization": f"Bearer {key}"},\n                         timeout=self.timeout)\n        if r.status_code != 200:\n            raise MagicHourError(f"balance check failed: HTTP {r.status_code}")\n        return r.json()\n\n    # ---- generation (gated) ----\n    def credits_per_image(self) -> int:\n        return MODEL_TABLE[self.model]["credits"]\n\n    def edit(self, source: Image.Image, prompt: str, negative_prompt: str,\n             seed: int, aspect: Aspect, reference: Image.Image | None = None,\n             extra_images: list[Image.Image] | None = None) -> Image.Image:\n        """One editor generation. Inputs: product photo first, then any extra\n        product photos and/or style references. Multi-image models take up to\n        max_inputs additional images; overflow is noted, never silently dropped.\n        """\n        import uuid\n        self.last_result = None\n        self._attempt_id = uuid.uuid4().hex\n        self._attempt_stage = \'before_submission\'\n        extras = list(extra_images or [])\n        if reference is not None:\n            extras = [reference] + extras\n        limit = MODEL_TABLE[self.model].get("max_inputs", 0)\n        self.dropped_inputs = []\n        if len(extras) > limit:\n            self.dropped_inputs = extras[limit:]\n            extras = extras[:limit]\n        try:\n            image = self._edit_once(source, prompt, negative_prompt, seed, aspect,\n                                    extra_images=extras)\n            self._attempt_stage = \'output_decoded\'\n            return image\n        except Exception:\n            # A transport error after POST can still be billed. Never retry here.\n            raise\n\n    def _edit_once(self, source, prompt, negative_prompt, seed, aspect, reference=None,\n                   extra_images=None):\n        if not self.live:\n            self.last_result = {\'mock\': True, \'final\': {\'status\': \'complete\', \'credits_charged\': 0}}\n            return self._mock_edit(source, seed, aspect)\n        import os\n        import requests\n        key = os.environ.get("MAGIC_HOUR_API_KEY")\n        if not key:\n            raise MagicHourError("live edit requested but MAGIC_HOUR_API_KEY is unset")\n        extras = list(extra_images or [])\n        if reference is not None:\n            extras = [reference] + extras\n        images = [source] + extras\n        headers = {"Authorization": f"Bearer {key}"}\n        paths = []\n        for image in images:\n            buf = io.BytesIO()\n            image.convert("RGB").save(buf, "PNG")\n            up = requests.post(f"{self.base_url}/v1/files/upload-urls",\n                               headers=headers,\n                               json={"items": [{"type": "image", "extension": "png"}]},\n                               timeout=self.timeout)\n            up.raise_for_status()\n            item = up.json()["items"][0]\n            requests.put(item["upload_url"], data=buf.getvalue(),\n                         headers={"Content-Type": "image/png"},\n                         timeout=self.timeout).raise_for_status()\n            paths.append(item["file_path"])\n        # Lowest supported resolution fixed explicitly: auto is tier-dependent\n        # and can silently cost more. API has no seed or negative_prompt fields.\n        entry = MODEL_TABLE[self.model]\n        payload = {"model": entry.get("api_model", self.model), "image_count": 1,\n                   "resolution": entry.get("resolution", "1k" if self.model == "nano-banana-pro" else "640px"),\n                   "aspect_ratio": _ASPECT_TO_API[aspect],\n                   "style": {"prompt": prompt},\n                   "assets": {"image_file_paths": paths}}\n        started = time.monotonic()\n        # Never automatically retry a generation POST, including transport errors:\n        # the server might already have accepted and charged it.\n        self._attempt_stage = \'submission_started\'\n        res = requests.post(f"{self.base_url}/v1/ai-image-editor",\n                            headers=headers, json=payload, timeout=self.timeout)\n        if res.status_code == 402:\n            raise MagicHourError("insufficient credits (402): no automatic retry")\n        if res.status_code in (401, 403):\n            raise MagicHourError(f"auth/plan gate (HTTP {res.status_code}): no retry")\n        res.raise_for_status()\n        accepted = res.json()\n        project_id = accepted.get("id")\n        if not project_id:\n            raise MagicHourError("submission response lacks project ID; do not resubmit blindly")\n        self.last_result = {"id": project_id, "submission": accepted}\n        self._attempt_stage = \'project_accepted\'\n        self.receipt_path.parent.mkdir(parents=True, exist_ok=True)\n        with self.receipt_path.open("a") as f:\n            f.write(json.dumps({"project_id": project_id, "model": self.model,\n                                "credits_charged": accepted.get("credits_charged"),\n                                "resolution": payload["resolution"]}) + "\\n")\n        # Persist receipt before polling/download so recovery never needs a second\n        # generation. Creation charge and final/refunded charge remain distinct.\n        self.last_result = {"id": project_id, "submission": accepted}\n        final = self.wait_project(project_id)\n        self.last_result = {"id": project_id, "submission": accepted, "final": final}\n        self._log_cost(seed, aspect, time.monotonic() - started, final)\n        return self._download(final)\n\n    def wait_project(self, project_id: str) -> dict:\n        """Read-only bounded status polling. Timeout preserves ID; no recreation."""\n        import os\n        import requests\n        from urllib.parse import quote\n        key = os.environ.get("MAGIC_HOUR_API_KEY")\n        if not key:\n            raise MagicHourError("MAGIC_HOUR_API_KEY not set")\n        deadline = time.monotonic() + self.poll_timeout\n        while True:\n            res = requests.get(f"{self.base_url}/v1/image-projects/{quote(project_id, safe=\'\')}",\n                               headers={"Authorization": f"Bearer {key}"},\n                               timeout=self.timeout)\n            res.raise_for_status()\n            data = res.json()\n            if isinstance(self.last_result, dict) and self.last_result.get(\'id\') == project_id:\n                self.last_result[\'latest\'] = data\n            state = data.get("status")\n            if state == "complete":\n                return data\n            if state in ("error", "canceled", "draft"):\n                raise MagicHourError(f"project {project_id} status={state}; recover this ID, do not resubmit")\n            if state not in ("queued", "rendering"):\n                raise MagicHourError(f"project {project_id} returned unknown status {state!r}")\n            if time.monotonic() >= deadline:\n                raise MagicHourError(f"project {project_id} still {state}; polling timed out, resume this ID")\n            time.sleep(min(self.poll_interval, max(0, deadline - time.monotonic())))\n\n    def _download(self, project: dict) -> Image.Image:\n        import requests\n        downloads = project.get("downloads") or []\n        if not downloads:\n            raise MagicHourError("complete project has no downloads; recover project, do not resubmit")\n        res = requests.get(downloads[0]["url"], timeout=self.timeout)\n        res.raise_for_status()\n        return Image.open(io.BytesIO(res.content)).convert("RGB")\n\n    def resume(self, project_id: str) -> Image.Image:\n        """Recover existing project by ID, read-only, never a new generation."""\n        final = self.wait_project(project_id)\n        self.last_result = {"id": project_id, "final": final}\n        return self._download(final)\n\n    def _mock_edit(self, source: Image.Image, seed: int, aspect: Aspect) -> Image.Image:\n        """Offline deterministic echo: resize + seed-dependent tint. Exercises\n        orchestration and cost accounting with zero network and zero credits."""\n        import numpy as np\n        arr = np.array(source.convert("RGB").resize(aspect.size,\n                                                    Image.LANCZOS)).astype(np.float32)\n        arr[..., 2] = np.clip(arr[..., 2] + (seed % 5) * 3.0, 0, 255)\n        self._log_cost(seed, aspect, 0.0, {"mock": True, "credits_charged": 0})\n        return Image.fromarray(arr.astype(np.uint8))\n\n    def _log_cost(self, seed: int, aspect: Aspect, latency_s: float,\n                  response: dict) -> None:\n        if self.cost_log is None:\n            return\n        self.cost_log.parent.mkdir(parents=True, exist_ok=True)\n        row = {\n            "ts": time.strftime("%Y-%m-%dT%H:%M:%S%z"),\n            "model": self.model, "seed": seed, "aspect": aspect.value,\n            "credits_listed": self.credits_per_image(),\n            "credits_charged": response.get("credits_charged", None if self.live else 0),\n            "mock": not self.live, "latency_s": round(latency_s, 3),\n        }\n        with self.cost_log.open("a") as f:\n            f.write(json.dumps(row) + "\\n")\n'
# ---- embedded module: renderello/backends/magichour.py ----
path = base / 'renderello/backends/magichour.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Qwen-Image-Edit-2511 local backend (Apache-2.0, owning repo:\nhuggingface.co/Qwen/Qwen-Image-Edit-2511). GPU only; imported lazily so CPU\ninstalls never touch torch/diffusers. Inference parameters follow the model\ncard quick-start (true_cfg_scale 4.0, guidance_scale 1.0, 40 steps)."""\nfrom __future__ import annotations\n\nimport threading\n\nfrom PIL import Image\n\nfrom ..config import (QWEN_EDIT_MODEL_ID, QWEN_DEFAULT_STEPS, QWEN_GUIDANCE_SCALE,\n                      QWEN_TRUE_CFG_SCALE, CACHE_DIR, DeviceConfig)\n\n\nclass QwenEditBackend:\n    name = "qwen-image-edit-2511"\n    _lock = threading.Lock()\n\n    def __init__(self, device: DeviceConfig | None = None, steps: int = QWEN_DEFAULT_STEPS,\n                 low_memory: bool = False):\n        self.device = device or DeviceConfig.resolve(prefer_gpu=True)\n        if self.device.device != "cuda":\n            raise RuntimeError("Qwen-Image-Edit-2511 needs a CUDA GPU here; use "\n                               "Route 2 (cutout_compose) on CPU or run on Kaggle/Colab")\n        self.steps = steps\n        # bf16 weights total ~56.6 GB (HF API file listing, 2026-11-09):\n        # they do NOT fit T4-class 15 GB cards in bf16. low_memory=True uses\n        # sequential CPU offload (needs accelerate): feasible but SLOW.\n        # UNTESTED as of v0.1.5 - first GPU run must validate and report.\n        self.low_memory = low_memory\n        self._pipe = None\n\n    def _load(self):\n        import torch\n        from diffusers import QwenImageEditPlusPipeline  # per model card\n        with self._lock:\n            if self._pipe is None:\n                dtype = getattr(torch, self.device.dtype_name)\n                self._pipe = QwenImageEditPlusPipeline.from_pretrained(\n                    QWEN_EDIT_MODEL_ID, torch_dtype=dtype, cache_dir=str(CACHE_DIR))\n                if self.low_memory:\n                    self._pipe.enable_sequential_cpu_offload()\n                else:\n                    self._pipe.to(self.device.device)\n                self._pipe.set_progress_bar_config(disable=True)\n        return self._pipe\n\n    def edit(self, source: Image.Image, prompt: str, negative_prompt: str,\n             seed: int, aspect, reference: Image.Image | None = None) -> Image.Image:\n        import torch\n        pipe = self._load()\n        images = [source.convert("RGB")]\n        if reference is not None:  # 2511 accepts multi-image input (model card)\n            images.append(reference.convert("RGB"))\n        with torch.inference_mode():\n            out = pipe(\n                image=images,\n                prompt=prompt,\n                negative_prompt=negative_prompt or " ",\n                generator=torch.manual_seed(seed),\n                true_cfg_scale=QWEN_TRUE_CFG_SCALE,\n                guidance_scale=QWEN_GUIDANCE_SCALE,\n                num_inference_steps=self.steps,\n                num_images_per_prompt=1,\n            )\n        return out.images[0]\n'
# ---- embedded module: renderello/backends/qwen_local.py ----
path = base / 'renderello/backends/qwen_local.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Benchmark harness. Consumes a manifest of 20+ source images (the plan\'s\nacceptance gate: varied sources incl. tiny-text skincare bottle, reflective can,\ntransparent glass, shoes, bags, apparel, food packaging, multi-product, complex\nedges, irregular shapes), runs the full pipeline per source x template, and\naggregates QA into scores.csv + a per-run report.\n\nManifest format (benchmark/manifest.json):\n{\n  "sources": [{"path": "benchmark/sources/skincare_bottle.jpg",\n               "label": "skincare bottle with tiny text",\n               "license": "own photo | CC0 | ...",   // recorded per plan\n               "notes": "optional"}],\n  "templates": ["clean_studio", "luxury_hero"],        // subset or "all"\n  "routes": ["cutout_compose", "pure_prompt", "hybrid"],\n  "variants": 2\n}\n"""\nfrom __future__ import annotations\n\nimport csv\nimport json\nfrom dataclasses import asdict, replace\nfrom pathlib import Path\n\nfrom PIL import Image  # noqa: F401 - validates sources up front\n\nfrom .pipeline import run_pipeline\nfrom .qa import report_to_csv_row\nfrom .config import RunConfig\nfrom .schemas import EditRequest, Route\nfrom .templates import all_templates, get_template\n\n\ndef load_manifest(path: Path) -> dict:\n    m = json.loads(path.read_text())\n    if len(m.get("sources", [])) < 1:\n        raise ValueError("manifest has no sources")\n    for s in m["sources"]:\n        if not (path.parent / s["path"]).exists():\n            raise FileNotFoundError(f"source missing: {s[\'path\']}")\n        if "license" not in s:\n            raise ValueError(f"source {s[\'path\']} lacks a license record (plan B6)")\n    from .source_contract import SourceContract\n    for source in m[\'sources\']:\n        SourceContract(**source.get(\'source_contract\', {}))\n        if \'templates\' in source:\n            if not source[\'templates\']:raise ValueError(\'source template list is empty\')\n            for key in source[\'templates\']:get_template(key)\n    if \'variants\' in m and (type(m[\'variants\']) is not int or not 1<=m[\'variants\']<=4):\n        raise ValueError(\'manifest variants must be an integer 1..4\')\n    return m\n\n\ndef run_benchmark(manifest_path: Path, cfg: RunConfig,\n                  editor=None, out_dir: Path | None = None,\n                  scene_fn=None, scene_kind: str = "", allow_colour_conversion: bool = False,\n                  scene_finish: bool = False, scene_accept=None, scene_max_attempts: int = 1) -> Path:\n    m = load_manifest(manifest_path)\n    templates = all_templates() if m.get("templates") == "all" else \\\n        [get_template(k) for k in m["templates"]]\n    routes = [Route(r) for r in m["routes"]]\n    out = out_dir or cfg.out_root / "benchmark"\n    out.mkdir(parents=True, exist_ok=True)\n\n    # Manifest declares repeat counts and reviewer inputs, never permission for\n    # external generation or a quality claim. Invalid declarations fail before\n    # invoking any source/model, rather than losing failures from the denominator.\n    from .source_contract import SourceContract\n    cfg=replace(cfg,variants=m.get(\'variants\',cfg.variants))\n    # Validate the whole source list before any generation/model work.\n    # A late ICC/depth/orientation error must not appear only after prior calls.\n    from .pipeline_intake import load_generation_image\n    for src in m["sources"]:\n        probe, _ = load_generation_image(manifest_path.parent / src["path"],\n                                        allow_colour_conversion=allow_colour_conversion)\n        probe.close()\n    rows: list[dict] = []\n    runs = 0\n    for src in m["sources"]:\n        src_path = manifest_path.parent / src["path"]\n        source_templates=([get_template(k) for k in src[\'templates\']]\n                          if \'templates\' in src else templates)\n        contract=SourceContract(**src.get(\'source_contract\',{}))\n        for tpl in source_templates:\n            req = EditRequest(source_path=src_path, template=tpl,\n                              custom_instruction=src.get("instruction", ""),\n                              seed=cfg.seed)\n            result = run_pipeline(req, routes, cfg, editor=editor, scene_fn=scene_fn,\n                                  glass_mode=bool(src.get("glass", False)), source_contract=contract,\n                                  require_complete=bool(src.get(\'require_complete\',False)),\n                                  allow_colour_conversion=allow_colour_conversion,\n                                  scene_finish=scene_finish,\n                                  scene_accept=scene_accept, scene_max_attempts=scene_max_attempts)\n            runs += 1\n            source_record = json.loads(result.manifest_path.read_text()).get("source_provenance", {})\n            for v in result.variants:\n                row = report_to_csv_row(v.qa)\n                row.update({"source": str(src_path.resolve()),\n                            "source_label": src["label"],\n                            "source_sha256": source_record.get("sha256"),\n                            "source_classification": source_record.get("classification", "UNKNOWN / unclassified"),\n                            "source_match_basis": source_record.get("match_basis", "unrecorded"),\n                            "source_license": src["license"],\n                            "backend": v.backend,\n                            "seconds": v.seconds,\n                            "credits_estimated": v.credits_estimated,\n                            "success": v.success and v.output_path.is_file(),\n                            "scene_kind": scene_kind if v.route is Route.SCENE_COMPOSE else "",\n                            "source_contract": json.dumps(asdict(contract),sort_keys=True),\n                            "require_complete": bool(src.get(\'require_complete\',False))})\n                rows.append(row)\n\n    csv_path = out / "scores.csv"\n    if rows:\n        with csv_path.open("w", newline="") as f:\n            w = csv.DictWriter(f, fieldnames=sorted(rows[0].keys()))\n            w.writeheader()\n            w.writerows(rows)\n\n    # Gate view per the problem statement, with explicit numerators/denominators:\n    # - >=80% of outputs with no critical product alteration\n    # - >=70% publish-ready (fidelity 4-5 AND no critical flags; rubric 1-5)\n    # - >=8 templates represented in the run\n    # - config comparison: best route vs cheaper fallback by mean fidelity\n    # - every critical-flagged variant lands in the human-review queue\n    summary = _summarize(rows, runs)\n    if scene_kind:\n        summary["scene_kind"] = scene_kind + (" (MOCK gradient: plumbing only, not scene quality)" if scene_kind == "mock" else "")\n    (out / "benchmark_report.json").write_text(json.dumps(summary, indent=2))\n    return csv_path\n\n\ndef _summarize(rows: list[dict], runs: int, human: dict | None = None) -> dict:\n    """Shared gate math for fresh benchmark runs and summarize_runs - one\n    implementation so the two paths can never drift apart."""\n    summary: dict = {"runs": runs, "variants": len(rows), "routes": {},\n                     "templates_represented": sorted({r["template"] for r in rows}),\n                     "gate_status": "NOT EVALUATED on real data - these numbers "\n                     "come from whatever sources the manifest held; the plan\'s "\n                     "targets are achieved only on the full 20+ real-source run "\n                     "with human eye review of flagged outputs",\n                     "quality_status": "NOT_EVALUATED"}\n    known_sources = {r["source"] for r in rows if r.get("source")}\n    source_hashes = {r["source_sha256"] for r in rows if r.get("source_sha256")}\n    content_complete = bool(rows) and all(bool(r.get("source_sha256")) for r in rows)\n    classes = {}\n    for row in rows:\n        kind = row.get("source_classification") or "UNKNOWN / unclassified"\n        identity = row.get("source_sha256") or row.get("source") or "unrecorded"\n        classes.setdefault(kind, set()).add(identity)\n    summary["coverage"] = {\n        "unique_sources": len(known_sources), "source_target": 20,\n        "unique_content_sources": len(source_hashes),\n        "source_identity_complete": content_complete,\n        "source_count_met": content_complete and len(source_hashes) >= 20,\n        "source_count_basis": "distinct recorded file SHA256, not paths or verified photo capture origin; different encodings of one image can still count separately",\n        "source_classifications": {k: len(v) for k, v in sorted(classes.items())},\n        "real_photo_count_met": None,\n        "real_photo_status": "NOT_VERIFIED: photographic appearance does not establish capture origin. Illustrations and unknown inputs must not be called verified photographs.",\n        "templates": len(summary["templates_represented"]), "template_target": 8,\n        "template_count_met": len(summary["templates_represented"]) >= 8,\n    }\n    # Scope and specialist evidence are independent of per-sample ratings.\n    summary["acceptance_status"] = "NOT_EVALUATED"\n    summary["acceptance_pass"] = None\n    summary["specialist_comparison_status"] = "NOT_EVALUATED"\n    n = len(rows)\n    if n:\n        no_crit = sum(1 for r in rows if r.get("success", True) and not r["critical_errors"])\n        publish = sum(1 for r in rows\n                      if r.get("success", True) and not r["critical_errors"]\n                      and (r.get("score_fidelity") or 0) >= 4)\n        summary["gates"] = {\n            "no_critical_alteration": {"numerator": no_crit, "denominator": n,\n                                       "rate": round(no_crit / n, 3),\n                                       "target": 0.80,\n                                       "pass": None,\n                                       "status": "NOT_EVALUATED",\n                                       "automated_proxy_pass": (no_crit / n) >= 0.80,\n                                       "basis": "automated tripwires only; a human "\n                                                "eye audit can only LOWER this rate "\n                                                "(see README known QA gap)"},\n            "publish_ready": {"numerator": publish, "denominator": n,\n                              "rate": round(publish / n, 3), "target": 0.70,\n                              "pass": None,\n                              "status": "NOT_EVALUATED",\n                              "automated_proxy_pass": (publish / n) >= 0.70,\n                              "basis": "STRUCTURAL PROXY (auto fidelity>=4 and no "\n                                       "flags) - NOT a human publishability rating; "\n                                       "the PDF gate needs human-scored 1-5 "\n                                       "publishability on real outputs"},\n            "templates_represented": {\n                "numerator": len(summary["templates_represented"]),\n                "denominator": 8,\n                "pass": len(summary["templates_represented"]) >= 8},\n            "human_review_queue": [\n                {"output": r["output"], "template": r["template"],\n                 "flags": r["critical_errors"],\n                 "reason": ("GENERATIVE_PRODUCT_EDIT_ALWAYS_REQUIRES_FIDELITY_REVIEW"\n                            if r.get("route") in ("pure_prompt", "hybrid") else "AUTOMATED_WARNING_REQUIRES_REVIEW")}\n                for r in rows if r.get("success", True) and (r["critical_errors"] or r.get("route") in ("pure_prompt", "hybrid"))],\n            "execution_failure_queue": [\n                {"output": r["output"], "template": r["template"], "errors": r["critical_errors"],\n                 "status": "FAILED_NOT_A_QUALITY_REVIEW_CANDIDATE"}\n                for r in rows if not r.get("success", True)],\n        }\n    if human and n and all(r["output"] in human for r in rows):\n        # human[output] = {"no_critical": bool, "publishable": 1..5, "rater": R}\n        # R must be "owner" or "independent_human" to count as HUMAN_EVALUATED.\n        # "agent_eye" (an agent\'s own visual review) is recorded but is NOT a\n        # human rating: status becomes AGENT_EYE_REVIEWED and pass stays None.\n        raters = {human[r["output"]].get("rater", "unspecified") for r in rows}\n        hc = sum(1 for r in rows if r.get("success", True) and human[r["output"]]["no_critical"])\n        hp = sum(1 for r in rows if r.get("success", True) and human[r["output"]]["no_critical"]\n                 and human[r["output"]]["publishable"] >= 4)\n        g = summary["gates"]\n        summary["raters"] = sorted(raters)\n        if raters <= {"owner", "independent_human"}:\n            status, passes = "HUMAN_EVALUATED", (hc / n >= 0.80, hp / n >= 0.70)\n        elif raters == {"agent_eye"}:\n            status, passes = "AGENT_EYE_REVIEWED", (None, None)\n        else:\n            status, passes = "NOT_EVALUATED", (None, None)  # mixed/unknown raters\n        if status != "NOT_EVALUATED":\n            g["no_critical_alteration"].update(numerator=hc, rate=round(hc / n, 3),\n                                               status=status, **{"pass": passes[0]})\n            g["publish_ready"].update(numerator=hp, rate=round(hp / n, 3),\n                                      status=status, **{"pass": passes[1]})\n        summary["quality_status"] = status\n    import statistics\n    for route in {r["route"] for r in rows}:\n        sub = [r for r in rows if r["route"] == route]\n        scored = [r["score_fidelity"] for r in sub if r.get("success", True) and r.get("score_fidelity")]\n        crit = sum(1 for r in sub if r["critical_errors"])\n        # PDF economics: median latency, cost per successful output, retries.\n        times = [r["seconds"] for r in sub if r.get("seconds")]\n        successes = sum(1 for r in sub if r.get("success"))\n        credits = sum(r.get("credits_estimated", 0) for r in sub)\n        summary["routes"][route] = {\n            "n": len(sub),\n            "mean_fidelity": round(sum(scored) / len(scored), 2) if scored else None,\n            "critical_rate": round(crit / len(sub), 3),\n            "gate_80pct_no_critical": None,\n            "automated_proxy_80pct_no_critical": (crit / len(sub)) <= 0.20,\n            "quality_status": "NOT_EVALUATED",\n\n            "median_seconds": round(statistics.median(times), 2) if times else None,\n            "success_rate": round(successes / len(sub), 3),\n            "success_rate_basis": "executor completion, not publication usability or product fidelity",\n            "completed_outputs": successes,\n            "failed_attempts": len(sub) - successes,\n            "retries_needed": len(sub) - successes,\n            "retries_needed_basis": "legacy field: failed attempts only; actual retry count is not recorded",\n            "credits_total": credits,\n            "credits_basis": "sum of credits_estimated; not a verified billing ledger",\n            "credits_per_publishable_output": None,\n            "publishable_cost_status": "NOT_EVALUATED: completed outputs are not verified publishable outputs",\n            "credits_per_successful_output":\n                round(credits / successes, 2) if credits and successes else None,\n        }\n    ranked = sorted(((v["mean_fidelity"], k) for k, v in summary["routes"].items()\n                     if v["mean_fidelity"] is not None), reverse=True)\n    if ranked:\n        summary["config_comparison"] = {\n            "best_quality_route": None,\n            "fallback_route": None,\n            "automated_fidelity_leader": ranked[0][1],\n            "recommendation_status": "NOT_EVALUATED",\n            "note": "Automated fidelity ordering only, not a quality or value "\n                    "recommendation. A cheaper viable fallback needs human "\n                    "quality and measured cost evidence."}\n    return summary\n\n\ndef summarize_runs(run_dirs: list[Path], out: Path,\n                   source_meta: dict[str, dict] | None = None) -> Path:\n    """Aggregate EXISTING run manifests into the same gate report without\n    re-running anything. Lets us re-score after QA changes (e.g. the matte\n    tripwires) and reuse expensive GPU runs. source_meta maps manifest source\n    path -> {label, license}; missing metadata is marked, not invented."""\n    out.mkdir(parents=True, exist_ok=True)\n    rows: list[dict] = []\n    for rd in run_dirs:\n        manifest = rd / "manifest.json"\n        if not manifest.exists():\n            continue\n        m = json.loads(manifest.read_text())\n        meta = (source_meta or {}).get(m.get("source"), {})\n        for v in m.get("variants", []):\n            q = v.get("qa", {})\n            from .completion import saved_variant_failed\n            failed = saved_variant_failed(v)\n            row = {"source": m.get("source"),\n                   "source_label": meta.get("label", "(unlabeled)"),\n                   "source_sha256": m.get("source_provenance", {}).get("sha256"),\n                   "source_classification": m.get("source_provenance", {}).get("classification", "UNKNOWN / unclassified legacy run"),\n                   "source_match_basis": m.get("source_provenance", {}).get("match_basis", "unrecorded"),\n                   "source_license": meta.get("license", "(unrecorded)"),\n                   "route": v.get("route"), "template": q.get("template"),\n                   "seed": v.get("seed"), "backend": v.get("backend", ""),\n                   "seconds": v.get("seconds"),\n                   "credits_estimated": v.get("credits_estimated", 0),\n                   "success": not failed and bool(v.get("success", Path(v.get("output", "")).is_file())),\n                   "score_fidelity": None if failed else q.get("score_fidelity"),\n                   "critical_errors": q.get("critical_errors") or [],\n                   "output": v.get("output")}\n            rows.append(row)\n    csv_path = out / "scores.csv"\n    if rows:\n        with csv_path.open("w", newline="") as f:\n            w = csv.DictWriter(f, fieldnames=sorted(rows[0].keys()))\n            w.writeheader()\n            w.writerows(rows)\n    summary = _summarize(rows, len(run_dirs))\n    (out / "benchmark_report.json").write_text(json.dumps(summary, indent=2))\n    return csv_path\n'
# ---- embedded module: renderello/benchmark.py ----
path = base / 'renderello/benchmark.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Explicit pinned BiRefNet-lite ONNX CPU experiment. No prompts/downloads."""\nfrom pathlib import Path\nimport hashlib,json,time\nimport numpy as np\nfrom PIL import Image\n\nMODEL_URL=\'https://github.com/danielgatis/rembg/releases/download/v0.0.0/BiRefNet-general-bb_swin_v1_tiny-epoch_232.onnx\'\nMODEL_SHA256=\'5600024376f572a557870a5eb0afb1e5961636bef4e1e22132025467d0f03333\'\nORT_VERSION=\'1.31.1\'\n\n\ndef _predict(rgb,checkpoint):\n    import importlib.metadata\n    if importlib.metadata.version(\'onnxruntime\')!=ORT_VERSION:raise RuntimeError(\'requires onnxruntime==\'+ORT_VERSION)\n    import onnxruntime as ort\n    opts=ort.SessionOptions();opts.enable_cpu_mem_arena=False;opts.enable_mem_pattern=False\n    opts.graph_optimization_level=ort.GraphOptimizationLevel.ORT_DISABLE_ALL\n    opts.intra_op_num_threads=2;opts.inter_op_num_threads=1\n    session=ort.InferenceSession(str(checkpoint.resolve()),sess_options=opts,providers=[\'CPUExecutionProvider\'])\n    # Match rembg conversion normalization, including max-channel normalization.\n    x=np.array(rgb.resize((1024,1024),Image.Resampling.LANCZOS)).astype(np.float32)\n    x/=max(float(x.max()),1e-6)\n    x=(x-np.array([.485,.456,.406],np.float32))/np.array([.229,.224,.225],np.float32)\n    raw=session.run(None,{session.get_inputs()[0].name:x.transpose(2,0,1)[None]})[0][:,0,:,:]\n    return normalize_logits(raw,rgb.size)\n\n\ndef normalize_logits(raw,size):\n    if raw.shape!=(1,1024,1024) or not np.isfinite(raw).all():raise ValueError(\'invalid logits shape or nonfinite values\')\n    pred=1/(1+np.exp(-np.clip(raw,-80,80)))\n    span=float(pred.max()-pred.min())\n    if span<1e-8:raise ValueError(\'constant prediction; no valid foreground\')\n    pred=(pred-pred.min())/span\n    return np.array(Image.fromarray((pred.squeeze()*255).astype(np.uint8)).resize(size,Image.Resampling.LANCZOS))\n\n\ndef infer_file(source:Path,out:Path,checkpoint:Path):\n    if out.exists():raise ValueError(\'output exists; preserve earlier attempts\')\n    with Image.open(source) as im:\n        if im.width*im.height>4_000_000:raise ValueError(\'source exceeds 4 million pixels; no silent source resize\')\n        if min(im.size)<3:raise ValueError(\'source too small\')\n        rgb=im.convert(\'RGB\');pixels=np.array(rgb)\n    if not checkpoint.is_file():raise ValueError(\'checkpoint missing; explicitly download the official free conversion first\')\n    if hashlib.sha256(checkpoint.read_bytes()).hexdigest()!=MODEL_SHA256:raise ValueError(\'checkpoint SHA256 mismatch\')\n    started=time.monotonic()\n    try:alpha=_predict(rgb,checkpoint)\n    except ImportError as e:raise RuntimeError(\'install optional requirements-bilateral.txt in an isolated Renderello environment\') from e\n    if alpha.dtype!=np.uint8 or alpha.shape!=pixels.shape[:2] or not np.any(alpha):raise ValueError(\'invalid or empty inferred alpha\')\n    out.mkdir(parents=True)\n    for name,data in [(\'source.png\',pixels),(\'estimated-alpha.png\',alpha),(\'cutout.png\',np.dstack([pixels,alpha]))]:Image.fromarray(data).save(out/name)\n    for label,color in [(\'black\',(20,20,20)),(\'white\',(250,250,250)),(\'magenta\',(190,0,150))]:\n        bg=Image.new(\'RGB\',rgb.size,color);bg.paste(rgb,mask=Image.fromarray(alpha));bg.save(out/(label+\'-review.png\'))\n    report=dict(model=\'BiRefNet general-lite\',conversion=\'rembg-maintained ONNX, not original author checkpoint bytes\',model_url=MODEL_URL,model_sha256=MODEL_SHA256,\n        runtime=\'onnxruntime\',runtime_version=ORT_VERSION,provider=\'CPUExecutionProvider\',internal_input=[1024,1024],graph_optimizations=False,cpu_mem_arena=False,mem_pattern=False,\n        source_sha256=hashlib.sha256(source.read_bytes()).hexdigest(),source_rgb_unchanged=True,semantic_completeness=\'NOT_VERIFIED\',source_framing=\'NOT_VERIFIED\',quality=\'NOT_EVALUATED\',seconds=time.monotonic()-started,\n        alpha_kind=\'dichotomous segmentation, not optical glass matting\',\n        warnings=[\'Masks can omit wanted objects, include unwanted props, or bridge foreground and backdrop. Review each source separately.\',\n          \'Source RGB is preserved, including source-color fringes and reflections. Cropped parts cannot be recovered.\',\n          \'This model was killed in the smaller research environment. Successful CPU tests used Kaggle. Owner-PC resources and execution unverified.\'],\n        sources=[\'https://github.com/ZhengPeng7/BiRefNet\',\'https://huggingface.co/ZhengPeng7/BiRefNet_lite\',\'https://github.com/danielgatis/rembg\'])\n    report[\'output_sha256\']={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in out.iterdir() if p.is_file()}\n    (out/\'bilateral-mask-manifest.json\').write_text(json.dumps(report,indent=2));return report\n\n\ndef guarded_file(source,out,checkpoint):\n    """Separate CPU worker, 180s timeout and Linux-only 8GiB RSS guard."""\n    import subprocess,sys,tempfile,os\n    # Standalone module script keeps core CLI imports out of the inference worker.\n    cmd=[sys.executable,str(Path(__file__).resolve()),\'--source\',str(source),\'--out\',str(out),\'--checkpoint\',str(checkpoint)]\n    env=os.environ.copy();env.update(OMP_NUM_THREADS=\'2\',MKL_NUM_THREADS=\'2\')\n    with tempfile.TemporaryFile(mode=\'w+\') as stdout,tempfile.TemporaryFile(mode=\'w+\') as stderr:\n        p=subprocess.Popen(cmd,stdout=stdout,stderr=stderr,text=True,env=env);start=time.monotonic()\n        try:\n            while p.poll() is None:\n                if time.monotonic()-start>180:raise RuntimeError(\'bilateral-mask 180 second timeout; attempt not accepted\')\n                try:rss=int(next(l.split()[1] for l in Path(f\'/proc/{p.pid}/status\').read_text().splitlines() if l.startswith(\'VmRSS:\')))\n                except (FileNotFoundError,StopIteration):rss=0\n                if rss>8192*1024:raise RuntimeError(\'bilateral-mask 8GiB RSS limit; attempt not accepted\')\n                time.sleep(.1)\n        finally:\n            if p.poll() is None:\n                p.terminate()\n                try:p.wait(timeout=5)\n                except subprocess.TimeoutExpired:p.kill();p.wait()\n        stdout.seek(0);stderr.seek(0)\n        if p.returncode:raise RuntimeError(\'bilateral-mask worker failed (exit \'+str(p.returncode)+\'): \'+stderr.read()[-2000:])\n        return json.load(stdout)\n\n\ndef main():\n    import argparse\n    p=argparse.ArgumentParser(description=__doc__)\n    for name in (\'source\',\'out\',\'checkpoint\'):p.add_argument(\'--\'+name,type=Path,required=True)\n    a=p.parse_args();print(json.dumps(infer_file(a.source,a.out,a.checkpoint)))\nif __name__==\'__main__\':main()\n'
# ---- embedded module: renderello/bilateral_mask.py ----
path = base / 'renderello/bilateral_mask.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Build a sealed blank blind pack from an explicit local JSON manifest."""\nimport argparse,json\nfrom pathlib import Path\nfrom .blind_review import build_pack,Candidate\n\n\ndef main(argv=None):\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--manifest\',type=Path,required=True);p.add_argument(\'--out\',type=Path,required=True)\n    a=p.parse_args(argv)\n    try:\n        if a.manifest.is_symlink() or not a.manifest.is_file() or a.manifest.stat().st_size>20000:raise ValueError(\'bounded manifest required\')\n        m=json.loads(a.manifest.read_text())\n        if not isinstance(m,dict) or set(m)-{\'source\',\'seed\',\'candidates\',\'regions\'}:raise ValueError(\'exact pack manifest fields required\')\n        root=a.manifest.resolve().parent\n        def path(v):return root/Path(v)\n        cs=[]\n        for c in m[\'candidates\']:\n            if not isinstance(c,dict) or set(c)-{\'method\',\'path\',\'failure\'}:raise ValueError(\'unknown candidate fields\')\n            cs.append(Candidate(c[\'method\'],None if c.get(\'path\') is None else path(c[\'path\']),c.get(\'failure\',\'\')))\n        out=build_pack(path(m[\'source\']),cs,a.out,seed=m[\'seed\'],regions=m.get(\'regions\'))\n    except (ValueError,TypeError,KeyError,OSError) as exc:p.exit(2,\'REFUSED: \'+str(exc)+\'\\n\')\n    print(\'Saved sealed blank pack: \'+str(out)+\'; share REVIEW only, never PRIVATE. No scores inferred.\')\n\n\nif __name__==\'__main__\':main()\n'
# ---- embedded module: renderello/blind_pack_cli.py ----
path = base / 'renderello/blind_pack_cli.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Local anonymized review packs. No ratings or human identity are inferred.\n\nCandidate order is seeded for reproducibility, not cryptographic secrecy. The\nprivate key must be kept away from raters. Missing/failed outputs remain entries.\nFull-resolution PNG derivatives accompany scaled overviews for edge/print review.\n"""\nfrom __future__ import annotations\nfrom dataclasses import dataclass\nfrom pathlib import Path\nimport csv\nimport hashlib\nimport json\nimport random\nfrom PIL import Image, ImageDraw\nfrom .image_intake import inspect_image, IntakePolicy\nfrom .review import DIMENSIONS\n\n\n@dataclass(frozen=True)\nclass Candidate:\n    method: str\n    path: Path | None\n    failure: str = \'\'\n\n    def __post_init__(self):\n        if not isinstance(self.method, str) or not self.method.strip():\n            raise ValueError(\'Method identity required in private key\')\n        if not isinstance(self.failure, str):\n            raise ValueError(\'Failure must be text\')\n        if self.path is None and not self.failure.strip():\n            raise ValueError(\'Missing output requires an explicit failure\')\n        if self.path is not None and self.failure:\n            raise ValueError(\'Candidate cannot both succeed and fail\')\n\n\ndef _sha(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef build_pack(source: Path, candidates: list[Candidate], out: Path, *, seed: int,\n               regions: dict[str, tuple[int, int, int, int]] | None = None) -> Path:\n    """Build one source comparison; invalid input is not silently skipped.\n\n    Regions use the shared full-image coordinate system. Candidates with different\n    dimensions are refused rather than making an ungrounded registration/print\n    fidelity claim. More general layouts need their own declared transforms.\n    """\n    if not 2 <= len(candidates) <= 12:\n        raise ValueError(\'Review pack requires 2-12 candidate entries\')\n    if len({c.method for c in candidates}) != len(candidates):\n        raise ValueError(\'Duplicate method identity\')\n    if isinstance(seed, bool) or not isinstance(seed, int):\n        raise ValueError(\'Integer seed required\')\n    if out.exists():\n        raise FileExistsError(\'Never replace an earlier review\')\n    source = Path(source)\n    original_hash = _sha(source)\n    policy = IntakePolicy(allow_colour_conversion=True)\n    src = inspect_image(source, policy)\n    order = list(candidates)\n    random.Random(seed).shuffle(order)\n    decoded = []\n    seen = set()\n    decoded_seen = set()\n    try:\n        for c in order:\n            if c.path is None:\n                decoded.append((c, None, None));continue\n            path = Path(c.path)\n            digest = _sha(path)\n            if digest in seen:\n                raise ValueError(\'Duplicate output content cannot inflate the comparison\')\n            seen.add(digest)\n            image = inspect_image(path, policy)\n            if image.image.size != src.image.size:\n                image.close()\n                raise ValueError(\'Candidate/source dimensions differ; registration is not established\')\n            canonical = image.image.convert(\'RGBA\')\n            try:\n                pixel_digest = hashlib.sha256(canonical.tobytes()).hexdigest()\n            finally:\n                canonical.close()\n            if pixel_digest in decoded_seen:\n                image.close()\n                raise ValueError(\'Duplicate output pixels cannot inflate the comparison\')\n            decoded_seen.add(pixel_digest)\n            decoded.append((c, image, digest))\n        regions = regions or {}\n        for label, box in regions.items():\n            if not label.strip() or len(box) != 4 or any(isinstance(x, bool) or not isinstance(x, int) for x in box):\n                raise ValueError(\'Named integer ROI required\')\n            x0,y0,x1,y1=box;w,h=src.image.size\n            if not (0 <= x0 < x1 <= w and 0 <= y0 < y1 <= h):\n                raise ValueError(\'ROI outside shared image coordinates\')\n        if _sha(source) != original_hash:\n            raise ValueError(\'Source changed during review preparation\')\n        for c,im,digest in decoded:\n            if im is not None and _sha(c.path) != digest:\n                raise ValueError(\'Candidate changed during review preparation\')\n        out.mkdir(parents=True)\n        # A pack is consumable only after COMPLETE.json exists and hashes verify.\n        (out/\'PRIVATE\').mkdir();(out/\'REVIEW\').mkdir()\n        src.image.save(out/\'REVIEW/source.png\')\n        rows=[];key=[];tiles=[(\'SOURCE\',src.image)]\n        for i,(c,im,digest) in enumerate(decoded,1):\n            token=\'C\'+str(i).zfill(2)\n            key.append(dict(token=token,method=c.method,path=str(c.path) if c.path else None,\n                            original_sha256=digest,failure=c.failure))\n            if im is not None:\n                im.image.save(out/\'REVIEW\'/f\'{token}.png\')\n                for j,box in enumerate(regions.values(),1):\n                    im.image.crop(box).save(out/\'REVIEW\'/f\'{token}-region{j}.png\')\n            tiles.append((token,im.image if im else None))\n            rows.append(dict(token=token,status=\'OUTPUT_PRESENT\' if im else \'NOT_GENERATED\',\n                             **{name:\'\' for name in DIMENSIONS},no_critical=\'\',critical_errors=\'\',rater=\'\',notes=\'\'))\n        for j,box in enumerate(regions.values(),1):\n            src.image.crop(box).save(out/\'REVIEW\'/f\'source-region{j}.png\')\n        board=Image.new(\'RGB\',(1200,430*((len(tiles)+2)//3)+40),\'white\');draw=ImageDraw.Draw(board)\n        draw.text((12,10),\'Scaled overview only. Inspect full-resolution images and native crops before rating.\',fill=\'black\')\n        for i,(token,im) in enumerate(tiles):\n            x=(i%3)*400;y=40+(i//3)*430;draw.text((x+10,y),token,fill=\'black\')\n            if im is None:\n                draw.text((x+10,y+50),\'NOT GENERATED / retained in denominator\',fill=\'red\');continue\n            # Checkerboard makes partial alpha visible, not a fake opaque white pass.\n            checker=Image.new(\'RGB\',(380,390),\'#ddd\');cd=ImageDraw.Draw(checker)\n            for yy in range(0,390,20):\n                for xx in range(0,380,20):\n                    if (xx//20+yy//20)%2:cd.rectangle((xx,yy,xx+19,yy+19),fill=\'#aaa\')\n            preview=im.copy();preview.thumbnail((380,390))\n            px=(380-preview.width)//2;py=(390-preview.height)//2\n            checker.paste(preview,(px,py),preview.getchannel(\'A\') if preview.mode==\'RGBA\' else None)\n            board.paste(checker,(x+10,y+25));preview.close()\n        board.save(out/\'REVIEW/overview.png\');board.close()\n        fields=list(rows[0])\n        with (out/\'REVIEW/scores.csv\').open(\'x\',newline=\'\') as f:\n            writer=csv.DictWriter(f,fieldnames=fields);writer.writeheader();writer.writerows(rows)\n        (out/\'PRIVATE/key.json\').write_text(json.dumps(dict(seed=seed,source_path=str(source),source_sha256=original_hash,candidates=key,regions=regions),indent=2))\n        notes=(\'Share only REVIEW, never PRIVATE, with raters. Token assignment is randomized but not proof of independent blinding. \'\n               \'No automatic score, winner or acceptance is inferred. Missing outputs stay in the denominator. \'\n               \'ICC conversion may alter colours; full-resolution images are decoded normalized derivatives, not original encoded bytes. \'\n               \'Independent rater identity must be confirmed outside this file.\\n\')\n        (out/\'REVIEW/READ_ME.txt\').write_text(notes)\n        hashes={str(p.relative_to(out)):_sha(p) for p in (out/\'REVIEW\').iterdir() if p.is_file()}\n        (out/\'COMPLETE.json\').write_text(json.dumps(dict(schema_version=2,private_key_sha256=_sha(out/\'PRIVATE/key.json\'),candidates=len(candidates),outputs=sum(im is not None for _,im,_ in decoded),ratings=\'BLANK\',acceptance=\'NOT_EVALUATED\',review_sha256=hashes),indent=2))\n        return out\n    finally:\n        src.close()\n        for _,im,_ in decoded:\n            if im is not None:im.close()\n\n\ndef verify_pack(out: Path) -> dict:\n    """Reject incomplete or altered packs without accepting any ratings."""\n    record=json.loads((out/\'COMPLETE.json\').read_text())\n    if record.get(\'schema_version\') == 2:\n        key_path = out/\'PRIVATE/key.json\'\n        if key_path.is_symlink() or key_path.resolve().parent != (out/\'PRIVATE\').resolve() or _sha(key_path) != record.get(\'private_key_sha256\'):\n            raise ValueError(\'Private candidate key changed or is missing\')\n    expected=record[\'review_sha256\']\n    if not isinstance(expected, dict) or not expected:\n        raise ValueError(\'Empty or invalid review manifest\')\n    review = (out/\'REVIEW\').resolve()\n    for name, digest in expected.items():\n        path = out / name\n        if path.is_symlink() or path.resolve().parent != review:\n            raise ValueError(\'Review manifest path escapes review directory\')\n        if _sha(path) != digest:\n            raise ValueError(\'Review content changed or is missing\')\n    actual = {str(p.relative_to(out)) for p in (out/\'REVIEW\').iterdir()}\n    if actual != set(expected):\n        raise ValueError(\'Unexpected review files\')\n    return record\n'
# ---- embedded module: renderello/blind_review.py ----
path = base / 'renderello/blind_review.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Import a separate returned score sheet without modifying the sealed pack.\n\nNo reviewer identity, independent review, product acceptance or winner is inferred.\nReports contain the private method key and must not be sent back to blind raters.\n"""\nfrom pathlib import Path\nimport argparse\nimport csv\nimport hashlib\nimport io\nimport json\nfrom .blind_review import verify_pack\nfrom .brief_acceptance import Rating, evaluate\nfrom .review import DIMENSIONS\n\n\ndef _read(path):\n    path = Path(path)\n    if path.is_symlink() or not path.is_file() or path.stat().st_size > 2_000_000:\n        raise ValueError(\'bounded regular file required\')\n    return path.read_bytes()\n\n\ndef import_return(pack, returned_csv, *, provenance, review_reference):\n    """Return hash-bound judgments and explicit missing-review entries.\n\n    provenance is supplied by the caller from trusted context, never selected by\n    a rater\'s text in a CSV. claimed_independent records a claim, not verification.\n    All tokens must appear exactly once, including failed generations. Blank or\n    partial rows stay incomplete; invalid values fail closed, never become zero.\n    Critical errors use a JSON string array, including explicit [] for none.\n    """\n    if provenance not in (\'assistant_judgment\', \'owner\', \'claimed_independent\'):\n        raise ValueError(\'explicit review provenance required\')\n    if not isinstance(review_reference, str) or not review_reference.strip() or len(review_reference) > 200:\n        raise ValueError(\'bounded review reference required\')\n    pack, returned_csv = Path(pack), Path(returned_csv)\n    sealed = verify_pack(pack)\n    if sealed.get(\'schema_version\') != 2:\n        raise ValueError(\'a schema-2 pack with a sealed private key is required; legacy packs remain unchanged\')\n    raw = _read(returned_csv)\n    if returned_csv.resolve().is_relative_to(pack.resolve()):\n        raise ValueError(\'returned score sheet must be separate from sealed pack\')\n    original = list(csv.DictReader(io.StringIO(_read(pack/\'REVIEW/scores.csv\').decode(\'utf-8-sig\'))))\n    key = json.loads(_read(pack/\'PRIVATE/key.json\'))\n    candidates = key[\'candidates\']\n    lookup = {c[\'token\']: c for c in candidates}\n    expected = {r[\'token\']: r for r in original}\n    if len(lookup) != len(candidates) or set(lookup) != set(expected):\n        raise ValueError(\'private key does not match sealed score tokens\')\n    reader = csv.DictReader(io.StringIO(raw.decode(\'utf-8-sig\')))\n    headers = list(original[0])\n    if reader.fieldnames != headers:\n        raise ValueError(\'returned score columns/order must match sealed sheet exactly\')\n    rows = list(reader)\n    if len(rows) != len(expected) or len({r.get(\'token\') for r in rows}) != len(rows) or {r.get(\'token\') for r in rows} != set(expected):\n        raise ValueError(\'each planned token must appear exactly once, including failures\')\n    results=[]\n    for row in rows:\n        if None in row or any(v is None for v in row.values()):\n            raise ValueError(\'malformed score row\')\n        token = row[\'token\']; c = lookup[token]\n        if row[\'status\'] != expected[token][\'status\']:\n            raise ValueError(\'generation status is sealed and cannot be changed by returned ratings\')\n        out = dict(token=token, method=c[\'method\'], original_output_sha256=c[\'original_sha256\'],\n                   generation_status=row[\'status\'], failure=c[\'failure\'],\n                   rater_text_unverified=row[\'rater\'], notes=row[\'notes\'], rating=None)\n        provided = {d: row[d].strip() for d in DIMENSIONS}\n        for v in provided.values():\n            if v and v not in (\'1\',\'2\',\'3\',\'4\',\'5\'):\n                raise ValueError(\'scores must be blank or integer 1..5\')\n        decision = row[\'no_critical\'].strip().lower()\n        if decision not in (\'\',\'true\',\'false\'):\n            raise ValueError(\'no_critical must be blank, true or false\')\n        error_text = row[\'critical_errors\'].strip()\n        errors = None if not error_text else json.loads(error_text)\n        if errors is not None and (not isinstance(errors,list) or any(not isinstance(x,str) or not x.strip() for x in errors)):\n            raise ValueError(\'critical_errors must be a JSON string array\')\n        if decision and errors is not None and (decision == \'true\') == bool(errors):\n            raise ValueError(\'critical decision conflicts with errors\')\n        any_judgment = any(provided.values()) or decision or error_text\n        if row[\'status\'] == \'NOT_GENERATED\':\n            if any_judgment: raise ValueError(\'failed generation cannot receive image scores\')\n            out[\'review_status\']=\'NOT_GENERATED_RETAINED_IN_DENOMINATOR\'\n        elif not all(provided.values()) or not decision or errors is None or not row[\'rater\'].strip():\n            out[\'review_status\']=\'INCOMPLETE_REVIEW\'\n            out[\'missing_fields\']=[d for d,v in provided.items() if not v]\n            out[\'missing_fields\'] += [name for name,ok in [(\'no_critical\',bool(decision)),(\'critical_errors\',errors is not None),(\'rater\',bool(row[\'rater\'].strip()))] if not ok]\n        else:\n            rating=Rating(token,c[\'original_sha256\'],{d:int(v) for d,v in provided.items()},decision==\'true\',tuple(errors),provenance,review_reference+\':\'+token)\n            out[\'review_status\']=\'COMPLETE_RECORDED_JUDGMENT\'\n            out[\'rating\']=dict(output_id=rating.output_id,output_sha256=rating.output_sha256,scores=dict(rating.scores),no_critical=rating.no_critical,critical_errors=list(rating.critical_errors),provenance=rating.provenance,review_reference=rating.review_reference)\n        results.append(out)\n    # Recheck sealed content after parsing. A hash proves consistency, not origin.\n    verify_pack(pack)\n    return dict(schema=\'blind-review-return.v1\',returned_csv_sha256=hashlib.sha256(raw).hexdigest(),\n                private_key_sha256=sealed[\'private_key_sha256\'],source_sha256=key[\'source_sha256\'],review_reference=review_reference,\n                provenance=provenance,denominator=len(candidates),\n                complete_reviews=sum(r[\'rating\'] is not None for r in results),results=results,\n                identity_semantics=\'caller_provenance_and_rater_text_not_identity_authentication\',\n                disclosure=\'PRIVATE_METHOD_KEY_INCLUDED_KEEP_AWAY_FROM_BLIND_RATERS\',\n                best_quality_model=None,value_fallback=None,production_go=False,live_calls=0)\n\n\n\ndef bind_ratings(return_report, plan, attempts, token_output_ids):\n    """Bind selected returned tokens to one method\'s preregistered attempt plan.\n\n    An explicit map avoids guessing template/alternative from a filename. This\n    validates consistency with the supplied report, not the report\'s authorship.\n    Call import_return on the sealed local pack first. Never reassign a judgment\n    to another native image, source or method, even when scores look plausible.\n    """\n    evaluate(plan, attempts, [])\n    if not isinstance(token_output_ids, dict) or not token_output_ids:\n        raise ValueError(\'explicit nonempty token-to-output mapping required\')\n    if len(set(token_output_ids.values())) != len(token_output_ids):\n        raise ValueError(\'one returned judgment cannot pad multiple planned slots\')\n    if return_report.get(\'schema\') != \'blind-review-return.v1\':\n        raise ValueError(\'returned-review report required\')\n    rows={r[\'token\']:r for r in return_report[\'results\']}\n    by_id={p.output_id:p for p in plan}; result=[]\n    for token, output_id in token_output_ids.items():\n        if token not in rows or output_id not in by_id:\n            raise ValueError(\'unknown review token or unplanned output\')\n        row=rows[token]; planned=by_id[output_id]\n        if planned.method != row[\'method\'] or planned.source_sha256 != return_report[\'source_sha256\']:\n            raise ValueError(\'returned judgment source/method does not match plan\')\n        candidate_attempts=[a for a in attempts if a.output_id==output_id]\n        if row[\'rating\'] is None:\n            if row[\'generation_status\']==\'NOT_GENERATED\' and any(a.status==\'success\' for a in candidate_attempts):\n                raise ValueError(\'failed review entry conflicts with successful attempt\')\n            continue\n        r=row[\'rating\']\n        if r[\'output_id\'] != token or r[\'output_sha256\'] != row[\'original_output_sha256\']:\n            raise ValueError(\'returned judgment does not bind the native candidate\')\n        result.append(Rating(output_id,r[\'output_sha256\'],r[\'scores\'],r[\'no_critical\'],tuple(r[\'critical_errors\']),r[\'provenance\'],r[\'review_reference\']))\n    # Existing evaluator enforces original hash/first-try/no-substituted-retry rules.\n    evaluate(plan, attempts, result)\n    return result\n\n\ndef main(argv=None):\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--pack\',type=Path,required=True)\n    p.add_argument(\'--returned-csv\',type=Path,required=True)\n    p.add_argument(\'--provenance\',required=True,choices=[\'assistant_judgment\',\'owner\',\'claimed_independent\'])\n    p.add_argument(\'--review-reference\',required=True)\n    p.add_argument(\'--output\',type=Path,required=True)\n    a=p.parse_args(argv)\n    try:\n        if a.output.resolve().is_relative_to(a.pack.resolve()):\n            raise ValueError(\'output must be outside sealed pack\')\n        result=import_return(a.pack,a.returned_csv,provenance=a.provenance,review_reference=a.review_reference)\n        with a.output.open(\'x\') as f:json.dump(result,f,indent=2)\n    except (ValueError,TypeError,KeyError,OSError,UnicodeError,csv.Error) as exc:\n        p.exit(2,\'REFUSED: \'+str(exc)+\'\\n\')\n    print(\'Saved private returned-review report: \'+str(a.output))\n\n\nif __name__==\'__main__\':main()\n'
# ---- embedded module: renderello/blind_review_return.py ----
path = base / 'renderello/blind_review_return.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Boundary review helper: crops the product edge band at full resolution.\n\nProduces a side-by-side strip (final | raw scene | mask band) per sample so a\nhuman can check halos, leftover silhouettes and contact shadows. The numbers\nare only screens; the strips are for the eye.\n"""\nfrom __future__ import annotations\nimport numpy as np\nfrom PIL import Image\nimport cv2\n\n\ndef edge_band(alpha: np.ndarray, inner: int = 3, outer: int = 12) -> np.ndarray:\n    a = (alpha > 127).astype(np.uint8)\n    k = lambda r: cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * r + 1, 2 * r + 1))\n    return (cv2.dilate(a, k(outer)) > 0) & ~(cv2.erode(a, k(inner)) > 0) & ~(a > 0) | \\\n           ((a > 0) & ~(cv2.erode(a, k(inner)) > 0))\n\n\ndef band_stats(final: np.ndarray, raw: np.ndarray, alpha: np.ndarray) -> dict:\n    band = edge_band(alpha)\n    outside = band & ~(alpha > 127)\n    if outside.sum() == 0:\n        return {"band_px": 0}\n    d = np.abs(final[..., :3].astype(np.int16) - raw[..., :3].astype(np.int16)).mean(-1)\n    return {"band_px": int(band.sum()), "outside_px": int(outside.sum()),\n            "mean_abs_final_vs_raw_outside": round(float(d[outside].mean()), 2)}\n\n\ndef boundary_strip(final: Image.Image, raw: Image.Image, alpha: np.ndarray,\n                   bbox: tuple[int, int, int, int], zoom: int = 3, pad: int = 24) -> Image.Image:\n    x0, y0, x1, y1 = bbox\n    H, W = alpha.shape\n    x0, y0, x1, y1 = max(0, x0 - pad), max(0, y0 - pad), min(W, x1 + pad), min(H, y1 + pad)\n    band = (edge_band(alpha) * 255).astype(np.uint8)\n    panels = []\n    for im in (np.array(final.convert("RGB")), np.array(raw.convert("RGB")),\n               np.stack([band] * 3, -1)):\n        c = im[y0:y1, x0:x1]\n        panels.append(cv2.resize(c, None, fx=zoom, fy=zoom, interpolation=cv2.INTER_NEAREST))\n    return Image.fromarray(np.concatenate(panels, axis=1))\n\n\ndef frame_edge_contact(matte, margin_px: int = 2, min_run_frac: float = 0.04) -> dict:\n    """Detect a product cut off by the photo frame.\n\n    The v0.4.9 GPU matrix (watch1) produced hard rectangular strap ends because the\n    product touches the source frame: the cut IS the photo edge, not a matting error.\n    Returns {side: fraction_of_that_side_covered} for sides where foreground (>127)\n    covers at least min_run_frac of the side within margin_px of it, plus\n    \'touches\' (list) and \'needs_review\' (True when any side touches).\n    """\n    import numpy as np\n    m = np.asarray(matte) > 127\n    h, w = m.shape[:2]\n    k = max(1, int(margin_px))\n    sides = {"top": m[:k, :].any(axis=0).mean(), "bottom": m[-k:, :].any(axis=0).mean(),\n             "left": m[:, :k].any(axis=1).mean(), "right": m[:, -k:].any(axis=1).mean()}\n    sides = {s: round(float(v), 4) for s, v in sides.items()}\n    touches = [s for s, v in sides.items() if v >= min_run_frac]\n    return {**sides, "touches": touches, "needs_review": bool(touches)}\n'
# ---- embedded module: renderello/boundary.py ----
path = base / 'renderello/boundary.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Free boundary/contact control. Native source RGB and reviewed label cores stay locked.\nNot a physical renderer, automatic mask reviewer, or transparent-material recovery.\n"""\nfrom pathlib import Path\nimport hashlib,json\nimport numpy as np\nfrom PIL import Image,ImageDraw\nfrom scipy.ndimage import distance_transform_edt,gaussian_filter\n\n\ndef refine_alpha(alpha, protected_regions, *, trim=0.7, feather=1.4, contour=None):\n    a=np.asarray(alpha.convert(\'L\'))\n    if trim<0 or feather<=0:raise ValueError(\'invalid boundary width\')\n    mask=a>0\n    if contour is not None:\n        c=Image.new(\'L\',alpha.size);ImageDraw.Draw(c).polygon(contour,fill=255)\n        mask &= np.array(c)>0\n    dist=distance_transform_edt(mask)\n    result=np.minimum(a,np.clip((dist-trim)/feather,0,1)*255).astype(\'uint8\')\n    for box in protected_regions.values():\n        x1,y1,x2,y2=box\n        if not (0<=x1<x2<=a.shape[1] and 0<=y1<y2<=a.shape[0]):raise ValueError(\'invalid protected box\')\n        if np.any(a[y1:y2,x1:x2]!=255):raise ValueError(\'protected region not opaque\')\n        if np.any(~mask[y1:y2,x1:x2]):raise ValueError(\'contour crosses protected region\')\n        result[y1:y2,x1:x2]=255\n    return Image.fromarray(result)\n\n\ndef compose_contact(source,alpha,output,*,protected_regions,offset,canvas=(1024,1024)):\n    output=Path(output)\n    if output.exists():raise ValueError(\'preserve previous output\')\n    rgb=np.array(source.convert(\'RGB\'));a=np.array(alpha.convert(\'L\'))\n    if rgb.shape[:2]!=a.shape or rgb.shape[0]*rgb.shape[1]>4000000:raise ValueError(\'invalid source/mask\')\n    if len(offset)!=2 or any(type(v)!=int for v in offset):raise ValueError(\'integer translation required\')\n    if not protected_regions:raise ValueError(\'reviewed label/texture regions required\')\n    dx,dy=offset;h,w=canvas[1],canvas[0];ys,xs=np.nonzero(a)\n    if not len(xs) or min(xs)+dx<0 or min(ys)+dy<0 or max(xs)+dx>=w or max(ys)+dy>=h:raise ValueError(\'empty or clipped source\')\n    yy,xx=np.mgrid[:h,:w];grad=np.clip((yy/h)*8+((xx-w*.4)/w)**2*8,0,15)\n    bg=np.stack([242-grad,239-grad,234-grad],axis=-1)\n    # Support comes from the lowest reviewed silhouette, not its box centre.\n    opaque=a>=128;bottom=np.full(a.shape[1],-1)\n    for x in np.where(opaque.any(axis=0))[0]:bottom[x]=np.where(opaque[:,x])[0][-1]\n    low=int(bottom.max());support_x=np.where(bottom>=low-5)[0]\n    cx=float(support_x.mean()+dx);cy=low+dy\n    support_width=max(float(np.ptp(support_x)),12)\n    # A compact contact term plus a soft directional tabletop shadow.\n    near=np.exp(-((xx-cx)/(support_width*.60+5))**2-((yy-cy-1)/3.8)**2)*.28\n    broad=np.exp(-((xx-cx-15)/(max(40,support_width*.85)))**2-((yy-cy-5)/12)**2)*.12\n    shade=np.clip(gaussian_filter(near,1)+gaussian_filter(broad,4),0,.40)\n    bg=bg*(1-shade[...,None])+np.array([70,65,60])*shade[...,None]\n    base=Image.fromarray(np.round(bg).astype(\'uint8\'));base.paste(source.convert(\'RGB\'),(dx,dy),alpha)\n    seen=np.array(base);checks=[]\n    for name,box in protected_regions.items():\n        x1,y1,x2,y2=box\n        if np.any(a[y1:y2,x1:x2]!=255):raise ValueError(\'protected region must be opaque\')\n        err=np.abs(seen[y1+dy:y2+dy,x1+dx:x2+dx].astype(int)-rgb[y1:y2,x1:x2].astype(int))\n        if np.any(err):raise RuntimeError(\'source label changed\')\n        checks.append({\'name\':name,\'source_box\':box,\'pixels\':int((x2-x1)*(y2-y1)),\'changed_pixels\':0,\'max_channel_error\':0})\n    oy,ox=np.nonzero(a==255);diff=seen[oy+dy,ox+dx].astype(int)-rgb[oy,ox].astype(int)\n    assert not np.any(diff)\n    output.mkdir(parents=True);base.save(output/\'output.png\');alpha.save(output/\'reviewed-alpha.png\');source.save(output/\'source-native.png\')\n    manifest={\'kind\':\'BOUNDARY_CONTACT_CONTROL_NOT_PRODUCTION_APPROVAL\',\'integer_translation\':[dx,dy],\'source_resampled\':False,\'source_rgb_repainted\':False,\'transparent_lid_recovered\':False,\'support_centre\':[cx,int(cy)],\'support_width\':support_width,\'protected_regions\':checks,\'opaque_pixels\':len(ox),\'opaque_changed_pixels\':0,\'output_sha256\':hashlib.sha256((output/\'output.png\').read_bytes()).hexdigest(),\'limitations\':[\'Reviewed alpha boundary may differ from original alpha.\',\'Contact shadow is a 2D support heuristic, not recovered scene geometry.\',\'Transparent lid RGB retains old-scene transmission and reflections. No measured clean plate exists.\',\'Unchanged source perspective may remain physically unsupported.\']}\n    (output/\'manifest.json\').write_text(json.dumps(manifest,indent=2));return manifest\n'
# ---- embedded module: renderello/boundary_contact_preview.py ----
path = base / 'renderello/boundary_contact_preview.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Opt-in diagnostics for explicit local boundary searches. Never accepts a matte."""\nimport numpy as np\n\n\ndef boundary_search_review(baseline, traced, *, corridor_radius):\n    """Flag search-limit saturation. No flags is not semantic quality acceptance.\n\n    Inputs are same-column native y coordinates. This report does not compute\n    accuracy or confidence and must not select a production alpha.\n    """\n    if type(corridor_radius) is not int or not 1 <= corridor_radius <= 128:\n        raise ValueError(\'integer corridor radius required\')\n    b, t = np.asarray(baseline), np.asarray(traced)\n    if (b.ndim != 1 or t.shape != b.shape or b.size < 2 or b.size > 10000\n            or b.dtype.kind not in \'iu\' or t.dtype.kind not in \'iu\'\n            or b.dtype.kind == \'b\' or t.dtype.kind == \'b\'):\n        raise ValueError(\'matching integer native path coordinates required\')\n    # Bound before subtraction to avoid integer overflow and meaningless geometry.\n    if np.any(b < 0) or np.any(t < 0) or np.any(b > 1_000_000) or np.any(t > 1_000_000):\n        raise ValueError(\'path coordinates out of bounds\')\n    shift = np.abs(t.astype(np.int64)-b.astype(np.int64))\n    if np.any(shift > corridor_radius):\n        raise ValueError(\'traced path leaves explicit corridor\')\n    hits = np.flatnonzero(shift == corridor_radius).tolist()\n    reasons = [\'SEARCH_LIMIT_REACHED\'] if hits else []\n    return {\'kind\':\'boundary-search-diagnostic\',\'columns\':int(b.size),\n            \'moved_columns\':int(np.count_nonzero(shift)),\n            \'corridor_limit_column_indices\':hits,\'refusal_flags\':reasons,\n            \'quality_accepted\':False,\'production_mask_selected\':False,\n            \'semantic_review_required\':True,\n            \'limitation\':\'No flags is not confidence, accuracy, or semantic boundary acceptance.\'}\n'
# ---- embedded module: renderello/boundary_search_review.py ----
path = base / 'renderello/boundary_search_review.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Original-brief acceptance/economics accounting, without invented ratings.\n\nThe brief proposes80% no critical alteration and70% publish-usable. Those are\nseparate gates. Every preregistered first-try output remains in the denominator,\nincluding failed generations and missing reviews. Software execution is not a\npublishable result. A reported rating/provenance is a claim, not authentication.\n"""\nfrom __future__ import annotations\nfrom dataclasses import dataclass\nfrom collections.abc import Mapping\nfrom types import MappingProxyType\nfrom decimal import Decimal\nimport math\nimport statistics\nfrom .review import DIMENSIONS\n\n\ndef _id(value, name):\n    if not isinstance(value,str) or not value.strip() or len(value)>240:\n        raise ValueError(name+\' must be a short nonempty identifier\')\n\n\ndef _number(value, name):\n    if isinstance(value,bool) or not isinstance(value,(int,float)) or not math.isfinite(value) or value<0:\n        raise ValueError(name+\' must be finite nonnegative number\')\n\n\n@dataclass(frozen=True)\nclass PlannedOutput:\n    output_id: str\n    source_id: str\n    template_id: str\n    method: str\n    alternative: int\n    first_try: bool = True\n    source_sha256: str | None = None\n\n    def __post_init__(self):\n        for name in (\'output_id\',\'source_id\',\'template_id\',\'method\'):_id(getattr(self,name),name)\n        if type(self.alternative) is not int or not 0<=self.alternative<=100:\n            raise ValueError(\'bounded integer alternative required\')\n        if type(self.first_try) is not bool:raise ValueError(\'first_try must be boolean\')\n        if self.source_sha256 is not None and (not isinstance(self.source_sha256,str) or len(self.source_sha256)!=64 or any(c not in \'0123456789abcdef\' for c in self.source_sha256)):\n            raise ValueError(\'source hash must be exact lowercase sha256\')\n\n\n@dataclass(frozen=True)\nclass Attempt:\n    output_id: str\n    attempt_id: str\n    status: str\n    seconds: float | None\n    credits_charged: float | None\n    output_sha256: str | None = None\n    failure: str = \'\'\n\n    def __post_init__(self):\n        _id(self.output_id,\'output_id\');_id(self.attempt_id,\'attempt_id\')\n        if self.status not in (\'success\',\'failed\',\'unknown\'):raise ValueError(\'explicit attempt status required\')\n        for name in (\'seconds\',\'credits_charged\'):\n            value=getattr(self,name)\n            if value is not None:_number(value,name)\n        if self.status==\'success\':\n            if not isinstance(self.output_sha256,str) or len(self.output_sha256)!=64 or any(c not in \'0123456789abcdef\' for c in self.output_sha256):\n                raise ValueError(\'successful artifact needs exact lowercase sha256\')\n            if self.failure:raise ValueError(\'success cannot have failure text\')\n        elif self.output_sha256 is not None:\n            raise ValueError(\'failed/unknown attempt cannot carry a successful output hash\')\n        elif not isinstance(self.failure,str) or not self.failure.strip():\n            raise ValueError(\'failed/unknown attempt needs reason\')\n\n\n@dataclass(frozen=True)\nclass Rating:\n    output_id: str\n    output_sha256: str\n    scores: dict\n    no_critical: bool\n    critical_errors: tuple[str,...]\n    provenance: str\n    review_reference: str\n\n    def __post_init__(self):\n        _id(self.output_id,\'output_id\');_id(self.review_reference,\'review_reference\')\n        if not isinstance(self.scores,Mapping) or set(self.scores)!=set(DIMENSIONS):\n            raise ValueError(\'all seven scores required exactly\')\n        if any(type(v) is not int or not 1<=v<=5 for v in self.scores.values()):\n            raise ValueError(\'integer scores1..5 required\')\n        object.__setattr__(self,\'scores\',MappingProxyType(dict(self.scores)))\n        if type(self.no_critical) is not bool:raise ValueError(\'explicit boolean critical decision required\')\n        if type(self.critical_errors) is not tuple or any(not isinstance(x,str) or not x.strip() for x in self.critical_errors):\n            raise ValueError(\'explicit critical-error tuple required\')\n        if self.no_critical == bool(self.critical_errors):raise ValueError(\'critical decision conflicts with errors\')\n        if self.provenance not in (\'assistant_judgment\',\'owner\',\'claimed_independent\'):\n            raise ValueError(\'explicit review provenance required\')\n\n\n\ndef _artifact_matching_count(graph):\n    """One distinct artifact per coverage identity; repeated bytes cannot pad it.\n\n    Iterative augmenting paths avoid recursion limits. This establishes byte\n    diversity only, not visual variety, source authorship or template quality.\n    """\n    from collections import deque\n    assigned = {}\n    for identity in sorted(graph):\n        queue=deque([identity]);seen={identity};parent={};end=None\n        while queue and end is None:\n            node=queue.popleft()\n            for artifact in sorted(graph[node]):\n                if artifact in parent:continue\n                parent[artifact]=node\n                if artifact not in assigned:\n                    end=artifact;break\n                other=assigned[artifact]\n                if other not in seen:seen.add(other);queue.append(other)\n        if end is not None:\n            reverse={node:artifact for artifact,node in assigned.items()}\n            while True:\n                node=parent[end];previous=reverse.get(node);assigned[end]=node\n                if previous is None:break\n                end=previous\n    return len(assigned)\n\n\ndef evaluate(plan, attempts, ratings, *, publish_threshold=4, minimum_sources=20,\n             minimum_templates=8, competing_specialists=()):\n    """Complete accounting, not an automatic deployment or spending decision.\n\n    A score>=4 is a declared operational interpretation of publish-usable, not a\n    threshold stated in the brief. Return it prominently so it cannot be hidden.\n    Plan methods must be analyzed separately; mixed methods are refused.\n    """\n    if not plan or any(not isinstance(x,PlannedOutput) for x in plan):raise ValueError(\'nonempty typed plan required\')\n    if len({x.output_id for x in plan})!=len(plan):raise ValueError(\'duplicate planned output\')\n    if len({(x.source_id,x.template_id,x.alternative) for x in plan})!=len(plan):\n        raise ValueError(\'duplicate logical source/template/alternative slot\')\n    source_hashes = {}\n    for x in plan:\n        if x.source_id in source_hashes and source_hashes[x.source_id] != x.source_sha256:\n            raise ValueError(\'same source identifier has conflicting hash\')\n        source_hashes[x.source_id] = x.source_sha256\n    if len({x.method for x in plan})!=1:raise ValueError(\'evaluate one method at a time\')\n    for name,value,upper in [(\'publish_threshold\',publish_threshold,5),(\'minimum_sources\',minimum_sources,10000),(\'minimum_templates\',minimum_templates,1000)]:\n        if type(value) is not int or not 1<=value<=upper:raise ValueError(\'invalid \'+name)\n    ids={x.output_id for x in plan};by_id={x.output_id:[] for x in plan};seen=set()\n    for x in attempts:\n        if not isinstance(x,Attempt) or x.output_id not in ids:raise ValueError(\'unplanned attempt\')\n        if x.attempt_id in seen:raise ValueError(\'duplicate attempt id\')\n        seen.add(x.attempt_id);by_id[x.output_id].append(x)\n    rated={}\n    for r in ratings:\n        if not isinstance(r,Rating) or r.output_id not in ids:raise ValueError(\'unplanned rating\')\n        if r.output_id in rated:raise ValueError(\'multiple ratings require an explicit aggregation policy\')\n        successes=[a for a in by_id[r.output_id] if a.status==\'success\']\n        if not any(a.output_sha256==r.output_sha256 for a in successes):raise ValueError(\'rating does not bind a successful output\')\n        planned = next(p for p in plan if p.output_id == r.output_id)\n        if planned.first_try and (not by_id[r.output_id] or by_id[r.output_id][0].status != \'success\' or by_id[r.output_id][0].output_sha256 != r.output_sha256):\n            raise ValueError(\'first-try plan cannot substitute a later successful retry\')\n        rated[r.output_id]=r\n    count=len(plan);complete=len(rated)==count\n    no_critical=sum(r.no_critical for r in rated.values())\n    usable=sum(r.scores[\'publish_ready_usefulness\']>=publish_threshold and r.no_critical for r in rated.values())\n    # Exact rational comparison avoids rounding79.95% up to80%.\n    fidelity_pass=no_critical*100>=80*count\n    useful_pass=usable*100>=70*count\n    bound_sources={h for h in source_hashes.values() if h is not None}\n    planned_coverage=len(bound_sources)>=minimum_sources and len({p.template_id for p in plan})>=minimum_templates\n    eligible={}\n    for p in plan:\n        history=by_id[p.output_id]\n        selected=history[:1] if p.first_try else history\n        eligible[p.output_id]={a.output_sha256 for a in selected if a.status==\'success\'}\n    generated_source_graph={};generated_template_graph={}\n    reviewed_source_graph={};reviewed_template_graph={}\n    for p in plan:\n        hashes=eligible[p.output_id]\n        if hashes:\n            generated_template_graph.setdefault(p.template_id,set()).update(hashes)\n            if p.source_sha256 is not None:\n                generated_source_graph.setdefault(p.source_sha256,set()).update(hashes)\n        if p.output_id in rated:\n            h=rated[p.output_id].output_sha256\n            reviewed_template_graph.setdefault(p.template_id,set()).add(h)\n            if p.source_sha256 is not None:\n                reviewed_source_graph.setdefault(p.source_sha256,set()).add(h)\n    source_evidence=_artifact_matching_count(reviewed_source_graph)\n    template_evidence=_artifact_matching_count(reviewed_template_graph)\n    coverage=source_evidence>=minimum_sources and template_evidence>=minimum_templates\n    successful_artifact_hashes=[a.output_sha256 for a in attempts if a.status==\'success\']\n    specialist_names=set(competing_specialists)\n    if len(specialist_names)!=len(competing_specialists) or any(not isinstance(x,str) or not x.strip() for x in specialist_names):\n        raise ValueError(\'unique explicit specialist identities required\')\n    # Merely naming competitors does not establish a matched blind study.\n    blockers=[]\n    if not complete:blockers.append(\'INCOMPLETE_REVIEWS\')\n    if not coverage:blockers.append(\'SOURCE_OR_TEMPLATE_COVERAGE_BELOW_BRIEF\')\n    if not fidelity_pass:blockers.append(\'NO_CRITICAL_ALTERATION_GATE_NOT_ESTABLISHED\')\n    if not useful_pass:blockers.append(\'PUBLISH_USABLE_GATE_NOT_ESTABLISHED\')\n    blockers.append(\'TEMPLATE_VISUAL_DISTINCTNESS_REQUIRES_REVIEW\')\n    blockers.append(\'MATCHED_SPECIALIST_COMPETITIVENESS_REQUIRES_REVIEW\')\n    times=[a.seconds for a in attempts if a.seconds is not None]\n    known_credits=sum(Decimal(str(a.credits_charged)) for a in attempts if a.credits_charged is not None)\n    costs_complete=all(a.credits_charged is not None for a in attempts) and bool(attempts)\n    missing_attempts=sum(not by_id[p.output_id] for p in plan)\n    generated=sum(any(a.status==\'success\' for a in by_id[p.output_id]) for p in plan)\n    success_times=[a.seconds for a in attempts if a.status==\'success\' and a.seconds is not None]\n    return dict(method=plan[0].method,planned_outputs=count,distinct_sources=len({p.source_id for p in plan}),\n        bound_distinct_source_hashes=len(bound_sources),unbound_source_ids=sum(h is None for h in source_hashes.values()),\n        source_hash_semantics=\'declared_file_hash_not_proof_of_source_variety_or_authorship\',\n        distinct_templates=len({p.template_id for p in plan}),first_try_planned=sum(p.first_try for p in plan),\n        planned_coverage_gate=planned_coverage,\n        generated_bound_source_hashes=len(generated_source_graph),generated_template_ids=len(generated_template_graph),\n        reviewed_bound_source_hashes=len(reviewed_source_graph),reviewed_template_ids=len(reviewed_template_graph),\n        generated_artifact_bound_sources=_artifact_matching_count(generated_source_graph),\n        generated_artifact_bound_templates=_artifact_matching_count(generated_template_graph),\n        reviewed_artifact_bound_sources=source_evidence,reviewed_artifact_bound_templates=template_evidence,\n        unique_successful_artifact_hashes=len(set(successful_artifact_hashes)),\n        repeated_successful_artifact_occurrences=len(successful_artifact_hashes)-len(set(successful_artifact_hashes)),\n        eligible_generated_output_slots=sum(bool(v)for v in eligible.values()),\n        coverage_semantics=\'reviewed_source_and_template_identities_each_matched_to_distinct_artifact_bytes_not_proof_of_visual_variety\',\n        distinct_templates_semantics=\'planned_labels_not_polished_or_visually_distinct_template_count\',\n        template_visual_distinctness=\'NOT_ESTABLISHED\', \n        generated_outputs=generated,missing_attempt_outputs=missing_attempts,\n        failed_attempts=sum(a.status==\'failed\' for a in attempts),unknown_attempts=sum(a.status==\'unknown\' for a in attempts),\n        retry_attempts=sum(max(0,len(v)-1) for v in by_id.values()),\n        rated_outputs=len(rated),unrated_outputs=count-len(rated),rating_provenance=sorted({r.provenance for r in ratings}),\n        no_critical_count=no_critical,publish_usable_count=usable,\n        no_critical_fraction=no_critical/count,publish_usable_fraction=usable/count,\n        fraction_semantics=\'confirmed_passes_over_all_planned_outputs_unrated_not_assumed_good\',\n        publish_score_threshold=publish_threshold,publish_threshold_origin=\'caller_policy_not_explicit_brief_threshold\',\n        fidelity_gate=fidelity_pass,usefulness_gate=useful_pass,coverage_gate=coverage,\n        all_reviews_complete=complete,named_specialists=sorted(specialist_names),\n        blind_competitive_acceptance=\'NOT_ESTABLISHED\',blockers=blockers,production_go=False,\n        attempts=len(attempts),latency_samples=len(times),latency_complete=len(times)==len(attempts) and bool(attempts),\n        median_attempt_seconds=statistics.median(times) if times else None,\n        successful_latency_samples=len(success_times),\n        median_success_attempt_seconds=statistics.median(success_times) if success_times else None,\n        charged_credits_known=float(known_credits),charged_credits_complete=costs_complete,\n        credits_per_generated_output=float(known_credits/generated) if costs_complete and generated else None,\n        credits_per_publish_usable_output=float(known_credits/usable) if costs_complete and usable and complete else None,\n        price_and_margin=\'NOT_COMPUTED_REQUIRES_CURRENT_RATE_AND_SALE_PRICE\',\n        authority=\'ACCOUNTING_ONLY_NO_SEND_SPEND_OR_DEPLOY_AUTHORITY\')\n'
# ---- embedded module: renderello/brief_acceptance.py ----
path = base / 'renderello/brief_acceptance.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Evidence-bound original-brief economics. Offline; never authorizes generation.\n\nLegacy costs.py remains a published-catalog projection. This path uses typed\nattempt records from brief_acceptance, keeps missing charges unknown, and\nrequires explicitly dated pricing evidence. It cannot turn ratings into sales.\n"""\nfrom dataclasses import dataclass, asdict\nfrom datetime import date\nfrom decimal import Decimal, InvalidOperation\nimport argparse\nimport json\nfrom pathlib import Path\nfrom .brief_acceptance import PlannedOutput, Attempt, Rating, evaluate\n\n\ndef amount(value, name, *, positive=False):\n    if isinstance(value, bool) or not isinstance(value, (str, int, float, Decimal)):\n        raise ValueError(name + \' must be a decimal amount\')\n    try:\n        d = Decimal(str(value))\n    except InvalidOperation as exc:\n        raise ValueError(name + \' must be a decimal amount\') from exc\n    if not d.is_finite() or d < 0 or (positive and d == 0):\n        raise ValueError(name + \' must be finite and \' + (\'positive\' if positive else \'nonnegative\'))\n    if d.adjusted() > 15 or d.as_tuple().exponent < -12:\n        raise ValueError(name + \' exceeds decimal precision budget\')\n    return d\n\n\ndef text(value, name):\n    if not isinstance(value, str) or not value.strip() or len(value) > 2000:\n        raise ValueError(name + \' requires a bounded evidence reference\')\n\n\n@dataclass(frozen=True)\nclass PriceEvidence:\n    currency: str\n    per_credit: str\n    rate_basis: str\n    source_reference: str\n    observed_on: str\n    valid_through: str\n    credit_system: str\n\n    def __post_init__(self):\n        if not isinstance(self.currency, str) or len(self.currency) != 3 or not self.currency.isascii() or not self.currency.isupper() or not self.currency.isalpha():\n            raise ValueError(\'explicit three-letter currency required\')\n        amount(self.per_credit, \'per_credit\', positive=True)\n        if self.rate_basis not in (\'included_subscription_allocation\', \'purchased_credit_pack\', \'usage_invoice\'):\n            raise ValueError(\'explicit non-free credit valuation basis required\')\n        text(self.source_reference, \'price source\')\n        text(self.credit_system, \'credit system\')\n        try:\n            start, end = date.fromisoformat(self.observed_on), date.fromisoformat(self.valid_through)\n        except (TypeError, ValueError) as exc:\n            raise ValueError(\'ISO observation/validity dates required\') from exc\n        if start > end:\n            raise ValueError(\'price validity ends before observation\')\n\n\ndef build_report(plan, attempts, ratings, *, price=None, as_of, sale_price=None,\n                 sale_reference=None, other_cost_per_usable=None,\n                 other_cost_reference=None, publish_threshold=4, charge_credit_system=None):\n    """Report a scenario, not actual profit, a current vendor quote or go decision.\n\n    Missing first attempts and unknown statuses prevent a complete-cost claim.\n    All attempted charges (including failed and retried jobs) stay in numerator.\n    Distinct physical artifact hashes are the denominators, never renamed slots.\n    Other per-usable costs are mandatory for a contribution-margin scenario;\n    explicit zero is accepted only with an accompanying evidence/assumption note.\n    """\n    try:\n        day = date.fromisoformat(as_of)\n    except (TypeError, ValueError) as exc:\n        raise ValueError(\'explicit ISO as_of date required\') from exc\n    plan, attempts, ratings = list(plan), list(attempts), list(ratings)\n    summary = evaluate(plan, attempts, ratings, publish_threshold=publish_threshold)\n    if price is not None and not isinstance(price, PriceEvidence):\n        raise ValueError(\'typed dated pricing evidence required\')\n    if charge_credit_system is not None: text(charge_credit_system, \'charge credit system\')\n    if price is not None and price.credit_system != charge_credit_system:\n        raise ValueError(\'price and charge credit systems must explicitly match\')\n    if (sale_price is None) != (sale_reference is None):\n        raise ValueError(\'sale scenario and reference must be supplied together\')\n    if (other_cost_per_usable is None) != (other_cost_reference is None):\n        raise ValueError(\'other-cost scenario and reference must be supplied together\')\n    sale = None if sale_price is None else amount(sale_price, \'sale price\', positive=True)\n    overhead = None if other_cost_per_usable is None else amount(other_cost_per_usable, \'other cost\')\n    if sale is not None: text(sale_reference, \'sale price\')\n    if overhead is not None: text(other_cost_reference, \'other cost\')\n    complete = (bool(attempts) and summary[\'missing_attempt_outputs\'] == 0\n                and summary[\'unknown_attempts\'] == 0 and summary[\'charged_credits_complete\'])\n    known = sum((amount(a.credits_charged, \'charged credits\') for a in attempts if a.credits_charged is not None), Decimal(0))\n    successful = {a.output_sha256 for a in attempts if a.status == \'success\'}\n    reviewed_hashes = {r.output_sha256 for r in ratings}\n    artifact_reviews_complete = bool(successful) and successful <= reviewed_hashes\n    usable = {r.output_sha256 for r in ratings if r.no_critical and r.scores[\'publish_ready_usefulness\'] >= publish_threshold}\n    eligible = {a.output_id: [] for a in attempts}\n    for a in attempts: eligible[a.output_id].append(a)\n    selected = []\n    for p in plan:\n        history = eligible.get(p.output_id, [])\n        selected.extend(history[:1] if p.first_try else history)\n    first_eligible = {a.output_sha256 for a in selected if a.status == \'success\'}\n    usable &= first_eligible\n    # Identical bytes with conflicting reviews are not a usable-unit shortcut.\n    rejected_hashes = {r.output_sha256 for r in ratings if not r.no_critical or r.scores[\'publish_ready_usefulness\'] < publish_threshold}\n    conflicting_hashes = usable & rejected_hashes\n    usable -= rejected_hashes\n    blockers = []\n    if conflicting_hashes: blockers.append(\'CONFLICTING_DUPLICATE_ARTIFACT_REVIEWS\')\n    if not complete: blockers.append(\'INCOMPLETE_ATTEMPT_OR_CHARGE_EVIDENCE\')\n    if not summary[\'all_reviews_complete\']: blockers.append(\'INCOMPLETE_PLANNED_SLOT_REVIEWS_SEE_ACCEPTANCE\')\n    if not artifact_reviews_complete: blockers.append(\'INCOMPLETE_SUCCESSFUL_ARTIFACT_REVIEWS\')\n    if not successful: blockers.append(\'NO_SUCCESSFUL_ARTIFACTS\')\n    if not usable: blockers.append(\'NO_REVIEWED_USABLE_ARTIFACTS\')\n    fresh = price is not None and date.fromisoformat(price.observed_on) <= day <= date.fromisoformat(price.valid_through)\n    if price is None: blockers.append(\'PRICE_EVIDENCE_MISSING\')\n    elif not fresh: blockers.append(\'PRICE_EVIDENCE_OUTSIDE_DECLARED_VALIDITY\')\n    total = known * amount(price.per_credit, \'per_credit\') if complete and fresh else None\n    per_generated = total / len(successful) if total is not None and successful else None\n    per_usable = total / len(usable) if total is not None and usable and artifact_reviews_complete else None\n    if sale is None: blockers.append(\'SALE_PRICE_SCENARIO_MISSING\')\n    if overhead is None: blockers.append(\'OTHER_COST_SCENARIO_MISSING\')\n    margin = sale - per_usable - overhead if sale is not None and per_usable is not None and overhead is not None else None\n    dec = lambda v: None if v is None else format(v, \'f\')\n    return dict(schema=\'brief-economics.v1\', as_of=as_of, method=plan[0].method,\n        acceptance=summary, pricing=None if price is None else asdict(price), price_in_declared_validity=fresh,\n        evidence_semantics=\'caller_supplied_records_not_provider_or_reviewer_authentication\',\n        charge_credit_system=charge_credit_system, credit_scope_semantics=\'caller_declared_single_credit_system_for_all_attempts; split_provider_or_account_denominations_before_reporting\',\n        charge_evidence_complete=complete, known_charged_credits=dec(known),\n        all_successful_artifacts_reviewed=artifact_reviews_complete,\n        distinct_successful_artifacts=len(successful), distinct_usable_artifacts=len(usable),\n        successful_artifact_semantics=\'any_recorded_attempt_including_retries_not_first_try_quality_rate\',\n        total_attempt_credit_cost=dec(total), cost_per_distinct_generated_artifact=dec(per_generated),\n        cost_per_distinct_reviewed_usable_artifact=dec(per_usable),\n        sale_price_scenario=dec(sale), sale_reference=sale_reference,\n        other_cost_per_usable_scenario=dec(overhead), other_cost_reference=other_cost_reference,\n        contribution_per_usable_scenario=dec(margin),\n        contribution_margin_fraction_scenario=dec(None if margin is None else margin / sale),\n        margin_semantics=\'hypothetical_per_reviewed_usable_delivery_not_actual_revenue_or_profit; other_cost_input_must_cover_relevant_infrastructure_support_fees\',\n        zero_recorded_credits_semantics=\'recorded_free_test_not_proof_of_zero_future_serving_cost\',\n        best_quality_model=None, value_fallback=None,\n        recommendation=\'DEFERRED_UNTIL_MATCHED_QUALITY_AND_SPECIALIST_REVIEW\',\n        blockers=blockers, production_go=False, live_calls=0,\n        authority=\'NO_SEND_SPEND_SUBSCRIBE_OR_DEPLOY_AUTHORITY\')\n\n\ndef from_payload(payload, *, evidence_path=None):\n    """Strict reusable adapter for existing brief_acceptance records."""\n    if not isinstance(payload, dict): raise ValueError(\'object payload required\')\n    if \'evidence_pair_required\' in payload:\n        if payload[\'evidence_pair_required\'] is not True or evidence_path is None:\n            raise ValueError(\'paired provider export requires evidence_path and complete-pair validation\')\n        from .evidence_pair import validate_pair\n        validate_pair(evidence_path)\n        if json.loads(Path(evidence_path).read_text())!=payload:\n            raise ValueError(\'supplied payload differs from validated pair\')\n        payload={k:v for k,v in payload.items() if k!=\'evidence_pair_required\'}\n    if set(payload) - {\'plan\', \'attempts\', \'ratings\', \'price\', \'as_of\', \'sale_price\', \'sale_reference\', \'other_cost_per_usable\', \'other_cost_reference\', \'publish_threshold\', \'charge_credit_system\'}:\n        raise ValueError(\'unknown report fields\')\n    plan = [PlannedOutput(**r) for r in payload[\'plan\']]\n    attempts = [Attempt(**r) for r in payload[\'attempts\']]\n    ratings = []\n    for r in payload[\'ratings\']:\n        r = dict(r); r[\'critical_errors\'] = tuple(r[\'critical_errors\']); ratings.append(Rating(**r))\n    opts = {k: v for k, v in payload.items() if k not in (\'plan\', \'attempts\', \'ratings\', \'price\')}\n    price = None if payload.get(\'price\') is None else PriceEvidence(**payload[\'price\'])\n    return build_report(plan, attempts, ratings, price=price, **opts)\n\n\ndef main(argv=None):\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--input\', type=Path, required=True)\n    p.add_argument(\'--output\', type=Path, required=True)\n    a = p.parse_args(argv)\n    try:\n        if a.input.is_symlink() or not a.input.is_file() or a.input.stat().st_size > 2_000_000:\n            raise ValueError(\'bounded regular input JSON required\')\n        report = from_payload(json.loads(a.input.read_text()), evidence_path=a.input)\n        with a.output.open(\'x\') as f: json.dump(report, f, indent=2)\n    except (ValueError, TypeError, KeyError, OSError) as exc:\n        p.exit(2, \'REFUSED: \' + str(exc) + \'\\n\')\n    print(\'Saved offline economics report: \' + str(a.output))\n\n\nif __name__ == \'__main__\': main()\n'
# ---- embedded module: renderello/brief_economics.py ----
path = base / 'renderello/brief_economics.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Repeatable native product paste onto a saved scene, with authored shadow.\n\nNo model calls or fit-to-canvas resizing. Matte is interpreted as ordinary\nstraight-alpha coverage in sRGB code values, matching Pillow source-over.\nShadow darkening uses linear light; this is not calibrated physical relighting.\n"""\nimport numpy as np\nfrom PIL import Image\nfrom .candidate_diagnostics import native_rgb, native_alpha, digest\nfrom .checked_research.contact_camera import linear, srgb\n\n\ndef compose_cached_native(source, alpha, scene, *, translation, shadow=None):\n    native_rgb(source); native_rgb(scene); native_alpha(alpha, source.shape[:2])\n    sh, sw = alpha.shape\n    dh, dw = scene.shape[:2]\n    if sh * sw > 1_000_000 or dh * dw > 2_000_000:\n        raise ValueError(\'native1MPproduct/2MPscene limits; no automatic resize\')\n    if not isinstance(translation, (tuple, list)) or len(translation) != 2 or any(type(v) is not int or v < 0 for v in translation):\n        raise ValueError(\'nonnegative integer native translation required\')\n    x, y = translation\n    if x + sw > dw or y + sh > dh:\n        raise ValueError(\'whole native product must fit; no silent clipping or resize\')\n    if not np.any(alpha):\n        raise ValueError(\'empty product alpha\')\n    if shadow is None:\n        dark = scene.copy()\n    else:\n        if not isinstance(shadow, np.ndarray) or shadow.dtype.kind != \'f\' or shadow.shape != (dh, dw) or not np.isfinite(shadow).all() or np.any((shadow < 0) | (shadow > .5)):\n            raise ValueError(\'finite native shadow attenuation0..0.5required\')\n        if np.any(shadow[y:y+sh, x:x+sw][alpha >= 128]):\n            raise ValueError(\'authored shadow may not cross hard product support\')\n        dark = srgb(linear(scene) * (1-shadow[..., None]))\n        dark[shadow == 0] = scene[shadow == 0]\n    canvas = Image.fromarray(dark).convert(\'RGBA\')\n    rgba = Image.fromarray(np.dstack((source, alpha)), \'RGBA\')\n    canvas.alpha_composite(rgba, (x, y))\n    output = np.array(canvas.convert(\'RGB\'))\n    interior = alpha == 255\n    if not np.array_equal(output[y:y+sh, x:x+sw][interior], source[interior]):\n        raise RuntimeError(\'native opaque product RGB invariant failed\')\n    return output, dict(schema=\'cached-native-scene-composite.v1\', source_sha256=digest(source),\n        alpha_sha256=digest(alpha), scene_sha256=digest(scene), result_sha256=digest(output),\n        shadow_sha256=digest(shadow) if shadow is not None else None,\n        translation=[x,y], source_native_size=[sw,sh], output_size=[dw,dh],\n        source_resampled=False, source_rgb_repainted=False,\n        opaque_source_rgb_exact_pixels=int(interior.sum()),\n        alpha_space=\'straight coverage, sRGB code-value source-over\',\n        shadow_space=\'linear RGB attenuation, authored not recovered\',\n        extra_api_calls=0, physical_relighting=False, photo_accepted=False,\n        limitations=[\'Fractional alpha blends source and scene; exactness applies only to fully opaque pixels.\',\n                     \'Matte quality, source lighting, material optics and scene compatibility are not established.\',\n                     \'Authored contact and shadows are not measured scene geometry.\'])\n'
# ---- embedded module: renderello/cached_native_scene.py ----
path = base / 'renderello/cached_native_scene.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Render an existing RGB photo and supplied alpha on an existing scene.\n\nOffline only. No automatic segmentation, generation, resizing, crop or\nphysical relighting. Fractional alpha is straight sRGB coverage, not optics.\n"""\nfrom pathlib import Path\nfrom io import BytesIO\nimport argparse\nimport hashlib\nimport json\nimport os\nimport stat\nimport tempfile\nimport shutil\nimport numpy as np\nfrom PIL import Image\nfrom .cached_native_scene import compose_cached_native\n\n\ndef snapshot(path, mode, pixels):\n    path = Path(path)\n    flags = os.O_RDONLY | getattr(os, \'O_NONBLOCK\', 0) | getattr(os, \'O_NOFOLLOW\', 0)\n    with os.fdopen(os.open(path, flags), \'rb\') as stream:\n        if not stat.S_ISREG(os.fstat(stream.fileno()).st_mode):\n            raise ValueError(\'regular native PNG input required\')\n        data = stream.read(16_000_001)\n    if len(data) > 16_000_000:\n        raise ValueError(\'encoded PNG input exceeds16MBbudget\')\n    # Pillow can truncate16bitRGB PNG silently, so inspect IHDR before decode.\n    if not data.startswith(b\'\\x89PNG\\r\\n\\x1a\\n\') or len(data) < 29 or data[12:16] != b\'IHDR\' or data[24] != 8:\n        raise ValueError(\'8bit native PNG required, no implicit conversion\')\n    with Image.open(BytesIO(data)) as image:\n        if image.mode != mode or getattr(image, \'n_frames\', 1) != 1:\n            raise ValueError(\'single-frame \'+mode+\' PNG required, no implicit conversion\')\n        if image.width * image.height > pixels:\n            raise ValueError(\'native input exceeds pixel budget, no resizing\')\n        if \'transparency\' in image.info:\n            raise ValueError(\'native RGB/L PNG must not carry hidden color-key transparency\')\n        if image.getexif().get(274, 1) != 1 or image.info.get(\'icc_profile\'):\n            raise ValueError(\'normalize orientation/profile before pairing reviewed pixels\')\n        image.load()\n        array = np.array(image)\n    return data, array\n\n\ndef run(source, alpha, scene, out, x, y):\n    out = Path(out)\n    if out.exists() or out.is_symlink():\n        raise FileExistsError(\'preserve earlier output; choose a new directory\')\n    inputs = {}\n    for name, path, mode, budget in [(\'source\',source,\'RGB\',1_000_000),\n                                     (\'alpha\',alpha,\'L\',1_000_000),\n                                     (\'scene\',scene,\'RGB\',2_000_000)]:\n        inputs[name] = snapshot(path, mode, budget)\n    result, report = compose_cached_native(inputs[\'source\'][1], inputs[\'alpha\'][1],\n                                           inputs[\'scene\'][1], translation=(x,y))\n    # Calculate and validate everything before creating a staged result.\n    out.parent.mkdir(parents=True, exist_ok=True)\n    stage = Path(tempfile.mkdtemp(prefix=\'.\'+out.name+\'-staging-\', dir=out.parent))\n    try:\n        for name, (data, _) in inputs.items():\n            (stage/(name+\'.png\')).write_bytes(data)\n        Image.fromarray(result).save(stage/\'composite.png\')\n        manifest = dict(schema=\'offline-native-scene-result.v1\',\n            status=\'EXECUTED_REVIEW_REQUIRED\', photo_accepted=False,\n            automatic_segmentation=False, provider_calls=0, credits_spent=0,\n            source_bytes_preserved=True, scene_bytes_preserved=True,\n            input_file_sha256={name:hashlib.sha256(data).hexdigest() for name,(data,_) in inputs.items()},\n            output_sha256=hashlib.sha256((stage/\'composite.png\').read_bytes()).hexdigest(),\n            report=report)\n        (stage/\'manifest.json\').write_text(json.dumps(manifest,indent=2))\n        # Reserve the destination rather than replacing a newly appeared result.\n        # Copy into the reserved directory with an incomplete marker. On failure\n        # leave evidence, not a misleading success manifest.\n        out.mkdir()\n        (out/\'INCOMPLETE\').touch()\n        for path in stage.iterdir():\n            if path.name != \'manifest.json\': shutil.copyfile(path,out/path.name)\n        shutil.copyfile(stage/\'manifest.json\',out/\'manifest.json\')\n        (out/\'INCOMPLETE\').unlink()\n        return manifest\n    finally:\n        shutil.rmtree(stage)\n\n\ndef main(argv=None):\n    parser=argparse.ArgumentParser(description=__doc__)\n    for name in (\'source\',\'alpha\',\'scene\',\'out\'):\n        parser.add_argument(\'--\'+name,type=Path,required=True)\n    parser.add_argument(\'--x\',type=int,required=True)\n    parser.add_argument(\'--y\',type=int,required=True)\n    args=parser.parse_args(argv)\n    result=run(**vars(args))\n    print(json.dumps(result,indent=2))\n    return 0\n\n\nif __name__ == \'__main__\':\n    raise SystemExit(main())\n'
# ---- embedded module: renderello/cached_native_scene_cli.py ----
path = base / 'renderello/cached_native_scene_cli.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Bounded native mask diagnostics and source-bound spatial intent constraints.\n\nOpenCV connected components are descriptive, not semantic object recognition.\nNothing here deletes small components, fills holes, merges model candidates or\nturns model agreement into accuracy. All coordinates refer to normalized pixels.\n"""\nfrom __future__ import annotations\nfrom dataclasses import asdict, dataclass\nimport hashlib\nimport math\nimport cv2\nimport numpy as np\n\nMAX_PIXELS = 4_000_000\n\n\ndef digest(array):\n    """Bind shape and dtype as well as bytes; raw bytes alone are ambiguous."""\n    a = np.ascontiguousarray(array)\n    h = hashlib.sha256()\n    h.update(str(a.shape).encode(\'ascii\'))\n    h.update(a.dtype.str.encode(\'ascii\'))\n    h.update(a.tobytes())\n    return h.hexdigest()\n\n\ndef native_rgb(rgb):\n    if not isinstance(rgb, np.ndarray) or rgb.dtype != np.uint8:\n        raise ValueError(\'native uint8 RGB required\')\n    if rgb.ndim != 3 or rgb.shape[2] != 3 or min(rgb.shape[:2]) < 3:\n        raise ValueError(\'RGB dimensions must be at least 3 by 3\')\n    if rgb.shape[0] * rgb.shape[1] > MAX_PIXELS:\n        raise ValueError(\'4MP allocation budget exceeded; no silent downscale\')\n    return rgb\n\n\ndef native_alpha(alpha, shape):\n    if not isinstance(alpha, np.ndarray) or alpha.dtype != np.uint8:\n        raise ValueError(\'native uint8 alpha required; no floating range guessing\')\n    if alpha.ndim != 2 or alpha.shape != shape:\n        raise ValueError(\'alpha must match normalized source; no resize\')\n    if alpha.size > MAX_PIXELS or min(alpha.shape) < 3:\n        raise ValueError(\'alpha dimensions outside allocation budget\')\n    return alpha\n\n\n@dataclass(frozen=True)\nclass RegionConstraint:\n    """A reviewer-specified region, not an automatically discovered object.\n\n    A small interior box is often safer than an object\'s full bounding rectangle:\n    object bounding boxes also contain background. Negative regions can preserve\n    handle holes or forbid a prop. Names describe intent, not identity proof.\n    """\n    name: str\n    box: tuple[int, int, int, int]\n    kind: str\n    minimum_fraction: float = .95\n    threshold: int = 128\n\n    def __post_init__(self):\n        if not isinstance(self.name, str) or not self.name.strip() or len(self.name) > 120:\n            raise ValueError(\'short region name required\')\n        if self.kind not in (\'include\', \'exclude\'):\n            raise ValueError(\'region kind must be include or exclude\')\n        if len(self.box) != 4 or any(type(v) is not int for v in self.box):\n            raise ValueError(\'integer x0,y0,x1,y1 region required\')\n        x0, y0, x1, y1 = self.box\n        if min(x0, y0) < 0 or x1 <= x0 or y1 <= y0:\n            raise ValueError(\'nonempty positive region required\')\n        if isinstance(self.minimum_fraction, bool) or not math.isfinite(self.minimum_fraction) or not 0 < self.minimum_fraction <= 1:\n            raise ValueError(\'minimum fraction must be in (0,1]\')\n        if type(self.threshold) is not int or not 1 <= self.threshold <= 255:\n            raise ValueError(\'threshold must be integer 1..255\')\n\n\ndef check_regions(alpha, constraints):\n    native_alpha(alpha, alpha.shape)\n    if len(constraints) > 256:\n        raise ValueError(\'too many regions\')\n    names = set()\n    result = []\n    for c in constraints:\n        if not isinstance(c, RegionConstraint) or c.name in names:\n            raise ValueError(\'typed, uniquely named region constraints required\')\n        names.add(c.name)\n        x0, y0, x1, y1 = c.box\n        if x1 > alpha.shape[1] or y1 > alpha.shape[0]:\n            raise ValueError(\'region outside native source\')\n        roi = alpha[y0:y1, x0:x1]\n        good = roi >= c.threshold if c.kind == \'include\' else roi < c.threshold\n        fraction = float(good.mean())\n        result.append(dict(**asdict(c), observed_fraction=fraction,\n                           passed=fraction >= c.minimum_fraction))\n    return result\n\n\ndef mask_diagnostics(rgb, alpha):\n    native_rgb(rgb)\n    native_alpha(alpha, rgb.shape[:2])\n    hard = alpha >= 128\n    support = alpha > 0\n    flags = []\n    if not support.any():\n        flags.append(\'EMPTY_MASK\')\n    if not hard.any():\n        flags.append(\'NO_HIGH_ALPHA_SUPPORT\')\n    if hard.all():\n        flags.append(\'FULL_FRAME_FOREGROUND\')\n    if min(alpha.shape) < 64:\n        flags.append(\'TINY_SOURCE_NO_DETAIL_RECOVERY\')\n    if max(alpha.shape) / min(alpha.shape) > 8:\n        flags.append(\'ODD_ASPECT_REVIEW\')\n    if support.any():\n        yy, xx = np.where(support)\n        box = [int(xx.min()), int(yy.min()), int(xx.max()) + 1, int(yy.max()) + 1]\n    else:\n        box = None\n    sides = {name: int(np.count_nonzero(edge)) for name, edge in (\n        (\'top\', hard[0]), (\'bottom\', hard[-1]),\n        (\'left\', hard[:, 0]), (\'right\', hard[:, -1]))}\n    if any(sides.values()):\n        flags.append(\'FRAME_TOUCH_NOT_PROOF_OF_COMPLETE_PRODUCT\')\n    count, labels, stats, centers = cv2.connectedComponentsWithStats(hard.astype(np.uint8), connectivity=8)\n    # Bound JSON size independently of native image size. Keep total count exact.\n    order = sorted(range(1, count), key=lambda i: (-int(stats[i, cv2.CC_STAT_AREA]), i))\n    components = []\n    for i in order[:256]:\n        x, y, w, h, area = map(int, stats[i])\n        components.append(dict(label=i, box=[x, y, x+w, y+h], area=area,\n                               centroid=[float(v) for v in centers[i]],\n                               touches_frame=x == 0 or y == 0 or x+w == alpha.shape[1] or y+h == alpha.shape[0]))\n    if count - 1 > 1:\n        flags.append(\'MULTICOMPONENT_REVIEW_EVERY_PART\')\n    if count - 1 > 256:\n        flags.append(\'COMPONENT_LIST_TRUNCATED_NOT_MASK\')\n    # Background components not touching the frame are holes. Never fill them:\n    # real handle/scissor gaps and a false internal cut look identical topologically.\n    nb, _, bs, _ = cv2.connectedComponentsWithStats((~hard).astype(np.uint8), connectivity=8)\n    holes = []\n    for x, y, w, h, area in bs[1:]:\n        if x > 0 and y > 0 and x+w < alpha.shape[1] and y+h < alpha.shape[0]:\n            holes.append(int(area))\n    if holes:\n        flags.append(\'INTERNAL_HOLES_REVIEW_NO_AUTOFILL\')\n    soft = int(((alpha > 0) & (alpha < 255)).sum())\n    if soft > max(16, int(hard.sum()) * .3):\n        flags.append(\'LARGE_SOFT_BAND_NOT_OPTICAL_TRANSMISSION\')\n    pixels = rgb[hard]\n    luminance = pixels.astype(float) @ np.array([.2126, .7152, .0722]) if len(pixels) else np.empty(0)\n    dark = float((luminance < 24).mean()) if len(pixels) else None\n    white = float((luminance > 230).mean()) if len(pixels) else None\n    if dark is not None and dark > .25:\n        flags.append(\'DARK_FOREGROUND_EDGE_REVIEW\')\n    if white is not None and white > .25:\n        flags.append(\'WHITE_FOREGROUND_EDGE_REVIEW\')\n    return dict(source_pixels_sha256=digest(rgb), alpha_pixels_sha256=digest(alpha),\n                shape=list(alpha.shape), support_box=box,\n                support_pixels=int(support.sum()), hard_pixels=int(hard.sum()),\n                soft_pixels=soft, opaque_pixels=int((alpha == 255).sum()),\n                frame_contact=sides, component_count=count-1,\n                components=components, component_list_truncated=count-1 > 256,\n                hole_count=len(holes), hole_areas=sorted(holes, reverse=True)[:256],\n                dark_fraction=dark, white_fraction=white, flags=flags,\n                semantic_completeness=\'NOT_EVALUATED\', optics_recovered=False,\n                automatic_acceptance=False)\n\n\ndef compare_masks(rgb, candidates):\n    native_rgb(rgb)\n    if not 2 <= len(candidates) <= 8:\n        raise ValueError(\'compare 2..8 separately named candidates\')\n    for name, a in candidates.items():\n        if not isinstance(name, str) or not name.strip() or len(name) > 80:\n            raise ValueError(\'short candidate name required\')\n        native_alpha(a, rgb.shape[:2])\n    result = []\n    names = list(candidates)\n    for i, left in enumerate(names):\n        for right in names[i+1:]:\n            a, b = candidates[left], candidates[right]\n            ah, bh = a >= 128, b >= 128\n            union = int((ah | bh).sum())\n            intersection = int((ah & bh).sum())\n            result.append(dict(left=left, right=right,\n                left_sha256=digest(a), right_sha256=digest(b),\n                changed_alpha_pixels=int((a != b).sum()),\n                mean_absolute_alpha_delta=float(np.abs(a.astype(np.int16)-b.astype(np.int16)).mean()),\n                hard_iou=intersection/union if union else None,\n                hard_disagreement_pixels=int((ah ^ bh).sum()),\n                left_only_pixels=int((ah & ~bh).sum()),\n                right_only_pixels=int((bh & ~ah).sum()),\n                agreement_is_accuracy=False, automatically_selected=None))\n    return result\n\n\ndef disagreement_image(left, right):\n    native_alpha(left, left.shape)\n    native_alpha(right, left.shape)\n    result = np.zeros((*left.shape, 3), dtype=np.uint8)\n    a, b = left >= 128, right >= 128\n    result[a & b] = (110, 110, 110)\n    result[a & ~b] = (255, 90, 40)\n    result[b & ~a] = (30, 170, 255)\n    # Color key: orange left-only, blue right-only, grey hard agreement.\n    return result\n'
# ---- embedded module: renderello/candidate_diagnostics.py ----
path = base / 'renderello/candidate_diagnostics.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Explicit open-source color harmonization candidates, never physical relighting.\n\nThe default route changes BACKGROUND ONLY with scikit-image histogram matching.\nThe optional OpenCV Poisson route changes foreground color, requires explicit\nopt-in and returns a changed-pixel map. Neither route recovers source lighting,\nmaterial, print accuracy, reflections or refraction. No automatic fallback.\n"""\nfrom __future__ import annotations\nimport importlib.metadata\nimport math\nimport cv2\nimport numpy as np\nfrom skimage.exposure import match_histograms\nfrom .candidate_diagnostics import native_rgb, native_alpha, digest\n\nSOURCES = [\n    \'https://scikit-image.org/docs/stable/api/skimage.exposure.html\',\n    \'https://scikit-image.org/docs/stable/license.html\',\n    \'https://docs.opencv.org/4.10.0/df/da0/group__photo__clone.html\',\n    \'https://opencv.org/license/\',\n]\n\n\ndef _fraction(value, upper, name):\n    if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value) or not 0 <= value <= upper:\n        raise ValueError(name + \' outside explicit bounds\')\n\n\ndef _binary(mask, shape, name, nonempty=True):\n    if not isinstance(mask, np.ndarray) or mask.dtype != np.bool_ or mask.shape != shape:\n        raise ValueError(name + \' must be native boolean mask\')\n    if nonempty and not mask.any():\n        raise ValueError(name + \' must not be empty\')\n\n\ndef background_match(background, source, alpha, reference_mask, *, amount=.2,\n                     maximum_delta=40, reference_reviewed=False):\n    """Adjust a generated background toward reviewed source BACKGROUND samples.\n\n    This preserves source RGB by changing the destination scene before paste.\n    The reviewer must supply genuine backdrop samples, not merely assume every\n    zero-alpha pixel is background. Sample errors can be caused by bad masks.\n    A limited strength and hard change budget prevent silent extreme grading.\n    """\n    native_rgb(background)\n    native_rgb(source)\n    native_alpha(alpha, source.shape[:2])\n    _binary(reference_mask, alpha.shape, \'reference mask\')\n    _fraction(amount, .5, \'amount\')\n    if type(maximum_delta) is not int or not 0 <= maximum_delta <= 80:\n        raise ValueError(\'maximum delta must be integer 0..80\')\n    if reference_reviewed is not True:\n        raise ValueError(\'reviewed source backdrop samples required\')\n    if np.any(reference_mask & (alpha != 0)):\n        raise ValueError(\'backdrop samples intersect candidate product support\')\n    n = int(reference_mask.sum())\n    if not 64 <= n <= 1_000_000:\n        raise ValueError(\'64..1000000 background sample pixels required\')\n    reference = source[reference_mask][:, None, :]\n    matched = match_histograms(background, reference, channel_axis=-1)\n    if matched.shape != background.shape or not np.isfinite(matched).all():\n        raise ValueError(\'invalid histogram matching result\')\n    candidate = np.rint(background.astype(float)*(1-amount)+matched*amount)\n    if np.any((candidate < 0) | (candidate > 255)):\n        raise ValueError(\'harmonization exceeds RGB gamut\')\n    candidate = candidate.astype(np.uint8)\n    delta = np.abs(candidate.astype(np.int16)-background.astype(np.int16))\n    if int(delta.max()) > maximum_delta:\n        raise ValueError(\'harmonization exceeds configured color-change budget\')\n    report = dict(method=\'skimage_background_histogram_matching\',\n        runtime_version=importlib.metadata.version(\'scikit-image\'),\n        background_sha256=digest(background), source_sha256=digest(source),\n        alpha_sha256=digest(alpha), reference_mask_sha256=digest(reference_mask),\n        reference_pixels=n, strength=amount, maximum_observed_delta=int(delta.max()),\n        changed_background_pixels=int(np.any(delta, axis=2).sum()),\n        source_rgb_changed=False, physical_relighting=False,\n        reference_review=\'caller_declaration_not_automatic_segmentation\',\n        automatic_acceptance=False, sources=SOURCES[:2])\n    return candidate, report\n\n\ndef linear_over(source, alpha, background):\n    native_rgb(source)\n    native_rgb(background)\n    native_alpha(alpha, source.shape[:2])\n    if source.shape != background.shape:\n        raise ValueError(\'native same-canvas background required\')\n    from .checked_research.contact_camera import linear, srgb\n    a = alpha[..., None].astype(float)/255\n    result = srgb(linear(source)*a+linear(background)*(1-a))\n    result[alpha == 255] = source[alpha == 255]\n    result[alpha == 0] = background[alpha == 0]\n    return result\n\n\ndef poisson_candidate(source, alpha, background, *, protected=None,\n                      color_change_permitted=False, mode=\'normal\', strength=.25):\n    """Bounded same-coordinate OpenCV cloning, explicitly not source-exact.\n\n    The alpha-over baseline is always retained by the caller. Protected pixels\n    (e.g. independently reviewed label interiors) are restored byte-for-byte.\n    This restoration can itself create a seam and does not certify print or\n    reflectance accuracy. Frame-touching sources are refused, not padded/cropped.\n    """\n    native_rgb(source)\n    native_rgb(background)\n    native_alpha(alpha, source.shape[:2])\n    if source.shape != background.shape:\n        raise ValueError(\'same native canvas required\')\n    if color_change_permitted is not True:\n        raise ValueError(\'foreground-color-changing research requires explicit opt-in\')\n    if mode not in (\'normal\', \'mixed\', \'monochrome\'):\n        raise ValueError(\'unsupported Poisson mode\')\n    _fraction(strength, .5, \'strength\')\n    hard = alpha >= 128\n    if not hard.any() or hard.all():\n        raise ValueError(\'nontrivial foreground required\')\n    if hard[0].any() or hard[-1].any() or hard[:, 0].any() or hard[:, -1].any():\n        raise ValueError(\'frame-touching mask not eligible for same-canvas Poisson route\')\n    if protected is None:\n        protected = np.zeros(alpha.shape, bool)\n    _binary(protected, alpha.shape, \'protected pixels\', nonempty=False)\n    if np.any(protected & (alpha != 255)):\n        raise ValueError(\'protected pixels must be fully opaque\')\n    y, x = np.where(hard)\n    if x.max()-x.min() < 3 or y.max()-y.min() < 3:\n        raise ValueError(\'foreground too thin for Poisson integration\')\n    mask = hard.astype(np.uint8)*255\n    center = (int((x.min()+x.max()+1)//2), int((y.min()+y.max()+1)//2))\n    flags = {\'normal\': cv2.NORMAL_CLONE, \'mixed\': cv2.MIXED_CLONE,\n             \'monochrome\': cv2.MONOCHROME_TRANSFER}\n    try:\n        cloned = cv2.seamlessClone(cv2.cvtColor(source, cv2.COLOR_RGB2BGR),\n            cv2.cvtColor(background, cv2.COLOR_RGB2BGR), mask, center, flags[mode])\n    except cv2.error as exc:\n        raise RuntimeError(\'OpenCV Poisson solver failed; no fallback promoted\') from exc\n    cloned = cv2.cvtColor(cloned, cv2.COLOR_BGR2RGB)\n    if cloned.shape != source.shape or cloned.dtype != np.uint8:\n        raise ValueError(\'unexpected Poisson output\')\n    baseline = linear_over(source, alpha, background)\n    result = baseline.copy()\n    # Never modify outside hard support or invent semi-transparent optical RGB.\n    mixed = np.rint(baseline.astype(float)*(1-strength)+cloned*strength).astype(np.uint8)\n    result[hard] = mixed[hard]\n    result[protected] = source[protected]\n    changed = np.any(result != baseline, axis=2)\n    delta = np.abs(result.astype(np.int16)-baseline.astype(np.int16))\n    report = dict(method=\'opencv_\'+mode+\'_poisson_color_candidate\',\n        opencv_version=cv2.__version__, strength=strength,\n        source_sha256=digest(source), alpha_sha256=digest(alpha),\n        background_sha256=digest(background), protected_sha256=digest(protected),\n        changed_pixels=int(changed.sum()), maximum_channel_delta=int(delta.max()),\n        protected_pixels=int(protected.sum()), protected_rgb_exact=bool(np.array_equal(result[protected], source[protected])),\n        entire_product_source_exact=False,\n        unchanged_from_alpha_over=not bool(changed.any()),\n        physical_relighting=False, optics_recovered=False,\n        automatic_acceptance=False, sources=SOURCES[2:])\n    return result, changed, report\n'
# ---- embedded module: renderello/candidate_harmonization.py ----
path = base / 'renderello/candidate_harmonization.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Executable multi-candidate studio: normalize, inspect, refine and render.\n\nRun `python -m renderello.candidate_studio --help`. Defaults produce diagnostics,\nnot a selected mask or publishable image. Existing estimated masks are separately\nretained. Review declarations are bound to exact normalized source/mask pixels;\nthey are local engineering inputs, never authenticated user approval.\n"""\nfrom __future__ import annotations\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport re\nimport numpy as np\nfrom PIL import Image\nfrom .candidate_diagnostics import (digest, native_rgb, native_alpha,\n    mask_diagnostics, compare_masks, check_regions, RegionConstraint,\n    disagreement_image)\nfrom .image_intake import inspect_image, IntakePolicy, save_intake\nfrom .candidate_harmonization import linear_over, background_match, poisson_candidate\nfrom .contact_spans import ContactSpan, ContactPolicy, render_contact\n\n\ndef _json(path, data):\n    path.write_text(json.dumps(data, indent=2, allow_nan=False), encoding=\'utf-8\')\n\n\ndef _name(name):\n    if not isinstance(name, str) or not re.fullmatch(r\'[A-Za-z][A-Za-z0-9_-]{0,63}\', name):\n        raise ValueError(\'candidate name must be a safe short identifier\')\n    return name\n\n\ndef read_alpha(path, shape):\n    # An alpha is not a photograph: conversion of RGB/16-bit/P inputs could hide\n    # swapped files or quantization. Orientation must already match normalized RGB.\n    with Image.open(path) as im:\n        if im.mode != \'L\' or getattr(im, \'n_frames\', 1) != 1:\n            raise ValueError(\'single-frame L alpha required; no mode guessing\')\n        if im.size != (shape[1], shape[0]):\n            raise ValueError(\'mask size differs from normalized source\')\n        if im.getexif().get(274, 1) != 1:\n            raise ValueError(\'alpha has EXIF transform; normalize coordinates explicitly\')\n        a = np.array(im)\n    return native_alpha(a, shape)\n\n\ndef load_spec(path, rgb, candidates):\n    data = json.loads(Path(path).read_text(encoding=\'utf-8\'))\n    allowed = {\'schema\', \'source_pixels_sha256\', \'candidate_sha256\', \'regions\',\n               \'selection\', \'material\', \'framing\', \'source_pose\', \'camera\',\n               \'mask_reviewed\', \'contact_reviewed\', \'contact_spans\', \'contact_options\',\n               \'background_match\', \'poisson\'}\n    if not isinstance(data, dict) or set(data)-allowed:\n        raise ValueError(\'unknown studio specification fields\')\n    if data.get(\'schema\') != \'renderello-candidate-studio-v1\':\n        raise ValueError(\'unsupported studio specification\')\n    if data.get(\'source_pixels_sha256\') != digest(rgb):\n        raise ValueError(\'specification belongs to different normalized source pixels\')\n    if set(data.get(\'candidate_sha256\', {})) != set(candidates):\n        raise ValueError(\'specification must bind exactly the supplied candidates\')\n    for name, a in candidates.items():\n        if data[\'candidate_sha256\'][name] != digest(a):\n            raise ValueError(\'specification alpha changed: \'+name)\n    regions = []\n    for item in data.get(\'regions\', []):\n        c = dict(item)\n        c[\'box\'] = tuple(c[\'box\'])\n        regions.append(RegionConstraint(**c))\n    return data, regions\n\n\ndef write_candidate(root, name, rgb, alpha):\n    directory = root / _name(name)\n    directory.mkdir()\n    Image.fromarray(alpha).save(directory/\'alpha.png\')\n    Image.fromarray(np.dstack([rgb, alpha])).save(directory/\'cutout.png\')\n    for label, color in [(\'dark\', (20, 20, 20)), (\'white\', (245, 245, 245)), (\'magenta\', (190, 0, 140))]:\n        bg = np.empty_like(rgb)\n        bg[:] = color\n        Image.fromarray(linear_over(rgb, alpha, bg)).save(directory/(label+\'.png\'))\n    return directory\n\n\ndef route_status(spec, diagnostics, region_results):\n    """Explain blockers without deleting failed candidates or fabricating fields."""\n    blocked = []\n    if spec is None:\n        return [\'SOURCE_BOUND_REVIEW_SPEC_REQUIRED\']\n    if spec.get(\'mask_reviewed\') is not True:\n        blocked.append(\'VISUAL_MASK_REVIEW_REQUIRED\')\n    if spec.get(\'material\') not in (\'opaque\', \'transparent\', \'reflective\', \'unknown\'):\n        blocked.append(\'MATERIAL_DECLARATION_REQUIRED\')\n    elif spec[\'material\'] == \'transparent\':\n        blocked.append(\'TRANSMISSION_FLOW_CLEAN_PLATE_REQUIRED_NOT_ALPHA\')\n    elif spec[\'material\'] == \'reflective\':\n        blocked.append(\'REFLECTION_MATCH_UNVERIFIED_SOURCE_LOCK_NOT_RELIGHTING\')\n    elif spec[\'material\'] == \'unknown\':\n        blocked.append(\'MATERIAL_UNKNOWN\')\n    if spec.get(\'framing\') != \'complete\':\n        blocked.append(\'COMPLETE_SOURCE_NOT_REVIEWED_NO_INVENTED_PARTS\')\n    if spec.get(\'source_pose\') in (None, \'unknown\', \'suspended\'):\n        blocked.append(\'POSE_NOT_ELIGIBLE_FOR_SURFACE_CONTACT\')\n    if any(not r[\'passed\'] for r in region_results):\n        blocked.append(\'SPATIAL_INTENT_CONSTRAINT_FAILED\')\n    severe = {\'EMPTY_MASK\', \'FULL_FRAME_FOREGROUND\', \'NO_HIGH_ALPHA_SUPPORT\',\n              \'FRAME_TOUCH_NOT_PROOF_OF_COMPLETE_PRODUCT\'}\n    blocked.extend(flag for flag in diagnostics[\'flags\'] if flag in severe)\n    return blocked\n\n\ndef run(source, output, masks, *, spec_path=None, background=None,\n        allow_colour_conversion=False, birefnet_checkpoint=None,\n        matting_candidate=None, trimap=None, matting_method=\'cf\', native_results=None, rgba_results=None,\n        instance_reviews=None):\n    output = Path(output)\n    if output.exists():\n        raise FileExistsError(\'preserve prior attempt; choose a fresh output directory\')\n    intake = inspect_image(source, IntakePolicy(max_pixels=4_000_000,\n        allow_colour_conversion=allow_colour_conversion))\n    try:\n        rgb = np.array(intake.image.convert(\'RGB\'))\n        native_rgb(rgb)\n        source_alpha = np.array(intake.image.getchannel(\'A\')) if intake.image.mode == \'RGBA\' else None\n        output.mkdir(parents=True)\n        (output/\'INCOMPLETE\').touch()\n        save_intake(intake, output/\'input\')\n        # Always use orientation/color-normalized RGB for all downstream workers.\n        normalized_source = output/\'source-rgb.png\'\n        Image.fromarray(rgb).save(normalized_source)\n        candidates = {}\n        for name, path in masks.items():\n            _name(name)\n            candidates[name] = read_alpha(path, rgb.shape[:2])\n        if not candidates and source_alpha is not None:\n            candidates[\'embedded-alpha\'] = source_alpha.copy()\n        if birefnet_checkpoint is not None:\n            if source_alpha is not None and np.any(source_alpha != 255):\n                raise ValueError(\'model RGB interpretation of transparent source requires an explicit flattening policy\')\n            if \'BiRefNet\' in candidates:\n                raise ValueError(\'duplicate BiRefNet candidate\')\n            from .bilateral_mask import guarded_file\n            guarded_file(normalized_source, output/\'birefnet-worker\', Path(birefnet_checkpoint))\n            candidates[\'BiRefNet\'] = read_alpha(output/\'birefnet-worker/estimated-alpha.png\', rgb.shape[:2])\n        if not 1 <= len(candidates) <= 8:\n            raise ValueError(\'supply 1..8 candidates or an embedded source alpha\')\n        if source_alpha is not None:\n            for a in candidates.values():\n                if np.any(a > source_alpha):\n                    raise ValueError(\'candidate exceeds embedded source alpha; no hidden RGB reveal or silent multiplication\')\n        refinement_reviews = {}\n        if matting_candidate is not None:\n            if matting_candidate not in candidates or trimap is None:\n                raise ValueError(\'matting needs a named initial candidate and explicit reviewed trimap\')\n            name = _name(matting_candidate+\'-refined\')\n            if name in candidates:\n                raise ValueError(\'refined candidate name collides\')\n            if len(candidates) >= 8:\n                raise ValueError(\'no room for additional refined candidate\')\n            from .matte_change_review import audit_trimap, review_change\n            trimap_array = read_alpha(trimap, rgb.shape[:2])\n            seed_review, seed_conflicts = audit_trimap(candidates[matting_candidate], trimap_array)\n            _json(output/\'trimap-review.json\', seed_review)\n            Image.fromarray(seed_conflicts.astype(np.uint8)*255).save(output/\'trimap-seed-conflicts.png\')\n            initial = output/\'initial-for-matting.png\'\n            Image.fromarray(candidates[matting_candidate]).save(initial)\n            from .matting import guarded_file\n            guarded_file(normalized_source, initial, output/\'matting-worker\',\n                trimap_path=Path(trimap), method=matting_method, edge_colors=False)\n            candidates[name] = read_alpha(output/\'matting-worker/refined-alpha.png\', rgb.shape[:2])\n            delta_review, delta_image = review_change(candidates[matting_candidate], candidates[name])\n            refinement_reviews[name] = dict(seeds=seed_review, changes=delta_review)\n            _json(output/\'matte-change-review.json\', delta_review)\n            Image.fromarray(delta_image).save(output/\'matte-alpha-change.png\')\n            if source_alpha is not None and np.any(candidates[name] > source_alpha):\n                raise ValueError(\'refined candidate exceeds embedded source alpha\')\n        imported_results = {}\n        for name, request in (native_results or {}).items():\n            _name(name)\n            if name in candidates or len(candidates) >= 8:\n                raise ValueError(\'imported candidate collides or exceeds eight-candidate budget\')\n            if not isinstance(request, dict) or set(request) != {\'directory\', \'initial\'}:\n                raise ValueError(\'native import needs directory and initial candidate\')\n            initial = request[\'initial\']\n            if initial not in candidates:\n                raise ValueError(\'native import initial alpha candidate is missing\')\n            from .native_result_import import load_verified_result\n            alpha, evidence = load_verified_result(request[\'directory\'], rgb, candidates[initial])\n            if source_alpha is not None and np.any(alpha > source_alpha):\n                raise ValueError(\'imported candidate exceeds embedded source alpha\')\n            candidates[name] = alpha\n            refinement_reviews[name] = dict(seeds=evidence[\'seeds\'], changes=evidence[\'changes\'])\n            imported_results[name] = evidence\n            _json(output/(\'native-import-\'+name+\'.json\'), evidence)\n        rgba_imports = {}\n        for name, request in (rgba_results or {}).items():\n            _name(name)\n            if name in candidates or len(candidates) >= 8:\n                raise ValueError(\'imported RGBA candidate collides or exceeds eight-candidate budget\')\n            if not isinstance(request, dict) or set(request) != {\'path\', \'sha256\', \'initial\'}:\n                raise ValueError(\'RGBA import needs path, sha256 and initial candidate\')\n            if request[\'initial\'] not in candidates:\n                raise ValueError(\'RGBA import initial alpha candidate is missing\')\n            from .source_locked_rgba_import import load_source_locked_rgba\n            from .matte_change_review import review_change\n            alpha, evidence = load_source_locked_rgba(request[\'path\'], rgb, request[\'sha256\'])\n            if source_alpha is not None and np.any(alpha > source_alpha):\n                raise ValueError(\'imported RGBA exceeds embedded source alpha\')\n            change, _ = review_change(candidates[request[\'initial\']], alpha)\n            candidates[name] = alpha\n            refinement_reviews[name] = dict(changes=change)\n            rgba_imports[name] = evidence\n            _json(output/(\'rgba-import-\'+name+\'.json\'), evidence)\n        spec, regions = load_spec(spec_path, rgb, candidates) if spec_path else (None, [])\n        # Optional API declarations bind exactly every final candidate, not only\n        # the selected one. A failed core remains available for diagnostics but\n        # cannot enter background rendering. No automatic instance detection.\n        if instance_reviews is not None and (not isinstance(instance_reviews, dict)\n                or set(instance_reviews) != set(candidates)):\n            raise ValueError(\'instance reviews must bind exactly every final candidate\')\n        root = output/\'candidates\'\n        root.mkdir()\n        report = dict(schema=\'renderello-candidate-studio-result-v1\',\n            source_encoded_sha256=intake.record[\'source_sha256\'],\n            normalized_source_sha256=digest(rgb), intake=intake.record,\n            candidates={}, comparisons=compare_masks(rgb, candidates) if len(candidates) > 1 else [],\n            automatically_selected=None, accepted_count=0, quality=\'NOT_EVALUATED\',\n            physical_relighting=False, real_glass_validated=False,\n            refinement_reviews=refinement_reviews, native_result_imports=imported_results,\n            source_locked_rgba_imports=rgba_imports)\n        for name, a in candidates.items():\n            directory = write_candidate(root, name, rgb, a)\n            diagnostic = mask_diagnostics(rgb, a)\n            region_results = check_regions(a, regions)\n            blocked = route_status(spec, diagnostic, region_results)\n            if name in refinement_reviews:\n                for stage, review in refinement_reviews[name].items():\n                    blocked.extend(\'REFINEMENT_\'+stage.upper()+\'_\'+flag for flag in review[\'blockers\'])\n            presence = None\n            if instance_reviews is not None:\n                from .reviewed_instance_presence import review_instance_presence\n                request = instance_reviews[name]\n                if not isinstance(request, dict) or set(request) != {\'regions\', \'bindings\', \'reviewed\', \'review_note\'}:\n                    raise ValueError(\'exact reviewed instance-presence request required\')\n                presence = review_instance_presence(rgb, np.dstack([rgb, a]), **request)\n                blocked.extend(\'INSTANCE_PRESENCE_\'+flag for flag in presence[\'blockers\'])\n                _json(directory/\'instance-presence.json\', presence)\n            record = dict(diagnostics=diagnostic, constraints=region_results, render_blockers=blocked)\n            if presence is not None:\n                record[\'instance_presence\'] = presence\n            report[\'candidates\'][name] = record\n            _json(directory/\'diagnostics.json\', record)\n        for row in report[\'comparisons\']:\n            Image.fromarray(disagreement_image(candidates[row[\'left\']], candidates[row[\'right\']])).save(\n                output/(\'disagreement-\'+row[\'left\']+\'-\'+row[\'right\']+\'.png\'))\n        selected = spec.get(\'selection\') if spec else None\n        if background is not None:\n            if selected not in candidates:\n                raise ValueError(\'render requires a source-bound explicit candidate selection\')\n            blocked = report[\'candidates\'][selected][\'render_blockers\']\n            if blocked:\n                report[\'render_status\'] = \'BLOCKED\'\n                report[\'render_blockers\'] = blocked\n            else:\n                # Strict source-sized scene. No automatic crop/resize of either.\n                with Image.open(background) as im:\n                    if im.mode != \'RGB\' or im.size != (rgb.shape[1], rgb.shape[0]) or im.info.get(\'icc_profile\') or im.getexif().get(274, 1) != 1:\n                        raise ValueError(\'normalized native-sized unprofiled RGB scene required\')\n                    bg = np.array(im)\n                a = candidates[selected]\n                policy = ContactPolicy(camera=spec[\'camera\'], material=spec[\'material\'],\n                    framing=spec[\'framing\'], source_pose=spec[\'source_pose\'],\n                    source_sha256=digest(rgb), alpha_sha256=digest(a),\n                    reviewed=spec.get(\'contact_reviewed\', False), **spec.get(\'contact_options\', {}))\n                spans = [ContactSpan(**s) for s in spec.get(\'contact_spans\', [])]\n                if spec.get(\'background_match\'):\n                    settings = dict(spec[\'background_match\'])\n                    sample_path = settings.pop(\'sample_mask\')\n                    samples = read_alpha(sample_path, a.shape)\n                    if not np.isin(samples, [0, 255]).all():\n                        raise ValueError(\'background sample mask must be binary\')\n                    bg, harmonization = background_match(bg, rgb, a, samples == 255, **settings)\n                    Image.fromarray(bg).save(output/\'harmonized-background.png\')\n                    report[\'background_harmonization\'] = harmonization\n                result, shade, meta = render_contact(rgb, a, bg, spans, policy)\n                Image.fromarray(result).save(output/\'source-locked-contact-candidate.png\')\n                Image.fromarray(np.rint(shade*255).astype(np.uint8)).save(output/\'contact-attenuation.png\')\n                report[\'contact_render\'] = meta\n                report[\'render_status\'] = \'RESEARCH_CANDIDATE_NOT_ACCEPTED\'\n                if spec.get(\'poisson\'):\n                    settings = dict(spec[\'poisson\'])\n                    protected_path = settings.pop(\'protected_mask\', None)\n                    protected = read_alpha(protected_path, a.shape) if protected_path else None\n                    if protected is not None and not np.isin(protected, [0, 255]).all():\n                        raise ValueError(\'protected mask must be binary\')\n                    from .checked_research.contact_camera import linear, srgb\n                    shaded_bg = srgb(linear(bg)*(1-shade[..., None]))\n                    result, changed, meta = poisson_candidate(rgb, a, shaded_bg,\n                        protected=protected == 255 if protected is not None else None, **settings)\n                    Image.fromarray(result).save(output/\'color-changing-poisson-candidate.png\')\n                    Image.fromarray(changed.astype(np.uint8)*255).save(output/\'poisson-changed-pixels.png\')\n                    report[\'poisson\'] = meta\n        _json(output/\'report.json\', report)\n        files = {str(p.relative_to(output)): hashlib.sha256(p.read_bytes()).hexdigest()\n                 for p in output.rglob(\'*\') if p.is_file() and p.name != \'INCOMPLETE\'}\n        _json(output/\'artifact-hashes.json\', files)\n        (output/\'INCOMPLETE\').unlink()\n        (output/\'DIAGNOSTICS_COMPLETE_NOT_ACCEPTED\').touch()\n        return report\n    except Exception as exc:\n        if output.is_dir():\n            _json(output/\'failure.json\', dict(status=\'FAILED_NOT_ACCEPTED\',\n                error_type=type(exc).__name__, message=str(exc)))\n        raise\n    finally:\n        intake.close()\n\n\ndef main(argv=None):\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--source\', type=Path, required=True)\n    p.add_argument(\'--out\', type=Path, required=True)\n    p.add_argument(\'--alpha\', action=\'append\', default=[], metavar=\'NAME=PATH\')\n    p.add_argument(\'--spec\', type=Path)\n    p.add_argument(\'--background\', type=Path)\n    p.add_argument(\'--birefnet-checkpoint\', type=Path)\n    p.add_argument(\'--allow-colour-conversion\', action=\'store_true\')\n    p.add_argument(\'--native-result\', action=\'append\', default=[], metavar=\'NAME=INITIAL=PATH\',\n        help=\'Replay and import a saved native-matting result, never automatically accept it\')\n    p.add_argument(\'--rgba-result\', action=\'append\', default=[], metavar=\'NAME=INITIAL=SHA256=PATH\',\n        help=\'Verify pinned native RGBA against source RGB; never automatically accept it\')\n    p.add_argument(\'--matting-candidate\')\n    p.add_argument(\'--trimap\', type=Path)\n    p.add_argument(\'--matting-method\', choices=[\'cf\', \'lkm\', \'guided\'], default=\'cf\')\n    a = p.parse_args(argv)\n    masks = {}\n    for item in a.alpha:\n        if \'=\' not in item:\n            p.error(\'--alpha needs NAME=PATH\')\n        name, path = item.split(\'=\', 1)\n        if name in masks:\n            p.error(\'duplicate candidate name\')\n        masks[name] = Path(path)\n    native_results = {}\n    for item in a.native_result:\n        fields = item.split(\'=\', 2)\n        if len(fields) != 3 or not all(fields):\n            p.error(\'--native-result needs NAME=INITIAL=PATH\')\n        name, initial, path = fields\n        if name in native_results:\n            p.error(\'duplicate native-result candidate name\')\n        native_results[name] = dict(initial=initial, directory=Path(path))\n    rgba_results = {}\n    for item in a.rgba_result:\n        fields = item.split(\'=\', 3)\n        if len(fields) != 4 or not all(fields):\n            p.error(\'--rgba-result needs NAME=INITIAL=SHA256=PATH\')\n        name, initial, sha256, path = fields\n        if name in rgba_results:\n            p.error(\'duplicate RGBA candidate name\')\n        rgba_results[name] = dict(initial=initial, sha256=sha256, path=Path(path))\n    report = run(a.source, a.out, masks, spec_path=a.spec, background=a.background,\n        allow_colour_conversion=a.allow_colour_conversion,\n        birefnet_checkpoint=a.birefnet_checkpoint, matting_candidate=a.matting_candidate,\n        trimap=a.trimap, matting_method=a.matting_method, native_results=native_results,\n        rgba_results=rgba_results)\n    print(json.dumps(dict(status=\'diagnostics_complete_not_accepted\',\n        candidates=len(report[\'candidates\']), accepted=0)))\n\n\nif __name__ == \'__main__\':\n    main()\n'
# ---- embedded module: renderello/candidate_studio.py ----
path = base / 'renderello/candidate_studio.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


### embedded source: `renderello/checked_research/`

In [ ]:
SRC = '"""Opt-in checked research helpers. Not the default product pipeline."""\n'
# ---- embedded module: renderello/checked_research/__init__.py ----
path = base / 'renderello/checked_research/__init__.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Experimental shared reference/latent-anchor coordinates. No image generation.\n\nThis does not recover a mask, remove the old background, relight the source or\nprove the model will respect placement. Native RGB is copied only into the INPUT\ncanvas, never onto a generated output. Use the returned canvas for both model\nreference and latent encoding; verify model output separately.\n"""\nfrom __future__ import annotations\nimport numpy as np\nfrom ..candidate_diagnostics import native_rgb, native_alpha, digest\n\n\ndef conditioning_canvas(source, mask, *, size=(896, 896), offset=None,\n                        padding=(127, 127, 127)):\n    """Return native source placement, same-coordinate anchor mask, provenance.\n\n    size is (width,height). The mask is caller-supplied experimental protection,\n    not evidence of object completeness or physical alpha/transparency. No crop,\n    resize, erosion, thresholding or hidden automatic region selection occurs.\n    """\n    native_rgb(source)\n    native_alpha(mask, source.shape[:2])\n    if not isinstance(size, (tuple, list)) or len(size) != 2 or any(type(v) is not int for v in size):\n        raise ValueError(\'two integer canvas dimensions required\')\n    w, h = size\n    if min(w, h) < 32 or w % 32 or h % 32 or w*h > 4_000_000:\n        raise ValueError(\'canvas must be positive multiples of32 within4MP\')\n    sh, sw = source.shape[:2]\n    if sw > w or sh > h:\n        raise ValueError(\'native source does not fit; no silent resize/crop\')\n    if offset is None:\n        offset = ((w-sw)//2, (h-sh)//2)\n    if not isinstance(offset, (tuple, list)) or len(offset) != 2 or any(type(v) is not int for v in offset):\n        raise ValueError(\'two integer placement coordinates required\')\n    x, y = offset\n    if min(x,y) < 0 or x+sw > w or y+sh > h:\n        raise ValueError(\'source rectangle outside canvas\')\n    if not isinstance(padding, (tuple, list)) or len(padding) != 3 or any(type(v) is not int or not 0<=v<=255 for v in padding):\n        raise ValueError(\'three integer padding channels0..255 required\')\n    if not mask.any():\n        raise ValueError(\'empty protection mask\')\n    canvas = np.empty((h,w,3), np.uint8)\n    canvas[:] = padding\n    canvas[y:y+sh,x:x+sw] = source\n    placed_mask = np.zeros((h,w),np.uint8)\n    placed_mask[y:y+sh,x:x+sw] = mask\n    yy,xx = np.where(mask>0)\n    support_box = [x+int(xx.min()), y+int(yy.min()), x+int(xx.max())+1, y+int(yy.max())+1]\n    metadata = dict(method=\'NATIVE_SHARED_INPUT_CONDITIONING_CANVAS\',\n        source_shape=list(source.shape), canvas_shape=list(canvas.shape),\n        source_rectangle=[x,y,x+sw,y+sh], protection_support_box=support_box,\n        protection_height_fraction=(support_box[3]-support_box[1])/h,\n        padding=list(padding), source_pixels_sha256=digest(source),\n        input_mask_pixels_sha256=digest(mask), canvas_pixels_sha256=digest(canvas),\n        placed_mask_pixels_sha256=digest(placed_mask), source_resampled=False,\n        source_cropped=False, old_background_removed=False,\n        input_rgb_native_exact=True, output_rgb_paste_performed=False,\n        geometry_preservation=\'NOT_EVALUATED\', semantic_mask_accuracy=\'NOT_EVALUATED\',\n        physical_relighting=False, quality_accepted=False,\n        warnings=[\'Use exactly this canvas for model reference and latent encoding.\',\n                  \'Source rectangle includes its original background.\',\n                  \'Protection mask can preserve wrong parts or miss details.\',\n                  \'Matching input coordinates does not prove generated geometry or text.\'])\n    return canvas, placed_mask, metadata\n'
# ---- embedded module: renderello/checked_research/conditioning_canvas.py ----
path = base / 'renderello/checked_research/conditioning_canvas.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Reviewer-declared camera/support study, not inferred geometry or glass optics.\n\nNo foreground resampling or colour changes. Scene illumination is compressed to\nan explicitly designed smooth field to prevent invented props in the final plate.\nThe unmodified generated plate is retained as evidence. Production flags fail\nclosed; they are not an automatic acceptance classifier.\n"""\nfrom dataclasses import dataclass\nimport numpy as np\nfrom scipy.ndimage import gaussian_filter, label\n\n\n@dataclass(frozen=True)\nclass SceneSpec:\n    camera: str\n    light: str = \'left\'\n    transparent: bool = False\n    cropped: bool = False\n    multiobject: bool = False\n    support_reviewed: bool = False\n\n    def __post_init__(self):\n        if self.camera not in (\'front\', \'elevated\', \'flatlay\'):\n            raise ValueError(\'explicit front/elevated/flatlay camera required\')\n        if self.light not in (\'left\', \'right\', \'diffuse\'):\n            raise ValueError(\'explicit light direction required\')\n        if any(type(getattr(self, k)) is not bool for k in\n               (\'transparent\', \'cropped\', \'multiobject\', \'support_reviewed\')):\n            raise ValueError(\'boolean review declarations required\')\n\n\ndef _image(rgb):\n    if not isinstance(rgb, np.ndarray) or rgb.dtype != np.uint8:\n        raise ValueError(\'uint8 RGB image required\')\n    if rgb.ndim != 3 or rgb.shape[2] != 3 or min(rgb.shape[:2]) < 2:\n        raise ValueError(\'nontrivial RGB image required\')\n    if rgb.shape[0]*rgb.shape[1] > 8_000_000:\n        raise ValueError(\'image exceeds research allocation limit\')\n\n\ndef linear(rgb):\n    x = np.asarray(rgb, dtype=np.float64)/255.\n    return np.where(x <= .04045, x/12.92, ((x+.055)/1.055)**2.4)\n\n\ndef srgb(x):\n    x = np.clip(x, 0., 1.)\n    return np.rint(np.where(x <= .0031308, x*12.92,\n                           1.055*x**(1/2.4)-.055)*255).astype(np.uint8)\n\n\ndef layout(alpha, occupancy=.72):\n    """Crop empty source margins only. Native pixels retained; integer translation."""\n    if (not isinstance(alpha, np.ndarray) or alpha.dtype != np.uint8\n            or alpha.ndim != 2 or min(alpha.shape) < 2):\n        raise ValueError(\'native uint8 alpha required\')\n    if alpha.size > 4_000_000 or not np.any(alpha >= 128):\n        raise ValueError(\'nonempty bounded opaque support required\')\n    if isinstance(occupancy, bool) or not np.isfinite(occupancy) or not .4 <= occupancy <= .8:\n        raise ValueError(\'occupancy must be .4 through .8\')\n    yy, xx = np.nonzero(alpha > 0)\n    box = (int(xx.min()), int(yy.min()), int(xx.max()+1), int(yy.max()+1))\n    width, height = box[2]-box[0], box[3]-box[1]\n    side = max(256, int(np.ceil(max(width, height)/occupancy)))\n    if side > 2400:\n        raise ValueError(\'native composition exceeds allocation limit\')\n    left = (side-width)//2\n    top = int((side-height)*.52)\n    return box, (side, side), (left, top)\n\n\ndef smooth_scene(raw, shape, spec):\n    """Robust quadratic illumination field; no generated object texture survives.\n\n    This intentionally simplifies the scene. It is not a generated photograph or\n    automatic semantic prop detection, and cannot certify source-light matching.\n    """\n    from PIL import Image\n    _image(raw)\n    h, w = shape\n    if h < 2 or w < 2 or h*w > 6_000_000:\n        raise ValueError(\'bounded target dimensions required\')\n    sample = np.asarray(Image.fromarray(raw).resize((32, 32))).astype(float)/255.\n    sy, sx = np.mgrid[-1:1:32j, -1:1:32j]\n    basis = np.stack([np.ones_like(sx), sx, sy, sx*sy, sx*sx, sy*sy], -1)\n    X = basis.reshape(-1, 6); Y = sample.reshape(-1, 3)\n    # Huber-style reweighting reduces local generated props before low-order fit.\n    weight = np.ones(len(X))\n    for _ in range(5):\n        co = np.linalg.lstsq(X*weight[:, None], Y*weight[:, None], rcond=None)[0]\n        residual = np.linalg.norm(X@co-Y, axis=1)\n        scale = max(.01, float(np.median(residual))*1.4826)\n        weight = np.minimum(1., 1.5*scale/np.maximum(residual, 1e-6))\n    y, x = np.mgrid[-1:1:complex(h), -1:1:complex(w)]\n    B = np.stack([np.ones_like(x), x, y, x*y, x*x, y*y], -1)\n    field = B@co\n    # Keep a restrained commercial studio field instead of large fitted blobs.\n    mean = np.median(Y, axis=0)\n    field = mean + .38*(field-mean)\n    lateral = (x if spec.light == \'right\' else -x) if spec.light != \'diffuse\' else 0*x\n    field += .025*lateral[..., None]\n    if spec.camera != \'flatlay\':\n        # Smooth continuous floor, deliberately no horizon/ledge to float above.\n        field -= .025*np.maximum(y, 0)[..., None]\n    return np.rint(np.clip(field, .08, .96)*255).astype(np.uint8)\n\n\ndef shadow_map(alpha, shape, position, spec):\n    """Flatlay full silhouette or upright lower-contact contour plus cast shadow.\n\n    Contact starts at the actual native lower contour. No fabricated 3D pose.\n    Upright cast uses a declared 2D height projection, never measured geometry.\n    """\n    h, w = shape; x, y = position\n    ah, aw = alpha.shape\n    if x < 0 or y < 0 or x+aw > w or y+ah > h:\n        raise ValueError(\'native alpha must fit canvas\')\n    a = alpha.astype(float)/255.; direction = 1 if spec.light == \'left\' else -1\n    if spec.camera == \'flatlay\':\n        full = np.zeros(shape); full[y:y+ah, x:x+aw] = a\n        offset = max(2, round(max(aw, ah)*.009))\n        from scipy.ndimage import shift\n        cast = shift(full, (offset, direction*offset), order=1, mode=\'constant\')\n        tight = gaussian_filter(cast, max(1.2, max(aw, ah)*.003))*.24\n        broad = gaussian_filter(cast, max(3., max(aw, ah)*.014))*.12\n        return np.clip(1-(1-tight)*(1-broad), 0, .45)\n    support = alpha >= 128\n    bottoms = np.full(aw, -1, dtype=int)\n    for col in np.flatnonzero(support.any(axis=0)):\n        bottoms[col] = np.flatnonzero(support[:, col])[-1]\n    if not np.any(bottoms >= 0):\n        raise ValueError(\'nonempty support required for upright shadow\')\n    floor = int(bottoms.max())\n    span = np.flatnonzero(bottoms >= floor-max(2, round(ah*.035)))\n    # The previous blurred one-pixel line lost contact density and its projected\n    # mass sat below the object. Build an attached footprint with a finite-width\n    # core, then a rapidly decaying tail. The first uncovered pixel is strongest.\n    contact = np.zeros(shape)\n    reach = max(4, round(ah*.025))\n    decay = max(1.2, ah*.004)\n    for col in span:\n        row = y+bottoms[col]+1\n        length = min(reach, h-row)\n        if length > 0:\n            contact[row:row+length, x+col] = .62*np.exp(-np.arange(length)/decay)\n    # Horizontal-only smoothing cannot move the footprint away from its base.\n    from scipy.ndimage import gaussian_filter1d\n    contact = gaussian_filter1d(contact, max(.6, aw*.0015), axis=1)\n    # A broad tail must start on the same support, not at an offset ellipse.\n    tail = gaussian_filter(contact, (max(2., ah*.008), max(2., aw*.012)))*.45\n    # Shadow length is an explicit 2D assumption. Compress toward the attached\n    # support plane and attenuate with height, avoiding a second detached island.\n    cast = np.zeros(shape)\n    for row in range(ah):\n        height = max(0, floor-row)/max(1, floor)\n        cols = np.flatnonzero(a[row] > 0)\n        xx = x+cols+np.rint(direction*height*ah*.12).astype(int)\n        yy = np.full(len(cols), y+floor+round(height*ah*.025), dtype=int)\n        valid = (xx >= 0)&(xx < w)&(yy >= 0)&(yy < h)\n        np.maximum.at(cast, (yy[valid], xx[valid]), a[row, cols[valid]]*.10*(1-height*.7))\n    cast = gaussian_filter(cast, max(1.2, aw*.006))\n    return np.clip(1-(1-contact)*(1-tail)*(1-cast), 0, .68)\n\n\ndef composite(source, alpha, raw_scene, spec):\n    _image(source); _image(raw_scene)\n    if not isinstance(spec, SceneSpec): raise ValueError(\'SceneSpec required\')\n    if alpha.shape != source.shape[:2]: raise ValueError(\'source/alpha pairing required\')\n    box, (w, h), position = layout(alpha)\n    x0, y0, x1, y1 = box\n    rgb = source[y0:y1, x0:x1]; a = alpha[y0:y1, x0:x1]\n    bg = smooth_scene(raw_scene, (h, w), spec)\n    shade = shadow_map(a, (h, w), position, spec)\n    bg_shadow = srgb(linear(bg)*(1-shade[..., None]))\n    x, y = position; ph, pw = a.shape\n    out = bg_shadow.copy(); dst = out[y:y+ph, x:x+pw]\n    weight = a[..., None].astype(float)/255.\n    dst[:] = srgb(linear(rgb)*weight+linear(dst)*(1-weight))\n    dst[a == 255] = rgb[a == 255]\n    _, number = label(alpha >= 128)\n    flags = []\n    if spec.transparent: flags.append(\'transparent-material-optics-not-recovered\')\n    if spec.cropped: flags.append(\'source-cropped-product-not-recovered\')\n    if spec.multiobject: flags.append(\'multi-object-membership-and-ghosts-need-review\')\n    if not spec.support_reviewed: flags.append(\'physical-support-not-reviewed\')\n    if number > 30: flags.append(\'fragmented-mask-needs-review\')\n    return out, bg_shadow, shade, dict(\n        sourceCrop=list(box), nativePlacement=list(position), canvas=[w, h],\n        foregroundResampled=False, changedOpaquePixels=int(np.any(dst[a == 255] != rgb[a == 255], axis=1).sum()),\n        opaquePixels=int((a == 255).sum()), camera=spec.camera, light=spec.light,\n        shadowGeometry=\'attached-footprint-declared-2D-approximation\', sceneRepresentation=\'robust-quadratic-generated-illumination-field\',\n        sourceRelit=False, glassOpticsRecovered=False, semanticPropDetection=False,\n        alphaUnchanged=True, maskComponents=int(number), unresolvedFlags=flags,\n        commercialAcceptance=None, acceptanceRequiresVisualReview=True)\n'
# ---- embedded module: renderello/checked_research/contact_camera.py ----
path = base / 'renderello/checked_research/contact_camera.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Experimental background-only fade around an exact reviewed context mask.\n\nNo alpha solve, relighting, new physical contact, glass optics or model call.\nThe preserved context includes old-scene RGB around all nonzero product alpha.\n"""\nimport numpy as np\nfrom scipy.ndimage import distance_transform_edt\nfrom ..candidate_diagnostics import native_rgb,native_alpha,digest\nfrom ..source_context_extension import extend_source_context\n\n\ndef transition_context(rgb,alpha,context,*,source_sha256,alpha_sha256,context_sha256,\n                       reviewed=False,note=\'\',material=\'unknown\',canvas=(1024,1024),offset=(0,0),feather=20):\n    native_rgb(rgb);native_alpha(alpha,rgb.shape[:2])\n    if reviewed is not True or material!=\'opaque\':raise ValueError(\'explicit opaque context review required\')\n    if not isinstance(note,str)or not note.strip()or len(note)>2000:raise ValueError(\'bounded review note required\')\n    if not isinstance(context,np.ndarray)or context.dtype!=np.bool_ or context.shape!=alpha.shape or not context.any():raise ValueError(\'native boolean context mask required\')\n    if digest(rgb)!=source_sha256 or digest(alpha)!=alpha_sha256 or digest(context)!=context_sha256:raise ValueError(\'source/alpha/context binding mismatch\')\n    if np.any((alpha>0)&~context):raise ValueError(\'context must retain every nonzero source-alpha pixel\')\n    if type(feather)is not int or not 2<=feather<=64:raise ValueError(\'bounded2..64pixel feather required\')\n    d=distance_transform_edt(~context);weight=np.clip(1-d/feather,0,1);weight=weight*weight*(3-2*weight)\n    if any(np.any(v>0)for v in(weight[0],weight[-1],weight[:,0],weight[:,-1])):\n        raise ValueError(\'protected context+feather reaches source frame; no hidden rectangular seam\')\n    extension,record=extend_source_context(rgb,alpha,material=material,reviewed_source_sha256=source_sha256,\n        reviewed_alpha_sha256=alpha_sha256,canvas=canvas,offset=offset,smooth_outer=True)\n    h,w=alpha.shape;cw,ch=canvas;dx,dy=offset;coef=np.array(record[\'fit_coefficients\']);yy,xx=np.mgrid[:ch,:cw]\n    plane=np.clip(coef[0]+np.clip((xx-dx)/max(1,w-1),0,1)[...,None]*coef[1]+np.clip((yy-dy)/max(1,h-1),0,1)[...,None]*coef[2],0,255)\n    output=np.rint(plane).astype(np.uint8);local=output[dy:dy+h,dx:dx+w]\n    local[:]=np.rint(rgb*weight[...,None]+plane[dy:dy+h,dx:dx+w]*(1-weight[...,None])).astype(np.uint8)\n    local[context]=rgb[context]\n    if not np.array_equal(local[context],rgb[context]):raise RuntimeError(\'protected context changed\')\n    change=np.any(local!=rgb,axis=2)\n    record.update(schema=\'reviewed-context-transition.v1\',context_sha256=context_sha256,review_note=note,\n        context_is_caller_hypothesis=True,source_pixels_preserved=int(context.sum()),source_changed_pixels=int(change.sum()),\n        background_changed_pixels=int(change.sum()),protected_context_changed_pixels=0,source_alpha_support_changed_pixels=0,\n        whole_source_frame_locked=False,source_product_and_context_rgb_locked=True,feather=feather,\n        source_contact_retained=\'ONLY_IF_CALLER_CONTEXT_INCLUDES_IT_NOT_VERIFIED\',background_replaced=\'FAR_BACKGROUND_ONLY\',\n        photo_accepted=False,new_template_executed=False)\n    record[\'limitations\']=[x for x in record[\'limitations\'] if x!=\'The complete old scene remains. This is added copy space, not a product staging template.\']\n    record[\'limitations\'].extend([\'Old-scene pixels remain around the whole protected product; a halo is still possible.\',\n        \'Far background RGB changes within the original photograph; entire-source RGB is not locked.\'])\n    return output,record\n'
# ---- embedded module: renderello/checked_research/context_transition.py ----
path = base / 'renderello/checked_research/context_transition.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Checked bilinear correspondence rendering. This is a component, not glass synthesis.\nCoordinates are normalized absolute background coordinates, x then y, align_corners=True.\nDefault refuses out-of-image requested coordinates. Caller may explicitly select mask or\nborder, with a manifest recording exclusions/clamping. No attenuation/reflections inferred.\n"""\nimport numpy as np\n\n\ndef render_correspondence(background, correspondence, valid, *, out_of_image=\'reject\'):\n    if not isinstance(background, np.ndarray) or background.dtype != np.uint8 or background.ndim != 3 or background.shape[2] != 3:\n        raise ValueError(\'uint8 RGB background required\')\n    if not isinstance(correspondence, np.ndarray) or correspondence.dtype.kind != \'f\' or correspondence.ndim != 3 or correspondence.shape[2] != 2 or not np.isfinite(correspondence).all():\n        raise ValueError(\'finite floating HxWx2 normalized correspondence required\')\n    h, w = correspondence.shape[:2]\n    bh, bw = background.shape[:2]\n    if min(h, w, bh, bw) < 2 or max(h*w, bh*bw) > 4194304:\n        raise ValueError(\'dimensions >=2 and <=4MP required\')\n    if not isinstance(valid, np.ndarray) or valid.dtype != np.bool_ or valid.shape != (h, w) or not valid.any():\n        raise ValueError(\'nonempty matching boolean valid mask required\')\n    if out_of_image not in (\'reject\', \'mask\', \'border\'):\n        raise ValueError(\'explicit reject, mask or border policy required\')\n    inside = ((correspondence >= 0) & (correspondence <= 1)).all(2)\n    outside = valid & ~inside\n    n = int(outside.sum())\n    if n and out_of_image == \'reject\':\n        raise ValueError(f\'{n} requested correspondences outside background; choose an explicit policy\')\n    keep = valid & inside if out_of_image == \'mask\' else valid.copy()\n    if not keep.any():\n        raise ValueError(\'no renderable pixels after policy\')\n    c = np.clip(correspondence, 0, 1)\n    x = c[:,:,0].astype(np.float64) * (bw-1)\n    y = c[:,:,1].astype(np.float64) * (bh-1)\n    x0, y0 = np.floor(x).astype(int), np.floor(y).astype(int)\n    x1, y1 = np.minimum(x0+1, bw-1), np.minimum(y0+1, bh-1)\n    fx, fy = (x-x0)[:,:,None], (y-y0)[:,:,None]\n    sample = (background[y0,x0]*(1-fx)*(1-fy) + background[y0,x1]*fx*(1-fy) + background[y1,x0]*(1-fx)*fy + background[y1,x1]*fx*fy)\n    rgb = np.zeros((h,w,3), np.uint8)\n    rgb[keep] = np.rint(sample[keep]).clip(0,255).astype(np.uint8)\n    alpha = keep.astype(np.uint8)*255\n    return rgb, alpha, dict(method=\'bilinear normalized absolute correspondence, align_corners=True\',out_of_image_policy=out_of_image,requested_pixels=int(valid.sum()),out_of_image_requested_pixels=n,masked_out_of_image_pixels=n if out_of_image==\'mask\' else 0,border_clamped_pixels=n if out_of_image==\'border\' else 0,rendered_pixels=int(keep.sum()),is_complete_glass_render=False,attenuation_recovered=False,reflections_recovered=False,publication_accepted=False)\n'
# ---- embedded module: renderello/checked_research/correspondence_render.py ----
path = base / 'renderello/checked_research/correspondence_render.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Experimental aligned-frame luminance transfer, not physical relighting.\nInputs require an independently checked registration. Shape checks cannot prove it.\nNo source resampling or generated RGB replacement. This deliberately changes RGB.\n"""\nimport numpy as np\nfrom scipy.ndimage import gaussian_filter\n\ndef decode(x):\n x=x.astype(float)/255\n return np.where(x<=.04045,x/12.92,((x+.055)/1.055)**2.4)\n\ndef encode(x):\n return np.round(np.clip(np.where(x<=.0031308,12.92*x,1.055*np.maximum(x,0)**(1/2.4)-.055),0,1)*255).astype(\'uint8\')\n\ndef transfer(source,raw,alpha,*,registration_reviewed=False,sigma=16.,gain_bounds=(.65,1.45)):\n if registration_reviewed is not True:raise ValueError(\'registration review required; array shape is not alignment evidence\')\n if source.dtype!=np.uint8 or raw.dtype!=np.uint8 or source.shape!=raw.shape or source.ndim!=3 or source.shape[2]!=3:raise ValueError(\'matching uint8 RGB source/raw required\')\n h,w=source.shape[:2]\n if h*w>4_194_304:raise ValueError(\'research limit4MP\')\n if alpha.shape!=(h,w) or alpha.dtype!=np.uint8:raise ValueError(\'matching uint8 alpha required\')\n if not np.any(alpha):raise ValueError(\'empty alpha\')\n if not np.isfinite(sigma) or sigma<=0 or sigma>100:raise ValueError(\'sigma out of bounds\')\n lo,hi=gain_bounds\n if not np.isfinite([lo,hi]).all() or not (0<lo<=1<=hi<=2):raise ValueError(\'gain bounds must straddle1, max2\')\n a=alpha/255.;s=decode(source);r=decode(raw);y=np.array([.2126,.7152,.0722]);den=gaussian_filter(a,sigma);sl=gaussian_filter((s*y).sum(2)*a,sigma)/np.maximum(den,1e-6);dl=gaussian_filter((r*y).sum(2)*a,sigma)/np.maximum(den,1e-6)\n gain=np.clip((dl+1e-8)/(sl+1e-8),lo,hi);requested=gain.copy();cap=1/np.maximum(s.max(2),1e-12);gain=np.minimum(gain,cap);gain[a==0]=1;fg=encode(s*gain[:,:,None]);out=np.round(fg*a[:,:,None]+raw*(1-a[:,:,None])).astype(\'uint8\')\n return out,gain,dict(method=\'alpha-normalized linear-luminance Gaussian ratio; per-pixel no-new-clipping cap\',source_rgb_exact=False,source_resampled=False,physical_relighting=False,publication_accepted=False,highlight_capped_pixels=int(((requested>cap)&(a>0)).sum()),preencode_out_of_gamut_pixels=int(((s*gain[:,:,None]>1+1e-12).any(2)&(a>0)).sum()),opaque_rgb_changed=int((np.any(out!=source,2)&(alpha==255)).sum()))\n'
# ---- embedded module: renderello/checked_research/detail_transfer.py ----
path = base / 'renderello/checked_research/detail_transfer.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Experimental fractional-edge colour estimation, not exact product recovery.\n\nThe input RGB is retained at alpha 0 and 255. Fractional edges may change RGB.\nNative shape is preserved, but the estimator uses an internal image pyramid.\n"""\nimport numpy as np\n\n\ndef estimate_fractional_edge(rgb, alpha, *, source_alpha_pair_reviewed=False, estimator=None):\n    if source_alpha_pair_reviewed is not True:\n        raise ValueError(\'explicit source/alpha pairing review required\')\n    if not isinstance(rgb,np.ndarray) or rgb.dtype!=np.uint8 or rgb.ndim!=3 or rgb.shape[2]!=3:\n        raise ValueError(\'native uint8 RGB required\')\n    if not isinstance(alpha,np.ndarray) or alpha.dtype!=np.uint8 or alpha.shape!=rgb.shape[:2]:\n        raise ValueError(\'matching native uint8 alpha required\')\n    if min(alpha.shape)<2 or alpha.size>1_000_000:\n        raise ValueError(\'native dimensions2..1MP required\')\n    fractional=(alpha>0)&(alpha<255)\n    if not (alpha==0).any() or not (alpha==255).any() or not fractional.any():\n        raise ValueError(\'known foreground/background and fractional edge required\')\n    injected=estimator is not None\n    if estimator is None:\n        from pymatting import estimate_foreground_ml\n        estimator=estimate_foreground_ml\n    a=alpha.astype(np.float32)/255\n    foreground,background=estimator(rgb.astype(np.float32)/255,a,return_background=True)\n    for value in [foreground,background]:\n        if not isinstance(value,np.ndarray) or value.shape!=rgb.shape or value.dtype.kind!=\'f\' or not np.isfinite(value).all() or ((value<0)|(value>1)).any():\n            raise ValueError(\'finite native floating foreground/background in[0,1] required\')\n    corrected=rgb.copy()\n    corrected[fractional]=np.round(foreground[fractional]*255).astype(np.uint8)\n    metrics=dict(method=\'multilevel foreground/background estimate at fractional alpha only\',\n                 fractional_pixels=int(fractional.sum()),rgb_changed_pixels=int(np.any(corrected!=rgb,2).sum()),\n                 opaque_rgb_exact=True,zero_alpha_rgb_exact=True,alpha_changed=False,\n                 native_output_shape=True,estimator_uses_internal_pyramid=True,\n                 source_rgb_fully_exact=False,physical_material_recovery=False,\n                 pairing_review_is_caller_declaration=True,injected_estimator=injected,\n                 publication_accepted=False,boundary_accuracy=None)\n    return corrected,metrics\n'
# ---- embedded module: renderello/checked_research/edge_colour.py ----
path = base / 'renderello/checked_research/edge_colour.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Opt-in local fractional-edge unmixing. Not true foreground recovery.\nNearest known background/interior RGB are assumptions, not observations under edge.\nAll alpha, alpha0/255RGB and coordinates unchanged. Linear/sRGB mixing selectable\nonly by an explicit caller choice; source camera compositing space is unknown.\n"""\nimport numpy as np\nfrom scipy.ndimage import distance_transform_edt\n\ndef unmix_edge(rgb,alpha,*,pairing_reviewed=False,mixing_space=None,regularization=.05,max_distance=12.):\n if pairing_reviewed is not True:raise ValueError(\'source/alpha pairing review required\')\n if not isinstance(rgb,np.ndarray) or rgb.dtype!=np.uint8 or rgb.ndim!=3 or rgb.shape[2]!=3:raise ValueError(\'uint8 RGB required\')\n if not isinstance(alpha,np.ndarray) or alpha.dtype!=np.uint8 or alpha.shape!=rgb.shape[:2]:raise ValueError(\'matching uint8alpha required\')\n if min(alpha.shape)<2 or alpha.size>1_000_000:raise ValueError(\'2pixel axes/1MP limit\')\n if mixing_space not in [\'linear\',\'srgb\']:raise ValueError(\'explicit linear/srgb mixing space required\')\n for v in [regularization,max_distance]:\n  if isinstance(v,(bool,np.bool_)) or not isinstance(v,(int,float,np.number)) or not np.isfinite(v):raise ValueError(\'finite scalar controls required\')\n if not 0<regularization<=1 or not 0<max_distance<=100:raise ValueError(\'control bounds\')\n f=alpha==255;b=alpha==0;edge=(alpha>0)&(alpha<255)\n if not f.any() or not b.any() or not edge.any():raise ValueError(\'known foreground/background/fractionalpixels required\')\n db,ib=distance_transform_edt(~b,return_indices=True);df,iff=distance_transform_edt(~f,return_indices=True)\n x=rgb.astype(float)/255\n if mixing_space==\'linear\':x=np.where(x<=.04045,x/12.92,((x+.055)/1.055)**2.4)\n prior=x[iff[0],iff[1]];background=x[ib[0],ib[1]];a=alpha[:,:,None].astype(float)/255\n estimate=(a*(x-(1-a)*background)+regularization*prior)/(a*a+regularization)\n eligible=edge&(db<=max_distance)&(df<=max_distance);clipped=((estimate<0)|(estimate>1)).any(2)&eligible\n estimate=np.clip(estimate,0,1)\n if mixing_space==\'linear\':estimate=np.where(estimate<=.0031308,12.92*estimate,1.055*estimate**(1/2.4)-.055)\n out=rgb.copy();out[eligible]=np.round(estimate[eligible]*255).astype(\'uint8\')\n return out,dict(method=\'regularized nearest-known local edge unmixing\',mixing_space=mixing_space,regularization=float(regularization),max_distance=float(max_distance),fractional_pixels=int(edge.sum()),eligible_pixels=int(eligible.sum()),distance_rejected_pixels=int((edge&~eligible).sum()),changed_pixels=int(np.any(out!=rgb,2).sum()),preclip_estimate_out_of_range_pixels=int(clipped.sum()),opaque_rgb_exact=True,zero_alpha_rgb_exact=True,alpha_changed=False,source_resampled=False,physical_recovery=False,boundary_accuracy=None,publication_accepted=False)\n'
# ---- embedded module: renderello/checked_research/edge_unmix.py ----
path = base / 'renderello/checked_research/edge_unmix.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Fail-closed research accounting. File validity and error metrics are not photo quality."""\nimport hashlib\nfrom pathlib import Path\nimport numpy as np\nfrom PIL import Image\n\ndef image_record(path,expected_sha256=None):\n p=Path(path);digest=hashlib.sha256(p.read_bytes()).hexdigest()\n if expected_sha256 is not None and digest!=expected_sha256:raise ValueError(\'image hash mismatch\')\n with Image.open(p) as im:im.verify()\n with Image.open(p) as im:\n  rgb=np.asarray(im.convert(\'RGB\'));size=list(im.size)\n return dict(path=str(p),sha256=digest,rgb_sha256=hashlib.sha256(rgb.tobytes()).hexdigest(),size=size)\n\ndef source_bound_alpha(source,alpha,expected_source_sha256,*,expected_alpha_sha256=None):\n rec=image_record(source,expected_source_sha256);p=Path(alpha)\n if expected_alpha_sha256 is not None and hashlib.sha256(p.read_bytes()).hexdigest()!=expected_alpha_sha256:raise ValueError(\'alpha hash mismatch\')\n with Image.open(p) as a:\n  if a.mode!=\'L\' or list(a.size)!=rec[\'size\']:raise ValueError(\'source-native L alpha required; no automatic resize\')\n  arr=np.asarray(a)\n if not arr.any():raise ValueError(\'empty alpha\')\n return dict(source=rec,alpha_sha256=hashlib.sha256(p.read_bytes()).hexdigest(),support_pixels=int((arr>0).sum()),opaque_pixels=int((arr==255).sum()),components_reviewed=False,boundary_accuracy=None)\n\ndef correspondence_metrics(pred,truth,valid):\n if not isinstance(pred,np.ndarray) or pred.ndim!=3 or pred.shape[2]!=2 or truth.shape!=pred.shape:raise ValueError(\'matching HxWx2 arrays required\')\n if pred.dtype.kind!=\'f\' or truth.dtype.kind!=\'f\' or not np.isfinite(pred).all() or not np.isfinite(truth).all():raise ValueError(\'finite floating correspondence required\')\n h,w=pred.shape[:2]\n if min(h,w)<2 or valid.dtype!=np.bool_ or valid.shape!=(h,w):raise ValueError(\'matching boolean valid mask and dimensions>=2 required\')\n inbounds=((truth>=0)&(truth<=1)).all(2);v=valid&inbounds\n if not v.any():raise ValueError(\'no in-bounds valid pixels\')\n err=np.linalg.norm((pred-truth)*np.array([w-1,h-1]),axis=2)[v]\n return dict(valid_pixels=int(v.sum()),excluded_truth_out_of_bounds=int((valid&~inbounds).sum()),pred_out_of_bounds_valid=int((v&~((pred>=0)&(pred<=1)).all(2)).sum()),mean_epe_pixels=float(err.mean()),median_epe_pixels=float(np.median(err)),over3px_fraction=float((err>3).mean()),over5px_fraction=float((err>5).mean()),over10px_fraction=float((err>10).mean()),product_accuracy=None,publication_gate=None)\n\ndef acceptance_summary(rows):\n # Software cannot establish rater identity or blinding from imported text.\n if not isinstance(rows,list):raise ValueError(\'list of review rows required\')\n dimensions=[\'fidelity\',\'photorealism\',\'contact\',\'material_edges\',\'composition\',\'distinctness\',\'publish_usefulness\'];seen=set()\n for row in rows:\n  k=(row.get(\'source_sha256\'),row.get(\'output_sha256\'),row.get(\'rater_id\'))\n  if not all(isinstance(v,str) and v for v in k):raise ValueError(\'source/output hashes and rater identity declarations required\')\n  if k in seen:raise ValueError(\'duplicate review row\')\n  seen.add(k)\n  if any(isinstance(row.get(d),bool) or not isinstance(row.get(d),int) or not 1<=row[d]<=5 for d in dimensions):raise ValueError(\'all seven integer1..5scores required\')\n  if type(row.get(\'critical_error\')) is not bool:raise ValueError(\'explicit critical-error decision required\')\n return dict(imported_rows=len(rows),no_critical_alteration_fraction=None,publish_ready_fraction=None,acceptance_status=\'NOT_ESTABLISHED\',reason=\'Imported declarations do not authenticate rater independence/blinding or matched template and source coverage.\')\n'
# ---- embedded module: renderello/checked_research/evidence_checks.py ----
path = base / 'renderello/checked_research/evidence_checks.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Opt-in text-grounded native-coordinate box proposals, not product masks.\n\nThe detector resizes its internal input explicitly. Saved source pixels and output\ncoordinates remain source-native. Scores are not accuracy; reflected objects and\nwrong categories can remain. This route never selects the largest object.\n"""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport time\nimport numpy as np\nfrom PIL import Image\n\nMODEL = \'IDEA-Research/grounding-dino-tiny\'\nREVISION = \'a2bb814dd30d776dcf7e30523b00659f4f141c71\'\nWEIGHTS_SHA256 = \'1a2412ef99bd74bcd3c2a246fa1e48581f8889a1300c9051974741314fc042f3\'\nMAX_PIXELS = 2_000_000\n\n\ndef sha256(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef validate_queries(queries):\n    if not isinstance(queries, list) or not 1 <= len(queries) <= 12:\n        raise ValueError(\'explicit list of 1..12 category queries required\')\n    for q in queries:\n        if not isinstance(q, str) or not q.strip() or q != q.strip() or len(q) > 256 or not q.endswith(\'.\') or q != q.lower():\n            raise ValueError(\'lowercase query ending in period, at most 256 characters required\')\n    if len(set(queries)) != len(queries):\n        raise ValueError(\'duplicate query\')\n    return queries\n\n\ndef checked_proposals(rows, width, height):\n    """Retain all predictions and their query provenance; never infer identity."""\n    if type(width) is not int or type(height) is not int or min(width, height) < 2:\n        raise ValueError(\'native integer dimensions required\')\n    result = []\n    for row in rows:\n        if not isinstance(row.get(\'query\'), str) or not row[\'query\']:\n            raise ValueError(\'query provenance required\')\n        boxes, scores, labels = row[\'boxes\'], row[\'scores\'], row[\'labels\']\n        if not (len(boxes) == len(scores) == len(labels)):\n            raise ValueError(\'box/score/label count differs\')\n        for box, score, label in zip(boxes, scores, labels):\n            b = np.asarray(box, dtype=np.float64)\n            if b.shape != (4,) or not np.isfinite(b).all() or not np.isfinite(score) or not 0 <= score <= 1:\n                raise ValueError(\'finite box and score in [0,1] required\')\n            if not (0 <= b[0] < b[2] <= width and 0 <= b[1] < b[3] <= height):\n                raise ValueError(\'source-native in-bounds nondegenerate box required; no clipping\')\n            if not isinstance(label, str) or not label:\n                raise ValueError(\'nonempty predicted text required\')\n            result.append(dict(proposal_id=len(result), query=row[\'query\'], box=b.tolist(),\n                               model_confidence=float(score), predicted_text=label))\n    return result\n\n\ndef group_overlaps(proposals, threshold=0.75):\n    """Greedy box overlap groups, NOT deduplicated product identities.\n\n    Every member survives, including incompatible category predictions. Grouping\n    order is stable confidence-descending, then input order. A group is matched\n    to its representative, not transitive-connected to all its other members.\n    """\n    if type(threshold) not in (int, float) or not np.isfinite(threshold) or not 0 < threshold <= 1:\n        raise ValueError(\'overlap IoU threshold in (0,1] required\')\n    groups = []\n    for p in sorted(proposals, key=lambda x: (-x[\'model_confidence\'], x[\'proposal_id\'])):\n        a = p[\'box\']\n        for g in groups:\n            b = g[\'representative\'][\'box\']\n            inter = max(0, min(a[2],b[2])-max(a[0],b[0])) * max(0,min(a[3],b[3])-max(a[1],b[1]))\n            union = (a[2]-a[0])*(a[3]-a[1])+(b[2]-b[0])*(b[3]-b[1])-inter\n            if inter / union >= threshold:\n                g[\'members\'].append(p)\n                break\n        else:\n            groups.append(dict(group_id=len(groups), representative=p, members=[p]))\n    return groups\n\n\nclass CpuPredictor:\n    def __init__(self, threads=2):\n        if type(threads) is not int or not 1 <= threads <= 4:\n            raise ValueError(\'CPU threads 1..4 required\')\n        import torch\n        import transformers\n        from transformers import GroundingDinoForObjectDetection, GroundingDinoProcessor\n        from huggingface_hub import hf_hub_download\n        torch.set_num_threads(threads)\n        self.torch = torch\n        weights = hf_hub_download(MODEL, \'model.safetensors\', revision=REVISION)\n        if sha256(weights) != WEIGHTS_SHA256:\n            raise ValueError(\'pinned checkpoint hash differs\')\n        self.processor = GroundingDinoProcessor.from_pretrained(MODEL, revision=REVISION)\n        self.processor.image_processor.size = {\'shortest_edge\':480, \'longest_edge\':800}\n        self.model, loading = GroundingDinoForObjectDetection.from_pretrained(\n            MODEL, revision=REVISION, use_safetensors=True, output_loading_info=True)\n        if any(loading.get(k) for k in (\'missing_keys\',\'unexpected_keys\',\'mismatched_keys\',\'error_msgs\')):\n            raise ValueError(\'strict checkpoint load failed\')\n        self.model.eval()\n        self.info = dict(model=MODEL, revision=REVISION, weights_sha256=WEIGHTS_SHA256,\n                         loading_info=loading, device=\'cpu\', threads=threads,\n                         torch_version=torch.__version__, transformers_version=transformers.__version__,\n                         detector_internal_resize={\'shortest_edge\':480,\'longest_edge\':800})\n\n    def __call__(self, image, query):\n        tokens = self.processor.tokenizer(query, truncation=False)[\'input_ids\']\n        if len(tokens) > self.model.config.max_text_len:\n            raise ValueError(\'query exceeds checkpoint token limit; no silent truncation\')\n        inputs = self.processor(images=image, text=query, return_tensors=\'pt\')\n        if inputs.input_ids.shape[-1] != len(tokens):\n            raise ValueError(\'processor changed query token length\')\n        t = time.monotonic()\n        with self.torch.inference_mode():\n            out = self.model(**inputs)\n        result = self.processor.post_process_grounded_object_detection(\n            out, inputs.input_ids, threshold=.25, text_threshold=.25,\n            target_sizes=[image.size[::-1]])[0]\n        return dict(query=query, boxes=result[\'boxes\'].tolist(), scores=result[\'scores\'].tolist(),\n                    labels=result[\'text_labels\'], input_shape=list(inputs.pixel_values.shape),\n                    seconds=time.monotonic()-t)\n\n\ndef run(source, output_dir, *, source_sha256, queries, threads=2, predictor=None):\n    queries = validate_queries(queries)\n    if not isinstance(source_sha256, str) or len(source_sha256) != 64 or any(x not in \'0123456789abcdef\' for x in source_sha256):\n        raise ValueError(\'explicit lowercase SHA-256 required\')\n    if sha256(source) != source_sha256:\n        raise ValueError(\'source hash differs\')\n    dest = Path(output_dir)\n    if dest.exists():\n        raise ValueError(\'output directory exists; no replacement\')\n    with Image.open(source) as im:\n        if im.mode != \'RGB\' or min(im.size) < 2 or im.width*im.height > MAX_PIXELS:\n            raise ValueError(\'native RGB source within 2M pixel CPU bound required\')\n        image = im.copy()\n    worker = predictor if predictor is not None else CpuPredictor(threads)\n    rows = [worker(image, q) for q in queries]\n    for expected, row in zip(queries, rows):\n        if row.get(\'query\') != expected:\n            raise ValueError(\'predictor changed query provenance\')\n    props = checked_proposals(rows, *image.size)\n    manifest = dict(source_sha256=source_sha256, native_size=list(image.size), source_resampled=False,\n                    predictor=getattr(worker,\'info\',{\'unverified_injected_predictor\':True}),\n                    box_threshold=.25,text_threshold=.25,queries=rows,proposals=props,\n                    overlap_groups=group_overlaps(props),grouping_iou=.75,\n                    status=\'PROPOSALS_NOT_SEGMENTATIONS\',accuracy=None,completeness=None,\n                    publication_accepted=False,production_default_changed=False)\n    dest.mkdir(parents=True)  # exclusive, only after complete validated inference\n    image.save(dest/\'source-native.png\')\n    (dest/\'manifest.json\').write_text(json.dumps(manifest,indent=2)+\'\\n\')\n    return manifest\n\n\ndef main():\n    p=argparse.ArgumentParser(description=__doc__)\n    for arg in (\'source\',\'source-sha256\',\'queries-json\',\'output-dir\'):\n        p.add_argument(\'--\'+arg,required=True)\n    p.add_argument(\'--threads\',type=int,default=2)\n    a=p.parse_args()\n    try:\n        value=run(a.source,a.output_dir,source_sha256=a.source_sha256,\n                  queries=json.loads(a.queries_json),threads=a.threads)\n    except (ValueError,OSError) as e:\n        p.exit(2,\'REFUSED: \'+str(e)+\'\\n\')\n    print(json.dumps(value,indent=2))\n\nif __name__==\'__main__\':\n    main()\n'
# ---- embedded module: renderello/checked_research/grounded_proposals.py ----
path = base / 'renderello/checked_research/grounded_proposals.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Opt-in CPU box-to-mask research. Retains all three masks, never claims identities.\n\nNative RGB is preserved in every RGBA export. Binary masks are not alpha matting.\nReflections, merged products, occlusion and missing bristles can survive this route.\n"""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport time\nimport numpy as np\nfrom PIL import Image\nfrom .grounded_proposals import sha256, checked_proposals, MAX_PIXELS\n\nMODEL = \'facebook/sam2.1-hiera-tiny\'\nREVISION = \'de431c4043854a71d8101e17995dfe596bf101a5\'\nWEIGHTS_SHA256 = \'48c14467e5cf9e51870511feb72c89688e82dd74523142c0538b663e193ac2a7\'\n\n\ndef checked_manifest(value, source_hash, native_size):\n    if not isinstance(value, dict) or value.get(\'source_sha256\') != source_hash or value.get(\'native_size\') != list(native_size):\n        raise ValueError(\'proposal source hash or native dimensions differ\')\n    if value.get(\'status\') != \'PROPOSALS_NOT_SEGMENTATIONS\' or value.get(\'source_resampled\') is not False:\n        raise ValueError(\'native checked proposal manifest required\')\n    props = value.get(\'proposals\')\n    if not isinstance(props, list) or not 1 <= len(props) <= 64:\n        raise ValueError(\'1..64 explicit proposals required; no silent limiting\')\n    result = []\n    for i, p in enumerate(props):\n        if not isinstance(p, dict) or type(p.get(\'proposal_id\')) is not int or p[\'proposal_id\'] != i:\n            raise ValueError(\'stable consecutive integer proposal IDs required\')\n        try:\n            one = checked_proposals([dict(query=p[\'query\'], boxes=[p[\'box\']], scores=[p[\'model_confidence\']], labels=[p[\'predicted_text\']])], *native_size)[0]\n        except (KeyError, TypeError) as e:\n            raise ValueError(\'proposal fields missing or malformed\') from e\n        one[\'proposal_id\'] = i\n        result.append(one)\n    return result\n\n\ndef checked_masks(masks, scores, native_size):\n    a, s = np.asarray(masks), np.asarray(scores)\n    w, h = native_size\n    if a.shape != (3, h, w) or a.dtype != np.bool_:\n        raise ValueError(\'exactly three native-size boolean masks required; no resizing or threshold guessing\')\n    if s.shape != (3,) or (not np.issubdtype(s.dtype, np.floating) and not np.issubdtype(s.dtype, np.integer)) or not np.isfinite(s).all():\n        raise ValueError(\'three finite model mask-quality scores required; not accuracy\')\n    area = a.sum(axis=(1,2))\n    if np.any(area == 0) or np.any(area == w*h):\n        raise ValueError(\'empty or all-image mask refused\')\n    return a, s.astype(np.float64)\n\n\nclass CpuSegmenter:\n    def __init__(self, threads=2, cache_embeddings=True):\n        if type(threads) is not int or not 1 <= threads <= 4:\n            raise ValueError(\'CPU threads 1..4 required\')\n        import torch\n        import transformers\n        import torchvision\n        from transformers import Sam2Model, Sam2Processor\n        from huggingface_hub import hf_hub_download\n        torch.set_num_threads(threads)\n        self.cache_embeddings = cache_embeddings\n        self.torch = torch\n        weights = hf_hub_download(MODEL, \'model.safetensors\', revision=REVISION)\n        if sha256(weights) != WEIGHTS_SHA256:\n            raise ValueError(\'pinned segmentation checkpoint hash differs\')\n        self.model, loading = Sam2Model.from_pretrained(MODEL, revision=REVISION, output_loading_info=True, use_safetensors=True)\n        if any(loading.get(k) for k in (\'missing_keys\',\'unexpected_keys\',\'mismatched_keys\',\'error_msgs\')):\n            raise ValueError(\'strict segmentation checkpoint load failed\')\n        self.model.eval()\n        self.processor = Sam2Processor.from_pretrained(MODEL, revision=REVISION)\n        self.embeddings = None\n        self.source_rgb_hash = None\n        self.info = dict(model=MODEL, revision=REVISION, weights_sha256=WEIGHTS_SHA256, loading_info=loading,\n                         device=\'cpu\', threads=threads, torch_version=torch.__version__,\n                         transformers_version=transformers.__version__, torchvision_version=torchvision.__version__,\n                         internal_resize=list(self.processor.image_processor.size.values()))\n\n    def __call__(self, image, proposal):\n        rgb_hash = hashlib.sha256(np.asarray(image).tobytes()).hexdigest()\n        if self.cache_embeddings and self.source_rgb_hash is not None and rgb_hash != self.source_rgb_hash:\n            raise ValueError(\'cannot reuse image embeddings for a different image\')\n        self.source_rgb_hash = rgb_hash\n        inputs = self.processor(images=image, input_boxes=[[proposal[\'box\']]], return_tensors=\'pt\')\n        cached = self.embeddings is not None\n        if cached:\n            del inputs[\'pixel_values\']\n            inputs[\'image_embeddings\'] = self.embeddings\n        start = time.monotonic()\n        with self.torch.inference_mode():\n            out = self.model(**inputs, multimask_output=True)\n        seconds = time.monotonic() - start\n        self.embeddings = out.image_embeddings if self.cache_embeddings else None\n        masks = self.processor.post_process_masks(out.pred_masks, inputs[\'original_sizes\'])[0][0].numpy()\n        return dict(proposal_id=proposal[\'proposal_id\'], masks=masks, scores=out.iou_scores[0,0].numpy(),\n                    seconds=seconds, includes_image_embedding=not cached)\n\n\ndef native_crop(image, proposal, margin):\n    """Explicit proposal-centred native crop; cannot recover missing proposal objects."""\n    if type(margin) is not int or not 1 <= margin <= 256:\n        raise ValueError(\'explicit crop margin must be integer 1..256\')\n    x0,y0,x1,y1 = proposal[\'box\']\n    if not np.isfinite([x0,y0,x1,y1]).all() or not (0 <= x0 < x1 <= image.width and 0 <= y0 < y1 <= image.height):\n        raise ValueError(\'native proposal bounds required\')\n    bounds = (max(0,int(np.floor(x0))-margin), max(0,int(np.floor(y0))-margin),\n              min(image.width,int(np.ceil(x1))+margin), min(image.height,int(np.ceil(y1))+margin))\n    local = dict(proposal, box=[x0-bounds[0],y0-bounds[1],x1-bounds[0],y1-bounds[1]])\n    return image.crop(bounds), local, bounds\n\n\ndef restore_native_masks(masks, scores, crop, native_size):\n    left,top,right,bottom = crop\n    w,h = native_size\n    if not (0 <= left < right <= w and 0 <= top < bottom <= h):\n        raise ValueError(\'crop must lie inside native source\')\n    small, scores = checked_masks(masks, scores, (right-left,bottom-top))\n    native = np.zeros((3,h,w),dtype=bool)\n    native[:,top:bottom,left:right] = small\n    return native, scores\n\n\ndef run(source, proposals_path, output_dir, *, source_sha256, proposals_sha256, threads=2, segmenter=None, crop_margin=None):\n    if crop_margin is not None and (type(crop_margin) is not int or not 1 <= crop_margin <= 256):\n        raise ValueError(\'explicit crop margin must be integer 1..256\')\n    if sha256(source) != source_sha256 or sha256(proposals_path) != proposals_sha256:\n        raise ValueError(\'source or proposal-file hash differs\')\n    dest = Path(output_dir)\n    if dest.exists():\n        raise ValueError(\'output directory exists; no replacement\')\n    with Image.open(source) as im:\n        if im.mode != \'RGB\' or min(im.size) < 2 or im.width*im.height > MAX_PIXELS:\n            raise ValueError(\'native RGB image within 2M-pixel CPU bound required\')\n        image = im.copy()\n    props = checked_manifest(json.loads(Path(proposals_path).read_text()), source_sha256, image.size)\n    worker = segmenter if segmenter is not None else CpuSegmenter(threads,cache_embeddings=crop_margin is None)\n    results = []\n    for p in props:\n        crop = None\n        if crop_margin is None:\n            row = worker(image, p)\n        else:\n            local_image, local_proposal, crop = native_crop(image,p,crop_margin)\n            row = worker(local_image,local_proposal)\n        if type(row.get(\'proposal_id\')) is not int or row[\'proposal_id\'] != p[\'proposal_id\']:\n            raise ValueError(\'segmenter changed proposal provenance\')\n        masks, scores = checked_masks(row[\'masks\'], row[\'scores\'], image.size) if crop is None else restore_native_masks(row[\'masks\'],row[\'scores\'],crop,image.size)\n        row = dict(row, native_crop_bounds=crop)\n        seconds = row.get(\'seconds\')\n        if type(seconds) not in (int, float) or not np.isfinite(seconds) or seconds < 0:\n            raise ValueError(\'finite nonnegative inference timing required\')\n        if type(row.get(\'includes_image_embedding\')) is not bool:\n            raise ValueError(\'explicit image-embedding timing provenance required\')\n        results.append((p, row, masks, scores))\n    dest.mkdir(parents=True)\n    image.save(dest/\'source-native.png\')\n    rgb = np.asarray(image)\n    records = []\n    for p, row, masks, scores in results:\n        i = p[\'proposal_id\']\n        np.savez_compressed(dest/f\'proposal-{i:02d}-raw.npz\', masks=masks, model_mask_quality=scores)\n        names = []\n        for j, mask in enumerate(masks):\n            name = f\'proposal-{i:02d}-candidate-{j}.png\'\n            Image.fromarray(np.dstack((rgb, mask.astype(np.uint8)*255))).save(dest/name)\n            names.append(name)\n        records.append(dict(proposal=p, candidates=names, model_mask_quality=scores.tolist(),\n                            areas=masks.sum(axis=(1,2)).tolist(), seconds=row[\'seconds\'],\n                            includes_image_embedding=row[\'includes_image_embedding\'], native_crop_bounds=row[\'native_crop_bounds\'], selected_candidate=None,\n                            status=\'MASK_CANDIDATES_NOT_VERIFIED_PRODUCTS\'))\n    info = dict(source_sha256=source_sha256, proposals_sha256=proposals_sha256, native_size=list(image.size),\n                crop_margin=crop_margin, crop_limits_context=crop_margin is not None, source_resampled=False, source_rgb_preserved=True, binary_masks_not_matting=True,\n                segmenter=getattr(worker,\'info\',{\'unverified_injected_segmenter\':True}), records=records,\n                production_default_changed=False, publication_accepted=False, accuracy=None, completeness=None)\n    (dest/\'manifest.json\').write_text(json.dumps(info,indent=2)+\'\\n\')\n    return info\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    for name in (\'source\',\'source-sha256\',\'proposals-path\',\'proposals-sha256\',\'output-dir\'):\n        parser.add_argument(\'--\'+name, required=True)\n    parser.add_argument(\'--threads\', type=int, default=2)\n    parser.add_argument(\'--crop-margin\',type=int,default=None,help=\'Opt-in native crop padding 1..256; can lose image context\')\n    args = parser.parse_args()\n    try:\n        result = run(args.source,args.proposals_path,args.output_dir,source_sha256=args.source_sha256,\n                     proposals_sha256=args.proposals_sha256,threads=args.threads,crop_margin=args.crop_margin)\n    except (ValueError,OSError) as e:\n        parser.exit(2,\'REFUSED: \'+str(e)+\'\\n\')\n    print(json.dumps(result,indent=2))\n\nif __name__ == \'__main__\':\n    main()\n'
# ---- embedded module: renderello/checked_research/grounded_segmentation.py ----
path = base / 'renderello/checked_research/grounded_segmentation.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Opt-in, source-detail-preserving intrinsic gain experiment, not physical relighting.\nOnly a bounded achromatic gain is applied to the original linear RGB. Estimated\nmodel RGB never replaces labels. This can still damage brightness/materials.\nCaller must review source/map alignment. No registration inference or source resize.\n"""\nimport numbers\nimport numpy as np\nfrom scipy.ndimage import gaussian_filter\nfrom scipy.optimize import nnls\nfrom .detail_transfer import decode, encode\n\ndef intrinsic_gain(source, alpha, prediction, *, alignment_reviewed=False, sigma=16., shading_strength=.5, residual_suppression=.25, gain_bounds=(.65,1.45)):\n if alignment_reviewed is not True:raise ValueError(\'reviewed source/map alignment required\')\n if not isinstance(source,np.ndarray) or source.dtype!=np.uint8 or source.ndim!=3 or source.shape[2]!=3:raise ValueError(\'uint8 RGB required\')\n h,w=source.shape[:2]\n if min(h,w)<2 or h*w>4_194_304:raise ValueError(\'source must be2pixels minimum per axis and4MP maximum\')\n if not isinstance(alpha,np.ndarray) or alpha.dtype!=np.uint8 or alpha.shape!=(h,w) or not (alpha==255).any():raise ValueError(\'matching uint8 alpha with opaque foreground required\')\n if not isinstance(prediction,np.ndarray) or prediction.shape!=(3,h,w,3) or prediction.dtype.kind!=\'f\' or not np.isfinite(prediction).all() or prediction.min()<0 or prediction.max()>1:raise ValueError(\'finite linear3target RGB maps in0to1required\')\n for v,low,high in [(sigma,0,100),(shading_strength,0,1),(residual_suppression,0,1)]:\n  if isinstance(v,(bool,np.bool_)) or not isinstance(v,numbers.Real) or not np.isfinite(v) or not low<=v<=high:raise ValueError(\'invalid scalar control\')\n if sigma==0:raise ValueError(\'positive sigma required\')\n try:lo,hi=gain_bounds\n except (TypeError,ValueError):raise ValueError(\'gain bounds pair required\') from None\n if any(isinstance(v,(bool,np.bool_)) or not isinstance(v,numbers.Real) for v in [lo,hi]) or not np.isfinite([lo,hi]).all() or not 0<lo<=1<=hi<=2:raise ValueError(\'gain bounds must straddle1,max2\')\n s=decode(source);weights=np.array([.2126,.7152,.0722]);y=(s*weights).sum(2);a,sh,r=prediction.astype(np.float64);opaque=alpha==255;support=alpha>0;mask=alpha/255.;den=gaussian_filter(mask,sigma)\n def blur(v):return gaussian_filter(v*mask,sigma)/np.maximum(den,1e-8)\n shade=blur((sh*weights).sum(2));res=blur((r*weights).sum(2));base=(a*sh*weights).sum(2)\n scales,_=nnls(np.stack([base[opaque],(r*weights).sum(2)[opaque]],axis=1),y[opaque]);median=float(np.median(shade[opaque]));identity=shading_strength==0 and residual_suppression==0\n if not identity and median<=1e-8:raise ValueError(\'unusable near-zero model shading\')\n gain=(np.maximum(median,1e-8)/np.maximum(shade,1e-8))**shading_strength\n ratio=np.minimum(scales[1]*res/np.maximum(blur(y),1e-8),1.)\n gain=np.clip(gain*(1-residual_suppression*ratio),lo,hi);requested=gain.copy();cap=1/np.maximum(s.max(2),1e-12);gain=np.minimum(gain,cap);gain[~support]=1\n rgb=encode(s*gain[:,:,None]);rgb[~support]=source[~support]\n if identity:rgb=source.copy();gain=np.ones((h,w),float)\n rgba=np.dstack([rgb,alpha]);return rgba,gain,dict(method=\'bounded original linear-RGB gain from blurred intrinsic shading/residual; in-sample scale fit\',source_resampled=False,source_rgb_exact=identity,estimated_rgb_replaces_source=False,physical_relighting=False,publication_accepted=False,independent_accuracy=None,gain_bounds=[lo,hi],sigma=float(sigma),shading_strength=float(shading_strength),residual_suppression=float(residual_suppression),fitted_luminance_scales=scales.tolist(),highlight_capped_pixels=int(((requested>cap)&support).sum()),preencode_out_of_gamut_pixels=int(((s*gain[:,:,None]>1+1e-12).any(2)&support).sum()),opaque_rgb_changed=int((np.any(rgb!=source,2)&opaque).sum()))\n'
# ---- embedded module: renderello/checked_research/intrinsic_gain.py ----
path = base / 'renderello/checked_research/intrinsic_gain.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Screen-space left/right gain diagnostic, not physical light estimation.\n\nRequires declared paired prompts and aligned gain maps. It does not infer surface\nnormals, light position, scene geometry, source identity or publication readiness.\n"""\nimport numpy as np\n\n\ndef paired_horizontal_gain(left_prompt_gain, right_prompt_gain, opaque, *, aligned_pair_reviewed=False):\n    if aligned_pair_reviewed is not True:\n        raise ValueError(\'explicit aligned pair review required; shapes cannot prove alignment\')\n    gains = [np.asarray(left_prompt_gain), np.asarray(right_prompt_gain)]\n    if not isinstance(opaque,np.ndarray) or opaque.dtype!=np.bool_ or opaque.ndim!=2:\n        raise ValueError(\'native boolean opaque mask required\')\n    if opaque.size>4_194_304 or min(opaque.shape)<4 or opaque.sum()<16:\n        raise ValueError(\'nontrivial native support within4MP required\')\n    if any(g.shape!=opaque.shape or g.dtype.kind!=\'f\' or not np.isfinite(g).all() or np.any(g<=0) for g in gains):\n        raise ValueError(\'finite positive native floating gain maps required\')\n    yy,xx=np.indices(opaque.shape)\n    centre=float(xx[opaque].mean());lo,hi=np.quantile(yy[opaque],[.15,.85])\n    left=opaque&(xx<centre)&(yy>=lo)&(yy<=hi)\n    right=opaque&(xx>=centre)&(yy>=lo)&(yy<=hi)\n    if min(left.sum(),right.sum())<4:\n        raise ValueError(\'both screen-space half regions need support\')\n    rows=[]\n    for name,g in zip([\'left_prompt\',\'right_prompt\'],gains):\n        l,r=float(g[left].mean()),float(g[right].mean())\n        rows.append(dict(prompt=name,left_mean_gain=l,right_mean_gain=r,left_minus_right=l-r))\n    return dict(method=\'mean gain on opaque horizontal halves, central15..85percent vertical support\',\n                left_pixels=int(left.sum()),right_pixels=int(right.sum()),centre_x=centre,\n                vertical_limits=[float(lo),float(hi)],rows=rows,\n                contrast_change_right_minus_left=rows[1][\'left_minus_right\']-rows[0][\'left_minus_right\'],\n                gain_mean_abs_difference=float(np.abs(gains[0]-gains[1])[opaque].mean()),\n                physical_lighting_established=False,independent_direction_accuracy=None,\n                publication_accepted=False,alignment_review_is_caller_declaration=True)\n'
# ---- embedded module: renderello/checked_research/lighting_diagnostics.py ----
path = base / 'renderello/checked_research/lighting_diagnostics.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Source-bound paired lighting diagnostic. Does not generate or accept photographs.\n\nHashes bind file bytes; alignment remains an explicit caller declaration, not\nindependent proof. Native-sized gain arrays must originate from the same source.\n"""\nimport argparse\nimport hashlib\nimport json\nimport shutil\nimport tempfile\nfrom pathlib import Path\nimport numpy as np\nfrom PIL import Image\nfrom .evidence_checks import source_bound_alpha\nfrom .lighting_diagnostics import paired_horizontal_gain\nfrom .research_cli import file_record\n\n\ndef execute(args):\n    dest = Path(args.output_dir)\n    if dest.exists():\n        raise ValueError(\'output directory already exists; preserve prior evidence\')\n    binding = source_bound_alpha(args.source, args.alpha, args.source_sha256,\n                                 expected_alpha_sha256=args.alpha_sha256)\n    records = {}\n    gains = []\n    for name in (\'left_gain\', \'right_gain\'):\n        path = getattr(args, name)\n        records[name] = file_record(path, getattr(args, name + \'_sha256\'))\n        if Path(path).stat().st_size > 64 * 1024 * 1024:\n            raise ValueError(\'gain file exceeds 64 MiB research limit\')\n        gain = np.load(path, allow_pickle=False)\n        if not isinstance(gain, np.ndarray):\n            gain.close()\n            raise ValueError(\'single native NumPy gain array required, not an archive\')\n        gains.append(gain)\n    with Image.open(args.alpha) as im:\n        opaque = np.array(im) == 255\n    metrics = paired_horizontal_gain(*gains, opaque,\n                                     aligned_pair_reviewed=args.aligned_pair_reviewed)\n    manifest = dict(command=\'lighting-pair\', inputs=dict(binding=binding, **records),\n                    metrics=metrics, publication_accepted=False, quality_accuracy=None,\n                    alignment_review_is_caller_declaration=True,\n                    source_identity_independently_verified=False)\n    # Validate before creating output; reserve exclusively and never overwrite evidence.\n    dest.parent.mkdir(parents=True, exist_ok=True)\n    temp = Path(tempfile.mkdtemp(prefix=\'renderello-lighting-pair-\', dir=dest.parent))\n    try:\n        (temp / \'manifest.json\').write_text(json.dumps(manifest, indent=2) + \'\\n\')\n        dest.mkdir()\n        try:\n            (temp / \'manifest.json\').rename(dest / \'manifest.json\')\n        except BaseException:\n            shutil.rmtree(dest)\n            raise\n    finally:\n        shutil.rmtree(temp)\n    return manifest\n\n\ndef parser():\n    p = argparse.ArgumentParser(description=__doc__)\n    for name in (\'source\', \'alpha\', \'left-gain\', \'right-gain\'):\n        p.add_argument(\'--\' + name, required=True)\n        p.add_argument(\'--\' + name + \'-sha256\', required=True)\n    p.add_argument(\'--aligned-pair-reviewed\', action=\'store_true\')\n    p.add_argument(\'--output-dir\', required=True)\n    return p\n\n\nif __name__ == \'__main__\':\n    p = parser()\n    args = p.parse_args()\n    try:\n        result = execute(args)\n    except (ValueError, OSError) as error:\n        p.exit(2, \'REFUSED: \' + str(error) + \'\\n\')\n    print(json.dumps(result, indent=2))\n'
# ---- embedded module: renderello/checked_research/lighting_pair_cli.py ----
path = base / 'renderello/checked_research/lighting_pair_cli.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Opt-in native ViTMatte research, never a production quality acceptance.\n\nNo source resizing, repainting or silent mode conversion. Hashes bind supplied\nbytes, not semantic product identity. A binary support mask and derived trimap\nare caller evidence, not independently annotated boundary truth.\n"""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport shutil\nimport tempfile\nimport time\n\nimport numpy as np\nfrom PIL import Image\n\nMODEL = \'hustvl/vitmatte-small-composition-1k\'\nREVISION = \'6a58ad7646403c1df626fbd746900aec7361ea1d\'\nWEIGHTS_SHA256 = \'bda9289db1bb6762d978b42d1c62ae3f34daf7497171a347a1d09657efd788cb\'\nMAX_PIXELS = 1_000_000\n\n\ndef sha256(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef read_bound(source, mask, source_sha256, mask_sha256):\n    for path, expected in ((source, source_sha256), (mask, mask_sha256)):\n        if not isinstance(expected, str) or len(expected) != 64 or any(c not in \'0123456789abcdef\' for c in expected):\n            raise ValueError(\'explicit lowercase SHA-256 required\')\n        if sha256(path) != expected:\n            raise ValueError(\'input file hash mismatch: \' + str(path))\n    with Image.open(source) as s, Image.open(mask) as m:\n        if s.mode != \'RGB\' or m.mode != \'L\':\n            raise ValueError(\'native RGB source and L mask required; no mode conversion\')\n        if s.size != m.size:\n            raise ValueError(\'native source/mask dimensions differ; no resizing\')\n        if min(s.size) < 2 or s.width * s.height > MAX_PIXELS:\n            raise ValueError(\'source dimensions outside bounded native research limit (2px minimum, 1000000 pixels maximum)\')\n        rgb, alpha = np.array(s), np.array(m)\n    if not np.isin(alpha, [0, 255]).all():\n        raise ValueError(\'explicit binary 0/255 support mask required; no hidden threshold\')\n    if not (alpha == 0).any() or not (alpha == 255).any():\n        raise ValueError(\'both source foreground and background support required\')\n    return rgb, alpha\n\n\ndef derive_trimap(mask, radius):\n    from scipy.ndimage import binary_erosion, binary_dilation\n    if type(radius) is not int or not 1 <= radius <= 16:\n        raise ValueError(\'integer trimap radius 1..16 required\')\n    if not isinstance(mask, np.ndarray) or mask.ndim != 2 or mask.dtype != np.uint8 or not np.isin(mask, [0, 255]).all():\n        raise ValueError(\'two-dimensional binary uint8 mask required\')\n    support = mask == 255\n    fg = binary_erosion(support, iterations=radius)\n    bg = ~binary_dilation(support, iterations=radius)\n    if not fg.any() or not bg.any():\n        raise ValueError(\'trimap has no known foreground or background\')\n    tri = np.full(mask.shape, 128, np.uint8)\n    tri[fg], tri[bg] = 255, 0\n    return tri\n\n\ndef checked_alpha(raw, trimap):\n    if not isinstance(trimap, np.ndarray) or trimap.ndim != 2 or trimap.dtype != np.uint8 or not np.isin(trimap, [0, 128, 255]).all():\n        raise ValueError(\'explicit 0/128/255 uint8 trimap required\')\n    if not isinstance(raw, np.ndarray) or raw.dtype.kind != \'f\' or raw.shape != trimap.shape:\n        raise ValueError(\'source-native floating alpha required\')\n    if not np.isfinite(raw).all() or (raw < 0).any() or (raw > 1).any():\n        raise ValueError(\'finite alpha in [0,1] required; no clipping invalid predictions\')\n    alpha = raw.copy()\n    alpha[trimap == 255], alpha[trimap == 0] = 1, 0\n    return alpha, np.round(alpha * 255).astype(np.uint8)\n\n\ndef composite(rgb, alpha8, background):\n    if not isinstance(rgb, np.ndarray) or rgb.dtype != np.uint8 or rgb.ndim != 3 or rgb.shape[2] != 3:\n        raise ValueError(\'native uint8 RGB required\')\n    if not isinstance(alpha8, np.ndarray) or alpha8.dtype != np.uint8 or alpha8.shape != rgb.shape[:2]:\n        raise ValueError(\'matching uint8 alpha required\')\n    if type(background) is not int or not 0 <= background <= 255:\n        raise ValueError(\'integer diagnostic background 0..255 required\')\n    a = alpha8.astype(np.float64)[:, :, None] / 255\n    return np.round(rgb * a + background * (1 - a)).astype(np.uint8)\n\n\nclass CpuPredictor:\n    """Pinned weights, strict load; dependencies are optional outside this route."""\n    def __init__(self, threads=2, *, device=\'cpu\'):\n        if device not in (\'cpu\', \'cuda\'):\n            raise ValueError(\'explicit cpu or cuda device required\')\n        if type(threads) is not int or not 1 <= threads <= 4:\n            raise ValueError(\'CPU threads 1..4 required\')\n        import torch\n        if device == \'cuda\' and not torch.cuda.is_available():\n            raise RuntimeError(\'CUDA requested but unavailable; no CPU fallback\')\n        self.device = device\n        from huggingface_hub import hf_hub_download\n        from transformers import VitMatteImageProcessor, VitMatteForImageMatting\n        import transformers\n        torch.set_num_threads(threads)\n        self.torch = torch\n        t = time.monotonic()\n        weights = hf_hub_download(MODEL, \'model.safetensors\', revision=REVISION)\n        if sha256(weights) != WEIGHTS_SHA256:\n            raise ValueError(\'pinned model weights hash mismatch\')\n        self.processor = VitMatteImageProcessor.from_pretrained(MODEL, revision=REVISION)\n        self.model, info = VitMatteForImageMatting.from_pretrained(\n            MODEL, revision=REVISION, use_safetensors=True, output_loading_info=True)\n        if any(info.get(k) for k in (\'missing_keys\', \'unexpected_keys\', \'mismatched_keys\', \'error_msgs\')):\n            raise ValueError(\'strict checkpoint load failed: \' + repr(info))\n        self.model.eval().to(device)\n        self.info = dict(model=MODEL, revision=REVISION, weights_sha256=WEIGHTS_SHA256,\n                         load_seconds=time.monotonic() - t, loading_info=info,\n                         torch_version=torch.__version__, transformers_version=transformers.__version__,\n                         device=device, threads=threads,\n                         gpu_name=torch.cuda.get_device_name(0) if device == \'cuda\' else None)\n\n    def __call__(self, rgb, trimap):\n        h, w = trimap.shape\n        padded = ((h + 31) // 32 * 32, (w + 31) // 32 * 32)\n        inp = self.processor(images=Image.fromarray(rgb), trimaps=Image.fromarray(trimap), return_tensors=\'pt\')\n        if tuple(inp[\'pixel_values\'].shape) != (1, 4, *padded):\n            raise ValueError(\'processor dimensions changed beyond expected padding\')\n        inp = {key: value.to(self.device) for key, value in inp.items()}\n        if self.device == \'cuda\': self.torch.cuda.synchronize()\n        t = time.monotonic()\n        with self.torch.inference_mode():\n            pred = self.model(**inp).alphas\n        if self.device == \'cuda\': self.torch.cuda.synchronize()\n        if tuple(pred.shape) != (1, 1, *padded):\n            raise ValueError(\'unexpected model output shape\')\n        return pred[0, 0, :h, :w].detach().cpu().numpy().copy(), dict(\n            inference_seconds=time.monotonic() - t, padded_shape=list(padded))\n\n\ndef run(source, mask, output_dir, *, source_sha256, mask_sha256, radius=3, threads=2, predictor=None):\n    dest = Path(output_dir)\n    if dest.exists():\n        raise ValueError(\'output directory exists; prior evidence cannot be replaced\')\n    rgb, prior = read_bound(source, mask, source_sha256, mask_sha256)\n    tri = derive_trimap(prior, radius)\n    worker = predictor if predictor is not None else CpuPredictor(threads)\n    raw, timing = worker(rgb, tri)\n    alpha, a8 = checked_alpha(raw, tri)\n    metrics = dict(unknown_pixels=int((tri == 128).sum()), fractional_alpha_pixels=int(((a8 > 0) & (a8 < 255)).sum()),\n                   changed_coverage=int((prior != a8).sum()), support_added=int(((prior == 0) & (a8 > 0)).sum()),\n                   support_removed=int(((prior > 0) & (a8 == 0)).sum()), source_rgb_repainted=0,\n                   source_rgb_resampled=False, boundary_accuracy=None, publication_accepted=False)\n    dest.parent.mkdir(parents=True, exist_ok=True)\n    temp = Path(tempfile.mkdtemp(prefix=\'native-matting-\', dir=dest.parent))\n    try:\n        Image.fromarray(rgb).save(temp / \'source-native.png\')\n        Image.fromarray(prior).save(temp / \'prior-support-mask.png\')\n        Image.fromarray(tri).save(temp / \'derived-trimap.png\')\n        Image.fromarray(a8).save(temp / \'alpha.png\')\n        Image.fromarray(np.dstack([rgb, a8])).save(temp / \'component-rgba.png\')\n        np.save(temp / \'raw-alpha.npy\', raw, allow_pickle=False)\n        np.save(temp / \'enforced-alpha.npy\', alpha, allow_pickle=False)\n        for name, background in ((\'dark\', 35), (\'light\', 242)):\n            Image.fromarray(composite(rgb, a8, background)).save(temp / (name + \'.png\'))\n        outputs = {p.name: dict(sha256=sha256(p), bytes=p.stat().st_size) for p in temp.iterdir()}\n        manifest = dict(inputs=dict(source_sha256=source_sha256, mask_sha256=mask_sha256, native_shape=list(rgb.shape)),\n                        radius=radius, predictor=getattr(worker, \'info\', {\'unverified_injected_predictor\': True}),\n                        timing=timing, metrics=metrics, outputs=outputs,\n                        status=\'EXPERIMENTAL_NOT_PHOTO_ACCEPTED\', trimap_is_derived_not_independent_truth=True,\n                        composite_alpha=\'exported_quantized_uint8\', production_default_changed=False)\n        (temp / \'manifest.json\').write_text(json.dumps(manifest, indent=2) + \'\\n\')\n        dest.mkdir()  # exclusive reservation, no overwriting a competing writer\n        try:\n            for p in temp.iterdir():\n                p.rename(dest / p.name)\n        except BaseException:\n            shutil.rmtree(dest)\n            raise\n        return manifest\n    finally:\n        shutil.rmtree(temp)\n\n\ndef main():\n    p = argparse.ArgumentParser(description=__doc__)\n    for name in (\'source\', \'mask\', \'source-sha256\', \'mask-sha256\', \'output-dir\'):\n        p.add_argument(\'--\' + name, required=True)\n    p.add_argument(\'--radius\', type=int, default=3)\n    p.add_argument(\'--threads\', type=int, default=2)\n    a = p.parse_args()\n    try:\n        result = run(a.source, a.mask, a.output_dir, source_sha256=a.source_sha256,\n                     mask_sha256=a.mask_sha256, radius=a.radius, threads=a.threads)\n    except (ValueError, OSError) as e:\n        p.exit(2, \'REFUSED: \' + str(e) + \'\\n\')\n    print(json.dumps(result, indent=2))\n\n\nif __name__ == \'__main__\':\n    main()\n'
# ---- embedded module: renderello/checked_research/native_matting.py ----
path = base / 'renderello/checked_research/native_matting.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Explicit identity-core opacity override for research, not model-alpha truth.\nOnly previously reviewed opaque source rectangles may be restored. This can\ncreate seams, must be inspected, and never turns a candidate into accepted work.\n"""\nimport numpy as np\nfrom .segmentation_candidate import import_rgb_exact_candidate\n\ndef restore_reviewed_opaque_regions(source,candidate,reference_alpha,regions,*,pairing_reviewed=False,internal_resampling_declared=False,opaque_regions_reviewed=False):\n    out,base=import_rgb_exact_candidate(source,candidate,pairing_reviewed=pairing_reviewed,internal_resampling_declared=internal_resampling_declared)\n    if opaque_regions_reviewed is not True:raise ValueError(\'explicit opaque source-region review required\')\n    h,w=source.shape[:2]\n    if not isinstance(reference_alpha,np.ndarray) or reference_alpha.dtype!=np.uint8 or reference_alpha.shape!=(h,w):raise ValueError(\'matching reviewed uint8 reference alpha required\')\n    if not isinstance(regions,dict) or not regions:raise ValueError(\'named reviewed region rectangles required\')\n    union=np.zeros((h,w),bool)\n    for name,box in regions.items():\n        if not isinstance(name,str) or not name.strip():raise ValueError(\'nonempty region name required\')\n        if not isinstance(box,(list,tuple)) or len(box)!=4 or any(type(v) is not int for v in box):raise ValueError(\'four integer source coordinates required\')\n        x0,y0,x1,y1=box\n        if not 0<=x0<x1<=w or not 0<=y0<y1<=h:raise ValueError(\'source region outside bounds\')\n        if np.any(reference_alpha[y0:y1,x0:x1]!=255):raise ValueError(\'reference region is not fully opaque\')\n        if np.any(candidate[y0:y1,x0:x1,3]<240):raise ValueError(\'model alpha contradicts opaque-region declaration below240; review rather than overwrite\')\n        union[y0:y1,x0:x1]=True\n    changed=union&(out[:,:,3]!=255);out[union,3]=255\n    return out,dict(**base,alpha_override=\'explicit previously reviewed opaque source rectangles only\',alpha_override_pixels=int(changed.sum()),unique_region_pixels=int(union.sum()),region_names=list(regions),model_alpha_truth=False,visible_region_exact_only_after_verified_composition=True)\n'
# ---- embedded module: renderello/checked_research/opaque_regions.py ----
path = base / 'renderello/checked_research/opaque_regions.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Declared reviewed-support envelope, not automatic segmentation or true alpha.\nA candidate can only lose alpha outside a dilated prior-positive support. A prior\nomission cannot be recovered. Native source coordinates and source RGB stay fixed.\n"""\nimport numpy as np\nfrom scipy.ndimage import distance_transform_edt\n\ndef constrain_alpha(candidate,reviewed_prior,*,native_pairing_reviewed=False,prior_support_reviewed=False,radius=2):\n    if native_pairing_reviewed is not True or prior_support_reviewed is not True:\n        raise ValueError(\'native pairing and prior support review required\')\n    if not isinstance(candidate,np.ndarray) or candidate.dtype!=np.uint8 or candidate.ndim!=2:\n        raise ValueError(\'native uint8 candidate alpha required\')\n    if not isinstance(reviewed_prior,np.ndarray) or reviewed_prior.dtype!=np.uint8 or reviewed_prior.shape!=candidate.shape:\n        raise ValueError(\'matching uint8 reviewed prior alpha required\')\n    h,w=candidate.shape\n    if min(h,w)<2 or h*w>2_000_000:raise ValueError(\'native dimensions2..2MPrequired\')\n    if type(radius) is not int or not 0<=radius<=12:raise ValueError(\'integer radius0..12required\')\n    support=reviewed_prior>0\n    if not support.any() or support.all() or not (candidate>0).any():raise ValueError(\'nontrivial reviewed support and candidate required\')\n    keep=distance_transform_edt(~support)<=radius\n    out=np.where(keep,candidate,0).astype(\'uint8\')\n    return out,dict(method=\'candidate alpha restricted to declared prior-support distance envelope\',radius=radius,\n        candidate_support_pixels=int((candidate>0).sum()),result_support_pixels=int((out>0).sum()),\n        removed_support_pixels=int(((candidate>0)&~keep).sum()),prior_support_pixels=int(support.sum()),\n        prior_positive_lost_pixels=int((support&(out==0)).sum()),alpha_never_increased=True,\n        native_coordinates_unchanged=True,prior_omission_recovery=False,semantic_completeness=None,\n        boundary_accuracy=None,publication_accepted=False,production_promoted=False)\n'
# ---- embedded module: renderello/checked_research/prior_envelope.py ----
path = base / 'renderello/checked_research/prior_envelope.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Reproducible source-bound research commands. Never promotes output to accepted quality.\nExample: python -m renderello.checked_research.research_cli lighting --help\nHashes bind exact bytes, not semantic identity or alignment. --registration-reviewed is\nan explicit caller declaration and is retained as such, never independent verification.\n"""\nimport argparse, hashlib, json, shutil, tempfile\nfrom pathlib import Path\nimport numpy as np\nfrom PIL import Image\nfrom .evidence_checks import image_record, source_bound_alpha\nfrom .detail_transfer import transfer\nfrom .correspondence_render import render_correspondence\n\ndef file_record(path, expected):\n p=Path(path);digest=hashlib.sha256(p.read_bytes()).hexdigest()\n if digest!=expected:raise ValueError(\'file hash mismatch: \'+str(p))\n return dict(path=str(p.resolve()),sha256=digest,bytes=p.stat().st_size)\n\ndef execute(args):\n dest=Path(args.output_dir)\n if dest.exists():raise ValueError(\'output directory already exists; preserve prior evidence\')\n if args.command==\'lighting\':\n  binding=source_bound_alpha(args.source,args.alpha,args.source_sha256,expected_alpha_sha256=args.alpha_sha256)\n  rawrec=image_record(args.raw,args.raw_sha256)\n  with Image.open(args.source) as im:src=np.array(im.convert(\'RGB\'))\n  with Image.open(args.raw) as im:raw=np.array(im.convert(\'RGB\'))\n  with Image.open(args.alpha) as im:alpha=np.array(im)\n  rgb,gain,metrics=transfer(src,raw,alpha,registration_reviewed=args.registration_reviewed)\n  inputs=dict(binding=binding,raw=rawrec);extras={\'gain.npy\':gain};rgba=None\n else:\n  inputs=dict(background=image_record(args.background,args.background_sha256),correspondence=file_record(args.correspondence,args.correspondence_sha256),valid_mask=file_record(args.valid,args.valid_sha256))\n  with Image.open(args.background) as im:bg=np.array(im.convert(\'RGB\'))\n  with Image.open(args.valid) as im:\n   if im.mode!=\'L\':raise ValueError(\'explicit luminance valid mask required\')\n   valid=np.array(im)>127\n  corr=np.load(args.correspondence,allow_pickle=False)\n  rgb,alpha,metrics=render_correspondence(bg,corr,valid,out_of_image=args.out_of_image)\n  rgba=np.dstack([rgb,alpha]);extras={}\n # All validation/calculation precedes creating output. Existing evidence never replaced.\n dest.parent.mkdir(parents=True,exist_ok=True)\n temp=Path(tempfile.mkdtemp(prefix=\'renderello-research-\',dir=dest.parent))\n try:\n  Image.fromarray(rgb).save(temp/\'output.png\')\n  if rgba is not None:Image.fromarray(rgba,\'RGBA\').save(temp/\'component-rgba.png\')\n  for name,arr in extras.items():np.save(temp/name,arr,allow_pickle=False)\n  outputs={p.name:file_record(p,hashlib.sha256(p.read_bytes()).hexdigest()) for p in temp.iterdir()}\n  for name,record in outputs.items():record[\'path\']=str((dest/name).resolve())\n  manifest=dict(command=args.command,inputs=inputs,metrics=metrics,outputs=outputs,registration_review_is_caller_declaration=bool(getattr(args,\'registration_reviewed\',False)),publication_accepted=False,quality_accuracy=None)\n  (temp/\'manifest.json\').write_text(json.dumps(manifest,indent=2)+\'\\n\')\n  # mkdir is an exclusive reservation, so competing/existing output cannot be overwritten.\n  dest.mkdir()\n  try:\n   for p in temp.iterdir():p.rename(dest/p.name)\n  except BaseException:\n   shutil.rmtree(dest);raise\n  return manifest\n finally:shutil.rmtree(temp)\n\ndef parser():\n p=argparse.ArgumentParser(description=__doc__);sub=p.add_subparsers(dest=\'command\',required=True)\n l=sub.add_parser(\'lighting\');f=sub.add_parser(\'correspondence\')\n for name in (\'source\',\'raw\',\'alpha\'):\n  l.add_argument(\'--\'+name,required=True);l.add_argument(\'--\'+name+\'-sha256\',required=True)\n l.add_argument(\'--registration-reviewed\',action=\'store_true\')\n for name in (\'background\',\'correspondence\',\'valid\'):\n  f.add_argument(\'--\'+name,required=True);f.add_argument(\'--\'+name+\'-sha256\',required=True)\n f.add_argument(\'--out-of-image\',choices=(\'reject\',\'mask\',\'border\'),default=\'reject\')\n for s in (l,f):s.add_argument(\'--output-dir\',required=True)\n return p\nif __name__==\'__main__\':\n p=parser();a=p.parse_args()\n try:m=execute(a)\n except (ValueError,OSError) as e:p.exit(2,\'REFUSED: \'+str(e)+\'\\n\')\n print(json.dumps(m,indent=2))\n'
# ---- embedded module: renderello/checked_research/research_cli.py ----
path = base / 'renderello/checked_research/research_cli.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Explicit manual native GrabCut research. No default backend or quality pass."""\nfrom pathlib import Path\nimport hashlib,json,time\nimport cv2\nimport numpy as np\nfrom PIL import Image\n\nCLASSES={\'probable_foreground\':cv2.GC_PR_FGD,\'definite_foreground\':cv2.GC_FGD,\'definite_background\':cv2.GC_BGD}\n\ndef seed_map(data, source_bytes, size):\n    if not isinstance(data,dict) or data.get(\'source_sha256\')!=hashlib.sha256(source_bytes).hexdigest():\n        raise ValueError(\'exact source file SHA256 required\')\n    if data.get(\'source_size\')!=list(size):raise ValueError(\'source dimensions differ\')\n    if data.get(\'origin\')!=\'assistant visual review coarse manual polygons, not ground truth\':\n        raise ValueError(\'explicit manual research origin required\')\n    coord=data.get(\'annotation_coordinate_space\')\n    if not isinstance(coord,list) or len(coord)!=2 or any(type(x)is not int or x<2 for x in coord):raise ValueError(\'integer coordinate dimensions required\')\n    w,h=size;cw,ch=coord\n    if w*ch!=h*cw:raise ValueError(\'annotation aspect differs\')\n    result=np.zeros((h,w),np.uint8)\n    for key,cls in CLASSES.items():\n        polys=data.get(key)\n        if not isinstance(polys,list) or not 1<=len(polys)<=32:raise ValueError(\'one to32polygons per class required\')\n        for poly in polys:\n            a=np.asarray(poly)\n            if a.dtype.kind not in \'fiu\' or a.ndim!=2 or a.shape[1]!=2 or not 3<=len(a)<=128 or not np.isfinite(a).all():raise ValueError(\'bounded finite polygon required\')\n            if np.any(a<0) or np.any(a[:,0]>=cw) or np.any(a[:,1]>=ch):raise ValueError(\'polygon outside reviewed coordinate frame\')\n            q=np.rint(a*np.array([w/cw,h/ch])).astype(np.int32)\n            if cv2.contourArea(q)<1:raise ValueError(\'degenerate polygon\')\n            cv2.fillPoly(result,[q],cls)\n    if any(not np.any(result==x) for x in (cv2.GC_FGD,cv2.GC_BGD,cv2.GC_PR_FGD)):raise ValueError(\'all foreground/background constraints must survive precedence\')\n    return result\n\ndef run(source,seeds,out):\n    source,seeds,out=map(Path,(source,seeds,out))\n    if out.exists() or out.is_symlink():raise FileExistsError(\'preserve prior research output\')\n    if not source.is_file() or source.is_symlink() or source.stat().st_size>32_000_000:raise ValueError(\'bounded regular source required\')\n    raw=source.read_bytes();data=json.loads(seeds.read_text())\n    with Image.open(source) as im:\n        if im.mode!=\'RGB\' or getattr(im,\'n_frames\',1)!=1 or not 9<=im.width*im.height<=4_000_000:raise ValueError(\'native single-frame RGB3px..4MP required\')\n        if im.info.get(\'icc_profile\') or im.getexif().get(274,1)!=1:raise ValueError(\'explicit color/orientation normalization required\')\n        rgb=np.array(im);size=im.size\n    classes=seed_map(data,raw,size);initial=classes.copy()\n    cv2.setNumThreads(1);cv2.setRNGSeed(1182026);start=time.monotonic()\n    cv2.grabCut(cv2.cvtColor(rgb,cv2.COLOR_RGB2BGR),classes,None,np.zeros((1,65),np.float64),np.zeros((1,65),np.float64),5,cv2.GC_INIT_WITH_MASK)\n    alpha=np.where((classes==cv2.GC_FGD)|(classes==cv2.GC_PR_FGD),255,0).astype(np.uint8)\n    if np.any(alpha[initial==cv2.GC_FGD]!=255) or np.any(alpha[initial==cv2.GC_BGD]!=0):raise RuntimeError(\'manual hard seeds changed\')\n    out.mkdir(parents=True)\n    Image.fromarray(alpha).save(out/\'alpha.png\');Image.fromarray(np.dstack([rgb,alpha])).save(out/\'rgba.png\')\n    saved=np.array(Image.open(out/\'rgba.png\'))\n    if not np.array_equal(saved[:,:,:3],rgb):raise RuntimeError(\'native source RGB changed\')\n    # No scene or lighting is synthesized by this diagnostic.\n    for name,color in [(\'white\',(255,255,255)),(\'magenta\',(190,0,150))]:\n        preview=np.where(alpha[:,:,None]>0,rgb,np.array(color,np.uint8));Image.fromarray(preview).save(out/(name+\'-check.png\'))\n    report=dict(source_sha256=hashlib.sha256(raw).hexdigest(),seed_sha256=hashlib.sha256(seeds.read_bytes()).hexdigest(),opencv=cv2.__version__,seconds=time.monotonic()-start,source_size=list(size),source_RGB_changed=0,source_resampled=False,paid_calls=0,manual_supervision=True,mask_is_binary_not_optical=True,semantic_completeness=\'NOT_GROUND_TRUTH\',quality_pass=False,default_backend_changed=False)\n    (out/\'report.json\').write_text(json.dumps(report,indent=2));return report\n'
# ---- embedded module: renderello/checked_research/reviewed_grabcut.py ----
path = base / 'renderello/checked_research/reviewed_grabcut.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Native selected-subset assembly. Not automatic enumeration or photo acceptance.\n\nAll components come from the same source frame. Max-alpha union preserves that\nframe\'s positions and original RGB, including old lighting and contamination.\nIt does not infer occlusion, independent objects, contact, reflections or truth.\n"""\nimport hashlib\nimport numpy as np\n\n\ndef assemble_subset(source, components, *, native_pairing_reviewed=False):\n    if native_pairing_reviewed is not True:\n        raise ValueError(\'explicit same-source native crop pairing review required\')\n    if not isinstance(source, np.ndarray) or source.dtype != np.uint8 or source.ndim != 3 or source.shape[2] != 3:\n        raise ValueError(\'native uint8 RGB source required\')\n    h,w = source.shape[:2]\n    if min(h,w)<2 or h*w>2_000_000:\n        raise ValueError(\'native source dimensions 2..2MP required\')\n    if not isinstance(components,list) or not 1 <= len(components) <= 16:\n        raise ValueError(\'one to sixteen explicitly selected components required\')\n    union=np.zeros((h,w),np.uint8);count=np.zeros((h,w),np.uint8);names=set();records=[]\n    for c in components:\n        if not isinstance(c,dict) or set(c)!={\'name\',\'alpha\',\'crop\',\'source_crop_rgb_sha256\',\'selection_note\'}:\n            raise ValueError(\'explicit component fields required\')\n        name=c[\'name\'];note=c[\'selection_note\'];crop=c[\'crop\'];alpha=c[\'alpha\']\n        if not isinstance(name,str) or not name.strip() or len(name)>128 or name in names:\n            raise ValueError(\'unique nonempty component name required\')\n        if not isinstance(note,str) or not note.strip() or len(note)>2048:\n            raise ValueError(\'source-review selection note required\')\n        if not isinstance(crop,(list,tuple)) or len(crop)!=4 or any(type(v) is not int for v in crop):\n            raise ValueError(\'integer source crop bounds required\')\n        x0,y0,x1,y1=crop\n        if not (0<=x0<x1<=w and 0<=y0<y1<=h):\n            raise ValueError(\'crop outside source\')\n        if not isinstance(alpha,np.ndarray) or alpha.dtype!=np.uint8 or alpha.shape!=(y1-y0,x1-x0):\n            raise ValueError(\'native uint8 crop alpha required; no resizing\')\n        if not np.any(alpha):\n            raise ValueError(\'empty selected component refused\')\n        actual=hashlib.sha256(source[y0:y1,x0:x1].tobytes()).hexdigest()\n        if c[\'source_crop_rgb_sha256\']!=actual:\n            raise ValueError(\'component source-crop RGB hash differs\')\n        names.add(name);v=union[y0:y1,x0:x1];np.maximum(v,alpha,out=v);count[y0:y1,x0:x1]+=(alpha>0).astype(np.uint8)\n        records.append(dict(name=name,crop=list(crop),selection_note=note,source_crop_rgb_sha256=actual,\n                            alpha_sha256=hashlib.sha256(alpha.tobytes()).hexdigest(),support_pixels=int((alpha>0).sum())))\n    rgba=np.dstack([source,union])\n    manifest=dict(method=\'explicit same-source native max-alpha subset union\',components=records,\n                  overlap_support_pixels=int((count>1).sum()),union_support_pixels=int((union>0).sum()),\n                  source_rgb_sha256=hashlib.sha256(source.tobytes()).hexdigest(),source_rgb_exact=True,\n                  source_resampled=False,source_positions_unchanged=True,alpha_rule=\'maximum, not additive opacity\',\n                  semantic_object_count=None,semantic_completeness=None,independent_boundary_accuracy=None,\n                  publication_accepted=False,production_default_changed=False,\n                  limitations=[\'Selected subset only; omitted source items remain omitted.\',\n                               \'Component names and pairing review are caller declarations, not verified identities.\',\n                               \'Max union is for the same photographed frame, not independently layered new arrangements.\',\n                               \'Source lighting, reflections and boundary contamination remain.\'])\n    return rgba,manifest\n'
# ---- embedded module: renderello/checked_research/reviewed_subset.py ----
path = base / 'renderello/checked_research/reviewed_subset.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Import a reviewed source-RGB-exact BEN2 candidate, never automatic acceptance.\nModel inference resizes to1024square and returns resampled native-sized alpha.\nThis wrapper validates source alignment by byte comparison, not semantic truth.\nNo model download, inference, mask cleanup or alpha thresholding occurs here.\n"""\nimport numpy as np\n\ndef import_rgb_exact_candidate(source, candidate, *, pairing_reviewed=False, internal_resampling_declared=False):\n    if pairing_reviewed is not True:\n        raise ValueError(\'explicit source/candidate pairing review required\')\n    if internal_resampling_declared is not True:\n        raise ValueError(\'internal model resize and returned-alpha resampling must be declared\')\n    if not isinstance(source,np.ndarray) or source.dtype!=np.uint8 or source.ndim!=3 or source.shape[2]!=3:\n        raise ValueError(\'native uint8 RGB source required\')\n    h,w=source.shape[:2]\n    if min(h,w)<2 or h*w>4_194_304:\n        raise ValueError(\'native source2pixelsminimum peraxis,4MPmaximum\')\n    if not isinstance(candidate,np.ndarray) or candidate.dtype!=np.uint8 or candidate.shape!=(h,w,4):\n        raise ValueError(\'matching native-sized uint8 RGBA candidate required\')\n    if not np.array_equal(source,candidate[:,:,:3]):\n        raise ValueError(\'candidate RGB differs from source; foreground-refined output is not source-locked\')\n    a=candidate[:,:,3]\n    if not (a>127).any() or not (a==0).any():\n        raise ValueError(\'candidate needs nontrivial foreground and zero-alpha background\')\n    return candidate.copy(),dict(source_rgb_exact=True,source_resampled_by_wrapper=False,\n        native_model_inference=False,internal_model_size=[1024,1024],alpha_resampling_declared=True,\n        opaque_pixels=int((a==255).sum()),fractional_pixels=int(((a>0)&(a<255)).sum()),\n        zero_pixels=int((a==0).sum()),foreground_pixels=int((a>127).sum()),\n        alpha_accuracy=None,publication_accepted=False,production_promoted=False,\n        pairing_review_is_caller_declaration=True)\n'
# ---- embedded module: renderello/checked_research/segmentation_candidate.py ----
path = base / 'renderello/checked_research/segmentation_candidate.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Explicit withoutBG learned-matting-only CPU diagnostic. Never auto fallback.\n\nThe normal router\'s chosen category is recorded, not obeyed. This route is for\nreviewing a smaller branch where the normal routed test could not complete.\nPredicted alpha is model-resized; source RGB and native export size are fixed.\nNo model downloads, API calls, default route changes or quality acceptance.\n"""\nfrom pathlib import Path\nimport hashlib\nimport json\nimport shutil\nimport tempfile\nimport time\nimport numpy as np\nfrom PIL import Image\n\nMAX_PIXELS = 1_000_000\n\n\ndef file_hash(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for block in iter(lambda: f.read(1024 * 1024), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef _bound(path, expected):\n    if not isinstance(expected, str) or len(expected) != 64 or any(c not in \'0123456789abcdef\' for c in expected):\n        raise ValueError(\'explicit lowercase SHA256 required\')\n    if file_hash(path) != expected:\n        raise ValueError(\'file binding mismatch\')\n\n\ndef quantize_native_alpha(raw, height, width, resize):\n    if not isinstance(raw, np.ndarray) or raw.dtype.kind != \'f\' or raw.ndim != 3 or raw.shape[0] != 1:\n        raise ValueError(\'floating single-channel coarse alpha required\')\n    if not np.isfinite(raw).all() or raw.min() < 0 or raw.max() > 1:\n        raise ValueError(\'coarse alpha must be finite in [0,1], no invalid clipping\')\n    if any(type(x) is not int or x < 2 for x in (height, width)) or height * width > MAX_PIXELS:\n        raise ValueError(\'native 2px minimum and 1MP maximum required\')\n    a = resize(raw, height, width)\n    if not isinstance(a, np.ndarray) or a.shape != (1, height, width) or not np.isfinite(a).all() or a.min() < 0 or a.max() > 1:\n        raise ValueError(\'native alpha resize returned invalid result\')\n    return np.floor(a[0] * 255 + .5).astype(np.uint8)\n\n\ndef local_pipeline(sidecar, model_dir):\n    import onnxruntime as ort\n    from withoutbg.routed import RoutedPipeline\n    options = ort.SessionOptions()\n    options.intra_op_num_threads = 2\n    options.inter_op_num_threads = 1\n    root = Path(model_dir).resolve()\n    def resolve(name):\n        candidate = (root / name).resolve()\n        if not candidate.is_relative_to(root) or not candidate.is_file():\n            raise ValueError(\'existing local model inside supplied bundle required\')\n        return candidate\n    return RoutedPipeline(sidecar, resolve, providers=[\'CPUExecutionProvider\'],\n        session_factory=lambda p, providers: ort.InferenceSession(p, sess_options=options, providers=providers))\n\n\ndef run(source, model_dir, output_dir, *, source_sha256, sidecar_sha256,\n        reviewed_diagnostic=False, review_note=\'\', pipeline_factory=None, resize=None):\n    if reviewed_diagnostic is not True or not isinstance(review_note, str) or not review_note.strip() or len(review_note) > 2000:\n        raise ValueError(\'explicit diagnostic review and bounded note required\')\n    source, model_dir, dest = Path(source), Path(model_dir), Path(output_dir)\n    if dest.exists():\n        raise ValueError(\'output exists; preserve prior evidence\')\n    sidecar_path = model_dir / \'withoutbg-open-weights.onnx.json\'\n    _bound(source, source_sha256)\n    _bound(sidecar_path, sidecar_sha256)\n    sidecar = json.loads(sidecar_path.read_text())\n    if sidecar.get(\'schema_version\') != 3 or sidecar.get(\'pipeline\') != \'routed\':\n        raise ValueError(\'explicit schema3 routed bundle required\')\n    with Image.open(source) as im:\n        if im.mode != \'RGB\' or min(im.size) < 2 or im.width * im.height > MAX_PIXELS:\n            raise ValueError(\'native RGB 2px..1MP source required; no silent conversion\')\n        rgb = np.array(im)\n        icc = im.info.get(\'icc_profile\')\n    if resize is None:\n        from withoutbg.routed import resize_bilinear\n        resize = resize_bilinear\n    worker = (pipeline_factory or local_pipeline)(sidecar, model_dir)\n    start = time.monotonic()\n    normalized = rgb.astype(np.float32) / 255\n    category, feeds, features = worker._route(normalized)\n    if category not in worker.categories:\n        raise ValueError(\'router category outside declared bundle\')\n    # Deliberate review route. Never replace a failed default result silently.\n    coarse = worker._coarse(normalized, feeds, features)\n    alpha = quantize_native_alpha(coarse, *rgb.shape[:2], resize)\n    providers = {name: session.get_providers() for name, session in worker.sessions.items()}\n    if set(providers) != {\'router\', \'coarse\'} or any(p != [\'CPUExecutionProvider\'] for p in providers.values()):\n        raise ValueError(\'only requested CPU router/coarse sessions permitted\')\n    result = dict(schema=\'withoutbg-matting-only-diagnostic.v1\', source_sha256=source_sha256,\n        sidecar_sha256=sidecar_sha256, model_version=sidecar.get(\'model_version\'),\n        router_category=category, normal_route=\'birefnet\' if category in worker.birefnet_categories else \'matting\',\n        executed_route=\'FORCED_LEARNED_MATTING_DIAGNOSTIC\', review_note=review_note,\n        native_size=[rgb.shape[1], rgb.shape[0]], providers=providers,\n        inference_seconds=time.monotonic()-start, source_rgb_repainted=0,\n        source_rgb_resampled=False, model_internally_resizes=True,\n        default_route_changed=False, automatic_fallback=False, boundary_ground_truth=False,\n        physical_relighting=False, photographic_acceptance=False, paid_calls=0)\n    # Structural flags must travel with every result. They never choose which\n    # component is the product or authorize removal of small parts.\n    if min(alpha.shape) >= 3:\n        from renderello.candidate_diagnostics import mask_diagnostics\n        result[\'mask_review\'] = mask_diagnostics(rgb, alpha)\n    else:\n        result[\'mask_review\'] = dict(flags=[\'TOO_SMALL_FOR_STRUCTURAL_REVIEW\'],\n            semantic_completeness=\'NOT_EVALUATED\', automatic_acceptance=False)\n    dest.parent.mkdir(parents=True, exist_ok=True)\n    temp = Path(tempfile.mkdtemp(prefix=\'matting-diagnostic-\', dir=dest.parent))\n    try:\n        kw = {\'icc_profile\': icc} if icc else {}\n        Image.fromarray(alpha).save(temp/\'alpha.png\')\n        Image.fromarray(np.dstack([rgb, alpha])).save(temp/\'component-rgba.png\', **kw)\n        saved = np.array(Image.open(temp/\'component-rgba.png\'))\n        if not np.array_equal(saved[:, :, :3], rgb) or not np.array_equal(saved[:, :, 3], alpha):\n            raise RuntimeError(\'native export invariant failed\')\n        result[\'rgba_sha256\'] = file_hash(temp/\'component-rgba.png\')\n        (temp/\'result.json\').write_text(json.dumps(result, indent=2))\n        if dest.exists():\n            raise ValueError(\'output appeared during run; preserve it\')\n        temp.rename(dest)\n    finally:\n        if temp.exists():\n            shutil.rmtree(temp)\n    return result\n\n\ndef main(argv=None):\n    """Explicit opt-in command, separate from production segmentation defaults."""\n    import argparse\n    parser = argparse.ArgumentParser(description=\'Reviewed CPU learned-matting-only diagnostic. Not a production fallback or quality acceptance.\')\n    parser.add_argument(\'--source\', type=Path, required=True)\n    parser.add_argument(\'--model-dir\', type=Path, required=True)\n    parser.add_argument(\'--output-dir\', type=Path, required=True)\n    parser.add_argument(\'--source-sha256\', required=True)\n    parser.add_argument(\'--sidecar-sha256\', required=True)\n    parser.add_argument(\'--review-note\', required=True)\n    parser.add_argument(\'--reviewed-diagnostic\', action=\'store_true\', required=True)\n    args = parser.parse_args(argv)\n    result = run(args.source, args.model_dir, args.output_dir,\n        source_sha256=args.source_sha256, sidecar_sha256=args.sidecar_sha256,\n        review_note=args.review_note, reviewed_diagnostic=args.reviewed_diagnostic)\n    print(json.dumps(result, indent=2))\n    return result\n\n\nif __name__ == \'__main__\':\n    main()\n'
# ---- embedded module: renderello/checked_research/withoutbg_matting_diagnostic.py ----
path = base / 'renderello/checked_research/withoutbg_matting_diagnostic.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Explicit lossless RGB-container lineage, never rights or photograph clearance.\n\nNot an upload-name heuristic or default match. Both bounded original and derived\nfiles are supplied by the caller. A report describes these snapshots only.\n"""\nfrom io import BytesIO\nfrom pathlib import Path\nimport hashlib,json,os,stat\nimport numpy as np\nfrom PIL import Image\n\nMAX_BYTES=16_000_000\nMAX_PIXELS=1_000_000\n\ndef _snapshot(path):\n    p=Path(path)\n    if p.is_symlink():raise ValueError(\'Regular bounded source file required\')\n    with os.fdopen(os.open(p,os.O_RDONLY|getattr(os,\'O_NOFOLLOW\',0)|getattr(os,\'O_NONBLOCK\',0)),\'rb\') as f:\n        st=os.fstat(f.fileno())\n        if not stat.S_ISREG(st.st_mode) or st.st_size>MAX_BYTES:raise ValueError(\'Regular bounded source file required\')\n        data=f.read(MAX_BYTES+1)\n    if len(data)>MAX_BYTES:raise ValueError(\'Source byte limit exceeded\')\n    return data\n\ndef _rgb(data):\n    if data.startswith(b\'\\x89PNG\\r\\n\\x1a\\n\'):\n        pos=8\n        while pos<len(data):\n            if len(data)-pos<12:raise ValueError(\'Incomplete PNG metadata stream\')\n            length=int.from_bytes(data[pos:pos+4],\'big\');kind=data[pos+4:pos+8]\n            if pos+12+length>len(data):raise ValueError(\'Incomplete PNG metadata stream\')\n            if kind in {b\'cICP\',b\'mDCV\',b\'cLLI\',b\'tRNS\',b\'acTL\',b\'fcTL\',b\'fdAT\'}:\n                raise ValueError(\'Unsupported colour, transparency, or animation metadata\')\n            pos+=12+length\n    with Image.open(BytesIO(data)) as im:\n        if im.format not in (\'JPEG\',\'PNG\') or im.mode!=\'RGB\':raise ValueError(\'Native RGB JPEG or PNG required; no conversion\')\n        if im.width*im.height>MAX_PIXELS or getattr(im,\'n_frames\',1)!=1:raise ValueError(\'Single RGB frame up to one megapixel required\')\n        if im.getexif().get(274,1)!=1:raise ValueError(\'Orientation transform not permitted\')\n        # Even visually equivalent metadata must not be silently discarded.\n        color={k:im.info.get(k) for k in (\'icc_profile\',\'gamma\',\'srgb\',\'chromaticity\')}\n        color[\'exif\']=dict(im.getexif())\n        return np.array(im),color,im.format\n\ndef verify_lossless_container_lineage(original,derived):\n    """Return explicit content ancestry only; leave default source lookup alone."""\n    a,b=_snapshot(original),_snapshot(derived)\n    rgb_a,color_a,fmt_a=_rgb(a);rgb_b,color_b,fmt_b=_rgb(b)\n    if fmt_b!=\'PNG\':raise ValueError(\'Derived container must be PNG\')\n    if rgb_a.shape!=rgb_b.shape or not np.array_equal(rgb_a,rgb_b):raise ValueError(\'Decoded native RGB changed\')\n    if color_a!=color_b:raise ValueError(\'Colour or EXIF interpretation metadata changed\')\n    ha,hb=(hashlib.sha256(v).hexdigest() for v in (a,b))\n    registry=Path(__file__).with_name(\'source_registry.json\')\n    records=json.loads(registry.read_text()) if registry.is_file() else []\n    matches=[r for r in records if r.get(\'sha256\')==ha]\n    if len(matches)>1:raise ValueError(\'Ambiguous parent source registry\')\n    return dict(schema=\'renderello-explicit-source-lineage.v1\',original_file_sha256=ha,\n        derived_file_sha256=hb,original_container=fmt_a,derived_container=fmt_b,\n        size=[rgb_a.shape[1],rgb_a.shape[0]],decoded_rgb_exact=True,\n        interpretation_metadata_exact=True,parent_record=matches[0] if matches else None,\n        parent_record_status=\'Historical content-hash lookup only; not authenticated authorship or clearance\',\n        source_photo_origin_verified=False,rights_verified=False,publication_approved=False,\n        default_source_lookup_changed=False,\n        caveat=\'Caller-supplied ancestry verified for these file snapshots only. RGB equality does not prove genuine capture, author identity, licensing, alpha accuracy, or scene quality.\')\n'
# ---- embedded module: renderello/checked_source_lineage.py ----
path = base / 'renderello/checked_source_lineage.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""CLI entry points. Defaults are safe: CPU device, mock Magic Hour, no live\ncalls, no GPU assumptions. Every flag that could spend or change behavior\noutside the local machine is explicit opt-in.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport sys\nfrom pathlib import Path\n\nfrom .benchmark import run_benchmark\nfrom .config import DeviceConfig, RunConfig\nfrom .gallery import build_gallery\nfrom .pipeline import run_pipeline\nfrom .schemas import EditRequest, Route\nfrom .templates import TEMPLATES, all_templates, get_template\n\n\ndef _fid(score):\n    return "NOT_EVALUATED (review needed)" if score is None else f"{score}/5"\n\n\ndef _routes(text: str) -> list[Route]:\n    names = [t.strip() for t in text.split(",") if t.strip()]\n    out = []\n    for n in names:\n        try:\n            out.append(Route(n))\n        except ValueError:\n            raise SystemExit(f"unknown route \'{n}\'. choices: "\n                             f"{\', \'.join(r.value for r in Route)}")\n    return out\n\n\ndef _editor(args, cfg: RunConfig):\n    if args.editor == "stub":\n        from .backends.base import StubEditor\n        return StubEditor()\n    if args.editor == "qwen":\n        from .backends.qwen_local import QwenEditBackend\n        return QwenEditBackend(device=cfg.device)\n    if args.editor == "magichour":\n        from .backends.magichour import MagicHourBackend\n        live = bool(args.live)\n        if live:\n            print("WARNING: --live spends real Magic Hour credits. Proceeding only "\n                  "because --live was passed explicitly.", file=sys.stderr)\n        return MagicHourBackend(model=args.mh_model, live=live,\n                                cost_log=cfg.out_root / "mh_cost_log.jsonl")\n    return None\n\n\ndef _scene_fn(args):\n    """Scene function for scene_compose. \'mock\' = labeled gradient (no GPU).\n    \'iclight\' = real IC-Light relight on CUDA (needs a GPU host + weights; never\n    exercised on CPU)."""\n    kind = getattr(args, "scene", "mock")\n    if kind == "iclight":\n        from .iclight_gpu import ICLight\n        ic = ICLight(getattr(args, "iclight_weights", "models/iclight_sd15_fc.safetensors"))\n        return (lambda gray, prompt: ic.relight(gray, prompt)[0]), ic\n    from .webui import _mock_scene_fn\n    return _mock_scene_fn, None\n\n\ndef main(argv: list[str] | None = None) -> int:\n    # The candidate route owns its parser and lazy optional imports. Existing\n    # commands remain usable without importing its color/matting dependencies.\n    forwarded = list(sys.argv[1:] if argv is None else argv)\n    research_commands = {\n        \'brief-economics\': \'brief_economics\',\n        \'provider-attempts\': \'provider_attempts\',\n        \'blind-review-return\': \'blind_review_return\',\n        \'native-region-matte\': \'native_region_cli\',\n        \'blind-review-pack\': \'blind_pack_cli\',\n    }\n    if forwarded and forwarded[0] in research_commands:\n        from importlib import import_module\n        import_module(\'.\' + research_commands[forwarded[0]], __package__).main(forwarded[1:])\n        return 0\n    if forwarded[:1] == [\'candidate-studio\']:\n        from .candidate_studio import main as candidate_main\n        candidate_main(forwarded[1:])\n        return 0\n    p = argparse.ArgumentParser(prog="renderello",\n                                description="AI product photography prototype pipeline")\n    sub = p.add_subparsers(dest="cmd", required=True)\n    for name in research_commands:\n        sub.add_parser(name, help=\'offline evidence/research workflow; use \'+name+\' --help; no automatic quality acceptance\')\n    sub.add_parser(\'candidate-studio\', help=\'source-bound multi-model masks, edge diagnostics, optional matting/contact/harmonization research; never automatic acceptance\')\n\n    run = sub.add_parser("run", help="run the pipeline on one source image")\n    run.add_argument("--source", required=True, type=Path)\n    run.add_argument("--allow-colour-conversion", action="store_true", help="explicit ICC-to-sRGB conversion for source/reference; original files retained, encoded RGB may change")\n    run.add_argument("--templates", default="clean_studio",\n                     help="comma list or \'all\'")\n    run.add_argument("--routes", default="cutout_compose")\n    run.add_argument("--variants", type=int, default=2, choices=[1, 2, 3, 4])\n    instructions=run.add_mutually_exclusive_group()\n    instructions.add_argument("--instruction", default="")\n    instructions.add_argument("--instruction-file", type=Path, help="UTF-8 multiline scene instruction; explicit 16000 character limit, no silent truncation")\n    run.add_argument("--use-embedded-alpha", action="store_true", help="explicit uploaded source alpha for composition only; not mask/material verification")\n    run.add_argument("--mask", type=Path, default=None, help="explicit source-sized grayscale alpha mask; manual assist, no auto segmentation")\n    run.add_argument("--source-pose", choices=[\'unknown\',\'front\',\'elevated\',\'overhead\',\'suspended\'], default=\'unknown\')\n    run.add_argument("--source-material", choices=[\'unknown\',\'opaque\',\'transparent\'], default=\'unknown\')\n    run.add_argument("--source-framing", choices=[\'unknown\',\'complete\',\'cropped\'], default=\'unknown\')\n    run.add_argument("--require-complete", action=\'store_true\')\n    run.add_argument("--reference", type=Path, default=None,\n                     help="optional reference/backdrop photo (cover-fitted; PDF section 4 optional)")\n    run.add_argument("--seed", type=int, default=0)\n    run.add_argument("--editor", default="stub",\n                     choices=["stub", "qwen", "magichour", "none"])\n    run.add_argument("--mh-model", default="qwen-edit")\n    run.add_argument("--mh-mock", action="store_true",\n                     help="force Magic Hour mock (default unless --live)")\n    run.add_argument("--live", action="store_true",\n                     help="REAL Magic Hour calls - spends credits (owner approval required)")\n    run.add_argument("--gpu", action="store_true", help="resolve CUDA device")\n    run.add_argument("--no-relight", action="store_true")\n    run.add_argument("--scene", choices=["mock", "iclight"], default="mock", help="scene_compose scene source: mock gradient (no GPU) or real IC-Light (GPU host only)")\n    run.add_argument("--scene-finish", action="store_true", help="experimental scene-only contact shadow and dark-scene lift; CPU-tested, not real-GPU validated")\n    run.add_argument("--glass-mode", action="store_true", help="scene_compose: clear-glass transmission mode (clean-backdrop sources only)")\n    run.add_argument("--iclight-weights", default="models/iclight_sd15_fc.safetensors")\n    run.add_argument("--out", type=Path, default=Path("runs"))\n\n    cost = sub.add_parser("costs", help="cost projection from published MH list data (no spend)")\n    cost.add_argument("--model", default="qwen-edit")\n    cost.add_argument("--plan", default="creator_monthly")\n    cost.add_argument("--credits", type=float, default=None,\n                      help="override credits per generation (higher res / batch)")\n    cost.add_argument("--images", type=int, default=1)\n\n    cmp_ = sub.add_parser("mh-compare", help="S2: compare 2-3 Magic Hour editor models "\n                          "on one source. Default prints a zero-spend plan; "\n                          "--live SPENDS credits and needs owner approval")\n    cmp_.add_argument("--source", type=Path)\n    cmp_.add_argument("--models", default="flux-2-klein,qwen-edit,seedream-v4")\n    cmp_.add_argument("--template", default="clean_studio")\n    cmp_.add_argument("--plan", default="creator_monthly")\n    cmp_.add_argument("--seed", type=int, default=0)\n    cmp_.add_argument("--aspect", default="1:1", choices=["1:1", "4:5", "3:2"])\n    cmp_.add_argument("--live", action="store_true",\n                      help="SPEND Magic Hour credits (owner approval required)")\n    cmp_.add_argument("--out", type=Path, default=Path("runs/mh-compare"))\n\n    recover = sub.add_parser("mh-recover", help="read-only recovery of an existing MH project; no generation")\n    recover.add_argument("--project-id", required=True)\n    recover.add_argument("--output", type=Path, required=True)\n    replay = sub.add_parser("scene-replay", help="CPU-only replay of saved exact layers, no model calls")\n    replay.add_argument("--cache", type=Path, required=True)\n    replay.add_argument("--output", type=Path, required=True)\n    replay.add_argument("--shadow", choices=["soft", "hard", "none"])\n    replay.add_argument("--lift", action="store_true")\n\n    native = sub.add_parser("native-reframe", help="smaller source-sized alternate, no upscaling or model call")\n    native.add_argument("--cache", type=Path, required=True)\n    native.add_argument("--output", type=Path, required=True)\n    native.add_argument("--fill", type=float, default=.76)\n\n    ground = sub.add_parser("ground-replay", help="experimental explicit designed foreground, CPU-only")\n    ground.add_argument("--cache", type=Path, required=True)\n    ground.add_argument("--output", type=Path, required=True)\n    ground.add_argument("--horizon", type=int, required=True)\n    ground.add_argument("--color", type=int, nargs=3, required=True)\n\n    bench = sub.add_parser("benchmark", help="run the acceptance benchmark")\n    bench.add_argument("--manifest", required=True, type=Path)\n    bench.add_argument("--allow-colour-conversion", action="store_true", help="explicit ICC-to-sRGB policy for all benchmark sources; preflight occurs before generation")\n    bench.add_argument("--variants", type=int, default=2, choices=[1, 2, 3, 4])\n    bench.add_argument("--editor", default="stub",\n                       choices=["stub", "qwen", "magichour", "none"])\n    bench.add_argument("--mh-model", default="qwen-edit")\n    bench.add_argument("--live", action="store_true")\n    bench.add_argument("--gpu", action="store_true")\n    bench.add_argument("--seed", type=int, default=0)\n    bench.add_argument("--scene", default="mock", choices=["mock", "iclight"])\n    bench.add_argument("--iclight-weights", default="models/iclight_sd15_fc.safetensors")\n    bench.add_argument("--out", type=Path, default=Path("runs"))\n\n    gal = sub.add_parser("gallery", help="build a private HTML gallery from run dirs")\n    gal.add_argument("--runs", nargs="+", required=True, type=Path)\n    gal.add_argument("--out", required=True, type=Path)\n\n    summ = sub.add_parser("summarize", help="aggregate existing run manifests "\n                          "through the benchmark gate math (no recompute)")\n    summ.add_argument("--runs", nargs="+", required=True, type=Path)\n    summ.add_argument("--out", required=True, type=Path)\n\n    review_export = sub.add_parser("review-export", help="export blank seven-dimension ratings for existing outputs")\n    review_export.add_argument("--runs", nargs="+", required=True, type=Path)\n    review_export.add_argument("--out", required=True, type=Path)\n    review_export.add_argument("--portable", action="store_true", help="write output paths relative to sheet folder")\n    review_check = sub.add_parser("review-check", help="validate complete scores and exact output hashes; no acceptance claim")\n    review_check.add_argument("--sheet", required=True, type=Path)\n\n    bal = sub.add_parser("mh-balance", help="read-only Magic Hour credit balance")\n    bal.add_argument("--json", action="store_true")\n\n    matte = sub.add_parser("matte-refine", help="opt-in trimap matting/edge colors; no model downloads or acceptance claim")\n    matte.add_argument("--source", type=Path, required=True)\n    matte.add_argument("--alpha", type=Path, required=True)\n    matte.add_argument("--trimap", type=Path, default=None)\n    matte.add_argument("--method", choices=["cf","lkm","guided"], default="cf")\n    matte.add_argument("--radius", type=int, default=3)\n    matte.add_argument("--estimate-edge-colors", action="store_true", help="changes boundary RGB; preserves only definite foreground core exactly")\n    matte.add_argument("--out", type=Path, required=True)\n\n    matte_comp = sub.add_parser("matte-compose", help="compose a verified matting bundle on a declared overhead background")\n    matte_comp.add_argument("--bundle", type=Path, required=True)\n    matte_comp.add_argument("--background", type=Path, required=True, help="reviewed overhead background, not an upright scene")\n    matte_comp.add_argument("--source-pose", choices=["overhead"], required=True)\n    matte_comp.add_argument("--source-material", choices=["opaque"], required=True)\n    matte_comp.add_argument("--source-framing", choices=["complete"], required=True)\n    matte_comp.add_argument("--fill", type=float, default=.68)\n    matte_comp.add_argument("--out", type=Path, required=True)\n\n    semantic = sub.add_parser("semantic-mask", help="explicit isolated CPU IS-Net trial, not automatic quality acceptance")\n    semantic.add_argument("--source", type=Path, required=True)\n    semantic.add_argument("--out", type=Path, required=True)\n    semantic.add_argument("--allow-download", action="store_true", help="allow official checksum-verified179MB IS-Net download inside Renderello")\n\n    prompted = sub.add_parser("prompted-mask", help="explicit reviewed boxes/points with CPU Light HQ-SAM")\n    for name in ("source", "out", "prompts", "checkpoint"):\n        prompted.add_argument("--"+name,type=Path,required=True)\n\n    pyramid = sub.add_parser("pyramid-mask", help="explicit CPU InSPyReNet fast experiment, not quality acceptance")\n    for name in ("source", "out", "checkpoint"):\n        pyramid.add_argument("--"+name,type=Path,required=True)\n\n    bilateral = sub.add_parser("bilateral-mask", help="explicit CPU BiRefNet-lite trial; high memory use, not automatic acceptance")\n    for name in ("source", "out", "checkpoint"):\n        bilateral.add_argument("--"+name,type=Path,required=True)\n\n    intent = sub.add_parser("intent-mask", help="explicit reviewed HQ-SAM intent intersected with existing automatic alpha")\n    for name in ("source", "automatic", "reviewed", "out"):\n        intent.add_argument("--"+name,type=Path,required=True)\n\n    opaque = sub.add_parser("opaque-recover",help="explicit reviewed opaque silhouette to trimap matting; can replace missing saliency support, not automatic")\n    for key in (\'source\',\'reviewed\',\'out\'):opaque.add_argument(\'--\'+key,type=Path,required=True)\n    opaque.add_argument(\'--reviewed-alpha-sha256\',required=True)\n    opaque.add_argument(\'--material\',choices=[\'opaque\',\'transparent\',\'unknown\'],required=True)\n    opaque.add_argument(\'--framing\',choices=[\'complete\',\'cropped\',\'unknown\'],required=True)\n    opaque.add_argument(\'--confirm-reviewed-silhouette\',action=\'store_true\')\n    opaque.add_argument(\'--method\',choices=[\'cf\',\'lkm\'],default=\'cf\')\n    opaque.add_argument(\'--radius\',type=int,default=3)\n    labels = sub.add_parser(\'label-retention\',help=\'source-bound same-coordinate pixel/opacity checks, not OCR acceptance\')\n    for key in (\'source\',\'cutout\',\'regions\',\'out\'):labels.add_argument(\'--\'+key,type=Path,required=True)\n\n    studio=sub.add_parser(\'native-studio\',help=\'eight designed 2D studio candidates with native1:1product, not photorealistic template acceptance\')\n    studio.add_argument(\'--bundle\',type=Path,required=True)\n    studio.add_argument(\'--template\',default=\'all\')\n    studio.add_argument(\'--out\',type=Path,required=True)\n    studio.add_argument(\'--regions\',type=Path,help=\'optional exact source-bound print regions, checked in final scene after native translation\')\n    studio.add_argument(\'--source-pose\',choices=[\'front\',\'elevated\',\'overhead\',\'unknown\'],required=True)\n    studio.add_argument(\'--material\',choices=[\'opaque\',\'transparent\',\'unknown\'],required=True)\n    studio.add_argument(\'--framing\',choices=[\'complete\',\'cropped\',\'unknown\'],required=True)\n\n    refr=sub.add_parser(\'refractive-transfer\',help=\'experimental clean-plate transfer with external reviewed flow/transmission, not a glass estimator\')\n    for field in (\'source\',\'plate\',\'target\',\'flow\',\'transmission\',\'support\',\'spec\',\'out\'):\n        refr.add_argument(\'--\'+field,type=Path,required=True)\n    refr.add_argument(\'--confirm-assumptions\',action=\'store_true\')\n\n    plate=sub.add_parser(\'scene-plate\',help=\'native product over an inspected 1024px RGB background with declared support geometry; not physical lighting or glass\')\n    for field in (\'prepared\',\'background\',\'out\'):\n        plate.add_argument(\'--\'+field,type=Path,required=True)\n    placement=plate.add_mutually_exclusive_group(required=True)\n    placement.add_argument(\'--baseline\',type=int)\n    placement.add_argument(\'--source-origin\',type=int,nargs=2,metavar=(\'X\',\'Y\'),help=\'original normalized source origin on canvas; keeps source coordinates fixed across alpha changes\')\n    plate.add_argument(\'--center-x\',type=int,default=512)\n    plate.add_argument(\'--cast-offset\',type=float,nargs=2,default=(0,0),metavar=(\'DX\',\'DY\'))\n    plate.add_argument(\'--no-contact\',action=\'store_true\')\n    plate.add_argument(\'--background-kind\',choices=[\'photograph\',\'generated\',\'procedural\',\'unspecified_inspected_background\'],default=\'unspecified_inspected_background\')\n    plate.add_argument(\'--support-plane\',type=Path,help=\'reviewed1024pxL support-plane mask; rejects unsupported feet and clips scene shadow\')\n    plate.add_argument(\'--contact-strength\',type=float,default=1.)\n    plate.add_argument(\'--shadow-model\',choices=[\'legacy\',\'height_soft_linear\'],default=\'legacy\')\n    plate.add_argument(\'--shadow-softness\',type=float,default=1.)\n    plate.add_argument(\'--reviewed-contact-span\',type=int,nargs=2,metavar=(\'LEFT\',\'RIGHT\'),help=\'explicit half-open original source columns for reviewed sloped-base contact; height_soft_linear only\')\n    plate.add_argument(\'--scene-exposure-ev\',type=float,default=0.,help=\'background-only tone matching, half-stop maximum; not physical relighting\')\n    plate.add_argument(\'--scene-white-balance\',type=float,nargs=3,default=(1.,1.,1.),metavar=(\'R\',\'G\',\'B\'))\n\n    args = p.parse_args(argv)\n    if args.cmd == \'run\':\n        from .prompts import validate_instruction\n        if args.instruction_file is not None:\n            args.instruction=args.instruction_file.read_text(encoding=\'utf-8\')\n        args.instruction=validate_instruction(args.instruction)\n    if args.cmd == \'scene-plate\':\n        from .scene_plate import compose_plate\n        shadow_opts={} if args.shadow_model==\'legacy\' and args.shadow_softness==1. else dict(shadow_model=args.shadow_model,shadow_softness=args.shadow_softness)\n        if args.reviewed_contact_span is not None:shadow_opts[\'reviewed_contact_span\']=args.reviewed_contact_span\n        if args.source_origin is not None:shadow_opts[\'source_origin\']=args.source_origin\n        result=compose_plate(args.prepared,args.background,args.out,baseline=args.baseline,\n            center_x=args.center_x,cast_offset=args.cast_offset,contact=not args.no_contact,\n            support_plane_path=args.support_plane,contact_strength=args.contact_strength,**shadow_opts,\n            scene_exposure_ev=args.scene_exposure_ev,scene_white_balance=args.scene_white_balance,background_kind=args.background_kind)\n        print(json.dumps(result,indent=2));return 0\n\n    if args.cmd == \'refractive-transfer\':\n        from .refractive_transfer import transfer_files\n        print(json.dumps(transfer_files(args.source,args.plate,args.target,args.flow,args.transmission,args.support,args.spec,args.out,confirm_assumptions=args.confirm_assumptions),indent=2));return 0\n\n    if args.cmd == \'native-studio\':\n        from .native_studio import compose_native,STYLES\n        keys=list(STYLES) if args.template==\'all\' else args.template.split(\',\')\n        if len(set(keys))!=len(keys) or any(k not in STYLES for k in keys):raise ValueError(\'invalid or duplicate native studio template\')\n        if args.out.exists():raise ValueError(\'output exists; preserve earlier candidates\')\n        print(json.dumps([compose_native(args.bundle,k,args.out/k,source_pose=args.source_pose,material=args.material,framing=args.framing,regions_path=args.regions) for k in keys],indent=2));return 0\n    if args.cmd == \'opaque-recover\':\n        from .opaque_recovery import recover_file\n        print(json.dumps(recover_file(args.source,args.reviewed,args.out,reviewed_alpha_sha256=args.reviewed_alpha_sha256,material=args.material,framing=args.framing,confirm_reviewed_silhouette=args.confirm_reviewed_silhouette,method=args.method,radius=args.radius),indent=2));return 0\n    if args.cmd == \'label-retention\':\n        from .label_retention import check_file\n        print(json.dumps(check_file(args.source,args.cutout,args.regions,args.out),indent=2));return 0\n    if args.cmd == "intent-mask":\n        from .intent_mask import gate_file\n        print(json.dumps(gate_file(args.source,args.automatic,args.reviewed,args.out),indent=2))\n        return 0\n    if args.cmd == "bilateral-mask":\n        from .bilateral_mask import guarded_file\n        print(json.dumps(guarded_file(args.source,args.out,args.checkpoint),indent=2))\n        return 0\n    if args.cmd == "pyramid-mask":\n        from .pyramid_mask import guarded_file\n        print(json.dumps(guarded_file(args.source,args.out,args.checkpoint),indent=2))\n        return 0\n    if args.cmd == "prompted-mask":\n        from .prompted_mask import guarded_file\n        print(json.dumps(guarded_file(args.source,args.out,args.prompts,args.checkpoint),indent=2))\n        return 0\n    if args.cmd == "semantic-mask":\n        from .semantic_mask import guarded_file\n        print(json.dumps(guarded_file(args.source,args.out,args.allow_download),indent=2))\n        return 0\n    if args.cmd == "matte-compose":\n        from .matting import compose_bundle\n        print(json.dumps(compose_bundle(args.bundle,args.background,args.out,source_pose=args.source_pose,\n              material=args.source_material,framing=args.source_framing,fill=args.fill),indent=2))\n        return 0\n    if args.cmd == "matte-refine":\n        from .matting import guarded_file\n        print(json.dumps(guarded_file(args.source,args.alpha,args.out,trimap_path=args.trimap,\n              method=args.method,radius=args.radius,edge_colors=args.estimate_edge_colors),indent=2))\n        return 0\n    if args.cmd == "ground-replay":\n        from .ground_plane import replay_ground_plane\n        print(json.dumps(replay_ground_plane(args.cache, output=args.output,\n              horizon=args.horizon, color=args.color), indent=2))\n        return 0\n\n    if args.cmd == "review-export":\n        from .review import export_sheet\n        print(export_sheet(args.runs, args.out, portable=args.portable))\n        print("Blank ratings only. No scores inferred; missing image files are not reviewable.")\n        return 0\n    if args.cmd == "review-check":\n        from .review import load_scores\n        records = load_scores(args.sheet)\n        print(f"Validated {len(records)} score records against image bytes. Rater labels are self-reported, not authenticated.")\n        print("Acceptance NOT_EVALUATED. Full source/template coverage and specialist comparison remain separate.")\n        return 0\n    if args.cmd == "mh-balance":\n        from .backends.magichour import MagicHourBackend\n        info = MagicHourBackend().balance()\n        if args.json:\n            print(json.dumps(info, indent=2))\n        else:\n            print(f"credits: {info.get(\'credits\')} (tier: {info.get(\'tier\')})")\n        return 0\n\n    if args.cmd == "mh-recover":\n        from .backends.magichour import MagicHourBackend\n        backend = MagicHourBackend()\n        image = backend.resume(args.project_id)\n        args.output.parent.mkdir(parents=True, exist_ok=True)\n        image.save(args.output)\n        args.output.with_suffix(".json").write_text(json.dumps(backend.last_result, indent=2))\n        print(f"Recovered existing project {args.project_id} -> {args.output}; no generation submitted")\n        return 0\n    if args.cmd == "native-reframe":\n        from .native_reframe import native_reframe\n        image, diagnostic = native_reframe(args.cache, fill=args.fill)\n        args.output.parent.mkdir(parents=True, exist_ok=True)\n        image.save(args.output)\n        args.output.with_suffix(".json").write_text(json.dumps(diagnostic, indent=2))\n        print(json.dumps(diagnostic))\n        return 0\n    if args.cmd == "scene-replay":\n        from .compose import replay_scene\n        overrides = {}\n        if args.shadow:\n            overrides["shadow_style"] = None if args.shadow == "none" else args.shadow\n        if args.lift:\n            overrides["scene_auto_lift"] = True\n        image, diagnostic = replay_scene(args.cache, **overrides)\n        args.output.parent.mkdir(parents=True, exist_ok=True)\n        image.save(args.output)\n        print(f"CPU replay -> {args.output}: {diagnostic}; visual quality NOT_EVALUATED")\n        return 0\n    if args.cmd == "costs":\n        from . import costs as _c\n        rate = _c.effective_usd_per_credit(args.plan)\n        per = _c.projected_usd_per_output(args.model, args.plan,\n                                          images_per_generation=args.images,\n                                          credits_per_generation=args.credits)\n        print(f"PROJECTION (published list data, no spend): {args.model} under {args.plan}")\n        print(f"  included-credit rate: ${rate:.6f}/credit (bulk average, not marginal)")\n        print(f"  projected cost per output: ${per:.6f}")\n        print("  gross margin = sale price - this cost; sale price is your input, not assumed")\n        return 0\n    if args.cmd == "mh-compare":\n        from .mhcompare import plan_compare, run_compare\n        models = [m.strip() for m in args.models.split(",") if m.strip()]\n        if not args.live:\n            plan = plan_compare(models, plan=args.plan)\n            print("PLAN ONLY - no credits spent (pass --live to execute; needs owner approval)")\n            for r in plan["models"]:\n                print(f"  {r[\'model\']:<18} {r[\'credits_listed_from\']:>4} credits "\n                      f"(tier {r[\'tier_required\']:<7}) ~${r[\'projected_usd_per_image\']:.4f}/img")\n            print(f"  total: {plan[\'total_credits_listed\']} credits, "\n                  f"~${plan[\'total_projected_usd\']:.4f} (bulk included-credit rate, projection)")\n            return 0\n        from .templates import get_template as _get_template\n        from .schemas import Aspect as _A\n        if args.source is None:\n            p.error("--source is required for --live")\n        manifest = run_compare(args.source, models, _get_template(args.template),\n                               args.seed, _A(args.aspect), args.out, live=True)\n        print(f"LIVE comparison complete: {len(manifest[\'results\'])} models -> {args.out}")\n        for r in manifest["results"]:\n            print(f"  {r[\'model\']:<18} {r[\'latency_s_measured\']:>7.1f}s  {r[\'output\']}")\n        return 0\n    cfg = RunConfig(seed=getattr(args, "seed", 0),\n                    variants=getattr(args, "variants", 2),\n                    out_root=args.out,\n                    device=DeviceConfig.resolve(prefer_gpu=getattr(args, "gpu", False)))\n\n    if args.cmd == "run":\n        editor = _editor(args, cfg)\n        templates = all_templates() if args.templates == "all" else \\\n            [get_template(k) for k in args.templates.split(",")]\n        routes = _routes(args.routes)\n        scene_fn, _ic = None, None\n        if Route.SCENE_COMPOSE in routes:\n            scene_fn, _ic = _scene_fn(args)\n        if any(r in (Route.PURE_PROMPT, Route.HYBRID) for r in routes) and editor is None:\n            raise SystemExit("pure_prompt/hybrid need --editor stub|qwen|magichour")\n        from .source_contract import SourceContract\n        contract = SourceContract(args.source_pose,args.source_material,args.source_framing)\n        for tpl in templates:\n            req = EditRequest(source_path=args.source, template=tpl,\n                              custom_instruction=args.instruction, seed=args.seed,\n                              reference_path=args.reference, mask_path=args.mask)\n            result = run_pipeline(req, routes, cfg, editor=editor,\n                                  relight_enabled=not args.no_relight, scene_fn=scene_fn, glass_mode=args.glass_mode, scene_finish=args.scene_finish, source_contract=contract, require_complete=args.require_complete, allow_colour_conversion=args.allow_colour_conversion, use_embedded_alpha=args.use_embedded_alpha)\n            print(f"[{tpl.key}] {len(result.variants)} variants -> {result.run_dir}")\n            for v in result.variants:\n                flag = " FLAGS" if v.qa.critical_errors else ""\n                print(f"  {v.route.value} seed{v.seed}: fidelity "\n                      f"{_fid(v.qa.score_fidelity)}{flag} -> {v.output_path}")\n        if _ic is not None:\n            _ic.close()\n        return 0\n\n    if args.cmd == "benchmark":\n        editor = _editor(args, cfg)\n        scene_fn, _ic, kind = None, None, ""\n        if "scene_compose" in json.loads(args.manifest.read_text()).get("routes", []):\n            scene_fn, _ic = _scene_fn(args)\n            kind = args.scene\n        csv_path = run_benchmark(args.manifest, cfg, editor=editor, scene_fn=scene_fn, scene_kind=kind, allow_colour_conversion=args.allow_colour_conversion)\n        if _ic is not None:\n            _ic.close()\n        print(f"benchmark scores -> {csv_path}")\n        return 0\n\n    if args.cmd == "gallery":\n        out = build_gallery(args.runs, args.out)\n        print(f"gallery -> {out}")\n        return 0\n\n    if args.cmd == "summarize":\n        from .benchmark import summarize_runs\n        csv_path = summarize_runs(args.runs, args.out)\n        print(f"gate summary -> {csv_path}")\n        return 0\n    return 1\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n\n\n'
# ---- embedded module: renderello/cli.py ----
path = base / 'renderello/cli.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Portable notebook helpers. Free preview by default; live work needs explicit scope.\nNo default key, account, credit allowance or approval is supplied by this module.\n"""\nfrom __future__ import annotations\nimport hashlib\nimport json\nimport time\nimport uuid\nfrom pathlib import Path\nfrom urllib.parse import quote\nfrom PIL import Image\nfrom .credit_ledger import CreditLedger\nfrom .supervised_mh_pilot import BASE, QUOTES_1K, build_request, submit_once\n\n\ndef validate_choices(template, instruction, alternatives):\n    from .templates import get_template\n    from .prompts import validate_instruction\n    spec = get_template(template)\n    validate_instruction(instruction)\n    if type(alternatives) is not int or alternatives not in (2, 3, 4):\n        raise ValueError(\'Choose 2, 3 or 4 alternatives\')\n    return spec\n\n\ndef preview(source, root, *, template=\'minimal_tabletop\', instruction=\'\',\n            alternatives=2, material=\'unknown\'):\n    """CPU GrabCut composition, not generative AI or production-quality proof."""\n    spec = validate_choices(template, instruction, alternatives)\n    if material == \'transparent\':\n        raise ValueError(\'Free opaque preview is not a transparency workflow. Use reviewed optical inputs or a separately approved editor test.\')\n    from .source_contract import SourceContract\n    from .cutout import cutout\n    from .config import RunConfig\n    from .schemas import EditRequest, Route\n    from .pipeline import run_pipeline\n    from .gallery import build_gallery\n    source, root = Path(source), Path(root)\n    root.mkdir(parents=True, exist_ok=False)\n    import shutil\n    stored_source = root / (\'source-original\' + source.suffix.lower())\n    shutil.copyfile(source, stored_source)\n    source = stored_source\n    original = Image.open(source).convert(\'RGB\')\n    # Explicit backend prevents incidental model downloads on a Colab runtime.\n    mask = cutout(original, root / \'mask\', backend=\'grabcut\', device=\'cpu\')\n    result = run_pipeline(EditRequest(source, spec, custom_instruction=instruction,\n        mask_path=mask.matte_path), [Route.CUTOUT_COMPOSE],\n        RunConfig(variants=alternatives, out_root=root / \'runs\'),\n        relight_enabled=False, source_contract=SourceContract(material=material))\n    gallery = build_gallery([result.run_dir], root / \'gallery.html\')\n    return {\'root\': str(root), \'gallery\': str(gallery),\n        \'outputs\': [str(v.output_path) for v in result.variants if v.output_path.exists()],\n        \'manifest\': str(result.manifest_path),\n        \'kind\': \'FREE_CPU_PROCEDURAL_PREVIEW_NOT_AI_EDITOR_QUALITY\'}\n\n\ndef submit_live(source, root, *, session, key, expected_account, cap,\n                authority_reference, source_upload_approved=False,\n                spending_approved=False, quote_confirmed=False,\n                template=\'minimal_tabletop\', instruction=\'\', alternatives=2,\n                model=\'nano-banana-2.1\'):\n    """One new batch only. Calls cannot retry an existing folder.\n\n    These booleans record a caller declaration, not verified consent. A notebook\n    must be operated by the account owner or under independently checked scope.\n    Quotes are the 11 Nov 2026 1k-square snapshot, not a live pricing service.\n    """\n    spec = validate_choices(template, instruction, alternatives)\n    if not all(x is True for x in (source_upload_approved, spending_approved, quote_confirmed)):\n        raise ValueError(\'Source upload, spending and current quote confirmations are all required\')\n    if model not in QUOTES_1K or not expected_account or not key:\n        raise ValueError(\'Supported model, intended account and runtime key required\')\n    if type(cap) is not int or cap < alternatives * QUOTES_1K[model]:\n        raise ValueError(\'Explicit batch credit limit is below the quoted cost\')\n    if not authority_reference or not authority_reference.strip():\n        raise ValueError(\'Record the scope of owner approval; this field grants no authority\')\n    from .schemas import EditRequest, Aspect\n    from .prompts import build_prompt\n    source, root = Path(source), Path(root)\n    # Verify source and account before uploading anything.\n    image = Image.open(source); image.verify()\n    headers = {\'Authorization\': \'Bearer \' + key}\n    response = session.get(BASE + \'/v1/account\', headers=headers, timeout=30)\n    response.raise_for_status()\n    account = response.json()\n    if account.get(\'id\') != expected_account or type(account.get(\'credits\')) is not int:\n        raise ValueError(\'Account identity or credit balance does not match\')\n    if account[\'credits\'] < alternatives * QUOTES_1K[model]:\n        raise ValueError(\'Account balance below batch quote\')\n    root.mkdir(parents=True, exist_ok=False)\n    data = source.read_bytes()\n    (root / (\'source-original\' + source.suffix.lower())).write_bytes(data)  # bytes preserved; actual type retained by PIL\n    ledger = CreditLedger(root / \'credit-ledger\', cap=cap, authority_reference=authority_reference)\n    prompt = build_prompt(EditRequest(source, spec, custom_instruction=instruction, aspect=Aspect.SQUARE))\n    plan = {\'source_sha256\': hashlib.sha256(data).hexdigest(), \'template\': template,\n        \'prompt\': prompt, \'alternatives\': alternatives, \'model\': model,\n        \'quote_per_output\': QUOTES_1K[model], \'quote_total\': alternatives * QUOTES_1K[model],\n        \'quote_date\': \'2026-11-11\', \'resolution\': \'1k\', \'aspect\': \'1:1\',\n        \'approval_reference\': authority_reference, \'submitted_at\': time.time(),\n        \'caller_confirmations_are_not_authentication\': True}\n    (root / \'plan.json\').write_text(json.dumps(plan, indent=2))\n    # Normalize uploaded encoding explicitly while preserving saved original bytes.\n    import io\n    buf = io.BytesIO(); Image.open(source).convert(\'RGB\').save(buf, \'PNG\')\n    up = session.post(BASE + \'/v1/files/upload-urls\', headers=headers,\n        json={\'items\': [{\'type\': \'image\', \'extension\': \'png\'}]}, timeout=30)\n    up.raise_for_status(); item = up.json()[\'items\'][0]\n    put = session.put(item[\'upload_url\'], data=buf.getvalue(),\n        headers={\'Content-Type\': \'image/png\'}, timeout=60)\n    put.raise_for_status()\n    for i in range(alternatives):\n        job = \'alternative-\' + str(i + 1)\n        request = build_request(job, model, prompt, item[\'file_path\'])\n        submit_once(session, key, request, ledger, job, root / \'jobs\', expected_account)\n    return {\'root\': str(root), \'ledger\': ledger.snapshot(), \'status\': \'SUBMITTED_NOT_COMPLETE\'}\n\n\ndef recover_live(root, *, session, key, minimum_interval=3600):\n    """Read existing IDs once. Never create an image; default hourly checks."""\n    root = Path(root)\n    stamp = root / \'last-check.json\'\n    if stamp.exists() and time.time() - json.loads(stamp.read_text())[\'unix\'] < minimum_interval:\n        raise ValueError(\'Wait until the hourly check is due; no faster automatic polling\')\n    stamp.write_text(json.dumps({\'unix\': time.time()}))\n    headers = {\'Authorization\': \'Bearer \' + key}\n    results = []\n    for folder in sorted((root / \'jobs\').iterdir()):\n        if not folder.is_dir(): continue\n        if not (folder / \'response.json\').exists():\n            raise ValueError(\'Submission outcome unknown. Inspect account/provider; never resubmit blindly\')\n        project_id = json.loads((folder / \'response.json\').read_text())[\'id\']\n        response = session.get(BASE + \'/v1/image-projects/\' + quote(project_id, safe=\'\'),\n            headers=headers, timeout=30)\n        response.raise_for_status(); detail = response.json()\n        (folder / \'latest-detail.json\').write_text(json.dumps(detail, indent=2))\n        output = folder / \'output.png\'\n        if detail.get(\'status\') == \'complete\' and not output.exists():\n            downloads = detail.get(\'downloads\') or []\n            if not downloads: raise ValueError(\'Complete project has no download; preserve ID\')\n            downloaded = session.get(downloads[0][\'url\'], timeout=90)\n            downloaded.raise_for_status()\n            import io\n            Image.open(io.BytesIO(downloaded.content)).verify()\n            output.write_bytes(downloaded.content)\n        results.append({\'job\': folder.name, \'project_id\': project_id,\n            \'status\': detail.get(\'status\'), \'credits_charged\': detail.get(\'credits_charged\'),\n            \'output\': str(output) if output.exists() else None})\n    return results\n'
# ---- embedded module: renderello/colab_demo.py ----
path = base / 'renderello/colab_demo.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Conservative completion of saved runs, including pre-status manifests.\n\nOnly executor failure markers are failures here. Product quality warnings are\nnot execution failures, and an existing file alone does not prove completion.\n"""\ndef saved_variant_failed(variant):\n    if variant.get(\'success\') is False:\n        return True\n    qa = variant.get(\'qa\') or {}\n    return any(str(error).startswith((\'variant failed:\', \'cutout failed:\'))\n               for error in (qa.get(\'critical_errors\') or []))\n'
# ---- embedded module: renderello/completion.py ----
path = base / 'renderello/completion.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Deterministic compositing (Route 2 core, CPU-fast, no models):\n\n  procedural background from the template\'s background_hint\n  -> product placement (scale to template framing, canvas from aspect)\n  -> synthetic contact shadow matched to the template\'s shadow spec\n  -> alpha blend; product pixels are pasted, never resampled twice, so the\n     untransformed interior stays bit-identical to the source cutout.\n\nThe generative routes never pass through here; this module is the fidelity-safe\npath the benchmark uses to separate \'model quality\' from \'pipeline correctness\'.\n"""\nfrom __future__ import annotations\n\nimport hashlib\n\nimport cv2\nimport numpy as np\nfrom PIL import Image\n\nfrom .schemas import Aspect, CutoutResult, TemplateSpec\n\n\ndef _rng_for(seed: int, *parts: str) -> np.random.Generator:\n    h = hashlib.sha256((str(seed) + "|" + "|".join(parts)).encode()).digest()\n    return np.random.default_rng(int.from_bytes(h[:8], "little"))\n\n\ndef _parse_hint_color(hint: str) -> tuple[int, int, int]:\n    """Map the template\'s background_hint language to a base color. Deliberately\n    small vocabulary - the hint text is ours, so keep the mapping explicit."""\n    table = [\n        ("white", (250, 250, 250)), ("light gray", (225, 225, 226)),\n        ("gray", (200, 200, 202)), ("beige", (232, 222, 208)),\n        ("pink", (240, 226, 224)), ("cream", (243, 234, 220)),\n        ("green", (168, 190, 158)), ("amber", (176, 124, 62)),\n        ("gold", (190, 150, 80)), ("black", (22, 22, 24)),\n        ("near-black", (30, 30, 33)), ("dark", (44, 44, 48)),\n        ("saturated", (210, 60, 70)),\n    ]\n    low = hint.lower()\n    for key, rgb in table:\n        if key in low:\n            return rgb\n    return (235, 235, 236)\n\n\ndef placement_params(seed: int, template_key: str) -> tuple[float, float, float]:\n    """Per-variant composition knobs (fill, x-jitter, baseline). This is what\n    makes 2-4 alternatives genuinely distinct compositions instead of re-noised\n    copies: only position/scale/background vary, with no generative repaint.\n    Downscaling can change product RGB; native exactness needs verification. Centralized so compositing and QA alignment can never drift\n    apart silently."""\n    rng = _rng_for(seed, template_key, "compose")\n    return (float(rng.uniform(0.60, 0.78)),    # fill fraction of canvas height\n            float(rng.uniform(-0.09, 0.09)),   # x jitter, fraction of canvas width\n            float(rng.uniform(0.84, 0.93)))    # baseline, fraction of canvas height\n\n\ndef procedural_background(size: tuple[int, int], template: TemplateSpec,\n                          seed: int) -> np.ndarray:\n    """Gradient background (seed-varying angle) with a soft radial halo behind\n    the product anchor and optional diagonal light band (\'social_ad\' style).\n    Deterministic per (seed, template)."""\n    w, h = size\n    base = np.array(_parse_hint_color(template.background_hint), np.float32)\n    rng = _rng_for(seed, template.key, "halo")\n    X, Y = np.meshgrid(np.arange(w, dtype=np.float32),\n                       np.arange(h, dtype=np.float32))\n    angle = float(rng.uniform(0.0, np.pi))\n    ca, sa = abs(np.cos(angle)), abs(np.sin(angle))\n    proj = (X * np.cos(angle) + Y * np.sin(angle)) / (w * ca + h * sa + 1e-6)\n    direction = -1.0 if "dark" in template.background_hint or "black" in template.background_hint else 1.0\n    grad = base[None, None, :] + direction * (proj[..., None] - 0.5) * 36.0\n    img = grad.astype(np.float32).copy()\n\n    # Radial halo near the product anchor; center jitters per seed.\n    cx = w * (0.5 + float(rng.uniform(-0.06, 0.06)))\n    cy = h * (0.42 + float(rng.uniform(-0.05, 0.05)))\n    r = np.sqrt((X - cx) ** 2 + (Y - cy) ** 2) / (0.6 * max(w, h))\n    halo_strength = 14.0 if "glow" in template.background_hint or "halo" in template.background_hint else 7.0\n    img += (1.0 - np.clip(r, 0, 1))[..., None] * halo_strength\n\n    if "diagonal" in template.background_hint:\n        band = np.clip(1.0 - np.abs((X / w) - (Y / h) - 0.15) * 3.0, 0, 1)\n        img += band[..., None] * 26.0\n    # Fine grain keeps gradients from banding after JPEG export.\n    img += rng.normal(0, 0.9, (h, w, 1)).astype(np.float32)\n    return np.clip(img, 0, 255).astype(np.uint8)\n\n\ndef fit_product(cutout_rgba: np.ndarray, bbox: tuple[int, int, int, int],\n                canvas: tuple[int, int], fill: float = 0.7, jx: float = 0.0,\n                baseline: float = 0.88) -> tuple[np.ndarray, tuple[int, int]]:\n    """Crop to the matte bbox and scale so the product height is `fill` of the\n    canvas height (template camera language says ~70%). Never upscales beyond\n    source resolution. jx shifts placement horizontally (fraction of canvas\n    width); baseline sets where the product\'s foot sits (fraction of canvas\n    height). Returns placed-crop RGBA and its (x, y) offset."""\n    x, y, w, h = bbox\n    crop = cutout_rgba[y:y + h, x:x + w]\n    cw, ch = canvas\n    scale = min((ch * fill) / h, (cw * 0.9) / w, 1.0)\n    nw, nh = max(1, round(w * scale)), max(1, round(h * scale))\n    if (nw, nh) != (w, h):\n        # fit_product never upscales. Average premultiplied colour so RGB\n        # hidden by zero alpha cannot leak into visible antialiased edges.\n        # Keep opaque resizes on the original byte-for-byte OpenCV path.\n        if np.all(crop[..., 3] == 255):\n            crop = cv2.resize(crop, (nw, nh), interpolation=cv2.INTER_AREA)\n        else:\n            alpha = crop[..., 3].astype(np.float64) / 255.0\n            premultiplied = crop[..., :3].astype(np.float64) * alpha[..., None]\n            resized_alpha = cv2.resize(alpha, (nw, nh), interpolation=cv2.INTER_AREA)\n            resized_rgb = cv2.resize(premultiplied, (nw, nh), interpolation=cv2.INTER_AREA)\n            straight = np.zeros_like(resized_rgb)\n            np.divide(resized_rgb, resized_alpha[..., None], out=straight,\n                      where=resized_alpha[..., None] > 0)\n            legacy = cv2.resize(crop, (nw, nh), interpolation=cv2.INTER_AREA)\n            rgb_bytes = np.clip(np.rint(straight), 0, 255).astype(np.uint8)\n            # Where every contributor is opaque, preserve the existing byte\n            # resampler exactly, including its integer rounding convention.\n            covered = resized_alpha >= 1.0 - 1e-10\n            rgb_bytes[covered] = legacy[..., :3][covered]\n            crop = np.dstack((rgb_bytes, legacy[..., 3]))\n    px = max(0, min((cw - nw) // 2 + int(jx * cw), cw - nw))\n    py = max(0, min(int(ch * baseline) - nh, ch - nh))\n    return crop, (px, py)\n\n\ndef snap_to_canvas_edges(alpha_full: np.ndarray, placement: tuple[int, int],\n                         prod_size: tuple[int, int], canvas: tuple[int, int]) -> tuple[int, int]:\n    """If the source matte touches the photo frame on a side, that side of the\n    product is a straight cut (the photo\'s own edge). Float it mid-canvas and it\n    reads as a rectangle (v0.4.9 watch1). Snap each such side to the matching\n    canvas border so the cut reads as an intentional crop. Pixels are untouched."""\n    from .boundary import frame_edge_contact\n    px, py = placement; pw, ph = prod_size; cw, ch = canvas\n    t = frame_edge_contact(alpha_full)["touches"]\n    if "left" in t and "right" not in t:\n        px = 0\n    elif "right" in t and "left" not in t:\n        px = cw - pw\n    if "top" in t and "bottom" not in t:\n        py = 0\n    elif "bottom" in t and "top" not in t:\n        py = ch - ph\n    return px, py\n\n\ndef cover_fit_if_cut_both_sides(rgba_full: np.ndarray, canvas: tuple[int, int]):\n    """When the matte touches BOTH opposite frame edges on an axis (watch1: strap\n    runs off two corners), no placement can hide the straight cuts. Scale the whole\n    source frame to cover the canvas (center crop) so the cuts fall off-canvas.\n    Returns a canvas-sized RGBA or None when this case does not apply. May upscale\n    the source (reported by the caller via the crop size, never silent in diag)."""\n    from .boundary import frame_edge_contact\n    t = frame_edge_contact(rgba_full[..., 3])["touches"]\n    if not (("left" in t and "right" in t) or ("top" in t and "bottom" in t)):\n        return None\n    cw, ch = canvas\n    H, W = rgba_full.shape[:2]\n    sc = max(cw / W, ch / H)\n    nw, nh = max(cw, round(W * sc)), max(ch, round(H * sc))\n    interp = cv2.INTER_AREA if sc < 1 else cv2.INTER_LANCZOS4\n    # This branch used to filter straight RGBA, allowing hidden background RGB\n    # to contaminate visible cover-fit edges, especially with Lanczos upscaling.\n    from .alpha_resample import resize_rgba\n    big = resize_rgba(rgba_full, (nw, nh), interpolation=interp)\n    x0, y0 = (nw - cw) // 2, (nh - ch) // 2\n    return big[y0:y0 + ch, x0:x0 + cw].copy()\n\n\ndef anchored_contact_shadow(canvas, placement, matte):\n    """Opt-in tight support-plane shadow per connected product component.\n\n    Uses each component\'s actual lowest alpha support, never the whole image or\n    a scene-inferred surface. This is a 2D contact cue, not a physical floor or\n    pose detector. Eye review remains necessary, especially for hanging objects.\n    """\n    cw,ch=canvas;px,py=placement\n    if matte.ndim!=2:raise ValueError(\'matte must be2D\')\n    h,w=matte.shape\n    if not (0<=px<=cw-w and 0<=py<=ch-h):raise ValueError(\'matte outside canvas\')\n    layer=np.zeros((ch,cw),np.float32)\n    n,labels,stats,_=cv2.connectedComponentsWithStats((matte>127).astype(np.uint8),8)\n    for label in range(1,n):\n        if stats[label,cv2.CC_STAT_AREA]<max(3,matte.size//10000):continue\n        ys,xs=np.where(labels==label);bottom=int(ys.max())\n        # Only the lowest narrow band defines contact, not the full lower sixth.\n        support=xs[ys>=bottom-max(1,int(stats[label,cv2.CC_STAT_HEIGHT]*.015))]\n        left,right=int(support.min()),int(support.max())\n        center=px+(left+right)/2;radius=max(3,(right-left+1)/2)\n        yy,xx=np.ogrid[:ch,:cw]\n        sigma_x=max(3,radius*.65);sigma_y=max(1.5,min(4,h*.009))\n        contact=np.exp(-.5*(((xx-center)/sigma_x)**2+((yy-(py+bottom+1))/sigma_y)**2))\n        layer=np.maximum(layer,contact*.50)\n    return (np.clip(layer,0,1)*255).astype(np.uint8)\n\n\ndef base_footprint_shadow(canvas, placement, matte):\n    """Opt-in soft local support cue using per-column lowest opaque alpha.\n\n    This is hand-selected 2D finishing for known upright floor-contact sources,\n    not inferred scene depth, light direction or physical cast-shadow recovery.\n    """\n    cw,ch=canvas;px,py=placement\n    if matte.ndim!=2:raise ValueError(\'matte must be2D\')\n    h,w=matte.shape\n    if not (0<=px<=cw-w and 0<=py<=ch-h):raise ValueError(\'matte outside canvas\')\n    visible=matte>127;layer=np.zeros((ch,cw),np.float32)\n    yy,xx=np.where(visible)\n    if not len(xx):return layer.astype(np.uint8)\n    floor=int(yy.max());band=max(2,round(h*.035))\n    # Actual bottom contours support indented feet without turning a whole\n    # lower-product rectangle into a detached block shadow.\n    for x in range(w):\n        ys=np.flatnonzero(visible[:,x])\n        if len(ys) and ys[-1]>=floor-band:\n            y=py+int(ys[-1])+1\n            if y<ch:layer[y,px+x]=1.\n    sigma_x=max(1.,min(5.,w*.015));sigma_y=max(1.,min(5.,h*.006))\n    layer=cv2.GaussianBlur(layer,(0,0),sigmaX=sigma_x,sigmaY=sigma_y)\n    peak=float(layer.max())\n    if peak>0:layer*=.25/peak\n    return np.clip(layer*255,0,64).astype(np.uint8)\n\n\ndef contact_shadow(canvas: tuple[int, int], placement: tuple[int, int],\n                   prod_size: tuple[int, int], matte: np.ndarray,\n                   style: str) -> np.ndarray:\n    """Shadow layer (grayscale alpha). Style words from the template shadow spec:\n    \'soft/faint/diffused/long\' -> large blur; \'crisp/hard/graphic\' -> tight blur\n    + offset; \'reflection\' handled separately by the caller."""\n    if style == "base footprint soft":\n        return base_footprint_shadow(canvas,placement,matte)\n    if "anchored" in style:\n        return anchored_contact_shadow(canvas,placement,matte)\n    cw, ch = canvas\n    px, py = placement\n    pw, ph = prod_size\n    layer = np.zeros((ch, cw), np.float32)\n    # Anchor on the actual matte extent; supports transparent padding supplied\n    # by callers. fit_product itself does NOT add padding.\n    rows = np.nonzero((matte > 16).sum(axis=1))[0] if len(matte) else []\n    if len(rows):\n        y1 = int(rows[-1])\n        foot_h = min(y1 + 1, max(2, (y1 - int(rows[0]) + 1) // 6))\n        foot = matte[y1 - foot_h + 1:y1 + 1, :]\n        fh, fw = foot.shape\n        region = layer[py + y1 - fh + 1:py + y1 + 1, px:px + pw]\n        if region.shape == foot.shape:\n            region[:] = np.maximum(region, foot.astype(np.float32) / 255.0)\n    hard = any(k in style for k in ("crisp", "hard", "graphic"))\n    blur = 7 if hard else 31\n    opacity, (dx, dy) = (0.55, (18, 10)) if hard else (0.30, (4, 8))\n    if "long" in style:\n        dx, dy = 40, 6\n    layer = cv2.GaussianBlur(layer, (blur | 1, blur | 1), 0)\n    # Translate without wraparound (np.roll made shadows reappear at the\n    # opposite canvas edge for frame-touching products).\n    layer = cv2.warpAffine(layer, np.float32([[1, 0, dx], [0, 1, dy]]),\n                           (cw, ch), borderMode=cv2.BORDER_CONSTANT, borderValue=0)\n    return (np.clip(layer, 0, 1) * opacity * 255).astype(np.uint8)\n\n\ndef cover_to_canvas(img_rgb: np.ndarray, canvas: tuple[int, int]) -> np.ndarray:\n    """Scale an RGB image to cover the canvas and center-crop the overflow.\n    Used to fit an owner-supplied reference/backdrop photo to the working aspect."""\n    cw, ch = canvas\n    H, W = img_rgb.shape[:2]\n    sc = max(cw / W, ch / H)\n    nw, nh = max(cw, round(W * sc)), max(ch, round(H * sc))\n    interp = cv2.INTER_AREA if sc < 1 else cv2.INTER_LANCZOS4\n    big = cv2.resize(img_rgb, (nw, nh), interpolation=interp)\n    x0, y0 = (nw - cw) // 2, (nh - ch) // 2\n    return big[y0:y0 + ch, x0:x0 + cw].copy()\n\n\ndef composite(cutout: CutoutResult, template: TemplateSpec, aspect: Aspect,\n              seed: int, reflection: bool = False,\n              bg_transform=None, reference: np.ndarray | None = None) -> Image.Image:\n    """Full Route 2 composite. Returns RGB canvas image. bg_transform, when\n    given, grades the BACKGROUND ONLY (e.g. relight fallback) before the\n    product is pasted. Native product RGB is used before optional downscaling;\n    output exactness is not guaranteed by this routine alone. When\n    reference (HxWx3 uint8) is given, it replaces the procedural background:\n    the reference photo is cover-fitted to the canvas (center crop)."""\n    cw, ch = aspect.size\n    rgba = np.array(Image.open(cutout.rgba_path).convert("RGBA"))\n    matte = np.array(Image.open(cutout.matte_path).convert("L"))\n    fill, jx, baseline = placement_params(seed, template.key)\n    crop, (px, py) = fit_product(rgba, cutout.bbox, (cw, ch),\n                                 fill=fill, jx=jx, baseline=baseline)\n    ph, pw = crop.shape[:2]\n    scaled_matte = cv2.resize(matte[cutout.bbox[1]:cutout.bbox[1] + cutout.bbox[3],\n                                    cutout.bbox[0]:cutout.bbox[0] + cutout.bbox[2]],\n                              (pw, ph), interpolation=cv2.INTER_AREA)\n\n    if reference is not None:\n        bg = cover_to_canvas(reference, (cw, ch)).astype(np.float32)\n    else:\n        bg = procedural_background((cw, ch), template, seed).astype(np.float32)\n    # Shadow under the product, matched to the template\'s shadow language.\n    shadow = contact_shadow((cw, ch), (px, py), (pw, ph), scaled_matte,\n                            template.shadow).astype(np.float32) / 255.0\n    bg *= (1.0 - shadow[..., None] * 0.85)\n\n    # Optional mirror reflection (reflective_surface template).\n    if reflection:\n        refl = cv2.flip(crop, 0)\n        ra = (refl[..., 3:4].astype(np.float32) / 255.0) * 0.35\n        fade = np.linspace(0.5, 0.0, ph, dtype=np.float32)[:, None, None]\n        ra *= fade\n        ry = py + ph\n        # A reflection may extend beyond the canvas. Draw its visible prefix\n        # rather than dropping the whole reflection when the floor is short.\n        # Keep the full-product fade so clipping does not change its falloff.\n        visible_h = min(ph, ch - ry)\n        if visible_h > 0:\n            region = bg[ry:ry + visible_h, px:px + pw]\n            visible_alpha = ra[:visible_h]\n            region[:] = (region * (1 - visible_alpha)\n                         + refl[:visible_h, :, :3].astype(np.float32) * visible_alpha)\n\n    if bg_transform is not None:\n        bg_img = bg_transform(Image.fromarray(np.clip(bg, 0, 255).astype(np.uint8)))\n        bg = np.array(bg_img.convert("RGB")).astype(np.float32)\n\n    # Final paste: straight alpha over, product interior untouched.\n    a = (crop[..., 3:4].astype(np.float32) / 255.0)\n    region = bg[py:py + ph, px:px + pw]\n    region[:] = region * (1 - a) + crop[..., :3].astype(np.float32) * a\n    # No global sharpening. fit_product may already downscale for framing;\n    # untransformed opaque pixels can be verified separately. Background grain\n    # is applied before the final paste.\n    return Image.fromarray(np.clip(bg, 0, 255).astype(np.uint8), "RGB")\n\n\n\nSCENE_NATIVE_ASPECT = 512 / 640  # IC-Light working canvas (w/h); relight() center-crops to this\n\n\ndef aspect_safe_scene_call(scene_fn, gray: np.ndarray, prompt: str, native: float = SCENE_NATIVE_ASPECT, pad_value: int = 127):\n    """Call scene_fn on a canvas whose aspect matches the model\'s native aspect.\n\n    ICLight.relight() center-crops its input to 512x640. On a non-4:5 canvas\n    (landscape/square) that crop shifts and rescales the product, so the\n    generated product no longer sits under the pasted one (ghost product\n    behind the paste). Fix: pad the gray-127 canvas to the native aspect\n    (product untouched, still at the same pixel scale relative to the pad),\n    run the model, then crop the original region back out. Canvases already at\n    the native aspect are passed through unchanged.\n    """\n    h, w = gray.shape[:2]\n    if abs(w / h - native) < 0.01:\n        return np.asarray(scene_fn(gray, prompt))\n    if w / h > native:      # too wide: pad top/bottom\n        H, W = int(round(w / native)), w\n    else:                   # too tall: pad left/right\n        H, W = h, int(round(h * native))\n    y0, x0 = (H - h) // 2, (W - w) // 2\n    padded = np.full((H, W, 3), pad_value, np.uint8)\n    padded[y0:y0 + h, x0:x0 + w] = gray\n    out = np.asarray(scene_fn(padded, prompt))\n    if out.shape[:2] != (H, W):\n        out = cv2.resize(out, (W, H), interpolation=cv2.INTER_LANCZOS4)\n    return out[y0:y0 + h, x0:x0 + w]\n\n\ndef support_plane_canvas(shape, support_row):\n    """Experimental empty horizontal plane conditioning. No source RGB or shape.\n\n    A single monochrome horizon is18% of height behind the desired foot row.\n    This is a layout hypothesis, never evidence of a generated usable surface.\n    """\n    h,w=shape[:2]\n    if not 0<=support_row<h:raise ValueError(\'support row outside canvas\')\n    horizon=max(0,min(h-1,int(round(support_row-.18*h))))\n    image=np.full((h,w,3),127,np.uint8)\n    image[horizon:]=176\n    return image,horizon\n\n\ndef composite_with_scene(cutout: CutoutResult, template: TemplateSpec, aspect: Aspect,\n                         seed: int, scene_fn, prompt: str | None = None,\n                         shadow_style: str | None = None,\n                         scene_grade: float = 0.0,\n                         snap_frame_cut: bool = False,\n                         glass_backdrop=None,\n                         scene_auto_lift: bool = False, cache_dir=None,\n                         scene_is_aligned: bool = False,\n                         scene_accept=None, scene_max_attempts: int = 1,\n                         conditioning: str = "product_gray"):\n    """Route 2b: scene generated by a relight/scene model (IC-Light) around the\n    placed product, then SOURCE-EXACT product pixels pasted back on top.\n\n    scene_fn(canvas_rgb_gray127: HxWx3 uint8, prompt: str) -> HxWx3 uint8 of the\n    same size as the canvas (use iclight_gpu.ICLight.relight + a resize).\n    Returns (image, diagnostics). The generated product silhouette/shadow can\n    survive around the pasted product, so diagnostics reports a ring statistic\n    for EYE REVIEW; it is a screen, not proof the result is clean.\n\n    conditioning selects what the model actually receives:\n    - "product_gray" (default, legacy): the gray-127 canvas carrying the\n      product\'s RGB-modulated silhouette. The model can see the product shape\n      and may render a second one around it (the 20 Nov 2026 duplicate fail).\n    - "empty_guide": a product-free two-tone floor/wall guide from\n      support_plane_canvas, the horizon pinned at the product\'s foot row.\n      The model never receives any product pixel or silhouette; the caller\n      MUST pass an environment-only prompt (renderello.empty_scene_prompt).\n      This restores the measured-clean 13 Nov 2026 conditioning.\n    """\n    if conditioning not in ("product_gray", "empty_guide"):\n        raise ValueError(f"unknown conditioning {conditioning!r}")\n    cw, ch = aspect.size\n    rgba = np.array(Image.open(cutout.rgba_path).convert("RGBA"))\n    fill, jx, baseline = placement_params(seed, template.key)\n    crop, (px, py) = fit_product(rgba, cutout.bbox, (cw, ch),\n                                 fill=fill, jx=jx, baseline=baseline)\n    if getattr(scene_fn,\'flatlay_surface\',False):\n        # Flatlay has no floor baseline. Center the actual alpha bounds, not\n        # the original photo frame; keep the existing no-upscale policy.\n        crop,(px,py)=fit_product(rgba,cutout.bbox,(cw,ch),fill=.76,jx=0.,baseline=.88)\n        px,py=flatlay_center(crop[...,3],(cw,ch))\n    ph, pw = crop.shape[:2]\n    cover_used = False\n    if snap_frame_cut:  # opt-in; default placement unchanged\n        cover = cover_fit_if_cut_both_sides(rgba, (cw, ch))\n        if cover is not None:\n            cover_used = True   # product runs off opposite frame edges: keep source framing\n            crop, (px, py) = cover, (0, 0)\n            ph, pw = crop.shape[:2]\n        else:\n            px, py = snap_to_canvas_edges(rgba[..., 3], (px, py), (pw, ph), (cw, ch))\n    a_full = np.zeros((ch, cw), np.float32)\n    a_full[py:py + ph, px:px + pw] = crop[..., 3].astype(np.float32) / 255.0\n    gray = np.full((ch, cw, 3), 127, np.float32)\n    gray[py:py + ph, px:px + pw] = (127 + (crop[..., :3].astype(np.float32) - 127)\n                                   * a_full[py:py + ph, px:px + pw, None])\n    prompt = prompt or f"{template.scene}, {template.lighting}, product photography"\n    canvas = gray.clip(0, 255).astype(np.uint8)\n    plane_horizon=None\n    if conditioning == "empty_guide":\n        # Product-free model input: only the two-tone floor/wall guide, horizon\n        # pinned at the product\'s lowest opaque row so the model renders the\n        # support surface exactly where the product will stand. The prompt\n        # MUST come from empty_scene_prompt.build_empty_scene_prompt: a\n        # hand-written prompt can carry a product description with a\n        # contradictory "no product" tail, which is exactly the measured\n        # 20 Nov 2026 duplicate-product failure shape.\n        from .empty_scene_prompt import EmptyScenePrompt\n        if not isinstance(prompt, EmptyScenePrompt):\n            raise ValueError(\n                "empty_guide conditioning requires an EmptyScenePrompt from "\n                "renderello.empty_scene_prompt.build_empty_scene_prompt")\n        support_rows = np.where((crop[..., 3] > 127).any(axis=1))[0]\n        if not len(support_rows):\n            raise ValueError(\'empty matte for support plane\')\n        canvas, plane_horizon = support_plane_canvas(\n            gray.shape, py + int(support_rows[-1]))\n    elif getattr(scene_fn,\'support_plane\',False):\n        support_rows=np.where((crop[...,3]>127).any(axis=1))[0]\n        if not len(support_rows):raise ValueError(\'empty matte for support plane\')\n        canvas,plane_horizon=support_plane_canvas(gray.shape,py+int(support_rows[-1]))\n        prompt=(f"{template.scene}, {template.lighting}, empty bare level tabletop spanning "\n                "the entire foreground, continuous flat horizontal support surface, "\n                "table surface clearly visible below a low distant horizon, studio set photography")\n\n    flatlay_used = bool(getattr(scene_fn,\'flatlay_surface\',False))\n    if flatlay_used:\n        canvas=np.full_like(canvas,176)\n        prompt=(f"{template.scene}, {template.lighting}, {template.camera}, "\n                "true overhead flat lay photograph, entire image is one continuous "\n                "horizontal textured surface seen directly from above, empty center, "\n                "no wall, no horizon, no product, no raised platform")\n        shadow_style=\'surface contact\'\n\n    if getattr(scene_fn,\'camera_matched\',False):\n        # Empty scene only. Never feed source silhouette/RGB into generation.\n        canvas=np.full_like(canvas,176)\n\n    # Direct backdrop is already fitted to the requested aspect. Only model\n    # callbacks need IC-Light native-aspect padding/crop correction.\n    pad_value = 176 if (flatlay_used or getattr(scene_fn, "camera_matched", False)) else 127\n\n    def _gen_scene():\n        s = (np.asarray(scene_fn(canvas, prompt)) if scene_is_aligned else\n             aspect_safe_scene_call(scene_fn, canvas, prompt, pad_value=pad_value))\n        if s.shape != (ch, cw, 3):\n            s = cv2.resize(s, (cw, ch), interpolation=cv2.INTER_LANCZOS4)\n        return s\n\n    scene = _gen_scene()\n    scene_attempts = 1\n    scene_acceptance = None\n    emptiness = None\n\n    def _screens():\n        """Combined acceptance dict: caller scene_accept plus the pre-paste\n        emptiness screen (empty_guide conditioning only). Either suspect\n        verdict counts as an intrusion so the retry budget regenerates."""\n        nonlocal emptiness\n        combined = {}\n        if scene_accept is not None:\n            combined.update(scene_accept(scene))\n        if conditioning == "empty_guide":\n            from .scene_emptiness import screen_scene_emptiness\n            emptiness = screen_scene_emptiness(scene, (px, py, pw, ph))\n            combined["emptiness"] = emptiness\n            if emptiness["suspect_object_in_scene"]:\n                combined["intrusion"] = True\n        return combined\n\n    if scene_accept is not None or conditioning == "empty_guide":\n        # Heuristic rejection of generated scenes with equipment/frame\n        # intrusions (see scene_acceptance.py) or a suspected object where\n        # the product will be pasted (see scene_emptiness.py). Regeneration\n        # is capped by the CALLER via scene_max_attempts; each attempt may\n        # be a paid generation.\n        scene_acceptance = _screens()\n        while scene_acceptance.get("intrusion") and scene_attempts < max(1, scene_max_attempts):\n            scene = _gen_scene()\n            scene_attempts += 1\n            scene_acceptance = _screens()\n    if cache_dir is not None:\n        # Lossless layers at the exact aligned output coordinates. No need to\n        # run GPU again to try shadows/scene grades or inspect the true alpha.\n        from pathlib import Path\n        import json\n        cd = Path(cache_dir); cd.mkdir(parents=True, exist_ok=True)\n        Image.fromarray(scene.astype(np.uint8)).save(cd / "scene.png")\n        Image.fromarray(crop).save(cd / "product_rgba.png")\n        (cd / "layers.json").write_text(json.dumps({\n            "placement": [px, py], "cover_used": cover_used,\n            "glass_backdrop": None if glass_backdrop is None else np.asarray(glass_backdrop).tolist(),\n            "shadow_style": shadow_style, "scene_grade": scene_grade,\n            "scene_auto_lift": scene_auto_lift}, indent=2))\n    grounded = False\n    if shadow_style and not cover_used and glass_backdrop is None:\n        # Generated scenes carry no real floor contact, so an ungrounded product\n        # reads as floating. Darken the SCENE ONLY with a per-component contact\n        # ellipse at the product\'s actual lowest alpha; the product is pasted\n        # afterwards by finish_scene and stays bit-exact.\n        anchor = anchored_contact_shadow((cw, ch), (px, py), crop[..., 3])\n        scene = scene.astype(np.float32) * (1.0 - anchor.astype(np.float32)[..., None] / 255.0)\n        grounded = True\n    out, scene_lift = finish_scene(scene, crop, (px,py), shadow_style=shadow_style,\n                                  scene_grade=scene_grade, glass_backdrop=glass_backdrop,\n                                  scene_auto_lift=scene_auto_lift, cover_used=cover_used)\n    # Ring diagnostic: generated pixels just outside the pasted product vs the\n    # scene far away. Large difference => likely leftover generated silhouette\n    # or shadow hugging the product (needs eye review).\n    m = (a_full > 0.5).astype(np.uint8)\n    k = max(3, int(round(0.01 * max(cw, ch))))\n    kern = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * k + 1, 2 * k + 1))\n    ring = cv2.dilate(m, kern) - m\n    far = 1 - cv2.dilate(m, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (6 * k + 1, 6 * k + 1)))\n    diag = {"ring_px": int(ring.sum()), "far_px": int(far.sum())}\n    if flatlay_used:\n        diag["scene_mode"] = "flatlay_surface_experimental"\n    if getattr(scene_fn,\'camera_matched\',False):\n        diag[\'scene_mode\']=\'camera_matched_experimental\'\n        diag[\'declared_camera_pose\']=scene_fn.camera_pose\n    if plane_horizon is not None:\n        diag["support_plane_horizon"] = plane_horizon\n        diag["scene_mode"] = ("empty_guide" if conditioning == "empty_guide"\n                              else "support_plane_experimental")\n    diag["conditioning"] = conditioning\n    if glass_backdrop is not None:\n        diag["glass_mode"] = True\n    if ring.sum() and far.sum():\n        rl = scene[ring.astype(bool)].mean()\n        fl = scene[far.astype(bool)].mean()\n        diag["ring_vs_far_luma_diff"] = round(float(abs(rl - fl)), 2)\n        diag["needs_eye_review"] = diag["ring_vs_far_luma_diff"] > 25.0\n    else:\n        diag["needs_eye_review"] = True\n    # A quiet ring only clears that heuristic. It cannot certify camera,\n    # lighting/contact realism, mask accuracy or publication usefulness.\n    diag["ring_alert"] = diag["needs_eye_review"]\n    diag["needs_eye_review"] = True\n    diag["publication_status"] = "NOT_EVALUATED"\n    diag["review_required"] = ["camera compatibility", "contact and light", "mask edges", "composition"]\n    out_u8 = np.clip(out, 0, 255).astype(np.uint8)\n    diag["placement"] = (int(px), int(py), int(pw), int(ph))\n    diag["scene_lift"] = scene_lift\n    diag["shadow_style"] = shadow_style\n    diag["grounding_shadow"] = "anchored_contact" if grounded else None\n    if scene_acceptance is not None:\n        diag["scene_acceptance"] = scene_acceptance\n        diag["scene_attempts"] = scene_attempts\n        if emptiness is not None:\n            diag["emptiness_screen"] = emptiness\n    diag["cover_fit"] = bool(cover_used)\n    if glass_backdrop is None:\n        inner = crop[..., 3] == 255\n        diag["paste_bit_exact"] = bool(np.array_equal(out_u8[py:py + ph, px:px + pw][inner], crop[..., :3][inner]))\n    return Image.fromarray(out_u8, "RGB"), diag\n\n\ndef finish_scene(scene, crop, placement, *, shadow_style=None, scene_grade=0.0,\n                 glass_backdrop=None, scene_auto_lift=False, cover_used=False):\n    """CPU-only finishing shared by live compose and exact cached replay."""\n    ch, cw = scene.shape[:2]\n    ph, pw = crop.shape[:2]\n    px, py = placement\n    a_full = np.zeros((ch, cw), np.float32)\n    a_full[py:py+ph, px:px+pw] = crop[...,3].astype(np.float32) / 255.0\n    out = scene.astype(np.float32)\n    a = a_full[..., None]\n    scene_lift = 1.0\n    if scene_auto_lift:\n        # Dark-scene guard: generated scenes that come back near-black (e.g. the\n        # barber slate scene) make dark products unreadable. Lift the SCENE ONLY\n        # toward a readable mean luma; product pixels are pasted afterwards and\n        # stay bit-exact. Recorded in diagnostics.\n        luma = (0.299 * out[..., 0] + 0.587 * out[..., 1] + 0.114 * out[..., 2])\n        mean_l = float(luma.mean())\n        if mean_l < 70.0:\n            target = 100.0\n            gamma = np.log(max(target, 1.0) / 255.0) / np.log(max(mean_l, 1.0) / 255.0)\n            gamma = float(np.clip(gamma, 1.0 / 1.6, 1.0))\n            out = 255.0 * (out / 255.0) ** gamma\n            scene_lift = round(1.0 / gamma, 3)\n    if scene_grade > 0:\n        # Scene-only chroma grade toward the product\'s colour balance. Never\n        # touches product pixels (pasted afterwards). Gain is clipped.\n        s_ = float(min(scene_grade, 0.5))\n        pm = crop[..., :3][crop[..., 3] > 200].reshape(-1, 3).astype(np.float32)\n        if len(pm):\n            pm = pm.mean(0); sm = out.reshape(-1, 3).mean(0) + 1e-3\n            gain = (pm / (pm.mean() + 1e-3)) / (sm / sm.mean())\n            out = out * np.clip(gain ** s_, 0.8, 1.25)\n    if shadow_style and not cover_used:\n        # Contact shadow darkens the generated scene only; product pixels are\n        # pasted afterwards so the interior stays bit-exact. Skipped for\n        # cover-fit placements (full-bleed source framing has no floor contact).\n        if shadow_style==\'surface contact\':\n            sh=surface_contact_shadow((cw,ch),(px,py),crop[...,3])\n        else:\n            sh = contact_shadow((cw, ch), (px, py), (pw, ph), crop[..., 3], shadow_style)\n        out = out * (1.0 - sh.astype(np.float32)[..., None] / 255.0)\n    if glass_backdrop is not None:\n        # Clear-glass mode: modulate the NEW scene by measured transmission instead of\n        # pasting source pixels (so the old backdrop cannot show through). NOT bit-exact.\n        full = gray_src_paste(crop, px, py, ch, cw)\n        T = np.clip(full / np.maximum(np.asarray(glass_backdrop, np.float32)[None, None, :], 1.0), 0.0, 1.0)\n        out = out * (1 - a) + out * T * a\n    else:\n        out = out * (1 - a) + gray_src_paste(crop, px, py, ch, cw) * a\n    return np.clip(out, 0, 255).astype(np.uint8), scene_lift\n\n\ndef gray_src_paste(crop: np.ndarray, px: int, py: int, ch: int, cw: int) -> np.ndarray:\n    """Full-canvas float RGB holding the source-exact product crop at (px,py)."""\n    full = np.zeros((ch, cw, 3), np.float32)\n    full[py:py + crop.shape[0], px:px + crop.shape[1]] = crop[..., :3]\n    return full\n\n\ndef estimate_clean_backdrop(rgb: np.ndarray, border_frac: float = 0.06, max_std: float = 28.0):\n    """Median colour of the photo border when it is a near-uniform backdrop.\n    Returns (B(3,), ok). ok=False for cluttered backdrops (glass2-style scenes),\n    where transmission compositing cannot be used."""\n    h, w = rgb.shape[:2]\n    t = max(2, int(min(h, w) * border_frac))\n    b = np.concatenate([rgb[:t].reshape(-1, 3), rgb[-t:].reshape(-1, 3),\n                        rgb[:, :t].reshape(-1, 3), rgb[:, -t:].reshape(-1, 3)]).astype(np.float32)\n    return np.median(b, axis=0), bool(b.std(axis=0).max() <= max_std)\n\n\ndef silhouette_from_backdrop(rgb: np.ndarray, backdrop: np.ndarray, thresh: float = 14.0) -> np.ndarray:\n    """Soft silhouette of an object on a CLEAN backdrop: difference to the backdrop,\n    holes filled by flooding from the border (clear glass interiors are near-backdrop)."""\n    d = np.abs(rgb.astype(np.float32) - backdrop[None, None, :]).max(axis=2)\n    m = (d > thresh).astype(np.uint8)\n    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8))\n    h, w = m.shape\n    ff = m.copy(); pad = np.zeros((h + 2, w + 2), np.uint8)\n    for seed in ((0, 0), (w - 1, 0), (0, h - 1), (w - 1, h - 1)):\n        if ff[seed[1], seed[0]] == 0:\n            cv2.floodFill(ff, pad, seed, 2)\n    filled = (ff != 2).astype(np.uint8)\n    k = cv2.GaussianBlur(filled.astype(np.float32), (0, 0), 1.2)\n    return np.clip(k, 0, 1)\n\n\ndef transmission_composite(src_rgb: np.ndarray, scene_rgb: np.ndarray, matte: np.ndarray,\n                           backdrop: np.ndarray, strength: float = 1.0) -> np.ndarray:\n    """Clear-glass mode (opt-in, clean-backdrop sources only). Inside the matte the\n    NEW scene is multiplied by the glass\'s measured transmission T = I / backdrop\n    (clipped 0..1), so the original background does not show through and the glass\n    keeps its darkening, tint and edge lines. Outside the matte: scene unchanged.\n    Specular highlights equal to the backdrop (T=1) are NOT recovered (known limit).\n    Not bit-exact by design: glass pixels are modulated, not pasted."""\n    T = np.clip(src_rgb.astype(np.float32) / np.maximum(backdrop[None, None, :], 1.0), 0.0, 1.0)\n    T = 1.0 - strength * (1.0 - T)\n    M = matte.astype(np.float32)[..., None]\n    sc = scene_rgb.astype(np.float32)\n    out = sc * (1 - M) + sc * T * M\n    return np.clip(out, 0, 255).astype(np.uint8)\n\n\ndef replay_scene(cache_dir, **overrides):\n    """CPU-only exact-layer replay. Overrides are finishing options, no model calls.\n    Returns an image and opaque RGB invariant, not a visual-quality verdict."""\n    from pathlib import Path\n    import json\n    cd = Path(cache_dir)\n    options = json.loads((cd / "layers.json").read_text())\n    placement = options.pop("placement")\n    options.pop("scene_mode", None)  # recorded callback provenance, not a finish argument\n    allowed = {"shadow_style", "scene_grade", "scene_auto_lift"}\n    if set(overrides) - allowed:\n        raise ValueError("Only shadow_style, scene_grade, scene_auto_lift may be overridden")\n    options.update(overrides)\n    scene = np.array(Image.open(cd / "scene.png").convert("RGB"))\n    crop = np.array(Image.open(cd / "product_rgba.png").convert("RGBA"))\n    out, lift = finish_scene(scene, crop, placement, **options)\n    px, py = placement; ph, pw = crop.shape[:2]\n    exact = None if options.get("glass_backdrop") is not None else bool(np.array_equal(\n        out[py:py+ph,px:px+pw][crop[...,3] == 255], crop[...,:3][crop[...,3] == 255]))\n    return Image.fromarray(out), {"paste_bit_exact": exact, "scene_lift": lift}\n\n\ndef surface_contact_shadow(canvas, placement, matte):\n    """Small full-silhouette shadow for explicitly topdown scenes only."""\n    w,h=canvas;x,y=placement;mh,mw=matte.shape\n    if x<0 or y<0 or x+mw>w or y+mh>h:raise ValueError(\'matte outside canvas\')\n    a=np.zeros((h,w),np.float32);a[y:y+mh,x:x+mw]=matte.astype(np.float32)/255\n    sigma=max(1.,min(w,h)*.003)\n    M=np.float32([[1,0,sigma],[0,1,2*sigma]])\n    shifted=cv2.warpAffine(a,M,(w,h),borderMode=cv2.BORDER_CONSTANT)\n    return (cv2.GaussianBlur(shifted,(0,0),sigma)*65).clip(0,65).astype(np.uint8)\n\n\ndef flatlay_center(matte, canvas):\n    w,h=canvas;mh,mw=matte.shape\n    yy,xx=np.where(matte>127)\n    if not len(xx):raise ValueError(\'empty matte\')\n    if mw>w or mh>h:raise ValueError(\'crop larger than canvas\')\n    x=round(w/2-(int(xx.min())+int(xx.max()))/2)\n    y=round(h/2-(int(yy.min())+int(yy.max()))/2)\n    return max(0,min(w-mw,x)),max(0,min(h-mh,y))\n'
# ---- embedded module: renderello/compose.py ----
path = base / 'renderello/compose.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Runtime configuration: paths, device, model identifiers, seeds.\n\nModel ids are pinned to the exact repos verified in the research plan\n(owning-source model cards). Heavy frameworks are imported lazily by the modules\nthat need them, so the CPU test path never requires torch/diffusers.\n"""\nfrom __future__ import annotations\n\nimport os\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\n\n\n# Owning-source model repos (see README license register).\nQWEN_EDIT_MODEL_ID = "Qwen/Qwen-Image-Edit-2511"          # Apache-2.0\nBIREFNET_MODEL_ID = "ZhengPeng7/BiRefNet"                 # MIT\nRMBG2_MODEL_ID = "briaai/RMBG-2.0"                        # CC BY-NC 4.0 - restricted, see README\nIC_LIGHT_REPO = "lllyasviel/ic-light"                     # code Apache-2.0 / weights OpenRAIL-M\n\nMAGICHOUR_API_BASE = os.environ.get("MAGICHOUR_API_BASE", "https://api.magichour.ai")\n\nDEFAULT_OUTPUT_ROOT = Path(os.environ.get("RENDERELLO_OUT", "runs"))\nCACHE_DIR = Path(os.environ.get("RENDERELLO_CACHE", ".cache/renderello"))\n\n# Qwen-Image-Edit-2511 inference defaults per the model card quick-start.\nQWEN_DEFAULT_STEPS = 40\nQWEN_TRUE_CFG_SCALE = 4.0\nQWEN_GUIDANCE_SCALE = 1.0\n\n# Working resolutions. Longest side is capped; product is never upscaled past source.\nMAX_WORKING_SIDE = 1024\nMIN_WORKING_SIDE = 512\n\n\n@dataclass(frozen=True)\nclass DeviceConfig:\n    """Resolved execution device. GPU work is opt-in; CPU is the default so the\n    deterministic route and the whole test suite run anywhere."""\n    device: str = "cpu"\n    dtype_name: str = "float32"  # bfloat16 on CUDA per Qwen model card\n\n    @staticmethod\n    def resolve(prefer_gpu: bool = False) -> "DeviceConfig":\n        if prefer_gpu:\n            try:\n                import torch  # lazy: GPU hosts only\n                if torch.cuda.is_available():\n                    return DeviceConfig(device="cuda", dtype_name="bfloat16")\n            except ImportError:\n                pass\n        return DeviceConfig()\n\n\n@dataclass(frozen=True)\nclass RunConfig:\n    """Per-run knobs. Seeds are explicit so benchmark runs are reproducible and\n    first-try results stay honestly attributable to a recorded seed."""\n    seed: int = 0\n    variants: int = 2                 # problem statement: generate 2-4 alternatives\n    max_variants: int = 4\n    out_root: Path = DEFAULT_OUTPUT_ROOT\n    device: DeviceConfig = field(default_factory=DeviceConfig)\n    save_intermediates: bool = True   # cutouts, backgrounds, pre-relight - auditability\n\n    def variant_seeds(self) -> list[int]:\n        n = max(1, min(self.variants, self.max_variants))\n        return [self.seed + i for i in range(n)]\n'
# ---- embedded module: renderello/config.py ----
path = base / 'renderello/config.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Native source-locked compositor with explicit per-object contact spans.\n\nContact spans are supplied in source coordinates and bound to the mask hash.\nThey are not estimated 3D support, cannot fix floating source poses, and never\nsilently bridge separate products onto one ground line. OpenCV distance/blur\nand NumPy implement a bounded 2D illustration, not a physical renderer.\n"""\nfrom __future__ import annotations\nfrom dataclasses import asdict, dataclass\nimport math\nimport cv2\nimport numpy as np\nfrom .candidate_diagnostics import native_rgb, native_alpha, digest\nfrom .candidate_harmonization import linear_over\nfrom .checked_research.contact_camera import linear, srgb\n\n\n@dataclass(frozen=True)\nclass ContactSpan:\n    name: str\n    x0: int\n    x1: int\n    bottom_min: int\n    bottom_max: int\n\n    def __post_init__(self):\n        if not isinstance(self.name, str) or not self.name.strip() or len(self.name) > 80:\n            raise ValueError(\'short contact name required\')\n        if any(type(getattr(self, v)) is not int for v in (\'x0\', \'x1\', \'bottom_min\', \'bottom_max\')):\n            raise ValueError(\'integer contact coordinates required\')\n        if self.x0 < 0 or self.x1 <= self.x0 or self.bottom_min < 0 or self.bottom_max < self.bottom_min:\n            raise ValueError(\'nonempty contact interval and ordered row range required\')\n\n\n@dataclass(frozen=True)\nclass ContactPolicy:\n    camera: str\n    material: str\n    framing: str\n    source_pose: str\n    source_sha256: str\n    alpha_sha256: str\n    reviewed: bool = False\n    opacity: float = .25\n    decay: float = 3.0\n    length: int = 20\n    lateral_sigma: float = 2.0\n    cast_dx: int = 5\n    flatlay_cast_offset: tuple[int, int] = (0, 0)\n    span_taper_px: int = 0\n\n    def __post_init__(self):\n        if self.camera not in (\'front\', \'elevated\', \'flatlay\'):\n            raise ValueError(\'unsupported camera\')\n        if self.material not in (\'opaque\', \'transparent\', \'reflective\', \'unknown\'):\n            raise ValueError(\'unsupported material\')\n        if self.framing not in (\'complete\', \'cropped\', \'unknown\'):\n            raise ValueError(\'unsupported framing\')\n        if self.source_pose not in (\'front\', \'elevated\', \'overhead\', \'suspended\', \'unknown\'):\n            raise ValueError(\'unsupported source pose\')\n        if type(self.reviewed) is not bool:\n            raise ValueError(\'reviewed must be boolean\')\n        for name, lo, hi in [(\'opacity\', 0, .5), (\'decay\', .5, 12), (\'lateral_sigma\', 0, 12)]:\n            v = getattr(self, name)\n            if isinstance(v, bool) or not isinstance(v, (float, int)) or not math.isfinite(v) or not lo <= v <= hi:\n                raise ValueError(name+\' outside configured bounds\')\n        if type(self.length) is not int or not 1 <= self.length <= 96:\n            raise ValueError(\'length must be 1..96\')\n        if type(self.cast_dx) is not int or not -64 <= self.cast_dx <= 64:\n            raise ValueError(\'cast_dx must be -64..64\')\n        if type(self.span_taper_px) is not int or not 0 <= self.span_taper_px <= 64:\n            raise ValueError(\'span taper must be integer 0..64\')\n        if self.camera == \'flatlay\' and self.span_taper_px:\n            raise ValueError(\'span taper applies only to upright contact spans\')\n        offset = self.flatlay_cast_offset\n        if not isinstance(offset, (tuple, list)) or len(offset) != 2 or any(type(v) is not int or not -16 <= v <= 16 for v in offset):\n            raise ValueError(\'flatlay cast offset requires two integer channels -16..16\')\n        if self.camera != \'flatlay\' and any(offset):\n            raise ValueError(\'flatlay cast offset is incompatible with upright camera\')\n        for h in (self.source_sha256, self.alpha_sha256):\n            if not isinstance(h, str) or len(h) != 64 or any(c not in \'0123456789abcdef\' for c in h):\n                raise ValueError(\'source and alpha pixel hashes required\')\n\n\ndef eligible(rgb, alpha, policy):\n    native_rgb(rgb)\n    native_alpha(alpha, rgb.shape[:2])\n    if not isinstance(policy, ContactPolicy):\n        raise ValueError(\'typed contact policy required\')\n    if digest(rgb) != policy.source_sha256 or digest(alpha) != policy.alpha_sha256:\n        raise ValueError(\'contact declaration belongs to different source or alpha\')\n    if not policy.reviewed:\n        raise ValueError(\'explicit source-bound contact review required\')\n    if policy.material != \'opaque\':\n        raise ValueError(\'source-locked contact route only accepts declared opaque materials\')\n    if policy.framing != \'complete\':\n        raise ValueError(\'cannot make complete-product contact claim for cropped/unreviewed source\')\n    if policy.source_pose in (\'suspended\', \'unknown\'):\n        raise ValueError(\'unknown or suspended source pose needs a different concept\')\n    if (policy.camera == \'flatlay\') != (policy.source_pose == \'overhead\'):\n        raise ValueError(\'source and target camera incompatible\')\n    hard = alpha >= 128\n    if not hard.any() or hard.all():\n        raise ValueError(\'nontrivial native mask required\')\n    if any(edge.any() for edge in (hard[0], hard[-1], hard[:, 0], hard[:, -1])):\n        raise ValueError(\'frame-touching support conflicts with complete contact route\')\n\n\ndef contact_map(rgb, alpha, spans, policy):\n    eligible(rgb, alpha, policy)\n    h, w = alpha.shape\n    if len(spans) > 64:\n        raise ValueError(\'too many contact spans\')\n    if policy.camera == \'flatlay\':\n        if spans:\n            raise ValueError(\'flatlay uses the whole silhouette, no upright spans\')\n        # A finite outside-distance field attaches at the silhouette, including\n        # real holes, without applying guessed upright geometry to a flatlay.\n        hard = alpha >= 128\n        distance = cv2.distanceTransform((~hard).astype(np.uint8), cv2.DIST_L2, cv2.DIST_MASK_PRECISE)\n        shade = policy.opacity*np.exp(-distance/policy.decay)\n        shade[distance > policy.length] = 0\n        offset = policy.flatlay_cast_offset\n        method = \'flatlay_finite_outside_distance\'\n        if any(offset):\n            # Explicit 2D art direction, not recovered light or height. Keep a\n            # weak attached rim, plus a translated silhouette outside hard RGB.\n            dx, dy = offset\n            shifted = cv2.warpAffine(hard.astype(np.uint8),\n                np.array([[1, 0, dx], [0, 1, dy]], dtype=float), (w, h),\n                flags=cv2.INTER_NEAREST, borderMode=cv2.BORDER_CONSTANT) > 0\n            if int(shifted.sum()) != int(hard.sum()):\n                raise ValueError(\'flatlay cast extends beyond canvas; supply explicit room\')\n            cast_distance = cv2.distanceTransform((~shifted).astype(np.uint8), cv2.DIST_L2, cv2.DIST_MASK_PRECISE)\n            cast = .8*policy.opacity*np.exp(-cast_distance/policy.decay)\n            cast[cast_distance > policy.length] = 0\n            shade = np.maximum(.3*shade, cast)\n            method = \'authored_flatlay_offset_silhouette\'\n        shade[hard] = 0\n        return shade, dict(method=method, spans=[], flatlay_cast_offset=list(offset),\n            light_inferred=False, source_height_inferred=False)\n    if not spans:\n        raise ValueError(\'upright route needs explicit per-object contact spans\')\n    if any(not isinstance(s, ContactSpan) for s in spans):\n        raise ValueError(\'typed contact spans required\')\n    if len({s.name for s in spans}) != len(spans):\n        raise ValueError(\'unique contact names required\')\n    # Native lower contour per column. Empty columns remain -1 and never bridge.\n    hard = alpha >= 128\n    lower = np.where(hard.any(axis=0), h-1-np.argmax(hard[::-1], axis=0), -1)\n    shade = np.zeros((h, w), np.float64)\n    used_columns = np.zeros(w, bool)\n    records = []\n    for span in spans:\n        if span.x1 > w or span.bottom_max >= h:\n            raise ValueError(\'contact span outside native image\')\n        cols = np.arange(span.x0, span.x1)\n        rows = lower[cols]\n        if np.any(rows < span.bottom_min) or np.any(rows > span.bottom_max):\n            raise ValueError(\'contact range contradicts native lower contour; no automatic projection\')\n        if used_columns[cols].any():\n            raise ValueError(\'overlapping contact spans would double-count support\')\n        if np.any(rows + policy.length >= h):\n            raise ValueError(\'native canvas too short for finite shadow; supply a larger explicit canvas\')\n        used_columns[cols] = True\n        taper = np.ones(len(cols), np.float64)\n        if policy.span_taper_px:\n            # Explicit art direction of contact endpoints, not inferred support.\n            # Smoothstep has zero slope at either end; no product pixels move.\n            edge = np.minimum(np.arange(len(cols)) + .5, np.arange(len(cols))[::-1] + .5)\n            t = np.minimum(1., edge / policy.span_taper_px)\n            taper = t*t*(3-2*t)\n        patch = np.zeros((h, w), np.float64)\n        # The first row below each declared lower contour gets the unblurred\n        # contact core. Lateral blur only; no vertical shift detaches it.\n        for step in range(1, policy.length+1):\n            weight = policy.opacity*math.exp(-(step-1)/policy.decay)\n            patch[rows+step, cols] = weight*taper\n        if policy.lateral_sigma:\n            patch = cv2.GaussianBlur(patch, (0, 1), sigmaX=policy.lateral_sigma, sigmaY=0,\n                                     borderType=cv2.BORDER_CONSTANT)\n        # A deliberately weak same-attached cast, separate from the finite core.\n        cast = np.zeros_like(patch)\n        for step in range(1, policy.length+1):\n            dx = round(policy.cast_dx*(step-1)/max(1, policy.length-1))\n            target = cols + dx\n            valid = (target >= 0) & (target < w)\n            cast[rows[valid]+step, target[valid]] = .28*policy.opacity*math.exp(-(step-1)/(policy.decay*2))*taper[valid]\n        patch = np.maximum(patch, cast)\n        shade = np.maximum(shade, patch)\n        records.append(dict(**asdict(span), columns=len(cols),\n                            observed_bottom_range=[int(rows.min()), int(rows.max())], span_taper_px=policy.span_taper_px))\n    # Soft edges still receive the background shadow underneath them.\n    # Suppressing every nonzero alpha creates a gap on noisy model masks.\n    shade[hard] = 0\n    if not np.isfinite(shade).all() or np.any((shade < 0) | (shade > .5)):\n        raise ValueError(\'invalid contact attenuation\')\n    return shade, dict(method=\'reviewed_native_contact_spans\', spans=records)\n\n\ndef render_contact(source, alpha, background, spans, policy):\n    native_rgb(background)\n    if background.shape != source.shape:\n        raise ValueError(\'explicit native same-canvas background required\')\n    shade, meta = contact_map(source, alpha, spans, policy)\n    shaded = srgb(linear(background)*(1-shade[..., None]))\n    result = linear_over(source, alpha, shaded)\n    meta.update(policy=asdict(policy), source_resampled=False, source_cropped=False,\n                source_rgb_sha256=digest(source), alpha_sha256=digest(alpha),\n                background_sha256=digest(background), result_sha256=digest(result),\n                opaque_rgb_exact=bool(np.array_equal(result[alpha == 255], source[alpha == 255])),\n                changed_background_pixels=int((shade > 0).sum()),\n                recovered_3d_geometry=False, automatic_acceptance=False,\n                limitation=\'Reviewed 2D support illustration. Cannot repair pose, source reflection, missing parts or material lighting.\')\n    return result, shade, meta\n'
# ---- embedded module: renderello/contact_spans.py ----
path = base / 'renderello/contact_spans.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Opt-in two-layer floor-shadow replay. No inferred light or product editing.\n\nA designer declares a flat-floor setting and a shadow direction. Alpha-only\nprojection cannot recover3D geometry, so outputs remain designed candidates.\n"""\nfrom pathlib import Path\nimport json\nimport math\nimport numpy as np\nfrom PIL import Image, ImageFilter\nfrom .intent_mask import sha\n\nFLOOR_STYLES = (\'porcelain_sweep\',\'sage_arch\',\'noir_halo\',\'amber_paper\',\'linen_daylight\',\'cobalt_aperture\')\n\n\ndef shadow_layers(alpha, size, placement, *, direction=0.45, depth=0.12,\n                  cast_opacity=0.08, contact_opacity=0.42):\n    if not isinstance(alpha,np.ndarray) or alpha.dtype!=np.uint8 or alpha.ndim!=2:\n        raise ValueError(\'alpha must be a uint8 plane\')\n    if len(size)!=2 or any(type(n)!=int or not 64<=n<=4096 for n in size):\n        raise ValueError(\'canvas dimensions must be64-4096\')\n    if len(placement)!=2 or any(type(n)!=int for n in placement):\n        raise ValueError(\'placement must be two integers\')\n    vals=(direction,depth,cast_opacity,contact_opacity)\n    if any(type(v) not in (int,float) or not math.isfinite(v) for v in vals):\n        raise ValueError(\'shadow parameters must be finite numbers\')\n    if not -1<=direction<=1 or not 0.02<=depth<=0.3 or not 0<=cast_opacity<=0.4 or not 0<=contact_opacity<=0.6:\n        raise ValueError(\'shadow parameters outside reviewed bounds\')\n    w,h=size;left,top=placement;ph,pw=alpha.shape\n    if not (0<=left and 0<=top and left+pw<=w and top+ph<=h):\n        raise ValueError(\'product outside canvas\')\n    ys,xs=np.where(alpha>127)\n    if not len(ys):raise ValueError(\'support pixels above127 required\')\n    foot=int(ys.max());floor=top+foot\n    if floor+3>=h:raise ValueError(\'insufficient floor room\')\n    # A small band of actual base columns, not an ellipse across empty foot gaps.\n    band=max(1,min(3,round(ph*.006)))\n    profile=alpha[max(0,foot-band+1):foot+1].max(axis=0).astype(float)/255\n    contact=np.zeros((h,w),np.uint8)\n    for dy,factor in ((0,1.),(1,.75),(2,.25)):\n        contact[floor+dy,left:left+pw]=np.round(profile*contact_opacity*255*factor).astype(\'uint8\')\n    contact=np.array(Image.fromarray(contact).filter(ImageFilter.GaussianBlur(1.5)))\n    # Designed2Dprojection. Higher rows travel farther; do not claim true optics.\n    cast=np.zeros((h,w),np.uint8)\n    for row in range(foot+1):\n        height=foot-row;dy=round(height*depth);dx=round(height*direction*depth)\n        target_y=floor+dy\n        if target_y>=h:continue\n        src0=max(0,-left-dx);src1=min(pw,w-left-dx)\n        if src1<=src0:continue\n        value=np.round(alpha[row,src0:src1].astype(float)*cast_opacity*math.exp(-height/max(1,foot*.22))).astype(\'uint8\')\n        a=left+dx+src0;b=left+dx+src1\n        cast[target_y,a:b]=np.maximum(cast[target_y,a:b],value)\n    cast=np.array(Image.fromarray(cast).filter(ImageFilter.GaussianBlur(5)))\n    # Blur first, then floor clipping. No blurred halo is allowed on the wall.\n    contact[:floor]=0;cast[:floor]=0\n    combined=np.round(255*(1-(1-contact.astype(float)/255)*(1-cast.astype(float)/255))).astype(\'uint8\')\n    return contact,cast,combined,dict(floor_row=floor,support_band_rows=band,\n        support_columns=int((profile>0).sum()),direction=direction,depth=depth,\n        cast_opacity=cast_opacity,contact_opacity=contact_opacity,\n        rows_above_floor_nonzero=int(np.count_nonzero(combined[:floor])),\n        physical_geometry_recovered=False)\n\n\ndef replay_contact(candidate:Path,out:Path,*,flat_floor_confirmed=False,direction=0.45,depth=0.12):\n    candidate=Path(candidate);out=Path(out)\n    if out.exists():raise ValueError(\'output exists; preserve earlier versions\')\n    if flat_floor_confirmed is not True:raise ValueError(\'explicit flat floor declaration required\')\n    m=json.loads((candidate/\'native-studio-manifest.json\').read_text())\n    if m.get(\'style\') not in FLOOR_STYLES:raise ValueError(\'platform support geometry requires separate review\')\n    if m.get(\'material\')!=\'opaque\' or m.get(\'framing\')!=\'complete\' or m.get(\'source_pose\') not in (\'front\',\'elevated\') or m.get(\'scale\')!=1 or m.get(\'rotation\')!=0:\n        raise ValueError(\'source-native opaque complete candidate required\')\n    for f in (\'background.png\',\'native-product-crop.png\'):\n        if sha(candidate/f)!=m.get(\'output_sha256\',{}).get(f):raise ValueError(\'candidate file hash mismatch: \'+f)\n    with Image.open(candidate/\'background.png\') as im:bg=im.convert(\'RGB\')\n    with Image.open(candidate/\'native-product-crop.png\') as im:\n        if im.mode!=\'RGBA\':raise ValueError(\'RGBA product required\')\n        crop=np.array(im)\n    alpha=crop[:,:,3];core=alpha==255\n    if not np.any(core):raise ValueError(\'opaque core required\')\n    contact,cast,combined,d=shadow_layers(alpha,bg.size,m[\'placement\'],direction=direction,depth=depth)\n    scene=Image.composite(Image.new(\'RGB\',bg.size,(0,0,0)),bg,Image.fromarray(combined))\n    scene.paste(Image.fromarray(crop[:,:,:3]),tuple(m[\'placement\']),Image.fromarray(alpha))\n    x,y=m[\'placement\'];ph,pw=alpha.shape;result=np.array(scene)[y:y+ph,x:x+pw]\n    d[\'opaque_core_rgb_max_error\']=int(np.abs(result[core].astype(\'int16\')-crop[:,:,:3][core].astype(\'int16\')).max())\n    if d[\'opaque_core_rgb_max_error\']:raise ValueError(\'source-native product invariant failed\')\n    d.update(style=m[\'style\'],parent_manifest_sha256=sha(candidate/\'native-studio-manifest.json\'),source_sha256=m[\'source_sha256\'],placement=m[\'placement\'],scale=1,rotation=0,opaque_core_pixels=int(core.sum()),quality=\'NOT_EVALUATED\',mode=\'DESIGNED_TWO_LAYER_SHADOW_NOT_INFERRED_LIGHTING\',warnings=[\'No product relighting or material repair.\',\'Source alpha errors survive unchanged.\',\'Only fully opaque core pixels remain exact in final RGB.\',\'Flat floor and light direction are designer declarations, not image inference.\',\'Platforms rejected, no attempt to paint a shadow across unsupported geometry.\'])\n    out.mkdir(parents=True);scene.save(out/\'candidate.png\')\n    for name,a in ((\'contact-alpha.png\',contact),(\'cast-alpha.png\',cast),(\'shadow-alpha.png\',combined)):Image.fromarray(a).save(out/name)\n    d[\'output_sha256\']={p.name:sha(p) for p in out.iterdir()}\n    (out/\'contact-studio-manifest.json\').write_text(json.dumps(d,indent=2));return d\n'
# ---- embedded module: renderello/contact_studio.py ----
path = base / 'renderello/contact_studio.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Explicit whole-photograph copyspace exports, not product staging.\n\nRun python -m renderello.context_reframe_cli --help. Does not call models,\nresample products, infer a review, relight, or create photoshoot templates.\n"""\nfrom pathlib import Path\nfrom io import BytesIO\nimport argparse,hashlib,json\nimport numpy as np\nfrom PIL import Image\nfrom .candidate_diagnostics import digest,native_rgb,native_alpha\nfrom .source_context_extension import extend_source_context\n\nASPECTS={\'square\':(1024,1024),\'portrait\':(896,1120),\'landscape\':(1056,704)}\n\n\ndef _bytes(path,limit):\n    with Path(path).open(\'rb\')as stream:data=stream.read(limit+1)\n    if len(data)>limit:raise ValueError(\'bounded input bytes required\')\n    return data\n\n\ndef _image(data,mode):\n    with Image.open(BytesIO(data))as im:\n        if im.mode!=mode or getattr(im,\'n_frames\',1)!=1:raise ValueError(\'single-frame \'+mode+\' required; no silent colour conversion\')\n        if im.getexif().get(274,1)!=1:raise ValueError(\'normalize orientation before review\')\n        if im.width*im.height>1_500_000:raise ValueError(\'native1.5MP budget; no downscale\')\n        return np.array(im)\n\n\ndef run(source,alpha,review,out,*,aspects=(\'square\',\'portrait\',\'landscape\')):\n    out=Path(out)\n    if out.exists():raise FileExistsError(\'preserve earlier exports; choose a new directory\')\n    if not aspects or len(set(aspects))!=len(aspects)or any(x not in ASPECTS for x in aspects):raise ValueError(\'unique supported aspect names required\')\n    blobs={n:_bytes(p,32_000_000 if n!=\'review\'else 64_000)for n,p in [(\'source\',source),(\'alpha\',alpha),(\'review\',review)]}\n    rgb=_image(blobs[\'source\'],\'RGB\');a=_image(blobs[\'alpha\'],\'L\');native_rgb(rgb);native_alpha(a,rgb.shape[:2])\n    spec=json.loads(blobs[\'review\'])\n    keys={\'schema\',\'source_pixels_sha256\',\'alpha_pixels_sha256\',\'material\',\'reviewed\',\'note\'}\n    if not isinstance(spec,dict)or set(spec)!=keys or spec[\'schema\']!=\'renderello-whole-photo-reframe-review-v1\':raise ValueError(\'exact supported review schema required\')\n    if spec[\'reviewed\']is not True or spec[\'material\']!=\'opaque\':raise ValueError(\'explicit opaque review required; no inferred material or glass recovery\')\n    if not isinstance(spec[\'note\'],str)or not spec[\'note\'].strip()or len(spec[\'note\'])>2000:raise ValueError(\'bounded nonempty review note required\')\n    if spec[\'source_pixels_sha256\']!=digest(rgb)or spec[\'alpha_pixels_sha256\']!=digest(a):raise ValueError(\'review belongs to different source/alpha pixels\')\n    out.mkdir(parents=True);(out/\'INCOMPLETE\').touch()\n    Image.fromarray(rgb).save(out/\'source.png\');Image.fromarray(a).save(out/\'alpha.png\');(out/\'review.json\').write_bytes(blobs[\'review\'])\n    rows=[];h,w=a.shape\n    for name in aspects:\n        canvas=ASPECTS[name];offset=((canvas[0]-w)//2,(canvas[1]-h)//2)\n        try:\n            result,report=extend_source_context(rgb,a,material=\'opaque\',reviewed_source_sha256=digest(rgb),reviewed_alpha_sha256=digest(a),canvas=canvas,offset=offset,smooth_outer=True)\n            dx,dy=offset\n            if not np.array_equal(result[dy:dy+h,dx:dx+w],rgb):raise RuntimeError(\'whole-source exactness failed\')\n            path=out/(name+\'.png\');Image.fromarray(result).save(path)\n            rows.append(dict(aspect=name,status=\'EXECUTED_REVIEW_REQUIRED\',file=path.name,sha256=hashlib.sha256(path.read_bytes()).hexdigest(),report=report))\n        except ValueError as exc:rows.append(dict(aspect=name,status=\'REFUSED\',reason=str(exc),canvas=list(canvas)))\n    manifest=dict(schema=\'renderello-whole-photo-reframe-result-v1\',purpose=\'ADDED_COPYSPACE_ONLY_NOT_PHOTOSHOOT_TEMPLATE\',\n        input_file_sha256={n:hashlib.sha256(b).hexdigest()for n,b in blobs.items()},input_pixels_sha256={\'source\':digest(rgb),\'alpha\':digest(a)},\n        input_semantics=\'bounded_snapshots_no_resampling_or_colour_conversion\',review_is_caller_declaration_not_owner_quality_approval=True,\n        planned_exports=len(rows),rendered_exports=sum(r[\'status\']==\'EXECUTED_REVIEW_REQUIRED\'for r in rows),\n        refused_exports=sum(r[\'status\']==\'REFUSED\'for r in rows),distinct_sources=1,new_template_accepts=0,photo_quality=\'NOT_EVALUATED\',rows=rows,\n        limitations=[\'Entire old photograph is retained, including its camera, light, background and contact.\',\n        \'Estimated alpha is not ground truth; source-border review and native visual inspection remain required.\',\n        \'No new scene, relighting, pose, transparent optics or polished-template acceptance.\'])\n    (out/\'manifest.json\').write_text(json.dumps(manifest,indent=2));(out/\'INCOMPLETE\').unlink();return manifest\n\n\ndef main(argv=None):\n    parser=argparse.ArgumentParser(description=__doc__)\n    for n in [\'source\',\'alpha\',\'review\',\'out\']:parser.add_argument(\'--\'+n,type=Path,required=True)\n    parser.add_argument(\'--aspects\',nargs=\'+\',choices=tuple(ASPECTS),default=list(ASPECTS));args=parser.parse_args(argv)\n    result=run(args.source,args.alpha,args.review,args.out,aspects=tuple(args.aspects));print(json.dumps(result,indent=2));return 0\n\n\nif __name__==\'__main__\':raise SystemExit(main())\n'
# ---- embedded module: renderello/context_reframe_cli.py ----
path = base / 'renderello/context_reframe_cli.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Cost/latency economics (PDF acceptance gate): median latency, cost per\nsuccessful output, retries, and the gross-margin formula. All Magic Hour\nfigures are PROJECTIONS from published list data (docs.magichour.ai,\nfetched 2026-11-14): model credit costs and subscription included-credit\nrates. These catalog constants are not historical job receipts or measured latency.\nActual run evidence must be supplied separately. Effective $/credit uses each plan\'s included credits (a bulk\naverage, NOT a marginal price); credit-pack/usage-based rates vary by offer\nand are not published numerically, so they are inputs, not defaults."""\nfrom __future__ import annotations\n\nimport statistics\nfrom dataclasses import dataclass\n\n\n# docs.magichour.ai/api-reference/models (2026-11-14): "From" = credits for one\n# image at the model\'s lowest resolution; larger resolutions cost more.\nMODEL_CREDITS_FROM = {\n    "flux-schnell": 5, "flux-2-klein": 5, "z-image-turbo": 5, "krea-2": 10,\n    "qwen-edit": 10, "qwen-image-2.1": 10, "seedream-v4": 40,\n    "seedream-v4.5": 50, "seedream-v5-flash": 30, "nano-banana": 50, "nano-banana-2-lite": 50,\n    "nano-banana-2.1": 50, "gpt-image-2": 50, "seedream-v5-pro": 75,\n    "nano-banana-2": 50, "gpt-image-2.5-flare": 100, "nano-banana-pro": 150,\n    # 13 Nov 2026 settled receipts: 200/image at res=1k, 1:1, single input.\n    "gpt-image-2.5-flare-1k": 200,\n}\n\n# docs.magichour.ai/billing/overview (2026-11-14): subscription included credits.\n# effective_usd_per_credit = monthly price / monthly credits (bulk included rate).\nSUBSCRIPTION_PLANS = {\n    "creator_monthly": {"usd": 19, "credits": 12_000},\n    "creator_annual": {"usd": 12, "credits": 12_000},\n    "pro_monthly": {"usd": 39, "credits": 25_000},\n    "pro_annual": {"usd": 25, "credits": 25_000},\n    "business_monthly": {"usd": 99, "credits": 70_000},\n    "business_annual": {"usd": 66, "credits": 70_000},\n}\n\n\n@dataclass(frozen=True)\nclass RouteEconomics:\n    route: str\n    runs: int\n    successes: int\n    retries: int                       # runs beyond the first for a success\n    median_seconds: float | None       # measured, real runs only\n    credits_total: float               # recorded on the runs (0 for self-hosted)\n    credits_per_successful_output: float | None\n\n\ndef route_economics(route: str, rows: list[dict]) -> RouteEconomics:\n    """rows: benchmark/result dicts with seconds, credits_estimated, success."""\n    times = [float(r["seconds"]) for r in rows if r.get("seconds") is not None]\n    successes = sum(1 for r in rows if r.get("success", True))\n    credits = sum(float(r.get("credits_estimated", 0)) for r in rows)\n    return RouteEconomics(\n        route=route, runs=len(rows), successes=successes,\n        retries=max(0, len(rows) - successes) if successes else len(rows),\n        median_seconds=round(statistics.median(times), 2) if times else None,\n        credits_total=credits,\n        credits_per_successful_output=round(credits / successes, 2) if credits and successes else None)\n\n\ndef effective_usd_per_credit(plan: str) -> float:\n    """Bulk included-credit rate for a subscription plan. NOT a marginal price."""\n    p = SUBSCRIPTION_PLANS[plan]\n    return p["usd"] / p["credits"]\n\n\ndef projected_usd_per_output(model: str, plan: str, images_per_generation: int = 1,\n                             credits_per_generation: float | None = None) -> float:\n    """Projected $ cost of one MH generation under a plan\'s included-credit rate.\n    credits_per_generation overrides the published \'from\' rate (e.g. higher\n    resolution or multi-image batches cost more than the single-image floor)."""\n    credits = credits_per_generation if credits_per_generation is not None \\\n        else MODEL_CREDITS_FROM[model] * images_per_generation\n    return round(credits * effective_usd_per_credit(plan), 6)\n\n\ndef gross_margin_per_output(sale_price_usd: float, cost_usd: float) -> float:\n    """Margin formula only: the sale price is a caller input, never assumed."""\n    return round(sale_price_usd - cost_usd, 6)\n'
# ---- embedded module: renderello/costs.py ----
path = base / 'renderello/costs.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Explicit experimental U2NETP masks. No download, default selection or quality pass.\nOnly accepts inspected original-publisher architecture and weights by SHA256.\nRun this command with a CPU torch interpreter in its own environment.\n"""\nfrom __future__ import annotations\nimport argparse,hashlib,json,subprocess,sys,time\nfrom pathlib import Path\n\nCODE_SHA256=\'96dd7a19c7de4f13520ccfc1075ded3350ff4946493be3561b9918a46218f415\'\nWEIGHTS_SHA256=\'e7567cde013fb64813973ce6e1ecc25a80c05c3ca7adbc5a54f3c3d90991b854\'\nFULL_WEIGHTS_SHA256=\'10025a17f49cd3208afc342b589890e402ee63123d6f2d289a4a0903695cce58\'\nCOMMIT=\'ac7e1c817ecab7c7dff5ce6b1abba61cd213ff29\'\n\n\ndef verified_file(path:Path,expected:str)->str:\n    actual=hashlib.sha256(path.read_bytes()).hexdigest()\n    if actual!=expected:raise ValueError(f\'publisher artifact hash mismatch: {path.name}\')\n    return actual\n\n\ndef estimate(source:Path,code:Path,weights:Path,out:Path,model:str="u2netp"):\n    if model not in ("u2netp","u2net"):raise ValueError("unsupported model")\n    weight_hash = WEIGHTS_SHA256 if model == "u2netp" else FULL_WEIGHTS_SHA256\n    verified_file(code,CODE_SHA256);verified_file(weights,weight_hash)\n    # Verification must precede module import and tensor deserialization.\n    import importlib.util\n    import resource\n    import numpy as np\n    import torch\n    from PIL import Image\n    if out.exists():raise ValueError(\'output directory exists; use a fresh directory\')\n    out.mkdir(parents=True)\n    spec=importlib.util.spec_from_file_location(\'_verified_u2net\',code)\n    module=importlib.util.module_from_spec(spec);spec.loader.exec_module(module)\n    torch.set_num_threads(1);torch.set_num_interop_threads(1)\n    net=(module.U2NETP if model == "u2netp" else module.U2NET)(3,1)\n    net.load_state_dict(torch.load(weights,map_location=\'cpu\',weights_only=True));net.eval()\n    im=Image.open(source).convert(\'RGB\')\n    x=np.asarray(im.resize((320,320),Image.Resampling.BILINEAR)).astype(\'float32\')\n    x/=max(float(x.max()),1.)\n    x=(x-np.array([.485,.456,.406],dtype=\'float32\'))/np.array([.229,.224,.225],dtype=\'float32\')\n    t=time.monotonic()\n    with torch.inference_mode():raw=net(torch.from_numpy(x.transpose(2,0,1).copy())[None])[0][0,0].numpy()\n    if not np.isfinite(raw).all():raise ValueError(\'non-finite model output\')\n    lo,hi=float(raw.min()),float(raw.max())\n    if hi-lo<1e-8:raise ValueError(\'constant saliency; no reliable foreground range\')\n    alpha=Image.fromarray(((raw-lo)/(hi-lo)*255).astype(\'uint8\')).resize(im.size,Image.Resampling.BILINEAR)\n    np.save(out/\'raw-saliency-320.npy\',raw)\n    alpha.save(out/\'estimated-alpha.png\')\n    rgba=im.copy();rgba.putalpha(alpha);rgba.save(out/\'estimated-cutout.png\')\n    preview=Image.new(\'RGB\',im.size,(218,236,223));preview.paste(im,(0,0),alpha);preview.save(out/\'review-preview.png\')\n    manifest={\'status\':\'REVIEW_REQUIRED\',\'backend\':model+\'_experimental\',\'source\':str(source.resolve()),\n              \'source_sha256\':hashlib.sha256(source.read_bytes()).hexdigest(),\'code_commit\':COMMIT,\n              \'code_sha256\':CODE_SHA256,\'weights_sha256\':weight_hash,\'input_size\':[320,320],\n              \'torch_version\':torch.__version__,\'inference_seconds\':time.monotonic()-t,\n              \'maxrss_kib\':resource.getrusage(resource.RUSAGE_SELF).ru_maxrss,\n              \'raw_probability_range\':[lo,hi],\'raw_saliency\':\'raw-saliency-320.npy\',\n              \'alpha_sha256\':hashlib.sha256((out/\'estimated-alpha.png\').read_bytes()).hexdigest(),\n              \'alpha\':\'estimated-alpha.png\',\n              \'warning\':\'New estimated saliency, not recovered GPU alpha or quality acceptance. Inspect missing objects, halos, disconnected fragments and transparency. No component removal or edge cleanup applied.\'}\n    (out/\'mask-manifest.json\').write_text(json.dumps(manifest,indent=2))\n    return manifest\n\n\ndef main():\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--source\',type=Path,required=True);p.add_argument(\'--code\',type=Path,required=True)\n    p.add_argument(\'--weights\',type=Path,required=True);p.add_argument(\'--out\',type=Path,required=True)\n    p.add_argument(\'--model\',choices=[\'u2netp\',\'u2net\'],default=\'u2netp\',help=\'explicit original-publisher variant; both experimental\')\n    p.add_argument(\'--worker\',action=\'store_true\',help=argparse.SUPPRESS)\n    args=p.parse_args()\n    # The parent does not import torch; it can stop the child before OOM.\n    if args.worker:\n        print(json.dumps(estimate(args.source,args.code,args.weights,args.out,args.model)),flush=True);return 0\n    verified_file(args.code,CODE_SHA256);verified_file(args.weights,WEIGHTS_SHA256 if args.model == "u2netp" else FULL_WEIGHTS_SHA256)\n    if args.out.exists():p.error(\'output directory exists; use a fresh directory\')\n    child=subprocess.Popen([sys.executable,\'-m\',\'renderello.cpu_mask\',*sys.argv[1:],\'--worker\'])\n    start=time.monotonic();reason=None\n    while child.poll() is None:\n        try:\n            text=Path(f\'/proc/{child.pid}/status\').read_text()\n            rss=int(next(line.split()[1] for line in text.splitlines() if line.startswith(\'VmRSS:\')))\n        except (FileNotFoundError,StopIteration):rss=0\n        if rss>1500*1024:reason=\'1500MiB RSS safety limit\'\n        if time.monotonic()-start>90:reason=\'90second safety timeout\'\n        if reason:\n            child.terminate()\n            try:child.wait(timeout=5)\n            except subprocess.TimeoutExpired:child.kill();child.wait()\n            print(\'stopped: \'+reason,file=sys.stderr);return 2\n        time.sleep(.08)\n    return child.returncode\n\nif __name__==\'__main__\':raise SystemExit(main())\n'
# ---- embedded module: renderello/cpu_mask.py ----
path = base / 'renderello/cpu_mask.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Append-only local credit reservations. Not a grant of spending authority.\n\nReserve a source-grounded quote BEFORE submission. A possibly submitted request\nkeeps its full reservation until actual charge or verified non-submission is known.\nNo retry, top-up, API call or permission inference is performed here.\n"""\nfrom __future__ import annotations\nimport hashlib\nimport json\nimport math\nimport os\nfrom pathlib import Path\nfrom .matched_trial import _canonical, _identifier, _number, read_json\n\n\nclass CreditLedger:\n    def __init__(self, root, *, cap=None, authority_reference=None):\n        self.root = Path(root)\n        self.meta = self.root / \'budget.json\'\n        if not self.root.exists():\n            if type(cap) is not int or cap < 0:\n                raise ValueError(\'explicit nonnegative integer approved cap required\')\n            if not isinstance(authority_reference, str) or not authority_reference.strip():\n                raise ValueError(\'original approval reference required; this does not authenticate it\')\n            self.root.mkdir(parents=True)\n            self.meta.write_bytes(_canonical(dict(cap=cap, authority_reference=authority_reference)))\n        if self.root.is_symlink():\n            raise ValueError(\'ledger symlink refused\')\n        self.config = read_json(self.meta)\n        if type(self.config.get(\'cap\')) is not int or self.config[\'cap\'] < 0:\n            raise ValueError(\'invalid ledger cap\')\n        if cap is not None and cap != self.config[\'cap\']:\n            raise ValueError(\'cannot silently change approved cap\')\n\n    def _read(self):\n        events = []\n        previous = \'0\' * 64\n        paths = sorted(self.root.glob(\'event-*.json\'))\n        for i, p in enumerate(paths, 1):\n            if p.name != f\'event-{i:06d}.json\':\n                raise ValueError(\'missing credit-ledger event\')\n            e = read_json(p)\n            digest = e.pop(\'digest\')\n            if e.get(\'previous\') != previous or hashlib.sha256(_canonical(e)).hexdigest() != digest:\n                raise ValueError(\'credit-ledger hash chain mismatch\')\n            previous = digest\n            events.append(e)\n        return events, previous\n\n    def _state(self):\n        events, previous = self._read()\n        jobs = {}\n        for e in events:\n            ident = e[\'job\']\n            if e[\'kind\'] == \'reserve\':\n                if ident in jobs:\n                    raise ValueError(\'duplicate reservation\')\n                jobs[ident] = dict(e, status=\'reserved\', charged=None)\n            elif e[\'kind\'] == \'submitted\':\n                if ident not in jobs or jobs[ident][\'status\'] != \'reserved\':\n                    raise ValueError(\'submission state conflict\')\n                jobs[ident].update(status=\'submitted\', project_id=e[\'project_id\'])\n            elif e[\'kind\'] == \'settle\':\n                if ident not in jobs or jobs[ident][\'status\'] not in {\'reserved\', \'submitted\'}:\n                    raise ValueError(\'settlement state conflict\')\n                jobs[ident].update(status=\'settled\', charged=e[\'charged\'])\n            else:\n                raise ValueError(\'unknown ledger event\')\n        return events, previous, jobs\n\n    def snapshot(self):\n        _, _, jobs = self._state()\n        committed = sum(j[\'charged\'] if j[\'status\'] == \'settled\' else j[\'quote\'] for j in jobs.values())\n        return dict(cap=self.config[\'cap\'], committed_or_reserved=committed,\n                    remaining=self.config[\'cap\'] - committed,\n                    over_cap=committed > self.config[\'cap\'], jobs=jobs,\n                    authority=\'REFERENCE_ONLY_NOT_PERMISSION\')\n\n    def _append(self, make_event):\n        lock = self.root / \'.writer.lock\'\n        fd = os.open(lock, os.O_CREAT | os.O_EXCL | os.O_WRONLY, 0o600)\n        try:\n            events, previous, jobs = self._state()\n            event = make_event(jobs)\n            event[\'previous\'] = previous\n            event[\'digest\'] = hashlib.sha256(_canonical(event)).hexdigest()\n            dest = self.root / f\'event-{len(events) + 1:06d}.json\'\n            with dest.open(\'xb\') as f:\n                f.write(_canonical(event));f.flush();os.fsync(f.fileno())\n            return self.snapshot()\n        finally:\n            os.close(fd)\n            lock.unlink()\n\n    def reserve(self, job, quote, *, cost_source, account_id, request_sha256):\n        _identifier(job);_number(quote, \'exact current credit quote\')\n        if not isinstance(cost_source, str) or not cost_source.strip():\n            raise ValueError(\'verified quote source required\')\n        if not isinstance(account_id, str) or not account_id.strip():\n            raise ValueError(\'verified intended account required\')\n        if not isinstance(request_sha256, str) or len(request_sha256) != 64 or any(c not in \'0123456789abcdef\' for c in request_sha256):\n            raise ValueError(\'exact request SHA256 required\')\n        def event(jobs):\n            if job in jobs:\n                raise ValueError(\'job exists; recover it instead of resubmitting\')\n            committed = sum(j[\'charged\'] if j[\'status\'] == \'settled\' else j[\'quote\'] for j in jobs.values())\n            if any(j[\'status\']==\'settled\' and j[\'charged\']>j[\'quote\'] for j in jobs.values()):\n                raise ValueError(\'unexpected overcharge: manual budget review required\')\n            if committed + quote > self.config[\'cap\']:\n                raise ValueError(\'approved credit cap would be exceeded\')\n            if any(j[\'status\'] != \'settled\' for j in jobs.values()):\n                raise ValueError(\'unsettled submission: recover before new spending\')\n            if jobs and any(j[\'account_id\'] != account_id for j in jobs.values()):\n                raise ValueError(\'ledger cannot change spending account\')\n            return dict(kind=\'reserve\',job=job,quote=quote,cost_source=cost_source,\n                        account_id=account_id,request_sha256=request_sha256)\n        return self._append(event)\n\n    def submitted(self, job, project_id):\n        _identifier(job)\n        if not isinstance(project_id, str) or not project_id.strip():\n            raise ValueError(\'provider project ID required\')\n        def event(jobs):\n            if job not in jobs or jobs[job][\'status\'] != \'reserved\':\n                raise ValueError(\'reservation required\')\n            return dict(kind=\'submitted\',job=job,project_id=project_id)\n        return self._append(event)\n\n    def settle(self, job, charged, *, receipt_reference):\n        _identifier(job);_number(charged, \'verified actual charged credits\')\n        if not isinstance(receipt_reference, str) or not receipt_reference.strip():\n            raise ValueError(\'actual charge or verified non-submission evidence required\')\n        def event(jobs):\n            if job not in jobs or jobs[job][\'status\'] not in {\'reserved\', \'submitted\'}:\n                raise ValueError(\'unsettled reservation required\')\n            # Record an unexpected overcharge honestly, then block later spends.\n            return dict(kind=\'settle\',job=job,charged=charged,receipt_reference=receipt_reference)\n        return self._append(event)\n'
# ---- embedded module: renderello/credit_ledger.py ----
path = base / 'renderello/credit_ledger.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Subject cutout. Backend order: BiRefNet (MIT) -> RMBG-2.0 (restricted license,\nonly if explicitly enabled) -> OpenCV GrabCut (CPU fallback, no download).\n\nModel backends lazy-load and run on the resolved device; the GrabCut fallback\nkeeps the whole pipeline runnable and testable on CPU with zero downloads. Matte\nquality differs by backend and is recorded on the CutoutResult for QA honesty.\n"""\nfrom __future__ import annotations\n\nimport warnings\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nfrom PIL import Image\n\nfrom .config import BIREFNET_MODEL_ID, RMBG2_MODEL_ID, CACHE_DIR\nfrom .schemas import CutoutResult\n\n\ndef _bbox_from_matte(matte: np.ndarray) -> tuple[int, int, int, int]:\n    # Bounding geometry must contain every saved nonzero alpha pixel. A display\n    # threshold here silently discards soft edges later when compose crops.\n    ys, xs = np.where(matte > 0)\n    if len(xs) == 0:\n        raise ValueError("empty cutout matte - no product found in source image")\n    x0, x1, y0, y1 = xs.min(), xs.max(), ys.min(), ys.max()\n    return int(x0), int(y0), int(x1 - x0 + 1), int(y1 - y0 + 1)\n\n\ndef _save_cutout(img_rgb: np.ndarray, matte: np.ndarray, out_dir: Path,\n                 stem: str, backend: str) -> CutoutResult:\n    out_dir.mkdir(parents=True, exist_ok=True)\n    rgba = np.dstack([img_rgb, matte])\n    rgba_path = out_dir / f"{stem}_cutout.png"\n    matte_path = out_dir / f"{stem}_matte.png"\n    Image.fromarray(rgba, "RGBA").save(rgba_path)\n    Image.fromarray(matte, "L").save(matte_path)\n    return CutoutResult(\n        rgba_path=rgba_path, matte_path=matte_path,\n        bbox=_bbox_from_matte(matte), backend=backend,\n        coverage=float((matte > 127).mean()),\n    )\n\n\ndef grabcut_matte(img_rgb: np.ndarray, iters: int = 8, *, max_pixels: int = 4_000_000) -> np.ndarray:\n    """CPU fallback. Two seedings: a centered border-rect for normal framing,\n    then a border-color mask for products touching the frame edge (the rect\n    seed alone starves GrabCut of foreground there). Raises ValueError when no\n    product can be isolated - loud failure beats a silently blank matte."""\n    h, w = img_rgb.shape[:2]\n    # OpenCV\'s graph allocation can kill the process before Python can record\n    # a failure. Keep a declared CPU working budget, never silently resize the\n    # photograph or upscale a proxy matte. Supplied native alpha bypasses this\n    # fallback, while explicit callers can choose a different allocation budget.\n    if type(max_pixels) is not int or max_pixels <= 0:\n        raise ValueError("positive integer GrabCut pixel budget required")\n    if h * w > max_pixels:\n        raise ValueError(f"CPU GrabCut allocation budget exceeded: {w}x{h} > {max_pixels} pixels; "\n                         "supply a reviewed native-size alpha or choose an explicit model backend; "\n                         "source was not resized")\n    bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)\n\n    def _rect_pass() -> np.ndarray:\n        mask = np.zeros((h, w), np.uint8)\n        rect = (max(1, w // 50), max(1, h // 50), w - 2 * max(1, w // 50),\n                h - 2 * max(1, h // 50))\n        bgd, fgd = np.zeros((1, 65), np.float64), np.zeros((1, 65), np.float64)\n        cv2.grabCut(bgr, mask, rect, bgd, fgd, iters, cv2.GC_INIT_WITH_RECT)\n        return np.where((mask == cv2.GC_FGD) | (mask == cv2.GC_PR_FGD),\n                        255, 0).astype(np.uint8)\n\n    def _border_pass() -> np.ndarray:\n        border = np.concatenate([\n            img_rgb[:5].reshape(-1, 3), img_rgb[-5:].reshape(-1, 3),\n            img_rgb[:, :5].reshape(-1, 3), img_rgb[:, -5:].reshape(-1, 3)])\n        med = np.median(border, axis=0)\n        dist = np.linalg.norm(img_rgb.astype(np.float32) - med, axis=2)\n        mask = np.where(dist > 25, cv2.GC_PR_FGD, cv2.GC_PR_BGD).astype(np.uint8)\n        bgd, fgd = np.zeros((1, 65), np.float64), np.zeros((1, 65), np.float64)\n        cv2.grabCut(bgr, mask, None, bgd, fgd, iters, cv2.GC_INIT_WITH_MASK)\n        return np.where((mask == cv2.GC_FGD) | (mask == cv2.GC_PR_FGD),\n                        255, 0).astype(np.uint8)\n\n    matte = _rect_pass()\n    if (matte > 127).mean() < 0.01:\n        matte = _border_pass()\n    if (matte > 127).mean() < 0.005:\n        raise ValueError("empty cutout matte - no product found in source image")\n    # Clean speckle noise and fill pinholes.\n    k = np.ones((5, 5), np.uint8)\n    matte = cv2.morphologyEx(matte, cv2.MORPH_OPEN, k)\n    matte = cv2.morphologyEx(matte, cv2.MORPH_CLOSE, k, iterations=2)\n    # Feather edges 1px for compositing.\n    return cv2.GaussianBlur(matte, (3, 3), 0)\n\n\ndef _u2netp_entry(img_rgb: np.ndarray, device: str = "cpu") -> np.ndarray:\n    from renderello.onnx_matte import u2netp_onnx_matte\n    return u2netp_onnx_matte(img_rgb)\n\n\ndef _load_birefnet(device: str):\n    from transformers import AutoModelForImageSegmentation  # lazy, GPU host\n    kwargs: dict = {}\n    if not device.startswith("cuda"):\n        # Published BiRefNet weights are fp16; on CPU the default load keeps\n        # fp16 biases while inputs are float32 and inference dies with\n        # "Input type (float) and bias type (c10::Half) should be the same".\n        import torch\n        kwargs["torch_dtype"] = torch.float32\n    model = AutoModelForImageSegmentation.from_pretrained(\n        BIREFNET_MODEL_ID, trust_remote_code=True, cache_dir=str(CACHE_DIR),\n        **kwargs)\n    model.to(device).eval()\n    if device.startswith("cuda"):\n        # fp16 halves VRAM and is ~2x faster; validated on Kaggle T4 x2\n        # (2026-11-09): 0.36 s/img steady-state, matte quality unchanged\n        # in the 5-photo comparison test.\n        model.half()\n    return model\n\n\ndef birefnet_matte(img_rgb: np.ndarray, device: str = "cpu") -> np.ndarray:\n    """BiRefNet inference (MIT, github.com/ZhengPeng7/BiRefNet). Input normalized\n    per the repo\'s inference recipe (1024x1024, ImageNet stats)."""\n    import torch\n    from torchvision import transforms\n    model = _load_birefnet(device)\n    tf = transforms.Compose([\n        transforms.ToPILImage(), transforms.Resize((1024, 1024)),\n        transforms.ToTensor(),\n        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),\n    ])\n    x = tf(img_rgb).unsqueeze(0).to(device)\n    if device.startswith("cuda"):\n        x = x.half()  # match model.half() above\n    with torch.inference_mode():\n        pred = model(x)[-1].sigmoid().float().cpu()\n    matte = (pred[0].squeeze().numpy() * 255).astype(np.uint8)\n    return cv2.resize(matte, (img_rgb.shape[1], img_rgb.shape[0]),\n                      interpolation=cv2.INTER_LINEAR)\n\n\ndef rmbg2_matte(img_rgb: np.ndarray, device: str = "cpu") -> np.ndarray:\n    """BRIA RMBG-2.0. License CC BY-NC 4.0 (non-commercial) - enabled only via\n    pipeline flag; restriction recorded in README/plan, owner decision pending."""\n    import torch\n    from transformers import AutoModelForImageSegmentation\n    from torchvision import transforms\n    model = AutoModelForImageSegmentation.from_pretrained(\n        RMBG2_MODEL_ID, trust_remote_code=True, cache_dir=str(CACHE_DIR))\n    model.to(device).eval()\n    tf = transforms.Compose([\n        transforms.ToPILImage(), transforms.Resize((1024, 1024)),\n        transforms.ToTensor(),\n        transforms.Normalize([0.5, 0.5, 0.5], [1.0, 1.0, 1.0]),\n    ])\n    with torch.inference_mode():\n        pred = model(tf(img_rgb).unsqueeze(0).to(device))[-1].sigmoid().cpu()\n    matte = (pred[0].squeeze().numpy() * 255).astype(np.uint8)\n    return cv2.resize(matte, (img_rgb.shape[1], img_rgb.shape[0]),\n                      interpolation=cv2.INTER_LINEAR)\n\n\n_BACKENDS = {"grabcut": grabcut_matte, "birefnet": birefnet_matte,\n             "rmbg2": rmbg2_matte, "u2netp_onnx": _u2netp_entry}\n\n\ndef cutout(source: Image.Image, out_dir: Path, stem: str = "product",\n           backend: str = "auto", device: str = "cpu",\n           allow_restricted: bool = False) -> CutoutResult:\n    """Produce an RGBA cutout + matte. backend=\'auto\' picks the best available: birefnet when torch is\n    installed, else u2netp_onnx when onnxruntime is installed, else grabcut. Restricted-license backends must\n    be requested explicitly (allow_restricted)."""\n    img_rgb = np.array(source.convert("RGB"))\n    order: list[str]\n    if backend == "auto":\n        order = []\n        try:\n            import torch  # noqa: F401\n            order.append("birefnet")\n        except ImportError:\n            pass\n        try:\n            import onnxruntime  # noqa: F401\n            order.append("u2netp_onnx")\n        except ImportError:\n            pass\n        order.append("grabcut")\n    else:\n        order = [backend]\n    if any(b == "rmbg2" for b in order) and not allow_restricted:\n        raise PermissionError("RMBG-2.0 is CC BY-NC 4.0; pass allow_restricted=True "\n                              "only after the owner resolves the license question")\n    last_err: Exception | None = None\n    for name in order:\n        try:\n            matte = _BACKENDS[name](img_rgb, device) if name != "grabcut" \\\n                else _BACKENDS[name](img_rgb)\n            return _save_cutout(img_rgb, matte, out_dir, stem, name)\n        except Exception as e:  # fall through to next backend\n            last_err = e\n            if backend != "auto":\n                raise\n            remaining = order[order.index(name) + 1:]\n            if remaining:\n                warnings.warn(\n                    f"cutout backend \'{name}\' failed "\n                    f"({type(e).__name__}: {e}); falling back to "\n                    f"\'{remaining[0]}\' - matte quality is reduced (crude "\n                    f"edges, not publish-quality). Fix the error above to "\n                    f"use the learned backend.",\n                    RuntimeWarning, stacklevel=2)\n    raise RuntimeError(f"all cutout backends failed; last error: {last_err}")\n\n\ndef supplied_mask_cutout(source: Image.Image, mask_path: Path, out_dir: Path,\n                         stem: str = "product") -> CutoutResult:\n    """Use explicit alpha with no model/fallback. Never resize a misaligned mask.\n    White=foreground, black=background; grayscale values are soft opacity.\n    Mask correctness still requires eye review; source RGB is not edited.\n    """\n    with Image.open(mask_path) as image:\n        if image.mode != "L":\n            raise ValueError("supplied mask must be an 8-bit grayscale L-mode image")\n        if image.size != source.size:\n            raise ValueError("supplied mask must exactly match source dimensions; no auto-resize")\n        matte = np.array(image)\n    if not np.any(matte > 127):\n        raise ValueError("supplied mask has no foreground")\n    if not np.any(matte < 128):\n        raise ValueError("supplied mask has no background")\n    return _save_cutout(np.array(source.convert("RGB")), matte, out_dir, stem, "supplied_mask")\n'
# ---- embedded module: renderello/cutout.py ----
path = base / 'renderello/cutout.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Conservative local-detail tripwire, not OCR or a product-identity verifier.\n\nWhole-product averages can hide a small relabeled area. Compare high-frequency\nluminance in overlapping, well-supported interior patches. Uniform brightness\nchanges are largely removed; resampling, misalignment and genuine texture edits\ncan still trigger review. Thresholds are provisional, not benchmark-calibrated.\n"""\nfrom __future__ import annotations\nimport cv2\nimport numpy as np\n\n\ndef detail_change_review(source, output, matte):\n    for image in (source, output):\n        if not isinstance(image, np.ndarray) or image.dtype != np.uint8 or image.ndim != 3 or image.shape[2] != 3:\n            raise ValueError(\'aligned native uint8 RGB arrays required\')\n    if source.shape != output.shape:\n        raise ValueError(\'aligned image dimensions must match; no implicit resampling\')\n    if not isinstance(matte, np.ndarray) or matte.dtype != np.uint8 or matte.shape != source.shape[:2]:\n        raise ValueError(\'aligned uint8 matte required\')\n    height, width = matte.shape\n    if height * width > 4_000_000:\n        raise ValueError(\'detail review limited to four million aligned pixels\')\n    report = dict(schema=\'local-detail-review.v1\', evaluated_patches=0,\n                  flagged_patches=[], flagged_patch_count=0, status=\'INSUFFICIENT_DETAIL_EVIDENCE\',\n                  text_verified=False, publication_accepted=False,\n                  thresholds_provisional=True,\n                  limitations=[\'Local high-frequency change is not proof of changed text or logo.\',\n                               \'Misalignment, blur and resampling may trigger review.\',\n                               \'No flag does not establish label fidelity or publication readiness.\'])\n    if min(height, width) < 32:\n        return report\n    gray = [cv2.cvtColor(image, cv2.COLOR_RGB2GRAY).astype(np.float32)\n            for image in (source, output)]\n    high = [g - cv2.GaussianBlur(g, (0, 0), 1.2) for g in gray]\n    # Do not treat fractional alpha or source-background boundaries as label detail.\n    interior = cv2.erode((matte == 255).astype(np.uint8),\n                        np.ones((5, 5), np.uint8), borderType=cv2.BORDER_CONSTANT,\n                        borderValue=0).astype(bool)\n    def positions(length):\n        return sorted(set(range(0, length - 31, 16)) | {length - 32})\n    flagged = []\n    for y in positions(height):\n        for x in positions(width):\n            section = np.s_[y:y + 32, x:x + 32]\n            valid = interior[section]\n            if valid.mean() < .9:\n                continue\n            energy = float(np.abs(high[0][section][valid]).mean())\n            if energy < 2:\n                continue\n            report[\'evaluated_patches\'] += 1\n            difference = np.abs(high[0][section] - high[1][section])[valid]\n            error = float(difference.mean())\n            changed = float((difference > 12).mean())\n            if error > 10 and changed > .25:\n                flagged.append(dict(box=[x, y, 32, 32], highpass_mae=round(error, 4),\n                                    changed_detail_fraction=round(changed, 4),\n                                    source_detail_energy=round(energy, 4)))\n    # Store bounded evidence, retaining the strongest changes rather than an\n    # unbounded set of near-duplicate overlapping boxes.\n    flagged.sort(key=lambda item: (-item[\'highpass_mae\'], item[\'box\']))\n    report[\'flagged_patch_count\'] = len(flagged)\n    report[\'flagged_patches\'] = flagged[:24]\n    if report[\'evaluated_patches\']:\n        report[\'status\'] = \'LOCAL_DETAIL_CHANGE_REVIEW_REQUIRED\' if flagged else \'NO_LOCAL_TRIPWIRE_NOT_A_PASS\'\n    return report\n'
# ---- embedded module: renderello/detail_qa.py ----
path = base / 'renderello/detail_qa.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Empty-scene prompt construction for the background-only generation route.\n\nThe scene model receives ONLY the environment. The product is never named,\ndescribed or implied anywhere in the prompt: a prompt that describes the\nproduct for a paragraph and then appends "no product" is contradictory, and\nmodels follow the dominant description - that is the measured 20 Nov 2026\nduplicate-product failure (a hallucinated second bottle behind the preserved\ncutout).\n\nDetailed prompts are kept LONG by design: length is not the defect,\ncontradiction is. All the detail describes the\nempty set - surface, wall, horizon, lighting, camera, material treatment,\nmood and the unobstructed placement area. Product-fidelity instructions live\nin the separate product-edit prompts (whole/hybrid routes), never here.\n"""\n\n# Hard exclusion block, grounded in the archived 13 Nov 2026 empty-scene\n# request (scoped/scripts/Renderello-MagicHour-empty-scene-pipeline-2146.py),\n# generalized beyond one product type.\n_EXCLUSIONS = (\n    "Completely empty set: no product, no bottle, no packaging, no pedestal, "\n    "no platform, no plinth, no riser, no props, no merchandise, no text, no "\n    "logo, no label, no object silhouette, and no cast shadow from an absent "\n    "object. Do not render any object. The placement area stays completely "\n    "unobstructed for later photographic placement of the real item."\n)\n\n_ASPECT_WORDS = {\n    "1:1": "Square commercial photograph",\n    "4:5": "Vertical 4:5 commercial photograph",\n    "3:4": "Vertical 3:4 commercial photograph",\n    "16:9": "Wide 16:9 commercial photograph",\n    "9:16": "Tall 9:16 commercial photograph",\n}\n\n\nclass EmptyScenePrompt(str):\n    """Marker type: composite_with_scene(conditioning=\\"empty_guide\\") only\n    accepts this, so a hand-written product-bearing prompt with a \\"no\n    product\\" tail (the 20 Nov 2026 failure shape) can never reach the\n    scene model through that path."""\n\n\ndef build_empty_scene_prompt(template, *, aspect_key: str = "1:1",\n                             horizon_frac: float = 0.60,\n                             instruction: str = "") -> str:\n    """Environment-only scene prompt from a template\'s declared set design.\n\n    template carries scene/lighting/camera/mood text (TemplateSpec fields).\n    horizon_frac is the floor-wall division as a fraction of image height,\n    matched to the two-tone conditioning guide the model also receives.\n    instruction is the user\'s environment note; it is included only after\n    product-naming terms are refused, keeping the prompt non-contradictory.\n    """\n    if not 0.3 <= horizon_frac <= 0.8:\n        raise ValueError("horizon fraction outside the usable set range")\n    scene = getattr(template, "scene", "") or "seamless studio set"\n    lighting = getattr(template, "lighting", "") or "soft diffused studio lighting"\n    camera = getattr(template, "camera", "") or "straight-on product-photography camera height"\n    mood = getattr(template, "mood", "") or "calm commercial finish"\n    opener = _ASPECT_WORDS.get(aspect_key, "Commercial photograph")\n    pct = round(horizon_frac * 100)\n    parts = [\n        f"{opener} of a completely EMPTY premium product-photography set.",\n        "The supplied two-tone guide only indicates the floor-wall division; "\n        "it is not an object and nothing sits on the surface.",\n        f"Set: {scene}.",\n        f"The floor-wall junction sits at approximately {pct} percent of "\n        "image height, with the support surface continuous, level and "\n        "uninterrupted across the full foreground.",\n        f"Lighting: {lighting}.",\n        f"Camera: {camera}.",\n        f"Finish: {mood}.",\n        _EXCLUSIONS,\n    ]\n    note = (instruction or "").strip()\n    if note:\n        lowered = note.lower()\n        for term in ("product", "bottle", "label", "logo", "packag",\n                     "jar", "tube", "box", "cream", "perfume", "spray"):\n            if term in lowered:\n                raise ValueError(\n                    "environment instruction names the product; keep product "\n                    "detail out of the empty-scene prompt")\n        parts.append(f"Additional environment-only note: {note}.")\n    return EmptyScenePrompt(" ".join(parts))\n'
# ---- embedded module: renderello/empty_scene_prompt.py ----
path = base / 'renderello/empty_scene_prompt.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Recoverable, no-overwrite pair publication. A marker commits both hashes.\n\nThree filenames cannot be atomically committed together. Consumers must validate\nbefore use. Staging and partial destinations stay on failure for explicit recovery.\n"""\nfrom pathlib import Path\nimport hashlib,json,os,tempfile\n\nSCHEMA=\'evidence-pair.v1\'\n\n\ndef _bytes(value):\n    return (json.dumps(value,indent=2,allow_nan=False)+\'\\n\').encode()\n\n\ndef _hash(data):return hashlib.sha256(data).hexdigest()\n\n\ndef _read(path):\n    if path.is_symlink() or not path.is_file() or path.stat().st_size>10_000_000:\n        raise ValueError(\'bounded regular evidence file required: \'+str(path))\n    return path.read_bytes()\n\n\ndef _directory_sync_mode():\n    # Python\'s Windows CRT cannot open directories for fsync. Do not catch\n    # arbitrary PermissionError: those still indicate real failures on POSIX.\n    return \'file_fsync_only_windows\' if os.name == \'nt\' else \'file_and_directory_fsync\'\n\n\ndef _sync_dir(path):\n    if _directory_sync_mode() == \'file_fsync_only_windows\':\n        return\n    fd=os.open(path,os.O_RDONLY)\n    try:os.fsync(fd)\n    finally:os.close(fd)\n\n\ndef _write(path,data):\n    with path.open(\'xb\') as f:\n        f.write(data);f.flush();os.fsync(f.fileno())\n\n\ndef _paths(output):\n    return [output,Path(str(output)+\'.references.json\'),Path(str(output)+\'.complete.json\')]\n\n\ndef _valid_commit(commit, output):\n    required = {\'schema\', \'output\', \'payload_sha256\', \'references_sha256\'}\n    return (isinstance(commit, dict) and set(commit) in (required, required | {\'persistence\'})\n        and commit[\'schema\'] == SCHEMA and commit[\'output\'] == str(output)\n        and (\'persistence\' not in commit or commit[\'persistence\'] in\n             (\'file_fsync_only_windows\', \'file_and_directory_fsync\')))\n\n\ndef validate_pair(output):\n    output=Path(output).absolute();payload,refs,marker=_paths(output)\n    commit=json.loads(_read(marker))\n    if not _valid_commit(commit, output):\n        raise ValueError(\'invalid pair completion marker\')\n    if _hash(_read(payload))!=commit[\'payload_sha256\'] or _hash(_read(refs))!=commit[\'references_sha256\']:\n        raise ValueError(\'pair hash mismatch; do not consume\')\n    return commit\n\n\ndef recover_pair(stage,output):\n    stage=Path(stage).absolute();output=Path(output).absolute()\n    if stage.is_symlink() or not stage.is_dir() or stage.parent!=output.parent:\n        raise ValueError(\'same-directory regular staging directory required\')\n    commit=json.loads(_read(stage/\'commit.json\'))\n    if not _valid_commit(commit, output):\n        raise ValueError(\'staging destination/commit mismatch\')\n    sources=[stage/\'payload.json\',stage/\'references.json\',stage/\'commit.json\']\n    data=[_read(s) for s in sources]\n    if _hash(data[0])!=commit[\'payload_sha256\'] or _hash(data[1])!=commit[\'references_sha256\']:\n        raise ValueError(\'staged evidence hash mismatch\')\n    # Preflight ALL conflicts before publishing any remaining file. Never replace.\n    destinations=_paths(output)\n    for target,expected in zip(destinations,data):\n        if os.path.lexists(target) and _read(target)!=expected:\n            raise ValueError(\'existing evidence differs; preserve it: \'+str(target))\n    for source,target,expected in zip(sources,destinations,data):\n        try:os.link(source,target)\n        except FileExistsError:\n            if _read(target)!=expected:raise ValueError(\'concurrent evidence conflict; preserve it: \'+str(target))\n        _sync_dir(output.parent)\n    return validate_pair(output)\n\n\ndef export_pair(output,payload,references):\n    output=Path(output).absolute()\n    if any(os.path.lexists(p) for p in _paths(output)):\n        raise ValueError(\'preserve earlier output/reference/completion files\')\n    # Serialize both BEFORE staging, so invalid values cannot leave half a pair.\n    p,r=_bytes(payload),_bytes(references)\n    stage=Path(tempfile.mkdtemp(prefix=\'.\'+output.name+\'.stage-\',dir=output.parent))\n    try:\n        _write(stage/\'payload.json\',p);_write(stage/\'references.json\',r)\n        _write(stage/\'commit.json\',_bytes(dict(schema=SCHEMA,output=str(output),payload_sha256=_hash(p),references_sha256=_hash(r),persistence=_directory_sync_mode())))\n        _sync_dir(stage);_sync_dir(output.parent)\n        recover_pair(stage,output)\n    except (OSError,ValueError) as exc:\n        raise ValueError(\'incomplete export; retained staging \'+str(stage)+\'; validate/recover explicitly: \'+str(exc)) from exc\n    return stage\n'
# ---- embedded module: renderello/evidence_pair.py ----
path = base / 'renderello/evidence_pair.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Recorded executor class is not a quality, identity or spend attestation."""\n\n\ndef describe_execution(route, backend=\'\', *, scene_mock=False, reference=False):\n    if backend==\'stub\' and route in (\'pure_prompt\',\'hybrid\'):\n        kind=\'TEST_DOUBLE_NOT_GENERATIVE_MODEL\'\n        caption=\'TEST DOUBLE: deterministic test editor, not an AI product edit or model-quality result.\'\n    elif isinstance(backend,str) and backend.startswith(\'magichour:\') and backend.endswith(\':mock\'):\n        kind=\'TEST_DOUBLE_NOT_GENERATIVE_MODEL\'\n        caption=\'MOCK MAGIC HOUR: deterministic offline image echo, not a model-quality result or actual charge.\'\n    elif isinstance(backend,str) and backend.startswith(\'magichour:\') and backend.endswith(\':live\'):\n        kind=\'RECORDED_LIVE_EDITOR_CONFIGURATION\'\n        caption=\'Recorded live Magic Hour configuration; inspect provider receipts for execution/charges and images for quality.\'\n    elif route==\'scene_compose\' and scene_mock:\n        kind=\'PROCEDURAL_MOCK_SCENE\'\n        caption=\'MOCK SCENE: procedural gradient, not a generated photoshoot or real relighting.\'\n    elif route==\'scene_compose\' and reference:\n        kind=\'SAVED_REFERENCE_BACKDROP\'\n        caption=\'SAVED BACKDROP: uploaded image compositing, no new scene-generation model.\'\n    elif route==\'cutout_compose\':\n        kind=\'DETERMINISTIC_COMPOSITE\'\n        caption=\'DETERMINISTIC COMPOSITE: source alpha pasted onto a backdrop; not new product generation.\'\n    else:\n        kind=\'EXECUTOR_NOT_CLASSIFIED\'\n        caption=\'Execution type not classified here. Consult the recorded backend and run evidence.\'\n    return dict(kind=kind,caption=caption,quality_evidence=False,credits_verified=False)\n'
# ---- embedded module: renderello/execution_kind.py ----
path = base / 'renderello/execution_kind.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Opt-in expanded unknown-band alpha/color recovery with source-bound core locks.\nCPU research only. Derived trimap is a hypothesis, not semantic or optical truth.\nOnly the declared core stays source-exact; estimated edge RGB/alpha can change.\nUse a guarded worker for untrusted/large jobs, as renderello.matting provides.\n"""\nimport numpy as np\nfrom .candidate_diagnostics import native_rgb,native_alpha,digest\nfrom .matting import make_trimap,refine_arrays\n\n\ndef recover_expanded_edge(rgb,initial_alpha,protected,*,source_sha256,alpha_sha256,\n        material,review_note,reviewed=False,radius=8):\n    native_rgb(rgb);native_alpha(initial_alpha,rgb.shape[:2])\n    if initial_alpha.size>1_000_000:raise ValueError(\'1MP source budget; no resize\')\n    if reviewed is not True or material!=\'opaque\':raise ValueError(\'explicit opaque source pairing review required\')\n    if not isinstance(review_note,str) or not review_note.strip() or len(review_note)>2000:\n        raise ValueError(\'source/core review note required\')\n    if source_sha256!=digest(rgb) or alpha_sha256!=digest(initial_alpha):\n        raise ValueError(\'source/alpha binding mismatch\')\n    if not isinstance(protected,np.ndarray) or protected.dtype!=np.bool_ or protected.shape!=initial_alpha.shape or not protected.any():\n        raise ValueError(\'nonempty native boolean protected core required\')\n    if np.any(initial_alpha[protected]!=255):raise ValueError(\'protected core must already be fully opaque\')\n    if type(radius) is not int or not 1<=radius<=12:raise ValueError(\'integer1..12band radius required\')\n    trimap=make_trimap(initial_alpha,radius)\n    locked_from_unknown=int((protected&(trimap==128)).sum())\n    if np.any(protected&(trimap==0)):raise ValueError(\'reviewed core contradicts definite background\')\n    trimap[protected]=255\n    foreground,alpha,solver=refine_arrays(rgb.copy(),initial_alpha.copy(),trimap.copy(),method=\'cf\',edge_colors=True)\n    native_rgb(foreground);native_alpha(alpha,initial_alpha.shape)\n    if foreground.shape!=rgb.shape:raise ValueError(\'solver foreground changed source coordinates\')\n    if not np.array_equal(foreground[trimap!=128],rgb[trimap!=128]):\n        raise ValueError(\'solver changed definite foreground/background RGB\')\n    if not np.all(alpha[trimap==255]==255) or not np.all(alpha[trimap==0]==0):\n        raise ValueError(\'solver violated trimap alpha locks\')\n    if not np.array_equal(foreground[protected],rgb[protected]):\n        raise ValueError(\'protected source RGB changed\')\n    changed=np.any(foreground!=rgb,axis=2)\n    return foreground,alpha,trimap,dict(schema=\'expanded-edge-recovery.v1\',\n        source_sha256=digest(rgb),initial_alpha_sha256=digest(initial_alpha),\n        protected_sha256=digest(protected),foreground_sha256=digest(foreground),\n        output_alpha_sha256=digest(alpha),trimap_sha256=digest(trimap),\n        review_note=review_note,material=material,radius=radius,\n        protected_pixels=int(protected.sum()),protected_rgb_exact=True,\n        protected_relocked_from_unknown=locked_from_unknown,solver=solver,\n        rgb_changed_pixels=int(changed.sum()),alpha_changed_pixels=int((alpha!=initial_alpha).sum()),\n        initially_opaque_rgb_changed=int((changed&(initial_alpha==255)).sum()),\n        source_resampled=False,fully_source_rgb_exact=not bool(changed.any()),\n        optical_recovery=False,physical_relighting=False,independent_ground_truth=False,\n        accepted=False,default_pipeline_changed=False,\n        limitations=[\'Input alpha and trimap seeds may be wrong; a wider band can erase detail or include background.\',\n          \'Foreground color estimation uses an internal pyramid. Native output dimensions do not mean every internal calculation is native resolution.\',\n          \'Protected core is a reviewer declaration, not independent label verification.\',\n          \'No geometry/pose/lighting/glass or cropped-part recovery.\'])\n'
# ---- embedded module: renderello/expanded_edge_recovery.py ----
path = base / 'renderello/expanded_edge_recovery.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Read-only preflight for local cases x seeds image experiments.\n\nValidates declared execution metadata, never source rights, quality, permission,\nprovider cost or successful execution. Frozen evidence is never repaired here.\n"""\nfrom __future__ import annotations\nimport argparse\nimport hashlib\nimport json\nfrom io import BytesIO\nfrom pathlib import Path\nfrom PIL import Image\nfrom .matched_trial import read_json, _relative\n\n\ndef _int(value, label, low, high):\n    if type(value) is not int or not low <= value <= high:\n        raise ValueError(label + \': bounded integer required\')\n    return value\n\n\ndef _text(value, label):\n    if not isinstance(value, str) or not value.strip() or len(value) > 32000:\n        raise ValueError(label + \': nonempty bounded text required\')\n    return value\n\n\ndef _pair(value, label, low, high):\n    if not isinstance(value, list) or len(value) != 2:\n        raise ValueError(label + \': two integers required\')\n    return [_int(x, label, low, high) for x in value]\n\n\ndef validate_plan(plan, root):\n    """Derive output count and audit source bytes/canvases, with no mutations.\n\n    Source-shaped canvases are allowed, but a conflicting template aspect is\n    reported as unmet, never silently counted as fulfilled. Multiple template\n    cases can share one source. Extra provenance rows are not executed sources.\n    """\n    if not isinstance(plan, dict):\n        raise ValueError(\'plan object required\')\n    cases = plan.get(\'cases\'); seeds = plan.get(\'seeds\'); sources = plan.get(\'sources\')\n    if not isinstance(cases, list) or not 1 <= len(cases) <= 500:\n        raise ValueError(\'one to 500 cases required\')\n    if not isinstance(seeds, list) or not 1 <= len(seeds) <= 16:\n        raise ValueError(\'one to 16 seeds required\')\n    for s in seeds: _int(s, \'seed\', 0, 2**63-1)\n    if len(set(seeds)) != len(seeds):\n        raise ValueError(\'duplicate seeds\')\n    count = len(cases)*len(seeds)\n    if _int(plan.get(\'planned_outputs\'), \'planned_outputs\', 1, 8000) != count:\n        raise ValueError(f\'planned_outputs contradicts cases x seeds: expected {count}\')\n    if not isinstance(sources, list) or not 1 <= len(sources) <= 500:\n        raise ValueError(\'source records required\')\n    records = {}\n    for row in sources:\n        if not isinstance(row, dict): raise ValueError(\'source object required\')\n        sid = _text(row.get(\'id\'), \'source id\')\n        if sid in records: raise ValueError(\'duplicate source id\')\n        records[sid] = row\n    rows=[];seen=set();used=set();cells=set()\n    root=Path(root)\n    for c in cases:\n        if not isinstance(c,dict): raise ValueError(\'case object required\')\n        cid=_text(c.get(\'id\'),\'case id\');sid=_text(c.get(\'source_id\'),\'case source id\')\n        if cid in seen: raise ValueError(\'duplicate case id\')\n        seen.add(cid);used.add(sid)\n        if sid not in records: raise ValueError(\'case has no source record\')\n        record=records[sid]\n        if c.get(\'source\') != record.get(\'input_file\'):\n            raise ValueError(\'case/source record path mismatch\')\n        path=_relative(root,c.get(\'source\'))\n        if path.stat().st_size>64_000_000: raise ValueError(\'source file exceeds limit\')\n        source_bytes=path.read_bytes()\n        if len(source_bytes)>64_000_000: raise ValueError(\'source file exceeds limit\')\n        digest=hashlib.sha256(source_bytes).hexdigest()\n        if digest != record.get(\'input_file_sha256\'):\n            raise ValueError(\'source bytes do not match input_file_sha256\')\n        with Image.open(BytesIO(source_bytes)) as im:\n            if im.format not in {\'PNG\',\'JPEG\',\'WEBP\'} or getattr(im,\'n_frames\',1)!=1:\n                raise ValueError(\'single-frame raster source required\')\n            size=list(im.size)\n            if size[0]*size[1]>40_000_000: raise ValueError(\'source pixel limit exceeded\')\n            if im.getexif().get(274,1)!=1: raise ValueError(\'normalize orientation before freezing\')\n            im.load()\n        with Image.open(BytesIO(source_bytes)) as check:\n            check.verify()\n        if size != _pair(record.get(\'decoded_size\'),\'decoded size\',1,16384):\n            raise ValueError(\'decoded source size mismatch\')\n        width=_int(c.get(\'width\'),\'width\',32,8192);height=_int(c.get(\'height\'),\'height\',32,8192)\n        if width*height>40_000_000 or width%32 or height%32:\n            raise ValueError(\'bounded multiple-of-32 canvas required\')\n        if [width,height] != _pair(record.get(\'output_canvas\'),\'output canvas\',32,8192):\n            raise ValueError(\'source/case canvas mismatch\')\n        xy=_pair(c.get(\'translation\'),\'translation\',0,8192)\n        if xy != _pair(record.get(\'native_translation\'),\'native translation\',0,8192):\n            raise ValueError(\'source/case translation mismatch\')\n        if record.get(\'source_resampled\') is not False or record.get(\'canvas_only_padding\') is not True:\n            raise ValueError(\'explicit native-padding declarations required\')\n        if xy[0]+size[0]>width or xy[1]+size[1]>height:\n            raise ValueError(\'native source extends outside canvas\')\n        prompt=_text(c.get(\'prompt\'),\'prompt\');template=c.get(\'template\')\n        if not isinstance(template,dict): raise ValueError(\'template object required\')\n        key=_text(template.get(\'key\'),\'template key\');aspect=template.get(\'aspect\')\n        if not isinstance(aspect,str) or aspect.count(\':\')!=1:\n            raise ValueError(\'integer template aspect required\')\n        try: aw,ah=(int(x) for x in aspect.split(\':\'))\n        except (TypeError,ValueError): raise ValueError(\'integer template aspect required\') from None\n        if not 1<=aw<=100 or not 1<=ah<=100: raise ValueError(\'bounded positive aspect required\')\n        cell=(sid,key,prompt,width,height)\n        if cell in cells: raise ValueError(\'duplicate logical case\')\n        cells.add(cell)\n        rows.append(dict(case_id=cid,source_id=sid,input_sha256=digest,source_size=size,\n                         canvas=[width,height],source_resampled=False,\n                         requested_aspect=aspect,requested_aspect_fulfilled=width*ah==height*aw))\n    return dict(schema=\'renderello.experiment-preflight.v1\',validation=\'PASSED\',\n                planned_outputs=count,case_count=len(cases),seed_count=len(seeds),\n                executed_source_ids=sorted(used),unused_provenance_source_ids=sorted(set(records)-used),\n                case_checks=rows,all_template_aspects_fulfilled=all(r[\'requested_aspect_fulfilled\'] for r in rows),\n                generation_executed=False,quality_accepted=False,source_rights_verified=False,\n                execution_authorized=False,cost_verified=False,files_modified=False)\n\n\ndef main(argv=None):\n    p=argparse.ArgumentParser(description=__doc__);p.add_argument(\'plan\');p.add_argument(\'--root\',required=True)\n    a=p.parse_args(argv)\n    try: result=validate_plan(read_json(a.plan),a.root)\n    except (ValueError,KeyError,OSError,Image.DecompressionBombError) as exc:\n        print(json.dumps(dict(validation=\'REJECTED\',error=str(exc),generation_executed=False,files_modified=False)))\n        return 2\n    print(json.dumps(result,indent=2));return 0\n\n\nif __name__==\'__main__\': raise SystemExit(main())\n'
# ---- embedded module: renderello/experiment_preflight.py ----
path = base / 'renderello/experiment_preflight.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Opt-in native alpha annotations. No semantic inference or quality approval.\n\nA source review note records the annotator\'s rationale; its presence does not\nprove correctness. Restoring opacity reuses photographed RGB, including any old\nscene contamination. Callers must inspect the resulting composites.\n"""\nimport hashlib\nimport numpy as np\nfrom PIL import Image, ImageDraw\n\n\ndef _orientation(a,b,c):\n    return (b[0]-a[0])*(c[1]-a[1])-(b[1]-a[1])*(c[0]-a[0])\n\n\ndef _intersects(a,b,c,d):\n    def on(a,b,c):\n        return _orientation(a,b,c)==0 and min(a[0],b[0])<=c[0]<=max(a[0],b[0]) and min(a[1],b[1])<=c[1]<=max(a[1],b[1])\n    x,y,z,t=_orientation(a,b,c),_orientation(a,b,d),_orientation(c,d,a),_orientation(c,d,b)\n    return ((x>0)!=(y>0) and (z>0)!=(t>0)) or on(a,b,c) or on(a,b,d) or on(c,d,a) or on(c,d,b)\n\n\ndef _polygon(points,width,height):\n    if not isinstance(points,(list,tuple)) or len(points)<3:\n        raise ValueError(\'polygon needs at least three vertices\')\n    for p in points:\n        if not isinstance(p,(list,tuple)) or len(p)!=2 or any(type(x)!=int for x in p):\n            raise ValueError(\'integer source coordinates required\')\n        if not (0<=p[0]<width and 0<=p[1]<height):\n            raise ValueError(\'polygon outside source\')\n    if len(set(map(tuple,points)))!=len(points):raise ValueError(\'repeated vertex\')\n    edges=list(zip(points,points[1:]+points[:1]))\n    if sum(a[0]*b[1]-b[0]*a[1] for a,b in edges)==0:raise ValueError(\'zero area polygon\')\n    for i,(a,b) in enumerate(edges):\n        for j,(c,d) in enumerate(edges):\n            if j<=i or j==i+1 or (i==0 and j==len(edges)-1):continue\n            if _intersects(a,b,c,d):raise ValueError(\'self-intersecting polygon\')\n    im=Image.new(\'L\',(width,height));ImageDraw.Draw(im).polygon(points,fill=255)\n    return np.asarray(im)>0\n\n\ndef apply_explicit_alpha_patches(alpha, protected_regions, patches):\n    """Return a new L alpha and exact delta ledger. Never modifies input/defaults.\n\n    Patches: {operation: \'exclude\'|\'restore_opaque\', polygon: [[x,y],...],\n              source_review_note: nonempty str}. Overlap and protected-region\n    intersection are refused. Native pixel coordinates include polygon borders.\n    """\n    if not isinstance(alpha,Image.Image) or alpha.mode!=\'L\':raise ValueError(\'native L alpha required\')\n    a=np.asarray(alpha);height,width=a.shape\n    if width*height>4_000_000:raise ValueError(\'source too large\')\n    if not isinstance(protected_regions,dict) or not protected_regions:raise ValueError(\'named protected regions required\')\n    protected=np.zeros(a.shape,bool)\n    for name,box in protected_regions.items():\n        if not isinstance(name,str) or not name.strip() or not isinstance(box,(list,tuple)) or len(box)!=4 or any(type(x)!=int for x in box):raise ValueError(\'invalid protected region\')\n        x1,y1,x2,y2=box\n        if not (0<=x1<x2<=width and 0<=y1<y2<=height):raise ValueError(\'invalid protected region bounds\')\n        if np.any(a[y1:y2,x1:x2]!=255):raise ValueError(\'protected region must be opaque\')\n        protected[y1:y2,x1:x2]=True\n    if not isinstance(patches,(list,tuple)):raise ValueError(\'explicit patch list required\')\n    b=a.copy();used=np.zeros(a.shape,bool);ledger=[]\n    for patch in patches:\n        if not isinstance(patch,dict) or set(patch)!={\'operation\',\'polygon\',\'source_review_note\'}:raise ValueError(\'explicit patch fields required\')\n        op=patch[\'operation\'];note=patch[\'source_review_note\']\n        if op not in (\'exclude\',\'restore_opaque\'):raise ValueError(\'invalid patch operation\')\n        if not isinstance(note,str) or not note.strip():raise ValueError(\'source review rationale required\')\n        mask=_polygon(patch[\'polygon\'],width,height)\n        if np.any(mask&protected):raise ValueError(\'patch crosses protected region\')\n        if np.any(mask&used):raise ValueError(\'overlapping patches require one unambiguous annotation\')\n        used|=mask;b[mask]=0 if op==\'exclude\' else 255\n        ledger.append({**patch,\'selected_pixels\':int(mask.sum())})\n    manifest={\'kind\':\'EXPLICIT_ALPHA_ANNOTATION_NOT_PRODUCTION_APPROVAL\',\'patches\':ledger,\'alpha_changed_pixels\':int((b!=a).sum()),\'support_removed\':int(((a>0)&(b==0)).sum()),\'support_added\':int(((a==0)&(b>0)).sum()),\'protected_alpha_changed\':int(((b!=a)&protected).sum()),\'source_resampled\':False,\'quality_accepted\':False,\'independent_ground_truth\':False,\'input_alpha_sha256\':hashlib.sha256(a.tobytes()).hexdigest(),\'output_alpha_sha256\':hashlib.sha256(b.tobytes()).hexdigest()}\n    return Image.fromarray(b),manifest\n\n\ndef export_explicit_alpha_review(source, alpha, protected_regions, patches, output):\n    """Write a new review-only bundle with native light/dark audit previews.\n\n    No acceptance marker or production-review bundle is created. Existing output\n    directories are preserved, and validation finishes before files are written.\n    """\n    from pathlib import Path\n    import json, tempfile\n    if not isinstance(source,Image.Image) or source.mode!=\'RGB\' or source.size!=alpha.size:\n        raise ValueError(\'native RGB source must match alpha dimensions\')\n    output=Path(output)\n    if output.exists():raise ValueError(\'preserve previous output\')\n    candidate,manifest=apply_explicit_alpha_patches(alpha,protected_regions,patches)\n    rgb=np.asarray(source);a=np.asarray(candidate)\n    previews={}\n    for name,color in [(\'light\',(245,245,240)),(\'dark\',(25,25,30))]:\n        im=Image.new(\'RGB\',source.size,color);im.paste(source,mask=candidate)\n        seen=np.asarray(im);opaque=a==255\n        if not np.array_equal(seen[opaque],rgb[opaque]):raise RuntimeError(\'opaque RGB altered\')\n        previews[name]=im\n    manifest.update(source_rgb_sha256=hashlib.sha256(rgb.tobytes()).hexdigest(),retained_opaque_rgb_changed=0,source_rgb_repainted=False,protected_regions=protected_regions,production_review_created=False,caption=\'CPU source-locked alpha-annotation preview, not Magic Hour. Requires visual review; not publication approval.\')\n    output.parent.mkdir(parents=True,exist_ok=True)\n    with tempfile.TemporaryDirectory(dir=output.parent,prefix=\'.alpha-review-\') as tmp:\n        tmp=Path(tmp);source.save(tmp/\'source-native.png\');alpha.save(tmp/\'baseline-alpha.png\');candidate.save(tmp/\'candidate-alpha.png\')\n        for name,im in previews.items():\n            im.save(tmp/f\'{name}-native.png\')\n            plate=Image.new(\'RGB\',(max(source.width,800),source.height+72),\'white\');plate.paste(im,(0,0));draw=ImageDraw.Draw(plate)\n            draw.text((8,source.height+8),f\'{name}: CPU source-locked alpha-annotation preview, not Magic Hour.\',fill=\'black\')\n            draw.text((8,source.height+30),\'Manual source-coordinate candidate; requires visual review, not publication approval.\',fill=\'black\')\n            draw.text((8,source.height+50),\'Native pixels. No RGB repainting/resampling. Protected alpha changes: 0.\',fill=\'black\')\n            plate.save(tmp/f\'{name}-captioned.png\')\n        (tmp/\'manifest.json\').write_text(json.dumps(manifest,indent=2))\n        # Final directory appears only after every file has been written.\n        if output.exists():raise ValueError(\'preserve previous output\')\n        tmp.rename(output)\n    return manifest\n'
# ---- embedded module: renderello/explicit_alpha_patch.py ----
path = base / 'renderello/explicit_alpha_patch.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Lossless artifact export for notebook sessions where /kaggle/working is\nnot persisted by Quick Save. Full-resolution PNGs are displayed inline UNRESIZED\n(IPython embeds the exact PNG bytes); a browser agent can then fetch the image\nblob from the saved version and write the same bytes locally.\n\nProven 2026-11-09 for 1800 px images via fetch() from the notebook output frame.\nverify_roundtrip() checks the encode/decode is pixel-exact.\n"""\nfrom __future__ import annotations\nimport base64\nimport hashlib\nimport io\nfrom pathlib import Path\nfrom PIL import Image\n\n\ndef png_bytes(path: Path) -> bytes:\n    b = Path(path).read_bytes()\n    assert b[:8] == b"\\x89PNG\\r\\n\\x1a\\n", "not a PNG"\n    return b\n\n\ndef manifest_line(path: Path) -> str:\n    b = png_bytes(path)\n    return f"{Path(path).name} {len(b)} bytes sha256={hashlib.sha256(b).hexdigest()}"\n\n\ndef display_full_res(path: Path):\n    """Notebook: display the PNG file unresized and print its sha256 so the\n    retrieved copy can be verified byte-for-byte."""\n    from IPython.display import display, Image as IPImage\n    print(manifest_line(path))\n    display(IPImage(data=png_bytes(path), format="png"))\n\n\ndef verify_roundtrip(path: Path) -> bool:\n    b = png_bytes(path)\n    back = base64.b64decode(base64.b64encode(b))\n    a = Image.open(io.BytesIO(back)).convert("RGBA")\n    o = Image.open(path).convert("RGBA")\n    return back == b and a.size == o.size and a.tobytes() == o.tobytes()\n'
# ---- embedded module: renderello/export.py ----
path = base / 'renderello/export.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Decide which saved candidates need exact-fidelity review, never acceptance.\n\nGenerative product edits need review even if every heuristic is quiet. Flags\nare conservative warnings, not semantic proof. All outputs remain downloadable.\n"""\nGENERATIVE_PRODUCT_ROUTES = frozenset((\'pure_prompt\', \'hybrid\'))\n\n\ndef review_requirement(route, flags=(), *, output_exists=True, completed=True):\n    if not completed:\n        return dict(required=False, status=\'FAILED_PARTIAL_ARTIFACT\' if output_exists else \'FAILED_NO_OUTPUT\', reasons=[],\n                    product_text_verified=False, photo_accepted=False)\n    if not output_exists:\n        return dict(required=False, status=\'NOT_GENERATED\', reasons=[],\n                    product_text_verified=False, photo_accepted=False)\n    reasons = []\n    if route in GENERATIVE_PRODUCT_ROUTES:\n        reasons.append(\'GENERATIVE_PRODUCT_EDIT_ALWAYS_REQUIRES_FIDELITY_REVIEW\')\n    if flags:\n        reasons.append(\'AUTOMATED_WARNING_REQUIRES_REVIEW\')\n    # Quiet deterministic composites also have unverified alpha/material/contact.\n    # Distinguish that general quality review from the product-redraw queue.\n    return dict(required=bool(reasons),\n                status=\'EXACT_FIDELITY_REVIEW_REQUIRED\' if reasons else \'NO_TRIPWIRE_NOT_A_QUALITY_PASS\',\n                reasons=reasons, product_text_verified=False, photo_accepted=False)\n\n\ndef review_caption(requirement):\n    if requirement.get(\'status\') in (\'FAILED_PARTIAL_ARTIFACT\', \'FAILED_NO_OUTPUT\'):\n        return \'FAILED: any saved image is a debug artifact, not a completed output or quality-rated candidate.\'\n    if requirement.get(\'status\') == \'NOT_GENERATED\':\n        return \'NOT GENERATED: no image to review.\'\n    if \'GENERATIVE_PRODUCT_EDIT_ALWAYS_REQUIRES_FIDELITY_REVIEW\' in requirement.get(\'reasons\', []):\n        return (\'FIDELITY REVIEW REQUIRED: this route generates product pixels. \'\n                \'Zero automated warnings does not verify label text, shape or logos. \'\n                \'Compare the original and full-size output before use.\')\n    if requirement.get(\'required\'):\n        return \'REVIEW REQUIRED: automated warnings need source/output inspection, not automatic acceptance.\'\n    return \'No automated tripwire. Mask, material, label readability and publication quality remain unverified.\'\n'
# ---- embedded module: renderello/fidelity_review.py ----
path = base / 'renderello/fidelity_review.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Framework-neutral FlowEdit Euler core, n_min=0 only.\n\nAlgorithm 1: https://arxiv.org/html/2412.08629\nDirect source/target velocity differences, not image conditioning or inversion.\nNo models, image IO, masks, RGB paste, payment or quality acceptance here.\n"""\nfrom __future__ import annotations\nimport math\n\n\ndef integrate(source, sigmas, velocity, noise_like, *, n_max, n_avg=1,\n              validate_array, on_step=None):\n    """Same numeric core for a NumPy oracle and a separately checked Torch run.\n\n    velocity(array, sigma, branch, step_index) returns source/target velocity.\n    noise_like(source, step_index, draw_index) returns a fresh Gaussian sample.\n    validate_array must reject wrong shape/type/nonfinite arrays. It is called\n    before any model call and on model outputs and states. Caller owns model\n    scheduling, CFG, dtype, token handling, latent encoding and generator.\n    These callbacks are trusted local code, never loaded from plan content.\n    """\n    if not callable(validate_array) or not callable(velocity) or not callable(noise_like):\n        raise ValueError(\'explicit validation, velocity and noise callbacks required\')\n    try: schedule=list(sigmas)\n    except TypeError: raise ValueError(\'finite decreasing sigma sequence required\') from None\n    if not 2<=len(schedule)<=501:raise ValueError(\'one to 500 integration steps required\')\n    if any(isinstance(s,bool) or not isinstance(s,(int,float)) or not math.isfinite(s) for s in schedule):\n        raise ValueError(\'finite numeric sigmas required\')\n    if schedule[0]>1 or schedule[-1]!=0 or any(a<=b for a,b in zip(schedule,schedule[1:])):\n        raise ValueError(\'strictly decreasing sigmas in [0,1] ending at zero required\')\n    steps=len(schedule)-1\n    if type(n_max)is not int or not 0<=n_max<=steps:raise ValueError(\'bounded integer n_max required\')\n    if type(n_avg)is not int or not 1<=n_avg<=16:raise ValueError(\'bounded integer n_avg required\')\n    if on_step is not None and not callable(on_step):raise ValueError(\'step callback required\')\n    validate_array(source);state=source+0\n    for i in range(steps-n_max,steps):\n        sigma=schedule[i];delta=source*0\n        for draw in range(n_avg):\n            noise=noise_like(source,i,draw);validate_array(noise)\n            src=(1-sigma)*source+sigma*noise\n            # This order gives an exact zero displacement when state==source.\n            tar=src+(state-source)\n            validate_array(src);validate_array(tar)\n            vs=velocity(src,sigma,\'source\',i);validate_array(vs)\n            vt=velocity(tar,sigma,\'target\',i);validate_array(vt)\n            delta=delta+(vt-vs)/n_avg\n        state=state+(schedule[i+1]-sigma)*delta;validate_array(state)\n        if on_step is not None:on_step(i,sigma,state)\n    return state\n'
# ---- embedded module: renderello/flowedit_delta.py ----
path = base / 'renderello/flowedit_delta.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Private gallery generator. Static HTML, no external assets, relative paths -\nopen the folder locally or share the Drive folder privately. Per the plan, the\ngallery stays private (no public Space/publish)."""\nfrom __future__ import annotations\n\nimport html\nimport hashlib\nimport json\nimport os\n\nfrom .artifact_paths import run_artifact\nfrom .fidelity_review import review_requirement, review_caption\nfrom pathlib import Path\n\n_CSS = """\n*{box-sizing:border-box}\nbody{font-family:system-ui,sans-serif;margin:24px;background:#fafafa;color:#222;\noverflow-x:hidden}\nh1{font-size:20px} h2{font-size:16px;margin-top:32px}\n.note{font-size:12px;color:#777;max-width:900px}\n.pair{display:inline-block;vertical-align:top;margin:8px;background:#fff;\nborder:1px solid #ddd;border-radius:8px;padding:8px;width:340px;max-width:94%}\n.pair img{width:100%;border-radius:4px}\n.meta{overflow-wrap:anywhere;font-size:12px;color:#555;margin-top:6px;white-space:pre-wrap}\n.bad{color:#a00}.good{color:#070}\n"""\n\n_DISCLAIMER = ("All scores are AUTOMATED STRUCTURAL metrics (pixel-level fidelity "\n               "checks), not human ratings and not publish-ready judgments. "\n               "Source classification, attribution and rights status are shown per source below. "\n               "Unknown inputs are not assumed synthetic or licensed. This gallery alone "\n               "does not establish the full benchmark gate.")\n\n\ndef _source_caption(m: dict) -> str:\n    intake = m.get(\'image_intake\') or {}\n    basis = (\'\\nIMAGE INTAKE: \' + \', \'.join(intake.get(\'transformations\') or [\'decoded RGB assumed sRGB\']) +\n             \'. Product pixel comparisons use normalized RGB, not original encoded colour values. Colour accuracy is not verified.\') if intake else \'\'\n    p = m.get(\'source_provenance\')\n    if not p:\n        return (\'SOURCE: UNKNOWN / unclassified legacy run. No content-bound source record. \'\n                \'Do not assume synthetic, licensed or publishable; rerun to record provenance.\' + basis)\n    return \'\\n\'.join([\'SOURCE: \' + p.get(\'classification\', \'UNKNOWN / unclassified\'),\n                      \'Content match: \' + p.get(\'match_basis\', \'unrecorded\'),\n                      \'SHA256: \' + p.get(\'sha256\', \'unrecorded\'),\n                      \'Attribution: \' + p.get(\'attribution\', \'unrecorded\'),\n                      \'Record: \' + p.get(\'record_origin\', \'unrecorded\'),\n                      \'Rights: \' + p.get(\'rights_status\', \'unrecorded\')]) + basis\n\n\ndef build_gallery(run_dirs: list[Path], out_html: Path, title: str = "Renderello gallery (private)",\n                  source_paths: dict[str, Path] | None = None) -> Path:\n    sections: list[str] = []\n    for rd in run_dirs:\n        manifest = rd / "manifest.json"\n        if not manifest.exists():\n            continue\n        m = json.loads(manifest.read_text())\n        tpl = m["template"]["name"]\n        rows: list[str] = [f"<h2>{html.escape(tpl)} - {html.escape(Path(m[\'source\']).name)}</h2>"]\n        source = Path(m["source"])\n        digest = m.get("source_provenance", {}).get("sha256")\n        if source_paths and digest in source_paths:\n            source = Path(source_paths[digest])\n            if not source.is_file() or hashlib.sha256(source.read_bytes()).hexdigest() != digest:\n                raise ValueError("source override missing or hash mismatch: " + digest)\n        if source.is_file():\n            rows.append(\'<div class="pair"><img src="{}" alt="source"><div class="meta">{}</div></div>\'.format(\n                html.escape(_rel(str(source), out_html.parent)), html.escape(_source_caption(m))))\n        else:\n            rows.append(\'<div class="pair"><div class="meta bad">Source image not included in this archive. \'\n                        \'No before/after comparison is claimed.</div><div class="meta">\' +\n                        html.escape(_source_caption(m)) + \'</div></div>\')\n        for v in m["variants"]:\n            q = v["qa"]\n            output = run_artifact(rd, v.get("output", ""))\n            output_exists = output is not None\n            from .completion import saved_variant_failed\n            failed = saved_variant_failed(v)\n            if not output_exists or failed:\n                failures = q.get("critical_errors") or [m.get("cutout_error") or "Output file is missing"]\n                text = (("FAILED / PARTIAL ARTIFACT - NOT A COMPLETED OUTPUT\\n" if output_exists else "FAILED / NOT GENERATED\\n") +\n                        f"route {v.get(\'route\', \'unknown\')} | seed {v.get(\'seed\', \'unknown\')}\\n" +\n                        ("Artifact retained for debugging. No quality rating.\\n" if output_exists else "No image output and no quality rating.\\n") + "; ".join(str(e) for e in failures))\n                rows.append(f\'<div class="pair"><div class="meta bad">{html.escape(text)}</div></div>\')\n                continue\n            score = q.get("score_fidelity")\n            errs = q.get("critical_errors") or []\n            cls = "bad" if errs or v.get("backend") == "grabcut" else ""\n            meta = (f"route {v[\'route\']} | seed {v[\'seed\']} | backend {v.get(\'backend\',\'\')}\\n"\n                    f"auto fidelity {\'NOT_EVALUATED (review needed)\' if score is None else str(score)+\'/5\'} (structural metric, not a human rating) | "\n                    f"ssim {_fmt(q.get(\'ssim_product\'))} | "\n                    f"drift {_fmt(q.get(\'color_drift\'))} | edges {_fmt(q.get(\'edge_retained\'))}")\n            if v.get("backend") == "grabcut":\n                meta += "\\nREVIEW REQUIRED: GrabCut fallback may retain source background or cut product edges. A high pixel-fidelity score does not validate the mask."\n            if m.get("mask_provenance") == "embedded_source_alpha":\n                meta += "\\nEMBEDDED SOURCE ALPHA: explicitly retained after intake; no segmentation model; mask and material semantics unverified."\n            elif v.get("backend") == "supplied_mask":\n                meta += ("\\nASSISTED SEEDED GRABCUT MASK: explicit correction regions/strokes, not automatic success or original GPU alpha."\n                         if m.get("mask_provenance") == "estimated_seeded_grabcut_assisted" else\n                         f"\\nESTIMATED {m.get(\'mask_provenance\',\'\').replace(\'estimated_\',\'\').upper()} MASK: not original GPU alpha; inspect missing objects, halos and transparency."\n                         if (m.get("mask_provenance") or "").startswith("estimated_") else\n                         "\\nSUPPLIED MASK: explicit assisted alpha, not automatic segmentation; inspect edges.")\n            from .execution_kind import describe_execution\n            execution = v.get(\'execution_kind\') or describe_execution(v.get(\'route\'),v.get(\'backend\',\'\'))\n            meta += \'\\n\' + execution[\'caption\']\n            requirement = review_requirement(v.get(\'route\'), errs)\n            meta += "\\n" + review_caption(requirement)\n            if requirement[\'required\']:\n                cls = "bad"\n            if errs:\n                meta += "\\nFLAGS: " + "; ".join(errs)\n            rows.append(\n                f\'<div class="pair"><img src="{html.escape(_rel(str(output), out_html.parent))}">\'\n                f\'<div class="meta {cls}">{html.escape(meta)}</div></div>\')\n        sections.append("\\n".join(rows))\n    doc = (f"<!doctype html><html><head><meta charset=\'utf-8\'>"\n           f"<meta name=\'viewport\' content=\'width=device-width,initial-scale=1\'>"\n           f"<title>{html.escape(title)}</title><style>{_CSS}</style></head>"\n           f"<body><h1>{html.escape(title)}</h1>"\n           f"<p class=\'note\'>{html.escape(_DISCLAIMER)}</p>"\n           + "\\n".join(sections) + "</body></html>")\n    out_html.parent.mkdir(parents=True, exist_ok=True)\n    out_html.write_text(doc)\n    return out_html\n\n\ndef _rel(p: str, base: Path) -> str:\n    # A basename fallback silently broke galleries outside the run directory.\n    # The gallery and referenced folders must travel together when sharing.\n    return Path(os.path.relpath(Path(p).resolve(), base.resolve())).as_posix()\n\n\ndef _fmt(x) -> str:\n    return f"{x:.3f}" if isinstance(x, (int, float)) else "-"\n'
# ---- embedded module: renderello/gallery.py ----
path = base / 'renderello/gallery.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Explicit studio foreground plane experiment, not inferred physical geometry.\n\nA designer chooses horizon and color. This replaces scene foreground pixels,\nnever moves the saved product, and never claims to detect a real tabletop.\n"""\nfrom pathlib import Path\nimport json\nimport numpy as np\nfrom PIL import Image\nfrom .compose import finish_scene\n\n\ndef studio_foreground(scene, crop, placement, *, horizon, color):\n    if scene.dtype != np.uint8 or scene.ndim != 3 or scene.shape[2] != 3:\n        raise ValueError(\'scene must be RGB uint8\')\n    if crop.dtype != np.uint8 or crop.ndim != 3 or crop.shape[2] != 4:\n        raise ValueError(\'product must be RGBA uint8\')\n    h,w=scene.shape[:2];x,y=map(int,placement);ph,pw=crop.shape[:2]\n    if x<0 or y<0 or x+pw>w or y+ph>h:raise ValueError(\'product clips canvas\')\n    rows=np.where((crop[...,3]>127).any(axis=1))[0]\n    if not len(rows):raise ValueError(\'empty product\')\n    foot=y+int(rows[-1])\n    if not isinstance(horizon,int) or not 0<=horizon<foot:raise ValueError(\'horizon must be above foot\')\n    rgb=np.asarray(color,dtype=float)\n    if rgb.shape!=(3,) or not np.isfinite(rgb).all() or (rgb<0).any() or (rgb>255).any():\n        raise ValueError(\'color must contain three finite values0-255\')\n    out=scene.copy()\n    # A mild near-to-far tonal gradient gives a deliberately designed studio\n    # surface. There is no generated texture or hidden inpainting here.\n    t=np.linspace(0,1,h-horizon)[:,None,None]\n    plane=np.clip(rgb[None,None,:]*(1.02-.13*t),0,255)\n    out[horizon:]=np.broadcast_to(plane,(h-horizon,w,3)).astype(np.uint8)\n    return out,{\'mode\':\'explicit_designed_foreground\',\'horizon\':horizon,\n                \'foot_row\':foot,\'color\':rgb.tolist(),\'automatic_geometry\':False}\n\n\ndef replay_ground_plane(cache, *, output, horizon, color):\n    cache=Path(cache);output=Path(output)\n    if output.exists():raise ValueError(\'output exists; choose a new path\')\n    scene=np.array(Image.open(cache/\'scene.png\').convert(\'RGB\'))\n    crop=np.array(Image.open(cache/\'product_rgba.png\').convert(\'RGBA\'))\n    m=json.loads((cache/\'layers.json\').read_text());place=tuple(m[\'placement\'])\n    if m.get(\'cover_used\'):raise ValueError(\'cover placement unsupported\')\n    plane,diag=studio_foreground(scene,crop,place,horizon=horizon,color=color)\n    final,_=finish_scene(plane,crop,place,shadow_style=\'anchored soft\',\n                        glass_backdrop=m.get(\'glass_backdrop\'),scene_auto_lift=False)\n    result=np.clip(final,0,255).astype(np.uint8)\n    x,y=place;ph,pw=crop.shape[:2];opaque=crop[...,3]==255\n    diag[\'opaque_rgb_exact\']=None if m.get(\'glass_backdrop\') is not None else bool(\n        np.array_equal(result[y:y+ph,x:x+pw][opaque],crop[...,:3][opaque]))\n    if diag[\'opaque_rgb_exact\'] is False:raise ValueError(\'product fidelity invariant failed\')\n    output.parent.mkdir(parents=True,exist_ok=True)\n    Image.fromarray(result).save(output)\n    output.with_suffix(\'.json\').write_text(json.dumps(diag,indent=2))\n    return diag\n'
# ---- embedded module: renderello/ground_plane.py ----
path = base / 'renderello/ground_plane.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""IC-Light GPU relight (foreground-conditioned, text-to-scene).\n\nOwning repo: github.com/lllyasviel/IC-Light (code Apache-2.0, weights\nOpenRAIL-M; attribution required in any derivative display). This is the\nbg_source=NONE variant of the official gradio_demo.py `process()`: the scene\nis generated from the text prompt around a foreground composited on gray 127.\n\nVALIDATED 2026-11-09 on the owner\'s Kaggle (2x Tesla T4, torch 2.11.0+cu130,\ndiffusers 0.42.0): 3 relights, 32.1 / 37.9 / 36.5 s each (512x640, 25 steps,\nhighres x1.5). See COVERAGE.md for the measured limits. IMPORTANT: the output\nis a REGENERATION, not a pixel edit - product details drifted in that test\n(sneaker LED sole lost, can graphic altered). Never ship these pixels as the\nproduct; use them as scene/lighting only and paste source-exact product pixels\nback (the pipeline\'s compose route does that for the product region).\n\nRequires CUDA + `diffusers safetensors transformers`. Nothing here runs on CPU\nhosts; the pure-numpy helpers are unit-tested, the model path is not.\n"""\nfrom __future__ import annotations\n\nimport math\nimport os\nimport time\n\nimport numpy as np\nfrom PIL import Image\n\nSD15_NAME = "stablediffusionapi/realistic-vision-v51"  # official demo default\nICLIGHT_URL = ("https://huggingface.co/lllyasviel/ic-light/resolve/main/"\n               "iclight_sd15_fc.safetensors")\nDEFAULTS = dict(width=512, height=640, steps=25, cfg=2.0, highres_scale=1.5,\n                highres_denoise=0.5, seed=12345,\n                a_prompt="best quality",\n                n_prompt="lowres, bad anatomy, bad hands, cropped, worst quality")\n\n\ndef composite_on_gray(img_rgb: np.ndarray, matte_u8: np.ndarray) -> np.ndarray:\n    """Official run_rmbg composite (sigma=0): 127 + (img-127)*alpha."""\n    alpha = matte_u8.astype(np.float32) / 255.0\n    return (127 + (img_rgb.astype(np.float32) - 127) * alpha[..., None]\n            ).clip(0, 255).astype(np.uint8)\n\n\ndef resize_and_center_crop(image: np.ndarray, tw: int, th: int) -> np.ndarray:\n    pil = Image.fromarray(image)\n    ow, oh = pil.size\n    s = max(tw / ow, th / oh)\n    rw, rh = int(round(ow * s)), int(round(oh * s))\n    r = pil.resize((rw, rh), Image.LANCZOS)\n    l, t = (rw - tw) // 2, (rh - th) // 2\n    return np.array(r.crop((l, t, l + tw, t + th)))\n\n\ndef highres_size(w: int, h: int, scale: float) -> tuple[int, int]:\n    return (int(round(w * scale / 64.0) * 64), int(round(h * scale / 64.0) * 64))\n\n\ndef lighting_gradient(width, height, direction):\n    """Official directional initial image; model output is not guaranteed direction-correct."""\n    if direction not in (\'none\', \'left\', \'right\', \'top\', \'bottom\'):\n        raise ValueError(\'unknown initial-light direction\')\n    if direction == \'none\':\n        return None\n    if type(width) is not int or type(height) is not int or width % 64 or height % 64 or not (256 <= width <= 1024 and 256 <= height <= 1024):\n        raise ValueError(\'initial-light dimensions must be multiples of64 within256..1024\')\n    if direction in (\'left\', \'right\'):\n        gradient = np.linspace(255, 0, width) if direction == \'left\' else np.linspace(0, 255, width)\n        image = np.tile(gradient, (height, 1))\n    else:\n        gradient = np.linspace(255, 0, height) if direction == \'top\' else np.linspace(0, 255, height)\n        image = np.tile(gradient[:, None], (1, width))\n    return np.stack((image,) * 3, axis=-1).astype(np.uint8)\n\n\nclass ICLight:\n    """Lazy-loaded IC-Light (SD1.5 fc). Call .relight(); call .close() after."""\n\n    def __init__(self, weights_path: str = "models/iclight_sd15_fc.safetensors"):\n        self.weights_path = weights_path\n        self._loaded = False\n\n    def _ensure_weights(self) -> None:\n        if os.path.exists(self.weights_path):\n            return\n        from torch.hub import download_url_to_file\n        os.makedirs(os.path.dirname(self.weights_path) or ".", exist_ok=True)\n        download_url_to_file(ICLIGHT_URL, dst=self.weights_path)\n\n    def load(self) -> None:\n        import torch\n        import safetensors.torch as sf\n        from diffusers import (AutoencoderKL, DPMSolverMultistepScheduler,\n                               StableDiffusionImg2ImgPipeline,\n                               StableDiffusionPipeline, UNet2DConditionModel)\n        from diffusers.models.attention_processor import AttnProcessor2_0\n        from transformers import CLIPTextModel, CLIPTokenizer\n        if not torch.cuda.is_available():\n            raise RuntimeError("IC-Light needs CUDA; no GPU available")\n        self._ensure_weights()\n        self.torch = torch\n        self.tokenizer = CLIPTokenizer.from_pretrained(SD15_NAME, subfolder="tokenizer")\n        te = CLIPTextModel.from_pretrained(SD15_NAME, subfolder="text_encoder")\n        vae = AutoencoderKL.from_pretrained(SD15_NAME, subfolder="vae")\n        unet = UNet2DConditionModel.from_pretrained(SD15_NAME, subfolder="unet")\n        with torch.no_grad():  # 8-channel conv_in hack from the official demo\n            nc = torch.nn.Conv2d(8, unet.conv_in.out_channels, unet.conv_in.kernel_size,\n                                 unet.conv_in.stride, unet.conv_in.padding)\n            nc.weight.zero_()\n            nc.weight[:, :4].copy_(unet.conv_in.weight)\n            nc.bias = unet.conv_in.bias\n            unet.conv_in = nc\n        orig_forward = unet.forward\n\n        def hooked(sample, timestep, encoder_hidden_states, **kw):\n            cc = kw["cross_attention_kwargs"]["concat_conds"].to(sample)\n            cc = torch.cat([cc] * (sample.shape[0] // cc.shape[0]), dim=0)\n            kw["cross_attention_kwargs"] = {}\n            return orig_forward(torch.cat([sample, cc], dim=1), timestep,\n                                encoder_hidden_states, **kw)\n        unet.forward = hooked\n        off = sf.load_file(self.weights_path)\n        base = unet.state_dict()\n        unet.load_state_dict({k: base[k] + off[k] for k in base}, strict=True)\n        del off, base\n        dev = torch.device("cuda")\n        self.device = dev\n        self.te = te.to(device=dev, dtype=torch.float16)\n        self.vae = vae.to(device=dev, dtype=torch.bfloat16)\n        self.unet = unet.to(device=dev, dtype=torch.float16)\n        self.unet.set_attn_processor(AttnProcessor2_0())\n        self.vae.set_attn_processor(AttnProcessor2_0())\n        sched = DPMSolverMultistepScheduler(\n            num_train_timesteps=1000, beta_start=0.00085, beta_end=0.012,\n            algorithm_type="sde-dpmsolver++", use_karras_sigmas=True, steps_offset=1)\n        kw = dict(vae=self.vae, text_encoder=self.te, tokenizer=self.tokenizer,\n                  unet=self.unet, scheduler=sched, safety_checker=None,\n                  requires_safety_checker=False, feature_extractor=None,\n                  image_encoder=None)\n        self.t2i = StableDiffusionPipeline(**kw)\n        self.i2i = StableDiffusionImg2ImgPipeline(**kw)\n        self._loaded = True\n\n    def _enc(self, txt):\n        torch = self.torch\n        ml = self.tokenizer.model_max_length\n        cl = ml - 2\n        s, e = self.tokenizer.bos_token_id, self.tokenizer.eos_token_id\n        toks = self.tokenizer(txt, truncation=False, add_special_tokens=False)["input_ids"]\n        chunks = [[s] + toks[i:i + cl] + [e] for i in range(0, len(toks), cl)]\n        chunks = [(c[:ml] if len(c) >= ml else c + [e] * (ml - len(c))) for c in chunks]\n        ids = torch.tensor(chunks).to(device=self.device, dtype=torch.int64)\n        return self.te(ids).last_hidden_state\n\n    def _pair(self, pos, neg):\n        torch = self.torch\n        c, uc = self._enc(pos), self._enc(neg)\n        n = max(len(c), len(uc))\n        c = torch.cat([c] * math.ceil(n / len(c)), dim=0)[:n]\n        uc = torch.cat([uc] * math.ceil(n / len(uc)), dim=0)[:n]\n        return (torch.cat([p[None] for p in c], dim=1),\n                torch.cat([p[None] for p in uc], dim=1))\n\n    def _np2pt(self, imgs):\n        h = self.torch.from_numpy(np.stack(imgs, axis=0)).float() / 127.0 - 1.0\n        return h.movedim(-1, 1)\n\n    def _pt2np(self, imgs):\n        return [(x.movedim(0, -1) * 127.5 + 127.5).detach().float().cpu().numpy()\n                .clip(0, 255).astype(np.uint8) for x in imgs]\n\n    def relight(self, fg_on_gray: np.ndarray, prompt: str, **over) -> tuple[np.ndarray, float]:\n        """fg_on_gray: HxWx3 uint8 (see composite_on_gray). Returns (image, seconds)."""\n        p = {**DEFAULTS, **over}\n        initial = lighting_gradient(p[\'width\'], p[\'height\'], p.get(\'initial_light\', \'none\'))\n        lowres_denoise = p.get(\'lowres_denoise\', .9)\n        if isinstance(lowres_denoise, bool) or not np.isfinite(lowres_denoise) or not .1 <= lowres_denoise <= 1:\n            raise ValueError(\'lowres_denoise must be finite0.1..1\')\n        if not self._loaded:\n            self.load()\n        torch = self.torch\n        t0 = time.time()\n        with torch.inference_mode():\n            rng = torch.Generator(device=self.device).manual_seed(int(p["seed"]))\n            w, h = p["width"], p["height"]\n            fg = resize_and_center_crop(fg_on_gray, w, h)\n            cc = self._np2pt([fg]).to(device=self.vae.device, dtype=self.vae.dtype)\n            cc = self.vae.encode(cc).latent_dist.mode() * self.vae.config.scaling_factor\n            conds, unconds = self._pair(prompt + ", " + p["a_prompt"], p["n_prompt"])\n            if initial is None:\n                lat = self.t2i(prompt_embeds=conds, negative_prompt_embeds=unconds,\n                               width=w, height=h, num_inference_steps=p["steps"],\n                               num_images_per_prompt=1, generator=rng, output_type="latent",\n                               guidance_scale=p["cfg"],\n                               cross_attention_kwargs={"concat_conds": cc}\n                               ).images.to(self.vae.dtype) / self.vae.config.scaling_factor\n            else:\n                # Official IC-Light directional initial-latent path, not post-hoc grading.\n                bg = self._np2pt([initial]).to(device=self.vae.device, dtype=self.vae.dtype)\n                bg_latent = self.vae.encode(bg).latent_dist.mode() * self.vae.config.scaling_factor\n                lat = self.i2i(image=bg_latent, strength=lowres_denoise,\n                    prompt_embeds=conds, negative_prompt_embeds=unconds,\n                    width=w, height=h, num_inference_steps=int(round(p[\'steps\'] / lowres_denoise)),\n                    num_images_per_prompt=1, generator=rng, output_type=\'latent\',\n                    guidance_scale=p[\'cfg\'], cross_attention_kwargs={\'concat_conds\': cc}\n                    ).images.to(self.vae.dtype) / self.vae.config.scaling_factor\n            px = self._pt2np(self.vae.decode(lat).sample)\n            hw, hh = highres_size(w, h, p["highres_scale"])\n            px = [np.array(Image.fromarray(x).resize((hw, hh), Image.LANCZOS)) for x in px]\n            px = self._np2pt(px).to(device=self.vae.device, dtype=self.vae.dtype)\n            lat = self.vae.encode(px).latent_dist.mode() * self.vae.config.scaling_factor\n            lat = lat.to(device=self.unet.device, dtype=self.unet.dtype)\n            h2, w2 = lat.shape[2] * 8, lat.shape[3] * 8\n            fg = resize_and_center_crop(fg_on_gray, w2, h2)\n            cc = self._np2pt([fg]).to(device=self.vae.device, dtype=self.vae.dtype)\n            cc = self.vae.encode(cc).latent_dist.mode() * self.vae.config.scaling_factor\n            lat = self.i2i(image=lat, strength=p["highres_denoise"], prompt_embeds=conds,\n                           negative_prompt_embeds=unconds, width=w2, height=h2,\n                           num_inference_steps=int(round(p["steps"] / p["highres_denoise"])),\n                           num_images_per_prompt=1, generator=rng, output_type="latent",\n                           guidance_scale=p["cfg"],\n                           cross_attention_kwargs={"concat_conds": cc}\n                           ).images.to(self.vae.dtype) / self.vae.config.scaling_factor\n            out = self._pt2np(self.vae.decode(lat).sample)[0]\n        return out, round(time.time() - t0, 2)\n\n    def close(self) -> None:\n        if self._loaded:\n            for a in ("t2i", "i2i", "unet", "vae", "te"):\n                delattr(self, a)\n            self.torch.cuda.empty_cache()\n            self._loaded = False\n\n\nclass SupportPlaneCallback:\n    """Experimental plane-only conditioning; compositor removes all product RGB.\n\n    No guarantee of a matching support plane. Must be tested on actual GPU output.\n    """\n    support_plane = True\n\n    def __init__(self, model, seed=0):\n        self.model,self.seed=model,int(seed)\n\n    def __call__(self, canvas, prompt):\n        if canvas.ndim!=3 or canvas.shape[2]!=3 or canvas.dtype!=np.uint8:\n            raise ValueError(\'plane canvas must be HxWx3 uint8\')\n        return self.model.relight(canvas,prompt,seed=self.seed,cfg=4.0,\n            n_prompt=\'bottle, jar, can, box, glass, cup, watch, product, packaging, \'\n                     \'logo, text, person, hands, building, doorway, stairs, pillars, \'\n                     \'floating platform, vertical tabletop, clutter, low quality, distortion\')[0]\n\n\nclass FlatlaySurfaceCallback:\n    """Opt-in matched-view surface with an explicit designer-selected material."""\n    flatlay_surface=True\n    MATERIALS={\n      \'linen\':\'very fine woven ivory linen fabric, microscopic weave, soft folds only at the far corners\',\n      \'slate\':\'smooth dark charcoal slate sheet, subtle fine stone grain, no cracks, no large rocks\',\n      \'paper\':\'smooth matte pale sage green photographic paper, nearly invisible fine grain\',\n    }\n    def __init__(self,model,seed=0,material=\'linen\'):\n        if material not in self.MATERIALS:raise ValueError(\'unknown flatlay material\')\n        self.model,self.seed,self.material=model,int(seed),material\n    def __call__(self,canvas,prompt):\n        if canvas.dtype!=np.uint8 or canvas.ndim!=3 or canvas.shape[2]!=3:\n            raise ValueError(\'flatlay canvas must be RGB uint8\')\n        prompt=(\'professional commercial product flatlay backdrop, \'+self.MATERIALS[self.material]+\n                \', true directly overhead camera, continuous flat surface filling every pixel, \'\n                \'empty center for later product placement, subtle soft diagonal window lighting, \'\n                \'gentle tonal gradient, low contrast, high resolution photograph, no walls, no horizon\')\n        return self.model.relight(canvas,prompt,seed=self.seed,cfg=6.0,\n          n_prompt=\'product, packaging, bottle, watch, glass, cup, text, logo, hands, \'\n                   \'person, wall, horizon, kitchen cabinets, vertical view, side view, \'\n                   \'pedestal, stairs, cracks, coarse texture, stucco, gravel, pebbles, \'\n                   \'leather, embossed pattern, busy texture, cartoon, low quality, distortion\')[0]\n\n\nclass CameraMatchedCallback:\n    """Reviewer-declared front/elevated empty sweep, without product conditioning.\n\n    Same-angle prompting is a hypothesis to test, never a pose-estimation claim.\n    """\n    camera_matched = True\n    CAMERAS = {\n        \'front\': \'eye-level camera parallel to the vertical backdrop, level camera, low horizon\',\n        \'elevated\': \'camera looking slightly down at the floor at 20 degrees, three-quarter product photography angle\',\n    }\n    STYLES = {\n        \'clean_studio\': (\'seamless matte ivory paper sweep\', \'broad softbox illumination from upper left, subtle natural brightness gradient\'),\n        \'luxury_hero\': (\'seamless smooth deep charcoal sweep, softly lit floor, restrained dark gray tonal gradient\', \'large softbox from upper left, gentle rim light from upper right, readable softly lit center\'),\n        \'editorial_beauty\': (\'seamless smooth pale blush pink sweep, muted warm pink floor, clean beauty editorial studio\', \'large diffused beauty softbox from upper left, bright soft daylight\')\n    }\n    def __init__(self,model,seed=0,pose=\'front\',style=\'clean_studio\'):\n        if pose not in self.CAMERAS:raise ValueError(\'matched sweep supports front or elevated sources only\')\n        if style not in self.STYLES:raise ValueError(\'unsupported camera studio style\')\n        self.model,self.seed,self.camera_pose,self.style=model,int(seed),pose,style\n    def __call__(self,canvas,prompt):\n        if canvas.dtype!=np.uint8 or canvas.ndim!=3 or canvas.shape[2]!=3:\n            raise ValueError(\'matched scene canvas must be RGB uint8\')\n        surface,lighting=self.STYLES[self.style]\n        prompt=(\'empty professional photographic studio cyclorama, \'+surface+\', \'\n                \'continuous flat level floor occupying lower two thirds of photograph, \'\n                \'smooth curved infinity backdrop without a visible wall joint, \'+self.CAMERAS[self.camera_pose]+\n                \', \'+lighting+\', \'\n                \'empty studio, no objects, commercially photographed clean backdrop\')\n        return self.model.relight(canvas,prompt,seed=self.seed,cfg=6.,\n            n_prompt=\'product, bottle, shoe, bag, packaging, text, logo, person, hands, \'\n                     \'table edge, furniture, cabinet, shelf, window, rocks, raised platform, \'\n                     \'pedestal, coarse fabric, busy texture, overhead camera, topdown, \'\n                     \'stairs, corners, dramatic shadows, low quality\')[0]\n\n\nclass DarkSweepConditioningCallback(CameraMatchedCallback):\n    """Experimental conditioning-only ablation. Same luxury prompt and CFG.\n\n    A dark smooth sweep replaces fixed bright-gray conditioning. No product RGB,\n    geometry, or generated-room retouch. Never a physical-lighting/quality claim.\n    """\n    @staticmethod\n    def conditioning_canvas(shape):\n        h,w=shape[:2]\n        yy,xx=np.mgrid[0:h,0:w].astype(np.float32)\n        x=(xx+.5)/w;y=(yy+.5)/h\n        spot=np.exp(-((x-.44)**2/.16+(y-.62)**2/.26))\n        floor=1/(1+np.exp(-(y-.68)*14))\n        lum=18+25*spot+12*floor\n        return np.repeat(lum[...,None],3,axis=2).clip(0,255).astype(np.uint8)\n    def __init__(self,model,seed=0,pose=\'front\',style=\'luxury_hero\'):\n        if style!=\'luxury_hero\':raise ValueError(\'dark sweep ablation is luxury only\')\n        super().__init__(model,seed,pose,style)\n    def __call__(self,canvas,prompt):\n        return super().__call__(self.conditioning_canvas(canvas.shape),prompt)\n'
# ---- embedded module: renderello/iclight_gpu.py ----
path = base / 'renderello/iclight_gpu.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Opt-in image intake, not automatic product/quality acceptance.\n\nOriginal bytes are never rewritten. Colour-changing conversions require explicit\npolicy; unknown profiles and high-bit-depth images fail rather than being silently\ntruncated. Image identity, material and print readability still need visual review.\n"""\nfrom __future__ import annotations\nfrom dataclasses import dataclass, asdict\nfrom io import BytesIO\nfrom pathlib import Path\nimport hashlib\nimport json\nimport math\nimport os\nimport tempfile\nimport stat\nfrom PIL import Image, ImageCms, ImageOps, UnidentifiedImageError\n\n\nclass IntakeError(ValueError):\n    """An input cannot be safely interpreted under the selected policy."""\n\n\n@dataclass(frozen=True)\nclass IntakePolicy:\n    max_bytes: int = 40_000_000\n    max_pixels: int = 40_000_000\n    max_side: int = 16000\n    max_aspect: float = 40.0\n    allow_colour_conversion: bool = False\n    formats: tuple[str, ...] = (\'JPEG\', \'PNG\', \'WEBP\', \'TIFF\', \'BMP\')\n\n    def __post_init__(self):\n        for name in (\'max_bytes\', \'max_pixels\', \'max_side\'):\n            x = getattr(self, name)\n            if isinstance(x, bool) or not isinstance(x, int) or x <= 0:\n                raise ValueError(name + \' must be a positive integer\')\n        if not math.isfinite(self.max_aspect) or self.max_aspect < 1:\n            raise ValueError(\'max_aspect must be finite and at least one\')\n        if not self.formats or any(not isinstance(x, str) for x in self.formats):\n            raise ValueError(\'formats must contain image format names\')\n\n\n@dataclass\nclass IntakeResult:\n    image: Image.Image\n    record: dict\n\n    def close(self):\n        self.image.close()\n\n\ndef _read_bounded(path: Path, limit: int) -> bytes:\n    # A stream read enforces the limit even if a file grows after stat().\n    if not stat.S_ISREG(path.stat().st_mode):\n        raise IntakeError(\'Input must be a regular file\')\n    # Nonblocking open plus descriptor recheck closes the FIFO replacement race.\n    flags = os.O_RDONLY | getattr(os, \'O_NONBLOCK\', 0)\n    with os.fdopen(os.open(path, flags), \'rb\') as f:\n        if not stat.S_ISREG(os.fstat(f.fileno()).st_mode):\n            raise IntakeError(\'Input must be a regular file\')\n        if os.fstat(f.fileno()).st_size > limit:\n            raise IntakeError(\'Encoded input exceeds byte limit\')\n        data = f.read(limit + 1)\n    if len(data) > limit:\n        raise IntakeError(\'Encoded input exceeds byte limit\')\n    return data\n\n\ndef _dimensions(image: Image.Image, p: IntakePolicy):\n    w, h = image.size\n    if min(w, h) < 1 or max(w, h) > p.max_side or w * h > p.max_pixels:\n        raise IntakeError(\'Image dimensions exceed configured budget\')\n    if max(w / h, h / w) > p.max_aspect:\n        raise IntakeError(\'Extreme aspect ratio requires a separate review path\')\n\n\ndef inspect_image(path: str | Path, policy: IntakePolicy | None = None) -> IntakeResult:\n    """Decode and normalize orientation only; never invent missing product pixels.\n\n    Returned alpha is retained including transparent-pixel RGB. Converting colour\n    space can change RGB, and is explicitly reported rather than called exact.\n    Animated and multipage images are refused rather than picking a hidden frame.\n    """\n    p = policy or IntakePolicy()\n    data = _read_bounded(Path(path), p.max_bytes)\n    if not data:\n        raise IntakeError(\'Empty input\')\n    # Pillow exposes 16-bit RGB PNG as RGB after truncation. Inspect IHDR first.\n    if data.startswith(b\'\\x89PNG\\r\\n\\x1a\\n\') and len(data) >= 29:\n        if data[12:16] != b\'IHDR\' or data[24] > 8:\n            raise IntakeError(\'High-bit-depth PNG needs an explicit tone-mapping path\')\n    digest = hashlib.sha256(data).hexdigest()\n    transformations = []\n    warnings = []\n    try:\n        with Image.open(BytesIO(data)) as verifier:\n            _dimensions(verifier, p)\n            verifier.verify()\n        with Image.open(BytesIO(data)) as source:\n            fmt, mode, original_size = source.format, source.mode, source.size\n            if fmt not in p.formats:\n                raise IntakeError(\'Unsupported format: \' + str(fmt))\n            _dimensions(source, p)\n            if fmt == \'TIFF\':\n                bits = source.tag_v2.get(258, (8,))\n                bits = (bits,) if isinstance(bits, int) else bits\n                if any(b > 8 for b in bits):\n                    raise IntakeError(\'High-bit-depth TIFF needs an explicit tone-mapping path\')\n            if getattr(source, \'n_frames\', 1) != 1:\n                raise IntakeError(\'Animated or multipage input requires explicit frame selection\')\n            # Do not quietly clamp 16-bit/float/HDR values into 8-bit RGB.\n            if mode not in (\'RGB\', \'RGBA\', \'L\', \'LA\', \'P\', \'CMYK\'):\n                raise IntakeError(\'Unsupported bit depth or colour mode: \' + mode)\n            orientation = source.getexif().get(274, 1)\n            if orientation not in range(1, 9):\n                raise IntakeError(\'Invalid EXIF orientation\')\n            profile = source.info.get(\'icc_profile\')\n            if profile and len(profile) > 4_000_000:\n                raise IntakeError(\'ICC profile exceeds size limit\')\n            # Structural verification used a fresh stream before metadata access.\n        with Image.open(BytesIO(data)) as source:\n            source.load()\n            image = ImageOps.exif_transpose(source)\n            if orientation != 1:\n                transformations.append(\'exif_orientation_\' + str(orientation))\n            _dimensions(image, p)\n            alpha = None\n            # PNG can carry tRNS colour-key transparency in RGB and L modes,\n            # not only palette images. Extract it before RGB/ICC normalization.\n            if image.mode in (\'RGBA\', \'LA\') or \'transparency\' in image.info:\n                alpha = image.convert(\'RGBA\').getchannel(\'A\')\n            if profile:\n                if not p.allow_colour_conversion:\n                    image.close()\n                    raise IntakeError(\'Embedded ICC profile requires explicit colour-conversion policy\')\n                try:\n                    src_profile = ImageCms.ImageCmsProfile(BytesIO(profile))\n                    dst_profile = ImageCms.createProfile(\'sRGB\')\n                    # alpha is passed separately; LittleCMS consumes colour channels.\n                    base = image.convert(\'RGB\') if image.mode in (\'RGBA\', \'P\') else image\n                    image2 = ImageCms.profileToProfile(base, src_profile, dst_profile, outputMode=\'RGB\')\n                except (ValueError, OSError, ImageCms.PyCMSError) as e:\n                    image.close()\n                    raise IntakeError(\'ICC conversion failed; no unprofiled fallback\') from e\n                image.close()\n                image = image2\n                transformations.append(\'icc_to_srgb\')\n            elif mode == \'CMYK\':\n                image.close()\n                raise IntakeError(\'Unprofiled CMYK is ambiguous; supply a colour-managed source\')\n            else:\n                image2 = image.convert(\'RGB\')\n                image.close()\n                image = image2\n                if mode in (\'L\', \'LA\', \'P\'):\n                    transformations.append(\'lossless_8bit_channel_expansion\')\n                warnings.append(\'No ICC profile: RGB treated as sRGB, not colour-accuracy verified\')\n            if alpha is not None:\n                image.putalpha(alpha)\n                alpha.close()\n            # Do not carry source EXIF or an obsolete ICC tag into normalized output.\n            image.info.clear()\n            if min(image.size) < 512:\n                warnings.append(\'Low-resolution source: no upscaling or detail recovery performed\')\n            if max(image.size) / min(image.size) > 8:\n                warnings.append(\'Wide/tall source: framing and template compatibility need review\')\n            if image.mode == \'RGBA\':\n                warnings.append(\'Existing alpha retained; semantic mask correctness is not verified\')\n            record = dict(schema_version=1, source_sha256=digest,\n                          encoded_bytes=len(data), input_format=fmt, input_mode=mode,\n                          input_size=list(original_size), output_size=list(image.size),\n                          output_mode=image.mode, orientation=orientation,\n                          icc_present=bool(profile), transformations=transformations,\n                          warnings=warnings, automatic_publication_pass=False,\n                          product_identity=\'NOT_EVALUATED\', print_accuracy=\'NOT_EVALUATED\',\n                          glass_optics=\'NOT_EVALUATED\', policy=asdict(p))\n            return IntakeResult(image, record)\n    except IntakeError:\n        raise\n    except (UnidentifiedImageError, OSError, SyntaxError, ValueError,\n            Image.DecompressionBombError) as e:\n        raise IntakeError(\'Image decoding/metadata validation failed: \' + type(e).__name__) from e\n\n\ndef save_intake(result: IntakeResult, destination: str | Path) -> Path:\n    """Stage verified output in a new directory. Never replace previous outputs.\n\n    A temporary sibling prevents a half-written PNG from looking like completion.\n    Caller owns destination choice; metadata is private unless explicitly shared.\n    """\n    dest = Path(destination)\n    if dest.exists():\n        raise FileExistsError(\'Choose a new output directory\')\n    dest.parent.mkdir(parents=True, exist_ok=True)\n    stage = Path(tempfile.mkdtemp(prefix=\'.\' + dest.name + \'-\', dir=dest.parent))\n    try:\n        result.image.save(stage / \'normalized.png\', format=\'PNG\')\n        with Image.open(stage / \'normalized.png\') as check:\n            check.load()\n            if check.mode != result.image.mode or check.size != result.image.size or check.tobytes() != result.image.tobytes():\n                raise IntakeError(\'PNG round-trip differs from normalized pixels\')\n        record = dict(result.record)\n        record[\'normalized_png_sha256\'] = hashlib.sha256((stage / \'normalized.png\').read_bytes()).hexdigest()\n        (stage / \'intake.json\').write_text(json.dumps(record, indent=2), encoding=\'utf-8\')\n        # mkdir protects existing destinations, unlike an overwriting rename.\n        dest.mkdir()\n        try:\n            for child in stage.iterdir():\n                child.rename(dest / child.name)\n        except BaseException:\n            # A failure marker distinguishes partial output from completed intake.\n            (dest / \'INTAKE_INCOMPLETE\').touch()\n            raise\n        return dest\n    finally:\n        # Only our own empty temporary directory is removed, never user files.\n        if stage.exists() and not any(stage.iterdir()):\n            stage.rmdir()\n\n\ndef main(argv=None):\n    """Explicit local CLI; this does not change existing generation defaults."""\n    import argparse\n    parser = argparse.ArgumentParser(description=\'Inspect and normalize a source derivative without overwriting the source\')\n    parser.add_argument(\'--source\', required=True, type=Path)\n    parser.add_argument(\'--out\', required=True, type=Path)\n    parser.add_argument(\'--allow-colour-conversion\', action=\'store_true\')\n    args = parser.parse_args(argv)\n    result = inspect_image(args.source, IntakePolicy(allow_colour_conversion=args.allow_colour_conversion))\n    try:\n        save_intake(result, args.out)\n        print(json.dumps(result.record, indent=2))\n    finally:\n        result.close()\n\n\nif __name__ == \'__main__\':\n    main()\n'
# ---- embedded module: renderello/image_intake.py ----
path = base / 'renderello/image_intake.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Explicit reviewed intent gate for an existing automatic alpha. No inference or acceptance."""\nfrom pathlib import Path\nimport hashlib,json\nimport numpy as np\nfrom PIL import Image\nfrom .prompted_mask import validate_prompts,MODEL_SHA256 as HQ_SHA256\n\ndef sha(p):return hashlib.sha256(p.read_bytes()).hexdigest()\n\ndef load_bundle(folder,manifest_name,source_hash,size):\n    m=json.loads((folder/manifest_name).read_text())\n    if m.get(\'source_sha256\')!=source_hash:raise ValueError(\'bundle source SHA256 mismatch\')\n    paths=(\'estimated-alpha.png\',\'source.png\',\'cutout.png\')\n    for name in paths:\n        if m.get(\'output_sha256\',{}).get(name)!=sha(folder/name):raise ValueError(\'bundle output SHA256 mismatch: \'+name)\n    with Image.open(folder/\'estimated-alpha.png\') as im:\n        if im.mode!=\'L\' or im.size!=size:raise ValueError(\'alpha must be source-sized L image\')\n        a=np.array(im)\n    if not np.any(a):raise ValueError(\'empty input alpha\')\n    return m,a\n\ndef gate_file(source:Path,automatic:Path,reviewed:Path,out:Path):\n    if out.exists():raise ValueError(\'output exists; preserve attempts\')\n    with Image.open(source) as im:\n        if im.width*im.height>4_000_000:raise ValueError(\'source exceeds4million pixels\')\n        rgb=im.convert(\'RGB\');pixels=np.array(rgb);size=im.size\n    source_hash=sha(source)\n    candidates=[n for n in (\'bilateral-mask-manifest.json\',\'pyramid-mask-manifest.json\',\'semantic-mask-manifest.json\') if (automatic/n).is_file()]\n    if len(candidates)!=1:raise ValueError(\'exactly one supported automatic manifest required\')\n    auto,a=load_bundle(automatic,candidates[0],source_hash,size)\n    prompt,b=load_bundle(reviewed,\'prompted-mask-manifest.json\',source_hash,size)\n    if prompt.get(\'model_sha256\')!=HQ_SHA256:raise ValueError(\'unsupported reviewed HQ-SAM model\')\n    pp=reviewed/\'reviewed-prompts.json\'\n    if sha(pp)!=prompt.get(\'prompt_sha256\'):raise ValueError(\'reviewed prompt SHA256 mismatch\')\n    data=json.loads(pp.read_text());objects=validate_prompts(data,source,*size)\n    if prompt.get(\'prompt_origin\')!=\'reviewed_source_geometry\' or prompt.get(\'objects\')!=[o[\'name\'] for o in objects]:raise ValueError(\'reviewed intent provenance mismatch\')\n    if not np.isin(b,[0,255]).all():raise ValueError(\'reviewed mask must be binary\')\n    for folder,expected_alpha in ((automatic,a),(reviewed,b)):\n        with Image.open(folder/\'source.png\') as im:\n            if not np.array_equal(np.array(im.convert(\'RGB\')),pixels):raise ValueError(\'bundle decoded RGB mismatch\')\n        with Image.open(folder/\'cutout.png\') as im:\n            if im.mode!=\'RGBA\' or not np.array_equal(np.array(im)[:,:,:3],pixels):raise ValueError(\'bundle cutout RGB mismatch\')\n            if not np.array_equal(np.array(im)[:,:,3],expected_alpha):raise ValueError(\'bundle cutout alpha mismatch\')\n    alpha=np.minimum(a,b)\n    if not np.any(alpha):raise ValueError(\'empty intersection; incompatible masks\')\n    out.mkdir(parents=True)\n    for name,v in [(\'source.png\',pixels),(\'estimated-alpha.png\',alpha),(\'automatic-alpha.png\',a),(\'reviewed-alpha.png\',b),(\'cutout.png\',np.dstack([pixels,alpha]))]:Image.fromarray(v).save(out/name)\n    for name,col in [(\'black\',(20,20,20)),(\'white\',(250,250,250)),(\'magenta\',(190,0,150))]:\n        bg=Image.new(\'RGB\',size,col);bg.paste(rgb,mask=Image.fromarray(alpha));bg.save(out/(name+\'-review.png\'))\n    (out/\'reviewed-prompts.json\').write_bytes(pp.read_bytes())\n    report=dict(source_sha256=source_hash,source_rgb_unchanged=True,method=\'minimum(automatic alpha, reviewed binary HQ-SAM intent)\',automatic_manifest_sha256=sha(automatic/candidates[0]),reviewed_manifest_sha256=sha(reviewed/\'prompted-mask-manifest.json\'),prompt_sha256=sha(pp),objects=[o[\'name\'] for o in objects],input_automatic_model=auto.get(\'model\'),removed_nonzero_pixels=int(np.count_nonzero((a>0)&(alpha==0))),retained_alpha_values_exact=True,quality=\'NOT_EVALUATED\',semantic_completeness=\'NOT_VERIFIED\',source_framing=\'NOT_VERIFIED\',warnings=[\'Manual reviewed intent, not an automatic success. Boxes/points need source review.\',\'Intersection only removes; it cannot recover any automatic-mask missing pixels.\',\'HQ-SAM errors and hard jagged boundaries transfer into the intersection. No optical glass matting.\',\'Source RGB stays exact, including source-color fringe. No quality acceptance or default switch.\'])\n    report[\'output_sha256\']={p.name:sha(p) for p in out.iterdir() if p.is_file()}\n    (out/\'intent-mask-manifest.json\').write_text(json.dumps(report,indent=2));return report\n'
# ---- embedded module: renderello/intent_mask.py ----
path = base / 'renderello/intent_mask.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Same-coordinate source-pixel and alpha retention checks, not OCR truth.\nA correct string cannot be inferred from byte equality or invented by this tool.\n"""\nfrom pathlib import Path\nimport json\nimport numpy as np\nfrom PIL import Image\nfrom .intent_mask import sha\n\n\ndef check_regions(source:Path,cutout:Path,regions:dict):\n    with Image.open(source) as im:\n        if im.width*im.height>4_000_000:raise ValueError(\'source exceeds four million pixels\')\n        rgb=np.array(im.convert(\'RGB\'));size=list(im.size)\n    if regions.get(\'source_sha256\')!=sha(source) or regions.get(\'image_size\')!=size:raise ValueError(\'regions must bind exact source SHA256 and dimensions\')\n    with Image.open(cutout) as im:\n        if im.mode!=\'RGBA\' or list(im.size)!=size:raise ValueError(\'cutout must be same-coordinate source-sized RGBA; resampled scenes are not supported\')\n        rgba=np.array(im)\n    items=regions.get(\'regions\')\n    if not isinstance(items,list) or not 1<=len(items)<=32:raise ValueError(\'one to32 regions required\')\n    names=set();results=[]\n    for item in items:\n        name=item.get(\'name\');b=item.get(\'box\');threshold=item.get(\'minimum_alpha\',255)\n        if not isinstance(name,str) or not name.strip() or name in names:raise ValueError(\'unique nonempty region names required\')\n        names.add(name)\n        if not isinstance(b,list) or len(b)!=4 or any(type(x)!=int for x in b) or not (0<=b[0]<b[2]<=size[0] and 0<=b[1]<b[3]<=size[1]):raise ValueError(\'region box must be valid integer source coordinates\')\n        if type(threshold)!=int or not 0<=threshold<=255:raise ValueError(\'minimum alpha must be integer0-255\')\n        x1,y1,x2,y2=b;a=rgba[y1:y2,x1:x2,3];diff=np.abs(rgba[y1:y2,x1:x2,:3].astype(\'int16\')-rgb[y1:y2,x1:x2].astype(\'int16\'));changed=np.any(diff,axis=2);lost=a<threshold\n        # Evaluate the actual RGB seen after source-over, not only hidden RGBA RGB.\n        # Preserve configurable retention semantics; exact compositing is separate.\n        rgb_region=rgb[y1:y2,x1:x2]\n        region_image=Image.fromarray(rgba[y1:y2,x1:x2], \'RGBA\')\n        composite_checks={}\n        for background,color in [(\'white\',(255,255,255)),(\'black\',(0,0,0)),(\'magenta\',(255,0,255))]:\n            bg=Image.new(\'RGBA\',region_image.size,(*color,255))\n            seen=np.array(Image.alpha_composite(bg,region_image).convert(\'RGB\'))\n            error=np.abs(seen.astype(\'int16\')-rgb_region.astype(\'int16\'))\n            composite_checks[background]=dict(changed_pixels=int(np.any(error,axis=2).sum()),max_channel_error=int(error.max()),mean_channel_error=float(error.mean()))\n        exact_composite=all(x[\'changed_pixels\']==0 for x in composite_checks.values())\n        results.append(dict(name=name,box=b,pixels=int(a.size),source_rgb_exact=not bool(changed.any()),rgb_changed_pixels=int(changed.sum()),rgb_max_error=int(diff.max()),minimum_alpha_observed=int(a.min()),minimum_alpha_required=threshold,below_required_alpha_pixels=int(lost.sum()),retention_check_passed=not bool(changed.any() or lost.any()),composite_checks=composite_checks,source_native_composite_exact=exact_composite))\n    return dict(source_sha256=sha(source),cutout_sha256=sha(cutout),regions_sha256=None,image_size=size,regions=results,all_retention_checks_passed=all(x[\'retention_check_passed\'] for x in results),all_source_native_composites_exact=all(x[\'source_native_composite_exact\'] for x in results),quality=\'NOT_EVALUATED\',transcription=\'NOT_PERFORMED\',warnings=[\'Same-coordinate byte and opacity checks only, not independent reading or verification of label text.\',\'RGBA RGB equality is not visible composite equality. Composite checks use actual Pillow source-over on white, black and magenta.\',\'A source may already be blurry, wrong or incomplete. Region selection and minimum opacity must be reviewed.\',\'Transparent regions should not be declared opaque. This tool does not estimate optical transmission.\'])\n\n\ndef check_file(source,cutout,regions_path,out):\n    if out.exists():raise ValueError(\'output exists; preserve previous reports\')\n    regions=json.loads(regions_path.read_text());report=check_regions(source,cutout,regions);report[\'regions_sha256\']=sha(regions_path);out.parent.mkdir(parents=True,exist_ok=True);out.write_text(json.dumps(report,indent=2));return report\n'
# ---- embedded module: renderello/label_retention.py ----
path = base / 'renderello/label_retention.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Optional local OCR mismatch tripwire, never exact-text/brand acceptance.\n\nUses an installed Tesseract binary only. No cloud upload, download or default\npipeline change. OCR can misread legitimate print, miss invented print, and\nignore small text. A quiet result never certifies identity or publication.\n"""\nfrom __future__ import annotations\nfrom collections import Counter\nimport csv, io, re, shutil, subprocess\nfrom PIL import Image\n\n\ndef parse_tsv(text: str, *, confidence: float=70., min_length: int=4) -> list[dict]:\n    if not isinstance(text,str):raise ValueError(\'TSV text required\')\n    if type(confidence) not in (int,float) or not 0<=confidence<=100:raise ValueError(\'confidence0..100 required\')\n    if type(min_length) is not int or not 1<=min_length<=40:raise ValueError(\'min_length1..40 required\')\n    if len(text)>8_000_000:raise ValueError(\'OCR report exceeds8MB budget\')\n    rows=csv.DictReader(io.StringIO(text),delimiter=\'\\t\')\n    required={\'level\',\'conf\',\'text\',\'left\',\'top\',\'width\',\'height\'}\n    if not required.issubset(rows.fieldnames or []):raise ValueError(\'Unexpected Tesseract TSV header\')\n    out=[]\n    for row in rows:\n        if row.get(\'level\')!=\'5\':continue\n        try:c=float(row[\'conf\']);box=[int(row[k])for k in (\'left\',\'top\',\'width\',\'height\')]\n        except (ValueError,TypeError):continue\n        if not confidence<=c<=100 or min(box)<0 or min(box[2:])<=0:continue\n        token=re.sub(\'[^A-Z0-9]\',\'\',(row.get(\'text\')or\'\').upper())\n        if len(token)<min_length:continue\n        out.append(dict(token=token,observed_text=row[\'text\'],confidence=c,box=box))\n    return out\n\n\ndef extract_words(image: Image.Image, *, confidence: float=70., timeout: int=30) -> list[dict]:\n    if not isinstance(image,Image.Image)or image.mode!=\'RGB\':raise ValueError(\'PIL RGB image required; no hidden colour conversion\')\n    if image.width*image.height>4_000_000:raise ValueError(\'OCR native4MPbudget; no silent downscale\')\n    if type(timeout)is not int or not 1<=timeout<=60:raise ValueError(\'timeout1..60 required\')\n    binary=shutil.which(\'tesseract\')\n    if not binary:raise RuntimeError(\'Optional Tesseract is not installed; no OCR result\')\n    b=io.BytesIO();image.save(b,format=\'PNG\')\n    result=subprocess.run([binary,\'stdin\',\'stdout\',\'-l\',\'eng\',\'--psm\',\'11\',\'tsv\'],input=b.getvalue(),stdout=subprocess.PIPE,stderr=subprocess.PIPE,timeout=timeout,check=False)\n    if result.returncode:raise RuntimeError(\'Tesseract failed: \'+result.stderr.decode(\'utf-8\',\'replace\')[:500])\n    return parse_tsv(result.stdout.decode(\'utf-8\',\'replace\'),confidence=confidence)\n\n\ndef compare_words(source: list[dict], output: list[dict], *, warning_recall: float=.8) -> dict:\n    if type(warning_recall)not in(int,float)or not 0<warning_recall<=1:raise ValueError(\'warning_recall must be0..1\')\n    def counts(rows):\n        if not isinstance(rows,list)or any(not isinstance(r,dict)or not isinstance(r.get(\'token\'),str)or not re.fullmatch(\'[A-Z0-9]+\',r[\'token\'])for r in rows):raise ValueError(\'typed normalized OCR word records required\')\n        return Counter(r[\'token\']for r in rows)\n    a,b=counts(source),counts(output);n=sum(a.values());same=sum((a&b).values());recall=same/n if n else None\n    # Fewer than3sourcewords cannot support a useful retention diagnostic.\n    status=\'INSUFFICIENT_SOURCE_OCR\'if n<3 else(\'OCR_MISMATCH_REVIEW_REQUIRED\'if recall<warning_recall else \'NO_OCR_TRIPWIRE_NOT_A_TEXT_PASS\')\n    return dict(schema=\'renderello-label-ocr-audit.v1\',status=status,source_recognized_words=n,output_recognized_words=sum(b.values()),matched_source_words=same,source_word_recall=recall,missing_source_words=list((a-b).elements()),new_output_words=list((b-a).elements()),source_words=source,output_words=output,product_text_verified=False,photo_accepted=False,default_pipeline_changed=False,limitations=[\'OCR is fallible and incomplete; exact normalized word overlap is not text-order, typography, logo or identity proof.\',\'Lighting, resolution and viewpoint can change OCR without actual label changes.\',\'High-confidence OCR may still misread print; manual native-pixel comparison remains necessary.\'])\n\n\ndef audit_label_text(source: Image.Image, output: Image.Image) -> dict:\n    return compare_words(extract_words(source),extract_words(output))\n'
# ---- embedded module: renderello/label_text_audit.py ----
path = base / 'renderello/label_text_audit.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Explicit long SDXL prompt encoding with Compel\'s dual-CLIP contract.\n\nNo silent 77-token truncation. Chunking includes the whole text but does not\npromise that a model obeys each clause or that scene quality improves.\n"""\n\n\ndef encode_long_sdxl(pipe, prompt, negative_prompt=\'\', *, device=\'cuda\'):\n    if not isinstance(prompt,str) or not prompt.strip():\n        raise ValueError(\'nonempty full prompt required\')\n    if not isinstance(negative_prompt,str):raise ValueError(\'negative prompt must be text\')\n    # Bound memory explicitly and report an error instead of silently truncating.\n    if len(prompt)>16000 or len(negative_prompt)>16000:\n        raise ValueError(\'prompt exceeds explicit16000character budget; no truncation performed\')\n    import torch\n    from compel import Compel,ReturnedEmbeddingsType\n    processor=Compel(tokenizer=[pipe.tokenizer,pipe.tokenizer_2],\n        text_encoder=[pipe.text_encoder,pipe.text_encoder_2],\n        returned_embeddings_type=ReturnedEmbeddingsType.PENULTIMATE_HIDDEN_STATES_NON_NORMALIZED,\n        requires_pooled=[False,True],truncate_long_prompts=False,device=device,\n        dtype_for_device_getter=lambda _:torch.float16)\n    with torch.inference_mode():\n        positive,positive_pooled=processor(prompt)\n        negative,negative_pooled=processor(negative_prompt)\n        empty,_=processor(\'\')\n        # Explicit empty conditioning avoids the installed multi-provider\'s\n        # missing empty_z attribute on its implicit batch-padding path.\n        positive,negative=processor.pad_conditioning_tensors_to_same_length(\n            [positive,negative],precomputed_padding=empty)\n        conditioning=torch.cat([positive,negative],dim=0)\n        pooled=torch.cat([positive_pooled,negative_pooled],dim=0)\n    counts=[len(t(prompt,add_special_tokens=False)[\'input_ids\']) for t in [pipe.tokenizer,pipe.tokenizer_2]]\n    diagnostic=dict(method=\'Compel dual CLIP nontruncating chunks\',positive_tokens_by_encoder=counts,\n        conditioning_shape=list(conditioning.shape),pooled_shape=list(pooled.shape),\n        truncate_long_prompts=False,full_prompt=prompt,negative_prompt=negative_prompt,\n        constraint_obedience=\'NOT_GUARANTEED\',quality_acceptance=\'NOT_EVALUATED\')\n    return dict(prompt_embeds=conditioning[0:1],negative_prompt_embeds=conditioning[1:2],\n                pooled_prompt_embeds=pooled[0:1],negative_pooled_prompt_embeds=pooled[1:2]),diagnostic\n'
# ---- embedded module: renderello/long_prompt_sdxl.py ----
path = base / 'renderello/long_prompt_sdxl.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Long, detailed prompt support.\n\nThe owner requirement (2026-11-20): the pipeline must accept very long,\ndetailed prompts, not just short instructions. Two constraints shape this\nmodule:\n\n1. CLIP-based diffusion models (SD1.5, SDXL) tokenize with a hard context\n   window (77 tokens for CLIP-L, 75 usable). Long prompts must be chunked\n   and the embeddings pooled, or the tail is silently dropped.\n2. API editors (Magic Hour, Qwen) take plain strings with their own limits;\n   for them the full text passes through unchanged.\n\nThis module never truncates silently: chunking preserves every clause, and\n`PromptPlan.report()` states exactly how the prompt will reach the model.\n"""\nfrom __future__ import annotations\n\nimport re\nfrom dataclasses import dataclass, field\n\n# CLIP BPE tokens roughly track word+punctuation pieces; this regex\n# approximation stays within ~10% of the real tokenizer, which is enough to\n# split safely below the 75-token usable window.\n_TOKEN_RE = re.compile(r"\\w+|[^\\w\\s]")\nCLAIP_USABLE_TOKENS = 75  # 77 minus BOS/EOS\nSDXL_DUAL_ENCODER = 2     # CLIP-L + OpenCLIP-G each get the same chunks\n\n\ndef approx_token_count(text: str) -> int:\n    """Approximate CLIP token count (overestimates slightly; safe side)."""\n    return len(_TOKEN_RE.findall(text))\n\n\ndef _split_clauses(text: str) -> list[str]:\n    """Split at sentence / clause boundaries, keeping the separators."""\n    parts = re.split(r"(?<=[.!?;])\\s+", text.strip())\n    clauses: list[str] = []\n    for part in parts:\n        if approx_token_count(part) <= CLAIP_USABLE_TOKENS - 8:\n            clauses.append(part)\n            continue\n        # Long sentence: fall back to comma boundaries, then hard word wraps.\n        sub = re.split(r"(?<=,)\\s+", part)\n        buf = ""\n        for piece in sub:\n            if approx_token_count(piece) > CLAIP_USABLE_TOKENS - 8:\n                words = piece.split()\n                for w in words:\n                    cand = f"{buf} {w}".strip()\n                    if approx_token_count(cand) > CLAIP_USABLE_TOKENS - 8:\n                        if buf:\n                            clauses.append(buf)\n                        buf = w\n                    else:\n                        buf = cand\n            else:\n                cand = f"{buf} {piece}".strip()\n                if approx_token_count(cand) > CLAIP_USABLE_TOKENS - 8:\n                    if buf:\n                        clauses.append(buf)\n                    buf = piece\n                else:\n                    buf = cand\n        if buf:\n            clauses.append(buf)\n    return [c for c in clauses if c.strip()]\n\n\ndef chunk_prompt(text: str, max_tokens: int = CLAIP_USABLE_TOKENS - 8) -> list[str]:\n    """Pack clauses into chunks that each fit one CLIP context window.\n\n    Returns at least one chunk. No clause is dropped or truncated; a clause\n    longer than the window is hard-wrapped at word boundaries.\n    """\n    clauses = _split_clauses(text)\n    if not clauses:\n        return [""]\n    chunks: list[str] = []\n    buf = ""\n    for clause in clauses:\n        cand = f"{buf} {clause}".strip()\n        if approx_token_count(cand) <= max_tokens:\n            buf = cand\n        else:\n            if buf:\n                chunks.append(buf)\n            buf = clause\n    if buf:\n        chunks.append(buf)\n    return chunks\n\n\n@dataclass\nclass PromptPlan:\n    """The full resolved prompt for one generation, plus how it will be fed."""\n    full_text: str\n    negative: str\n    chunks: list[str] = field(default_factory=list)\n    provider: str = "unknown"\n    approx_tokens: int = 0\n    notes: list[str] = field(default_factory=list)\n\n    def report(self) -> dict:\n        return {\n            "provider": self.provider,\n            "approx_tokens": self.approx_tokens,\n            "chunk_count": len(self.chunks),\n            "chars": len(self.full_text),\n            "notes": list(self.notes),\n        }\n\n\ndef plan_prompt(full_text: str, negative: str, provider: str) -> PromptPlan:\n    """Build the feeding plan for a provider.\n\n    - diffusers/CLIP providers (sdxl_lightning, iclight): chunked at the CLIP\n      window; the GPU side must embed per chunk and pool (see\n      `encode_long_prompt`).\n    - string providers (magichour, hf_inference, qwen, procedural): the full\n      text is passed unchanged; only the leading section matters most.\n    """\n    plan = PromptPlan(full_text=full_text.strip(), negative=negative.strip(),\n                      provider=provider)\n    plan.approx_tokens = approx_token_count(plan.full_text)\n    if provider in {"sdxl_lightning", "iclight", "sd15"}:\n        plan.chunks = chunk_prompt(plan.full_text)\n        if len(plan.chunks) > 1:\n            plan.notes.append(\n                f"long prompt split into {len(plan.chunks)} CLIP-window chunks; "\n                "chunk embeddings are sequence-concatenated per encoder before "\n                "inference (padding excluded, no clause dropped)")\n    else:\n        plan.chunks = [plan.full_text]\n        plan.notes.append("provider consumes the prompt as a single string")\n    return plan\n\n\ndef encode_long_prompt(pipe, prompt: str, device: str, negative: str = ""):\n    """Sequence-concatenated multi-chunk text encoding for diffusers pipelines.\n\n    Works for SDXL (two text encoders) and SD1.5 (one). Each chunk fits the\n    model\'s context window, so no tokens are dropped; the per-chunk hidden\n    states (padding excluded via the attention mask) are concatenated along\n    the sequence axis, so the UNet attends over every clause - the previous\n    version mean-pooled each chunk to one token and then averaged chunks,\n    collapsing any prompt to a single conditioning token. Prompt and\n    negative sequences are length-equalized by repeating the shorter one\'s\n    final valid token (SDXL requires equal sequence lengths). Requires\n    torch + the pipeline\'s own tokenizer(s); import-lazy so CPU installs\n    never need torch.\n\n    NOTE (2026-11-21): this fixes the conditioning STRUCTURE only.\n    Photographic quality improvement is unproven - free-tier Colab runs\n    crashed before producing comparable pixels, so no before/after visual\n    validation exists yet.\n    """\n    import torch\n\n    chunks = chunk_prompt(prompt)\n    neg_chunks = chunk_prompt(negative) if negative else [""]\n\n    def _encode(chunks_: list[str], tokenizer, encoder) -> "torch.Tensor":\n        embs = []\n        for ch in chunks_:\n            tokens = tokenizer(ch, padding="max_length", truncation=True,\n                               max_length=tokenizer.model_max_length,\n                               return_tensors="pt").to(device)\n            with torch.no_grad():\n                out = encoder(tokens.input_ids, output_hidden_states=True)\n            hidden = out.hidden_states[-2] if hasattr(out, "hidden_states") else out[0]\n            valid = int(tokens.attention_mask[0].sum().item())\n            embs.append(hidden[:, :valid].float())\n        return torch.cat(embs, dim=1).to(dtype=encoder.dtype)\n\n    def _pad_to(t: "torch.Tensor", length: int) -> "torch.Tensor":\n        if t.shape[1] >= length:\n            return t\n        pad = t[:, -1:].expand(-1, length - t.shape[1], -1)\n        return torch.cat([t, pad], dim=1)\n\n    if hasattr(pipe, "tokenizer_2") and pipe.tokenizer_2 is not None:\n        pe1 = _encode(chunks, pipe.tokenizer, pipe.text_encoder)\n        pe2 = _encode(chunks, pipe.tokenizer_2, pipe.text_encoder_2)\n        ne1 = _encode(neg_chunks, pipe.tokenizer, pipe.text_encoder)\n        ne2 = _encode(neg_chunks, pipe.tokenizer_2, pipe.text_encoder_2)\n        seq = max(pe1.shape[1], ne1.shape[1])\n        pe1, ne1 = _pad_to(pe1, seq), _pad_to(ne1, seq)\n        pe2, ne2 = _pad_to(pe2, seq), _pad_to(ne2, seq)\n        prompt_embeds = torch.cat([pe1, pe2], dim=-1)\n        negative_embeds = torch.cat([ne1, ne2], dim=-1)\n        pooled = pe2.mean(dim=1)\n        neg_pooled = ne2.mean(dim=1)\n        return prompt_embeds, negative_embeds, pooled, neg_pooled\n    pe = _encode(chunks, pipe.tokenizer, pipe.text_encoder)\n    ne = _encode(neg_chunks, pipe.tokenizer, pipe.text_encoder)\n    seq = max(pe.shape[1], ne.shape[1])\n    return _pad_to(pe, seq), _pad_to(ne, seq), None, None\n'
# ---- embedded module: renderello/longprompt.py ----
path = base / 'renderello/longprompt.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Opt-in Lucida alpha research. Not a refractive model or accepted glass result."""\nfrom pathlib import Path\nimport hashlib,json,time\nimport numpy as np\nfrom PIL import Image\nREVISION=\'6cbedc9722652dc9a3df91dd871f0c4f3334e922\'\nMODEL=\'egeorcun/lucida\'\n# Pin publisher files, not a moving main revision or unrestricted remote code.\nFILES={\n \'__init__.py\':(\'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\',0),\n \'BiRefNet_config.py\':(\'e7b8c2a74f6cea6a59553d517f71d47f2c1d90e670a13416af17c25fe2f3dc52\',298),\n \'model.safetensors\':(\'2f1aa6913426537d4b93dd5f7138ae5c6664e99abda98c95f3d5b9101283e7d5\',884878856),\n \'birefnet.py\':(\'2a45b4e0ece72d7c4212bca1a988e7d7e52bfe9f98ec59c58b8809c8a8b7a831\',92134),\n}\n\ndef digest(path):\n h=hashlib.sha256()\n with Path(path).open(\'rb\') as f:\n  for b in iter(lambda:f.read(4*1024*1024),b\'\'):h.update(b)\n return h.hexdigest()\n\ndef verify_model(directory):\n directory=Path(directory)\n for name,(sha,size) in FILES.items():\n  p=directory/name\n  if p.is_symlink() or not p.is_file():raise ValueError(\'missing or symbolic publisher file: \'+name)\n  if p.stat().st_size!=size or digest(p)!=sha:raise ValueError(\'publisher file checksum mismatch: \'+name)\n return directory\n\ndef export_review(source,alpha,out,*,inference=None):\n """Write RGB-preserving review artifacts; inferred alpha alone never earns acceptance."""\n source,out=Path(source),Path(out)\n if out.exists():raise ValueError(\'output exists; preserve earlier attempts\')\n with Image.open(source) as im:\n  if im.width*im.height>4_000_000:raise ValueError(\'source exceeds four million pixels\')\n  rgb=np.array(im.convert(\'RGB\'))\n alpha=np.asarray(alpha)\n if alpha.dtype!=np.uint8 or alpha.shape!=rgb.shape[:2]:raise ValueError(\'alpha must be source-sized uint8\')\n if not alpha.any():raise ValueError(\'empty inferred alpha\')\n out.mkdir(parents=True)\n Image.fromarray(rgb).save(out/\'source.png\');Image.fromarray(alpha).save(out/\'estimated-alpha.png\')\n rgba=Image.fromarray(np.dstack([rgb,alpha]));rgba.save(out/\'cutout.png\')\n for title,color in [(\'white\',(255,255,255)),(\'black\',(0,0,0)),(\'magenta\',(255,0,255))]:\n  bg=Image.new(\'RGBA\',rgba.size,(*color,255));Image.alpha_composite(bg,rgba).convert(\'RGB\').save(out/(title+\'-review.png\'))\n report=dict(model=MODEL,revision=REVISION,source_sha256=digest(source),source_rgb_unchanged=True,\n  alpha_sha256=hashlib.sha256(alpha.tobytes()).hexdigest(),quality=\'NOT_EVALUATED\',semantic_completeness=\'NOT_VERIFIED\',\n  source_framing=\'NOT_VERIFIED\',refraction_estimated=False,background_color_removed=False,\n  license_note=\'Publisher declares MIT weights/code. Training dataset rights are not independently cleared.\',\n  warnings=[\'Soft alpha is not refractive flow or transmission.\',\'Source RGB retains reflected/refracted old background.\',\'Source crops remain incomplete.\',\'Inspect actual pixels on every review background.\'],\n  inference=inference or {})\n report[\'output_sha256\']={p.name:digest(p) for p in out.iterdir() if p.is_file()}\n (out/\'lucida-mask-manifest.json\').write_text(json.dumps(report,indent=2))\n return report\n\ndef infer_file(source,out,model_dir):\n """Use pre-reviewed pinned local code and safetensors, never auto-download."""\n import sys,importlib.util\n source,out=Path(source),Path(out)\n if out.exists():raise ValueError(\'output exists; preserve earlier attempts\')\n with Image.open(source) as im:\n  if im.width*im.height>4_000_000:raise ValueError(\'source exceeds four million pixels\')\n  rgb=im.convert(\'RGB\')\n root=verify_model(model_dir)\n import torch,torchvision\n from safetensors.torch import load_file\n if not torch.cuda.is_available():raise RuntimeError(\'isolated CUDA trial only; no implicit CPU fallback\')\n started=time.monotonic()\n spec=importlib.util.spec_from_file_location(\'_renderello_pinned_lucida\',root/\'__init__.py\',submodule_search_locations=[str(root)])\n package=importlib.util.module_from_spec(spec);sys.modules[spec.name]=package;spec.loader.exec_module(package)\n from importlib import import_module\n Model=import_module(spec.name+\'.birefnet\').BiRefNet\n Config=import_module(spec.name+\'.BiRefNet_config\').BiRefNetConfig\n model=Model(config=Config(bb_pretrained=False))\n model.load_state_dict(load_file(str(root/\'model.safetensors\'),device=\'cpu\'),strict=True)\n model.eval().to(\'cuda\')\n transform=torchvision.transforms.Compose([torchvision.transforms.Resize((1024,1024)),torchvision.transforms.ToTensor(),torchvision.transforms.Normalize([.485,.456,.406],[.229,.224,.225])])\n with torch.inference_mode():prediction=model(transform(rgb).unsqueeze(0).to(\'cuda\'))[-1].sigmoid().cpu()[0,0]\n a=Image.fromarray((prediction.numpy()*255).astype(np.uint8)).resize(rgb.size,Image.Resampling.BILINEAR)\n return export_review(source,np.array(a),out,inference=dict(device=torch.cuda.get_device_name(0),seconds=time.monotonic()-started,weight_sha256=FILES[\'model.safetensors\'][0]))\n\ndef guarded_file(source,out,model_dir):\n import subprocess,sys\n cmd=[sys.executable,\'-m\',\'renderello.lucida_mask\',\'--source\',str(source),\'--out\',str(out),\'--model-dir\',str(model_dir)]\n try:p=subprocess.run(cmd,capture_output=True,text=True,timeout=180)\n except subprocess.TimeoutExpired as e:raise RuntimeError(\'Lucida 180 second worker timeout; partial output is not accepted\') from e\n if p.returncode:raise RuntimeError(\'Lucida worker failed: \'+p.stderr[-2000:])\n return json.loads(p.stdout)\n\ndef main():\n import argparse\n p=argparse.ArgumentParser(description=__doc__)\n for key in [\'source\',\'out\',\'model-dir\']:p.add_argument(\'--\'+key,type=Path,required=True)\n a=p.parse_args();print(json.dumps(infer_file(a.source,a.out,a.model_dir)))\nif __name__==\'__main__\':main()\n'
# ---- embedded module: renderello/lucida_mask.py ----
path = base / 'renderello/lucida_mask.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Frozen matched trials and read-only evidence accounting.\n\nNo generation, payment, upload, identity authentication or quality approval.\nA plan is NOT execution permission. Missing first tries remain in denominators.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport math\nimport statistics\nimport struct\nfrom pathlib import Path\n\nfrom .image_intake import inspect_image, IntakePolicy\nfrom .review import DIMENSIONS\n\nSCHEMA = \'renderello.matched-trial.v1\'\nKINDS = {\'local\', \'magic_hour\', \'specialist\'}\nSTATES = {\'output\', \'failed\', \'unconfirmed\'}\nTAGS = {\'tiny_text\', \'reflective\', \'transparent\', \'shoes\', \'bags\', \'apparel\',\n        \'food_packaging\', \'multiple_products\', \'complex_edges\', \'irregular_shape\'}\n\n\ndef _sha(data):\n    return hashlib.sha256(data).hexdigest()\n\n\ndef _canonical(obj):\n    return json.dumps(obj, sort_keys=True, separators=(\',\', \':\'),\n                      ensure_ascii=True, allow_nan=False).encode(\'utf-8\')\n\n\ndef _unique_pairs(pairs):\n    obj = {}\n    for key, value in pairs:\n        if key in obj:\n            raise ValueError(\'duplicate JSON key: \' + key)\n        obj[key] = value\n    return obj\n\n\ndef read_json(path):\n    p = Path(path)\n    if p.is_symlink() or not p.is_file() or p.stat().st_size > 8_000_000:\n        raise ValueError(\'bounded regular JSON file required\')\n    return json.loads(p.read_text(encoding=\'utf-8\'), object_pairs_hook=_unique_pairs,\n                      parse_constant=lambda v: (_ for _ in ()).throw(ValueError(v)))\n\n\ndef _text(value, label, limit=32000):\n    if not isinstance(value, str) or not value.strip() or len(value) > limit:\n        raise ValueError(label + \': nonempty bounded text required\')\n    return value\n\n\ndef _identifier(value):\n    _text(value, \'identifier\', 80)\n    if not value.isascii() or not all(c.isalnum() or c in \'_-\' for c in value):\n        raise ValueError(\'identifier must be ASCII letters, digits, underscore or hyphen\')\n    return value\n\n\ndef _number(value, label):\n    if isinstance(value, bool) or not isinstance(value, (int, float)):\n        raise ValueError(label + \': numeric value required\')\n    if not math.isfinite(value) or value < 0:\n        raise ValueError(label + \': finite nonnegative value required\')\n    return value\n\n\ndef _relative(root, name):\n    if not isinstance(name, str) or not name or \'\\\\\' in name or \':\' in name:\n        raise ValueError(\'portable relative artifact path required\')\n    p = Path(name)\n    if p.is_absolute() or any(x in (\'.\', \'..\') for x in name.split(\'/\')):\n        raise ValueError(\'artifact path traversal refused\')\n    root = Path(root).resolve()\n    current = root\n    for part in p.parts:\n        current = current / part\n        if current.is_symlink():\n            raise ValueError(\'symlink artifact refused\')\n    if not current.is_file() or not current.resolve().is_relative_to(root):\n        raise ValueError(\'artifact missing or outside evidence root\')\n    return current\n\n\ndef pixel_identity(image):\n    """EXIF/profile-normalized RGBA samples, including size; not perceptual identity.\n\n    Different JPEG encodings can decode differently; near duplicates still need\n    curator review. Hidden transparent RGB is retained, not silently discarded.\n    """\n    rgba = image.convert(\'RGBA\')\n    try:\n        return _sha(struct.pack(\'>II\', *rgba.size) + rgba.tobytes())\n    finally:\n        rgba.close()\n\n\ndef freeze_plan(spec, out):\n    """Validate all inputs, then save immutable normalized source snapshots.\n\n    spec carries source paths and declared tags, not proof of photo origin or\n    publication rights. A complete marker binds all files. Failed preparation\n    leaves an incomplete new directory, never replaces an earlier experiment.\n    """\n    out = Path(out)\n    if out.exists():\n        raise FileExistsError(\'new destination required; preserve prior experiment\')\n    cases = spec.get(\'cases\')\n    methods = spec.get(\'methods\')\n    if not isinstance(cases, list) or not 1 <= len(cases) <= 500:\n        raise ValueError(\'one to 500 predeclared cases required\')\n    if not isinstance(methods, list) or not 2 <= len(methods) <= 12:\n        raise ValueError(\'two to twelve methods required\')\n    method_ids = set()\n    for method in methods:\n        mid = _identifier(method[\'id\'])\n        if mid in method_ids or method.get(\'kind\') not in KINDS:\n            raise ValueError(\'unique methods and known method kinds required\')\n        method_ids.add(mid)\n        if type(method.get(\'selection_complete\')) is not bool:\n            raise ValueError(\'explicit method-selection state required\')\n        if not isinstance(method.get(\'settings\'), dict) or not method[\'settings\']:\n            raise ValueError(\'method settings required, including unresolved choices\')\n        _canonical(method)  # Refuse NaN or unserializable configuration now.\n    decoded = []\n    seen_ids = set()\n    unique_pixels = {}\n    try:\n        for case in cases:\n            cid = _identifier(case[\'id\'])\n            if cid in seen_ids:\n                raise ValueError(\'duplicate case ID\')\n            seen_ids.add(cid)\n            _text(case[\'template\'], \'template\', 80)\n            _text(case[\'prompt\'], \'prompt\')\n            _text(case[\'rights_record\'], \'rights record\')\n            if case.get(\'aspect\') not in {\'1:1\', \'4:5\', \'3:2\'}:\n                raise ValueError(\'declared supported aspect required\')\n            tags = case.get(\'tags\')\n            if not isinstance(tags, list) or len(set(tags)) != len(tags) or not set(tags) <= TAGS:\n                raise ValueError(\'unique supported category declarations required\')\n            if case.get(\'origin\') not in {\'photograph_claimed\', \'illustration\', \'unknown\'}:\n                raise ValueError(\'explicit claimed image origin required\')\n            source = Path(case[\'source\'])\n            if source.is_symlink():\n                raise ValueError(\'symlink source refused\')\n            intake = inspect_image(source, IntakePolicy(allow_colour_conversion=True))\n            digest = pixel_identity(intake.image)\n            # One source may be used for different templates, but cannot become\n            # two source identities or two repetitions of the same case cell.\n            cell = (digest, case[\'template\'], case[\'aspect\'], case[\'prompt\'])\n            if cell in unique_pixels:\n                intake.close()\n                raise ValueError(\'duplicate normalized source/template/prompt cell\')\n            unique_pixels[cell] = cid\n            # Keep metadata, not every decoded image, for a 500-case plan.\n            # Reopen and reverify each input when saving its frozen snapshot.\n            decoded.append((case, dict(intake.record), digest, list(intake.image.size)))\n            intake.close()\n        out.mkdir(parents=True)\n        (out / \'inputs\').mkdir()\n        records = []\n        for case, record, digest, size in decoded:\n            name = \'inputs/\' + case[\'id\'] + \'.png\'\n            intake = inspect_image(Path(case[\'source\']), IntakePolicy(allow_colour_conversion=True))\n            try:\n                if intake.record[\'source_sha256\'] != record[\'source_sha256\'] or pixel_identity(intake.image) != digest:\n                    raise ValueError(\'source changed during plan preparation\')\n                intake.image.save(out / name)\n            finally:\n                intake.close()\n            records.append(dict(id=case[\'id\'], source=name,\n                                original_file_sha256=record[\'source_sha256\'],\n                                normalized_pixel_sha256=digest,\n                                source_size=size,\n                                intake=record,\n                                template=case[\'template\'], prompt=case[\'prompt\'],\n                                aspect=case[\'aspect\'], tags=case[\'tags\'],\n                                origin=case[\'origin\'], rights_record=case[\'rights_record\']))\n        plan = dict(schema=SCHEMA, state=\'PLAN_ONLY_NOT_EXECUTION_PERMISSION\',\n                    methods=methods, cases=records,\n                    tasks=[dict(case=c[\'id\'], method=m[\'id\'], first_attempt=1)\n                           for c in records for m in methods],\n                    quality_status=\'NOT_EVALUATED\',\n                    selection_complete=all(m[\'selection_complete\'] for m in methods),\n                    caution=\'Tags, origins and rights are declarations, not independently verified facts.\')\n        (out / \'plan.json\').write_bytes(_canonical(plan))\n        hashes = {p.relative_to(out).as_posix(): _sha(p.read_bytes())\n                  for p in out.rglob(\'*\') if p.is_file()}\n        (out / \'COMPLETE.json\').write_bytes(_canonical(dict(schema=SCHEMA, files=hashes)))\n        return plan\n    finally:\n        decoded.clear()\n\n\ndef verify_plan(root):\n    """Verify exact saved snapshots and no unexpected files before accounting."""\n    root = Path(root)\n    complete = read_json(root / \'COMPLETE.json\')\n    hashes = complete.get(\'files\')\n    if complete.get(\'schema\') != SCHEMA or not isinstance(hashes, dict) or not hashes:\n        raise ValueError(\'invalid completion record\')\n    present = {p.relative_to(root).as_posix() for p in root.rglob(\'*\') if p.is_file()}\n    if present != set(hashes) | {\'COMPLETE.json\'}:\n        raise ValueError(\'unexpected or missing experiment files\')\n    for name, expected in hashes.items():\n        if _sha(_relative(root, name).read_bytes()) != expected:\n            raise ValueError(\'frozen experiment bytes changed: \' + name)\n    plan = read_json(root / \'plan.json\')\n    if plan.get(\'schema\') != SCHEMA:\n        raise ValueError(\'unknown plan schema\')\n    # Rebuild original task cells instead of trusting a hand-edited task list.\n    tasks = [dict(case=c[\'id\'], method=m[\'id\'], first_attempt=1)\n             for c in plan[\'cases\'] for m in plan[\'methods\']]\n    if tasks != plan.get(\'tasks\'):\n        raise ValueError(\'plan task matrix is inconsistent\')\n    for case in plan[\'cases\']:\n        intake = inspect_image(_relative(root, case[\'source\']))\n        try:\n            if pixel_identity(intake.image) != case[\'normalized_pixel_sha256\']:\n                raise ValueError(\'normalized input identity mismatch\')\n        finally:\n            intake.close()\n    return plan\n\n\ndef _rating(record):\n    if record is None:\n        return None\n    if not isinstance(record, dict) or set(record.get(\'dimensions\', {})) != set(DIMENSIONS):\n        raise ValueError(\'all seven quality dimensions required\')\n    if any(type(v) is not int or not 1 <= v <= 5 for v in record[\'dimensions\'].values()):\n        raise ValueError(\'quality ratings must be integer one to five\')\n    errors = record.get(\'critical_errors\')\n    if not isinstance(errors, list) or any(not isinstance(e, str) or not e.strip() for e in errors):\n        raise ValueError(\'explicit critical-error list required\')\n    if type(record.get(\'no_critical\')) is not bool or record[\'no_critical\'] != (not errors):\n        raise ValueError(\'critical-error declarations conflict\')\n    if record.get(\'rater\') not in {\'owner\', \'independent_human\', \'agent_eye\'}:\n        raise ValueError(\'explicit rater claim required\')\n    return record\n\n\ndef audit_results(plan_root, evidence_root, events, *, usd_per_credit=None):\n    """Read-only matched accounting. Never sends, generates or infers ratings.\n\n    Rater labels in imported files remain claims. Even complete reported ratings\n    do not authenticate human identity, blinding, rights or competitive quality.\n    This function consequently NEVER sets acceptance_pass=True.\n    """\n    plan = verify_plan(plan_root)\n    if not isinstance(events, list) or len(events) > 50000:\n        raise ValueError(\'bounded event list required\')\n    if usd_per_credit is not None:\n        _number(usd_per_credit, \'explicit current USD per credit\')\n    allowed = {(t[\'case\'], t[\'method\']) for t in plan[\'tasks\']}\n    by_cell = {cell: {} for cell in allowed}\n    output_hashes = {}\n    repeated = []\n    for event in events:\n        cell = (event[\'case\'], event[\'method\'])\n        if cell not in by_cell:\n            raise ValueError(\'event is not a predeclared matched task\')\n        attempt = event.get(\'attempt\')\n        if type(attempt) is not int or not 1 <= attempt <= 100:\n            raise ValueError(\'attempt must be an integer one to100\')\n        if attempt in by_cell[cell]:\n            raise ValueError(\'duplicate attempt record\')\n        if event.get(\'state\') not in STATES:\n            raise ValueError(\'explicit output, failed or unconfirmed state required\')\n        _number(event.get(\'seconds\'), \'measured attempt seconds\')\n        charge = event.get(\'credits_charged\')\n        if charge is not None:\n            _number(charge, \'actual charged credits\')\n        if event.get(\'credits_estimated\') is not None:\n            _number(event[\'credits_estimated\'], \'estimated credits\')\n        if event[\'state\'] == \'output\':\n            p = _relative(evidence_root, event.get(\'output\'))\n            if _sha(p.read_bytes()) != event.get(\'output_sha256\'):\n                raise ValueError(\'output bytes changed\')\n            intake = inspect_image(p, IntakePolicy(allow_colour_conversion=True))\n            try:\n                identity = pixel_identity(intake.image)\n            finally:\n                intake.close()\n            if identity in output_hashes:\n                repeated.append([output_hashes[identity], [*cell, attempt]])\n            output_hashes[identity] = [*cell, attempt]\n        else:\n            if event.get(\'output\') or event.get(\'rating\') is not None:\n                raise ValueError(\'failed/unconfirmed attempt cannot carry output or quality rating\')\n            _text(event.get(\'failure\'), \'explicit failed/unconfirmed reason\')\n        _rating(event.get(\'rating\'))\n        by_cell[cell][attempt] = event\n    for attempts in by_cell.values():\n        if attempts and sorted(attempts) != list(range(1, max(attempts) + 1)):\n            raise ValueError(\'missing earlier attempt: first tries may not disappear\')\n    report = dict(schema=SCHEMA, quality_status=\'NOT_EVALUATED\', acceptance_pass=None,\n                  planned_tasks=len(allowed), methods={},\n                  unique_normalized_sources=len({c[\'normalized_pixel_sha256\'] for c in plan[\'cases\']}),\n                  templates=sorted({c[\'template\'] for c in plan[\'cases\']}),\n                  declared_tags=sorted({tag for c in plan[\'cases\'] for tag in c[\'tags\']}),\n                  missing_declared_tags=sorted(TAGS - {tag for c in plan[\'cases\'] for tag in c[\'tags\']}),\n                  repeated_output_pixels=repeated,\n                  warnings=[\'Declared categories and photo origins require curator verification.\',\n                            \'Exact normalized hashes do not detect all crops or near duplicates.\',\n                            \'Imported rater identity/blinding is not authenticated by a JSON label.\',\n                            \'First-try failures, unconfirmed calls and missing attempts stay in planned denominators.\',\n                            \'Successful file generation is not publication usefulness.\',\n                            \'No model recommendation or specialist parity is inferred.\'])\n    for method in plan[\'methods\']:\n        cells = [by_cell[(c[\'id\'], method[\'id\'])] for c in plan[\'cases\']]\n        attempts = [v for cell in cells for v in cell.values()]\n        first = [cell[1] for cell in cells if 1 in cell]\n        outputs = [e for e in first if e[\'state\'] == \'output\']\n        ratings = [e[\'rating\'] for e in outputs if e.get(\'rating\') is not None]\n        human_claims = [r for r in ratings if r[\'rater\'] in {\'owner\', \'independent_human\'}]\n        complete = len(human_claims) == len(outputs) and bool(first) and len(first) == len(cells)\n        no_critical = sum(r[\'no_critical\'] for r in human_claims)\n        useful = sum(r[\'no_critical\'] and r[\'dimensions\'][\'publish_ready_usefulness\'] >= 4\n                     for r in human_claims)\n        known_charges = [e[\'credits_charged\'] for e in attempts if e.get(\'credits_charged\') is not None]\n        charged = sum(known_charges) if attempts and len(known_charges) == len(attempts) else None\n        # The cost includes failed tries and retries, but an accepted-output\n        # denominator requires complete claimed human ratings. It remains a\n        # declared-rating calculation, never authenticated production economics.\n        cost_per_useful = charged / useful if charged is not None and complete and useful else None\n        n = len(cells)\n        report[\'methods\'][method[\'id\']] = dict(\n            planned_first_tries=n, recorded_first_tries=len(first),\n            not_started=n - len(first), first_outputs=len(outputs),\n            first_failures=sum(e[\'state\'] == \'failed\' for e in first),\n            first_unconfirmed=sum(e[\'state\'] == \'unconfirmed\' for e in first),\n            retries_recorded=sum(max(0, len(cell) - 1) for cell in cells),\n            first_generation_rate=len(outputs) / n,\n            first_attempt_median_seconds=statistics.median(e[\'seconds\'] for e in first) if first else None,\n            all_attempts=len(attempts), unknown_charge_attempts=len(attempts) - len(known_charges),\n            actual_credits_total=charged, known_partial_credits=sum(known_charges),\n            usd_cost_total=charged * usd_per_credit if charged is not None and usd_per_credit is not None else None,\n            claimed_human_rated_first_outputs=len(human_claims),\n            claimed_no_critical=no_critical if complete else None,\n            claimed_publishable=useful if complete else None,\n            claimed_no_critical_rate=no_critical / n if complete else None,\n            claimed_publishable_rate=useful / n if complete else None,\n            claimed_targets_met=(no_critical / n >= .8 and useful / n >= .7) if complete else None,\n            actual_credits_per_claimed_publishable=cost_per_useful,\n            rating_status=\'CLAIMED_HUMAN_LABELS_NOT_AUTHENTICATED\' if human_claims else \'NOT_HUMAN_RATED\',\n            acceptance_pass=None)\n    return report\n\n\ndef main():\n    p = argparse.ArgumentParser(description=__doc__)\n    sub = p.add_subparsers(dest=\'command\', required=True)\n    freeze = sub.add_parser(\'freeze\', help=\'normalize snapshots; no generation or spend\')\n    freeze.add_argument(\'--spec\', type=Path, required=True)\n    freeze.add_argument(\'--out\', type=Path, required=True)\n    audit = sub.add_parser(\'audit\', help=\'read-only audit of existing attempt records\')\n    audit.add_argument(\'--plan\', type=Path, required=True)\n    audit.add_argument(\'--evidence-root\', type=Path, required=True)\n    audit.add_argument(\'--events\', type=Path, required=True)\n    audit.add_argument(\'--out\', type=Path, required=True)\n    a = p.parse_args()\n    if a.command == \'freeze\':\n        result = freeze_plan(read_json(a.spec), a.out)\n    else:\n        result = audit_results(a.plan, a.evidence_root, read_json(a.events))\n        with a.out.open(\'x\', encoding=\'utf-8\') as f:\n            json.dump(result, f, indent=2, allow_nan=False)\n    print(json.dumps(dict(schema=SCHEMA, command=a.command,\n                          quality_status=result[\'quality_status\']), allow_nan=False))\n\n\nif __name__ == \'__main__\':\n    main()\n'
# ---- embedded module: renderello/matched_trial.py ----
path = base / 'renderello/matched_trial.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Native trimap and matte-change review, never an accuracy score.\n\nOpenCV topology is used to expose dropped parts, filled holes and new bridges.\nA baseline can be wrong. Preserving it does not establish semantic correctness.\nNo small component is discarded, no hole is filled, and no candidate is selected.\n"""\nfrom __future__ import annotations\nfrom dataclasses import asdict, dataclass\nimport math\nimport cv2\nimport numpy as np\nfrom .candidate_diagnostics import native_alpha, digest\n\n\n@dataclass(frozen=True)\nclass ChangeBudget:\n    threshold: int = 128\n    minimum_part_retained: float = .9\n    maximum_added_fraction: float = .1\n    maximum_removed_fraction: float = .1\n    maximum_hole_fill: float = .1\n    maximum_alpha_mean_delta: float = .05\n    detail_limit: int = 256\n\n    def __post_init__(self):\n        if type(self.threshold) is not int or not 1 <= self.threshold <= 255:\n            raise ValueError(\'threshold must be integer 1..255\')\n        for key in (\'minimum_part_retained\', \'maximum_added_fraction\',\n                    \'maximum_removed_fraction\', \'maximum_hole_fill\',\n                    \'maximum_alpha_mean_delta\'):\n            value = getattr(self, key)\n            if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value) or not 0 <= value <= 1:\n                raise ValueError(key+\' must be finite fraction\')\n        if type(self.detail_limit) is not int or not 1 <= self.detail_limit <= 256:\n            raise ValueError(\'detail limit must be integer 1..256\')\n\n\ndef _components(binary):\n    return cv2.connectedComponentsWithStats(binary.astype(np.uint8), connectivity=8)\n\n\ndef _require_pair(before, after):\n    if not isinstance(before, np.ndarray) or before.ndim != 2:\n        raise ValueError(\'native two dimensional alpha required\')\n    native_alpha(before, before.shape)\n    native_alpha(after, before.shape)\n\n\ndef audit_trimap(alpha, trimap, *, unknown_limit=200_000):\n    """Audit explicit seeds against an estimated alpha, without trusting either.\n\n    Contradictions are review blockers, not proof the reviewer or model is wrong.\n    Values 0/255 fix solver seeds;128 means unknown. No implicit threshold cast.\n    """\n    _require_pair(alpha, trimap)\n    if type(unknown_limit) is not int or not 1 <= unknown_limit <= 4_000_000:\n        raise ValueError(\'positive bounded integer unknown limit required\')\n    if not np.isin(trimap, [0, 128, 255]).all():\n        raise ValueError(\'trimap needs only 0,128,255\')\n    fg, bg, unknown = trimap == 255, trimap == 0, trimap == 128\n    blockers = []\n    if not fg.any(): blockers.append(\'NO_FOREGROUND_SEED\')\n    if not bg.any(): blockers.append(\'NO_BACKGROUND_SEED\')\n    if int(unknown.sum()) > unknown_limit: blockers.append(\'UNKNOWN_BAND_EXCEEDS_BUDGET\')\n    contradictions = (fg & (alpha < 128)) | (bg & (alpha >= 128))\n    if contradictions.any(): blockers.append(\'SEED_DISAGREES_WITH_INITIAL_ALPHA_REVIEW_BOTH\')\n    # Thin disconnected parts can lose all definite foreground during erosion.\n    count, labels, stats, _ = _components(alpha >= 128)\n    seeded = np.bincount(labels[fg].ravel(), minlength=count)\n    unseeded = [i for i in range(1, count) if seeded[i] == 0]\n    if unseeded: blockers.append(\'INITIAL_PART_WITHOUT_FOREGROUND_SEED\')\n    report = dict(alpha_sha256=digest(alpha), trimap_sha256=digest(trimap),\n        foreground_seed_pixels=int(fg.sum()), background_seed_pixels=int(bg.sum()),\n        unknown_pixels=int(unknown.sum()), contradictory_seed_pixels=int(contradictions.sum()),\n        initial_parts=count-1, unseeded_part_count=len(unseeded),\n        unseeded_part_labels=unseeded[:256], detail_truncated=len(unseeded)>256,\n        blockers=blockers, seed_semantics=\'NOT_VERIFIED\', automatic_acceptance=False)\n    return report, contradictions\n\n\ndef review_change(before, after, budget=None):\n    """Compare alpha revisions at identical native coordinates.\n\n    Fractions of added/removed support use initial hard foreground area, not the\n    whole canvas. Per-part retention prevents a large object hiding a lost small\n    accessory. Hole fill uses each original enclosed background component. New\n    connected components spanning multiple original parts are flagged as bridges.\n    This does not certify that separated parts should remain separate in reality.\n    """\n    _require_pair(before, after)\n    budget = ChangeBudget() if budget is None else budget\n    if not isinstance(budget, ChangeBudget):\n        raise ValueError(\'typed change budget required\')\n    b, a = before >= budget.threshold, after >= budget.threshold\n    nb, lb, sb, _ = _components(b)\n    na, la, sa, _ = _components(a)\n    blockers = []\n    area = int(b.sum())\n    if not area: blockers.append(\'EMPTY_INITIAL_HARD_MASK\')\n    if not a.any(): blockers.append(\'EMPTY_REVISED_HARD_MASK\')\n    added, removed = a & ~b, b & ~a\n    denominator = max(1, area)\n    added_fraction, removed_fraction = float(added.sum())/denominator, float(removed.sum())/denominator\n    if added_fraction > budget.maximum_added_fraction: blockers.append(\'ADDED_SUPPORT_EXCEEDS_BUDGET\')\n    if removed_fraction > budget.maximum_removed_fraction: blockers.append(\'REMOVED_SUPPORT_EXCEEDS_BUDGET\')\n    # Bincount is linear in pixels, rather than scanning the canvas per component.\n    retained = np.bincount(lb[a].ravel(), minlength=nb)\n    parts = []\n    damaged = []\n    for i in range(1, nb):\n        part_area = int(sb[i, cv2.CC_STAT_AREA])\n        fraction = int(retained[i])/part_area\n        if fraction < budget.minimum_part_retained: damaged.append(i)\n        if len(parts) < budget.detail_limit:\n            x, y, w, h, _ = map(int, sb[i])\n            parts.append(dict(label=i, box=[x,y,x+w,y+h], area=part_area,\n                retained_pixels=int(retained[i]), retained_fraction=fraction))\n    if damaged: blockers.append(\'INITIAL_PART_DAMAGED_OR_REMOVED\')\n    # Count unique (revised,initial) overlap pairs using a compact integer key.\n    pairs = np.unique(la[b & a].astype(np.int64)*nb+lb[b & a])\n    overlap_counts = np.bincount(pairs//nb, minlength=na)\n    bridges = np.flatnonzero(overlap_counts > 1).tolist()\n    isolated = [i for i in range(1,na) if overlap_counts[i] == 0]\n    if bridges: blockers.append(\'REVISED_COMPONENT_BRIDGES_INITIAL_PARTS\')\n    if isolated: blockers.append(\'NEW_DISCONNECTED_FOREGROUND\')\n    nh, lh, sh, _ = _components(~b)\n    filled = np.bincount(lh[a].ravel(), minlength=nh)\n    holes, changed_holes = [], []\n    h, w = b.shape\n    hole_count = 0\n    for i in range(1, nh):\n        x, y, cw, ch, hole_area = map(int, sh[i])\n        if x == 0 or y == 0 or x+cw == w or y+ch == h: continue\n        hole_count += 1\n        fraction = int(filled[i])/hole_area\n        if fraction > budget.maximum_hole_fill: changed_holes.append(i)\n        if len(holes) < budget.detail_limit:\n            holes.append(dict(label=i, box=[x,y,x+cw,y+ch], area=hole_area,\n                filled_pixels=int(filled[i]), filled_fraction=fraction))\n    if changed_holes: blockers.append(\'INITIAL_HOLE_FILLED\')\n    delta = after.astype(np.int16)-before.astype(np.int16)\n    mean_delta = float(np.abs(delta).mean()/255)\n    if mean_delta > budget.maximum_alpha_mean_delta: blockers.append(\'ALPHA_CHANGE_EXCEEDS_BUDGET\')\n    truncated = nb-1 > budget.detail_limit or na-1 > budget.detail_limit or hole_count > budget.detail_limit\n    if truncated: blockers.append(\'TOPOLOGY_DETAIL_LIMIT_REQUIRES_REVIEW\')\n    report = dict(before_sha256=digest(before), after_sha256=digest(after),\n        budget=asdict(budget), initial_hard_pixels=area, revised_hard_pixels=int(a.sum()),\n        initial_parts=nb-1, revised_parts=na-1, original_holes=hole_count,\n        added_hard_pixels=int(added.sum()), removed_hard_pixels=int(removed.sum()),\n        added_fraction=added_fraction, removed_fraction=removed_fraction,\n        mean_normalized_alpha_delta=mean_delta, maximum_alpha_delta=int(np.abs(delta).max()),\n        changed_alpha_pixels=int(np.count_nonzero(delta)), parts=parts, holes=holes,\n        damaged_part_count=len(damaged), damaged_part_labels=damaged[:budget.detail_limit],\n        bridge_count=len(bridges), bridge_labels=bridges[:budget.detail_limit],\n        new_component_count=len(isolated), new_component_labels=isolated[:budget.detail_limit],\n        filled_hole_count=len(changed_holes), filled_hole_labels=changed_holes[:budget.detail_limit],\n        detail_truncated=truncated, blockers=blockers,\n        within_configured_change_budget=not blockers, baseline_accuracy=\'NOT_VERIFIED\',\n        automatic_acceptance=False)\n    # Red means alpha increased, blue means decreased. Magnitudes are byte deltas,\n    # not an error map. The original/revised alpha arrays are never mutated.\n    view = np.zeros((*before.shape,3),np.uint8)\n    view[:,:,0] = np.clip(delta,0,255).astype(np.uint8)\n    view[:,:,2] = np.clip(-delta,0,255).astype(np.uint8)\n    return report, view\n'
# ---- embedded module: renderello/matte_change_review.py ----
path = base / 'renderello/matte_change_review.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Opt-in trimap matting and foreground recovery, not automatic segmentation.\n\nCF/LKM and multi-level foreground estimation: official MIT PyMatting 1.1.16.\nGuided filter: He, Sun, Tang (ECCV2010), RGB local linear equations 14-16.\nNo network access. This stage cannot recover a missing semantic object or pose.\n"""\nfrom __future__ import annotations\nimport hashlib,json,time\nfrom pathlib import Path\nimport cv2\nimport numpy as np\nfrom PIL import Image\n\nSOURCES=[\'https://github.com/pymatting/pymatting/\',\n         \'https://pymatting.github.io/alpha.html\',\n         \'https://pymatting.github.io/foreground.html\',\n         \'https://arxiv.org/abs/2006.14970\',\n         \'https://people.csail.mit.edu/kaiming/publications/eccv10guidedfilter.pdf\']\n\n\ndef validate(rgb,alpha):\n    if rgb.dtype!=np.uint8 or alpha.dtype!=np.uint8 or rgb.ndim!=3 or rgb.shape[2]!=3 or alpha.shape!=rgb.shape[:2]:\n        raise ValueError(\'source RGB and source-sized alpha must be uint8\')\n    if alpha.size>1_000_000:raise ValueError(\'matting is capped at one million source pixels; no silent resize\')\n    if alpha.size<9:raise ValueError(\'source too small\')\n\n\ndef make_trimap(alpha,radius=3):\n    """Conservative heuristic seeds, NOT ground-truth confidence or semantic proof."""\n    if alpha.dtype!=np.uint8 or alpha.ndim!=2:raise ValueError(\'alpha must be uint8 grayscale\')\n    if not isinstance(radius,int) or not 1<=radius<=12:raise ValueError(\'radius must be integer1-12\')\n    k=np.ones((2*radius+1,2*radius+1),np.uint8)\n    fg=cv2.erode((alpha>240).astype(np.uint8),k,borderType=cv2.BORDER_CONSTANT,borderValue=0)>0\n    bg=cv2.erode((alpha<15).astype(np.uint8),k,borderType=cv2.BORDER_CONSTANT,borderValue=1)>0\n    tri=np.full(alpha.shape,128,np.uint8);tri[fg]=255;tri[bg]=0\n    check_trimap(tri,alpha.shape)\n    return tri\n\n\ndef check_trimap(tri,shape):\n    if tri.dtype!=np.uint8 or tri.shape!=shape or not set(np.unique(tri)).issubset({0,128,255}):\n        raise ValueError(\'trimap must be source-sized uint8 with0,128,255\')\n    if not (np.any(tri==0) and np.any(tri==255)):\n        raise ValueError(\'trimap needs definite background and foreground\')\n\n\ndef guided_alpha(rgb,alpha,radius=4,eps=.001):\n    """RGB guided filter; edge-aware feathering only, no foreground-color recovery."""\n    I=rgb.astype(np.float64)/255;p=alpha.astype(np.float64)/255\n    box=lambda a:cv2.boxFilter(a,-1,(2*radius+1,2*radius+1),normalize=True,borderType=cv2.BORDER_REFLECT)\n    meanI=box(I);meanP=box(p);covIp=box(I*p[...,None])-meanI*meanP[...,None]\n    cov=np.empty((*p.shape,3,3),np.float64)\n    for i in range(3):\n        for j in range(3):cov[...,i,j]=box(I[...,i]*I[...,j])-meanI[...,i]*meanI[...,j]+(eps if i==j else 0)\n    coeff=np.linalg.solve(cov,covIp[...,None])[...,0];b=meanP-(coeff*meanI).sum(2)\n    return np.clip((box(coeff)*I).sum(2)+box(b),0,1)\n\n\ndef refine_arrays(rgb,alpha,trimap,method=\'cf\',edge_colors=False):\n    validate(rgb,alpha);check_trimap(trimap,alpha.shape)\n    if method not in (\'cf\',\'lkm\',\'guided\'):raise ValueError(\'unsupported matting method\')\n    if int((trimap==128).sum())>200_000:raise ValueError(\'unknown trimap band exceeds200000pixels; supply a tighter reviewed trimap\')\n    image=rgb.astype(np.float64)/255;tri=trimap.astype(np.float64)/255\n    tri[trimap==128]=.5\n    if method==\'guided\':a=guided_alpha(rgb,alpha)\n    else:\n        try:\n            from pymatting import estimate_alpha_cf,estimate_alpha_lkm\n        except ImportError as e:raise RuntimeError(\'optional PyMatting1.1.16 is not installed; install official matting requirements in an isolated project environment\') from e\n        a=(estimate_alpha_cf if method==\'cf\' else estimate_alpha_lkm)(image,tri)\n    if not np.isfinite(a).all() or a.shape!=alpha.shape:raise ValueError(\'invalid estimated alpha\')\n    a=np.clip(a,0,1);a[trimap==0]=0;a[trimap==255]=1\n    result=rgb.copy()\n    if edge_colors:\n        try:from pymatting import estimate_foreground_ml\n        except ImportError as e:raise RuntimeError(\'foreground recovery needs optional official PyMatting1.1.16\') from e\n        f=estimate_foreground_ml(image,a)\n        if f.shape!=image.shape or not np.isfinite(f).all():raise ValueError(\'invalid estimated foreground\')\n        result=np.round(np.clip(f,0,1)*255).astype(np.uint8)\n        # Never estimate the protected interior. Restore byte-for-byte.\n        result[trimap==255]=rgb[trimap==255]\n        result[trimap==0]=rgb[trimap==0]\n    matte=np.round(a*255).astype(np.uint8)\n    changed=np.any(result!=rgb,axis=2)\n    return result,matte,dict(protected_core_pixels=int((trimap==255).sum()),\n        protected_core_rgb_max_error=int(np.abs(result[trimap==255].astype(\'int16\')-rgb[trimap==255].astype(\'int16\')).max()),\n        changed_rgb_pixels=int(changed.sum()),alpha_changed_pixels=int((matte!=alpha).sum()),\n        source_rgb_unchanged=not edge_colors,entire_product_source_exact=False,edge_rgb_estimated=bool(edge_colors),\n        semantic_completeness=\'NOT_VERIFIED\',quality=\'NOT_EVALUATED\')\n\n\ndef refine_file(source:Path,alpha:Path,out:Path,*,trimap_path=None,method=\'cf\',radius=3,edge_colors=False):\n    if out.exists():raise ValueError(\'output directory exists; preserve original attempts\')\n    with Image.open(source) as im:rgb=np.array(im.convert(\'RGB\'))\n    with Image.open(alpha) as im:\n        if im.mode!=\'L\':raise ValueError(\'alpha must be8-bit grayscale\')\n        a=np.array(im)\n    validate(rgb,a)\n    if trimap_path:\n        with Image.open(trimap_path) as im:\n            if im.mode!=\'L\':raise ValueError(\'trimap must be8-bit grayscale\')\n            tri=np.array(im)\n    else:tri=make_trimap(a,radius)\n    check_trimap(tri,a.shape);t=time.monotonic();foreground,matte,meta=refine_arrays(rgb,a,tri,method,edge_colors)\n    out.mkdir(parents=True)\n    for name,data in [(\'source.png\',rgb),(\'initial-alpha.png\',a),(\'trimap.png\',tri),(\'foreground.png\',foreground),(\'refined-alpha.png\',matte),(\'refined-cutout.png\',np.dstack([foreground,matte])),(\'changed-rgb.png\',(np.any(foreground!=rgb,2)*255).astype(\'uint8\'))]:\n        Image.fromarray(data).save(out/name)\n    for name,color in [(\'black\',(20,20,20)),(\'white\',(250,250,250)),(\'magenta\',(190,0,150))]:\n        bg=Image.new(\'RGB\',(rgb.shape[1],rgb.shape[0]),color);bg.paste(Image.fromarray(foreground),mask=Image.fromarray(matte));bg.save(out/(name+\'-review.png\'))\n    meta.update(method=method,backend=\'pymatting1.1.16\' if method!=\'guided\' or edge_colors else \'local_RGB_guided_filter\',\n        trimap_origin=\'supplied_reviewer_input\' if trimap_path else \'heuristic_from_estimated_alpha_not_ground_truth\',\n        radius=radius,seconds=time.monotonic()-t,source=str(source.resolve()),source_sha256=hashlib.sha256(source.read_bytes()).hexdigest(),\n        warnings=[\'Opt-in research stage, not automatic quality acceptance.\',\'Cannot repair missing objects, change camera pose, recover true glass optics or lighting.\',\n        \'Heuristic trimap can lock wrong foreground/background; inspect every part and hole.\',\n        \'Foreground recovery changes boundary RGB; only the saved definite-foreground core is restored exactly.\'] if edge_colors else [\'Alpha changed; source RGB unchanged. Camera/material/missing-object accuracy not verified.\'],sources=SOURCES)\n    meta[\'output_sha256\']={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in out.iterdir() if p.is_file()}\n    (out/\'matting-manifest.json\').write_text(json.dumps(meta,indent=2));return meta\n\n\nMAX_WORKER_RSS_BYTES=1500*1024*1024\nMAX_WORKER_SECONDS=90\n\n\ndef _stop_worker(child):\n    import subprocess\n    if child.poll() is None:\n        child.terminate()\n        try:child.wait(timeout=5)\n        except subprocess.TimeoutExpired:child.kill();child.wait(timeout=5)\n\n\ndef guarded_file(source,alpha,out,*,trimap_path=None,method=\'cf\',radius=3,edge_colors=False):\n    """Sampled90s/1500MiB single-worker guard. Not an OS allocation ceiling."""\n    import subprocess,sys,os,tempfile\n    from .process_memory import ResidentMemory\n    # Fail before starting work if this host cannot measure its own resident memory.\n    preflight=ResidentMemory(os.getpid())\n    try:preflight.sample()\n    finally:preflight.close()\n    cmd=[sys.executable,\'-m\',\'renderello.matting\',\'--source\',str(source),\'--alpha\',str(alpha),\'--out\',str(out),\'--method\',method,\'--radius\',str(radius)]\n    if trimap_path:cmd+=[\'--trimap\',str(trimap_path)]\n    if edge_colors:cmd+=[\'--estimate-edge-colors\']\n    # Files avoid a full stdout/stderr pipe blocking the child before it can exit.\n    # Imported callers can run outside the repository. Child module discovery\n    # must not depend on the caller\'s current working directory.\n    env = os.environ.copy()\n    package_root = str(Path(__file__).resolve().parent.parent)\n    env[\'PYTHONPATH\'] = package_root + os.pathsep + env.get(\'PYTHONPATH\', \'\')\n    with tempfile.TemporaryFile(mode=\'w+t\') as stdout_file, tempfile.TemporaryFile(mode=\'w+t\') as stderr_file:\n        child=subprocess.Popen(cmd,stdout=stdout_file,stderr=stderr_file,text=True,env=env)\n        monitor=None\n        exit_reap_grace_used=False\n        start=time.monotonic()\n        try:\n            if child.poll() is None:\n                try:monitor=ResidentMemory(child.pid)\n                except Exception as exc:\n                    if child.poll() is None:raise RuntimeError(\'memory monitoring unavailable; attempt not accepted\') from exc\n            while child.poll() is None:\n                if time.monotonic()-start>MAX_WORKER_SECONDS:raise RuntimeError(\'90second matting timeout; attempt not accepted\')\n                try:rss=monitor.sample()\n                except Exception as exc:\n                    if child.poll() is not None:break  # exited between poll and sample\n                    # Linux may remove VmRSS while the process is tearing down,\n                    # just before waitpid sees its exit. Reap once, bounded200ms.\n                    # A still-running unmonitorable worker remains refused; a\n                    # nonzero reaped exit is rejected below, never legitimized.\n                    try:child.wait(timeout=.2)\n                    except subprocess.TimeoutExpired:\n                        raise RuntimeError(\'memory monitoring unavailable; attempt not accepted\') from exc\n                    exit_reap_grace_used=True\n                    break\n                if rss>MAX_WORKER_RSS_BYTES:raise RuntimeError(\'1500MiB RSS safety limit; attempt not accepted\')\n                time.sleep(.08)\n            if time.monotonic()-start>MAX_WORKER_SECONDS:raise RuntimeError(\'90second matting timeout; attempt not accepted\')\n            stdout_file.seek(0);stderr_file.seek(0)\n            stdout,stderr=stdout_file.read(),stderr_file.read()\n            if child.returncode:raise RuntimeError(\'matting worker failed: \'+stderr[-2000:])\n            result=json.loads(stdout)\n            if exit_reap_grace_used:result[\'worker_guard_exit_reap_wait_seconds\']=.2\n            return result\n        finally:\n            try:_stop_worker(child)\n            finally:\n                if monitor is not None:monitor.close()\n\n\ndef main():\n    import argparse,os\n    os.environ.setdefault(\'NUMBA_NUM_THREADS\',\'1\')\n    p=argparse.ArgumentParser(description=__doc__)\n    for name in (\'source\',\'alpha\',\'out\'):p.add_argument(\'--\'+name,type=Path,required=True)\n    p.add_argument(\'--trimap\',type=Path);p.add_argument(\'--method\',choices=[\'cf\',\'lkm\',\'guided\'],default=\'cf\');p.add_argument(\'--radius\',type=int,default=3);p.add_argument(\'--estimate-edge-colors\',action=\'store_true\');a=p.parse_args()\n    print(json.dumps(refine_file(a.source,a.alpha,a.out,trimap_path=a.trimap,method=a.method,radius=a.radius,edge_colors=a.estimate_edge_colors)))\n\nif __name__==\'__main__\':main()\n\n\ndef compose_bundle(bundle:Path,background:Path,out:Path,*,source_pose,material,framing,fill=.68):\n    """Explicit top-down opaque route with original-source and estimated-edge provenance."""\n    from .compose import fit_product,flatlay_center,finish_scene\n    from .cutout import _bbox_from_matte\n    if out.exists():raise ValueError(\'output directory exists; preserve attempts\')\n    if source_pose!=\'overhead\':raise ValueError(\'this route requires a reviewed overhead source, not inferred pose\')\n    if material!=\'opaque\':raise ValueError(\'this route cannot paste transparent material\')\n    if framing!=\'complete\':raise ValueError(\'this route requires a reviewed complete product; cropped parts cannot be recovered\')\n    if not .2<=fill<=.85:raise ValueError(\'fill must be0.2-0.85\')\n    meta=json.loads((bundle/\'matting-manifest.json\').read_text())\n    for name in (\'source.png\',\'foreground.png\',\'refined-alpha.png\',\'refined-cutout.png\',\'trimap.png\'):\n        digest=hashlib.sha256((bundle/name).read_bytes()).hexdigest()\n        if digest!=meta[\'output_sha256\'].get(name):raise ValueError(\'matting bundle hash mismatch: \'+name)\n    p=np.array(Image.open(bundle/\'refined-cutout.png\').convert(\'RGBA\'))\n    with Image.open(background) as im:scene=np.array(im.convert(\'RGB\'))\n    H,W=scene.shape[:2]\n    if max(H,W)>2048 or min(H,W)<64:raise ValueError(\'background must be64-2048pixels per side\')\n    crop,xy=fit_product(p,_bbox_from_matte(p[...,3]),(W,H),fill);xy=flatlay_center(crop[...,3],(W,H));result,_=finish_scene(scene,crop,xy,shadow_style=\'surface contact\')\n    out.mkdir(parents=True);Image.fromarray(result).save(out/\'composite.png\')\n    report=dict(original_source_sha256=meta[\'source_sha256\'],matting_manifest_sha256=hashlib.sha256((bundle/\'matting-manifest.json\').read_bytes()).hexdigest(),\n        background_sha256=hashlib.sha256(background.read_bytes()).hexdigest(),background_camera=\'reviewer_declared_overhead\',\n        source_pose=source_pose,material=material,framing=framing,placement=list(xy),\n        source_rgb_unchanged=meta.get(\'source_rgb_unchanged\',not meta[\'edge_rgb_estimated\']),edge_rgb_estimated=meta[\'edge_rgb_estimated\'],\n        protected_core_rgb_max_error_before_placement=meta[\'protected_core_rgb_max_error\'],\n        warning=\'Source RGB may be resampled once for placement. Edge RGB estimation is not exact-source product fidelity.2D shadow is not recovered physical contact.\',\n        quality=\'NOT_EVALUATED\',composite_sha256=hashlib.sha256((out/\'composite.png\').read_bytes()).hexdigest())\n    (out/\'compose-manifest.json\').write_text(json.dumps(report,indent=2));return report\n\n\ndef refine_native_region(rgb, alpha, roi, *, source_sha256, alpha_sha256,\n                         reviewed=False, review_note=\'\', material=None,\n                         trimap=None, method=\'cf\', edge_colors=False, feather=8):\n    """Native-resolution local matting for a larger source, without global solves.\n\n    ROI and trimap are explicit reviewer hypotheses. Taper to the original prior\n    at the rectangle border instead of pasting a hard solver seam. This cannot\n    recover missing parts or prove semantic/optical correctness. No default route\n    is changed, and no scene/camera/lighting or acceptance is inferred.\n    """\n    from .candidate_diagnostics import digest\n    if type(edge_colors)is not bool:raise ValueError(\'explicit boolean edge-colour mode required\')\n    if not isinstance(rgb,np.ndarray) or rgb.dtype!=np.uint8 or rgb.ndim!=3 or rgb.shape[2]!=3:\n        raise ValueError(\'native uint8 RGB required\')\n    if not isinstance(alpha,np.ndarray) or alpha.dtype!=np.uint8 or alpha.shape!=rgb.shape[:2]:\n        raise ValueError(\'native uint8 alpha required\')\n    if alpha.size>8_000_000:raise ValueError(\'source exceeds bounded8MP copy budget\')\n    if digest(rgb)!=source_sha256 or digest(alpha)!=alpha_sha256:\n        raise ValueError(\'source/alpha binding mismatch\')\n    if reviewed is not True or material not in (\'opaque\',\'opaque_fibres\'):\n        raise ValueError(\'explicit reviewed opaque region required\')\n    if not isinstance(review_note,str) or not review_note.strip() or len(review_note)>2000:\n        raise ValueError(\'bounded region review note required\')\n    if not isinstance(roi,(tuple,list)) or len(roi)!=4 or any(type(v)is not int for v in roi):\n        raise ValueError(\'native integer ROI required\')\n    x0,y0,x1,y1=roi;h,w=alpha.shape\n    if not (0<=x0<x1<=w and 0<=y0<y1<=h):raise ValueError(\'ROI outside source\')\n    if type(feather)is not int or not 1<=feather<=32 or 2*feather>=min(x1-x0,y1-y0):\n        raise ValueError(\'bounded feather must fit native ROI\')\n    r=rgb[y0:y1,x0:x1];a=alpha[y0:y1,x0:x1]\n    validate(r,a)\n    tri=make_trimap(a,2) if trimap is None else trimap\n    check_trimap(tri,a.shape)\n    estimated_rgb,estimated_alpha,solver=refine_arrays(r,a,tri,method,edge_colors)\n    yy,xx=np.indices(a.shape)\n    distance=np.minimum.reduce([xx,yy,a.shape[1]-1-xx,a.shape[0]-1-yy])\n    weight=np.minimum(1.,distance/feather)*(tri==128)\n    blended_alpha=np.rint(a*(1-weight)+estimated_alpha*weight).astype(np.uint8)\n    blended_rgb=np.rint(r*(1-weight[...,None])+estimated_rgb*weight[...,None]).astype(np.uint8)\n    out_rgb=rgb.copy();out_alpha=alpha.copy()\n    out_rgb[y0:y1,x0:x1]=blended_rgb;out_alpha[y0:y1,x0:x1]=blended_alpha\n    # Invariants use only ROI-sized comparison arrays, not full-image float work.\n    if not np.array_equal(blended_rgb[tri!=128],r[tri!=128]) or not np.array_equal(blended_alpha[tri!=128],a[tri!=128]):\n        raise RuntimeError(\'definite trimap seeds changed\')\n    return out_rgb,out_alpha,dict(schema=\'native-region-matting.v1\',source_sha256=source_sha256,\n        prior_alpha_sha256=alpha_sha256,output_rgb_sha256=digest(out_rgb),output_alpha_sha256=digest(out_alpha),\n        roi_xyxy=list(roi),roi_pixels=int(a.size),source_pixels=int(alpha.size),\n        trimap_sha256=digest(tri),trimap_origin=\'supplied_reviewer_hypothesis\' if trimap is not None else \'derived_2px_not_ground_truth\',\n        review_note=review_note,material=material,method=method,feather=feather,\n        changed_rgb_pixels=int(np.any(blended_rgb!=r,2).sum()),changed_alpha_pixels=int((blended_alpha!=a).sum()),\n        estimated_boundary_rgb=bool(edge_colors),source_rgb_exact=not bool(np.any(blended_rgb!=r)),\n        outside_roi_rgb_alpha_exact=True,rectangle_border_rgb_alpha_exact=True,definite_trimap_seeds_exact=True,\n        resampled=False,solver=solver,accepted=False,default_pipeline_changed=False,\n        limitations=[\'Reviewer ROI/trimap are hypotheses, not semantic or optical truth.\',\n                    \'Feathering retains the prior at the rectangle border and can retain its errors.\',\n                    \'Boundary colour recovery is an estimate; product RGB is not globally source-exact when enabled.\',\n                    \'No full-source isolation, hidden-detail recovery, relighting, camera or physical-contact claim.\'])\n'
# ---- embedded module: renderello/matting.py ----
path = base / 'renderello/matting.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Opt-in bridge: verified estimated alpha with original normalized RGB only.\n\nThis does not accept estimated foreground colors or certify mask semantics.\nLegacy composition and defaults are unchanged.\n"""\nfrom pathlib import Path\nfrom io import BytesIO\nimport hashlib,json,tempfile,shutil\nimport numpy as np\nfrom PIL import Image\nfrom .image_intake import IntakeError,IntakePolicy,_read_bounded,inspect_image\nfrom .reviewed_source import prepare_reviewed_source\n\n\ndef prepare_alpha_only(source,bundle,destination):\n    source,bundle,dest=Path(source),Path(bundle),Path(destination)\n    if dest.exists() or dest.is_symlink():raise FileExistsError(\'Preserve previous prepared sources\')\n    if bundle.is_symlink() or not bundle.is_dir():raise IntakeError(\'Regular matting bundle directory required\')\n    def read(name):\n        p=bundle/name\n        if p.is_symlink() or not p.is_file():raise IntakeError(\'Regular bundle file required: \'+name)\n        return _read_bounded(p,IntakePolicy().max_bytes)\n    manifest_raw=read(\'matting-manifest.json\')\n    try:m=json.loads(manifest_raw)\n    except (ValueError,TypeError) as e:raise IntakeError(\'Invalid matting manifest\') from e\n    if not isinstance(m,dict) or not isinstance(m.get(\'output_sha256\'),dict):raise IntakeError(\'Matting output hashes required\')\n    raw=_read_bounded(source,IntakePolicy().max_bytes)\n    if hashlib.sha256(raw).hexdigest()!=m.get(\'source_sha256\'):raise IntakeError(\'Original source hash mismatch\')\n    files={}\n    for name in [\'source.png\',\'refined-alpha.png\',\'trimap.png\']:\n        data=read(name)\n        if hashlib.sha256(data).hexdigest()!=m[\'output_sha256\'].get(name):raise IntakeError(\'Matting bundle hash mismatch: \'+name)\n        files[name]=data\n    # Decode the exact bytes whose hash was checked, never reopen a mutable\n    # external source path after validation. Close before opening on Windows.\n    with tempfile.TemporaryDirectory(prefix=\'renderello-matting-source-\') as td:\n        snapshot=Path(td)/\'source.png\'\n        snapshot.write_bytes(raw)\n        original=inspect_image(snapshot)\n    try:\n        # The matting implementation works on encoded pixels. Do not guess an\n        # EXIF transform or silently reinterpret a raw/normalized coordinate map.\n        if original.record[\'orientation\']!=1 or original.record[\'input_mode\']!=\'RGB\' or original.record[\'transformations\']:\n            raise IntakeError(\'Use normalized RGB source with no orientation or color transform\')\n        if original.image.width*original.image.height>1_000_000:raise IntakeError(\'Matting bridge capped at one million source pixels; no resizing\')\n        rgb=np.array(original.image);size=original.image.size\n    finally:original.close()\n    arrays={}\n    for name,mode in [(\'source.png\',\'RGB\'),(\'refined-alpha.png\',\'L\'),(\'trimap.png\',\'L\')]:\n        data=files[name]\n        if not data.startswith(b\'\\x89PNG\\r\\n\\x1a\\n\') or len(data)<29 or data[24]!=8:raise IntakeError(\'Plain 8-bit PNG required\')\n        with Image.open(BytesIO(data)) as im:\n            if im.format!=\'PNG\' or im.mode!=mode or im.size!=size or getattr(im,\'n_frames\',1)!=1:raise IntakeError(\'Matting bundle size/mode mismatch\')\n            if im.getexif().get(274,1)!=1 or im.info.get(\'icc_profile\') or \'transparency\' in im.info:raise IntakeError(\'Ambiguous matting metadata\')\n            im.load();arrays[name]=np.array(im)\n    if not np.array_equal(arrays[\'source.png\'],rgb):raise IntakeError(\'Matting pixels disagree with original source\')\n    tri,alpha=arrays[\'trimap.png\'],arrays[\'refined-alpha.png\']\n    if not set(np.unique(tri)).issubset({0,128,255}) or not (np.any(tri==0) and np.any(tri==255)):raise IntakeError(\'Valid known foreground/background trimap required\')\n    if np.any(alpha[tri==255]!=255) or np.any(alpha[tri==0]!=0):raise IntakeError(\'Refined alpha conflicts with protected trimap core\')\n    evidence=dict(method=\'verified_matting_alpha_only\',source_sha256=hashlib.sha256(raw).hexdigest(),matting_manifest_sha256=hashlib.sha256(manifest_raw).hexdigest(),refined_alpha_sha256=hashlib.sha256(files[\'refined-alpha.png\']).hexdigest(),source_rgb_policy=\'ORIGINAL_RGB_ONLY_ESTIMATED_FOREGROUND_IGNORED\',protected_core_pixels=int((tri==255).sum()),alpha_semantics=\'ESTIMATED_NOT_GROUND_TRUTH\',physical_transmission=\'NOT_INFERRED\',quality_acceptance=\'NOT_EVALUATED\',defaults_changed=False)\n    dest.parent.mkdir(parents=True,exist_ok=True)\n    stage=Path(tempfile.mkdtemp(prefix=\'.\'+dest.name+\'-alpha-\',dir=dest.parent))\n    try:\n        # Copy verified mask bytes, not an unchecked path read again later.\n        (stage/\'alpha.png\').write_bytes(files[\'refined-alpha.png\'])\n        (stage/\'source.png\').write_bytes(raw)\n        record=prepare_reviewed_source(stage/\'source.png\',stage/\'prepared\',mask=stage/\'alpha.png\',mask_coordinates=\'display\')\n        record[\'estimated_alpha_provenance\']=evidence\n        (stage/\'prepared/prepared-source.json\').write_text(json.dumps(record,indent=2))\n        # Never overwrite a destination that appeared while validating.\n        dest.mkdir()\n        try:\n            for child in (stage/\'prepared\').iterdir():child.rename(dest/child.name)\n        except BaseException:\n            (dest/\'PREPARATION_INCOMPLETE\').touch();raise\n        return record\n    finally:shutil.rmtree(stage)\n\n\ndef main(argv=None):\n    import argparse\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--source\',required=True,type=Path);p.add_argument(\'--bundle\',required=True,type=Path);p.add_argument(\'--out\',required=True,type=Path)\n    a=p.parse_args(argv);print(json.dumps(prepare_alpha_only(a.source,a.bundle,a.out),indent=2));return 0\n\nif __name__==\'__main__\':raise SystemExit(main())\n'
# ---- embedded module: renderello/matting_alpha_prepare.py ----
path = base / 'renderello/matting_alpha_prepare.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Durable Magic Hour spend ledger backed by a private, isolated GitLab\nproject (Option B).\n\nWhy this exists: the on-host JSONL ledger lives on ephemeral storage and\nresets on every redeploy, and the balance anchor alone cannot see partial\ntop-ups. The durable ledger stores the cumulative committed spend as a single\nJSON document in a ledger-only GitLab project with NO deployment hook, updated\nwith strict compare-and-swap through the Repository Files API:\n\n  1. GET the file -> current content + last_commit_id.\n  2. PUT new content naming that last_commit_id (optimistic concurrency).\n  3. On 409 conflict, re-read and retry (bounded); any other failure raises.\n\nHard rules:\n  - Reserve BEFORE any paid call; a failed or unreachable reserve fails\n    CLOSED (the paid call never happens).\n  - Reconcile moves the reservation TO the observed charge: down when the\n    provider charged less, UP when it charged more (the listed price is a\n    lowest-resolution floor, not a ceiling - an overcharge is real spend\n    the durable ledger must record). A failed reconcile keeps the\n    reservation; the balance anchor then catches any real overcharge and\n    freezes spend until the owner reconciles the counters.\n  - The token is a minimum project-scoped, expiring access token supplied\n    via the server environment only. It is never logged, never included in\n    error messages, and never written to disk by this module.\n  - The ledger stores aggregate counters only: no user data, no prompts,\n    no image references, no account identifiers (no PII).\n\nConfiguration (all server-side env):\n  RENDERELLO_MH_DURABLE_LEDGER   "1"/"on" to enable; when enabled, ANY\n                                 reserve/reconcile failure refuses the spend.\n  RENDERELLO_MH_LEDGER_PROJECT   URL-encoded project path, e.g.\n                                 "your-group%2Fyour-ledger-project"\n  RENDERELLO_MH_LEDGER_TOKEN     project access token (api scope, <=30 days)\n  RENDERELLO_MH_LEDGER_FILE      ledger path in repo (default "spend.json")\n  RENDERELLO_MH_LEDGER_BRANCH    branch (default "main")\n  RENDERELLO_MH_LEDGER_API       API base (default "https://gitlab.com/api/v4")\n"""\nfrom __future__ import annotations\n\nimport base64\nimport json\nimport os\nimport time\nfrom urllib.parse import quote\n\nDEFAULT_FILE = "spend.json"\nDEFAULT_BRANCH = "main"\nDEFAULT_API = "https://gitlab.com/api/v4"\nMAX_CAS_RETRIES = 6\n\n\nclass LedgerError(RuntimeError):\n    """Any durable-ledger failure. Callers treat this as fail-closed: the\n    paid call must not proceed. Messages never carry the token or response\n    bodies."""\n\n\ndef enabled() -> bool:\n    return os.environ.get("RENDERELLO_MH_DURABLE_LEDGER", "").strip().lower() \\\n        in ("1", "on", "true", "yes")\n\n\ndef _config() -> dict:\n    project = os.environ.get("RENDERELLO_MH_LEDGER_PROJECT", "").strip()\n    token = os.environ.get("RENDERELLO_MH_LEDGER_TOKEN", "").strip()\n    if not project or not token:\n        raise LedgerError(\n            "durable ledger enabled but RENDERELLO_MH_LEDGER_PROJECT / "\n            "RENDERELLO_MH_LEDGER_TOKEN unset; live spend refused, zero "\n            "credits billed.")\n    return {\n        "api": os.environ.get("RENDERELLO_MH_LEDGER_API", DEFAULT_API).rstrip("/"),\n        "project": quote(project, safe="") if "/" in project else project,\n        "token": token,\n        "file": os.environ.get("RENDERELLO_MH_LEDGER_FILE", DEFAULT_FILE),\n        "branch": os.environ.get("RENDERELLO_MH_LEDGER_BRANCH", DEFAULT_BRANCH),\n    }\n\n\ndef _request(method: str, url: str, token: str, payload: dict | None = None):\n    """One GitLab API call. Returns (status, parsed-json-or-None). Never\n    raises on HTTP errors; raises LedgerError on transport/parse failure.\n    Error paths do not propagate response bodies (could echo request data)."""\n    import requests\n    headers = {"PRIVATE-TOKEN": token}\n    try:\n        resp = requests.request(method, url, headers=headers, json=payload,\n                                timeout=20)\n    except Exception as e:\n        raise LedgerError(\n            f"durable ledger unreachable ({type(e).__name__}); live spend "\n            "refused, zero credits billed.")\n    body = None\n    if resp.content:\n        try:\n            body = resp.json()\n        except ValueError:\n            body = None\n    return resp.status_code, body\n\n\ndef _file_url(cfg: dict) -> str:\n    return (f"{cfg[\'api\']}/projects/{cfg[\'project\']}/repository/files/"\n            f"{quote(cfg[\'file\'], safe=\'\')}")\n\n\ndef read_state(cfg: dict | None = None) -> dict:\n    """Current durable state: {"spent": int, "updated": str, "commit": str}.\n    Raises LedgerError on any failure (fail closed)."""\n    cfg = cfg or _config()\n    status, body = _request(\n        "GET", f"{_file_url(cfg)}?ref={quote(cfg[\'branch\'], safe=\'\')}",\n        cfg["token"])\n    if status == 404:\n        raise LedgerError(\n            "durable ledger file missing in the ledger project; live spend "\n            "refused, zero credits billed. Seed it before enabling.")\n    if status != 200 or not isinstance(body, dict):\n        raise LedgerError(\n            f"durable ledger read failed (HTTP {status}); live spend "\n            "refused, zero credits billed.")\n    try:\n        content = base64.b64decode(body["content"]).decode("utf-8")\n        state = json.loads(content)\n        spent = int(state["spent"])\n        commit = str(body["last_commit_id"])\n    except (KeyError, ValueError, TypeError, json.JSONDecodeError):\n        raise LedgerError(\n            "durable ledger content is malformed; live spend refused, zero "\n            "credits billed.")\n    return {"spent": spent, "updated": str(state.get("updated", "")),\n            "commit": commit}\n\n\ndef _write_state(cfg: dict, spent: int, last_commit_id: str) -> None:\n    """CAS write: update the ledger only if it is still at last_commit_id.\n    Raises LedgerError on conflict (caller retries) or any failure."""\n    payload = {\n        "branch": cfg["branch"],\n        "content": json.dumps({\n            "spent": int(spent),\n            "updated": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),\n        }, indent=2) + "\\n",\n        "commit_message": f"ledger: spent -> {int(spent)}",\n        "last_commit_id": last_commit_id,\n    }\n    status, _ = _request("PUT", _file_url(cfg), cfg["token"], payload)\n    if status == 200:\n        return\n    if status in (400, 409):\n        # 400 is GitLab\'s "file changed" validation failure on a stale\n        # last_commit_id; 409 is the explicit conflict. Both mean: retry.\n        raise LedgerError("cas-conflict")\n    raise LedgerError(\n        f"durable ledger write failed (HTTP {status}); live spend refused, "\n        "zero credits billed.")\n\n\ndef _mutate(delta: int) -> int:\n    """Apply `delta` to the durable spent counter with CAS retries. Returns\n    the new committed total. Raises LedgerError on persistent failure."""\n    import random\n    cfg = _config()\n    for attempt in range(MAX_CAS_RETRIES):\n        state = read_state(cfg)\n        new_spent = state["spent"] + delta\n        if new_spent < 0:\n            new_spent = 0\n        try:\n            # The commit id sent here always comes from this client\'s own\n            # read, so it always names a REAL commit. GitLab rejects a\n            # stale real id with HTTP 400 (verified live 2026-11-20),\n            # which is what makes this CAS exactly-one-wins. GitLab does\n            # NOT reject a nonexistent commit id (live-verified gap), so\n            # never construct one: only ids returned by read_state.\n            _write_state(cfg, new_spent, state["commit"])\n            return new_spent\n        except LedgerError as e:\n            if str(e) == "cas-conflict" and attempt < MAX_CAS_RETRIES - 1:\n                time.sleep(0.2 * (attempt + 1) + random.uniform(0, 0.3))\n                continue\n            if str(e) == "cas-conflict":\n                raise LedgerError(\n                    "durable ledger update kept conflicting; live spend "\n                    "refused, zero credits billed.")\n            raise\n    raise LedgerError("durable ledger update failed; refused.")  # unreachable\n\n\ndef reserve(credits: int) -> int:\n    """Reserve `credits` BEFORE the paid call. Returns the new durable\n    total. Any failure raises LedgerError -> caller must refuse the spend\n    (fail closed). No-op caller responsibility when not enabled()."""\n    if credits <= 0:\n        raise LedgerError("reserve amount must be positive")\n    return _mutate(int(credits))\n\n\ndef reconcile(reserved: int, observed: int) -> int:\n    """After the paid call, move the reservation TO the observed charge.\n    Downward when the observed charge is lower; UPWARD when the provider\n    charged more than the listed reservation - the overcharge is real\n    spend and the durable ledger must record it, otherwise a restart\n    (which drops the local reservation) would permanently under-count.\n    Unknown observed charges must NOT reach this function: callers keep\n    the reservation instead. Raises LedgerError on failure - callers keep\n    the reservation; the balance anchor then catches the discrepancy and\n    freezes spend until the owner reconciles the counters."""\n    delta = int(observed) - int(reserved)\n    if delta == 0:\n        cfg = _config()\n        return read_state(cfg)["spent"]\n    return _mutate(delta)\n\n\ndef spent() -> int:\n    """Durable committed spend. Raises LedgerError on failure."""\n    return read_state()["spent"]\n'
# ---- embedded module: renderello/mh_ledger.py ----
path = base / 'renderello/mh_ledger.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""S2 comparison runner: run the SAME source + prompt through 2-3 Magic Hour\nimage-editor models and record outputs, latency and credits side by side.\n\nTwo layers, kept strictly separate:\n- plan_compare(): pure projection from the published table. Zero network, zero\n  spend. This is the default CLI behavior.\n- run_compare(): executes one edit per model. With live backends this SPENDS\n  Magic Hour credits and therefore requires explicit --live on the CLI (owner\n  approval). Tests drive it with stub backends (zero network, zero spend).\n"""\nfrom __future__ import annotations\n\nimport json\nimport time\nfrom pathlib import Path\n\nfrom PIL import Image\n\nfrom .backends.magichour import MODEL_TABLE, MagicHourBackend\nfrom .costs import effective_usd_per_credit\nfrom .prompts import build_negative_prompt, build_prompt\nfrom .schemas import Aspect, EditRequest, TemplateSpec\n\nDEFAULT_COMPARE_SET = ["flux-2-klein", "qwen-edit", "seedream-v4"]\n\n\ndef plan_compare(models: list[str], plan: str = "creator_monthly") -> dict:\n    """Zero-spend projection of what a live comparison would do and cost."""\n    rows = []\n    for m in models:\n        if m not in MODEL_TABLE:\n            raise KeyError(f"unknown model \'{m}\'. known: {\', \'.join(sorted(MODEL_TABLE))}")\n        t = MODEL_TABLE[m]\n        rows.append({\n            "model": m,\n            "credits_listed_from": t["credits"],\n            "tier_required": t["tier"],\n            "max_inputs_editor": t["max_inputs"],\n            "projected_usd_per_image": round(t["credits"] * effective_usd_per_credit(plan), 6),\n        })\n    return {\n        "kind": "plan_only_no_spend",\n        "plan_basis": plan,\n        "note": "projections use published \'From\' single-image lowest-res rates "\n                "and the plan\'s bulk included-credit rate; not marginal cost, "\n                "not measured latency",\n        "models": rows,\n        "total_credits_listed": sum(r["credits_listed_from"] for r in rows),\n        "total_projected_usd": round(sum(r["projected_usd_per_image"] for r in rows), 6),\n    }\n\n\ndef run_compare(source_path: Path, models: list[str], template: TemplateSpec,\n                seed: int, aspect: Aspect, out_dir: Path,\n                backend_factory=None, live: bool = False) -> dict:\n    """One edit per model, same source/prompt/aspect; seed local/mock only. Returns the manifest\n    and writes it to out_dir/compare_manifest.json plus one PNG per model."""\n    if backend_factory is None:\n        def backend_factory(model: str):\n            return MagicHourBackend(model=model, live=live,\n                                    cost_log=out_dir / "cost_log.jsonl",\n                                    receipt_path=out_dir / "project_receipts.jsonl")\n    plan_compare(models)  # validate every model before any network/spend\n    out_dir.mkdir(parents=True, exist_ok=True)\n    source = Image.open(source_path).convert("RGB")\n    req = EditRequest(source_path=source_path, template=template, seed=seed)\n    prompt = build_prompt(req)\n    negative = build_negative_prompt()\n    manifest_path = out_dir / "compare_manifest.json"\n    if manifest_path.exists() or (out_dir / "project_receipts.jsonl").exists():\n        raise FileExistsError("Comparison evidence already exists. Recover recorded project IDs; "\n                              "never repeat a possibly charged run in this directory.")\n    manifest = {\n        "kind": "mh_model_comparison", "state": "running",\n        "source": str(source_path), "models_requested": models,\n        "template": template.key, "seed": seed, "aspect": aspect.value,\n        "seed_note": "local/mock seed only; MH API does not accept a seed",\n        "negative_prompt_note": "recorded only; MH API has no negative_prompt field",\n        "prompt": prompt, "negative_prompt": negative,\n        "note": "listed credits are price floors, not measured charges; missing actual charge is unknown",\n        "results": [],\n    }\n    def checkpoint():\n        tmp = manifest_path.with_suffix(".tmp")\n        tmp.write_text(json.dumps(manifest, indent=2))\n        tmp.replace(manifest_path)\n    checkpoint()  # directory is claimed BEFORE the first possibly charged call\n    for m in models:\n        backend = backend_factory(m)\n        row = {"model": m, "live": getattr(backend, "live", False),\n               "state": "submitting", "credits_listed_from": MODEL_TABLE[m]["credits"]}\n        manifest["results"].append(row)\n        checkpoint()\n        t0 = time.monotonic()\n        try:\n            img = backend.edit(source, prompt, negative, seed, aspect)\n            out_path = out_dir / f"compare_{m}_seed{seed}.png"\n            img.save(out_path)\n            row.update(state="complete", output=str(out_path))\n        except Exception as exc:\n            row.update(state="failed_or_unconfirmed", error=str(exc),\n                       recovery="Check receipts/provider projects. Do not resubmit automatically.")\n            manifest["state"] = "interrupted"\n            raise\n        finally:\n            result = getattr(backend, "last_result", None) or {}\n            row.update(latency_s_measured=round(time.monotonic()-t0,3),\n                       project_id=result.get("id"),\n                       credits_charged=(result.get("final") or {}).get("credits_charged"),\n                       credits_at_submission=(result.get("submission") or {}).get("credits_charged"))\n            checkpoint()\n    manifest["state"] = "complete"\n    checkpoint()\n    return manifest\n'
# ---- embedded module: renderello/mhcompare.py ----
path = base / 'renderello/mhcompare.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Multi-image input: several product photos composed into one shot, and\nreference/inspiration images that steer style.\n\nOwner requirement (2026-11-20): the pipeline takes multiple product images\nand reference images. Two distinct roles, kept separate everywhere:\n\n- PRODUCT images: every one is cut out and composited; each product\'s pixels\n  stay source-exact (Route 2 construction). A group shot arranges all of\n  them on one ground line with matched scale.\n- REFERENCE images: never composited. They contribute a measured style\n  (palette, warmth, brightness, contrast) that steers the scene prompt and\n  the procedural/generative background.\n"""\nfrom __future__ import annotations\n\nimport colorsys\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image\n\nfrom .compose import (composite, contact_shadow, fit_product, placement_params,\n                      procedural_background)\nfrom .cutout import cutout\nfrom .schemas import Aspect, CutoutResult, TemplateSpec\n\n\n@dataclass\nclass ReferenceStyle:\n    """Measured style of a reference/inspiration photo."""\n    source: str\n    dominant_rgb: tuple[int, int, int]\n    warmth: float          # -1 cool .. +1 warm\n    brightness: float      # 0..1 mean luminance\n    saturation: float      # 0..1 mean saturation\n    contrast: float        # luminance std, 0..1-ish\n\n    def descriptors(self) -> list[str]:\n        d: list[str] = []\n        d.append("warm color cast" if self.warmth > 0.12 else\n                 "cool color cast" if self.warmth < -0.12 else "neutral color balance")\n        d.append("bright high-key" if self.brightness > 0.66 else\n                 "dark low-key" if self.brightness < 0.33 else "mid-key exposure")\n        d.append("saturated palette" if self.saturation > 0.35 else\n                 "muted desaturated palette" if self.saturation < 0.18 else\n                 "moderately saturated palette")\n        d.append("high contrast" if self.contrast > 0.24 else\n                 "low soft contrast" if self.contrast < 0.12 else "moderate contrast")\n        r, g, b = self.dominant_rgb\n        d.append(f"dominant backdrop color around rgb({r},{g},{b})")\n        return d\n\n    def prompt_fragment(self) -> str:\n        return ("Style reference: " + ", ".join(self.descriptors()) +\n                ". Apply this palette and tonal treatment to the scene only, "\n                "never to the product.")\n\n\ndef extract_reference_style(path: Path, sample: int = 160) -> ReferenceStyle:\n    """Measure palette/tonal statistics of a reference image. Cheap and CPU."""\n    img = Image.open(path).convert("RGB")\n    img.thumbnail((sample, sample))\n    a = np.asarray(img).astype(np.float32) / 255.0\n    r, g, b = a[..., 0], a[..., 1], a[..., 2]\n    lum = 0.2126 * r + 0.7152 * g + 0.0722 * b\n    flat = a.reshape(-1, 3)\n    # Dominant color via coarse 4x4x4 histogram binning (robust to outliers).\n    bins = (flat * 3.999).astype(np.int32)\n    keys = bins[:, 0] * 16 + bins[:, 1] * 4 + bins[:, 2]\n    vals, counts = np.unique(keys, return_counts=True)\n    top = vals[counts.argmax()]\n    dom = tuple(min(255, int(round((c + 0.5) / 4 * 255))) for c in\n                ((top // 16) % 4, (top // 4) % 4, top % 4))\n    mx = flat.max(axis=1)\n    mn = flat.min(axis=1)\n    sat = float(np.mean(np.where(mx > 0, (mx - mn) / np.clip(mx, 1e-6, 1), 0)))\n    warmth = float(np.mean(r - b))\n    return ReferenceStyle(\n        source=str(path), dominant_rgb=dom,\n        warmth=warmth, brightness=float(lum.mean()),\n        saturation=sat, contrast=float(lum.std()))\n\n\ndef merged_reference_style(paths: list[Path]) -> ReferenceStyle | None:\n    """Average several references into one style (mean of measurements)."""\n    if not paths:\n        return None\n    styles = [extract_reference_style(p) for p in paths]\n    dom = tuple(int(round(np.mean([s.dominant_rgb[i] for s in styles])))\n                for i in range(3))\n    return ReferenceStyle(\n        source=", ".join(s.source for s in styles),\n        dominant_rgb=dom,\n        warmth=float(np.mean([s.warmth for s in styles])),\n        brightness=float(np.mean([s.brightness for s in styles])),\n        saturation=float(np.mean([s.saturation for s in styles])),\n        contrast=float(np.mean([s.contrast for s in styles])))\n\n\n@dataclass\nclass GroupMember:\n    path: Path\n    cutout: CutoutResult | None = None\n    error: str | None = None\n\n\ndef cutout_products(paths: list[Path], out_dir: Path, device: str = "cpu"\n                    ) -> list[GroupMember]:\n    """Cut out every product photo; per-item failures are recorded, not fatal."""\n    members: list[GroupMember] = []\n    for i, p in enumerate(paths):\n        m = GroupMember(path=p)\n        try:\n            img = Image.open(p).convert("RGB")\n            m.cutout = cutout(img, out_dir, stem=f"product{i+1}",\n                              backend="auto", device=device)\n        except Exception as e:  # recorded honestly, group can still render\n            m.error = f"{type(e).__name__}: {e}"\n        members.append(m)\n    return members\n\n\ndef compose_group(members: list[GroupMember], template: TemplateSpec,\n                  aspect: Aspect, seed: int, *,\n                  reference_style: ReferenceStyle | None = None) -> np.ndarray:\n    """Arrange every successful cutout on one shared ground line.\n\n    Scale is normalized so all products share a matched visual height (tall\n    items slightly taller), spaced evenly across the middle 80% of the\n    canvas, each sitting on the same baseline. Each product\'s interior\n    pixels stay bit-exact from its source cutout.\n    """\n    ok = [m for m in members if m.cutout is not None]\n    if not ok:\n        raise ValueError("no product cutouts available for the group shot")\n    W, H = aspect.size\n    bg = procedural_background((W, H), template, seed)\n    if reference_style is not None:\n        bg = _tint_background(bg, reference_style)\n    n = len(ok)\n    base_y = int(H * 0.86)\n    target_h = int(H * (0.52 if n == 1 else 0.44 if n == 2 else 0.36))\n    band_w = int(W * 0.8 / n)\n    out = bg.copy()\n    for i, m in enumerate(ok):\n        rgba = np.asarray(Image.open(m.cutout.rgba_path).convert("RGBA"))\n        h0 = max(m.cutout.bbox[3], 1)\n        scale = target_h / h0\n        new_w = max(int(rgba.shape[1] * scale), 8)\n        new_h = max(int(rgba.shape[0] * scale), 8)\n        prod = np.asarray(Image.fromarray(rgba).resize(\n            (new_w, new_h), Image.LANCZOS))\n        cx = int(W * 0.1 + band_w * (i + 0.5))\n        px = min(max(cx - new_w // 2, 0), W - new_w)\n        py = min(max(base_y - new_h, 0), H - new_h)\n        alpha = prod[..., 3:4].astype(np.float32) / 255.0\n        region = out[py:py + new_h, px:px + new_w]\n        out[py:py + new_h, px:px + new_w] = (\n            prod[..., :3] * alpha + region * (1 - alpha)).astype(np.uint8)\n    return out\n\n\ndef _tint_background(bg: np.ndarray, style: ReferenceStyle) -> np.ndarray:\n    """Pull the procedural backdrop toward the reference palette, gently.\n    Product pixels are not in the canvas yet, so this cannot touch them."""\n    dom = np.array(style.dominant_rgb, dtype=np.float32)\n    flat = bg.reshape(-1, 3).astype(np.float32)\n    lum = flat.mean(axis=1, keepdims=True)\n    tinted = np.clip(lum * 0.55 + dom * 0.45 * (flat.mean() / 255.0 + 0.55),\n                     0, 255)\n    return tinted.reshape(bg.shape).astype(np.uint8)\n'
# ---- embedded module: renderello/multinput.py ----
path = base / 'renderello/multinput.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Free native-pixel graphic variants, explicitly not photographic scenes."""\nfrom pathlib import Path\nimport hashlib\nimport json\nimport numpy as np\nfrom PIL import Image, ImageDraw, PngImagePlugin\n\nPRESETS = (\'navy_diagonal\', \'cream_editorial\', \'cobalt_circle\', \'sage_frame\')\n\ndef render_variants(source, rgba_path, output):\n    output = Path(output)\n    if output.exists():\n        raise FileExistsError(\'Preserve prior variant runs; use a new directory\')\n    with Image.open(source) as im:\n        if im.mode != \'RGB\' or im.getexif().get(274, 1) != 1 or getattr(im,\'n_frames\',1) != 1:\n            raise ValueError(\'Single orientation-normalized RGB source required\')\n        rgb = np.array(im)\n        profile = im.info.get(\'icc_profile\')\n    with Image.open(rgba_path) as im:\n        if im.mode != \'RGBA\' or im.size != (rgb.shape[1], rgb.shape[0]):\n            raise ValueError(\'Exact native RGBA dimensions required\')\n        rgba = np.array(im)\n    if not np.array_equal(rgb, rgba[:, :, :3]):\n        raise ValueError(\'RGBA repaints source pixels\')\n    a = rgba[:, :, 3]\n    yy, xx = np.where(a > 0)\n    if len(xx) == 0:\n        raise ValueError(\'Empty predicted mask\')\n    if rgb.shape[0] * rgb.shape[1] > 1_000_000:\n        raise ValueError(\'One-megapixel source budget\')\n    box = (xx.min(), yy.min(), xx.max()+1, yy.max()+1)\n    crop = rgba[box[1]:box[3], box[0]:box[2]]\n    ph, pw = crop.shape[:2]\n    w, h = max(800, pw+48), max(800, ph+180)\n    if w*h > 2_000_000:\n        raise ValueError(\'Canvas exceeds bounded native variant budget\')\n    output.mkdir(parents=True)\n    rows = []\n    for key in PRESETS:\n        bg = Image.new(\'RGB\', (w,h), \'#f7f3ed\')\n        d = ImageDraw.Draw(bg)\n        if key == \'navy_diagonal\':\n            d.rectangle((0,0,w,h), fill=\'#142236\')\n            d.polygon([(0,int(h*.25)),(w,int(h*.06)),(w,int(h*.86)),(0,h)], fill=\'#d9e0e7\')\n            d.line([(int(w*.06),int(h*.11)),(int(w*.41),int(h*.11))], fill=\'#cb9b58\', width=5)\n        elif key == \'cream_editorial\':\n            d.rectangle((int(w*.08),int(h*.08),int(w*.92),int(h*.92)), fill=\'#e5d8c6\')\n            d.rectangle((int(w*.13),int(h*.13),int(w*.87),int(h*.87)), outline=\'#9e805d\',width=2)\n            d.rectangle((0,int(h*.73),w,h), fill=\'#f6eee3\')\n        elif key == \'cobalt_circle\':\n            d.rectangle((0,0,w,h), fill=\'#183a87\')\n            d.ellipse((int(-w*.10),int(h*.12),int(w*1.10),int(h*1.24)), fill=\'#b8d6ed\')\n            d.rectangle((int(w*.77),0,int(w*.84),int(h*.21)), fill=\'#e4a48b\')\n        else:\n            d.rectangle((0,0,w,h), fill=\'#c8d4c5\')\n            d.rectangle((int(w*.07),int(h*.07),int(w*.93),int(h*.93)), fill=\'#edf0e8\')\n            d.rectangle((0,int(h*.79),w,h), fill=\'#6f8978\')\n            d.ellipse((int(w*.72),int(h*.07),int(w*.91),int(h*.26)), fill=\'#d6b98c\')\n        pixels = np.array(bg)\n        px, py = (w-pw)//2, (h-ph)//2+15\n        region = pixels[py:py+ph, px:px+pw]\n        alpha = crop[:,:,3:4].astype(np.float64)/255\n        region[:] = np.rint(crop[:,:,:3]*alpha+region*(1-alpha)).clip(0,255).astype(\'uint8\')\n        opaque = crop[:,:,3] == 255\n        changed = int(np.count_nonzero(np.any(region[opaque] != crop[:,:,:3][opaque],axis=1)))\n        assert changed == 0\n        meta = PngImagePlugin.PngInfo()\n        meta.add_text(\'status\', \'DESIGNED_CPU_GRAPHIC_NOT_PHOTOGRAPHIC_NOT_ACCEPTED\')\n        path = output/(key+\'.png\')\n        Image.fromarray(pixels).save(path,pnginfo=meta,**({\'icc_profile\':profile} if profile else {}))\n        rows.append(dict(preset=key,output=path.name,sha256=hashlib.sha256(path.read_bytes()).hexdigest(),opaque_changed_pixels=changed,placement=[px,py,pw,ph]))\n    record = dict(schema=\'renderello-native-graphic-variants-v1\',source_sha256=hashlib.sha256(Path(source).read_bytes()).hexdigest(),rgba_sha256=hashlib.sha256(Path(rgba_path).read_bytes()).hexdigest(),accepted=0,source_icc_preserved=bool(profile),source_icc_sha256=hashlib.sha256(profile).hexdigest() if profile else None,physical_relighting=False,physical_contact=False,product_resampled=False,quality=\'NOT_EVALUATED\',mask_provenance=\'IMPORTED_MODEL_ESTIMATE_NOT_GROUND_TRUTH\',variants=rows)\n    (output/\'manifest.json\').write_text(json.dumps(record,indent=2),encoding=\'utf-8\')\n    return record\n\nif __name__ == \'__main__\':\n    import argparse\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--source\',required=True,type=Path)\n    p.add_argument(\'--rgba\',required=True,type=Path)\n    p.add_argument(\'--out\',required=True,type=Path)\n    a=p.parse_args()\n    print(json.dumps(render_variants(a.source,a.rgba,a.out),indent=2))\n'
# ---- embedded module: renderello/native_graphic_variants.py ----
path = base / 'renderello/native_graphic_variants.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Unregistered native-layout review packs, without stretching product geometry.\n\nUnlike blind_review\'s shared-coordinate packs, these packs allow different native\nsizes. They preserve original encoded bytes and prohibit pixelwise/ROI alignment\nclaims. Tokenization does not prove independence or hide recognizable methods.\nVerification detects accidental changes against COMPLETE.json, not authenticity\nagainst a party able to replace both the files and the manifest.\n"""\nfrom pathlib import Path\nimport csv\nimport hashlib\nimport json\nimport random\nimport shutil\nfrom .blind_review import Candidate\nfrom .image_intake import IntakePolicy, inspect_image\nfrom .review import DIMENSIONS\n\n\ndef digest(path):\n    h=hashlib.sha256();total=0\n    with Path(path).open(\'rb\') as f:\n        while True:\n            chunk=f.read(1024*1024)\n            if not chunk:break\n            total+=len(chunk)\n            if total>40_000_000:raise ValueError(\'Review input exceeds encoded byte budget\')\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef build_native_pack(source, candidates, out, *, seed):\n    source=Path(source);out=Path(out)\n    if out.exists():raise FileExistsError(\'Never replace an earlier review\')\n    if type(seed) is not int:raise ValueError(\'Integer seed required\')\n    if not 2<=len(candidates)<=12:raise ValueError(\'Need2..12 candidates\')\n    if any(not isinstance(c,Candidate) for c in candidates):raise ValueError(\'Typed candidate required\')\n    if len({c.method for c in candidates})!=len(candidates):raise ValueError(\'Duplicate method\')\n    policy=IntakePolicy(allow_colour_conversion=True)\n    def inspect(path):\n        path=Path(path)\n        if path.is_symlink() or not path.is_file():raise ValueError(\'Regular nonsymlink input required\')\n        before=digest(path);image=inspect_image(path,policy)\n        try:return {\'path\':path,\'sha256\':before,\'size\':list(image.record[\'input_size\']),\'display_size\':list(image.image.size),\'mode\':image.image.mode,\'format\':image.record[\'input_format\']}\n        finally:image.close()\n    src=inspect(source);records=[];seen={src[\'sha256\']}\n    order=list(candidates);random.Random(seed).shuffle(order)\n    for c in order:\n        if c.path is None:records.append((c,None));continue\n        info=inspect(c.path)\n        if info[\'sha256\'] in seen:raise ValueError(\'Duplicate source/output content\')\n        seen.add(info[\'sha256\']);records.append((c,info))\n    # Validate every input before creating the output, then recheck during copy.\n    for info in [src,*[i for _,i in records if i]]:\n        if digest(info[\'path\'])!=info[\'sha256\']:raise ValueError(\'Input changed during preparation\')\n    out.mkdir(parents=True);review=out/\'REVIEW\';review.mkdir();private=out/\'PRIVATE\';private.mkdir()\n    files={};key=[];rows=[]\n    suffix={\'JPEG\':\'.jpg\',\'PNG\':\'.png\',\'WEBP\':\'.webp\',\'TIFF\':\'.tiff\',\'BMP\':\'.bmp\'}\n    def copy(info,token):\n        name=token+suffix[info[\'format\']];target=review/name\n        shutil.copyfile(info[\'path\'],target)\n        if digest(target)!=info[\'sha256\'] or digest(info[\'path\'])!=info[\'sha256\']:\n            raise ValueError(\'Input changed while copying; pack incomplete\')\n        files[name]=info[\'sha256\'];return name\n    source_name=copy(src,\'source\')\n    for index,(candidate,info) in enumerate(records,1):\n        token=f\'C{index:02d}\';name=copy(info,token) if info else None\n        key.append({\'token\':token,\'method\':candidate.method,\'source_path\':str(candidate.path) if candidate.path else None,\'failure\':candidate.failure})\n        rows.append({\'token\':token,\'file\':name,\'native_size\':info[\'size\'] if info else None,\'status\':\'NATIVE_ARTIFACT_PRESENT\' if info else \'MISSING_OUTPUT\',\'failure\':candidate.failure if not info else \'\'})\n    manifest={\'schema\':\'native-layout-review.v1\',\'source_file\':source_name,\'source_size\':src[\'size\'],\'candidates\':rows,\n      \'registration\':\'NOT_ESTABLISHED\',\'pixelwise_comparison_permitted\':False,\n      \'native_encoded_bytes_preserved\':True,\'independent_blinding\':\'NOT_ESTABLISHED\',\n      \'recognizable_methods_may_be_known\':True,\'ratings\':\'BLANK\',\'acceptance\':\'NOT_EVALUATED\'}\n    (review/\'manifest.json\').write_text(json.dumps(manifest,indent=2))\n    with (review/\'scores-blank.csv\').open(\'w\',newline=\'\') as f:\n        fields=[\'token\',*DIMENSIONS,\'no_critical\',\'critical_errors\',\'provenance\',\'notes\'];writer=csv.DictWriter(f,fieldnames=fields);writer.writeheader();writer.writerows({\'token\':r[\'token\']} for r in rows)\n    (private/\'key.json\').write_text(json.dumps({\'seed\':seed,\'source_path\':str(source),\'candidates\':key},indent=2))\n    for p in review.iterdir():\n        if p.is_file():files[p.name]=digest(p)\n    (out/\'COMPLETE.json\').write_text(json.dumps({\'review_sha256\':files,\'outputs\':sum(i is not None for _,i in records),\'planned\':len(records)},indent=2))\n    return out\n\n\ndef verify_native_pack(out):\n    out=Path(out);record=json.loads((out/\'COMPLETE.json\').read_text());review=out/\'REVIEW\'\n    hashes=record[\'review_sha256\']\n    if not isinstance(hashes,dict) or not hashes:raise ValueError(\'Empty manifest\')\n    if review.is_symlink() or {p.name for p in review.iterdir()}!=set(hashes):raise ValueError(\'Unexpected review contents\')\n    for name,sha in hashes.items():\n        path=review/name\n        if Path(name).name!=name or path.is_symlink() or not path.is_file():raise ValueError(\'Unsafe review path\')\n        if digest(path)!=sha:raise ValueError(\'Review content changed\')\n    return record\n'
# ---- embedded module: renderello/native_layout_review.py ----
path = base / 'renderello/native_layout_review.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Native whole-photo editorial ad layouts, separate from product staging.\n\nNo matting, scene generation, crop, resize, relighting or automatic approval.\n"""\nfrom dataclasses import dataclass,asdict\nimport hashlib\nfrom io import BytesIO\nimport numpy as np\nfrom PIL import Image,ImageDraw,ImageFont\nfrom .candidate_diagnostics import native_rgb,digest\n\nLAYOUTS={\n \'square\':dict(size=(1024,1024),well=(72,330,952,928),text=(76,88,948,294),footer=(76,970)),\n \'portrait\':dict(size=(1024,1280),well=(72,438,952,1036),text=(76,112,948,390),footer=(76,1160)),\n \'landscape\':dict(size=(1440,800),well=(528,80,1408,720),text=(48,132,480,650),footer=(48,742)),\n}\n\n@dataclass(frozen=True)\nclass AdCopy:\n    eyebrow:str\n    headline:str\n    disclosure:str\n    attribution:str\n    def __post_init__(self):\n        for n,limit in [(\'eyebrow\',60),(\'headline\',100),(\'disclosure\',100),(\'attribution\',120)]:\n            value=getattr(self,n)\n            if not isinstance(value,str)or not value.strip()or len(value)>limit:\n                raise ValueError(\'bounded nonempty copy required\')\n            if any(ord(c)<32 and c!=\'\\n\'for c in value):raise ValueError(\'unsupported copy control character\')\n        if \'\\n\'in self.eyebrow or \'\\n\'in self.disclosure or \'\\n\'in self.attribution or len(self.headline.split(\'\\n\'))>2:\n            raise ValueError(\'one-line labels and at most two explicit headline lines required\')\n\n\ndef render_native_ad(rgb,copy,*,source_sha256,layout,display_font,text_font,bold_font,palette):\n    native_rgb(rgb)\n    if rgb.shape[0]*rgb.shape[1]>2_000_000:raise ValueError(\'2MP native source cap\')\n    if source_sha256!=digest(rgb):raise ValueError(\'source pixel binding changed\')\n    if not isinstance(copy,AdCopy)or layout not in LAYOUTS:raise ValueError(\'typed copy and named layout required\')\n    if not isinstance(palette,dict)or set(palette)!={\'ground\',\'surface\',\'ink\',\'secondary\',\'accent\'}:\n        raise ValueError(\'explicit five-color palette required\')\n    for value in palette.values():\n        if not isinstance(value,str)or len(value)!=7 or value[0]!=\'#\':raise ValueError(\'hex palette required\')\n        int(value[1:],16)\n    spec=LAYOUTS[layout];x0,y0,x1,y1=spec[\'well\'];h,w=rgb.shape[:2]\n    if w>x1-x0 or h>y1-y0:raise ValueError(\'whole source does not fit photo well; no crop or resize\')\n    fonts={}\n    for n,path in [(\'display\',display_font),(\'text\',text_font),(\'bold\',bold_font)]:\n        with open(path,\'rb\')as stream:fonts[n]=stream.read(5_000_001)\n    if any(len(v)>5_000_000 for v in fonts.values()):raise ValueError(\'bounded font files required\')\n    canvas=Image.new(\'RGB\',spec[\'size\'],palette[\'ground\']);draw=ImageDraw.Draw(canvas)\n    tx,ty,tr,tb=spec[\'text\'];eyefont=ImageFont.truetype(BytesIO(fonts[\'bold\']),20)\n    if draw.textbbox((0,0),copy.eyebrow,font=eyefont)[2]>tr-tx:raise ValueError(\'eyebrow does not fit; no truncation\')\n    draw.text((tx,ty),copy.eyebrow,font=eyefont,fill=palette[\'accent\'])\n    top=ty+48\n    for size in range(100,47,-1):\n        f=ImageFont.truetype(BytesIO(fonts[\'display\']),size)\n        box=draw.multiline_textbbox((0,0),copy.headline,font=f,spacing=4)\n        if box[2]-box[0]<=tr-tx and box[3]-box[1]<=tb-top:break\n    else:raise ValueError(\'headline does not fit; edit copy rather than silently shorten\')\n    draw.multiline_text((tx-box[0],top-box[1]),copy.headline,font=f,spacing=4,fill=palette[\'ink\'])\n    draw.rectangle(spec[\'well\'],fill=palette[\'surface\'])\n    dx=x0+(x1-x0-w)//2;dy=y0+(y1-y0-h)//2\n    canvas.paste(Image.fromarray(rgb),(dx,dy))\n    footfont=ImageFont.truetype(BytesIO(fonts[\'text\']),16);fx,fy=spec[\'footer\']\n    fb=draw.textbbox((0,0),copy.disclosure,font=footfont)\n    if fb[2]-fb[0]>spec[\'size\'][0]-fx-32:raise ValueError(\'disclosure does not fit\')\n    draw.text((fx,fy),copy.disclosure,font=footfont,fill=palette[\'secondary\'])\n    creditbox=draw.textbbox((0,0),copy.attribution,font=footfont)\n    if creditbox[2]-creditbox[0]>spec[\'size\'][0]-fx-32:raise ValueError(\'attribution does not fit\')\n    draw.text((fx,fy+24),copy.attribution,font=footfont,fill=palette[\'secondary\'])\n    out=np.array(canvas)\n    if not np.array_equal(out[dy:dy+h,dx:dx+w],rgb):raise RuntimeError(\'whole native photograph changed\')\n    return out,dict(schema=\'native-photo-ad-layout.v1\',layout=layout,source_sha256=digest(rgb),result_sha256=digest(out),\n        copy=asdict(copy),font_sha256={n:hashlib.sha256(v).hexdigest()for n,v in fonts.items()},palette=palette,\n        source_size=[w,h],canvas_size=list(spec[\'size\']),integer_translation=[dx,dy],source_rgb_exact_pixels=w*h,\n        source_cropped=False,source_resampled=False,source_relighting=False,matting_used=False,\n        generated_scene=False,original_background_and_shadow_retained=True,headline_font_size=size,\n        copy_is_draft_not_brand_or_owner_approved=True,photographic_acceptance=False,polished_template_acceptance=False,\n        category=\'GRAPHIC_DESIGN_CONCEPT_NOT_NEW_PRODUCT_PHOTOGRAPHY\',\n        limitations=[\'Original source photo, lighting, background and shadow remain unchanged.\',\n          \'Graphic copy is draft input, not approved brand messaging or verified product claims.\',\n          \'Three layouts of the same source are not three source photos or distinct template categories.\',\n          \'No source rights clearance, photoshoot acceptance or production default is implied.\'])\n'
# ---- embedded module: renderello/native_photo_ad.py ----
path = base / 'renderello/native_photo_ad.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Decoded, unresampled opaque-RGB check at explicit source/output coordinates.\n\nThis is neither an edge/mask evaluation nor a photographic quality score.\nA size mismatch returns unavailable, never an interpolated pixel-exact pass.\n"""\nimport numpy as np\n\n\ndef _box(value, width, height):\n    if not isinstance(value,(tuple,list)) or len(value)!=4 or any(type(x) is not int for x in value):\n        raise ValueError(\'Box requires four integer coordinates, not booleans\')\n    x,y,w,h=value\n    if min(x,y)<0 or min(w,h)<1 or x+w>width or y+h>height:\n        raise ValueError(\'Box outside decoded image\')\n    return x,y,w,h\n\n\ndef audit_native_pixels(source, alpha, output, source_box, output_box):\n    """Compare uint8 RGB arrays only where original alpha equals exactly 255."""\n    for name,array,channels in [(\'source\',source,3),(\'output\',output,3)]:\n        if not isinstance(array,np.ndarray) or array.dtype!=np.uint8 or array.ndim!=3 or array.shape[2]!=channels or min(array.shape[:2])<1:\n            raise ValueError(name+\' must be a nonempty uint8 RGB array\')\n    if not isinstance(alpha,np.ndarray) or alpha.dtype!=np.uint8 or alpha.shape!=source.shape[:2]:\n        raise ValueError(\'Alpha must be source-sized uint8 grayscale\')\n    sx,sy,sw,sh=_box(source_box,source.shape[1],source.shape[0])\n    ox,oy,ow,oh=_box(output_box,output.shape[1],output.shape[0])\n    mask=alpha[sy:sy+sh,sx:sx+sw]==255\n    count=int(mask.sum())\n    result={\'scope\':\'FULLY_OPAQUE_NATIVE_RGB_ONLY\',\'source_box\':[sx,sy,sw,sh],\n            \'output_box\':[ox,oy,ow,oh],\'fully_opaque_source_pixels\':count,\n            \'changed_pixels\':None,\'max_channel_error\':None,\'native_rgb_exact\':None,\n            \'mask_semantics\':\'NOT_EVALUATED\',\'edges\':\'NOT_EVALUATED\',\n            \'lighting_and_grounding\':\'NOT_EVALUATED\',\'quality_acceptance\':\'NOT_EVALUATED\'}\n    if (sw,sh)!=(ow,oh):\n        result[\'status\']=\'NATIVE_SIZE_CHANGED\';return result\n    if count==0:\n        result[\'status\']=\'NO_FULLY_OPAQUE_PIXELS\';return result\n    original=source[sy:sy+sh,sx:sx+sw][mask].astype(np.int16)\n    placed=output[oy:oy+oh,ox:ox+ow][mask].astype(np.int16)\n    difference=np.abs(original-placed)\n    changed=int(np.any(difference!=0,axis=1).sum())\n    result.update(status=\'MATCH\' if changed==0 else \'MISMATCH\',changed_pixels=changed,\n                  max_channel_error=int(difference.max()),native_rgb_exact=changed==0)\n    return result\n'
# ---- embedded module: renderello/native_pixel_audit.py ----
path = base / 'renderello/native_pixel_audit.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Smaller source-sized alternate compositions. No upscale or model call."""\nfrom pathlib import Path\nimport json\nimport math\nimport numpy as np\nfrom PIL import Image\nfrom .compose import finish_scene, flatlay_center\n\n\ndef native_reframe(cache_dir, *, fill=.76):\n    """Center an unchanged cached product on a smaller, same-aspect scene crop.\n\n    This deliberately exports fewer pixels rather than inventing product detail.\n    Camera angle and illumination are unchanged and still need human review.\n    Glass/cover-fit caches are excluded because their layer semantics differ.\n    """\n    if not math.isfinite(fill) or not .25 <= fill <= .9:\n        raise ValueError(\'fill must be finite and between .25 and .9\')\n    cd=Path(cache_dir)\n    options=json.loads((cd/\'layers.json\').read_text())\n    if options.get(\'cover_used\') or options.get(\'glass_backdrop\') is not None:\n        raise ValueError(\'native reframe requires an ordinary opaque-product cache\')\n    scene=np.asarray(Image.open(cd/\'scene.png\').convert(\'RGB\'))\n    product=np.asarray(Image.open(cd/\'product_rgba.png\').convert(\'RGBA\'))\n    yy,xx=np.where(product[...,3]>0)\n    if not len(xx):raise ValueError(\'empty product alpha\')\n    # Trim only fully transparent padding; all nonzero-alpha pixels retained.\n    product=product[yy.min():yy.max()+1,xx.min():xx.max()+1].copy()\n    ph,pw=product.shape[:2];h,w=scene.shape[:2]\n    if pw>w or ph>h:raise ValueError(\'product larger than scene\')\n    factor=min(1.,max(pw/(w*fill),ph/(h*fill)))\n    nw=min(w,max(pw,math.ceil(w*factor)))\n    nh=min(h,max(ph,math.ceil(h*factor)))\n    x=(w-nw)//2;y=(h-nh)//2\n    scene=scene[y:y+nh,x:x+nw].copy()\n    px,py=flatlay_center(product[...,3],(nw,nh))\n    out,lift=finish_scene(scene,product,(px,py),\n        shadow_style=options.get(\'shadow_style\'),scene_grade=options.get(\'scene_grade\',0.),\n        scene_auto_lift=options.get(\'scene_auto_lift\',False))\n    inner=product[...,3]==255\n    exact=bool(inner.any() and np.array_equal(out[py:py+ph,px:px+pw][inner],product[...,:3][inner]))\n    return Image.fromarray(out),dict(output_size=[nw,nh],original_size=[w,h],\n        product_size=[pw,ph],placement=[px,py],paste_bit_exact=exact,\n        product_resampled=False,scene_resampled=False,source_resolution_limited=True,\n        scene_lift=lift,quality_status=\'REVIEW_REQUIRED\',\n        warnings=[\'Smaller export, not a higher-resolution product.\',\n                  \'Camera angle, light direction, mask and contact need visual review.\'])\n'
# ---- embedded module: renderello/native_reframe.py ----
path = base / 'renderello/native_reframe.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""File entry point for opt-in reviewed native-region matting. No network."""\nfrom pathlib import Path\nimport argparse,json,hashlib\nimport numpy as np\nfrom PIL import Image\nfrom .matting import refine_native_region\nfrom .image_intake import inspect_image,IntakePolicy\nfrom .candidate_diagnostics import digest\n\n\ndef _sha(p):\n    from .image_intake import _read_bounded\n    p=Path(p)\n    if p.is_symlink():raise ValueError(\'regular non-symlink input required\')\n    return hashlib.sha256(_read_bounded(p,20_000_000)).hexdigest()\n\n\ndef run(source,alpha,spec,out,*,trimap=None):\n    source,alpha,spec,out=map(Path,(source,alpha,spec,out))\n    if out.exists():raise FileExistsError(\'preserve previous output directory\')\n    if spec.is_symlink() or not spec.is_file() or spec.stat().st_size>20000:raise ValueError(\'bounded regular JSON spec required\')\n    request=json.loads(spec.read_text())\n    required={\'source_file_sha256\',\'alpha_file_sha256\',\'roi\',\'reviewed\',\'review_note\',\'material\',\'method\',\'edge_colors\',\'feather\'}\n    if not isinstance(request,dict) or set(request)!=required|({\'trimap_file_sha256\'} if trimap else set()):\n        raise ValueError(\'exact explicit native-region spec fields required\')\n    before={source:_sha(source),alpha:_sha(alpha),spec:_sha(spec)}\n    if before[source]!=request[\'source_file_sha256\'] or before[alpha]!=request[\'alpha_file_sha256\']:\n        raise ValueError(\'encoded source/alpha hash mismatch\')\n    policy=IntakePolicy(max_bytes=20_000_000,max_pixels=8_000_000,allow_colour_conversion=False)\n    src=inspect_image(source,policy);mask=inspect_image(alpha,policy);tri=None\n    try:\n        # Reject transformations because ROI/masks name the original coordinate grid.\n        if src.record.get(\'transformations\') or set(mask.record.get(\'transformations\',[])) - {\'lossless_8bit_channel_expansion\'}:\n            raise ValueError(\'normalize source and masks explicitly before source-space region review\')\n        with Image.open(source) as im:\n            if im.mode!=\'RGB\' or im.getexif().get(274,1)!=1:raise ValueError(\'orientation-normalized RGB source required\')\n        with Image.open(alpha) as im:\n            if im.mode!=\'L\' or im.getexif().get(274,1)!=1:raise ValueError(\'native unrotated grayscale alpha required\')\n            aa=np.array(im)\n        rgb=np.array(src.image)\n        if trimap:\n            trimap=Path(trimap);before[trimap]=_sha(trimap)\n            if before[trimap]!=request[\'trimap_file_sha256\']:raise ValueError(\'trimap hash mismatch\')\n            with inspect_image(trimap,policy).image as checked:\n                if checked.size!=(request[\'roi\'][2]-request[\'roi\'][0],request[\'roi\'][3]-request[\'roi\'][1]):raise ValueError(\'ROI-sized trimap required\')\n            with Image.open(trimap) as im:\n                if im.mode!=\'L\' or im.getexif().get(274,1)!=1 or im.info.get(\'icc_profile\'):raise ValueError(\'plain unrotated grayscale trimap required\')\n                tri=np.array(im)\n        fg,a,meta=refine_native_region(rgb,aa,request[\'roi\'],source_sha256=digest(rgb),alpha_sha256=digest(aa),\n            reviewed=request[\'reviewed\'],review_note=request[\'review_note\'],material=request[\'material\'],method=request[\'method\'],\n            edge_colors=request[\'edge_colors\'],feather=request[\'feather\'],trimap=tri)\n        if any(_sha(p)!=h for p,h in before.items()):raise ValueError(\'input changed during matting\')\n        out.mkdir(parents=True)\n        # Save native source RGB as a plain PNG for the source-exact UI mask path.\n        Image.fromarray(rgb).save(out/\'normalized-source.png\');Image.fromarray(a).save(out/\'alpha.png\')\n        Image.fromarray(np.dstack([fg,a])).save(out/\'candidate.png\')\n        (out/\'original-source.bin\').write_bytes(source.read_bytes());(out/\'request.json\').write_bytes(spec.read_bytes())\n        x0,y0,x1,y1=request[\'roi\'];patch=fg[y0:y1,x0:x1];z=a[y0:y1,x0:x1,None].astype(float)/255\n        for label,color in [(\'white\',(255,255,255)),(\'dark\',(30,30,30)),(\'magenta\',(190,0,150))]:\n            Image.fromarray(np.rint(patch*z+np.array(color)*(1-z)).astype(np.uint8)).save(out/(label+\'-roi.png\'))\n        meta.update(input_file_sha256={str(p):h for p,h in before.items()},\n            dashboard_handoff=\'normalized-source.png plus alpha.png in existing explicit mask input; review source pose/material/framing separately\',\n            estimated_color_handoff=\'candidate.png is research-only when edge colours enabled; explicit alpha upload keeps source RGB, not estimated foreground\',\n            external_rgba_import=\'existing UI RGBA importer is capped at1MP; use explicit mask input for larger sources\',\n            runtime_guard=\'size/unknown-band checks only, no file-worker wall-clock/RSS enforcement\',live_calls=0)\n        meta[\'outputs_sha256\']={p.name:_sha(p) for p in out.iterdir() if p.is_file()}\n        (out/\'report.json\').write_text(json.dumps(meta,indent=2));return meta\n    finally:src.close();mask.close()\n\n\ndef main(argv=None):\n    p=argparse.ArgumentParser(description=__doc__)\n    for n in [\'source\',\'alpha\',\'spec\',\'out\']:p.add_argument(\'--\'+n,type=Path,required=True)\n    p.add_argument(\'--trimap\',type=Path)\n    a=p.parse_args(argv)\n    try:r=run(a.source,a.alpha,a.spec,a.out,trimap=a.trimap)\n    except (ValueError,TypeError,KeyError,OSError) as exc:p.exit(2,\'REFUSED: \'+str(exc)+\'\\n\')\n    print(json.dumps(r,indent=2))\n\n\nif __name__==\'__main__\':main()\n'
# ---- embedded module: renderello/native_region_cli.py ----
path = base / 'renderello/native_region_cli.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Replay a saved native-matting result before it becomes a studio candidate.\n\nThis proves internal consistency and source pixel pairing, not model authorship,\nsemantic segmentation truth, optical correctness or publication acceptance.\n"""\nfrom pathlib import Path\nimport hashlib\nimport json\nimport numpy as np\nfrom PIL import Image\nfrom .candidate_diagnostics import native_rgb, native_alpha, digest\nfrom .checked_research.native_matting import derive_trimap, checked_alpha, composite\nfrom .matte_change_review import audit_trimap, review_change\n\nEXPECTED = frozenset((\'source-native.png\', \'prior-support-mask.png\', \'derived-trimap.png\',\n    \'alpha.png\', \'component-rgba.png\', \'raw-alpha.npy\', \'enforced-alpha.npy\', \'dark.png\', \'light.png\'))\n\n\ndef load_verified_result(directory, source_rgb, initial_alpha):\n    native_rgb(source_rgb)\n    native_alpha(initial_alpha, source_rgb.shape[:2])\n    if source_rgb.shape[0] * source_rgb.shape[1] > 1_000_000:\n        raise ValueError(\'Native result exceeds one-megapixel replay budget\')\n    root = Path(directory)\n    if root.is_symlink() or not root.is_dir():\n        raise ValueError(\'Regular result directory required\')\n\n    def bounded(name, limit=20_000_000):\n        p = root / name\n        if p.is_symlink() or not p.is_file() or p.stat().st_size > limit:\n            raise ValueError(\'Missing, unsafe or oversized result file: \' + name)\n        data = p.read_bytes()\n        if len(data) > limit:\n            raise ValueError(\'Result file grew beyond read budget\')\n        return data\n\n    manifest_bytes = bounded(\'manifest.json\', 65_536)\n    manifest = json.loads(manifest_bytes)\n    if not isinstance(manifest, dict) or set(manifest.get(\'outputs\', {})) != EXPECTED:\n        raise ValueError(\'Exactly the nine native result outputs are required\')\n    if manifest.get(\'status\') != \'EXPERIMENTAL_NOT_PHOTO_ACCEPTED\':\n        raise ValueError(\'Unexpected acceptance/status claim in research result\')\n    if manifest.get(\'inputs\', {}).get(\'native_shape\') != list(source_rgb.shape):\n        raise ValueError(\'Manifest native shape differs from source\')\n    buffers = {}\n    for name in sorted(EXPECTED):\n        data = bounded(name)\n        entry = manifest[\'outputs\'][name]\n        if not isinstance(entry, dict) or entry.get(\'bytes\') != len(data) or entry.get(\'sha256\') != hashlib.sha256(data).hexdigest():\n            raise ValueError(\'Native result hash or size mismatch: \' + name)\n        buffers[name] = data\n    # Decode only the verified bytes, so later on-disk changes cannot switch input.\n    from io import BytesIO\n    def image(name, mode):\n        with Image.open(BytesIO(buffers[name])) as im:\n            if im.mode != mode or im.size != (source_rgb.shape[1], source_rgb.shape[0]) or getattr(im, \'n_frames\', 1) != 1:\n                raise ValueError(\'Native result image mode/shape changed: \' + name)\n            if im.getexif().get(274, 1) != 1 or im.info.get(\'icc_profile\'):\n                raise ValueError(\'Result orientation/profile transform is not permitted\')\n            return np.array(im)\n    src = image(\'source-native.png\', \'RGB\')\n    rgba = image(\'component-rgba.png\', \'RGBA\')\n    alpha = image(\'alpha.png\', \'L\')\n    binary = image(\'prior-support-mask.png\', \'L\')\n    trimap = image(\'derived-trimap.png\', \'L\')\n    if not np.array_equal(src, source_rgb) or not np.array_equal(rgba[:, :, :3], source_rgb):\n        raise ValueError(\'Result belongs to different or repainted source RGB\')\n    if not np.array_equal(rgba[:, :, 3], alpha):\n        raise ValueError(\'RGBA alpha differs from standalone alpha\')\n    if not np.array_equal(binary, np.where(initial_alpha >= 128, 255, 0).astype(np.uint8)):\n        raise ValueError(\'Result support mask differs from supplied initial alpha threshold\')\n    if not np.array_equal(derive_trimap(binary, manifest.get(\'radius\')), trimap):\n        raise ValueError(\'Derived trimap replay differs\')\n    raw = np.load(BytesIO(buffers[\'raw-alpha.npy\']), allow_pickle=False, max_header_size=1024)\n    enforced = np.load(BytesIO(buffers[\'enforced-alpha.npy\']), allow_pickle=False, max_header_size=1024)\n    expected_enforced, expected_alpha = checked_alpha(raw, trimap)\n    if not np.array_equal(expected_enforced, enforced) or not np.array_equal(expected_alpha, alpha):\n        raise ValueError(\'Alpha enforcement replay differs\')\n    for name, background in [(\'dark.png\', 35), (\'light.png\', 242)]:\n        if not np.array_equal(composite(source_rgb, alpha, background), image(name, \'RGB\')):\n            raise ValueError(\'Composite replay differs: \' + name)\n    seeds, _ = audit_trimap(initial_alpha, trimap)\n    changes, _ = review_change(initial_alpha, alpha)\n    return alpha.copy(), dict(\n        schema=\'verified-native-result-import.v1\',\n        source_pixels_sha256=digest(source_rgb), initial_alpha_sha256=digest(initial_alpha),\n        imported_alpha_sha256=digest(alpha), manifest_sha256=hashlib.sha256(manifest_bytes).hexdigest(),\n        files_verified=len(EXPECTED), source_rgb_exact=True, replay_exact=True,\n        model_execution_provenance=\'MANIFEST_CLAIM_NOT_AUTHENTICATED_BY_REPLAY\',\n        no_new_inference=True, seeds=seeds, changes=changes,\n        automatic_acceptance=False, physical_relighting=False, glass_optics_verified=False)\n'
# ---- embedded module: renderello/native_result_import.py ----
path = base / 'renderello/native_result_import.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Designed 2D studio candidates with native-size source-exact opaque product cores.\nProcedural illustration, not photographed scenery, recovered lighting or pose.\n"""\nfrom pathlib import Path\nimport json\nimport numpy as np\nfrom PIL import Image,ImageDraw,ImageFilter\nfrom .intent_mask import sha\n\nSTYLES={\n \'porcelain_sweep\':dict(name=\'Porcelain sweep\',top=(236,234,227),bottom=(247,247,244),accent=(217,214,203),shape=\'sweep\'),\n \'sage_arch\':dict(name=\'Sage arch\',top=(143,161,148),bottom=(201,211,194),accent=(225,228,213),shape=\'arch\'),\n \'clay_plinth\':dict(name=\'Clay plinth\',top=(199,151,127),bottom=(226,188,159),accent=(167,115,94),shape=\'plinth\'),\n \'noir_halo\':dict(name=\'Noir halo\',top=(23,28,32),bottom=(54,61,65),accent=(110,120,123),shape=\'halo\'),\n \'amber_paper\':dict(name=\'Amber paper\',top=(223,176,94),bottom=(239,204,136),accent=(183,127,56),shape=\'paper\'),\n \'linen_daylight\':dict(name=\'Linen daylight\',top=(218,208,189),bottom=(238,232,216),accent=(167,154,129),shape=\'window\'),\n \'cobalt_aperture\':dict(name=\'Cobalt aperture\',top=(31,58,106),bottom=(61,96,145),accent=(151,182,203),shape=\'aperture\'),\n \'blush_steps\':dict(name=\'Blush steps\',top=(219,181,177),bottom=(239,211,204),accent=(185,138,133),shape=\'steps\'),\n}\n\n\ndef background(style,size=(1024,1024),seed=0,baseline=860):\n    if style not in STYLES:raise ValueError(\'unknown studio style\')\n    w,h=size\n    if type(w)!=int or type(h)!=int or not (512<=w<=2048 and 512<=h<=2048):raise ValueError(\'canvas dimensions must be512-2048\')\n    spec=STYLES[style];y,x=np.mgrid[0:h,0:w];t=y/max(1,h-1);top=np.array(spec[\'top\']);bottom=np.array(spec[\'bottom\']);a=top[None,None,:]*(1-t[:,:,None])+bottom[None,None,:]*t[:,:,None]\n    # Gentle broad illumination and fixed-seed subpixel paper grain only in scenery.\n    light=np.exp(-(((x/w-.35)/.6)**2+((y/h-.35)/.7)**2));a+=light[:,:,None]*7\n    a+=np.random.default_rng(seed).normal(0,.55,(h,w,1));im=Image.fromarray(np.clip(a,0,255).astype(\'uint8\'));d=ImageDraw.Draw(im);c=spec[\'accent\'];shape=spec[\'shape\'];sx=w/1024;sy=h/1024\n    def box(v):return tuple(round(n*(sx if i%2==0 else sy)) for i,n in enumerate(v))\n    if shape==\'sweep\':\n        # No hard horizon; a wide soft neutral floor roll.\n        layer=Image.new(\'RGBA\',(w,h));ld=ImageDraw.Draw(layer);ld.ellipse(box((-400,750,1500,1450)),fill=(*c,90));layer=layer.filter(ImageFilter.GaussianBlur(80*sx));im=Image.alpha_composite(im.convert(\'RGBA\'),layer).convert(\'RGB\')\n    elif shape==\'arch\':\n        d.rounded_rectangle(box((145,110,880,1250)),radius=round(360*sx),fill=c);d=ImageDraw.Draw(im);d.rectangle(box((145,500,880,1250)),fill=c)\n    elif shape==\'plinth\':\n        d.polygon(box((230,860,760,860,960,1150,80,1150)),fill=c);d.polygon(box((230,860,760,860,790,885,205,885)),fill=(231,190,162))\n    elif shape==\'halo\':\n        arr=np.array(im).astype(float);v=np.exp(-(((x/w-.5)/.30)**2+((y/h-.47)/.34)**2));arr+=v[:,:,None]*40;im=Image.fromarray(np.clip(arr,0,255).astype(\'uint8\'))\n    elif shape==\'paper\':\n        d.polygon(box((0,260,1024,40,1024,700,0,920)),fill=(236,194,117));layer=Image.new(\'RGBA\',(w,h));ld=ImageDraw.Draw(layer);ld.polygon(box((0,260,1024,40,1024,85,0,305)),fill=(*c,50));im=Image.alpha_composite(im.convert(\'RGBA\'),layer.filter(ImageFilter.GaussianBlur(12*sx))).convert(\'RGB\')\n    elif shape==\'window\':\n        layer=Image.new(\'RGBA\',(w,h));ld=ImageDraw.Draw(layer)\n        for shift in [0,270,540]:ld.polygon(box((-250+shift,0,-160+shift,0,500+shift,1024,410+shift,1024)),fill=(*c,45))\n        layer=layer.filter(ImageFilter.GaussianBlur(16*sx));im=Image.alpha_composite(im.convert(\'RGBA\'),layer).convert(\'RGB\')\n    elif shape==\'aperture\':\n        d.rounded_rectangle(box((180,120,845,970)),radius=round(300*sx),fill=c);d.rounded_rectangle(box((215,155,810,970)),radius=round(265*sx),fill=(90,129,167))\n    elif shape==\'steps\':\n        d.rectangle(box((80,900,944,1024)),fill=c);d.rectangle(box((180,baseline,844,900)),fill=(226,188,182));d.rectangle(box((180,baseline,844,baseline+10)),fill=(242,216,206))\n    return im\n\n\ndef load_product(bundle):\n    m=json.loads((bundle/\'matting-manifest.json\').read_text())\n    if m.get(\'source_rgb_unchanged\') is not True or m.get(\'edge_rgb_estimated\') is not False:raise ValueError(\'native studio requires RGB-unchanged matting bundle, not estimated edge colors\')\n    for f in (\'source.png\',\'refined-alpha.png\',\'refined-cutout.png\'):\n        if sha(bundle/f)!=m.get(\'output_sha256\',{}).get(f):raise ValueError(\'matting bundle output hash mismatch: \'+f)\n    with Image.open(bundle/\'source.png\') as im:rgb=np.array(im.convert(\'RGB\'))\n    with Image.open(bundle/\'refined-cutout.png\') as im:\n        if im.mode!=\'RGBA\':raise ValueError(\'RGBA cutout required\')\n        rgba=np.array(im)\n    with Image.open(bundle/\'refined-alpha.png\') as im:\n        if im.mode!=\'L\':raise ValueError(\'L alpha required\')\n        alpha=np.array(im)\n    if rgba.shape[:2]!=rgb.shape[:2] or not np.array_equal(rgba[:,:,:3],rgb) or not np.array_equal(rgba[:,:,3],alpha):raise ValueError(\'cutout RGB/alpha inconsistent\')\n    if not np.any(alpha==255):raise ValueError(\'opaque product core required\')\n    yy,xx=np.where(alpha>0)\n    if not len(xx):raise ValueError(\'empty alpha\')\n    # The one-pixel border must be background. This is a mechanical warning\n    # boundary, not a proof the source framing is complete.\n    if np.any(alpha[0]) or np.any(alpha[-1]) or np.any(alpha[:,0]) or np.any(alpha[:,-1]):raise ValueError(\'mask touches source frame; complete source needs review before native studio\')\n    return rgba,(int(xx.min()),int(yy.min()),int(xx.max()+1),int(yy.max()+1)),m\n\n\ndef compose_native(bundle:Path,style,out:Path,*,source_pose,material,framing,regions_path=None):\n    if out.exists():raise ValueError(\'output exists; preserve earlier candidates\')\n    if source_pose not in (\'front\',\'elevated\'):raise ValueError(\'front or elevated studio source declaration required; no pose change\')\n    if material!=\'opaque\' or framing!=\'complete\':raise ValueError(\'opaque complete source required; no transparent paste or uncropping\')\n    rgba,bbox,source_meta=load_product(bundle);x1,y1,x2,y2=bbox;crop=rgba[y1:y2,x1:x2];ph,pw=crop.shape[:2]\n    if ph>760 or pw>760:raise ValueError(\'native product exceeds760px; no silent resizing. Use a smaller source or larger authored workflow explicitly\')\n    if (style==\'clay_plinth\' and pw>450) or (style==\'blush_steps\' and pw>580):raise ValueError(\'native product too wide for this designed platform; choose a sweep or halo without a narrow support\')\n    left=(1024-pw)//2\n    baseline=875 if style==\'clay_plinth\' else (865 if style==\'blush_steps\' else 860)\n    support_bottom=int(np.where(crop[:,:,3]>127)[0].max())+1\n    top=baseline-support_bottom\n    bg=background(style);alpha=Image.fromarray(crop[:,:,3]);sh=Image.new(\'L\',(1024,1024));sd=ImageDraw.Draw(sh)\n    sd.ellipse((left-10,baseline-5,left+pw+25,baseline+9),fill=70)\n    sh=sh.filter(ImageFilter.GaussianBlur(10));scene=Image.composite(Image.new(\'RGB\',bg.size,(16,17,18)),bg,sh)\n    # One source-native translation, never scale, rotate, warp or relight product.\n    scene.paste(Image.fromarray(crop[:,:,:3]),(left,top),alpha);result=np.array(scene);placed=result[top:top+ph,left:left+pw];core=crop[:,:,3]==255\n    error=int(np.abs(placed[core].astype(\'int16\')-crop[:,:,:3][core].astype(\'int16\')).max())\n    if error:raise ValueError(\'native opaque core changed unexpectedly\')\n    out.mkdir(parents=True);bg.save(out/\'background.png\');sh.save(out/\'shadow-alpha.png\');scene.save(out/\'candidate.png\');Image.fromarray(crop).save(out/\'native-product-crop.png\')\n    report=dict(style=style,name=STYLES[style][\'name\'],scene_kind=\'designed_2D_procedural_studio_candidate_NOT_photographed_or_AI_generated\',source_sha256=source_meta[\'source_sha256\'],matting_manifest_sha256=sha(bundle/\'matting-manifest.json\'),source_crop_box=list(bbox),placement=[left,top],support_alpha_threshold=127,support_baseline=baseline,source_to_canvas_translation=[left-x1,top-y1],scale=1,rotation=0,opaque_core_pixels=int(core.sum()),opaque_core_rgb_max_error=error,source_pose=source_pose,material=material,framing=framing,quality=\'NOT_EVALUATED\',fine_print=\'SOURCE_NATIVE_CORE_RETENTION_NOT_INDEPENDENT_TRANSCRIPTION\',warnings=[\'Designed candidate, not polished-template acceptance or a recovered physical scene.\',\'Source lighting is not changed; dark/colored scenes may have lighting mismatch.\',\'Only alpha255pixels remain identical in the final RGB scene; edge pixels mix with the new background.\',\'No upscaling, no camera rotation, no glass optics, no crop completion.\',\'The soft floor shadow is a2Ddesign approximation, not recovered geometry.\'])\n    if regions_path:\n        report[\'native_region_checks\']=check_native_regions(rgba,result,[left-x1,top-y1],json.loads(regions_path.read_text()),source_meta[\'source_sha256\'])\n        report[\'regions_sha256\']=sha(regions_path)\n    report[\'output_sha256\']={p.name:sha(p) for p in out.iterdir() if p.is_file()};(out/\'native-studio-manifest.json\').write_text(json.dumps(report,indent=2));return report\n\n\ndef check_native_regions(rgba,scene,translation,regions,source_sha256):\n    """Exact final-scene pixel check only for declared fully opaque source regions."""\n    # Share the same rigor for shape/name/bounds validation without re-encoding\n    # source bytes or pretending the PNG hash equals the original photo hash.\n    h,w=rgba.shape[:2]\n    if regions.get(\'source_sha256\')!=source_sha256 or regions.get(\'image_size\')!=[w,h]:raise ValueError(\'region source binding mismatch\')\n    items=regions.get(\'regions\')\n    if not isinstance(items,list) or not 1<=len(items)<=32:raise ValueError(\'one to32 regions required\')\n    dx,dy=translation;names=set();rows=[]\n    for item in items:\n        name=item.get(\'name\');b=item.get(\'box\')\n        if not isinstance(name,str) or not name.strip() or name in names:raise ValueError(\'unique nonempty region names required\')\n        names.add(name)\n        if not isinstance(b,list) or len(b)!=4 or any(type(x)!=int for x in b) or not (0<=b[0]<b[2]<=w and 0<=b[1]<b[3]<=h):raise ValueError(\'invalid source region box\')\n        x1,y1,x2,y2=b\n        if not (0<=x1+dx<x2+dx<=scene.shape[1] and 0<=y1+dy<y2+dy<=scene.shape[0]):raise ValueError(\'region outside scene\')\n        src=rgba[y1:y2,x1:x2,:3];dest=scene[y1+dy:y2+dy,x1+dx:x2+dx];a=rgba[y1:y2,x1:x2,3];err=np.abs(src.astype(\'int16\')-dest.astype(\'int16\'))\n        rows.append(dict(name=name,source_box=b,scene_box=[x1+dx,y1+dy,x2+dx,y2+dy],pixels=int(a.size),nonopaque_source_pixels=int((a!=255).sum()),final_rgb_changed_pixels=int(np.any(err,2).sum()),final_rgb_max_error=int(err.max()),source_native_retained=not bool(np.any(a!=255) or np.any(err))))\n    return dict(all_source_native_regions_retained=all(r[\'source_native_retained\'] for r in rows),regions=rows,transcription=\'NOT_PERFORMED\',quality=\'NOT_EVALUATED\')\n'
# ---- embedded module: renderello/native_studio.py ----
path = base / 'renderello/native_studio.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""U2Net-p ONNX learned matting - the free-host learned cutout backend.\n\nWhy this exists: full BiRefNet needs torch + a 444 MB fp16 checkpoint (Colab)\nand the BiRefNet-tiny ONNX conversion is 224 MB - neither fits a 512 MB free\nRender instance. U2Net-p is a 4.6 MB original-publisher saliency model whose\nONNX conversion runs in ~0.3 s on CPU in well under 200 MB extra RAM. Quality\nsits between OpenCV grabcut and BiRefNet: smooth learned edges, weaker on\nfine/translucent detail than BiRefNet. The backend name is recorded honestly\nin manifests and shown in the UI; never presented as BiRefNet.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport os\nimport urllib.request\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image\n\nMODEL_URL = ("https://github.com/danielgatis/rembg/releases/download/"\n             "v0.0.0/u2netp.onnx")\nMODEL_SHA256 = "309c8469258dda742793dce0ebea8e6dd393174f89934733ecc8b14c76f4ddd8"\n_INPUT = 320\n_session = None\n\n\ndef _model_path() -> Path:\n    d = Path(os.environ.get("RENDERELLO_MODEL_DIR", "/tmp/renderello-models"))\n    d.mkdir(parents=True, exist_ok=True)\n    p = d / "u2netp.onnx"\n    if not p.is_file():\n        tmp = p.with_suffix(".download")\n        urllib.request.urlretrieve(MODEL_URL, tmp)\n        if hashlib.sha256(tmp.read_bytes()).hexdigest() != MODEL_SHA256:\n            tmp.unlink()\n            raise RuntimeError("u2netp.onnx sha256 mismatch; refusing")\n        tmp.rename(p)\n    if hashlib.sha256(p.read_bytes()).hexdigest() != MODEL_SHA256:\n        raise RuntimeError("cached u2netp.onnx sha256 mismatch; refusing")\n    return p\n\n\ndef _get_session():\n    global _session\n    if _session is None:\n        import onnxruntime as ort\n        opts = ort.SessionOptions()\n        opts.enable_cpu_mem_arena = False\n        opts.intra_op_num_threads = 2\n        opts.inter_op_num_threads = 1\n        _session = ort.InferenceSession(str(_model_path()), sess_options=opts,\n                                        providers=["CPUExecutionProvider"])\n    return _session\n\n\ndef u2netp_onnx_matte(img_rgb: np.ndarray) -> np.ndarray:\n    """Return a uint8 matte for an HxWx3 uint8 RGB array."""\n    h, w = img_rgb.shape[:2]\n    session = _get_session()\n    im = Image.fromarray(img_rgb).resize((_INPUT, _INPUT), Image.Resampling.BILINEAR)\n    x = np.asarray(im).astype(np.float32)\n    x /= max(float(x.max()), 1.0)\n    x = (x - np.array([.485, .456, .406], np.float32)) / np.array([.229, .224, .225], np.float32)\n    raw = session.run(None, {session.get_inputs()[0].name: x.transpose(2, 0, 1)[None]})[0][0, 0]\n    if not np.isfinite(raw).all():\n        raise ValueError("non-finite u2netp output")\n    lo, hi = float(raw.min()), float(raw.max())\n    if hi - lo < 1e-8:\n        raise ValueError("constant saliency; no reliable foreground")\n    alpha = (raw - lo) / (hi - lo) * 255.0\n    # Saliency outputs a soft plateau, not a binary matte: a 0.98-alpha interior\n    # would blend the whole product with the new backdrop (uniform color\n    # shift), and faint nonzero background speckle pollutes the bbox. Stretch\n    # so the interior saturates to full opacity and the floor to zero, keeping\n    # only a narrow soft band at edges.\n    alpha = np.clip((alpha - 32.0) * (255.0 / 192.0), 0, 255).astype(np.uint8)\n    return np.asarray(Image.fromarray(alpha).resize((w, h), Image.Resampling.BILINEAR))\n'
# ---- embedded module: renderello/onnx_matte.py ----
path = base / 'renderello/onnx_matte.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Explicit source-bound opaque recovery from an inspected semantic silhouette.\nNot automatic acceptance, not a union of guesses, not transparent-object matting.\n"""\nfrom pathlib import Path\nimport json\nimport numpy as np\nfrom PIL import Image\nfrom .intent_mask import sha,load_bundle\nfrom .prompted_mask import validate_prompts,MODEL_SHA256\nfrom .matting import guarded_file\n\n\ndef verify_reviewed(source,reviewed,expected_alpha_sha256):\n    with Image.open(source) as im:\n        if im.width*im.height>1_000_000:raise ValueError(\'opaque recovery capped at one million pixels; no resize\')\n        pixels=np.array(im.convert(\'RGB\'));size=im.size\n    manifest,alpha=load_bundle(reviewed,\'prompted-mask-manifest.json\',sha(source),size)\n    if sha(reviewed/\'estimated-alpha.png\')!=expected_alpha_sha256:raise ValueError(\'reviewed silhouette hash mismatch\')\n    if manifest.get(\'model_sha256\')!=MODEL_SHA256:raise ValueError(\'unsupported reviewed model\')\n    pp=reviewed/\'reviewed-prompts.json\'\n    if sha(pp)!=manifest.get(\'prompt_sha256\'):raise ValueError(\'prompt hash mismatch\')\n    objects=validate_prompts(json.loads(pp.read_text()),source,*size)\n    if manifest.get(\'prompt_origin\')!=\'reviewed_source_geometry\' or manifest.get(\'objects\')!=[o[\'name\'] for o in objects]:raise ValueError(\'reviewed provenance mismatch\')\n    if any(1 not in o[\'labels\'] for o in objects):raise ValueError(\'explicit positive points required on every product\')\n    if not np.isin(alpha,[0,255]).all() or not np.any(alpha==0):raise ValueError(\'reviewed silhouette must be binary and include background\')\n    with Image.open(reviewed/\'source.png\') as im:\n        if not np.array_equal(np.array(im.convert(\'RGB\')),pixels):raise ValueError(\'reviewed source RGB mismatch\')\n    with Image.open(reviewed/\'cutout.png\') as im:\n        rgba=np.array(im)\n        if im.mode!=\'RGBA\' or not np.array_equal(rgba[:,:,:3],pixels) or not np.array_equal(rgba[:,:,3],alpha):raise ValueError(\'reviewed cutout mismatch\')\n    return manifest\n\n\ndef recover_file(source:Path,reviewed:Path,out:Path,*,reviewed_alpha_sha256,material,framing,confirm_reviewed_silhouette=False,method=\'cf\',radius=3):\n    if out.exists():raise ValueError(\'output exists; preserve attempts\')\n    if material!=\'opaque\':raise ValueError(\'opaque-only recovery; transparent or unknown material requires a different workflow\')\n    if framing!=\'complete\':raise ValueError(\'complete source required; this cannot recover cropped parts\')\n    if not confirm_reviewed_silhouette:raise ValueError(\'explicit inspection of this exact silhouette required\')\n    if method not in (\'cf\',\'lkm\'):raise ValueError(\'opaque recovery requires cf or lkm\')\n    verify_reviewed(source,reviewed,reviewed_alpha_sha256)\n    # Different trust decision from min-alpha gate: use reviewed silhouette as\n    # initial semantic support, never lock the known-bad automatic alpha as BG.\n    result=guarded_file(source,reviewed/\'estimated-alpha.png\',out,method=method,radius=radius,edge_colors=False)\n    result.update(route=\'explicit_reviewed_opaque_recovery\',initial_support=\'inspected_HQ_SAM_silhouette_NOT_automatic_alpha\',reviewed_manifest_sha256=sha(reviewed/\'prompted-mask-manifest.json\'),reviewed_alpha_sha256=reviewed_alpha_sha256,material_declaration=material,framing_declaration=framing,silhouette_confirmation=\'explicit_caller_confirmation_not_proof_of_human_review\',quality=\'NOT_EVALUATED\',fine_print=\'NOT_INDEPENDENTLY_VERIFIED\',warnings=result[\'warnings\']+[\'This route can restore source pixels missing from an automatic saliency mask only if the supplied reviewed silhouette includes them.\',\'The trimap is still heuristic; it can lock incorrect foreground and background. Inspect final edges on all backgrounds.\',\'Opaque material and complete framing are caller declarations, not inferred facts. No glass transmission or uncropping.\'])\n    (out/\'opaque-recovery-manifest.json\').write_text(json.dumps(result,indent=2));return result\n'
# ---- embedded module: renderello/opaque_recovery.py ----
path = base / 'renderello/opaque_recovery.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Pipeline orchestration. One run = every requested route x every variant seed,\nwith artifacts and a manifest written for audit. The full-run shape mirrors the\nproblem statement flow: source -> (cutout) -> candidates -> QA -> report.\n\nRoute 2 is always computable (CPU). Routes 1/3 need an editor backend; the CLI\nsupplies stub (tests), qwen (GPU) or magichour (mock/live). A backend raising on\na missing GPU fails only its own variants - the deterministic route still\ndelivers, and the manifest records the failure verbatim.\n"""\nfrom __future__ import annotations\n\nimport json\nimport hashlib\nimport time\nimport tempfile\nfrom dataclasses import asdict\nfrom pathlib import Path\n\nfrom PIL import Image\n\nfrom . import compose, prompts, qa\nfrom .fidelity_review import review_requirement\nfrom .backends.base import EditorBackend\nfrom .config import RunConfig\nfrom .execution_kind import describe_execution\nfrom .cutout import cutout, supplied_mask_cutout\nfrom .relight import relight\nfrom .source_provenance import identify_source\nfrom .schemas import (EditRequest, QAReport, Route, RunResult, VariantResult)\n\n\ndef _run_dir(cfg: RunConfig, request: EditRequest) -> Path:\n    stamp = time.strftime("%Y%m%d-%H%M%S")\n    # Every attempt needs its own artifacts, even with the same timestamp.\n    # Atomic creation also protects concurrent calls from reusing stale output.\n    cfg.out_root.mkdir(parents=True, exist_ok=True)\n    return Path(tempfile.mkdtemp(\n        prefix=f"{request.source_path.stem}_{request.template.key}_{stamp}_",\n        dir=cfg.out_root))\n\n\ndef run_pipeline(request: EditRequest, routes: list[Route], cfg: RunConfig,\n                 editor: EditorBackend | None = None,\n                 relight_enabled: bool = True, scene_fn=None,\n                 snap_frame_cut: bool = True, glass_mode: bool = False,\n                 scene_finish: bool = False, source_contract=None,\n                 require_complete: bool = False,\n                 allow_colour_conversion: bool = False, use_embedded_alpha: bool = False,\n                 scene_accept=None, scene_max_attempts: int = 1) -> RunResult:\n    prompts.validate_instruction(request.custom_instruction)\n    from .pipeline_intake import load_generation_image\n    if use_embedded_alpha and request.mask_path:\n        raise ValueError("choose embedded alpha or separate mask, not both")\n    if use_embedded_alpha and any(r not in (Route.CUTOUT_COMPOSE, Route.SCENE_COMPOSE) for r in routes):\n        raise ValueError("embedded alpha is for source-product composition, not product regeneration")\n    decoded = load_generation_image(request.source_path, allow_colour_conversion=allow_colour_conversion, use_embedded_alpha=use_embedded_alpha)\n    source, intake_record = decoded[:2]\n    embedded_alpha = decoded[2] if use_embedded_alpha else None\n    if request.mask_path and intake_record["orientation"] != 1:\n        raise ValueError("EXIF-oriented source with a separate mask requires prior normalized source/mask pairing; no automatic mask rotation")\n    source_provenance = identify_source(request.source_path)\n    ref_rgb = None\n    ref_pil = None\n    reference_intake = None\n    if request.reference_path:\n        import numpy as _np\n        ref_pil, reference_intake = load_generation_image(request.reference_path, allow_colour_conversion=allow_colour_conversion)\n        ref_rgb = _np.array(ref_pil)\n    run_dir = _run_dir(cfg, request)\n    if intake_record["transformations"]:\n        source.save(run_dir / "normalized-source.png")\n    if reference_intake and reference_intake["transformations"]:\n        ref_pil.save(run_dir / "normalized-reference.png")\n    aspect = request.effective_aspect\n\n    # Cutout once per run: Routes 2/3 consume it, and QA uses its matte+bbox.\n    # A cutout failure must degrade the run to recorded failures, never crash\n    # the whole batch (benchmark honesty).\n    inter_dir = run_dir / "intermediates" if cfg.save_intermediates else run_dir\n    effective_mask_path = request.mask_path\n    if embedded_alpha is not None:\n        effective_mask_path = run_dir / "embedded-source-alpha.png"\n        embedded_alpha.save(effective_mask_path)\n        embedded_alpha.close()\n    variants: list[VariantResult] = []\n    product_transforms = {}\n    co = None\n    matte_np = None\n    try:\n        co = (supplied_mask_cutout(source, effective_mask_path, inter_dir)\n              if effective_mask_path is not None else\n              cutout(source, inter_dir, stem="product", backend="auto",\n                     device=cfg.device.device))\n        import numpy as np\n        matte_np = np.array(Image.open(co.matte_path).convert("L"))\n    except Exception as e:\n        for route in routes:\n            for seed in cfg.variant_seeds():\n                report = QAReport(\n                    source=str(request.source_path),\n                    output=str(run_dir / f"{route.value}_seed{seed}.png"),\n                    route=route.value, template=request.template.key, seed=seed,\n                    critical_errors=[f"cutout failed: {type(e).__name__}: {e}"],\n                    score_fidelity=None)\n                variants.append(VariantResult(route, seed,\n                                              report.output and Path(report.output),\n                                              report, success=False))\n        manifest = run_dir / "manifest.json"\n        manifest.write_text(json.dumps({\n            "source": str(request.source_path),\n        "source_provenance": source_provenance,\n        "image_intake": intake_record,\n        "reference_intake": reference_intake,\n            "template": asdict(request.template),\n            "cutout_error": str(e),\n            "variants": [{"route": v.route.value, "seed": v.seed,\n                          "output": str(v.output_path), "success": False,\n                          "backend": "cutout_failed", "provider_evidence": None,\n                          "provider_call_status": "NOT_CALLED_CUTOUT_PREFLIGHT_FAILED",\n                          "qa": json.loads(v.qa.to_json())} for v in variants],\n        }, indent=2, default=str))\n        return RunResult(request=request, variants=variants, run_dir=run_dir,\n                         manifest_path=manifest)\n\n    mask_provenance = "embedded_source_alpha" if use_embedded_alpha else "supplied_mask" if request.mask_path else None\n    if request.mask_path:\n        # Sidecar is artifact provenance only, never a quality/permission grant.\n        sidecar = request.mask_path.parent / "mask-manifest.json"\n        if sidecar.is_file():\n            try:\n                mp = json.loads(sidecar.read_text())\n                if (mp.get("backend") in ("u2netp_experimental", "u2net_experimental", "seeded_grabcut_assisted")\n                        and mp.get("alpha_sha256") == hashlib.sha256(request.mask_path.read_bytes()).hexdigest()\n                        and mp.get("source_sha256") == hashlib.sha256(request.source_path.read_bytes()).hexdigest()):\n                    mask_provenance = "estimated_" + mp["backend"]\n            except (ValueError, OSError):\n                pass\n    from .source_contract import source_mask_preflight\n    import numpy as np\n    source_check = source_mask_preflight(np.array(source), matte_np, co.bbox,\n        source_contract, overhead_scene=(request.template.key == \'flat_lay\' or\n            bool(getattr(scene_fn, \'flatlay_surface\', False))),\n        glass_mode=glass_mode, require_complete=require_complete)\n    matched_pose = getattr(scene_fn,\'camera_pose\',None)\n    if matched_pose and source_contract is not None and source_contract.pose not in (\'unknown\',matched_pose):\n        source_check[\'blockers\'].append(\'Declared source and requested scene camera poses disagree.\')\n        source_check[\'passed\']=False\n    glass_backdrop = None\n    if glass_mode:\n        B, clean = compose.estimate_clean_backdrop(np.array(source))\n        if not clean:\n            source_check[\'blockers\'].append(\'Transmission requires a clean uniform source backdrop; refusing opaque fallback.\')\n        elif float(np.min(B)) < 128:\n            source_check[\'blockers\'].append(\'Transmission from a dark source backdrop is underdetermined; use a clean bright backdrop capture.\')\n        else:\n            glass_backdrop = B\n            source_check[\'warnings\'].append(\'Single-image transmission is approximate: highlights and refracted geometry are not recovered; no exact RGB or material fidelity claim.\')\n        source_check[\'passed\'] = not source_check[\'blockers\']\n    (inter_dir / \'source-mask-preflight.json\').write_text(json.dumps(source_check, indent=2))\n    prompt = prompts.build_prompt(request)\n    neg = prompts.build_negative_prompt()\n\n    for route in routes:\n        for seed in cfg.variant_seeds():\n            out_path = run_dir / f"{route.value}_seed{seed}.png"\n            t0 = time.monotonic()\n            editor_called = False\n            prior_attempt_id = getattr(editor, \'_attempt_id\', None)\n            try:\n                if route in (Route.CUTOUT_COMPOSE, Route.SCENE_COMPOSE) and not source_check[\'passed\']:\n                    raise ValueError(\'Source compatibility preflight: \' + \'; \'.join(source_check[\'blockers\']))\n                if route in (Route.CUTOUT_COMPOSE, Route.HYBRID) and source_contract is not None and source_contract.material == \'transparent\':\n                    raise ValueError(\'Transparent source cannot use an opaque composite route, even when glass_mode is requested; use scene_compose with validated transmission.\')\n                if route is Route.CUTOUT_COMPOSE:\n                    bg_tx = None\n                    if relight_enabled:\n                        # Grade the background only; product stays source-exact.\n                        bg_tx = lambda im: relight(im, request.template.lighting,\n                                                   prefer_gpu=False)[0]\n                    img = compose.composite(co, request.template, aspect, seed,\n                                            reflection="reflect" in request.template.scene.lower(),\n                                            bg_transform=bg_tx, reference=ref_rgb)\n                    img.save(out_path)\n                    placed = _placed_region(co, aspect, seed, request.template.key)\n                    transform = _product_transform_record(co.bbox, placed)\n                    product_transforms[f"{route.value}:{seed}"] = transform\n                    report = qa.evaluate(\n                        source, out_path, route.value, request.template.key, seed,\n                        src_crop_box=co.bbox, out_region=_placed_region(co, aspect, seed, request.template.key),\n                        matte=matte_np,\n                        notes=("route2: source RGB composited, no model repaint; " +\n                               ("product resampled to template framing, native pixel-exactness NOT claimed"\n                                if transform["product_resampled"] else\n                                "native product dimensions retained; opaque RGB subject to output verification")))\n                    # Pixel comparisons can\'t see a bad matte (a kept slab is\n                    # source-exact); score the matte itself.\n                    report.critical_errors.extend(qa.matte_sanity(matte_np, co.bbox, flag_frame_cut=True))\n                    report.score_fidelity = qa.fidelity_score(report)\n                    variants.append(VariantResult(route, seed, out_path, report,\n                                                  {"cutout": co.rgba_path},\n                                                  backend=co.backend,\n                                                  seconds=round(time.monotonic() - t0, 2)))\n                elif route is Route.SCENE_COMPOSE:\n                    if not source_check[\'passed\']:\n                        raise ValueError(\'Source compatibility preflight: \' + \'; \'.join(source_check[\'blockers\']))\n                    # Route 2b: scene_fn(gray_canvas_rgb, prompt) -> rgb (IC-Light on GPU).\n                    # Product pixels are pasted back source-exact. snap_frame_cut keeps\n                    # frame-cut products from showing hard straight cuts (cover-fit or snap).\n                    eff_scene_fn = scene_fn\n                    if eff_scene_fn is None and ref_rgb is not None:\n                        # Reference backdrop instead of a generated scene: cover-fit to\n                        # whatever padded canvas aspect_safe_scene_call hands us.\n                        eff_scene_fn = lambda g, pr: compose.cover_to_canvas(ref_rgb, (g.shape[1], g.shape[0]))\n                    if eff_scene_fn is None:\n                        raise RuntimeError("scene_compose needs a scene_fn (GPU relight) or a reference image")\n                    gb, gnote = glass_backdrop, ""\n                    img, diag = compose.composite_with_scene(\n                        co, request.template, aspect, seed, eff_scene_fn,\n                        shadow_style=request.template.shadow if scene_finish else None,\n                        scene_auto_lift=scene_finish,\n                        snap_frame_cut=snap_frame_cut, glass_backdrop=gb,\n                        cache_dir=(inter_dir / f"scene_layers_seed{seed}") if cfg.save_intermediates else None,\n                        scene_is_aligned=(scene_fn is None and ref_rgb is not None),\n                        scene_accept=scene_accept, scene_max_attempts=scene_max_attempts)\n                    img.save(out_path)\n                    # Same-coordinate QA: compare the output region where the product was\n                    # actually placed (diag["placement"]) against the source crop, instead of\n                    # the whole canvas resized onto the source (invalid when the placed\n                    # product is scaled/moved). Cover-fit placements compare the full source.\n                    _pl = diag.get("placement")\n                    _crop_box = (0, 0, source.width, source.height) if diag.get("cover_fit") else co.bbox\n                    report = qa.evaluate(\n                        source, out_path, route.value, request.template.key, seed,\n                        src_crop_box=_crop_box, matte=matte_np, out_region=_pl,\n                        notes=("route2b REFERENCE-BACKDROP (no generative scene model); "\n                               if (scene_fn is None and ref_rgb is not None)\n                               else "route2b: scene generated around source-exact product; ")\n                              + f"ring diag {diag}; QA compares the actually placed region (same coordinates)"\n                              + (" | GLASS MODE: product pixels modulated, not bit-exact" if gb is not None else "") + gnote)\n                    # Structural matte checks (slabs, frame cut) stay on in every mode.\n                    structural_flags = qa.matte_sanity(matte_np, co.bbox, flag_frame_cut=True)\n                    if gb is None:\n                        # A claimed exact paste is not permission to erase\n                        # independently measured detail/colour-change warnings.\n                        report.critical_errors.extend(structural_flags)\n                    else:\n                        # Glass modulation deliberately changes RGB, so its\n                        # pixel-change warnings are not applicable. Structural\n                        # warnings remain; the output stays explicitly unscored.\n                        report.critical_errors = structural_flags\n                    report.notes += f" | same-coordinate QA region={_pl} paste_bit_exact={diag.get(\'paste_bit_exact\')} (pixel metrics use a resampled source baseline, not exact)"\n                    if diag.get("paste_bit_exact") is False:\n                        report.critical_errors.append("product paste NOT bit-exact at placed coordinates")\n                    if gb is not None:\n                        # Only the pixel-change metrics are inapplicable (pixels modulated by\n                        # design). Say so explicitly and mark the output for eye review.\n                        report.notes += (" | REVIEW NEEDED (glass mode): pixel-equality/colour/edge-retention "\n                                         "metrics NOT APPLICABLE; matte checks kept; eye-check glass edges")\n                    report.score_fidelity = None if gb is not None else qa.fidelity_score(report)\n                    variants.append(VariantResult(route, seed, out_path, report,\n                                                  {"cutout": co.rgba_path},\n                                                  backend=co.backend,\n                                                  seconds=round(time.monotonic() - t0, 2)))\n                elif route is Route.PURE_PROMPT:\n                    if editor is None:\n                        raise RuntimeError("pure_prompt needs an editor backend")\n                    editor_called = True\n                    img = editor.edit(source, prompt, neg, seed, aspect, reference=ref_pil)\n                    img.save(out_path)\n                    report = qa.evaluate(\n                        source, out_path, route.value, request.template.key, seed,\n                        src_crop_box=co.bbox, matte=matte_np,\n                        notes="route1: whole-image edit; alignment approximate")\n                    variants.append(VariantResult(route, seed, out_path, report,\n                                                  backend=getattr(editor, "name", "editor"),\n                                                  seconds=round(time.monotonic() - t0, 2)))\n                elif route is Route.HYBRID:\n                    if editor is None:\n                        raise RuntimeError("hybrid needs an editor backend")\n                    base_img = compose.composite(co, request.template, aspect, seed,\n                                                 reflection="reflect" in request.template.scene.lower())\n                    harm = prompts.build_harmonize_prompt(request)\n                    editor_called = True\n                    img = editor.edit(base_img, harm, neg, seed, aspect, reference=ref_pil)\n                    img.save(out_path)\n                    report = qa.evaluate(\n                        source, out_path, route.value, request.template.key, seed,\n                        src_crop_box=co.bbox, out_region=_placed_region(co, aspect, seed, request.template.key),\n                        matte=matte_np,\n                        notes="route3: harmonize over correct composite; drift flagged")\n                    report.critical_errors.extend(qa.matte_sanity(matte_np, co.bbox, flag_frame_cut=True))\n                    report.score_fidelity = qa.fidelity_score(report)\n                    variants.append(VariantResult(route, seed, out_path, report,\n                                                  {"cutout": co.rgba_path},\n                                                  backend=getattr(editor, "name", "editor"),\n                                                  seconds=round(time.monotonic() - t0, 2)))\n            except Exception as e:\n                # Failed variant is recorded, not hidden (benchmark honesty).\n                report = QAReport(source=str(request.source_path),\n                                  output=str(out_path), route=route.value,\n                                  template=request.template.key, seed=seed,\n                                  critical_errors=[f"variant failed: {type(e).__name__}: {e}"],\n                                  score_fidelity=None)\n                variants.append(VariantResult(route, seed, out_path, report,\n                                              seconds=round(time.monotonic() - t0, 2), success=False))\n\n            if editor_called and callable(getattr(editor,\'attempt_evidence\',None)):\n                evidence = editor.attempt_evidence()\n                if isinstance(evidence,dict) and evidence.get(\'attempt_id\') and evidence[\'attempt_id\'] != prior_attempt_id:\n                    evidence = dict(evidence)\n                    evidence.update(pipeline_completed=variants[-1].success,\n                        output_sha256=hashlib.sha256(out_path.read_bytes()).hexdigest() if out_path.is_file() else None,\n                        pipeline_artifact_exists=out_path.is_file(),automatic_retry=False)\n                    variants[-1].provider_evidence = evidence\n\n    for variant in variants:\n        variant.qa.notes += \' | SOURCE/MASK PREFLIGHT: \' + \'; \'.join(source_check[\'warnings\'])\n    if use_embedded_alpha:\n        for variant in variants:\n            variant.qa.notes += " | EMBEDDED SOURCE ALPHA: explicitly retained; no segmentation; mask semantics/material/publication unverified"\n    if request.mask_path is not None:\n        for variant in variants:\n            variant.qa.notes += (" | ASSISTED SEEDED GRABCUT MASK: explicit correction regions/strokes; not automatic success or original GPU alpha; inspect edges/frame cuts"\n                                 if mask_provenance == "estimated_seeded_grabcut_assisted" else\n                                 f" | ESTIMATED {mask_provenance.replace(\'estimated_\', \'\').upper()} MASK: new model estimate, not original GPU alpha; review missing objects/halos/transparency"\n                                 if mask_provenance and mask_provenance.startswith("estimated_") else\n                                 " | SUPPLIED MASK: explicit assisted alpha, not automatic segmentation; edges require review")\n    manifest = run_dir / "manifest.json"\n    manifest.write_text(json.dumps({\n        "source": str(request.source_path),\n        "source_provenance": source_provenance,\n        "image_intake": intake_record,\n        "reference_intake": reference_intake,\n        "template": asdict(request.template),\n        "custom_instruction": request.custom_instruction,\n        "source_mask_preflight": source_check,\n        "mask_path": str(effective_mask_path) if effective_mask_path else None,\n        "mask_provenance": mask_provenance,\n        "mask_sha256": hashlib.sha256(effective_mask_path.read_bytes()).hexdigest() if effective_mask_path else None,\n        "aspect": aspect.value,\n        "prompt": prompt if any(r is not Route.CUTOUT_COMPOSE for r in routes) else None,\n        "cutout": asdict(co) | {"rgba_path": str(co.rgba_path),\n                                "matte_path": str(co.matte_path)},\n        "variants": [{\n            "route": v.route.value, "seed": v.seed, "output": str(v.output_path),\n            "backend": v.backend, "seconds": v.seconds,\n            "credits_estimated": v.credits_estimated,\n            "credits_estimated_semantics": "legacy_projection_not_actual_charge; use provider_evidence or unknown",\n            "provider_evidence": v.provider_evidence,\n            "success": v.success and v.output_path.is_file(),\n            "artifact_exists": v.output_path.is_file(),\n            "execution_kind": describe_execution(v.route.value, v.backend, scene_mock=getattr(scene_fn, "mock_scene", False) is True, reference=(scene_fn is None and ref_rgb is not None)),\n            "fidelity_review": review_requirement(v.route.value, v.qa.critical_errors, output_exists=v.output_path.is_file(), completed=v.success),\n            "product_transform": product_transforms.get(f"{v.route.value}:{v.seed}"),\n            "qa": json.loads(v.qa.to_json()),\n        } for v in variants],\n    }, indent=2, default=str))\n    return RunResult(request=request, variants=variants, run_dir=run_dir,\n                     manifest_path=manifest)\n\n\ndef _placed_region(co, aspect, seed: int, template_key: str) -> tuple[int, int, int, int]:\n    """Recompute where compose placed the crop for this variant seed. Uses the\n    same placement_params/fit_product math as compose.composite so QA\n    alignment and compositing can never drift apart silently."""\n    cw, ch = aspect.size\n    fill, jx, baseline = compose.placement_params(seed, template_key)\n    x, y, w, h = co.bbox\n    scale = min((ch * fill) / h, (cw * 0.9) / w, 1.0)\n    nw, nh = max(1, round(w * scale)), max(1, round(h * scale))\n    px = max(0, min((cw - nw) // 2 + int(jx * cw), cw - nw))\n    py = max(0, min(int(ch * baseline) - nh, ch - nh))\n    return (px, py, nw, nh)\n\n\ndef _product_transform_record(source_bbox, placed_region):\n    """Declared geometry only, not alpha truth or output pixel verification."""\n    x,y,w,h=source_bbox\n    px,py,nw,nh=placed_region\n    return dict(schema=\'renderello-product-transform.v1\',\n        source_bbox=list(source_bbox),output_region=list(placed_region),\n        source_crop_size=[w,h],placed_product_size=[nw,nh],\n        product_resampled=(nw,nh)!=(w,h),\n        native_rgb_exactness_claimed=False,\n        alpha_semantics=\'NOT_VERIFIED\',photographic_acceptance=False,\n        note=\'No generative RGB repaint on cutout_compose. Resampling and fractional blending can change visible RGB. Unchanged dimensions alone do not certify native RGB.\')\n'
# ---- embedded module: renderello/pipeline.py ----
path = base / 'renderello/pipeline.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Shared guarded image decode for generation. No photo-quality acceptance.\n\nColour conversion is explicit. Pixel identity after normalization is a different\nclaim from equality with an original file\'s encoded RGB values.\n"""\nimport hashlib\nfrom .image_intake import inspect_image, IntakePolicy\n\n\ndef load_generation_image(path, *, allow_colour_conversion=False, use_embedded_alpha=False):\n    result = inspect_image(path, IntakePolicy(allow_colour_conversion=allow_colour_conversion))\n    try:\n        image = result.image.convert(\'RGB\')\n        image.info.clear()\n        record = dict(result.record)\n        record[\'normalized_rgb_sha256\'] = hashlib.sha256(image.tobytes()).hexdigest()\n        record[\'normalized_rgb_size\'] = list(image.size)\n        record[\'pixel_comparison_basis\'] = \'normalized RGB derivative, not original encoded colour values\'\n        if type(use_embedded_alpha) is not bool:\n            raise ValueError(\'embedded alpha choice must be boolean\')\n        if use_embedded_alpha and result.image.mode != \'RGBA\':\n            raise ValueError(\'use embedded alpha requires a source with alpha/transparency\')\n        alpha = result.image.getchannel(\'A\') if use_embedded_alpha else None\n        record[\'embedded_alpha_used\'] = bool(use_embedded_alpha)\n        record[\'embedded_alpha_sha256\'] = hashlib.sha256(alpha.tobytes()).hexdigest() if alpha else None\n        record[\'legacy_alpha_discarded\'] = result.image.mode == \'RGBA\' and not use_embedded_alpha\n        if record[\'legacy_alpha_discarded\']:\n            record[\'warnings\'] = list(record[\'warnings\']) + [\'Legacy generation decodes opaque RGB; embedded alpha is not used as its segmentation mask. Supply a separate reviewed mask.\']\n        record[\'colour_accuracy_verified\'] = False\n        return (image, record, alpha) if use_embedded_alpha else (image, record)\n    finally:\n        result.close()\n'
# ---- embedded module: renderello/pipeline_intake.py ----
path = base / 'renderello/pipeline_intake.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Opt-in plain-backdrop CPU silhouette control. No optics or scene reconstruction."""\nfrom pathlib import Path\nimport json,hashlib\nimport cv2,numpy as np\nfrom PIL import Image\n\ndef plain_backdrop_alpha(image,*,material,threshold=24,background_points=(),protected_regions=None):\n    if material!=\'opaque\':raise ValueError(\'This route requires an explicitly opaque product; no glass optics recovery\')\n    if threshold not in (8,16,24):raise ValueError(\'Choose a tested threshold: 8,16,24\')\n    a=np.array(image.convert(\'RGB\'));h,w=a.shape[:2]\n    if h*w>16000000:raise ValueError(\'Source too large for bounded CPU route\')\n    band=max(1,min(h,w)//100);border=np.concatenate([a[:band].reshape(-1,3),a[-band:].reshape(-1,3),a[:,:band].reshape(-1,3),a[:,-band:].reshape(-1,3)]);med=np.median(border,axis=0);spread=float(np.percentile(np.linalg.norm(border-med,axis=1),90))\n    if spread>=28:raise ValueError(\'Nonuniform source border: refuse this plain-backdrop route\')\n    candidate=(np.linalg.norm(a.astype(np.float32)-med,axis=2)<=threshold).astype(\'uint8\')\n    _,lab=cv2.connectedComponents(candidate,connectivity=8)\n    ids=np.unique(np.concatenate([lab[0],lab[-1],lab[:,0],lab[:,-1]]));ids=set(int(v) for v in ids if v!=0)\n    # Explicit image-coordinate background seeds can remove enclosed source\n    # backdrop, such as a handle opening. Never infer these holes from colour.\n    if not isinstance(background_points,(list,tuple)) or len(background_points)>64:\n        raise ValueError(\'At most 64 explicit background points required\')\n    validated=[]\n    for point in background_points:\n        if not isinstance(point,(list,tuple)) or len(point)!=2 or any(type(v) is not int for v in point):\n            raise ValueError(\'Background points must be integer source coordinates\')\n        x,y=point\n        if not (0<=x<w and 0<=y<h):raise ValueError(\'Background point outside source\')\n        if not candidate[y,x]:raise ValueError(\'Background point is not in the estimated backdrop colour\')\n        ids.add(int(lab[y,x]));validated.append([x,y])\n    alpha=(~np.isin(lab,list(ids))).astype(\'uint8\')*255\n    protected_regions={} if protected_regions is None else protected_regions\n    if not isinstance(protected_regions,dict):raise ValueError(\'Protected regions must be a named mapping\')\n    for name,box in protected_regions.items():\n        if not isinstance(name,str) or not name or not isinstance(box,(list,tuple)) or len(box)!=4 or any(type(v) is not int for v in box):\n            raise ValueError(\'Named protected boxes require integer coordinates\')\n        x1,y1,x2,y2=box\n        if not (0<=x1<x2<=w and 0<=y1<y2<=h):raise ValueError(\'Protected box outside source\')\n        if np.any(alpha[y1:y2,x1:x2]!=255):raise ValueError(\'Removal crosses protected region; review mask and seeds\')\n    coverage=float(np.mean(alpha>0))\n    if not .005<coverage<.95:raise ValueError(\'Implausible coverage; manual mask review required\')\n    return Image.fromarray(alpha),{\'background_points\':validated,\'protected_regions\':protected_regions,\'seed_origin\':\'EXPLICIT_CALLER_COORDINATES_NOT_AUTOMATIC_REVIEW\',\'enclosed_background_removed_only_when_seeded\':True,\'border90_rgb_distance\':spread,\'threshold\':threshold,\'coverage\':coverage,\'limitations\':[\'Binary silhouette can retain pale fringes and old shadows.\',\'Near-background-color product parts touching the outside can be lost.\',\'No protected label regions or independent silhouette ground truth supplied.\',\'Not suitable for transparent optics or complex scene separation.\']}\n\ndef preview_plain_backdrop(source,root,*,material,template=\'minimal_tabletop\',instruction=\'\',alternatives=2,background_points=(),protected_regions=None):\n    from .colab_demo import validate_choices\n    from .compose import procedural_background\n    import shutil\n    spec=validate_choices(template,instruction,alternatives);source=Path(source);root=Path(root)\n    if root.exists():raise ValueError(\'Preserve earlier output\')\n    im=Image.open(source).convert(\'RGB\');alpha,audit=plain_backdrop_alpha(im,material=material,background_points=background_points,protected_regions=protected_regions);root.mkdir();shutil.copyfile(source,root/(\'source-original\'+source.suffix));alpha.save(root/\'experimental-alpha.png\');outputs=[];a=np.asarray(alpha);ys,xs=np.nonzero(a);cx=(xs.min()+xs.max())//2;cy=(ys.min()+ys.max())//2;canvas=(max(1024,im.width),max(1024,im.height));dx=int(canvas[0]//2-cx);dy=int(canvas[1]//2-cy)\n    if xs.min()+dx<0 or ys.min()+dy<0 or xs.max()+dx>=canvas[0] or ys.max()+dy>=canvas[1]:raise ValueError(\'Source placement clipped\')\n    src=np.asarray(im)\n    for i in range(alternatives):\n        bg=Image.fromarray(procedural_background(canvas,spec,i)).convert(\'RGB\');bg.paste(im,(dx,dy),alpha);path=root/(\'alternative-\'+str(i+1)+\'.png\');bg.save(path);saved=np.asarray(Image.open(path));assert np.array_equal(src[ys,xs],saved[ys+dy,xs+dx]);outputs.append(str(path))\n    manifest={\'kind\':\'EXPERIMENTAL_PLAIN_BACKDROP_OPAQUE_PREVIEW_NOT_ACCEPTED\',\'mask\':audit,\'opaque_rgb_changed\':0,\'kept_source_pixels\':len(xs),\'integer_translation\':[dx,dy],\'source_resampled\':False,\'outputs\':outputs,\'source_sha256\':hashlib.sha256(source.read_bytes()).hexdigest(),\'custom_instruction_recorded_not_generatively_executed\':instruction,\'paid_credits\':0};(root/\'manifest.json\').write_text(json.dumps(manifest,indent=2));return {\'kind\':manifest[\'kind\'],\'manifest\':str(root/\'manifest.json\'),\'outputs\':outputs,\'root\':str(root)}\n'
# ---- embedded module: renderello/plain_backdrop_preview.py ----
path = base / 'renderello/plain_backdrop_preview.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Explicit prepared-source bridge into CPU composition, not automatic quality approval.\n\nImmutable input-byte snapshots prevent a changed prepared file from being used after\nits hash check. Legacy entry points, publication and source-rights decisions stay\nunchanged. This command never downloads a model or selects a paid backend.\n"""\nfrom pathlib import Path\nfrom io import BytesIO\nimport hashlib,json,os,stat,tempfile\nfrom PIL import Image\nfrom .image_intake import IntakeError,IntakePolicy,_dimensions\n\n\ndef _read(path,limit):\n    path=Path(path)\n    if path.is_symlink():raise IntakeError(\'Prepared input symlink refused\')\n    fd=os.open(path,os.O_RDONLY|getattr(os,\'O_NONBLOCK\',0)|getattr(os,\'O_NOFOLLOW\',0))\n    with os.fdopen(fd,\'rb\') as stream:\n        if not stat.S_ISREG(os.fstat(stream.fileno()).st_mode):raise IntakeError(\'Prepared input must be regular\')\n        if os.fstat(stream.fileno()).st_size>limit:raise IntakeError(\'Prepared input exceeds byte budget\')\n        data=stream.read(limit+1)\n    if len(data)>limit:raise IntakeError(\'Prepared input exceeds byte budget\')\n    return data\n\n\ndef _pairs(rows):\n    result={}\n    for k,v in rows:\n        if k in result:raise IntakeError(\'Duplicate prepared manifest key\')\n        result[k]=v\n    return result\n\n\ndef load_prepared(folder):\n    """Return verified bytes, not mutable paths. No writes on validation failure."""\n    folder=Path(folder)\n    if folder.is_symlink() or not folder.is_dir():raise IntakeError(\'Prepared directory required, no symlink\')\n    if (folder/\'PREPARATION_INCOMPLETE\').exists():raise IntakeError(\'Incomplete preparation refused\')\n    raw=_read(folder/\'prepared-source.json\',128_000)\n    try:\n        record=json.loads(raw,object_pairs_hook=_pairs,parse_constant=lambda x:(_ for _ in ()).throw(IntakeError(\'Nonfinite manifest value\')))\n        size=record[\'intake\'][\'output_size\']\n        if not isinstance(size,list) or len(size)!=2 or any(type(x) is not int or x<1 for x in size):raise IntakeError(\'Invalid prepared dimensions\')\n    except (KeyError,TypeError,json.JSONDecodeError) as e:raise IntakeError(\'Invalid prepared manifest\') from e\n    files={}\n    for name,mode,key in [(\'normalized-source.png\',\'RGB\',\'source_png_sha256\'),(\'reviewed-alpha.png\',\'L\',\'alpha_png_sha256\')]:\n        data=_read(folder/name,40_000_000)\n        if hashlib.sha256(data).hexdigest()!=record.get(key):raise IntakeError(\'Prepared bytes changed: \'+name)\n        if not data.startswith(b\'\\x89PNG\\r\\n\\x1a\\n\') or len(data)<29 or data[12:16]!=b\'IHDR\' or data[24]!=8:raise IntakeError(\'Expected 8-bit PNG\')\n        with Image.open(BytesIO(data)) as im:\n            _dimensions(im,IntakePolicy())\n            if im.format!=\'PNG\' or im.mode!=mode or list(im.size)!=size or getattr(im,\'n_frames\',1)!=1:raise IntakeError(\'Prepared image mode or coordinates disagree\')\n            if im.getexif().get(274,1)!=1 or im.info.get(\'icc_profile\') or \'transparency\' in im.info:raise IntakeError(\'Hidden prepared interpretation metadata\')\n            im.load()\n            if mode==\'L\' and im.getextrema()[1]==0:raise IntakeError(\'Prepared alpha empty\')\n        files[name]=data\n    return record,files\n\n\ndef compose_prepared(folder,out_root,template=\'clean_studio\',seed=0):\n    from .templates import get_template\n    from .config import RunConfig\n    from .schemas import EditRequest,Route\n    from .pipeline import run_pipeline\n    if type(seed) is not int:raise ValueError(\'Integer seed required\')\n    selected=get_template(template)\n    record,files=load_prepared(folder)\n    out_root=Path(out_root);out_root.mkdir(parents=True,exist_ok=True)\n    job=Path(tempfile.mkdtemp(prefix=\'prepared-compose-\',dir=out_root));snap=job/\'input\';snap.mkdir()\n    try:\n        for name,data in files.items():(snap/name).write_bytes(data)\n        # Retain provenance without upgrading the source rights or review status.\n        (snap/\'prepared-source.json\').write_text(json.dumps(record,indent=2,allow_nan=False))\n        request=EditRequest(snap/\'normalized-source.png\',selected,mask_path=snap/\'reviewed-alpha.png\')\n        result=run_pipeline(request,[Route.CUTOUT_COMPOSE],RunConfig(variants=1,seed=seed,out_root=job/\'runs\'),relight_enabled=False)\n        checks=_audit_outputs(result,files,selected)\n        audit=dict(prepared_record=record,source_rights=\'NOT_UPGRADED\',semantic_mask_accuracy=\'NOT_VERIFIED\',publication_status=\'NOT_EVALUATED\',legacy_routes_changed=False,relighting=False,paid_backend=False,run_manifest=str(result.manifest_path),composition_may_resize_source=True,pixel_fidelity=\'SEE_SCOPED_NATIVE_PIXEL_CHECKS\',native_pixel_checks=checks)\n        (job/\'prepared-compose-audit.json\').write_text(json.dumps(audit,indent=2,allow_nan=False))\n        if any(check[\'status\']==\'MISMATCH\' for check in checks):\n            raise IntakeError(\'Saved composition changed fully opaque native RGB pixels\')\n        return result\n    except BaseException:\n        (job/\'FAILED\').touch();raise\n\n\ndef _audit_outputs(result,files,selected):\n    """Check saved outputs, not a resampled QA baseline or only the input hash."""\n    import numpy as np\n    from .native_pixel_audit import audit_native_pixels\n    from .pipeline import _placed_region\n    from .schemas import CutoutResult\n    manifest=json.loads(result.manifest_path.read_text())\n    c=manifest[\'cutout\']\n    co=CutoutResult(Path(c[\'rgba_path\']),Path(c[\'matte_path\']),tuple(c[\'bbox\']),c[\'backend\'],c[\'coverage\'])\n    with Image.open(BytesIO(files[\'normalized-source.png\'])) as im:source=np.array(im)\n    with Image.open(BytesIO(files[\'reviewed-alpha.png\'])) as im:alpha=np.array(im)\n    checks=[]\n    for variant in result.variants:\n        raw=variant.output_path.read_bytes()\n        with Image.open(BytesIO(raw)) as im:\n            if im.mode!=\'RGB\':raise IntakeError(\'Expected decoded RGB composition for native audit\')\n            output=np.array(im)\n        region=_placed_region(co,selected.aspect,variant.seed,selected.key)\n        check=audit_native_pixels(source,alpha,output,co.bbox,region)\n        check.update(output=str(variant.output_path),output_sha256=hashlib.sha256(raw).hexdigest(),\n                     placement_basis=\'EXPECTED_CPU_COMPOSITION_COORDINATES\')\n        checks.append(check)\n    return checks\n\n\ndef main(argv=None):\n    import argparse\n    p=argparse.ArgumentParser(description=\'Explicit CPU composition from verified prepared source/mask bytes; resizing may change pixels; no automatic quality approval\')\n    p.add_argument(\'--prepared\',required=True,type=Path);p.add_argument(\'--out\',required=True,type=Path);p.add_argument(\'--template\',default=\'clean_studio\');p.add_argument(\'--seed\',type=int,default=0)\n    a=p.parse_args(argv);r=compose_prepared(a.prepared,a.out,a.template,a.seed);print(r.manifest_path)\n\nif __name__==\'__main__\':main()\n'
# ---- embedded module: renderello/prepared_compose.py ----
path = base / 'renderello/prepared_compose.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Preset library: the 8-12 reusable photoshoot templates required by the\nproblem statement, each with a genuinely detailed, researched prompt.\n\nPrompt structure follows the six-part commercial-brief pattern documented in\ndocs/PROMPT_RESEARCH.md (product, purpose, composition, lighting,\nsurface/environment, constraints), with material-specific overlays so glass,\nmetal, fabric and packaging each get physically-correct lighting language.\n\nPresets are data, not code paths: they sit on top of templates.py (scene\nfragments) and prompts.py (fidelity guard). The studio layer assembles the\nfinal long prompt from these parts plus the owner\'s free-form instruction.\n"""\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, field\n\nfrom .schemas import Aspect\nfrom .templates import TEMPLATES, get_template\n\n# ---------------------------------------------------------------------------\n# Material overlays: physically accurate lighting/texture language per\n# material family. Appended when the owner declares (or the intake detects)\n# the product material. Grounded in the product-photography prompt research:\n# glass needs controlled highlights + refraction language, metal needs\n# grain/reflection relationship, matte needs tonal gradient, etc.\n# ---------------------------------------------------------------------------\nMATERIAL_OVERLAYS: dict[str, str] = {\n    "glass": (\n        "The product is transparent or translucent glass: render believable "\n        "refraction and glass thickness, a controlled vertical strip highlight "\n        "on one edge, subtle rim light separating the silhouette from the "\n        "backdrop, and restrained internal reflections. Never frost, recolor "\n        "or opacify the glass."),\n    "metal": (\n        "The product is metal: trace the brushed or polished grain with a long "\n        "soft key so the reflection gradient runs along the grain, keep edge "\n        "highlights crisp, and preserve the exact anodized or plated color. "\n        "No fingerprints, no warped edges."),\n    "matte_plastic": (\n        "The product is matte plastic or soft-touch material: use a broad "\n        "soft source that lays a smooth tonal gradient across the surface so "\n        "the shape stays readable; avoid specular glare and keep the texture "\n        "fine and realistic."),\n    "glossy": (\n        "The product is glossy or lacquered: manage reflections deliberately "\n        "- one clean elongated highlight per curved face, deep saturated base "\n        "color, and mirror-sharp but physically plausible surroundings "\n        "reflected at reduced contrast."),\n    "fabric": (\n        "The product is fabric, apparel or textile: preserve weave, stitching, "\n        "seams, buttons, straps and zippers exactly; light to reveal fiber "\n        "texture without flattening; natural drape only, no invented folds."),\n    "leather": (\n        "The product is leather: emphasize grain and crease authenticity with "\n        "raking soft light, keep hardware (buckles, clasps, zippers) "\n        "pixel-faithful, and preserve exact dye color and finish."),\n    "food_packaging": (\n        "The product is food or beverage packaging: keep every label element "\n        "- brand mark, variant, weight, nutrition block, barcode, MRP - "\n        "bit-exact; render the wrapper\'s real material (foil, laminate, "\n        "paperboard) and natural folds; appetizing but restrained color."),\n    "skincare": (\n        "The product is skincare or cosmetics packaging: airy highlights, "\n        "gentle gradients on pumps/caps, clean refraction if the vessel is "\n        "translucent, and perfectly legible label typography at macro "\n        "fidelity."),\n    "beverage_can": (\n        "The product is a beverage can: manage the cylindrical highlight as "\n        "one continuous strip, preserve condensation only if the scene calls "\n        "for it, keep the printed artwork - every letter and graphic - "\n        "exactly as on the source."),\n    "ceramic": (\n        "The product is ceramic or porcelain: soft broad key with gentle "\n        "specular bloom on the glaze, exact glaze color, no surface "\n        "repainting."),\n    "electronics": (\n        "The product is electronics: keep screens, ports, buttons, LEDs and "\n        "printed markings exact; screens show exactly what the source shows; "\n        "brushed or anodized surfaces keep their real finish."),\n    "jewelry": (\n        "The product is jewelry or a watch: macro-grade edge fidelity, "\n        "controlled specular fire on facets, exact metal tone, legible dial "\n        "or engraving, no invented stones or links."),\n}\n\nMATERIAL_OVERLAYS["glossy_plastic"] = MATERIAL_OVERLAYS["glossy"]\nMATERIAL_OVERLAYS["wood"] = (\n    "The product is wood: preserve the real grain direction, tone and finish "\n    "exactly; light with a soft raking key that reveals texture without "\n    "plastic-looking gloss; keep edges, joints and any engraving crisp; no "\n    "repainted or re-colored surfaces.")\n\nGENERIC_MATERIAL = (\n    "Preserve the product\'s true material response - its real texture, "\n    "reflectivity, translucency and finish - exactly as in the source photo.")\n\n\n@dataclass(frozen=True)\nclass Knob:\n    """One user-adjustable preset control."""\n    key: str\n    label: str\n    kind: str                    # select | slider | toggle | text\n    options: tuple[str, ...] = ()\n    default: str = ""\n    hint: str = ""\n\n\n@dataclass(frozen=True)\nclass Preset:\n    """A production-candidate photoshoot template with its detailed prompt."""\n    key: str\n    name: str\n    template_key: str            # links to templates.py scene fragments\n    tagline: str                 # one-line click-appeal for the picker UI\n    best_for: tuple[str, ...]    # product families this preset serves\n    default_materials: tuple[str, ...]\n    detailed_prompt: str         # the long researched prompt body\n    knobs: tuple[Knob, ...] = ()\n    cover_note: str = ""         # what the hero/before-after cover shows\n\n    @property\n    def template(self):\n        return get_template(self.template_key)\n\n\n_K_ASPECT = Knob("aspect", "Output shape", "select",\n                 ("1:1", "4:5", "3:2"), "1:1", "square, portrait or landscape")\n_K_VARIANTS = Knob("variants", "Variants", "select", ("1", "2", "3", "4"), "4",\n                   "alternatives to generate")\n_K_SHADOW = Knob("shadow_strength", "Shadow strength", "select",\n                 ("subtle", "natural", "dramatic"), "natural")\n_K_PROPS = Knob("props", "Props", "toggle", (), "off",\n                "allow restrained supporting props at frame edges")\n\nPRESETS: dict[str, Preset] = {p.key: p for p in [\n\n    Preset(\n        key="clean_studio", name="Clean Studio", template_key="clean_studio",\n        tagline="Marketplace-ready packshot on a seamless sweep",\n        best_for=("any product", "marketplace listings", "catalog"),\n        default_materials=("matte_plastic",),\n        detailed_prompt=(\n            "Commercial catalog photograph of the product standing on an "\n            "invisible surface against a seamless soft-white studio sweep, "\n            "shot as a clean ecommerce hero. Composition: front three-quarter "\n            "product view, the full product visible with comfortable even "\n            "margins, occupying about 70 percent of frame height, centered "\n            "with generous breathing room. Lighting: a broad diffused key "\n            "from the upper left and a gentle fill card from the opposite "\n            "side, 5500K neutral white balance, soft shadowless wrap with a "\n            "single faint contact shadow directly beneath the product. "\n            "Surface and environment: seamless white-to-very-light-gray "\n            "vertical gradient, no props, no set dressing. Palette and "\n            "treatment: honest color, crisp micro-contrast on the product, "\n            "calm neutral background. Constraints: the product\'s shape, "\n            "proportions, colors, logos and every label character remain "\n            "identical to the source; no duplicate product, no invented "\n            "text, no watermark, no border, no hands."),\n        knobs=(_K_ASPECT, _K_VARIANTS, _K_SHADOW),\n        cover_note="paired source vs white-sweep packshot"),\n\n    Preset(\n        key="luxury_hero", name="Luxury Hero", template_key="luxury_hero",\n        tagline="Dramatic dark-stage hero with rim light",\n        best_for=("fragrance", "watches", "premium electronics", "spirits"),\n        default_materials=("glass", "metal"),\n        detailed_prompt=(\n            "Premium campaign hero photograph of the product on a polished "\n            "black stone slab against a dark charcoal void. Composition: "\n            "slightly low hero angle with a 100mm macro product lens, tight "\n            "premium crop, product slightly right of center, faint warm rim "\n            "halo behind the silhouette. Lighting: one dramatic strip "\n            "softbox from above and behind carving a controlled edge "\n            "highlight, a warm 4300K accent kicker from the left rear, deep "\n            "controlled falloff into black, and a crisp short contact shadow "\n            "with a subtle mirrored reflection on the slab. Environment: "\n            "near-black gradient background with no visible horizon, fine "\n            "stone texture under the product. Treatment: high-contrast "\n            "aspirational grade, deep blacks, specular highlights kept "\n            "clean. Constraints: product geometry, finish, engravings and "\n            "label typography exactly as the source; no extra products, no "\n            "text, no smoke, no lens flare covering the label."),\n        knobs=(_K_ASPECT, _K_VARIANTS, _K_SHADOW),\n        cover_note="dark hero with edge-lit silhouette"),\n\n    Preset(\n        key="editorial_beauty", name="Editorial Beauty", template_key="editorial_beauty",\n        tagline="Soft skincare-editorial pedestal scene",\n        best_for=("skincare", "cosmetics", "wellness"),\n        default_materials=("skincare", "glass"),\n        detailed_prompt=(\n            "Editorial beauty photograph of the product on a matte pastel "\n            "stone pedestal in a soft beige studio. Composition: three-quarter "\n            "angle at product height with a 70mm lens, generous negative "\n            "space above the product, one out-of-focus botanical sprig far "\n            "behind at the frame edge. Lighting: diffused north-window "\n            "softbox from the left, gentle top fill, 5000K, airy highlights, "\n            "a long soft shadow falling right at low contrast. Environment: "\n            "soft beige-pink seamless backdrop with a gentle vignette, honed "\n            "stone pedestal with subtle mineral texture. Treatment: calm "\n            "editorial palette, lifted but controlled highlights, delicate "\n            "contrast. Constraints: vessel, cap, pump, label typography and "\n            "fill level exactly as the source; no hands, no extra bottles, "\n            "no generated label text, no water droplets unless the source "\n            "shows them."),\n        knobs=(_K_ASPECT, _K_VARIANTS, _K_PROPS),\n        cover_note="pastel pedestal beauty editorial"),\n\n    Preset(\n        key="lifestyle_kitchen", name="Lifestyle Kitchen", template_key="lifestyle_kitchen",\n        tagline="Warm morning-counter lifestyle context",\n        best_for=("food", "beverage", "kitchen goods", "supplements"),\n        default_materials=("food_packaging", "beverage_can"),\n        detailed_prompt=(\n            "Lifestyle photograph of the product on a light oak countertop "\n            "in a bright kitchen interior. Composition: 50mm at counter "\n            "height, shallow depth of field so the product stays tack-sharp "\n            "while the kitchen falls into warm bokeh, product on the left "\n            "third with counter space running right. Lighting: warm morning "\n            "window light from the rear-left wrapping around the product, a "\n            "white bounce card filling the front, 4800K, a natural "\n            "directional shadow to the right with a diffused edge. "\n            "Environment: blurred bright kitchen with cream tones, soft "\n            "horizontal counter line, circular bokeh in the top third, no "\n            "readable packaging or brands in the background. Treatment: "\n            "warm, lived-in, appetizing color with honest saturation. "\n            "Constraints: package artwork, cap, seal and every printed "\n            "character identical to the source; one background ingredient "\n            "cue maximum, heavily defocused; no hands, no duplicate product."),\n        knobs=(_K_ASPECT, _K_VARIANTS, _K_PROPS),\n        cover_note="counter-top lifestyle with bokeh kitchen"),\n\n    Preset(\n        key="natural_outdoor", name="Natural Outdoor", template_key="natural_outdoor",\n        tagline="Fresh open-shade outdoor setting",\n        best_for=("beverages", "wellness", "eco brands", "snacks"),\n        default_materials=("beverage_can", "glass"),\n        detailed_prompt=(\n            "Outdoor lifestyle photograph of the product on a weathered "\n            "stone surface with softly blurred green foliage behind. "\n            "Composition: 60mm at surface level, low horizon, the product "\n            "grounded on the stone with organic texture leading toward it, "\n            "background fully defocused. Lighting: open-shade daylight with "\n            "dappled sun accents, 5600K, gentle sky fill lifting the shadow "\n            "side, a soft dappled shadow consistent with foliage light. "\n            "Environment: muted green wash backdrop with soft dappled light "\n            "patches, no sky blown out, no litter or man-made clutter. "\n            "Treatment: fresh organic grade, true greens, clean whites on "\n            "the product. Constraints: label, cap, color and proportions "\n            "identical to the source; condensation only if physically "\n            "plausible for the scene; no insects, no watermarks, no extra "\n            "products."),\n        knobs=(_K_ASPECT, _K_VARIANTS, _K_SHADOW),\n        cover_note="stone ledge with foliage bokeh"),\n\n    Preset(\n        key="minimal_tabletop", name="Minimal Tabletop", template_key="minimal_tabletop",\n        tagline="Quiet architectural podium study",\n        best_for=("design objects", "cosmetics", "home goods"),\n        default_materials=("ceramic", "matte_plastic"),\n        detailed_prompt=(\n            "Minimalist tabletop photograph of the product on a single matte "\n            "neutral podium against a plain light-gray sweep. Composition: "\n            "85mm straight-on at product center, symmetrical framing, the "\n            "podium edge low in frame, large areas of quiet negative space. "\n            "Lighting: one large overhead softbox with minimal fill, 5200K, "\n            "sculpted but simple, a single soft elliptical shadow under the "\n            "podium edge. Environment: uniform light gray with a very subtle "\n            "floor-wall seam low in frame, no props, no texture noise. "\n            "Treatment: architectural, desaturated-neutral palette, precise "\n            "edges. Constraints: exact product silhouette, finish and "\n            "markings preserved; no color cast on the product; no extra "\n            "objects; no text."),\n        knobs=(_K_ASPECT, _K_VARIANTS),\n        cover_note="single podium minimal study"),\n\n    Preset(\n        key="seasonal_holiday", name="Seasonal Holiday", template_key="seasonal_holiday",\n        tagline="Festive warm-bokeh campaign look",\n        best_for=("gifting", "food & beverage", "limited editions"),\n        default_materials=("glossy", "food_packaging"),\n        detailed_prompt=(\n            "Festive campaign photograph of the product on a holiday "\n            "tabletop with restrained seasonal accents. Composition: 70mm "\n            "slightly elevated 15 degrees, product as the undisputed hero "\n            "center-frame, one pine sprig and warm bokeh lights confined to "\n            "the far background edges only. Lighting: warm tungsten key at "\n            "3800K with a cool rim light for separation, gentle sparkle "\n            "highlights in the bokeh, a warm soft shadow with a subtle "\n            "festive glow at the base. Environment: deep warm amber gradient "\n            "backdrop with distant gold bokeh, matte tabletop with fine "\n            "texture. Treatment: festive but premium - rich ambers, deep "\n            "greens, never cluttered, never kitsch. Constraints: product "\n            "artwork, ribbons and labels identical to the source; no "\n            "ornaments touching or occluding the product; no generated "\n            "greeting text; no snow on the product unless requested."),\n        knobs=(_K_ASPECT, _K_VARIANTS, _K_PROPS),\n        cover_note="warm bokeh festive hero"),\n\n    Preset(\n        key="social_ad", name="Social Ad", template_key="social_ad",\n        tagline="Bold graphic scroll-stopper with copy space",\n        best_for=("social campaigns", "launches", "ads"),\n        default_materials=("glossy",),\n        detailed_prompt=(\n            "Vertical 4:5 social advertisement photograph of the product "\n            "against a bold single-color backdrop with a graphic diagonal "\n            "light band. Composition: 50mm with a dynamic slight tilt, the "\n            "product upper-center and filling enough of the frame to stay "\n            "legible on a phone screen, deliberate open copy space at the "\n            "bottom-left for later headline placement. Lighting: hard "\n            "directional key producing a defined graphic drop shadow offset "\n            "down-right, a colored gel accent from the rear, high contrast "\n            "with a crisp product silhouette. Environment: saturated solid "\n            "backdrop with a diagonal lighter band, product floating "\n            "slightly above a implied surface. Treatment: energetic modern "\n            "ad grade, punchy but believable color. Constraints: product "\n            "geometry, logos and label text exactly as the source; no "\n            "generated words anywhere in frame; no extra products; nothing "\n            "in the reserved copy space; no watermark."),\n        knobs=(_K_VARIANTS, _K_SHADOW,\n               Knob("backdrop_color", "Backdrop color", "text", (), "brand red",\n                    "a solid color or hex, e.g. #C8102E")),\n        cover_note="bold diagonal-band ad layout"),\n\n    Preset(\n        key="reflective_surface", name="Reflective Mirror", template_key="reflective_surface",\n        tagline="Sleek black-acrylic mirror reflection",\n        best_for=("electronics", "fragrance", "watches", "tech"),\n        default_materials=("metal", "glass", "glossy"),\n        detailed_prompt=(\n            "Sleek product photograph on glossy black acrylic with a clear "\n            "mirror reflection below the product. Composition: 90mm at "\n            "surface level, product centered with its reflection extending "\n            "naturally below, dark gradient studio behind with no visible "\n            "horizon. Lighting: two strip softboxes left and right creating "\n            "symmetric edge highlights, 5000K, the reflection fading "\n            "gradually with distance, minimal ambient shadow. Environment: "\n            "dark neutral gradient, the acrylic\'s fine reflection haze kept "\n            "physically plausible. Treatment: polished tech grade, deep "\n            "blacks, controlled speculars. Constraints: the reflection must "\n            "mirror the real product geometry - never an invented bottom "\n            "side; product markings, ports and labels exactly as the source; "\n            "no duplicated product beside the reflection; no text."),\n        knobs=(_K_ASPECT, _K_VARIANTS),\n        cover_note="black acrylic mirror product study"),\n\n    Preset(\n        key="flat_lay", name="Flat Lay", template_key="flat_lay",\n        tagline="Top-down editorial arrangement",\n        best_for=("apparel", "accessories", "stationery", "multi-product sets"),\n        default_materials=("fabric", "leather"),\n        detailed_prompt=(\n            "Top-down flat lay photograph of the product on textured linen "\n            "viewed directly from above. Composition: camera at exactly 90 "\n            "degrees overhead, product centered with even margins, at most "\n            "two minimal side props resting at the corners, everything "\n            "aligned to a quiet grid. Lighting: a large overhead diffused "\n            "source, perfectly even with no directional bias, 5500K, only a "\n            "very soft ambient-occlusion ring under the product. "\n            "Environment: uniform light textile texture, natural weave "\n            "visible at close range. Treatment: tidy graphic editorial "\n            "grade, true fabric color, crisp edges. Constraints: fabric "\n            "weave, stitching, hardware and print exactly as the source; no "\n            "folds invented across logos; no extra items beyond the corner "\n            "props; no text."),\n        knobs=(_K_ASPECT, _K_VARIANTS, _K_PROPS),\n        cover_note="top-down linen arrangement"),\n\n    Preset(\n        key="macro_detail", name="Macro Detail", template_key="minimal_tabletop",\n        tagline="Raking-light texture close-up",\n        best_for=("craft details", "materials", "quality proof"),\n        default_materials=("leather", "metal", "fabric"),\n        detailed_prompt=(\n            "Macro commercial photograph focusing on the product\'s defining "\n            "material or construction feature - its weave, grain, engraving, "\n            "stitching or mechanism. Composition: tight macro crop on the "\n            "feature with smooth background falloff, the rest of the product "\n            "dissolving into soft blur. Lighting: raking soft side light "\n            "revealing surface texture, sharp detail across the important "\n            "feature, neutral color reproduction. Environment: minimal "\n            "out-of-focus neutral backdrop. Treatment: honest macro texture, "\n            "no oversharpening halo. Constraints: every texture element must "\n            "match the real product - no altered stitching, no changed "\n            "hardware, no invented engraving, no text, no dust or scratches "\n            "added."),\n        knobs=(_K_VARIANTS, _K_SHADOW),\n        cover_note="macro texture detail"),\n\n    Preset(\n        key="hero_negative_space", name="Hero + Copy Space", template_key="luxury_hero",\n        tagline="Website hero with room for your headline",\n        best_for=("landing pages", "banners", "campaigns"),\n        default_materials=("glossy",),\n        detailed_prompt=(\n            "Wide website-hero photograph of the product positioned on the "\n            "right third of the frame on a minimal stone surface. "\n            "Composition: landscape banner framing, the product anchored "\n            "right-of-center with a clean open area on the left reserved for "\n            "later headline and button placement - absolutely no objects or "\n            "texture clutter in the reserved area. Lighting: soft "\n            "directional light from the upper left, a realistic contact "\n            "shadow, refined falloff toward the left. Environment: minimal "\n            "surface with a quiet gradient wall, subtle vignette pulling "\n            "attention to the product. Treatment: refined commercial "\n            "campaign grade. Constraints: product exactly as the source; no "\n            "generated text or placeholder glyphs anywhere; no duplicate "\n            "product; the left half stays clean enough for white headline "\n            "type."),\n        knobs=(_K_VARIANTS,),\n        cover_note="banner with clean headline zone"),\n]}\n\nassert len(PRESETS) >= 8, "problem statement requires at least 8 templates"\n\n\ndef get_preset(key: str) -> Preset:\n    try:\n        return PRESETS[key]\n    except KeyError:\n        raise KeyError(\n            f"unknown preset \'{key}\'. available: {\', \'.join(sorted(PRESETS))}"\n        ) from None\n\n\ndef material_overlay(material: str | None) -> str:\n    """Resolve a declared/detected material to its overlay text."""\n    if not material:\n        return GENERIC_MATERIAL\n    return MATERIAL_OVERLAYS.get(material.lower().strip(), GENERIC_MATERIAL)\n\n\ndef build_detailed_prompt(preset: Preset, *, product_description: str = "",\n                          material: str | None = None,\n                          instruction: str = "",\n                          reference_style: str = "",\n                          knob_values: dict[str, str] | None = None) -> str:\n    """Assemble the full long prompt for one generation.\n\n    Order is deliberate: product identity and fidelity constraints lead\n    (editor models attend to the front), then the researched scene body,\n    then material physics, reference style, knobs and the owner\'s free-form\n    instruction. Nothing is length-capped here; longprompt.plan_prompt\n    handles provider-specific feeding.\n    """\n    parts: list[str] = []\n    if product_description:\n        parts.append(f"The product: {product_description.strip()}.")\n    parts.append(preset.detailed_prompt.strip())\n    parts.append(material_overlay(material))\n    if reference_style:\n        parts.append(f"Match this reference style: {reference_style.strip()}.")\n    kv = knob_values or {}\n    color = (kv.get("backdrop_color") or "").strip()\n    if color:\n        parts.append(f"The backdrop color is {color}.")\n    if kv.get("shadow_strength") == "dramatic":\n        parts.append("Push the shadow deeper and more directional.")\n    elif kv.get("shadow_strength") == "subtle":\n        parts.append("Keep the shadow barely-there and soft.")\n    if instruction.strip():\n        parts.append(f"Owner instruction (scene and environment only, never "\n                     f"the product): {instruction.strip()}.")\n    parts.append("Photorealistic commercial product photography, high "\n                 "resolution, sharp focus on the product.")\n    return " ".join(parts)\n\n\ndef presets_for_ui() -> list[dict]:\n    """JSON-able preset cards for the website / notebook pickers."""\n    out = []\n    for p in PRESETS.values():\n        t = p.template\n        out.append({\n            "key": p.key, "name": p.name, "tagline": p.tagline,\n            "category": t.category, "aspect": t.aspect.value,\n            "best_for": list(p.best_for),\n            "materials": list(p.default_materials),\n            "knobs": [{"key": k.key, "label": k.label, "kind": k.kind,\n                       "options": list(k.options), "default": k.default,\n                       "hint": k.hint} for k in p.knobs],\n            "cover_note": p.cover_note,\n        })\n    return out\n'
# ---- embedded module: renderello/presets.py ----
path = base / 'renderello/presets.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Fail-closed sampled resident-memory monitor, no optional dependencies.\n\nWindows WorkingSetSize is resident bytes, not commit or peak allocation.\nThis is a polling guard, not an OS allocation ceiling or process-tree limit.\n"""\nimport ctypes\nimport sys\nfrom pathlib import Path\n\n\nclass ProcessMemoryCounters(ctypes.Structure):\n    _fields_ = [(\'cb\', ctypes.c_uint32), (\'PageFaultCount\', ctypes.c_uint32)] + [\n        (name, ctypes.c_size_t) for name in (\n            \'PeakWorkingSetSize\', \'WorkingSetSize\', \'QuotaPeakPagedPoolUsage\',\n            \'QuotaPagedPoolUsage\', \'QuotaPeakNonPagedPoolUsage\',\n            \'QuotaNonPagedPoolUsage\', \'PagefileUsage\', \'PeakPagefileUsage\')]\n\n\ndef _windows_api():\n    kernel = ctypes.WinDLL(\'kernel32\', use_last_error=True)\n    psapi = ctypes.WinDLL(\'psapi\', use_last_error=True)\n    kernel.OpenProcess.argtypes = [ctypes.c_uint32, ctypes.c_int32, ctypes.c_uint32]\n    kernel.OpenProcess.restype = ctypes.c_void_p\n    kernel.CloseHandle.argtypes = [ctypes.c_void_p]\n    kernel.CloseHandle.restype = ctypes.c_int32\n    psapi.GetProcessMemoryInfo.argtypes = [ctypes.c_void_p,\n        ctypes.POINTER(ProcessMemoryCounters), ctypes.c_uint32]\n    psapi.GetProcessMemoryInfo.restype = ctypes.c_int32\n    return kernel, psapi\n\n\nclass ResidentMemory:\n    def __init__(self, pid, *, platform=None, windows_api=None):\n        if type(pid) is not int or not 0 < pid <= 0xffffffff:\n            raise ValueError(\'positive DWORD process id required\')\n        self.pid = pid\n        self.platform = sys.platform if platform is None else platform\n        self.handle = None\n        self.closed = False\n        if self.platform == \'win32\':\n            self.kernel, self.psapi = (windows_api or _windows_api)()\n            # Query only, non-inherited. Modern Windows supports limited query access.\n            self.handle = self.kernel.OpenProcess(0x1000, False, pid)\n            if not self.handle:\n                raise RuntimeError(\'Windows process memory handle unavailable\')\n        elif not self.platform.startswith(\'linux\'):\n            raise RuntimeError(\'resident-memory monitor unsupported on this platform\')\n\n    def sample(self):\n        if self.closed:\n            raise RuntimeError(\'resident-memory monitor is closed\')\n        if self.platform == \'win32\':\n            counters = ProcessMemoryCounters()\n            counters.cb = ctypes.sizeof(counters)\n            if not self.psapi.GetProcessMemoryInfo(self.handle, ctypes.byref(counters), counters.cb):\n                raise RuntimeError(\'Windows process memory sample unavailable\')\n            value = counters.WorkingSetSize\n        else:\n            text = Path(f\'/proc/{self.pid}/status\').read_text()\n            line = next((line for line in text.splitlines() if line.startswith(\'VmRSS:\')), None)\n            fields = line.split() if line else []\n            if len(fields) != 3 or fields[2] != \'kB\':\n                raise RuntimeError(\'Linux resident-memory sample unavailable\')\n            value = int(fields[1]) * 1024\n        if value <= 0:\n            raise RuntimeError(\'invalid resident-memory sample\')\n        return value\n\n    def close(self):\n        if not self.closed:\n            self.closed = True\n            if self.handle is not None:\n                handle, self.handle = self.handle, None\n                if not self.kernel.CloseHandle(handle):\n                    raise RuntimeError(\'Windows process memory handle close failed\')\n'
# ---- embedded module: renderello/process_memory.py ----
path = base / 'renderello/process_memory.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Local source-bound prompt preparation. No segmentation/model/network download."""\nfrom pathlib import Path\nimport hashlib,io,json,uuid\nfrom PIL import Image,UnidentifiedImageError\n\n\ndef prepare_source(data:bytes,out_root:Path):\n    if not data or len(data)>25_000_000:raise ValueError(\'source must be 1 byte to 25 MB\')\n    try:\n        with Image.open(io.BytesIO(data)) as im:\n            if im.width*im.height>4_000_000:raise ValueError(\'source exceeds 4 million pixels; no silent resize\')\n            if min(im.size)<3:raise ValueError(\'source too small\')\n            # Match the CLI decode exactly, without browser EXIF rotation.\n            rgb=im.convert(\'RGB\');size=list(im.size)\n    except (UnidentifiedImageError,OSError) as e:raise ValueError(\'source is not a supported image\') from e\n    key=\'prompt-\'+uuid.uuid4().hex[:12];root=out_root/key;root.mkdir(parents=True)\n    (root/\'original-source.bin\').write_bytes(data);rgb.save(root/\'source-preview.png\')\n    d=dict(source_sha256=hashlib.sha256(data).hexdigest(),image_size=size,preview_url=\'/file/\'+key+\'/source-preview.png\',\n           job=key,decode=\'PIL RGB, no EXIF transpose; same coordinates as prompted-mask CLI\')\n    (root/\'source-manifest.json\').write_text(json.dumps(d,indent=2));return d\n\n\nPAGE=r\'\'\'<!doctype html><html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>Renderello mask prompts</title>\n<style>*{box-sizing:border-box}body{font-family:system-ui,sans-serif;max-width:1100px;margin:24px auto;padding:0 16px;color:#1c1c1e}h1{font-size:28px}p{line-height:1.5}.note{padding:12px;background:#f2f2f4;border-radius:8px}label{display:block;margin:12px 0 4px;font-weight:600}input,select,button{font:inherit;max-width:100%}input[type=text],select{width:100%;padding:10px}button,a.download{padding:10px 14px;border-radius:6px;border:1px solid #777;background:white;color:#1c1c1e;min-height:44px;cursor:pointer}button:disabled{opacity:.5;cursor:default}.controls{display:flex;gap:10px;flex-wrap:wrap;align-items:center;margin:12px 0}.controls>*{flex:0 1 auto}.panel{border:1px solid #ccc;padding:16px;border-radius:8px;margin:18px 0}#viewport{overflow:auto;max-height:650px;width:100%;border:1px solid #aaa}canvas{display:block;width:100%;max-width:none;height:auto;border:1px solid #aaa;touch-action:none;background:#eee}#status{overflow-wrap:anywhere}pre{white-space:pre-wrap;overflow-wrap:anywhere;background:#f2f2f4;padding:12px}#objects{width:100%}a{color:#0756b8}a.download{display:inline-block}#reviewed{width:20px;height:20px;vertical-align:middle}@media(max-width:500px){body{margin:16px auto}h1{font-size:24px}.panel{padding:10px}button{flex-grow:1}}</style></head><body>\n<a href="/">Back to Renderello</a><h1>Review mask prompts</h1>\n<p class="note">Preparation only. Draw boxes around each object, then add positive points on the product and negative points in empty holes or background. Drawing and export are preparation only. The separate explicit CPU button below can run the reviewed pipeline when configured; neither route certifies a correct mask.</p>\n<label for="source">Original source photo</label><input id="source" type="file" accept="image/*"><div class="controls"><button id="prepare">Load source for review</button></div>\n<p>Photo bytes stay in this local Renderello server\'s output folder. The preview uses the same decoded pixels as the CLI, without browser EXIF rotation. Do not edit or recompress the original after exporting prompts.</p>\n<p id="status" role="status" aria-live="polite">Choose an image. Maximum 4 million pixels / 25 MB.</p>\n<div class="panel"><label for="objects">Selected object</label><select id="objects" disabled></select><label for="name">Object name</label><input id="name" type="text" maxlength="80" disabled><div class="controls"><button id="add" disabled>Add another object</button><button id="remove" disabled>Remove selected object</button><button id="undo" disabled>Undo last edit</button></div>\n<label for="mode">Drawing mode</label><select id="mode" disabled><option value="pan">Pan</option><option value="box" selected>Draw box</option><option value="positive">Positive point</option><option value="negative">Negative point</option></select>\n<p>Blue boxes and numbered labels identify selected objects. Green points keep product; red points exclude background. Up to 8 objects and 32 points per object. The combined mask includes only objects you explicitly select.</p><p id="modehint">Box: drag around the whole object.</p><label for="zoom">Image zoom</label><select id="zoom"><option value="100">Fit width</option><option value="200">2x</option><option value="400">4x</option></select><p>For small details, zoom in and choose Pan to move around the image. Return to Draw box or a point mode to annotate. Desktop review is recommended for precise edges. Mobile layout and pointer coordinates were tested; real touch annotation is not verified.</p><div id="viewport"><canvas id="canvas" width="600" height="300" aria-label="Source image annotation canvas"></canvas></div><p id="geometry">No source loaded.</p>\n</div><label><input id="reviewed" type="checkbox" disabled> I inspected these boxes and points against this source image.</label>\n<div class="controls"><button id="export" disabled>Export reviewed prompt JSON</button></div><p id="download"></p><pre id="json" hidden></pre>\n<h2>Run reviewed intent + automatic alpha</h2><p>Step 1 runs CPU HQ-SAM only. Inspect its actual silhouette on all three backgrounds. Step 2 requires your explicit acceptance of that exact preview before intersecting it with the automatic alpha. A reviewed box is not a reviewed silhouette. Add at least one positive point on each product. If the mask has holes, selects background or loses product, change the prompts and retry instead of accepting. It does not run BiRefNet, download weights or certify quality. Intersection cannot recover automatic-mask missing parts. Inspect all three result backgrounds. Manual review stays required.</p><p id="runconfig">Checking server checkpoint setup...</p><label for="automatic">Existing automatic result ZIP (flat files)</label><input id="automatic" type="file" accept=".zip"><p>ZIP must contain source.png, cutout.png, estimated-alpha.png and its original bilateral/pyramid/semantic mask manifest, all at ZIP root.20MBcompressed/40MBunpacked cap. Must match the exact source.</p><div class="controls"><button id="runmask" disabled>1. Preview inferred silhouette (CPU)</button></div><p id="runstatus" role="status" aria-live="polite">Export reviewed prompts first.</p><div id="maskresults"></div><label><input id="silhouette" type="checkbox" disabled> I inspected this exact inferred silhouette on white, black and magenta. It keeps the selected product without unwanted holes or background.</label><div class="controls"><button id="combine" disabled>2. Accept silhouette and combine alpha</button></div><div id="combinedresults"></div><h2>Or run the optional mask stage by CLI</h2><p>This page does not install packages or download weights. Use the exact original photo and the reviewed JSON with the pinned checkpoint:</p><pre>python -m renderello.cli prompted-mask --source ORIGINAL_PHOTO --prompts reviewed-prompts.json --checkpoint sam_hq_vit_tiny.pth --out NEW_ATTEMPT</pre><p>Then inspect all black/white/magenta mask previews. Scores do not verify holes or completeness. Cropped parts cannot be recovered; source background colors may remain on edges. Owner-PC model execution is not verified by this editor.</p>\n<script>\n\'use strict\';\nconst $=id=>document.getElementById(id),c=$(\'canvas\'),ctx=c.getContext(\'2d\');\nlet state=null,img=null,selected=0,drag=null,history=[],generation=0,url=null,pan=null,job=null,exported=null,configured=false,pending=null,revision=0;\nfunction status(s){$(\'status\').textContent=s}\nfunction clearPreview(){revision++;pending=null;$(\'silhouette\').checked=false;$(\'silhouette\').disabled=true;$(\'combine\').disabled=true;$(\'combinedresults\').replaceChildren();$(\'maskresults\').replaceChildren()}\nfunction invalidate(){clearPreview();exported=null;$("runmask").disabled=true;$("maskresults").replaceChildren();if(url){URL.revokeObjectURL(url);url=null}$(\'download\').replaceChildren();$(\'json\').hidden=true;$(\'reviewed\').checked=false;$(\'export\').disabled=true}\nfunction active(){return state&&state.objects[selected]}\nfunction refresh(){const options=$(\'objects\');options.replaceChildren();if(!state)return;state.objects.forEach((o,i)=>{let e=document.createElement(\'option\');e.value=i;e.textContent=(i+1)+\'. \'+o.name;options.append(e)});options.value=selected;$(\'name\').value=active().name;$(\'geometry\').textContent=state.image_size.join(\' x \')+\' source pixels. Selected box: \'+(active().box?active().box.join(\', \'):\'not drawn\')+\'. Points: \'+active().points.length;draw();$(\'export\').disabled=!$(\'reviewed\').checked||!state.objects.every(o=>o.box&&o.name.trim())}\nfunction draw(){ctx.clearRect(0,0,c.width,c.height);if(!img)return;ctx.drawImage(img,0,0);const scale=c.width/Math.max(1,c.getBoundingClientRect().width);ctx.lineWidth=scale*2;ctx.font=scale*16+\'px sans-serif\';state.objects.forEach((o,i)=>{ctx.strokeStyle=i===selected?\'#0756b8\':\'#777\';ctx.fillStyle=\'#0756b8\';if(o.box){let b=o.box;ctx.strokeRect(b[0],b[1],b[2]-b[0],b[3]-b[1]);ctx.fillText(String(i+1),b[0]+4,b[1]+scale*20)}o.points.forEach((p,j)=>{ctx.beginPath();ctx.fillStyle=o.labels[j]?\'#008535\':\'#d91830\';ctx.arc(p[0],p[1],scale*4,0,Math.PI*2);ctx.fill()})});if(drag){ctx.strokeStyle=\'#0756b8\';ctx.strokeRect(drag[0],drag[1],drag[2]-drag[0],drag[3]-drag[1])}}\nfunction coord(e){const b=c.getBoundingClientRect();return [Math.min(c.width-1,Math.max(0,(e.clientX-b.left)*c.width/b.width)),Math.min(c.height-1,Math.max(0,(e.clientY-b.top)*c.height/b.height))].map(Math.round)}\nfunction snapshot(){history.push(JSON.stringify({objects:state.objects,selected}));invalidate()}\nfunction reset(){generation++;state=null;img=null;drag=null;pan=null;history=[];$(\'zoom\').value=\'100\';c.style.width=\'100%\';$(\'mode\').value=\'box\';$(\'modehint\').textContent=\'Box: drag around the whole object.\';invalidate();for(let id of [\'objects\',\'name\',\'add\',\'remove\',\'undo\',\'mode\',\'reviewed\'])$(id).disabled=true;c.width=600;c.height=300;draw();$(\'objects\').replaceChildren();$(\'name\').value=\'\';$(\'geometry\').textContent=\'No source loaded.\'}\n$(\'mode\').onchange=()=>{drag=null;pan=null;$(\'modehint\').textContent={pan:\'Pan: drag to move a zoomed image.\',box:\'Box: drag around the whole object.\',positive:\'Positive point: click on product.\',negative:\'Negative point: click inside an empty hole or background.\'}[$(\'mode\').value];draw()};\n$(\'zoom\').onchange=()=>{c.style.width=$(\'zoom\').value+\'%\';draw()};\n$(\'source\').onchange=()=>{reset();status(\'Source changed. Load it before drawing.\')};\n$(\'prepare\').onclick=async()=>{const file=$(\'source\').files[0];if(!file){status(\'Choose an image first.\');return}reset();const token=generation;status(\'Preparing source preview...\');let f=new FormData();f.append(\'photo\',file);try{const res=await fetch(\'/prompt-prepare\',{method:\'POST\',body:f});if(!res.ok)throw new Error(await res.text());const data=await res.json();if(token!==generation)return;job=data.job;const preview=new Image();preview.onload=()=>{if(token!==generation)return;img=preview;state={source_sha256:data.source_sha256,image_size:data.image_size,origin:\'reviewed_source_geometry\',objects:[{name:\'product\',points:[],labels:[]}]};selected=0;c.width=data.image_size[0];c.height=data.image_size[1];for(let id of [\'objects\',\'name\',\'add\',\'remove\',\'undo\',\'mode\',\'reviewed\'])$(id).disabled=false;refresh();status(\'Source loaded. Draw a box. SHA256 \'+data.source_sha256)};preview.onerror=()=>status(\'Preview could not load; do not export.\');preview.src=data.preview_url}catch(e){if(token===generation)status(String(e.message))}};\n$(\'objects\').onchange=()=>{selected=Number($(\'objects\').value);drag=null;refresh()};$(\'name\').oninput=()=>{if(!state)return;active().name=$(\'name\').value;invalidate();refresh()};\n$(\'add\').onclick=()=>{if(!state||state.objects.length>=8){status(\'Maximum 8 objects.\');return}snapshot();state.objects.push({name:\'object \'+(state.objects.length+1),points:[],labels:[]});selected=state.objects.length-1;refresh()};\n$(\'remove\').onclick=()=>{if(!state||state.objects.length===1){status(\'Keep at least one selected object.\');return}snapshot();state.objects.splice(selected,1);selected=Math.min(selected,state.objects.length-1);refresh()};\n$(\'undo\').onclick=()=>{if(!history.length)return;const old=JSON.parse(history.pop());state.objects=old.objects;selected=old.selected;invalidate();refresh()};\nc.onpointerdown=e=>{if(!state)return;e.preventDefault();let p=coord(e);c.setPointerCapture(e.pointerId);if($(\'mode\').value===\'pan\'){pan=[e.clientX,e.clientY,$(\'viewport\').scrollLeft,$(\'viewport\').scrollTop];return}if($(\'mode\').value===\'box\'){drag=[...p,...p];draw()}else{if(active().points.length>=32){status(\'Maximum 32 points per object.\');return}snapshot();active().points.push(p);active().labels.push($(\'mode\').value===\'positive\'?1:0);refresh()}};\nc.onpointermove=e=>{if(pan){$(\'viewport\').scrollLeft=pan[2]+pan[0]-e.clientX;$(\'viewport\').scrollTop=pan[3]+pan[1]-e.clientY;return}if(drag){const p=coord(e);drag[2]=p[0];drag[3]=p[1];draw()}};\nc.onpointerup=e=>{if(pan){pan=null;return}if(!drag)return;let p=coord(e),b=[Math.min(drag[0],p[0]),Math.min(drag[1],p[1]),Math.max(drag[0],p[0]),Math.max(drag[1],p[1])];drag=null;if(b[2]-b[0]<2||b[3]-b[1]<2){status(\'Box too small; drag around the whole object.\');draw();return}snapshot();active().box=b;refresh()};c.onpointercancel=()=>{drag=null;pan=null;draw()};\nwindow.addEventListener(\'resize\',draw);\n$(\'reviewed\').onchange=refresh;\n$(\'export\').onclick=()=>{if(!state||!$(\'reviewed\').checked)return;const names=state.objects.map(o=>o.name.trim());if(!state.objects.every(o=>o.box)||names.some(n=>!n)||new Set(names).size!==names.length){status(\'Draw every box and use unique object names.\');return}let data=structuredClone(state);data.objects.forEach(o=>o.name=o.name.trim());exported=data;$("runmask").disabled=!configured||!$("automatic").files[0];const text=JSON.stringify(data,null,2);if(url)URL.revokeObjectURL(url);url=URL.createObjectURL(new Blob([text],{type:\'application/json\'}));const a=document.createElement(\'a\');a.href=url;a.download=\'reviewed-prompts.json\';a.className=\'download\';a.textContent=\'Download reviewed-prompts.json\';$(\'download\').replaceChildren(a);$(\'json\').textContent=text;$(\'json\').hidden=false;status(\'Prompts exported for this exact source. No model has run. Inspect resulting masks after the CLI run.\')};\n\nfetch(\'/reviewed-status\').then(r=>r.json()).then(d=>{configured=d.configured;$(\'runconfig\').textContent=configured?\'Pinned checkpoint configured on this local server.\':\'Disabled: start this server with --reviewed-checkpoint PATH_TO_EXISTING_sam_hq_vit_tiny.pth in the optional HQ-SAM environment. No download is performed.\';}).catch(()=>{$(\'runconfig\').textContent=\'Server setup could not be checked.\'});\n\nfunction renderPreviews(root,d){root.replaceChildren();if(d.download_url){const a=document.createElement(\'a\');a.href=d.download_url;a.textContent=\'Download reviewed-mask result ZIP\';a.className=\'download\';root.append(a)}for(const color of [\'white\',\'black\',\'magenta\']){const p=document.createElement(\'p\');p.textContent=color+\' review\';const im=document.createElement(\'img\');im.src=d.preview_urls[color];im.alt=color+\' background mask review\';im.style.width=\'100%\';im.style.maxWidth=\'700px\';root.append(p,im)}}\n$(\'automatic\').onchange=()=>{clearPreview();$(\'runmask\').disabled=!configured||!exported||!$(\'automatic\').files[0]};\n$(\'silhouette\').onchange=()=>{$(\'combine\').disabled=!pending||!$(\'silhouette\').checked};\n$(\'runmask\').onclick=async()=>{if(!exported||!configured)return;if(exported.objects.some(o=>!o.labels.includes(1))){$(\'runstatus\').textContent=\'Add at least one positive point on each product, inspect prompts and export again. A box alone is not enough.\';return}clearPreview();const token=revision;const f=new FormData();f.append(\'job\',job);f.append(\'prompts\',JSON.stringify(exported));f.append(\'automatic\',$(\'automatic\').files[0]);$(\'runmask\').disabled=true;$(\'runstatus\').textContent=\'Inferring HQ-SAM silhouette on CPU only. No intersection yet. Up to180seconds.\';try{const res=await fetch(\'/reviewed-run\',{method:\'POST\',body:f});if(!res.ok)throw new Error(await res.text());const d=await res.json();if(token!==revision)return;renderPreviews($(\'maskresults\'),d);await Promise.all([...$(\'maskresults\').querySelectorAll(\'img\')].map(im=>new Promise((resolve,reject)=>{if(im.complete)return im.naturalWidth?resolve():reject(Error(\'preview image failed\'));im.onload=resolve;im.onerror=()=>reject(Error(\'preview image failed\'))})));if(token!==revision)return;pending=d;$(\'silhouette\').disabled=false;$(\'runstatus\').textContent=\'Silhouette ready, NOT combined. Inspect every preview. Reject holes or unwanted background by editing prompts and retrying. Only accept a correct silhouette.\';}catch(e){if(token===revision){clearPreview();$(\'runstatus\').textContent=\'Preview failed: \'+e.message}}finally{$(\'runmask\').disabled=!configured||!exported||!$(\'automatic\').files[0]}};\n$(\'combine\').onclick=async()=>{if(!pending||!$(\'silhouette\').checked)return;const token=revision;const f=new FormData();f.append(\'job\',pending.job);f.append(\'fingerprint\',pending.fingerprint);f.append(\'acceptance\',\'I inspected this exact inferred silhouette\');$(\'combine\').disabled=true;$(\'silhouette\').disabled=true;$(\'runstatus\').textContent=\'Combining the explicitly reviewed silhouette with automatic alpha.\';try{const res=await fetch(\'/reviewed-confirm\',{method:\'POST\',body:f});if(!res.ok)throw new Error(await res.text());const d=await res.json();if(token!==revision)return;pending=null;renderPreviews($(\'combinedresults\'),d);$(\'runstatus\').textContent=\'Combined after explicit silhouette review. NOT quality-accepted. Inspect final edges, missing parts, glass and source completeness. Intersection cannot restore lost parts.\';}catch(e){if(token===revision){clearPreview();$(\'runstatus\').textContent=\'Combination failed: \'+e.message}}};\n</script></body></html>\'\'\'\n'
# ---- embedded module: renderello/prompt_editor.py ----
path = base / 'renderello/prompt_editor.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Reviewed box/point Light HQ-SAM masks. Never automatic product acceptance."""\nfrom pathlib import Path\nimport hashlib,json,time\nimport numpy as np\nfrom PIL import Image\n\nMODEL_SHA256=\'0f32c075ccdd870ae54db2f7630e7a0878ede5a2b06d05d6fe02c65a82fb7196\'\nMODEL_REV=\'09b02a333b37772133eff3997bdba997867374b7\'\nMODEL_URL=\'https://huggingface.co/lkeab/hq-sam/resolve/\'+MODEL_REV+\'/sam_hq_vit_tiny.pth\'\n\n\ndef validate_prompts(data,source,width,height):\n    if not isinstance(data,dict) or data.get(\'source_sha256\')!=hashlib.sha256(source.read_bytes()).hexdigest():raise ValueError(\'prompts must bind to exact source SHA256\')\n    if data.get(\'image_size\')!=[width,height]:raise ValueError(\'prompt image_size must match source width,height\')\n    if data.get(\'origin\')!=\'reviewed_source_geometry\':raise ValueError(\'prompt origin must explicitly declare reviewed_source_geometry; do not label automatic prompts reviewed\')\n    objects=data.get(\'objects\')\n    if not isinstance(objects,list) or not 1<=len(objects)<=8:raise ValueError(\'one to eight explicitly selected objects required\')\n    names=set()\n    for item in objects:\n        if not isinstance(item,dict) or not isinstance(item.get(\'name\'),str) or not item[\'name\'].strip() or len(item[\'name\'])>80 or item[\'name\'] in names:raise ValueError(\'each object needs a unique short name\')\n        names.add(item[\'name\']);box=np.asarray(item.get(\'box\'),dtype=float)\n        if box.shape!=(4,) or not np.isfinite(box).all() or not (0<=box[0]<box[2]<=width and 0<=box[1]<box[3]<=height):raise ValueError(\'invalid source-pixel box\')\n        points=item.get(\'points\',[]);labels=item.get(\'labels\',[])\n        if not isinstance(points,list) or len(points)>32 or not isinstance(labels,list) or len(points)!=len(labels) or any(type(l)!=int or l not in (0,1) for l in labels):raise ValueError(\'points need equal0/1labels, max32\')\n        if points:\n            p=np.asarray(points,dtype=float)\n            if p.shape!=(len(points),2) or not np.isfinite(p).all() or np.any(p<0) or np.any(p[:,0]>=width) or np.any(p[:,1]>=height):raise ValueError(\'points must be finite source-pixel coordinates\')\n    return objects\n\n\ndef infer_file(source,out,prompts,checkpoint):\n    if out.exists():raise ValueError(\'output exists; preserve attempts\')\n    with Image.open(source) as im:\n        if im.width*im.height>4_000_000:raise ValueError(\'source exceeds4million pixels; no silent resize\')\n        image=np.array(im.convert(\'RGB\'));W,H=im.size\n    data=json.loads(prompts.read_text());objects=validate_prompts(data,source,W,H)\n    if hashlib.sha256(checkpoint.read_bytes()).hexdigest()!=MODEL_SHA256:raise ValueError(\'checkpoint does not match official pinned Light HQ-SAM SHA256\')\n    import torch\n    from segment_anything_hq import sam_model_registry,SamPredictor\n    torch.set_num_threads(2);started=time.monotonic();model=sam_model_registry[\'vit_tiny\'](checkpoint=None)\n    model.load_state_dict(torch.load(checkpoint,map_location=\'cpu\',weights_only=True));model.eval();predictor=SamPredictor(model)\n    with torch.inference_mode():predictor.set_image(image)\n    masks=[];scores=[]\n    for item in objects:\n        points=item.get(\'points\',[])\n        with torch.inference_mode():m,s,_=predictor.predict(box=np.asarray(item[\'box\']),point_coords=np.asarray(points) if points else None,point_labels=np.asarray(item.get(\'labels\')) if points else None,multimask_output=False)\n        if m.shape!=(1,H,W) or not np.any(m):raise ValueError(\'empty/invalid prompted mask\')\n        masks.append(m[0].astype(bool));scores.append(s.tolist())\n    alpha=np.logical_or.reduce(masks).astype(\'uint8\')*255\n    out.mkdir(parents=True)\n    for i,a in enumerate(masks):Image.fromarray(a.astype(\'uint8\')*255).save(out/f\'object-{i+1}-alpha.png\')\n    for name,a in [(\'source.png\',image),(\'estimated-alpha.png\',alpha),(\'cutout.png\',np.dstack([image,alpha]))]:Image.fromarray(a).save(out/name)\n    for name,col in [(\'black\',(20,20,20)),(\'white\',(250,250,250)),(\'magenta\',(190,0,150))]:\n        bg=Image.new(\'RGB\',(W,H),col);bg.paste(Image.fromarray(image),mask=Image.fromarray(alpha));bg.save(out/(name+\'-review.png\'))\n    (out/\'reviewed-prompts.json\').write_text(json.dumps(data,indent=2))\n    report=dict(source_sha256=data[\'source_sha256\'],prompt_sha256=hashlib.sha256(prompts.read_bytes()).hexdigest(),prompt_origin=data[\'origin\'],\n                objects=[i[\'name\'] for i in objects],union_rule=\'explicit object masks OR, not automatic semantic grouping\',model=\'Light HQ-SAM vit_tiny\',model_revision=MODEL_REV,model_url=MODEL_URL,model_sha256=MODEL_SHA256,\n                source_rgb_unchanged=True,device=\'cpu\',scores_unvalidated=scores,quality=\'NOT_EVALUATED\',semantic_completeness=\'NOT_VERIFIED\',seconds=time.monotonic()-started,\n                warnings=[\'Reviewed prompts are manual input, not an automatic mask success.\',\'High predicted scores do not prove correct holes, thin parts or semantic completeness.\',\'Binary mask is not optical alpha. Background reflections and hard edge aliasing can remain.\',\'Union includes only explicitly selected objects; omitted accessories remain omitted.\'])\n    report[\'output_sha256\']={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in out.iterdir() if p.is_file()};(out/\'prompted-mask-manifest.json\').write_text(json.dumps(report,indent=2));return report\n\n\ndef guarded_file(source,out,prompts,checkpoint):\n    import subprocess,sys,tempfile,os\n    cmd=[sys.executable,\'-m\',\'renderello.prompted_mask\',\'--source\',str(source),\'--out\',str(out),\'--prompts\',str(prompts),\'--checkpoint\',str(checkpoint)]\n    env=os.environ.copy();env[\'OMP_NUM_THREADS\']=\'2\'\n    with tempfile.TemporaryFile(mode=\'w+\') as stdout,tempfile.TemporaryFile(mode=\'w+\') as stderr:\n        p=subprocess.Popen(cmd,stdout=stdout,stderr=stderr,text=True,env=env);start=time.monotonic()\n        try:\n            while p.poll() is None:\n                if time.monotonic()-start>180:raise RuntimeError(\'prompted-mask180second timeout; not accepted\')\n                try:rss=int(next(l.split()[1] for l in Path(f\'/proc/{p.pid}/status\').read_text().splitlines() if l.startswith(\'VmRSS:\')))\n                except (FileNotFoundError,StopIteration):rss=0\n                if rss>2560*1024:raise RuntimeError(\'prompted-mask2.5GiB RSS limit; not accepted\')\n                time.sleep(.08)\n        finally:\n            if p.poll() is None:\n                p.terminate()\n                try:p.wait(timeout=5)\n                except subprocess.TimeoutExpired:p.kill();p.wait()\n        stdout.seek(0);stderr.seek(0)\n        if p.returncode:raise RuntimeError(\'prompted-mask worker failed: \'+stderr.read()[-2000:])\n        return json.load(stdout)\n\n\ndef main():\n    import argparse\n    p=argparse.ArgumentParser(description=__doc__)\n    for n in (\'source\',\'out\',\'prompts\',\'checkpoint\'):p.add_argument(\'--\'+n,type=Path,required=True)\n    a=p.parse_args();print(json.dumps(infer_file(a.source,a.out,a.prompts,a.checkpoint)))\n\nif __name__==\'__main__\':main()\n'
# ---- embedded module: renderello/prompted_mask.py ----
path = base / 'renderello/prompted_mask.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Prompt assembly. Every generative call gets the same hard product-fidelity\nguard plus the template\'s environment fragments plus an optional owner\ninstruction. The guard is non-negotiable per the problem statement: change only\nscene/lighting/context, never the product.\n\nThe full owner instruction is retained. Model-specific encoders must handle\nlong text explicitly; including text does not guarantee model obedience.\n"""\nfrom __future__ import annotations\n\nfrom .schemas import EditRequest\n\nFIDELITY_GUARD = (\n    "Keep the exact product unchanged: identical shape, packaging, color, logos, "\n    "label text and typography, proportions and material. Do not redraw, restyle, "\n    "retext or relabel the product. Change only the background, surface, lighting "\n    "and scene context around it."\n)\n\nNEGATIVE_PROMPT = (\n    "altered logo, changed label text, invented text, misspelled brand, deformed "\n    "packaging, warped bottle, wrong product, extra products, duplicated product, "\n    "melted edges, plastic-looking material change, watermark, frame, border"\n)\n\nMAX_INSTRUCTION_CHARS = 16000  # explicit application budget, never silent truncation\n\n\ndef build_prompt(request: EditRequest) -> str:\n    """Assemble the final edit prompt for a generative route."""\n    t = request.template\n    parts = [\n        FIDELITY_GUARD,\n        f"Place the product in this scene: {t.scene}.",\n        f"Lighting: {t.lighting}.",\n        f"Camera: {t.camera}.",\n        f"Mood: {t.mood}.",\n        f"Background: {t.background_hint}.",\n        f"Shadow and contact: {t.shadow}.",\n    ]\n    instruction = validate_instruction(request.custom_instruction).strip()\n    if instruction:\n        parts.append(f"Additional scene instruction (environment only): "\n                     f"{instruction}.")\n    parts.append("Photorealistic commercial product photograph.")\n    return " ".join(parts)\n\n\ndef build_negative_prompt() -> str:\n    return NEGATIVE_PROMPT\n\n\ndef build_harmonize_prompt(request: EditRequest) -> str:\n    """Route 3: gentler pass over an already-correct composite. The product is\n    already pixel-correct; ask only for light/shadow blending."""\n    t = request.template\n    instruction = validate_instruction(request.custom_instruction).strip()\n    extra = f" Additional scene instruction (environment only): {instruction}." if instruction else ""\n    return (\n        f"{FIDELITY_GUARD} This is already the correct product cutout on a new "\n        f"background. Only harmonize lighting, shadow and color grading so the "\n        f"product sits naturally in the scene ({t.lighting}; {t.scene}). "\n        "Do not move, resize, rotate or repaint the product." + extra\n    )\n\n\ndef validate_instruction(text: str) -> str:\n    """Reject instructions that ask to change the product itself - the problem\n    statement forbids it, so fail loudly instead of quietly corrupting labels."""\n    if not isinstance(text, str):\n        raise ValueError("custom instruction must be text")\n    if len(text) > MAX_INSTRUCTION_CHARS:\n        raise ValueError(f"custom instruction exceeds {MAX_INSTRUCTION_CHARS} characters; no truncation performed")\n    banned = ("change the label", "edit the logo", "remove the text", "rebrand",\n              "different product", "swap the product", "redesign the packaging")\n    low = text.lower()\n    for b in banned:\n        if b in low:\n            raise ValueError(f"custom instruction \'{b}\' would alter the product; "\n                             "only scene/lighting/context instructions are allowed")\n    return text\n'
# ---- embedded module: renderello/prompts.py ----
path = base / 'renderello/prompts.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Read recorded pipeline provider evidence into existing typed economics inputs.\n\nExplicit preregistered output mapping is required. This authenticates neither\nprovider responses, account denomination, reviewer identity nor current prices.\n"""\nfrom pathlib import Path\nimport hashlib,json,argparse,math\nfrom dataclasses import asdict\nfrom .brief_acceptance import PlannedOutput,Attempt,evaluate\n\n\ndef import_attempts(plan, runs):\n    """runs is [(manifest_path, {integer_variant_index: planned_output_id}), ...].\n\n    List runs in actual chronological attempt order. No retry is inferred from\n    filenames. Repeated project IDs are rejected to avoid charging recovery twice.\n    Failed/partial artifacts remain failed; unknown provider outcome stays unknown.\n    Legacy variants without per-call evidence are refused rather than priced at0.\n    """\n    plan=list(plan);evaluate(plan,[],[]);planned={p.output_id:p for p in plan}\n    attempts=[];seen_ids=set();seen_projects=set();references=[]\n    for manifest_path,mapping in runs:\n        path=Path(manifest_path)\n        if path.is_symlink() or not path.is_file() or path.stat().st_size>5_000_000:\n            raise ValueError(\'bounded regular pipeline manifest required\')\n        raw=path.read_bytes();m=json.loads(raw)\n        if not isinstance(mapping,dict) or not mapping or any(type(k)is not int for k in mapping):\n            raise ValueError(\'explicit integer variant-index mapping required\')\n        recorded={i for i,v in enumerate(m[\'variants\']) if isinstance(v.get(\'provider_evidence\'),dict)}\n        if recorded and set(mapping)!=recorded:\n            raise ValueError(\'map every recorded provider attempt, including failures; no cherry-picked billing denominator\')\n        source_hash=m.get(\'image_intake\',{}).get(\'source_sha256\')\n        for i,output_id in mapping.items():\n            if output_id not in planned or not 0<=i<len(m[\'variants\']):raise ValueError(\'unknown planned output or variant index\')\n            p=planned[output_id];v=m[\'variants\'][i];e=v.get(\'provider_evidence\')\n            if not isinstance(e,dict) or e.get(\'schema\')!=\'provider-attempt.v1\':raise ValueError(\'legacy/missing per-call provider evidence is not a zero-cost run\')\n            if p.source_sha256 is None or p.source_sha256!=source_hash or p.template_id!=m[\'template\'][\'key\'] or p.method!=e.get(\'model\'):\n                raise ValueError(\'preregistered source/template/model mismatch\')\n            aid=e.get(\'attempt_id\');project=e.get(\'project_id\')\n            if not isinstance(aid,str) or not aid or aid in seen_ids:raise ValueError(\'duplicate/missing provider attempt ID\')\n            seen_ids.add(aid)\n            if project:\n                if project in seen_projects:raise ValueError(\'repeated provider project: recovery cannot be counted as a new charge\')\n                seen_projects.add(project)\n            credit=e.get(\'credits_charged\')\n            if credit is not None and (type(credit) not in (int,float) or not math.isfinite(credit) or credit<0):raise ValueError(\'invalid recorded charge\')\n            terminal=e.get(\'terminal_detail_observed\') is True and e.get(\'provider_status\') in (\'complete\',\'error\',\'canceled\')\n            if not terminal and credit is not None:raise ValueError(\'nonterminal charge cannot be treated as settled\')\n            complete=v.get(\'success\') is True and e.get(\'pipeline_completed\') is True\n            digest=None\n            if complete:\n                image=Path(v[\'output\']);digest=e.get(\'output_sha256\')\n                if not image.is_absolute():raise ValueError(\'absolute recorded artifact path required; relocate through explicit evidence recovery first\')\n                if image.is_symlink() or not image.is_file() or image.stat().st_size>40_000_000 or hashlib.sha256(image.read_bytes()).hexdigest()!=digest:\n                    raise ValueError(\'successful artifact bytes are missing or changed\')\n                if e.get(\'provider_status\')!=\'complete\':raise ValueError(\'successful artifact conflicts with provider outcome\')\n            status=\'success\' if complete else \'failed\' if terminal else \'unknown\'\n            attempts.append(Attempt(output_id,aid,status,v.get(\'seconds\'),credit,digest,\'\' if complete else \'Recorded pipeline failure/partial artifact\' if terminal else \'Provider outcome unresolved; do not resubmit automatically\'))\n            references.append(dict(attempt_id=aid,manifest=str(path),manifest_sha256=hashlib.sha256(raw).hexdigest(),variant_index=i,project_id=project,mock=e.get(\'mock\'),receipt_path=e.get(\'receipt_path\')))\n    evaluate(plan,attempts,[])\n    return attempts,references\n\n\ndef main(argv=None):\n    p=argparse.ArgumentParser(description=__doc__)\n    mode=p.add_mutually_exclusive_group(required=True)\n    mode.add_argument(\'--recover-stage\',type=Path,help=\'explicitly recover this retained staging directory to --output without overwriting\')\n    mode.add_argument(\'--validate-output\',action=\'store_true\',help=\'verify --output and references against their completion marker\')\n    mode.add_argument(\'--input\',type=Path,help=\'JSON with explicit plan, ordered runs (manifest, mapping), as_of; optional charge_credit_system declaration\')\n    p.add_argument(\'--output\',type=Path,required=True)\n    a=p.parse_args(argv)\n    try:\n        from .evidence_pair import export_pair,recover_pair,validate_pair\n        if a.validate_output:\n            commit=validate_pair(a.output);print(\'Verified complete payload/reference pair hashes. Persistence: \'+commit.get(\'persistence\',\'legacy_unspecified\')+\'; no power-loss guarantee.\');return\n        if a.recover_stage:\n            commit=recover_pair(a.recover_stage,a.output);print(\'Recovered and verified complete pair; staging retained. Original persistence: \'+commit.get(\'persistence\',\'legacy_unspecified\')+\'; no power-loss guarantee.\');return\n        if a.input.is_symlink() or not a.input.is_file() or a.input.stat().st_size>2_000_000:raise ValueError(\'bounded input required\')\n        request=json.loads(a.input.read_text())\n        if set(request)-{\'plan\',\'runs\',\'as_of\',\'charge_credit_system\'}:raise ValueError(\'unknown import fields\')\n        plan=[PlannedOutput(**v) for v in request[\'plan\']]\n        runs=[]\n        for r in request[\'runs\']:\n            if set(r)!={\'manifest\',\'mapping\'}:raise ValueError(\'exact run manifest/mapping fields required\')\n            pairs=r[\'mapping\']\n            if not isinstance(pairs,list) or any(set(x)!={\'variant_index\',\'output_id\'} for x in pairs):raise ValueError(\'explicit mapping pairs required\')\n            mapping={x[\'variant_index\']:x[\'output_id\'] for x in pairs}\n            if len(mapping)!=len(pairs):raise ValueError(\'duplicate mapped variant\')\n            runs.append((a.input.resolve().parent/Path(r[\'manifest\']),mapping))\n        attempts,refs=import_attempts(plan,runs)\n        result=dict(plan=[asdict(v) for v in plan],attempts=[asdict(v) for v in attempts],ratings=[],as_of=request[\'as_of\'])\n        if \'charge_credit_system\'in request:result[\'charge_credit_system\']=request[\'charge_credit_system\']\n        # Validate through existing strict adapter before saving compatible payload.\n        from .brief_economics import from_payload\n        from_payload(result)\n        result[\'evidence_pair_required\']=True\n        stage=export_pair(a.output,result,dict(records=refs,semantics=\'recorded_evidence_not_provider_or_account_authentication; ratings_and_price_not_inferred\'))\n    except (ValueError,TypeError,KeyError,OSError) as exc:p.exit(2,\'REFUSED: \'+str(exc)+\'\\n\')\n    print(\'Saved and hash-verified complete payload/reference pair; staging retained at \'+str(stage)+\'; no ratings/price/winner inferred. Persistence: \'+validate_pair(a.output)[\'persistence\']+\'; no power-loss guarantee.\')\n\n\nif __name__==\'__main__\':main()\n'
# ---- embedded module: renderello/provider_attempts.py ----
path = base / 'renderello/provider_attempts.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Explicit InSPyReNet fast CPU experiment. No automatic acceptance or download."""\nfrom pathlib import Path\nimport hashlib,json,os,time\nimport numpy as np\nfrom PIL import Image\n\nMODEL_URL=\'https://github.com/plemeri/transparent-background/releases/download/1.2.12/ckpt_fast.pth\'\nMODEL_SHA256=\'e61ed7e85763dd83a796e1d650ea78604fff524121750858c54a6b560ac58c82\'\nPACKAGE_VERSION=\'1.3.4\'\n\n\ndef _predict(rgb,checkpoint):\n    import tempfile,contextlib,sys,importlib.metadata\n    if importlib.metadata.version(\'transparent-background\')!=PACKAGE_VERSION:\n        raise RuntimeError(\'requires transparent-background==\'+PACKAGE_VERSION)\n    os.environ[\'NO_ALBUMENTATIONS_UPDATE\']=\'1\'\n    import torch\n    from transparent_background import Remover\n    torch.set_num_threads(2)\n    # Keep package config inside this project, with a fresh known publisher config.\n    # Explicit checkpoint avoids the package automatic downloader. Map output avoids\n    # its default foreground recoloring; our saved RGB always comes from the source.\n    with tempfile.TemporaryDirectory(prefix=\'.inspyrenet-\',dir=Path(__file__).resolve().parent.parent) as td:\n        old=os.environ.get(\'TRANSPARENT_BACKGROUND_FILE_PATH\')\n        os.environ[\'TRANSPARENT_BACKGROUND_FILE_PATH\']=td\n        try:\n            with contextlib.redirect_stdout(sys.stderr):\n                model=Remover(mode=\'fast\',device=\'cpu\',ckpt=str(checkpoint.resolve()),resize=\'static\',jit=False)\n                return np.array(model.process(rgb,type=\'map\'))[:,:,0]\n        finally:\n            if old is None:os.environ.pop(\'TRANSPARENT_BACKGROUND_FILE_PATH\',None)\n            else:os.environ[\'TRANSPARENT_BACKGROUND_FILE_PATH\']=old\n\n\ndef infer_file(source:Path,out:Path,checkpoint:Path):\n    if out.exists():raise ValueError(\'output exists; preserve earlier attempts\')\n    with Image.open(source) as im:\n        if im.width*im.height>4_000_000:raise ValueError(\'source exceeds 4 million pixels; no silent source resize\')\n        if min(im.size)<3:raise ValueError(\'source too small\')\n        rgb=im.convert(\'RGB\');pixels=np.array(rgb)\n    if not checkpoint.is_file():raise ValueError(\'checkpoint missing; explicitly download the official free file first\')\n    if hashlib.sha256(checkpoint.read_bytes()).hexdigest()!=MODEL_SHA256:raise ValueError(\'checkpoint SHA256 mismatch; only tested fast model accepted\')\n    started=time.monotonic()\n    try:alpha=_predict(rgb,checkpoint)\n    except ImportError as e:raise RuntimeError(\'install optional requirements-pyramid.txt in an isolated Renderello environment\') from e\n    if alpha.dtype!=np.uint8 or alpha.shape!=pixels.shape[:2] or not np.any(alpha):raise ValueError(\'invalid or empty inferred alpha\')\n    out.mkdir(parents=True)\n    for name,data in [(\'source.png\',pixels),(\'estimated-alpha.png\',alpha),(\'cutout.png\',np.dstack([pixels,alpha]))]:Image.fromarray(data).save(out/name)\n    for label,color in [(\'black\',(20,20,20)),(\'white\',(250,250,250)),(\'magenta\',(190,0,150))]:\n        bg=Image.new(\'RGB\',rgb.size,color);bg.paste(rgb,mask=Image.fromarray(alpha));bg.save(out/(label+\'-review.png\'))\n    report=dict(model=\'InSPyReNet SwinB\',mode=\'fast\',input_preprocessing=\'static 384x384 internal model scale; alpha interpolated to unchanged source size\',\n        model_url=MODEL_URL,model_sha256=MODEL_SHA256,package=\'transparent-background\',package_version=PACKAGE_VERSION,device=\'cpu\',\n        source_sha256=hashlib.sha256(source.read_bytes()).hexdigest(),source_rgb_unchanged=True,semantic_completeness=\'NOT_VERIFIED\',source_framing=\'NOT_VERIFIED\',quality=\'NOT_EVALUATED\',\n        seconds=time.monotonic()-started,alpha_kind=\'saliency map, not optical matting\',\n        warnings=[\'Fast model loses the yellow notebook cover. Do not treat this as a general low-contrast solution.\',\n          \'Silver scissors holes opened automatically, but edge color/softness and fine neck details still need review.\',\n          \'Base 1024 mode was killed without an output in this environment. This command supports only the tested fast model.\',\n          \'Original RGB is preserved, including source reflections and background color fringes. Cropped parts cannot be recovered.\'],\n        sources=[\'https://github.com/plemeri/InSPyReNet\',\'https://github.com/plemeri/transparent-background\',\'https://arxiv.org/abs/2209.09475\'])\n    report[\'output_sha256\']={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in out.iterdir() if p.is_file()}\n    (out/\'pyramid-mask-manifest.json\').write_text(json.dumps(report,indent=2));return report\n\n\ndef guarded_file(source,out,checkpoint):\n    """CPU worker, 180 second timeout and Linux-only 2.5 GiB RSS guard."""\n    import subprocess,sys,tempfile\n    cmd=[sys.executable,\'-m\',\'renderello.pyramid_mask\',\'--source\',str(source),\'--out\',str(out),\'--checkpoint\',str(checkpoint)]\n    env=os.environ.copy();env.update(OMP_NUM_THREADS=\'2\',MKL_NUM_THREADS=\'2\',NO_ALBUMENTATIONS_UPDATE=\'1\')\n    with tempfile.TemporaryFile(mode=\'w+\') as stdout,tempfile.TemporaryFile(mode=\'w+\') as stderr:\n        p=subprocess.Popen(cmd,stdout=stdout,stderr=stderr,text=True,env=env);start=time.monotonic()\n        try:\n            while p.poll() is None:\n                if time.monotonic()-start>180:raise RuntimeError(\'pyramid-mask 180 second timeout; attempt not accepted\')\n                try:rss=int(next(l.split()[1] for l in Path(f\'/proc/{p.pid}/status\').read_text().splitlines() if l.startswith(\'VmRSS:\')))\n                except (FileNotFoundError,StopIteration):rss=0\n                if rss>2560*1024:raise RuntimeError(\'pyramid-mask 2.5 GiB RSS limit; attempt not accepted\')\n                time.sleep(.08)\n        finally:\n            if p.poll() is None:\n                p.terminate()\n                try:p.wait(timeout=5)\n                except subprocess.TimeoutExpired:p.kill();p.wait()\n        stdout.seek(0);stderr.seek(0)\n        if p.returncode:raise RuntimeError(\'pyramid-mask worker failed (exit \'+str(p.returncode)+\'): \'+stderr.read()[-2000:])\n        return json.load(stdout)\n\n\ndef main():\n    import argparse\n    p=argparse.ArgumentParser(description=__doc__)\n    for name in (\'source\',\'out\',\'checkpoint\'):p.add_argument(\'--\'+name,type=Path,required=True)\n    a=p.parse_args();print(json.dumps(infer_file(a.source,a.out,a.checkpoint)))\nif __name__==\'__main__\':main()\n'
# ---- embedded module: renderello/pyramid_mask.py ----
path = base / 'renderello/pyramid_mask.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Fidelity QA. The problem statement\'s scoring rubric (1-5, fidelity primary)\nplus automated drift metrics. Metrics are computed inside the aligned product\nregion only; background differences are expected and never penalized.\n\nAlignment: Route 2 outputs embed the (possibly scaled) source crop at a known\noffset recorded by the pipeline, so we compare crop-vs-crop. Generative routes\nre-segment the output (same cutout backend) and align by bbox before comparing -\napproximate, and flagged as such in notes, because generative pixels can shift.\n"""\nfrom __future__ import annotations\n\nfrom dataclasses import asdict\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nfrom PIL import Image\n\nfrom .schemas import QAReport\n\n\ndef _rgb(img: Image.Image | np.ndarray) -> np.ndarray:\n    return np.array(img.convert("RGB")) if isinstance(img, Image.Image) else img[..., :3]\n\n\ndef masked_ssim(a: np.ndarray, b: np.ndarray, mask: np.ndarray) -> float:\n    from skimage.metrics import structural_similarity\n    m = mask > 127\n    side = min(a.shape[:2])\n    if m.sum() < 64 or side < 3:\n        return float("nan")\n    ssim_map = structural_similarity(a, b, channel_axis=2, full=True,\n                                     data_range=255, win_size=min(7, side if side % 2 else side - 1))[1]\n    return float(ssim_map[m].mean())\n\n\ndef masked_l1(a: np.ndarray, b: np.ndarray, mask: np.ndarray) -> float:\n    m = mask > 127\n    if m.sum() == 0:\n        return float("nan")\n    return float(np.abs(a.astype(np.float32) - b.astype(np.float32))[m].mean())\n\n\ndef color_drift(a: np.ndarray, b: np.ndarray, mask: np.ndarray) -> float:\n    """Bhattacharyya distance between per-channel histograms in the product\n    region. 0 = identical color distribution; label recolors push this up."""\n    m = (mask > 127).astype(np.uint8)\n    if m.sum() == 0:\n        return float("nan")\n    total = 0.0\n    for ch in range(3):\n        ha = cv2.calcHist([a], [ch], m, [64], [0, 256])\n        hb = cv2.calcHist([b], [ch], m, [64], [0, 256])\n        total += cv2.compareHist(ha, hb, cv2.HISTCMP_BHATTACHARYYA)\n    return float(total / 3.0)\n\n\ndef edge_retention(a: np.ndarray, b: np.ndarray, mask: np.ndarray) -> float:\n    """Fraction of source product edges still present in the output. Catches\n    \'melted\' label text and deformed packaging better than L1 alone."""\n    m = mask > 127\n    ea = cv2.Canny(cv2.cvtColor(a, cv2.COLOR_RGB2GRAY), 60, 160) > 0\n    eb = cv2.Canny(cv2.cvtColor(b, cv2.COLOR_RGB2GRAY), 60, 160) > 0\n    eb_d = cv2.dilate(eb.astype(np.uint8), np.ones((3, 3), np.uint8)) > 0\n    src = ea & m\n    if src.sum() == 0:\n        return float("nan")\n    return float((src & eb_d).sum() / src.sum())\n\n\ndef critical_error_checks(src: np.ndarray, out: np.ndarray,\n                          mask: np.ndarray) -> list[str]:\n    """Heuristic tripwires for the plan\'s critical-error classes. These FLAG\n    for human eye review; they do not auto-fail, per the problem statement\'s\n    \'record critical errors separately\' instruction."""\n    errors: list[str] = []\n    m = mask > 127\n    if m.sum() == 0:\n        return ["product matte empty in output alignment - manual review required"]\n    l1 = masked_l1(src, out, mask)\n    if l1 > 28:\n        errors.append(f"large product-region pixel change (L1={l1:.1f}) - possible "\n                      "redrawn label/logo; eye-check before use")\n    # Localized repaint detector: a mean over the whole matte dilutes small but\n    # fatal changes (a relabeled region). Flag when a meaningful fraction of\n    # product pixels changed substantially.\n    diff = np.abs(src.astype(np.float32) - out.astype(np.float32)).mean(axis=2)\n    pct_changed = float(((diff > 60) & m).sum() / m.sum())\n    if pct_changed > 0.08:\n        errors.append(f"{pct_changed:.0%} of product pixels changed substantially - "\n                      "localized repaint (label/logo area?) likely; eye-check")\n    drift = color_drift(src, out, mask)\n    if drift > 0.35:\n        errors.append(f"strong product color shift (Bhattacharyya={drift:.2f}) - "\n                      "packaging color may have changed")\n    er = edge_retention(src, out, mask)\n    if er == er and er < 0.55:  # NaN-safe\n        errors.append(f"only {er:.0%} of product edges retained - label text or "\n                      "packaging geometry may be deformed")\n    # A low global average can conceal a small but important label repaint.\n    # This is a review warning, never an OCR or identity verdict.\n    from .detail_qa import detail_change_review\n    detail = detail_change_review(src, out, mask)\n    if detail.get("flagged_patch_count", 0):\n        patch = detail["flagged_patches"][0]\n        errors.append(f"local product-detail change in {detail[\'flagged_patch_count\']} interior patches "\n                      f"(strongest crop-relative box={patch[\'box\']}, high-pass MAE={patch[\'highpass_mae\']:.1f}) - "\n                      "possible fine-label/texture change; compare original at full size; not verified OCR")\n    return errors\n\n\ndef matte_sanity(matte: np.ndarray, bbox: tuple[int, int, int, int],\n                 flag_frame_cut: bool = False) -> list[str]:\n    """Cutout-quality tripwires. Route 2\'s pixel comparisons cannot see a bad\n    matte (a kept background slab is source-exact by construction), so the\n    matte itself must be scored. Flags feed critical_errors (fidelity cap 2).\n    Heuristics tuned on the 21-photo licensed CPU stress set: isolated\n    products sit well under these bounds; GrabCut failures on busy or\n    low-contrast scenes trip them."""\n    flags: list[str] = []\n    H, W = matte.shape\n    x, y, w, h = bbox\n    if w == 0 or h == 0:\n        return ["cutout empty - no subject isolated"]\n    coverage = (w * h) / (W * H)\n    fill = float((matte[y:y + h, x:x + w] > 127).mean())\n    if coverage > 0.80 and fill > 0.80:\n        flags.append(f"cutout kept {coverage:.0%} of the frame as a solid slab - "\n                     "subject isolation failed (busy or low-contrast background)")\n    elif coverage > 0.65 and fill > 0.85:\n        flags.append(f"cutout covers {coverage:.0%} of frame at {fill:.0%} fill - "\n                     "probable background inclusion; eye-check before use")\n    if flag_frame_cut:  # opt-in: default behavior unchanged\n        from .boundary import frame_edge_contact\n        fe = frame_edge_contact(matte)\n        if fe["needs_review"]:\n            flags.append("product touches the source frame on " + "/".join(fe["touches"]) +\n                         " - hard straight cut edges are the photo\'s own edge; eye-check")\n    return flags\n\n\ndef fidelity_score(report: QAReport) -> int:\n    """Map metrics to the 1-5 fidelity rubric (5 = pixel-faithful). Thresholds\n    are starting points from the plan; Phase 0 measured data recalibrates them\n    (recorded honestly here as provisional)."""\n    vals = []\n    if report.ssim_product is not None and report.ssim_product == report.ssim_product:\n        vals.append(min(5, max(1, int(round((report.ssim_product - 0.70) / 0.06)) + 1)))\n    if report.color_drift is not None and report.color_drift == report.color_drift:\n        vals.append(5 if report.color_drift < 0.08 else\n                    4 if report.color_drift < 0.18 else\n                    3 if report.color_drift < 0.30 else\n                    2 if report.color_drift < 0.45 else 1)\n    if report.edge_retained is not None and report.edge_retained == report.edge_retained:\n        vals.append(5 if report.edge_retained > 0.95 else\n                    4 if report.edge_retained > 0.85 else\n                    3 if report.edge_retained > 0.70 else\n                    2 if report.edge_retained > 0.55 else 1)\n    score = int(round(float(np.mean(vals)))) if vals else 3\n    if report.critical_errors:\n        score = min(score, 2)  # any critical flag caps fidelity per the rubric\n    return max(1, min(5, score))\n\n\ndef evaluate(source: Image.Image, output_path: Path, route: str, template: str,\n             seed: int, src_crop_box: tuple[int, int, int, int] | None = None,\n             out_region: tuple[int, int, int, int] | None = None,\n             matte: np.ndarray | None = None, notes: str = "") -> QAReport:\n    """Compare source crop to the aligned output region inside the matte."""\n    report = QAReport(source=str(source), output=str(output_path), route=route,\n                      template=template, seed=seed, notes=notes)\n    out = np.array(Image.open(output_path).convert("RGB"))\n    src_full = _rgb(source)\n    if src_crop_box is None or matte is None:\n        report.notes += " | no alignment supplied; metrics skipped, eye review only"\n        report.critical_errors.append("unscored: no product alignment")\n        report.score_fidelity = fidelity_score(report)\n        return report\n    x, y, w, h = src_crop_box\n    src = src_full[y:y + h, x:x + w]\n    msk = matte[y:y + h, x:x + w]\n    if out_region is not None:\n        ox, oy, ow, oh = out_region\n        region = out[oy:oy + oh, ox:ox + ow]\n        region = cv2.resize(region, (w, h), interpolation=cv2.INTER_AREA)\n        out = np.zeros_like(src_full)\n        out[:] = 0\n        out[y:y + h, x:x + w] = region\n        out = out[y:y + h, x:x + w]\n    else:\n        out = cv2.resize(out, (src_full.shape[1], src_full.shape[0]),\n                         interpolation=cv2.INTER_AREA)[y:y + h, x:x + w]\n    report.ssim_product = masked_ssim(src, out, msk)\n    report.l1_product = masked_l1(src, out, msk)\n    report.color_drift = color_drift(src, out, msk)\n    report.edge_retained = edge_retention(src, out, msk)\n    report.critical_errors = critical_error_checks(src, out, msk)\n    report.score_fidelity = fidelity_score(report)\n    return report\n\n\ndef report_to_csv_row(r: QAReport) -> dict:\n    d = asdict(r)\n    d["critical_errors"] = " ; ".join(r.critical_errors)\n    return d\n'
# ---- embedded module: renderello/qa.py ----
path = base / 'renderello/qa.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Explicit CPU seed-map correction. Human/engineer input, never automatic success.\nSeed values: 0 definite BG,64 probable BG,128 unchanged,192 probable FG,255 definite FG.\n"""\nfrom __future__ import annotations\nimport argparse,hashlib,json,time\nfrom pathlib import Path\nimport cv2\nimport numpy as np\nfrom PIL import Image\n\n\ndef _gray(path:Path,size:tuple[int,int])->np.ndarray:\n    with Image.open(path) as im:\n        if im.mode!=\'L\' or im.size!=size:\n            raise ValueError(\'alpha and seed map must be source-sized 8-bit grayscale; no resize\')\n        return np.array(im)\n\n\ndef refine(source:Path,alpha:Path,seeds:Path,out:Path,iters:int=5)->dict:\n    if out.exists():raise ValueError(\'output directory exists; use a fresh directory\')\n    if not 1<=iters<=8:raise ValueError(\'iterations must be1-8\')\n    with Image.open(source) as image:im=np.array(image.convert(\'RGB\'))\n    if im.shape[0]*im.shape[1]>4_000_000:raise ValueError(\'seed refinement capped at4million source pixels\')\n    size=(im.shape[1],im.shape[0]);a=_gray(alpha,size);s=_gray(seeds,size)\n    if not set(np.unique(s)).issubset({0,64,128,192,255}):raise ValueError(\'seed values must be0,64,128,192,255\')\n    if not np.any(s==0) or not np.any(s==255):raise ValueError(\'explicit definite foreground AND background seeds required\')\n    m=np.where(a>127,cv2.GC_PR_FGD,cv2.GC_PR_BGD).astype(\'uint8\')\n    m[a>245]=cv2.GC_FGD\n    for value,label in [(0,cv2.GC_BGD),(64,cv2.GC_PR_BGD),(192,cv2.GC_PR_FGD),(255,cv2.GC_FGD)]:m[s==value]=label\n    t=time.monotonic()\n    cv2.grabCut(cv2.cvtColor(im,cv2.COLOR_RGB2BGR),m,None,np.zeros((1,65),np.float64),np.zeros((1,65),np.float64),iters,cv2.GC_INIT_WITH_MASK)\n    result=np.where((m==1)|(m==3),255,0).astype(\'uint8\')\n    if not np.any(result>127) or not np.any(result<128):raise ValueError(\'refinement produced empty/full mask\')\n    # Only one pixel blur, then restore exact definite labels. No color correction.\n    result=cv2.GaussianBlur(result,(3,3),0);result[s==0]=0;result[s==255]=255\n    out.mkdir(parents=True)\n    Image.fromarray(a).save(out/\'initial-alpha.png\');Image.fromarray(s).save(out/\'seed-map.png\')\n    Image.fromarray(result).save(out/\'refined-alpha.png\');rgba=Image.fromarray(im);rgba.putalpha(Image.fromarray(result));rgba.save(out/\'refined-cutout.png\')\n    bg=Image.new(\'RGB\',size,(218,236,223));bg.paste(Image.fromarray(im),(0,0),Image.fromarray(result));bg.save(out/\'review-preview.png\')\n    digest=lambda p:hashlib.sha256(p.read_bytes()).hexdigest()\n    manifest={\'status\':\'REVIEW_REQUIRED\',\'backend\':\'seeded_grabcut_assisted\',\'source\':str(source.resolve()),\n              \'source_sha256\':digest(source),\'initial_alpha_sha256\':digest(alpha),\'seed_map_sha256\':digest(seeds),\n              \'alpha_sha256\':digest(out/\'refined-alpha.png\'),\'alpha\':\'refined-alpha.png\',\n              \'iterations\':iters,\'seconds\':time.monotonic()-t,\n              \'warning\':\'Engineer/human-seeded correction, not automatic model success or recovered GPU alpha. Inspect clipping, missing products, halos and frame cuts.\'}\n    (out/\'mask-manifest.json\').write_text(json.dumps(manifest,indent=2));return manifest\n\n\ndef main():\n    p=argparse.ArgumentParser(description=__doc__)\n    for name in [\'source\',\'alpha\',\'seeds\',\'out\']:p.add_argument(\'--\'+name,type=Path,required=True)\n    p.add_argument(\'--iters\',type=int,default=5);a=p.parse_args()\n    print(json.dumps(refine(a.source,a.alpha,a.seeds,a.out,a.iters)));return 0\n\nif __name__==\'__main__\':raise SystemExit(main())\n'
# ---- embedded module: renderello/refine_mask.py ----
path = base / 'renderello/refine_mask.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Experimental clean-plate refractive-difference compositor, not an optics estimator.\nOriginal implementation of a limited image formation model in linear sRGB.\nNo third-party network, dataset or implementation is bundled.\n"""\nfrom pathlib import Path\nimport json\nimport hashlib\nfrom io import BytesIO\nimport numpy as np\nfrom PIL import Image\nfrom .intent_mask import sha\n\nLIMITATIONS=[\n \'Experimental transfer requires externally supplied inspected refractive flow and transmission, not inferred from a product photo.\',\n \'Single background-plane correspondence and fixed additive residual only; not full 3D refraction, changed reflections, caustics, relighting or pose.\',\n \'Clean plate must match source camera, framing, exposure and lighting. Hash checks bind files, not physical alignment.\',\n \'Transparent product pixels necessarily change with the background; only zero-transmission pixels are source-exact.\',\n \'Synthetic validation is not a real-glass quality pass.\'\n]\n\n\ndef decode(rgb):\n a=np.asarray(rgb,dtype=np.float64)/255\n return np.where(a<=.04045,a/12.92,((a+.055)/1.055)**2.4)\n\n\ndef encode(a):\n a=np.asarray(a,dtype=np.float64)\n return np.rint(np.clip(np.where(a<=.0031308,12.92*a,1.055*np.maximum(a,0)**(1/2.4)-.055),0,1)*255).astype(np.uint8)\n\n\ndef warp(image,flow,support):\n h,w=image.shape[:2]\n if flow.shape!=(h,w,2) or not np.issubdtype(flow.dtype,np.floating) or not np.isfinite(flow).all():raise ValueError(\'finite floating HxWx2 flow required\')\n yy,xx=np.mgrid[:h,:w];x=xx+flow[:,:,0];y=yy+flow[:,:,1]\n if np.any(support & ((x<0)|(x>w-1)|(y<0)|(y>h-1))):raise ValueError(\'refracted sampling outside background plane; no border fabrication\')\n x=np.clip(x,0,w-1);y=np.clip(y,0,h-1);x0=np.floor(x).astype(int);y0=np.floor(y).astype(int);x1=np.minimum(x0+1,w-1);y1=np.minimum(y0+1,h-1);wx=(x-x0)[:,:,None];wy=(y-y0)[:,:,None]\n return (image[y0,x0]*(1-wx)+image[y0,x1]*wx)*(1-wy)+(image[y1,x0]*(1-wx)+image[y1,x1]*wx)*wy\n\n\ndef transfer_arrays(source,plate,target,flow,transmission,support):\n if source.shape!=plate.shape or source.shape!=target.shape or source.ndim!=3 or source.shape[2]!=3:raise ValueError(\'matching RGB images required\')\n if source.dtype!=np.uint8 or plate.dtype!=np.uint8 or target.dtype!=np.uint8:raise ValueError(\'uint8 sRGB image arrays required\')\n h,w=source.shape[:2]\n if transmission.shape!=(h,w,3) or not np.isfinite(transmission).all() or np.any((transmission<0)|(transmission>1)):raise ValueError(\'finite HxWx3 transmission in0..1 required\')\n if support.shape!=(h,w) or support.dtype!=np.bool_ or not np.any(support):raise ValueError(\'nonempty boolean support required\')\n src=decode(source);old=warp(decode(plate),flow,support);new=warp(decode(target),flow,support)\n residual=src-transmission*old\n # A negative radiance residual indicates incompatible plate/transmission under this model.\n if np.any(residual[support]<-.02):raise ValueError(\'negative source residual; plate/transmission incompatible with additive model\')\n prediction=src+transmission*(new-old)\n clipped=np.any((prediction<0)|(prediction>1),axis=2)&support\n if np.any(clipped):raise ValueError(\'target exceeds linear radiance gamut; no silent clipping\')\n result=target.copy();result[support]=encode(prediction[support]);protected=support & np.all(transmission==0,axis=2)\n result[protected]=source[protected]\n return result,dict(support_pixels=int(support.sum()),zero_transmission_pixels=int(protected.sum()),zero_transmission_rgb_exact=bool(np.array_equal(result[protected],source[protected])),out_of_gamut_pixels=0,linear_residual_min=float(residual[support].min()),flow_nonzero_pixels=int((support & np.any(flow!=0,axis=2)).sum()))\n\n\ndef _snapshot(path,limit):\n # Decode exactly the bytes whose hash was checked, not a later file read.\n with Path(path).open(\'rb\') as stream:data=stream.read(limit+1)\n if len(data)>limit:raise ValueError(\'input exceeds bounded byte snapshot: \'+Path(path).name)\n return data\n\n\ndef transfer_files(source,plate,target,flow,transmission,support,spec,out,*,confirm_assumptions=False):\n if not confirm_assumptions:raise ValueError(\'explicit limited-model confirmation required\')\n if out.exists():raise ValueError(\'output exists; preserve previous experiment\')\n paths=dict(source=source,plate=plate,target=target,flow=flow,transmission=transmission,support=support)\n spec_bytes=_snapshot(spec,64_000)\n d=json.loads(spec_bytes)\n if d.get(\'schema\')!=\'renderello-refractive-transfer-v1\' or d.get(\'color_space\')!=\'srgb\' or d.get(\'flow_convention\')!=\'source_pixel_to_background_dxdy_pixels\':raise ValueError(\'unsupported optics specification\')\n if d.get(\'capture_alignment_reviewed\') is not True or d.get(\'maps_reviewed\') is not True:raise ValueError(\'capture and map review declarations required\')\n if d.get(\'material\')!=\'transparent\' or d.get(\'framing\')!=\'complete\':raise ValueError(\'complete transparent source declaration required\')\n if d.get(\'map_origin\') not in (\'synthetic_known\',\'external_reviewed\'):raise ValueError(\'explicit map origin required\')\n snapshots={name:_snapshot(p,80_000_000 if name==\'flow\'else 32_000_000)for name,p in paths.items()}\n for name,data in snapshots.items():\n  if hashlib.sha256(data).hexdigest()!=d.get(\'input_sha256\',{}).get(name):raise ValueError(\'source-bound input hash mismatch: \'+name)\n arrays={}\n for name in (\'source\',\'plate\',\'target\',\'transmission\',\'support\'):\n  with Image.open(BytesIO(snapshots[name])) as im:\n   if im.width*im.height>4_194_304:raise ValueError(\'image exceeds4MP experimental bound\')\n   if im.mode!=(\'L\' if name==\'support\' else \'RGB\'):raise ValueError(\'RGB images/transmission and L binary support required\')\n   arrays[name]=np.array(im)\n if d.get(\'image_size\')!=[arrays[\'source\'].shape[1],arrays[\'source\'].shape[0]]:raise ValueError(\'source dimensions mismatch\')\n if not np.isin(arrays[\'support\'],[0,255]).all():raise ValueError(\'support must be binary0/255; transmission carries transparency\')\n f=np.load(BytesIO(snapshots[\'flow\']),allow_pickle=False)\n if not isinstance(f,np.ndarray):raise ValueError(\'plain NPY flow required\')\n result,stats=transfer_arrays(arrays[\'source\'],arrays[\'plate\'],arrays[\'target\'],f,arrays[\'transmission\'].astype(float)/255,arrays[\'support\']==255)\n report=dict(schema=\'renderello-refractive-transfer-result-v1\',spec_sha256=hashlib.sha256(spec_bytes).hexdigest(),input_read_semantics=\'bounded_immutable_byte_snapshots_hashed_before_decode\',input_sha256=d[\'input_sha256\'],map_origin=d[\'map_origin\'],method=\'linear_sRGB_source_plus_transmission_times_warped_background_difference\',quality=\'NOT_EVALUATED\',real_glass_validated=False,limitations=LIMITATIONS,**stats)\n out.mkdir(parents=True);Image.fromarray(result).save(out/\'candidate.png\');report[\'output_sha256\']={\'candidate.png\':sha(out/\'candidate.png\')};(out/\'refractive-transfer-manifest.json\').write_text(json.dumps(report,indent=2));return report\n'
# ---- embedded module: renderello/refractive_transfer.py ----
path = base / 'renderello/refractive_transfer.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""IC-Light relighting wrapper (GPU, optional). Owning repo:\nhuggingface.co/lllyasviel/ic-light - code Apache-2.0, weights OpenRAIL-M.\n\nIC-Light\'s public release ships as a Gradio demo over a diffusers fork rather\nthan a pip package, so this wrapper isolates the integration behind one\nfunction. Until a GPU host with the IC-Light environment exists, callers get a\ndeterministic CPU \'grading fallback\' (directional luminance + warmth shift)\nand the result is labeled fallback=True so QA never mistakes it for the real\nrelight pass.\n"""\nfrom __future__ import annotations\n\nimport numpy as np\nfrom PIL import Image\n\n\ndef relight_available() -> bool:\n    try:\n        import torch  # noqa: F401\n        return torch.cuda.is_available()\n    except ImportError:\n        return False\n\n\ndef grading_fallback(img: Image.Image, light_direction: str = "left",\n                     warmth: float = 6.0, strength: float = 0.18) -> Image.Image:\n    """CPU stand-in: directional luminance gradient + mild color-temperature\n    shift. Labeled everywhere as a fallback, never as IC-Light output."""\n    arr = np.array(img.convert("RGB")).astype(np.float32)\n    h, w = arr.shape[:2]\n    if light_direction in ("left", "right"):\n        g = np.linspace(1, -1, w, dtype=np.float32)[None, :, None]\n        if light_direction == "right":\n            g = -g\n    else:\n        g = np.linspace(1, -1, h, dtype=np.float32)[:, None, None]\n    arr *= 1.0 + g * strength\n    arr[..., 0] = np.clip(arr[..., 0] + warmth, 0, 255)   # warm: +R\n    arr[..., 2] = np.clip(arr[..., 2] - warmth * 0.6, 0, 255)\n    return Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))\n\n\ndef relight(img: Image.Image, template_lighting: str, prefer_gpu: bool = False\n            ) -> tuple[Image.Image, bool]:\n    """Returns (image, used_fallback). Real IC-Light integration lands with the\n    GPU host; until then every caller receives the labeled fallback."""\n    if prefer_gpu and relight_available():\n        # Integration point: IC-Light foreground-conditioned relight with the\n        # template\'s lighting description as the light prompt. Kept explicit so\n        # the swap is one edit on the GPU host.\n        raise NotImplementedError("IC-Light environment not installed on this "\n                                  "host; run with prefer_gpu=False (fallback) "\n                                  "or set up the ic-light environment first")\n    direction = "left" if "left" in template_lighting else (\n        "right" if "right" in template_lighting else "top")\n    warmth = 8.0 if any(k in template_lighting for k in\n                        ("warm", "tungsten", "3800K", "4300K", "4800K")) else 2.0\n    return grading_fallback(img, direction, warmth), True\n'
# ---- embedded module: renderello/relight.py ----
path = base / 'renderello/relight.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Portable generated-scene evidence gallery, separate from benchmark acceptance.\n\nInputs are local review declarations, never authenticated user permission.\nAll referenced bytes are verified before output; scene attempts never pad the\nproduct, benchmark, template or source-photo denominator. No model invocation.\n"""\nfrom pathlib import Path\nimport hashlib,html,json,re\n\nDIMENSIONS=(\'product_label_fidelity\',\'photorealism\',\'shadow_contact_realism\',\n    \'material_edges\',\'composition\',\'template_distinctness\',\'publish_ready_usefulness\')\n\n\ndef _text(value,name,limit=4000):\n    if not isinstance(value,str)or not value.strip()or len(value)>limit:\n        raise ValueError(\'bounded nonempty \'+name+\' required\')\n    return value\n\n\ndef _asset(root,entry):\n    if not isinstance(entry,dict)or set(entry)!={\'path\',\'sha256\'}:\n        raise ValueError(\'asset path and exact SHA256 required\')\n    raw=entry[\'path\'];h=entry[\'sha256\']\n    if not isinstance(raw,str)or not raw or \'\\\\\'in raw or Path(raw).is_absolute():\n        raise ValueError(\'relative portable asset path required\')\n    if any(p in (\'\',\'.\',\'..\')for p in raw.split(\'/\')):\n        raise ValueError(\'non-traversing asset path required\')\n    path=(root/raw).resolve()\n    if not path.is_relative_to(root):raise ValueError(\'asset escapes gallery root\')\n    if not isinstance(h,str)or not re.fullmatch(\'[a-f0-9]{64}\',h):\n        raise ValueError(\'encoded asset SHA256 required\')\n    if path.stat().st_size>20_000_000:raise ValueError(\'bounded20MBgalleryasset\')\n    data=path.read_bytes()\n    if hashlib.sha256(data).hexdigest()!=h:raise ValueError(\'asset bytes changed\')\n    # No conversion, resizing or hidden image orientation normalization here.\n    from PIL import Image\n    from io import BytesIO\n    with Image.open(BytesIO(data))as im:\n        if im.format!=\'PNG\'or im.mode not in (\'RGB\',\'RGBA\')or im.width*im.height>4_000_000 or getattr(im,\'n_frames\',1)!=1 or im.getexif().get(274,1)!=1:\n            raise ValueError(\'single-frame normalized boundedRGB/RGBA PNG required\')\n        im.load()\n    return raw\n\n\ndef build_research_gallery(manifest_path,out_html):\n    manifest_path=Path(manifest_path);root=manifest_path.parent.resolve();out_html=Path(out_html)\n    if out_html.parent.resolve()!=root:raise ValueError(\'gallery must travel beside manifest\')\n    if out_html.exists():raise FileExistsError(\'preserve prior gallery\')\n    if manifest_path.stat().st_size>1_000_000:raise ValueError(\'bounded1MBmanifest\')\n    m=json.loads(manifest_path.read_text())\n    if set(m)!={\'schema\',\'title\',\'note\',\'cases\'}or m[\'schema\']!=\'renderello-research-gallery.v1\':\n        raise ValueError(\'research gallery schema required\')\n    _text(m[\'title\'],\'title\',200);_text(m[\'note\'],\'note\',8000)\n    if not isinstance(m[\'cases\'],list)or not 1<=len(m[\'cases\'])<=256:\n        raise ValueError(\'1..256cases required\')\n    cards=[];seen=set();scene_hashes=set();product_hashes=set();sources=set();scene_ids=set();product_ids=set();scored=0;first_scene=0;first_product=0\n    for case in m[\'cases\']:\n        allowed={\'id\',\'kind\',\'attempt\',\'intent\',\'image\',\'source\',\'source_id\',\'note\',\'caption\',\'scores\',\'reviewer\',\'critical_errors_status\'}\n        if not isinstance(case,dict)or set(case)!=allowed:raise ValueError(\'explicit case fields required\')\n        ident=case[\'id\']\n        if not isinstance(ident,str)or not re.fullmatch(\'[a-z0-9][a-z0-9_-]{0,79}\',ident)or ident in seen:\n            raise ValueError(\'unique safe case id required\')\n        seen.add(ident)\n        kind=case[\'kind\'];attempt=case[\'attempt\']\n        if kind not in (\'scene_only\',\'product_composite\')or attempt not in (\'first_try\',\'retry\'):\n            raise ValueError(\'case kind/attempt required\')\n        for key in (\'intent\',\'note\',\'caption\',\'critical_errors_status\'):_text(case[key],key)\n        image=_asset(root,case[\'image\']);source=case[\'source\'];score=case[\'scores\']\n        if kind==\'scene_only\':\n            if source is not None or score is not None or case[\'source_id\']is not None or case[\'reviewer\']!=\'assistant_scene_inspection\':\n                raise ValueError(\'scene-only cannot claim product scores/source coverage\')\n            scene_ids.add(ident);scene_hashes.add(case[\'image\'][\'sha256\']);first_scene+=attempt==\'first_try\'\n        else:\n            _text(case[\'source_id\'],\'source identity\',100);source_path=_asset(root,source)\n            if case[\'reviewer\']!=\'assistant_not_blind_or_independent\':raise ValueError(\'explicit assistant scoring label required\')\n            if not isinstance(score,dict)or set(score)!=set(DIMENSIONS)or any(type(v)is not int or not 1<=v<=5 for v in score.values()):\n                raise ValueError(\'all seven assistant dimensions require integer1..5scores\')\n            product_ids.add(ident);product_hashes.add(case[\'image\'][\'sha256\']);sources.add(source[\'sha256\']);scored+=1;first_product+=attempt==\'first_try\'\n        esc=html.escape\n        pics=(f\'<figure><img loading="lazy" src="{esc(source_path,quote=True)}" alt="Original source product"><figcaption>Source input; capture origin not verified</figcaption></figure>\'if kind==\'product_composite\'else\'\')\n        pics+=f\'<figure><img loading="lazy" src="{esc(image,quote=True)}" alt="{esc(ident,quote=True)}"><figcaption>{esc(case["caption"])}</figcaption></figure>\'\n        scores=(\'<dl>\'+\'\'.join(f\'<dt>{esc(k.replace("_"," "))}</dt><dd>{v}/5</dd>\'for k,v in score.items())+\'</dl>\')if score is not None else\'<p>No product score. Scene-only attempt.</p>\'\n        cards.append(f\'<article id="{ident}"><h2>{esc(case["intent"])}</h2><p class="tag">{esc(kind)} | {esc(attempt)} | {esc(ident)}</p><p class="status">RESEARCH ONLY - NOT ACCEPTED</p><div class="pair">{pics}</div><p>{esc(case["note"])}</p>{scores}<p class="fine">Reviewer: {esc(case["reviewer"])}. Critical errors: {esc(case["critical_errors_status"])}</p></article>\')\n    counts=dict(scene_attempts=len(scene_ids),scene_distinct_encoded_artifacts=len(scene_hashes),first_try_scene_attempts=first_scene,\n        product_composite_attempts=len(product_ids),product_distinct_encoded_artifacts=len(product_hashes),first_try_product_composite_attempts=first_product,\n        distinct_source_encoded_artifacts=len(sources),assistant_seven_dimension_reviews=scored,\n        benchmark_accepts=0,polished_template_accepts=0,benchmark_evaluated=False,\n        caveat=\'Artifact hashes are byte identity, not visually distinct templates or verified photo capture. All gallery cases are research-only and NOT ACCEPTED. Counts are not full original benchmark coverage.\')\n    doc=\'\'\'<!doctype html><html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>\'\'\'+html.escape(m[\'title\'])+\'\'\'</title><style>\n*{box-sizing:border-box}body{max-width:1180px;margin:32px auto;padding:0 18px;font:16px/1.5 system-ui,sans-serif;color:#222;background:#f5f4ef}h1{font-size:30px;line-height:1.15}h2{font-size:21px}header,article{background:white;border:1px solid #d3d3cd;border-radius:12px;padding:20px;margin:22px 0}figure{margin:0;min-width:0}img{display:block;max-width:100%;width:100%;object-fit:contain;max-height:620px;background:#fafafa}figcaption,.fine{font-size:13px;color:#555;overflow-wrap:anywhere}.pair{display:grid;grid-template-columns:repeat(auto-fit,minmax(min(320px,100%),1fr));gap:18px}.status{color:#8b3529;font-weight:650}.tag{font-size:13px;color:#555;overflow-wrap:anywhere}dl{display:grid;grid-template-columns:minmax(140px,1fr) 50px;max-width:420px;font-size:14px}dt,dd{margin:0;padding:3px 0}summary{cursor:pointer;padding:8px 0}.counts{display:grid;grid-template-columns:repeat(auto-fit,minmax(150px,1fr));gap:12px;margin:22px 0}.counts div{padding:12px;background:#efeee8;border-radius:8px}.counts b{display:block;font-size:28px}nav{display:flex;flex-wrap:wrap;gap:12px}a{color:#174b7c}pre{font:13px/1.45 ui-monospace,monospace;white-space:pre-wrap;overflow-wrap:anywhere}@media(max-width:500px){h1{font-size:25px}header,article{padding:14px}body{margin:18px auto}}</style></head><body><header><h1>\'\'\'+html.escape(m[\'title\'])+\'</h1><p>\'+html.escape(m[\'note\'])+\'</p><p class="status">No accepted template or product-photo benchmark result in this gallery.</p><p>Generated scenes and source-product composites are counted separately. Repeated attempts and failed prompts remain visible. Scores are assistant judgments, not blind or independent review.</p><div class="counts"><div><b>\'+str(counts[\'scene_attempts\'])+\'</b> scene attempts</div><div><b>\'+str(counts[\'product_composite_attempts\'])+\'</b> product composites</div><div><b>\'+str(counts[\'distinct_source_encoded_artifacts\'])+\'</b> source files</div><div><b>0</b> accepted templates</div></div><details><summary>Exact counts and counting limits</summary><pre>\'+html.escape(json.dumps(counts,indent=2))+\'</pre></details><details><summary>Jump to an attempt</summary><nav>\'+\'\'.join(f\'<a href="#{c["id"]}">{html.escape(c["id"])}</a>\'for c in m[\'cases\'])+\'</nav></details></header>\'+\'\'.join(cards)+\'</body></html>\'\n    # Exclusive create only after all validation and byte checks have passed.\n    with out_html.open(\'x\',encoding=\'utf-8\')as f:f.write(doc)\n    return counts\n\n\nif __name__==\'__main__\':\n    import argparse\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'manifest\');parser.add_argument(\'output\')\n    args=parser.parse_args();print(json.dumps(build_research_gallery(args.manifest,args.output),indent=2))\n'
# ---- embedded module: renderello/research_gallery.py ----
path = base / 'renderello/research_gallery.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Read process and container memory counters without a monitoring service."""\nfrom pathlib import Path\nimport resource\nimport sys\n\n\ndef memoryUsage():\n    """Return counters with explicit scope; no claim of per-job isolation."""\n    scale = 1 if sys.platform == "darwin" else 1024\n    data = {\n        "processPeakBytes": int(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss * scale),\n        "scope": "process lifetime peak; container counters cover all processes",\n    }\n    for label, locations in {\n        "containerCurrentBytes": ["/sys/fs/cgroup/memory.current", "/sys/fs/cgroup/memory/memory.usage_in_bytes"],\n        "containerPeakBytes": ["/sys/fs/cgroup/memory.peak", "/sys/fs/cgroup/memory/memory.max_usage_in_bytes"],\n        "containerLimitBytes": ["/sys/fs/cgroup/memory.max", "/sys/fs/cgroup/memory/memory.limit_in_bytes"],\n    }.items():\n        for location in locations:\n            try:\n                value = Path(location).read_text().strip()\n                data[label] = int(value) if value != "max" else None\n                break\n            except (OSError, ValueError):\n                continue\n    return data\n'
# ---- embedded module: renderello/resources.py ----
path = base / 'renderello/resources.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Blank review-sheet export and strict scored-sheet import. No automatic ratings.\nRater labels record a claim; they do not authenticate a human or grant authority.\n"""\nfrom __future__ import annotations\n\nimport csv\nimport hashlib\nimport json\nimport os\n\nfrom .artifact_paths import run_artifact\nfrom pathlib import Path\n\nDIMENSIONS = ("product_label_fidelity", "photorealism", "shadow_contact_realism",\n              "material_edges", "composition", "template_distinctness",\n              "publish_ready_usefulness")\nIDENTITY = ("source", "output", "output_sha256", "template", "route", "seed")\nFIELDS = (*IDENTITY, *DIMENSIONS, "no_critical", "critical_errors", "rater", "notes")\nRATERS = {"owner", "independent_human", "agent_eye"}\n\n\ndef export_sheet(run_dirs: list[Path], output: Path, portable: bool = False) -> Path:\n    """Export existing readable outputs once. Never overwrite a scored sheet."""\n    rows = []\n    seen = set()\n    for run in run_dirs:\n        manifest = json.loads((run / "manifest.json").read_text())\n        for variant in manifest.get("variants", []):\n            p = run_artifact(run, variant.get("output", ""))\n            if p is None:\n                continue  # No image to rate; generation failures stay in manifests.\n            p = p.resolve()\n            if str(p) in seen:\n                raise ValueError(f"duplicate output: {p}")\n            seen.add(str(p))\n            rows.append({"source": manifest.get("source", ""), "output": (Path(os.path.relpath(p, output.parent.resolve())).as_posix() if portable else str(p)),\n                         "output_sha256": hashlib.sha256(p.read_bytes()).hexdigest(),\n                         "template": variant.get("qa", {}).get("template", ""),\n                         "route": variant.get("route", ""), "seed": variant.get("seed", "")})\n    if not rows:\n        raise ValueError("no existing outputs to review")\n    output.parent.mkdir(parents=True, exist_ok=True)\n    with output.open("x", newline="") as f:\n        writer = csv.DictWriter(f, fieldnames=FIELDS)\n        writer.writeheader()\n        writer.writerows(rows)  # All rating fields intentionally blank.\n    return output\n\n\ndef load_scores(sheet: Path) -> dict:\n    """Validate complete ratings and exact image bytes; never infer blank values.\n\n    Returns records compatible with benchmark._summarize(human=...), plus all\n    seven dimensions. Caller still owns trusted-channel rater provenance.\n    """\n    scores = {}\n    with sheet.open(newline="") as f:\n        reader = csv.DictReader(f)\n        if not set(FIELDS) <= set(reader.fieldnames or []):\n            raise ValueError("review sheet lacks required fields")\n        for row in reader:\n            key = row["output"]\n            if key in scores:\n                raise ValueError(f"duplicate output: {key}")\n            p = Path(key)\n            if not p.is_absolute():\n                p = sheet.parent / p\n            p = p.resolve()\n            if not p.is_file() or hashlib.sha256(p.read_bytes()).hexdigest() != row["output_sha256"]:\n                raise ValueError(f"output missing or bytes changed: {key}")\n            dims = {}\n            for name in DIMENSIONS:\n                if row[name] not in {"1", "2", "3", "4", "5"}:\n                    raise ValueError(f"{key}: {name} must be a complete integer rating 1-5")\n                dims[name] = int(row[name])\n            if row["no_critical"] not in {"true", "false"}:\n                raise ValueError(f"{key}: no_critical must be true or false")\n            if row["rater"] not in RATERS:\n                raise ValueError(f"{key}: explicit rater provenance required")\n            errors = row["critical_errors"].strip()\n            no_critical = row["no_critical"] == "true"\n            if no_critical == bool(errors):\n                raise ValueError(f"{key}: critical error text conflicts with no_critical")\n            canonical_key = str(p)\n            if canonical_key in scores:\n                raise ValueError(f"duplicate output: {key}")\n            scores[canonical_key] = {"no_critical": no_critical,\n                           "publishable": dims["publish_ready_usefulness"],\n                           "rater": row["rater"], "dimensions": dims,\n                           "critical_errors": errors, "notes": row["notes"],\n                           "provenance_verified": False}\n    if not scores:\n        raise ValueError("review sheet contains no ratings")\n    return scores\n'
# ---- embedded module: renderello/review.py ----
path = base / 'renderello/review.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Opt-in reviewed alpha locks. Preserves a prior matte, not segmentation truth.\n\nOpaque-source research only, no default pipeline integration. A reviewed native\nregion can retain its initial alpha including fractional boundary coverage.\nThe solver receives binary seeds in that region; fractional pixels are restored\nexplicitly after the solve. Such restoration can create seams and needs review.\nUse prepare + guarded_file + finish for bounded real work. The array convenience\nfunction is synchronous and deliberately does not promise a memory/time guard.\n"""\nimport numpy as np\nfrom .candidate_diagnostics import native_rgb,native_alpha,digest\nfrom .matting import make_trimap,refine_arrays\n\n\ndef prepare_alpha_lock(rgb,initial_alpha,frozen,*,source_sha256,alpha_sha256,\n        frozen_sha256,reviewed=False,material=None,review_note=\'\',radius=8):\n    native_rgb(rgb);native_alpha(initial_alpha,rgb.shape[:2])\n    if initial_alpha.size>1_000_000:raise ValueError(\'1MP source budget; no resize\')\n    if reviewed is not True or material!=\'opaque\':raise ValueError(\'explicit opaque pairing/region review required\')\n    if not isinstance(review_note,str) or not review_note.strip() or len(review_note)>2000:\n        raise ValueError(\'nonempty review note up to2000characters required\')\n    if not isinstance(frozen,np.ndarray) or frozen.dtype!=np.bool_ or frozen.shape!=initial_alpha.shape or not frozen.any():\n        raise ValueError(\'nonempty native boolean frozen region required\')\n    if source_sha256!=digest(rgb) or alpha_sha256!=digest(initial_alpha) or frozen_sha256!=digest(frozen):\n        raise ValueError(\'source/alpha/region binding mismatch\')\n    if type(radius) is not int or not 1<=radius<=12:raise ValueError(\'integer1..12radius required\')\n    if not np.any(frozen&(initial_alpha>0)):raise ValueError(\'region must protect some prior foreground\')\n    tri=make_trimap(initial_alpha,radius)\n    tri[frozen&(initial_alpha==0)]=0\n    tri[frozen&(initial_alpha==255)]=255\n    # Fractional prior coverage is not a binary foreground/background claim.\n    tri[frozen&(initial_alpha>0)&(initial_alpha<255)]=128\n    if int((tri==128).sum())>200_000:raise ValueError(\'200000unknown-pixel budget exceeded\')\n    return tri,dict(schema=\'reviewed-alpha-lock.v1\',source_sha256=digest(rgb),\n        initial_alpha_sha256=digest(initial_alpha),frozen_sha256=digest(frozen),\n        trimap_sha256=digest(tri),review_note=review_note,material=material,radius=radius,\n        frozen_pixels=int(frozen.sum()),frozen_foreground_pixels=int((frozen&(initial_alpha>0)).sum()),\n        frozen_fractional_pixels=int((frozen&(initial_alpha>0)&(initial_alpha<255)).sum()),\n        source_resampled=False,accepted=False,default_pipeline_changed=False,\n        independent_ground_truth=False,optical_recovery=False,physical_relighting=False)\n\n\ndef finish_alpha_lock(rgb,initial_alpha,frozen,foreground,solved_alpha,trimap,**review):\n    expected,meta=prepare_alpha_lock(rgb,initial_alpha,frozen,**review)\n    native_rgb(foreground);native_alpha(solved_alpha,initial_alpha.shape)\n    if foreground.shape!=rgb.shape or not np.array_equal(foreground,rgb):\n        raise ValueError(\'alpha-only solver must preserve all source RGB\')\n    if not isinstance(trimap,np.ndarray) or trimap.dtype!=np.uint8 or not np.array_equal(trimap,expected):\n        raise ValueError(\'solver trimap differs from reviewed bound trimap\')\n    if np.any(solved_alpha[expected==0]!=0) or np.any(solved_alpha[expected==255]!=255):\n        raise ValueError(\'solver violated definite alpha seeds\')\n    result=solved_alpha.copy();result[frozen]=initial_alpha[frozen]\n    meta.update(foreground_sha256=digest(foreground),solved_alpha_sha256=digest(solved_alpha),\n        output_alpha_sha256=digest(result),all_source_rgb_exact=True,frozen_alpha_exact=True,\n        frozen_pixels_restored_after_solve=int((frozen&(solved_alpha!=initial_alpha)).sum()),\n        alpha_changed_pixels=int((result!=initial_alpha).sum()),\n        limitations=[\'Prior alpha is preserved in a reviewer region, not recovered or verified as truth.\',\n          \'Fractional restoration can create a transition seam; inspect the region perimeter.\',\n          \'Outside the region the solver can still erase details or include background.\',\n          \'Unchanged RGB does not prove visible product completeness or photographic quality.\',\n          \'No pose, geometry, lighting, glass or missing-part reconstruction.\'])\n    return foreground.copy(),result,expected.copy(),meta\n\n\ndef recover_locked_alpha(rgb,initial_alpha,frozen,**review):\n    tri,_=prepare_alpha_lock(rgb,initial_alpha,frozen,**review)\n    foreground,alpha,solver=refine_arrays(rgb.copy(),initial_alpha.copy(),tri.copy(),method=\'cf\',edge_colors=False)\n    out=finish_alpha_lock(rgb,initial_alpha,frozen,foreground,alpha,tri,**review)\n    out[3][\'solver\']=solver\n    return out\n'
# ---- embedded module: renderello/reviewed_alpha_lock.py ----
path = base / 'renderello/reviewed_alpha_lock.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Executable reviewed alpha-lock route with guarded CF and hash-bound inputs.\n\nResearch only. Requires explicit reviewer declarations. Never accepts quality.\nRegion file is binary L PNG; partial-valued labels are refused, not thresholded.\n"""\nfrom pathlib import Path\nimport argparse,hashlib,json\nimport numpy as np\nfrom PIL import Image\nfrom .reviewed_alpha_lock import prepare_alpha_lock,finish_alpha_lock\nfrom .matting import guarded_file\nfrom .candidate_diagnostics import digest\nfrom .reviewed_parts import audit_parts\n\n\ndef execute(source,alpha,region,out,*,source_file_sha256,alpha_file_sha256,\n        region_file_sha256,review_note,reviewed=False,material=None,radius=8):\n    source,alpha,region,out=map(Path,[source,alpha,region,out])\n    if out.exists():raise ValueError(\'output exists; preserve prior attempt\')\n    for p,h in [(source,source_file_sha256),(alpha,alpha_file_sha256),(region,region_file_sha256)]:\n        if not p.is_file() or p.stat().st_size>32*1024*1024:raise ValueError(\'input file missing or exceeds32MiB\')\n        if hashlib.sha256(p.read_bytes()).hexdigest()!=h:raise ValueError(\'input file hash mismatch: \'+p.name)\n    with Image.open(source) as im:\n        if im.mode!=\'RGB\':raise ValueError(\'normalized RGB source required; no implicit orientation/color conversion\')\n        rgb=np.array(im)\n    arrays=[]\n    for p in [alpha,region]:\n        with Image.open(p) as im:\n            if im.mode!=\'L\':raise ValueError(\'alpha and region must be native8bitL\')\n            arrays.append(np.array(im))\n    a,f=arrays\n    if not set(np.unique(f)).issubset({0,255}):raise ValueError(\'binary0/255region file required; no threshold guessing\')\n    frozen=f==255\n    review=dict(source_sha256=digest(rgb),alpha_sha256=digest(a),frozen_sha256=digest(frozen),\n        review_note=review_note,reviewed=reviewed,material=material,radius=radius)\n    tri,prep=prepare_alpha_lock(rgb,a,frozen,**review)\n    # Exclusive reserve only after all input/region declarations validate.\n    out.mkdir(parents=True,exist_ok=False)\n    (out/\'attempt.json\').write_text(json.dumps(dict(status=\'STARTED_NOT_ACCEPTED\',review=review,\n        source_file_sha256=source_file_sha256,alpha_file_sha256=alpha_file_sha256,region_file_sha256=region_file_sha256),indent=2))\n    try:\n        Image.fromarray(tri).save(out/\'reviewed-trimap.png\')\n        worker=guarded_file(source,alpha,out/\'worker\',trimap_path=out/\'reviewed-trimap.png\',method=\'cf\',radius=radius,edge_colors=False)\n        fg=np.array(Image.open(out/\'worker/foreground.png\'));solved=np.array(Image.open(out/\'worker/refined-alpha.png\'))\n        fg,matte,tri,meta=finish_alpha_lock(rgb,a,frozen,fg,solved,tri,**review)\n        part_audit=audit_parts(rgb,a,fg,matte,[dict(name=\'reviewed-alpha-region\',mask=frozen,mask_sha256=digest(frozen))],\n            source_sha256=digest(rgb),prior_alpha_sha256=digest(a),reviewed=reviewed,review_note=review_note)\n        if not part_audit[\'all_named_parts_preserved\']:raise RuntimeError(\'reviewed region preservation failed\')\n        (out/\'part-preservation.json\').write_text(json.dumps(part_audit,indent=2))\n        meta[\'part_preservation\']=part_audit\n        Image.fromarray(matte).save(out/\'locked-alpha.png\');Image.fromarray(np.dstack([fg,matte])).save(out/\'locked-cutout.png\')\n        for name,c in [(\'light\',(250,250,250)),(\'dark\',(20,20,20))]:\n            im=Image.new(\'RGB\',(rgb.shape[1],rgb.shape[0]),c);im.paste(Image.fromarray(fg),mask=Image.fromarray(matte));im.save(out/(name+\'.png\'))\n        meta.update(worker=worker,status=\'EXECUTED_NOT_ACCEPTED\',input_files=dict(source=str(source.resolve()),alpha=str(alpha.resolve()),region=str(region.resolve())),\n            input_file_sha256=dict(source=source_file_sha256,alpha=alpha_file_sha256,region=region_file_sha256),\n            output_sha256={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in out.glob(\'*.png\')})\n        (out/\'manifest.json\').write_text(json.dumps(meta,indent=2));return meta\n    except Exception as exc:\n        (out/\'refused.json\').write_text(json.dumps(dict(status=\'REFUSED_NOT_ACCEPTED\',error=str(exc),accepted=False),indent=2))\n        raise\n\n\ndef main():\n    p=argparse.ArgumentParser(description=__doc__)\n    for n in [\'source\',\'alpha\',\'region\']:\n        p.add_argument(\'--\'+n,required=True,type=Path);p.add_argument(\'--\'+n+\'-file-sha256\',required=True)\n    p.add_argument(\'--out\',type=Path,required=True);p.add_argument(\'--review-note\',required=True)\n    p.add_argument(\'--reviewed\',action=\'store_true\');p.add_argument(\'--material\',choices=[\'opaque\'],required=True);p.add_argument(\'--radius\',type=int,default=8)\n    args=vars(p.parse_args())\n    try:print(json.dumps(execute(**args),indent=2))\n    except (ValueError,RuntimeError,OSError) as e:p.exit(2,\'REFUSED: \'+str(e)+\'\\n\')\n\nif __name__==\'__main__\':main()\n'
# ---- embedded module: renderello/reviewed_alpha_lock_cli.py ----
path = base / 'renderello/reviewed_alpha_lock_cli.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Separate feathered attenuation of reviewed distant background, not segmentation.\nNo zero-mask seam override: the selection boundary and a clearance collar are identity.\nReal low-alpha detail can be erased by a wrong annotation. No default integration.\n"""\nimport numpy as np\nfrom scipy.ndimage import distance_transform_edt,binary_dilation\nfrom .candidate_diagnostics import native_rgb,native_alpha,digest\n\ndef attenuate_reviewed_background(rgb,alpha,background,protected,*,source_sha256,\n alpha_sha256,background_sha256,protected_sha256,reviewed=False,material=None,\n review_note=\'\',ceiling=32,clearance=12,feather=32):\n native_rgb(rgb);native_alpha(alpha,rgb.shape[:2])\n if alpha.size>1_000_000:raise ValueError(\'1MP native cap; no resize\')\n if reviewed is not True or material!=\'opaque\':raise ValueError(\'explicit opaque background review required\')\n if not isinstance(review_note,str) or not review_note.strip() or len(review_note)>2000:raise ValueError(\'bounded review note required\')\n for m in (background,protected):\n  if not isinstance(m,np.ndarray) or m.dtype!=np.bool_ or m.shape!=alpha.shape or not m.any():raise ValueError(\'nonempty native boolean masks required\')\n for arr,h in [(rgb,source_sha256),(alpha,alpha_sha256),(background,background_sha256),(protected,protected_sha256)]:\n  if digest(arr)!=h:raise ValueError(\'source/alpha/mask binding mismatch\')\n for v,lo,hi in [(ceiling,1,32),(clearance,4,64),(feather,16,64)]:\n  if type(v)is not int or not lo<=v<=hi:raise ValueError(\'bounded integer controls required\')\n if np.any(background&protected) or not np.any(protected&(alpha>0)):raise ValueError(\'disjoint background and retained foreground required\')\n if np.any(alpha[background]>ceiling):raise ValueError(\'selected background exceeds alpha ceiling; no partial selection\')\n anchors=(alpha>=128)|protected\n if not np.any(alpha>=128):raise ValueError(\'high-alpha foreground anchor required\')\n # EDT to outside the selected region, plus an explicit frame collar. The frame\n # is identity too; this avoids inventing an unobserved beyond-frame selection.\n padded=np.pad(background,1,constant_values=False)\n selection_distance=distance_transform_edt(padded)[1:-1,1:-1]\n anchor_distance=distance_transform_edt(~anchors)\n t=np.clip((np.minimum(selection_distance,anchor_distance)-clearance)/feather,0,1)\n weight=t*t*(3-2*t);weight[~background]=0\n out=np.rint(alpha.astype(float)*(1-weight)).astype(np.uint8)\n collar=background&(np.minimum(selection_distance,anchor_distance)<=clearance)\n if not np.array_equal(out[collar],alpha[collar]) or not np.array_equal(out[~background],alpha[~background]) or not np.array_equal(out[protected],alpha[protected]) or np.any(out>alpha):raise RuntimeError(\'attenuation identity/monotonicity invariant failed\')\n changed=out!=alpha;seam=background&binary_dilation(~background,structure=np.ones((3,3)))\n if np.any(changed&seam):raise RuntimeError(\'selection boundary changed\')\n return rgb.copy(),out,dict(schema=\'reviewed-background-attenuation.v1\',source_sha256=digest(rgb),alpha_sha256=digest(alpha),background_sha256=digest(background),protected_sha256=digest(protected),output_alpha_sha256=digest(out),review_note=review_note,ceiling=ceiling,clearance=clearance,feather=feather,alpha_changed_pixels=int(changed.sum()),maximum_removed_alpha=int((alpha-out).max()),removed_alpha_units=int((alpha.astype(np.int64)-out).sum()),selection_boundary_alpha_exact=True,selection_and_frame_collar_exact=True,protected_alpha_exact=True,outside_selection_alpha_exact=True,source_rgb_exact=True,source_resampled=False,accepted=False,default_changed=False,physical_recovery=False,limitations=[\'Reviewer background annotation is not semantic truth; real low-alpha detail can be erased.\',\'A broad feather can retain visible background haze near the protected enclosure or frame.\',\'This does not repair source edge color, geometry, lighting or glass transmission.\',\'No photographic acceptance or default promotion.\'])\n'
# ---- embedded module: renderello/reviewed_background_attenuation.py ----
path = base / 'renderello/reviewed_background_attenuation.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""File-bound reviewed background exclusion. Never an automatic mask cleanup."""\nfrom pathlib import Path\nimport argparse\nimport json\nimport os\nimport shutil\nimport tempfile\nimport numpy as np\nfrom PIL import Image\nfrom .image_intake import _read_bounded\nfrom .candidate_diagnostics import digest\nfrom .reviewed_edge_cli import read_image, sha, _json_pairs\nfrom .reviewed_background_exclusion import exclude_reviewed_background\n\nNAMES = (\'source\', \'alpha\', \'background\', \'protected\')\n\n\ndef load_spec(path, expected):\n    if Path(path).is_symlink():\n        raise ValueError(\'regular spec file required\')\n    raw = _read_bounded(Path(path), 65_536)\n    if sha(raw) != expected:\n        raise ValueError(\'spec file hash mismatch\')\n    def nonfinite(value):\n        raise ValueError(\'nonfinite JSON\')\n    s = json.loads(raw, object_pairs_hook=_json_pairs, parse_constant=nonfinite)\n    fields = {\'schema\', \'files\', \'reviewed\', \'material\', \'review_note\', \'controls\'}\n    if not isinstance(s, dict) or set(s) != fields or s[\'schema\'] != \'reviewed-background-files.v1\':\n        raise ValueError(\'exact background-exclusion spec required\')\n    if not isinstance(s[\'files\'], dict) or set(s[\'files\']) != set(NAMES):\n        raise ValueError(\'all four file bindings required\')\n    for h in s[\'files\'].values():\n        if not isinstance(h, str) or len(h) != 64 or any(c not in \'0123456789abcdef\' for c in h):\n            raise ValueError(\'lowercase file SHA-256 required\')\n    if not isinstance(s[\'controls\'], dict) or set(s[\'controls\']) != {\'ceiling\', \'clearance\', \'seam_ceiling\'}:\n        raise ValueError(\'exact explicit exclusion controls required\')\n    return s\n\n\ndef execute(*, source, alpha, background, protected, spec, spec_sha256, output_dir):\n    dest = Path(output_dir)\n    if dest.exists() or dest.is_symlink():\n        raise ValueError(\'output exists; preserve earlier evidence\')\n    s = load_spec(spec, spec_sha256)\n    arrays, records = [], {}\n    for n, path in zip(NAMES, (source, alpha, background, protected)):\n        if Path(path).is_symlink():\n            raise ValueError(\'regular image file required\')\n        a, record = read_image(path, s[\'files\'][n], n)\n        arrays.append(a); records[n] = record\n    rgb, a, bg, keep = arrays\n    out_rgb, out_alpha, metrics = exclude_reviewed_background(rgb, a, bg, keep,\n        source_sha256=digest(rgb), alpha_sha256=digest(a), background_sha256=digest(bg),\n        protected_sha256=digest(keep), reviewed=s[\'reviewed\'], material=s[\'material\'],\n        review_note=s[\'review_note\'], **s[\'controls\'])\n    if not np.array_equal(out_rgb, rgb) or not np.array_equal(out_alpha[~bg], a[~bg]):\n        raise RuntimeError(\'source or outside-region invariant failed\')\n    dest.parent.mkdir(parents=True, exist_ok=True)\n    temp = Path(tempfile.mkdtemp(prefix=\'.background-stage-\', dir=dest.parent))\n    try:\n        Image.fromarray(out_rgb).save(temp/\'source-rgb.png\')\n        Image.fromarray(out_alpha).save(temp/\'alpha.png\')\n        Image.fromarray(np.dstack([out_rgb, out_alpha])).save(temp/\'component-rgba.png\')\n        for label, value in [(\'dark\', 35), (\'light\', 242)]:\n            aa = out_alpha.astype(np.uint32)[:, :, None]\n            comp = ((out_rgb.astype(np.uint32)*aa + value*(255-aa)+127)//255).astype(np.uint8)\n            Image.fromarray(comp).save(temp/(label+\'.png\'))\n        manifest = dict(schema=\'reviewed-background-file-result.v1\', spec_sha256=spec_sha256,\n            inputs=records, controls=s[\'controls\'], metrics=metrics,\n            outputs={p.name: sha(p.read_bytes()) for p in temp.iterdir()},\n            review_is_caller_declaration=True, source_rgb_exact=True, accepted=False,\n            source_resampled=False, default_pipeline_changed=False)\n        (temp/\'manifest.json\').write_text(json.dumps(manifest, indent=2, allow_nan=False)+\'\\n\')\n        dest.mkdir()\n        try:\n            for p in temp.iterdir(): os.rename(p, dest/p.name)\n        except BaseException:\n            shutil.rmtree(dest)\n            raise\n        return manifest\n    finally:\n        shutil.rmtree(temp)\n\n\ndef main(argv=None):\n    p = argparse.ArgumentParser(description=__doc__)\n    for n in NAMES + (\'spec\', \'spec-sha256\', \'output-dir\'): p.add_argument(\'--\'+n, required=True)\n    a = p.parse_args(argv)\n    try: result = execute(**vars(a))\n    except (ValueError, OSError) as exc: p.exit(2, \'REFUSED: \'+str(exc)+\'\\n\')\n    print(json.dumps(result, indent=2, allow_nan=False))\n\n\nif __name__ == \'__main__\': main()\n'
# ---- embedded module: renderello/reviewed_background_cli.py ----
path = base / 'renderello/reviewed_background_cli.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Reviewed native low-alpha background exclusion, not automatic segmentation.\n\nAn explicit reviewer mask declares known background. Nothing is selected from\nalpha values alone. The ceiling and distance checks are safety refusals, not\nproof that low coverage is unimportant: real fine details can have low alpha.\nUse on reviewed opaque sources only. No default pipeline integration.\n"""\nimport numpy as np\nfrom scipy.ndimage import distance_transform_edt, binary_dilation\nfrom .candidate_diagnostics import native_rgb, native_alpha, digest\n\n\ndef exclude_reviewed_background(rgb, alpha, background, protected, *,\n        source_sha256, alpha_sha256, background_sha256, protected_sha256,\n        reviewed=False, material=None, review_note=\'\', ceiling=32, clearance=12, seam_ceiling=2):\n    """Zero exactly the reviewed background mask, or refuse the entire request.\n\n    Source RGB and alpha outside the mask remain exact. Protected fractional\n    alpha is supported. Inputs are never modified. No RGB decontamination,\n    foreground recovery, model-quality ranking, or publication acceptance.\n    """\n    native_rgb(rgb);native_alpha(alpha,rgb.shape[:2])\n    if alpha.size>1_000_000:raise ValueError(\'1MP native allocation budget; no resize\')\n    if reviewed is not True or material!=\'opaque\':\n        raise ValueError(\'explicit opaque source/background review required\')\n    if not isinstance(review_note,str) or not review_note.strip() or len(review_note)>2000:\n        raise ValueError(\'nonempty review note up to2000characters required\')\n    for mask in (background,protected):\n        if not isinstance(mask,np.ndarray) or mask.dtype!=np.bool_ or mask.shape!=alpha.shape or not mask.any():\n            raise ValueError(\'nonempty native boolean background/protected masks required\')\n    for array,expected in ((rgb,source_sha256),(alpha,alpha_sha256),\n                           (background,background_sha256),(protected,protected_sha256)):\n        if digest(array)!=expected:raise ValueError(\'source/alpha/region binding mismatch\')\n    if type(ceiling) is not int or not 1<=ceiling<=32:\n        raise ValueError(\'integer ceiling1..32 required\')\n    if type(clearance) is not int or not 4<=clearance<=64:\n        raise ValueError(\'integer clearance4..64 native pixels required\')\n    if type(seam_ceiling) is not int or not 1<=seam_ceiling<=4:\n        raise ValueError(\'integer seam ceiling1..4 required; no high-jump override\')\n    if np.any(background&protected):raise ValueError(\'background crosses protected region\')\n    if not np.any(protected&(alpha>0)):\n        raise ValueError(\'protected mask must retain some prior foreground\')\n    if np.any(alpha[background]>ceiling):\n        raise ValueError(\'background exceeds low-alpha ceiling; no partial silent exclusion\')\n    high=alpha>=128\n    if not high.any():raise ValueError(\'high-alpha foreground anchor required\')\n    # Include protected low-alpha details as clearance anchors, not just opacity.\n    anchors=high|protected\n    distances=distance_transform_edt(~anchors)\n    if np.any(distances[background]<clearance):\n        raise ValueError(\'background violates foreground/detail clearance\')\n    # Only internal selection borders create an edit seam. The image frame\n    # has no untouched neighbor and is deliberately not treated as a seam.\n    seam=background&binary_dilation(~background,structure=np.ones((3,3),bool),border_value=0)\n    seam_peak=int(alpha[seam].max()) if seam.any() else 0\n    if seam_peak>seam_ceiling:\n        raise ValueError(\'reviewed background creates an abrupt alpha seam; no high-jump override\')\n    out=alpha.copy();out[background]=0\n    changed=out!=alpha\n    if np.any(changed&~background) or not np.array_equal(out[protected],alpha[protected]):\n        raise RuntimeError(\'background exclusion invariant failed\')\n    return rgb.copy(),out,dict(schema=\'reviewed-background-exclusion.v1\',\n        source_sha256=digest(rgb),alpha_sha256=digest(alpha),\n        background_sha256=digest(background),protected_sha256=digest(protected),\n        output_alpha_sha256=digest(out),review_note=review_note,material=material,\n        ceiling=ceiling,clearance=clearance,seam_ceiling=seam_ceiling,\n        seam_peak_removed_alpha=seam_peak,reviewed_seam_pixels=int(seam.sum()),\n        reviewed_background_pixels=int(background.sum()),\n        minimum_background_clearance=float(distances[background].min()),\n        alpha_changed_pixels=int(changed.sum()),removed_alpha_units=int(alpha[changed].astype(np.int64).sum()),\n        maximum_removed_alpha=int(alpha[changed].max()) if changed.any() else 0,\n        protected_alpha_exact=True,outside_reviewed_background_exact=True,all_source_rgb_exact=True,\n        source_resampled=False,accepted=False,default_pipeline_changed=False,\n        independent_ground_truth=False,physical_recovery=False,\n        limitations=[\'Reviewer-declared background is not segmentation truth.\',\n         \'Fine low-alpha details can be erased by a wrong region; ceiling is not a semantic safeguard.\',\n         \'Near-edge halos and source color contamination remain unchanged.\',\n         \'Explicit zeroing can reveal a region seam. Inspect native light/dark outputs.\',\n         \'Not for glass, transmission, soft shadows, smoke or inferred background.\'])\n'
# ---- embedded module: renderello/reviewed_background_exclusion.py ----
path = base / 'renderello/reviewed_background_exclusion.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Exact source-bound matte intervention gate. Never repairs or selects a matte.\n\nCaller-reviewed edit/protection regions are declarations, not semantic truth.\nThe whole candidate is refused rather than silently clamped to the prior.\nPassing means only the declared intervention boundary was respected.\n"""\nimport numpy as np\nfrom .candidate_diagnostics import native_rgb, native_alpha, digest\nfrom .matte_change_review import ChangeBudget, review_change\n\n\ndef review_candidate(rgb, before, rgba, allowed, protected, *, bindings,\n                     reviewed=False, material=None, framing=None,\n                     review_note=\'\', budget=None):\n    native_rgb(rgb); native_alpha(before, rgb.shape[:2])\n    if before.size > 1_000_000:\n        raise ValueError(\'native1MP review limit; no resize\')\n    if reviewed is not True or material != \'opaque\' or framing != \'complete\':\n        raise ValueError(\'explicit complete opaque source/region review required\')\n    if not isinstance(review_note, str) or not review_note.strip() or len(review_note) > 2000:\n        raise ValueError(\'nonempty bounded source/region review note required\')\n    if not isinstance(rgba, np.ndarray) or rgba.dtype != np.uint8 or rgba.shape != (*before.shape, 4):\n        raise ValueError(\'exact native uint8 RGBA candidate required\')\n    for mask in (allowed, protected):\n        if not isinstance(mask, np.ndarray) or mask.dtype != np.bool_ or mask.shape != before.shape or not mask.any():\n            raise ValueError(\'nonempty native boolean edit/protected regions required\')\n    if np.any(allowed & protected):\n        raise ValueError(\'edit and protected declarations overlap\')\n    if not np.any(protected & (before > 0)):\n        raise ValueError(\'protected declaration must retain prior foreground\')\n    arrays = dict(source=rgb, before=before, candidate=rgba, allowed=allowed, protected=protected)\n    if not isinstance(bindings, dict) or set(bindings) != set(arrays):\n        raise ValueError(\'exact five source/candidate/region bindings required\')\n    if any(not isinstance(bindings[n], str) or bindings[n] != digest(a) for n, a in arrays.items()):\n        raise ValueError(\'source/candidate/region binding mismatch\')\n    if budget is not None and not isinstance(budget, ChangeBudget):\n        raise ValueError(\'typed topology change budget required\')\n    after = rgba[:, :, 3]\n    signed_delta = after.astype(np.int16) - before.astype(np.int16)\n    changed = after != before\n    rgb_changed = np.any(rgba[:, :, :3] != rgb, axis=2)\n    outside = changed & ~allowed\n    locked = changed & protected\n    topology, delta = review_change(before, after, budget)\n    blockers = []\n    if rgb_changed.any(): blockers.append(\'CANDIDATE_SOURCE_RGB_CHANGED\')\n    if locked.any(): blockers.append(\'PROTECTED_ALPHA_CHANGED\')\n    if outside.any(): blockers.append(\'ALPHA_CHANGED_OUTSIDE_REVIEWED_EDIT_REGION\')\n    blockers.extend(\'TOPOLOGY_\' + b for b in topology[\'blockers\'])\n    return dict(schema=\'reviewed-candidate-intervention-gate.v1\', bindings=bindings.copy(),\n        material=material, framing=framing, review_note=review_note,\n        review_is_caller_declaration=True, edit_pixels=int(allowed.sum()),\n        protected_pixels=int(protected.sum()), changed_alpha_pixels=int(changed.sum()),\n        changed_protected_alpha_pixels=int(locked.sum()),\n        maximum_protected_alpha_decrease=int(np.maximum(-signed_delta[protected], 0).max()),\n        maximum_protected_alpha_increase=int(np.maximum(signed_delta[protected], 0).max()),\n        protected_before_fully_opaque_pixels=int(((before == 255) & protected).sum()),\n        protected_candidate_fully_opaque_pixels=int(((after == 255) & protected).sum()),\n        changed_outside_edit_pixels=int(outside.sum()),\n        changed_source_rgb_pixels=int(rgb_changed.sum()),\n        originally_opaque_pixels_removed_to_zero=int(((before == 255) & (after == 0)).sum()),\n        topology=topology, blockers=blockers,\n        intervention_constraints_satisfied=not blockers,\n        baseline_accuracy=\'NOT_VERIFIED\', candidate_accuracy=\'NOT_VERIFIED\',\n        photo_accepted=False, candidate_selected=False, default_changed=False,\n        source_resampled=False, output_repaired=False,\n        limitations=[\'Region intent and source material are caller declarations, not semantic detection.\',\n            \'A wrong prior can remain wrong. Passing this gate is not a quality score.\',\n            \'No candidate is selected, clamped, composited or silently repaired.\',\n            \'No lighting, scene/contact, fine-print readability or glass optics validation.\']), delta\n'
# ---- embedded module: renderello/reviewed_candidate_gate.py ----
path = base / 'renderello/reviewed_candidate_gate.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""File-bound exact intervention audit. No candidate image export or promotion."""\nfrom pathlib import Path\nfrom io import BytesIO\nimport argparse, hashlib, json, os, shutil, tempfile\nimport numpy as np\nfrom PIL import Image\nfrom .image_intake import _read_bounded\nfrom .candidate_diagnostics import digest\nfrom .reviewed_edge_cli import _json_pairs\nfrom .reviewed_candidate_gate import review_candidate\nfrom .matte_change_review import ChangeBudget\n\nNAMES = (\'source\', \'before\', \'candidate\', \'allowed\', \'protected\')\nsha = lambda data: hashlib.sha256(data).hexdigest()\n\n\ndef load_policy(path, expected):\n    raw = _read_bounded(Path(path), 65536)\n    if sha(raw) != expected: raise ValueError(\'policy file hash mismatch\')\n    try:\n        p = json.loads(raw, object_pairs_hook=_json_pairs,\n            parse_constant=lambda x: (_ for _ in ()).throw(ValueError(\'nonfinite JSON\')))\n    except (json.JSONDecodeError, UnicodeDecodeError) as e:\n        raise ValueError(\'invalid policy JSON\') from e\n    fields = {\'schema\', \'files\', \'reviewed\', \'material\', \'framing\', \'review_note\', \'budget\'}\n    if not isinstance(p, dict) or set(p) != fields or p[\'schema\'] != \'reviewed-candidate-files.v1\':\n        raise ValueError(\'exact candidate policy schema required\')\n    if not isinstance(p[\'files\'], dict) or set(p[\'files\']) != set(NAMES):\n        raise ValueError(\'exact five file bindings required\')\n    for h in p[\'files\'].values():\n        if not isinstance(h, str) or len(h) != 64 or any(c not in \'0123456789abcdef\' for c in h):\n            raise ValueError(\'lowercase file SHA256 required\')\n    keys = set(ChangeBudget.__dataclass_fields__)\n    if not isinstance(p[\'budget\'], dict) or set(p[\'budget\']) != keys:\n        raise ValueError(\'every topology change control must be explicit\')\n    try: budget = ChangeBudget(**p[\'budget\'])\n    except (TypeError, ValueError) as e: raise ValueError(\'invalid topology budget\') from e\n    return p, budget\n\n\ndef read_bound(path, expected, name):\n    raw = _read_bounded(Path(path), 8_000_000)\n    if sha(raw) != expected: raise ValueError(name + \' file hash mismatch\')\n    if not raw.startswith(b\'\\x89PNG\\r\\n\\x1a\\n\') or len(raw) < 29 or raw[24] != 8:\n        raise ValueError(name + \' requires8bit PNG\')\n    with Image.open(BytesIO(raw)) as im:\n        mode = \'RGB\' if name == \'source\' else (\'RGBA\' if name == \'candidate\' else \'L\')\n        if im.format != \'PNG\' or im.mode != mode: raise ValueError(name + \' exact \' + mode + \' required\')\n        if getattr(im, \'n_frames\', 1) != 1 or im.getexif() or any(k in im.info for k in (\'icc_profile\',\'gamma\',\'srgb\',\'transparency\')):\n            raise ValueError(name + \' metadata requires separate intake\')\n        if min(im.size) < 3 or im.width * im.height > 1_000_000:\n            raise ValueError(name + \' native3px..1MP limit; no resize\')\n        arr = np.array(im)\n    if name in (\'allowed\',\'protected\'):\n        if not np.isin(arr,[0,255]).all(): raise ValueError(name + \' requires binary0/255; no threshold\')\n        arr = arr == 255\n    return arr, dict(file_sha256=sha(raw), pixel_sha256=digest(arr), bytes=len(raw), shape=list(arr.shape))\n\n\ndef execute(*, source, before, candidate, allowed, protected, policy, policy_sha256, output_dir):\n    dest = Path(output_dir)\n    if dest.exists() or dest.is_symlink(): raise ValueError(\'output exists; preserve earlier evidence\')\n    p, budget = load_policy(policy, policy_sha256)\n    arrays = {}; records = {}\n    for n, path in zip(NAMES,(source,before,candidate,allowed,protected)):\n        arrays[n], records[n] = read_bound(path,p[\'files\'][n],n)\n    copies = {n:a.copy()for n,a in arrays.items()}\n    report, delta = review_candidate(arrays[\'source\'], arrays[\'before\'], arrays[\'candidate\'],\n        arrays[\'allowed\'], arrays[\'protected\'], bindings={n:digest(a)for n,a in arrays.items()},\n        reviewed=p[\'reviewed\'], material=p[\'material\'], framing=p[\'framing\'],\n        review_note=p[\'review_note\'], budget=budget)\n    if any(not np.array_equal(a,copies[n])for n,a in arrays.items()): raise RuntimeError(\'input mutation\')\n    dest.parent.mkdir(parents=True,exist_ok=True)\n    temp = Path(tempfile.mkdtemp(prefix=\'.candidate-audit-stage-\',dir=dest.parent))\n    try:\n        Image.fromarray(delta).save(temp/\'alpha-change-visualization.png\')\n        report.update(policy_file_sha256=policy_sha256, files=records,\n            candidate_image_exported=False, audit_only=True,\n            decision=\'INTERVENTION_CONSTRAINTS_ONLY\' if report[\'intervention_constraints_satisfied\'] else \'REFUSED_INTERVENTION_SCOPE\',\n            visualization_semantics=\'Red alpha increase, blue decrease. Magnitude is not error or quality.\')\n        report[\'output_sha256\']={\'alpha-change-visualization.png\':sha((temp/\'alpha-change-visualization.png\').read_bytes())}\n        (temp/\'audit.json\').write_text(json.dumps(report,indent=2,allow_nan=False)+\'\\n\')\n        dest.mkdir()\n        try:\n            for q in temp.iterdir(): os.rename(q,dest/q.name)\n        except BaseException:\n            shutil.rmtree(dest);raise\n        return report\n    finally: shutil.rmtree(temp)\n\n\ndef main():\n    p=argparse.ArgumentParser(description=__doc__)\n    for n in (*NAMES,\'policy\',\'policy-sha256\',\'output-dir\'):p.add_argument(\'--\'+n,required=True)\n    a=p.parse_args()\n    try: report=execute(**vars(a))\n    except (ValueError,OSError)as e:p.exit(2,\'REFUSED INPUT: \'+str(e)+\'\\n\')\n    print(json.dumps(report,indent=2,allow_nan=False))\n    # A valid audit may refuse the candidate. The report is evidence, not approval.\n    if not report[\'intervention_constraints_satisfied\']:p.exit(3)\n\nif __name__==\'__main__\':main()\n'
# ---- embedded module: renderello/reviewed_candidate_gate_cli.py ----
path = base / 'renderello/reviewed_candidate_gate_cli.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Opt-in illustrated transfer of reviewed color-bearing white-floor reflections.\n\nNot a shadow, light, emission or material recovery model. A separate experiment:\nnever weakens the neutral-only source-floor tool and never changes default routes.\n"""\nimport numpy as np\nfrom .candidate_diagnostics import native_rgb, native_alpha, digest\n\n\ndef tint_reviewed_chromatic_floor(rgb, alpha, background, selection, *,\n        source_sha256, alpha_sha256, background_sha256, selection_sha256,\n        reviewed=False, source_floor=None, review_note=\'\'):\n    native_rgb(rgb); native_rgb(background); native_alpha(alpha, rgb.shape[:2])\n    native_alpha(selection, alpha.shape)\n    if rgb.shape != background.shape or alpha.size > 1_000_000:\n        raise ValueError(\'matching native target and 1MP budget required\')\n    if reviewed is not True or source_floor != \'white_opaque_with_color_reflection\':\n        raise ValueError(\'explicit color-bearing white opaque floor review required\')\n    if not isinstance(review_note, str) or not review_note.strip() or len(review_note)>2000:\n        raise ValueError(\'bounded review note required\')\n    for value, expected in [(rgb, source_sha256), (alpha, alpha_sha256),\n                            (background, background_sha256), (selection, selection_sha256)]:\n        if digest(value) != expected:\n            raise ValueError(\'source/alpha/background/selection binding mismatch\')\n    if background.min()<160 or not np.all(background==background[0,0]):\n        raise ValueError(\'light flat target only\')\n    observed=(selection>0)&(alpha==0)\n    if not observed.any() or not np.any(alpha==255) or (selection>0).sum()>alpha.size*.20:\n        raise ValueError(\'bounded local selection, observed floor and opaque anchor required\')\n    spread=rgb.max(axis=2).astype(int)-rgb.min(axis=2).astype(int)\n    white=(rgb.min(axis=2)>=235)&(spread<=12)\n    if np.mean(white[observed])<.10:\n        raise ValueError(\'insufficient near-white floor context; no dark-floor extrapolation\')\n    a=alpha[:,:,None].astype(float)/255\n    m=selection[:,:,None].astype(float)/255\n    baseline=np.rint(rgb*a+background*(1-a)).astype(np.uint8)\n    floor=background*((1-m)+m*(rgb.astype(float)/255))\n    output=np.rint(rgb*a+floor*(1-a)).clip(0,255).astype(np.uint8)\n    if not np.array_equal(output[alpha==255], rgb[alpha==255]) or not np.array_equal(output[selection==0],baseline[selection==0]):\n        raise RuntimeError(\'native RGB or outside-selection invariant failed\')\n    return output,dict(schema=\'reviewed-chromatic-floor-illustration.v1\',\n        source_sha256=digest(rgb),alpha_sha256=digest(alpha),background_sha256=digest(background),\n        selection_sha256=digest(selection),output_sha256=digest(output),review_note=review_note,\n        changed_pixels=int(np.any(output!=baseline,axis=2).sum()),opaque_source_rgb_exact=True,\n        outside_selection_baseline_exact=True,source_resampled=False,alpha_changed=False,\n        default_pipeline_changed=False,accepted=False,physical_relighting=False,\n        recovered_shadow=False,recovered_emission=False,near_white_observed_fraction=float(np.mean(white[observed])),\n        limitations=[\'Manual floor selection is not semantic truth.\',\n          \'Uncalibrated sRGB channel-wise tint is an illustration, not physical light or emission recovery.\',\n          \'Observed source floor may carry unwanted texture, artifacts, objects or color casts.\',\n          \'Fractional-alpha boundaries may double-darken; inspect each native output.\',\n          \'Light flat targets only; no automatic acceptance or default integration.\'])\n'
# ---- embedded module: renderello/reviewed_chromatic_floor.py ----
path = base / 'renderello/reviewed_chromatic_floor.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Offline native color-bearing floor illustration. Explicit reviewed inputs only.\n\nDoes not identify a floor, infer a shadow, relight a product or authorize publishing.\n"""\nfrom pathlib import Path\nimport argparse\nimport hashlib\nimport json\nimport numpy as np\nfrom PIL import Image\nfrom .cached_native_scene_cli import snapshot\nfrom .candidate_diagnostics import digest\nfrom .reviewed_chromatic_floor import tint_reviewed_chromatic_floor\n\n\ndef run(source, alpha, selection, out, *, source_sha256, alpha_sha256,\n        selection_sha256, background_rgb, reviewed=False, review_note=\'\'):\n    out=Path(out)\n    if out.exists() or out.is_symlink():\n        raise FileExistsError(\'preserve prior result; choose new output directory\')\n    if not isinstance(background_rgb,(tuple,list)) or len(background_rgb)!=3 or any(type(v)is not int or not 160<=v<=255 for v in background_rgb):\n        raise ValueError(\'three integer light background channels160..255 required\')\n    inputs={}\n    for name,path,mode,expected in [(\'source\',source,\'RGB\',source_sha256),\n            (\'alpha\',alpha,\'L\',alpha_sha256),(\'selection\',selection,\'L\',selection_sha256)]:\n        if not isinstance(expected,str) or len(expected)!=64 or any(c not in \'0123456789abcdef\' for c in expected):\n            raise ValueError(\'explicit lowercase input file SHA256 required\')\n        data,array=snapshot(path,mode,1_000_000)\n        if hashlib.sha256(data).hexdigest()!=expected:\n            raise ValueError(\'input file binding mismatch\')\n        inputs[name]=(data,array)\n    rgb,a,s=(inputs[n][1] for n in [\'source\',\'alpha\',\'selection\'])\n    bg=np.full_like(rgb,background_rgb)\n    output,review=tint_reviewed_chromatic_floor(rgb,a,bg,s,\n        source_sha256=digest(rgb),alpha_sha256=digest(a),\n        background_sha256=digest(bg),selection_sha256=digest(s),reviewed=reviewed,\n        source_floor=\'white_opaque_with_color_reflection\',review_note=review_note)\n    baseline=np.rint(rgb*(a[:,:,None]/255)+bg*(1-a[:,:,None]/255)).astype(np.uint8)\n    # Reserve only after validation. A failed write retains the inputs and an\n    # INCOMPLETE marker, never a misleading successful manifest.\n    out.mkdir(parents=True)\n    (out/\'INCOMPLETE\').touch()\n    for name,(data,_) in inputs.items():(out/(name+\'.png\')).write_bytes(data)\n    for name,array in [(\'no-floor.png\',baseline),(\'reviewed-floor.png\',output)]:\n        Image.fromarray(array).save(out/name)\n        with Image.open(out/name)as im:\n            if im.mode!=\'RGB\' or not np.array_equal(np.array(im),array):\n                raise RuntimeError(\'saved RGB export mismatch\')\n    record=dict(schema=\'offline-reviewed-chromatic-floor-result.v1\',\n        status=\'EXECUTED_REVIEW_REQUIRED\',provider_calls=0,credits_spent=0,\n        input_file_sha256={n:hashlib.sha256(data).hexdigest() for n,(data,_) in inputs.items()},\n        output_file_sha256=hashlib.sha256((out/\'reviewed-floor.png\').read_bytes()).hexdigest(),\n        background_rgb=list(background_rgb),review=review)\n    (out/\'manifest.json\').write_text(json.dumps(record,indent=2,allow_nan=False))\n    (out/\'INCOMPLETE\').unlink()\n    return record\n\n\ndef main(argv=None):\n    p=argparse.ArgumentParser(description=__doc__)\n    for n in [\'source\',\'alpha\',\'selection\',\'out\']:p.add_argument(\'--\'+n,type=Path,required=True)\n    for n in [\'source-sha256\',\'alpha-sha256\',\'selection-sha256\',\'review-note\']:p.add_argument(\'--\'+n,required=True)\n    p.add_argument(\'--background-rgb\',type=int,nargs=3,required=True)\n    p.add_argument(\'--reviewed\',action=\'store_true\',required=True)\n    args=p.parse_args(argv)\n    print(json.dumps(run(**vars(args)),indent=2))\n    return 0\n\n\nif __name__==\'__main__\':raise SystemExit(main())\n'
# ---- embedded module: renderello/reviewed_chromatic_floor_cli.py ----
path = base / 'renderello/reviewed_chromatic_floor_cli.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Bounded RGB transfer from explicitly reviewed same-material near-opaque donors.\n\nResearch hypothesis, not optical recovery. Alpha and all non-target RGB stay exact.\n"""\nimport numpy as np\nfrom scipy.ndimage import distance_transform_edt\nfrom .candidate_diagnostics import native_rgb,native_alpha,digest\nfrom .reviewed_model_edge import import_reviewed_model_edge\n\n\ndef transfer_reviewed_donor_edge(rgb,alpha,target,donor,protected,*,donor_sha256,\n        max_donor_distance=24,**controls):\n    native_rgb(rgb);native_alpha(alpha,rgb.shape[:2])\n    if alpha.size>1_000_000:raise ValueError(\'1MP source budget\')\n    if not isinstance(target,np.ndarray)or target.dtype!=np.bool_ or target.shape!=alpha.shape or not target.any():raise ValueError(\'nonempty native boolean target required\')\n    if not isinstance(donor,np.ndarray)or donor.dtype!=np.bool_ or donor.shape!=alpha.shape or not donor.any():raise ValueError(\'nonempty native boolean donors required\')\n    if digest(donor)!=donor_sha256:raise ValueError(\'donor binding mismatch\')\n    if np.any(donor&target)or np.any(alpha[donor]<254):raise ValueError(\'disjoint reviewed same-material donors require alpha>=254\')\n    if type(max_donor_distance)is not int or not 1<=max_donor_distance<=48:raise ValueError(\'integer donor distance1..48\')\n    distances,indices=distance_transform_edt(~donor,return_indices=True)\n    if np.any(distances[target]>max_donor_distance):raise ValueError(\'target has no nearby reviewed donor\')\n    estimate=rgb.copy();estimate[target]=rgb[indices[0][target],indices[1][target]]\n    if \'estimate_sha256\'in controls:raise ValueError(\'estimate is derived, not supplied\')\n    out,report=import_reviewed_model_edge(rgb,alpha,estimate,target,protected,estimate_sha256=digest(estimate),**controls)\n    report.update(schema=\'reviewed-donor-edge.v1\',donor_sha256=digest(donor),max_donor_distance=max_donor_distance,estimate_origin=\'nearest explicitly reviewed same-material near-opaque source donor\',limitations=report[\'limitations\']+[\'Alpha>=254 donor support is not proof of physical opacity. Donor RGB transfer can flatten real shading or erase genuine boundary colour. Nearest donor pairing is not material or lighting truth.\'])\n    return out,report\n'
# ---- embedded module: renderello/reviewed_donor_edge.py ----
path = base / 'renderello/reviewed_donor_edge.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Opt-in annotated edge chroma transfer, not foreground/lighting recovery.\n\nTargets and same-material donors are supplied by a reviewer in native source\ncoordinates. Original luminance proxy is retained before gamut clipping. Opaque\nedge RGB CAN change; alpha and protected source pixels never do. No default use.\n"""\nimport math\nfrom numbers import Real\nimport numpy as np\nfrom scipy.ndimage import distance_transform_edt\nfrom .candidate_diagnostics import native_rgb, native_alpha, digest\n\n\ndef transfer_reviewed_edge_chroma(rgb, alpha, target, donor, protected, *,\n        source_sha256, alpha_sha256, review_note, reviewed=False,\n        edge_width=8., donor_distance=32., strength=.75):\n    native_rgb(rgb)\n    native_alpha(alpha, rgb.shape[:2])\n    if alpha.size > 1_000_000:\n        raise ValueError(\'1MP research allocation budget; no resize\')\n    if reviewed is not True or not isinstance(review_note, str) or not review_note.strip() or len(review_note)>2000:\n        raise ValueError(\'explicit source/material review note required\')\n    if digest(rgb) != source_sha256 or digest(alpha) != alpha_sha256:\n        raise ValueError(\'annotation source/alpha binding mismatch\')\n    for mask in (target, donor, protected):\n        if not isinstance(mask, np.ndarray) or mask.dtype != np.bool_ or mask.shape != alpha.shape:\n            raise ValueError(\'native boolean target, donor and protected masks required\')\n        if not mask.any():\n            raise ValueError(\'nonempty target, donor and protected masks required\')\n    if np.any(target & (donor | protected)) or np.any(donor & protected):\n        raise ValueError(\'annotation masks must be disjoint\')\n    if np.any(alpha[target] < 128) or np.any(alpha[donor] != 255):\n        raise ValueError(\'target needs high-alpha support and donors need opaque support\')\n    if np.any(alpha[protected] != 255):\n        raise ValueError(\'protected source regions must be opaque\')\n    for v, lo, hi in ((edge_width, 1, 24), (donor_distance, 1, 96), (strength, 0, 1)):\n        if isinstance(v, (bool, np.bool_)) or not isinstance(v, Real) or not math.isfinite(v) or not lo < v <= hi:\n            raise ValueError(\'finite bounded controls required\')\n    hard = alpha >= 128\n    if hard.all():\n        raise ValueError(\'explicit background required for edge distance\')\n    boundary_distance = distance_transform_edt(hard)\n    if np.any(boundary_distance[target] > edge_width):\n        raise ValueError(\'target extends beyond declared edge band; no interior repaint\')\n    distances, indices = distance_transform_edt(~donor, return_indices=True)\n    if np.any(distances[target] > donor_distance):\n        raise ValueError(\'target lacks nearby reviewed donors; no partial silent correction\')\n    # Same-material chroma is a review hypothesis. A weighted sRGB-code luminance\n    # proxy is NOT calibrated scene luminance. Preserve it before gamut clipping.\n    x = rgb[target].astype(np.float64)\n    near = rgb[indices[0][target], indices[1][target]].astype(np.float64)\n    weights = np.array([.2126, .7152, .0722])\n    candidate = near + ((x @ weights) - (near @ weights))[:, None]\n    mixed = x * (1 - strength) + candidate * strength\n    clipped = np.any((mixed < 0) | (mixed > 255), axis=1)\n    out = rgb.copy()\n    out[target] = np.rint(np.clip(mixed, 0, 255)).astype(np.uint8)\n    changed = np.any(out != rgb, axis=2)\n    if np.any(changed & ~target) or not np.array_equal(out[protected], rgb[protected]):\n        raise RuntimeError(\'source protection invariant failed\')\n    return out, dict(schema=\'reviewed-edge-chroma.v1\', source_sha256=digest(rgb),\n        alpha_sha256=digest(alpha), target_sha256=digest(target), donor_sha256=digest(donor),\n        protected_sha256=digest(protected), output_sha256=digest(out), review_note=review_note,\n        edge_width=float(edge_width), donor_distance=float(donor_distance), strength=float(strength),\n        target_pixels=int(target.sum()), changed_pixels=int(changed.sum()),\n        changed_opaque_pixels=int((changed & (alpha == 255)).sum()),\n        clipped_pixels=int(clipped.sum()), protected_rgb_exact=True, alpha_changed=False,\n        source_resampled=False, fully_source_rgb_exact=not bool(changed.any()),\n        physical_recovery=False, automatic_acceptance=False,\n        limitation=\'Annotated same-material chroma hypothesis. Can erase genuine edge color and retain wrong geometry. Opaque edge RGB is edited; no physical relighting.\')\n'
# ---- embedded module: renderello/reviewed_edge_chroma.py ----
path = base / 'renderello/reviewed_edge_chroma.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""File-bound opt-in edge gain/lift replay. No inferred annotation or quality promotion."""\nfrom pathlib import Path\nfrom io import BytesIO\nimport argparse,hashlib,json,os,shutil,tempfile\nimport numpy as np\nfrom PIL import Image\nfrom .image_intake import _read_bounded\nfrom .candidate_diagnostics import digest\nfrom .reviewed_edge_gain import darken_reviewed_edge\nfrom .reviewed_edge_lift import lift_reviewed_edge\n\nNAMES=(\'source\',\'alpha\',\'target\',\'donor\',\'protected\')\ndef sha(data):return hashlib.sha256(data).hexdigest()\ndef _json_pairs(pairs):\n d={}\n for k,v in pairs:\n  if k in d:raise ValueError(\'duplicate JSON field: \'+k)\n  d[k]=v\n return d\n\ndef load_spec(path,expected):\n raw=_read_bounded(Path(path),65536)\n if sha(raw)!=expected:raise ValueError(\'spec file hash mismatch\')\n try:s=json.loads(raw,object_pairs_hook=_json_pairs,parse_constant=lambda x:(_ for _ in ()).throw(ValueError(\'nonfinite JSON\')))\n except (json.JSONDecodeError,UnicodeDecodeError) as e:raise ValueError(\'invalid spec JSON\') from e\n fields={\'schema\',\'operation\',\'files\',\'reviewed\',\'material\',\'review_note\',\'controls\'}\n if not isinstance(s,dict) or set(s)!=fields or s[\'schema\']!=\'reviewed-edge-files.v1\':raise ValueError(\'exact spec schema required\')\n if s[\'operation\'] not in (\'darken\',\'lift\'):raise ValueError(\'explicit darken/lift operation required\')\n if not isinstance(s[\'files\'],dict) or set(s[\'files\'])!=set(NAMES):raise ValueError(\'all five file bindings required\')\n for h in s[\'files\'].values():\n  if not isinstance(h,str) or len(h)!=64 or any(c not in \'0123456789abcdef\' for c in h):raise ValueError(\'lowercase file SHA256 required\')\n shared={\'edge_width\',\'donor_distance\',\'strength\',\'max_code_delta\'};keys=shared|({\'minimum_gain\'} if s[\'operation\']==\'darken\' else {\'maximum_gain\'})\n if not isinstance(s[\'controls\'],dict) or set(s[\'controls\'])!=keys:raise ValueError(\'exact explicit operation controls required\')\n return s\n\ndef read_image(path,expected,name):\n raw=_read_bounded(Path(path),8_000_000)\n if sha(raw)!=expected:raise ValueError(name+\' file hash mismatch\')\n # Explicit 8-bit PNG avoids hidden color conversion, high-depth quantization or orientation.\n if not raw.startswith(b\'\\x89PNG\\r\\n\\x1a\\n\') or len(raw)<29 or raw[24]!=8:raise ValueError(name+\' must be8bit PNG\')\n with Image.open(BytesIO(raw)) as im:\n  if im.format!=\'PNG\' or im.mode!=(\'RGB\' if name==\'source\' else \'L\'):raise ValueError(name+\' exact RGB/L mode required\')\n  if getattr(im,\'n_frames\',1)!=1 or im.getexif() or any(k in im.info for k in (\'icc_profile\',\'gamma\',\'srgb\',\'transparency\')):raise ValueError(name+\' color/orientation/frame metadata requires separate intake\')\n  if min(im.size)<2 or im.width*im.height>1_000_000:raise ValueError(name+\' native1MP limit; no resize\')\n  a=np.array(im)\n if name not in (\'source\',\'alpha\'):\n  if not np.isin(a,[0,255]).all():raise ValueError(name+\' mask must use only0/255; no threshold guessing\')\n  a=a==255\n return a,dict(file_sha256=sha(raw),pixel_sha256=digest(a),bytes=len(raw),shape=list(a.shape))\n\ndef execute(*,source,alpha,target,donor,protected,spec,spec_sha256,output_dir):\n dest=Path(output_dir)\n if dest.exists() or dest.is_symlink():raise ValueError(\'output exists; preserve earlier evidence\')\n s=load_spec(spec,spec_sha256);arrays=[];records={}\n for n,p in zip(NAMES,(source,alpha,target,donor,protected)):\n  a,r=read_image(p,s[\'files\'][n],n);arrays.append(a);records[n]=r\n rgb,al,t,d,p=arrays;before=[v.copy() for v in arrays]\n f=darken_reviewed_edge if s[\'operation\']==\'darken\' else lift_reviewed_edge\n out,metrics=f(*arrays,source_sha256=digest(rgb),alpha_sha256=digest(al),target_sha256=digest(t),donor_sha256=digest(d),protected_sha256=digest(p),reviewed=s[\'reviewed\'],material=s[\'material\'],review_note=s[\'review_note\'],**s[\'controls\'])\n if any(not np.array_equal(a,b) for a,b in zip(arrays,before)):raise RuntimeError(\'input mutated\')\n if not np.array_equal(out[p],rgb[p]) or not np.array_equal(out[~t],rgb[~t]):raise RuntimeError(\'protected/outside RGB changed\')\n # Validation and computation precede reservation. Stage first; reserve with exclusive mkdir.\n dest.parent.mkdir(parents=True,exist_ok=True);temp=Path(tempfile.mkdtemp(prefix=\'.edge-stage-\',dir=dest.parent));reserved=False\n try:\n  Image.fromarray(out).save(temp/\'foreground.png\');Image.fromarray(al).save(temp/\'alpha.png\');Image.fromarray(np.dstack([out,al])).save(temp/\'component-rgba.png\')\n  outputs={q.name:sha(q.read_bytes()) for q in sorted(temp.iterdir())}\n  manifest=dict(schema=\'reviewed-edge-file-result.v1\',spec_sha256=spec_sha256,operation=s[\'operation\'],inputs=records,controls=s[\'controls\'],metrics=metrics,outputs=outputs,review_is_caller_declaration=True,source_resampled=False,alpha_changed=False,protected_rgb_exact=True,accepted=False,default_changed=False)\n  (temp/\'manifest.json\').write_text(json.dumps(manifest,indent=2,allow_nan=False)+\'\\n\')\n  dest.mkdir();reserved=True\n  try:\n   for q in temp.iterdir():os.rename(q,dest/q.name)\n  except BaseException:\n   shutil.rmtree(dest);reserved=False;raise\n  return manifest\n finally:\n  shutil.rmtree(temp)\n\ndef parser():\n p=argparse.ArgumentParser(description=__doc__)\n for n in NAMES:p.add_argument(\'--\'+n,required=True)\n for n in (\'spec\',\'spec-sha256\',\'output-dir\'):p.add_argument(\'--\'+n,required=True)\n return p\nif __name__==\'__main__\':\n p=parser();a=p.parse_args()\n try:m=execute(**vars(a))\n except (ValueError,OSError) as e:p.exit(2,\'REFUSED: \'+str(e)+\'\\n\')\n print(json.dumps(m,indent=2,allow_nan=False))\n'
# ---- embedded module: renderello/reviewed_edge_cli.py ----
path = base / 'renderello/reviewed_edge_cli.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Opt-in achromatic edge darkening from reviewed same-material donor RGB.\n\nA local brightness hypothesis, not inverse compositing or physical recovery.\nOpaque edge RGB CAN change. Alpha and RGB outside the target stay exact.\nGenuine highlights can be damaged. There is no default pipeline integration.\n"""\nimport math\nfrom numbers import Real\nimport numpy as np\nfrom scipy.ndimage import distance_transform_edt\nfrom .candidate_diagnostics import native_rgb,native_alpha,digest\nfrom .checked_research.detail_transfer import decode,encode\n\n\ndef darken_reviewed_edge(rgb,alpha,target,donor,protected,*,source_sha256,\n        alpha_sha256,target_sha256,donor_sha256,protected_sha256,\n        reviewed=False,material=None,review_note=\'\',edge_width=4.,\n        donor_distance=24.,strength=.5,minimum_gain=.65,max_code_delta=24):\n    native_rgb(rgb);native_alpha(alpha,rgb.shape[:2])\n    if alpha.size>1_000_000:raise ValueError(\'1MP native budget; no resize\')\n    if reviewed is not True or material!=\'opaque\':raise ValueError(\'explicit opaque same-material review required\')\n    if not isinstance(review_note,str) or not review_note.strip() or len(review_note)>2000:\n        raise ValueError(\'bounded same-material review note required\')\n    for mask in (target,donor,protected):\n        if not isinstance(mask,np.ndarray) or mask.dtype!=np.bool_ or mask.shape!=alpha.shape or not mask.any():\n            raise ValueError(\'nonempty native boolean target/donor/protected masks required\')\n    for a,h in [(rgb,source_sha256),(alpha,alpha_sha256),(target,target_sha256),(donor,donor_sha256),(protected,protected_sha256)]:\n        if digest(a)!=h:raise ValueError(\'source/alpha/region binding mismatch\')\n    if np.any(target&(donor|protected)) or np.any(donor&protected):raise ValueError(\'review masks must be disjoint\')\n    if np.any(alpha[target]<128) or np.any(alpha[donor]!=255) or not np.any(alpha[protected]>0):\n        raise ValueError(\'high-alpha targets, opaque donors and protected foreground required\')\n    for v,lo,hi in [(edge_width,0,8),(donor_distance,0,48),(strength,0,1)]:\n        if isinstance(v,(bool,np.bool_)) or not isinstance(v,Real) or not math.isfinite(v) or not lo<v<=hi:\n            raise ValueError(\'finite bounded real controls required\')\n    if isinstance(minimum_gain,(bool,np.bool_)) or not isinstance(minimum_gain,Real) or not math.isfinite(minimum_gain) or not .5<=minimum_gain<1:\n        raise ValueError(\'minimum linear gain0.5..below1 required\')\n    if type(max_code_delta) is not int or not 1<=max_code_delta<=32:raise ValueError(\'integer code delta1..32 required\')\n    hard=alpha>=128\n    if hard.all():raise ValueError(\'background required for boundary distance\')\n    edge=distance_transform_edt(hard)\n    if np.any(edge[target]>edge_width):raise ValueError(\'target outside reviewed edge band\')\n    dist,idx=distance_transform_edt(~donor,return_indices=True)\n    if np.any(dist[target]>donor_distance):raise ValueError(\'target lacks nearby reviewed same-material donors\')\n    s=decode(rgb[target]);near=decode(rgb[idx[0][target],idx[1][target]])\n    weights=np.array([.2126,.7152,.0722]);ys=s@weights;yd=near@weights\n    # Never brighten an edge. A darker donor alone does not prove contamination.\n    ratio=np.minimum(1.,yd/np.maximum(ys,1e-12));ratio[ys<=1e-12]=1.\n    bounded=np.maximum(ratio,minimum_gain);gain=1-strength*(1-bounded)\n    result=encode(s*gain[:,None]);delta=rgb[target].astype(np.int16)-result.astype(np.int16)\n    if np.any(delta<0):raise RuntimeError(\'darken-only invariant failed\')\n    if np.any(delta>max_code_delta):raise ValueError(\'requested edge correction exceeds code-delta budget\')\n    out=rgb.copy();out[target]=result;changed=np.any(out!=rgb,axis=2)\n    if np.any(changed&~target) or not np.array_equal(out[protected],rgb[protected]):raise RuntimeError(\'source protection invariant failed\')\n    return out,dict(schema=\'reviewed-edge-gain.v1\',source_sha256=digest(rgb),alpha_sha256=digest(alpha),\n        target_sha256=digest(target),donor_sha256=digest(donor),protected_sha256=digest(protected),\n        output_sha256=digest(out),review_note=review_note,material=material,edge_width=float(edge_width),\n        donor_distance=float(donor_distance),strength=float(strength),minimum_gain=float(minimum_gain),\n        max_code_delta=max_code_delta,observed_maximum_code_delta=int(delta.max()),\n        target_pixels=int(target.sum()),changed_pixels=int(changed.sum()),\n        changed_opaque_pixels=int((changed&(alpha==255)).sum()),\n        donor_gain_capped_pixels=int((ratio<minimum_gain).sum()),\n        actual_minimum_linear_gain=float(gain.min()),darken_only=True,\n        alpha_changed=False,protected_rgb_exact=True,outside_target_rgb_exact=True,\n        fully_source_rgb_exact=not bool(changed.any()),source_resampled=False,\n        accepted=False,default_pipeline_changed=False,physical_recovery=False,\n        independent_ground_truth=False,\n        limitations=[\'Same-material donors and spill are reviewer hypotheses, not detected facts.\',\n         \'Darkening can destroy genuine highlights, material variation and edge texture.\',\n         \'Source foreground RGB is edited, including opaque pixels; it is not source-RGB locked.\',\n         \'Alpha errors, geometry, background ghosts and label semantics are not corrected.\',\n         \'Native light/dark inspection is required; numeric bounds are not quality acceptance.\'])\n'
# ---- embedded module: renderello/reviewed_edge_gain.py ----
path = base / 'renderello/reviewed_edge_gain.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Opt-in achromatic edge brightening from reviewed same-material donor RGB.\n\nA local brightness hypothesis, not inverse compositing or physical recovery.\nOpaque edge RGB CAN change. Alpha and RGB outside the target stay exact.\nGenuine contour shadows can be erased. There is no default pipeline integration.\n"""\nimport math\nfrom numbers import Real\nimport numpy as np\nfrom scipy.ndimage import distance_transform_edt\nfrom .candidate_diagnostics import native_rgb,native_alpha,digest\nfrom .checked_research.detail_transfer import decode,encode\n\n\ndef lift_reviewed_edge(rgb,alpha,target,donor,protected,*,source_sha256,\n        alpha_sha256,target_sha256,donor_sha256,protected_sha256,\n        reviewed=False,material=None,review_note=\'\',edge_width=4.,\n        donor_distance=24.,strength=.5,maximum_gain=1.25,max_code_delta=24):\n    native_rgb(rgb);native_alpha(alpha,rgb.shape[:2])\n    if alpha.size>1_000_000:raise ValueError(\'1MP native budget; no resize\')\n    if reviewed is not True or material!=\'opaque\':raise ValueError(\'explicit opaque same-material review required\')\n    if not isinstance(review_note,str) or not review_note.strip() or len(review_note)>2000:\n        raise ValueError(\'bounded same-material review note required\')\n    for mask in (target,donor,protected):\n        if not isinstance(mask,np.ndarray) or mask.dtype!=np.bool_ or mask.shape!=alpha.shape or not mask.any():\n            raise ValueError(\'nonempty native boolean target/donor/protected masks required\')\n    for a,h in [(rgb,source_sha256),(alpha,alpha_sha256),(target,target_sha256),(donor,donor_sha256),(protected,protected_sha256)]:\n        if digest(a)!=h:raise ValueError(\'source/alpha/region binding mismatch\')\n    if np.any(target&(donor|protected)) or np.any(donor&protected):raise ValueError(\'review masks must be disjoint\')\n    if np.any(alpha[target]<128) or np.any(alpha[donor]!=255) or not np.any(alpha[protected]>0):\n        raise ValueError(\'high-alpha targets, opaque donors and protected foreground required\')\n    for v,lo,hi in [(edge_width,0,8),(donor_distance,0,48),(strength,0,1)]:\n        if isinstance(v,(bool,np.bool_)) or not isinstance(v,Real) or not math.isfinite(v) or not lo<v<=hi:\n            raise ValueError(\'finite bounded real controls required\')\n    if isinstance(maximum_gain,(bool,np.bool_)) or not isinstance(maximum_gain,Real) or not math.isfinite(maximum_gain) or not 1<maximum_gain<=1.5:\n        raise ValueError(\'maximum linear gain above1..1.5 required\')\n    if type(max_code_delta) is not int or not 1<=max_code_delta<=32:raise ValueError(\'integer code delta1..32 required\')\n    hard=alpha>=128\n    if hard.all():raise ValueError(\'background required for boundary distance\')\n    edge=distance_transform_edt(hard)\n    if np.any(edge[target]>edge_width):raise ValueError(\'target outside reviewed edge band\')\n    dist,idx=distance_transform_edt(~donor,return_indices=True)\n    if np.any(dist[target]>donor_distance):raise ValueError(\'target lacks nearby reviewed same-material donors\')\n    s=decode(rgb[target]);near=decode(rgb[idx[0][target],idx[1][target]])\n    weights=np.array([.2126,.7152,.0722]);ys=s@weights;yd=near@weights\n    # Never darken an edge. A brighter donor alone does not prove contamination.\n    ratio=np.maximum(1.,yd/np.maximum(ys,1e-12));ratio[ys<=1e-12]=1.\n    bounded=np.minimum(ratio,maximum_gain);gain=1+strength*(bounded-1)\n    result=encode(s*gain[:,None]);delta=result.astype(np.int16)-rgb[target].astype(np.int16)\n    if np.any(delta<0):raise RuntimeError(\'brighten-only invariant failed\')\n    if np.any(delta>max_code_delta):raise ValueError(\'requested edge correction exceeds code-delta budget\')\n    out=rgb.copy();out[target]=result;changed=np.any(out!=rgb,axis=2)\n    if np.any(changed&~target) or not np.array_equal(out[protected],rgb[protected]):raise RuntimeError(\'source protection invariant failed\')\n    return out,dict(schema=\'reviewed-edge-lift.v1\',source_sha256=digest(rgb),alpha_sha256=digest(alpha),\n        target_sha256=digest(target),donor_sha256=digest(donor),protected_sha256=digest(protected),\n        output_sha256=digest(out),review_note=review_note,material=material,edge_width=float(edge_width),\n        donor_distance=float(donor_distance),strength=float(strength),maximum_gain=float(maximum_gain),\n        max_code_delta=max_code_delta,observed_maximum_code_delta=int(delta.max()),\n        target_pixels=int(target.sum()),changed_pixels=int(changed.sum()),\n        changed_opaque_pixels=int((changed&(alpha==255)).sum()),\n        donor_gain_capped_pixels=int((ratio>maximum_gain).sum()),\n        actual_maximum_linear_gain=float(gain.max()),brighten_only=True,\n        alpha_changed=False,protected_rgb_exact=True,outside_target_rgb_exact=True,\n        fully_source_rgb_exact=not bool(changed.any()),source_resampled=False,\n        preencode_gamut_clipped_pixels=int(np.any(s*gain[:,None]>1,axis=1).sum()),\n        accepted=False,default_pipeline_changed=False,physical_recovery=False,\n        independent_ground_truth=False,\n        limitations=[\'Same-material donors and spill are reviewer hypotheses, not detected facts.\',\n         \'Brightening can erase genuine contour shadows, material variation and edge texture.\',\n         \'Source foreground RGB is edited, including opaque pixels; it is not source-RGB locked.\',\n         \'Gamut clipping can alter chromatic ratios; this is not a color-recovery guarantee.\',\n         \'Alpha errors, geometry, background ghosts and label semantics are not corrected.\',\n         \'Native light/dark inspection is required; numeric bounds are not quality acceptance.\'])\n'
# ---- embedded module: renderello/reviewed_edge_lift.py ----
path = base / 'renderello/reviewed_edge_lift.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Opt-in native, source-bound intent/coverage guard. No object detection/repair.\n\nNamed core masks and thresholds are explicit reviewer hypotheses. This checks\ncoverage of those pixels only, independently of a possibly incorrect prior.\nPassing never verifies full-object completeness, opacity, optics or quality.\n"""\nimport math\nimport re\nimport numpy as np\nfrom .candidate_diagnostics import native_rgb, native_alpha, digest\n\ndef review_instance_presence(rgb, rgba, regions, *, bindings, reviewed=False, review_note=\'\'):\n    native_rgb(rgb)\n    if not isinstance(rgba, np.ndarray) or rgba.dtype != np.uint8 or rgba.shape != (*rgb.shape[:2], 4):\n        raise ValueError(\'exact native uint8 RGBA required; no resize\')\n    native_alpha(rgba[:, :, 3], rgb.shape[:2])\n    if reviewed is not True or not isinstance(review_note, str) or (not review_note.strip()) or (len(review_note) > 2000):\n        raise ValueError(\'explicit reviewed intent and bounded note required\')\n    if not isinstance(bindings, dict) or set(bindings) != {\'source\', \'candidate\'} or bindings[\'source\'] != digest(rgb) or (bindings[\'candidate\'] != digest(rgba)):\n        raise ValueError(\'exact source/candidate bindings required\')\n    if not isinstance(regions, list) or not 1 <= len(regions) <= 32:\n        raise ValueError(\'one to32 reviewed regions required\')\n    names = set()\n    rows = []\n    blockers = []\n    changed = int(np.any(rgb != rgba[:, :, :3], 2).sum())\n    if changed:\n        blockers.append(\'CANDIDATE_SOURCE_RGB_CHANGED\')\n    fields = {\'name\', \'mask\', \'mask_sha256\', \'intent\', \'material\', \'alpha_threshold\', \'covered_fraction\', \'mean_alpha\'}\n    for region in regions:\n        if not isinstance(region, dict) or set(region) != fields:\n            raise ValueError(\'exact region declaration fields required\')\n        name = region[\'name\']\n        mask = region[\'mask\']\n        intent = region[\'intent\']\n        material = region[\'material\']\n        if not isinstance(name, str) or re.fullmatch(\'[a-z][a-z0-9_-]{0,79}\', name) is None or name in names:\n            raise ValueError(\'unique lowercase bounded instance slug required\')\n        names.add(name)\n        if not isinstance(mask, np.ndarray) or mask.dtype != np.bool_ or mask.shape != rgb.shape[:2] or (not mask.any()):\n            raise ValueError(\'nonempty native boolean core required\')\n        if region[\'mask_sha256\'] != digest(mask):\n            raise ValueError(\'region binding mismatch\')\n        if intent not in (\'retain\', \'exclude\') or material not in (\'opaque\', \'transparent\', \'mixed\', \'unknown\'):\n            raise ValueError(\'explicit region intent/material required\')\n        threshold = region[\'alpha_threshold\']\n        fraction = region[\'covered_fraction\']\n        mean = region[\'mean_alpha\']\n        if type(threshold) is not int or not 1 <= threshold <= 255:\n            raise ValueError(\'alpha threshold1..255 required\')\n        for x in [fraction, mean]:\n            if type(x) not in (float, int) or not math.isfinite(x) or (not 0 <= x <= 1):\n                raise ValueError(\'finite coverage/mean limits0..1 required\')\n        if intent == \'retain\' and (fraction <= 0 or mean <= 0):\n            raise ValueError(\'nonzero retain limits required\')\n        for previous in regions[:len(rows)]:\n            if previous[\'intent\'] != intent and np.any(previous[\'mask\'] & mask):\n                raise ValueError(\'contradictory retain/exclude cores overlap\')\n        a = rgba[:, :, 3][mask]\n        count = int((a >= threshold).sum())\n        n = int(mask.sum())\n        mass = int(a.astype(np.uint64).sum())\n        covered_ok = count >= fraction * n if intent == \'retain\' else count <= fraction * n\n        mean_ok = mass >= mean * 255 * n if intent == \'retain\' else mass <= mean * 255 * n\n        reasons = []\n        if not covered_ok:\n            reasons.append(\'RETAIN_COVERAGE_LOW\' if intent == \'retain\' else \'EXCLUDED_COVERAGE_HIGH\')\n        if not mean_ok:\n            reasons.append(\'RETAIN_MEAN_ALPHA_LOW\' if intent == \'retain\' else \'EXCLUDED_MEAN_ALPHA_HIGH\')\n        blockers.extend((name + \':\' + x for x in reasons))\n        rows.append(dict(name=name, mask_sha256=digest(mask), intent=intent, material=material, pixels=n, alpha_threshold=threshold, required_covered_fraction=fraction, required_mean_alpha=mean, covered_pixels=count, observed_covered_fraction=count / n, observed_mean_alpha=mass / (255 * n), constraint_satisfied=not reasons, blockers=reasons))\n    return dict(schema=\'reviewed-instance-presence.v1\', bindings=bindings.copy(), review_note=review_note, regions=rows, blockers=blockers, declared_coverage_satisfied=not blockers, source_rgb_changed_pixels=changed, photo_accepted=False, object_completeness_verified=False, instance_detection=False, independent_accuracy=None, physical_opacity_verified=False, transparent_optics_verified=False, candidate_selected=False, output_repaired=False, source_resampled=False, default_pipeline_changed=False, limitations=[\'Names, core masks, material and thresholds are reviewer declarations, not detected instances or truth.\', \'Unreviewed objects, boundaries, labels and full-object completeness are not checked.\', \'A correct low-alpha transparent object can fail a mistaken retention declaration.\', \'Coverage above a threshold is not measured optical opacity or scene appearance.\', \'No prior-mask agreement, inpainting, alpha clamping, repair, compositing or publication acceptance.\'])\n\ndef require_instance_presence(rgb, rgba, regions, **kwargs):\n    """Refuse the entire candidate on a failed declared-coverage audit.\n\n    No altered/partly repaired image is returned. On pass the caller still owns\n    other importer/model/material/quality checks. Arrays are never modified.\n    """\n    report = review_instance_presence(rgb, rgba, regions, **kwargs)\n    if not report[\'declared_coverage_satisfied\']:\n        error = ValueError(\'reviewed instance coverage refused: \' + \',\'.join(report[\'blockers\']))\n        error.review_report = report\n        raise error\n    return report\n'
# ---- embedded module: renderello/reviewed_instance_presence.py ----
path = base / 'renderello/reviewed_instance_presence.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Opt-in reviewed local two-colour edge fit, not optical or alpha truth.\n\nExplicit foreground/background donors and mixing space are hypotheses. Only a\nbounded reviewed fractional edge can change; no support or protected RGB changes.\n"""\nimport math\nimport numpy as np\nfrom scipy.ndimage import distance_transform_edt\nfrom .candidate_diagnostics import native_rgb,native_alpha,digest\n\n\ndef fit_reviewed_local_edge(rgb,alpha,target,foreground_donor,background_donor,protected,*,\n        source_sha256,alpha_sha256,target_sha256,foreground_sha256,background_sha256,protected_sha256,\n        reviewed=False,material=None,review_note=\'\',mixing_space=None,strength=.75,feather=3,\n        edge_width=4,max_distance=16,max_alpha_delta=96,max_rgb_delta=64,max_residual=12,minimum_contrast=48):\n    native_rgb(rgb);native_alpha(alpha,rgb.shape[:2])\n    if alpha.size>1_000_000:raise ValueError(\'1MP budget\')\n    if reviewed is not True or material!=\'opaque\':raise ValueError(\'explicit opaque source and donor review required\')\n    if not isinstance(review_note,str)or not review_note.strip()or len(review_note)>2000:raise ValueError(\'bounded nonempty review note required\')\n    masks=[target,foreground_donor,background_donor,protected]\n    for m in masks:\n        if not isinstance(m,np.ndarray)or m.dtype!=np.bool_ or m.shape!=alpha.shape or not m.any():raise ValueError(\'nonempty native boolean review masks required\')\n    for i,m in enumerate(masks):\n        if any(np.any(m&n)for n in masks[i+1:]):raise ValueError(\'review masks must be disjoint\')\n    for v,h in zip([rgb,alpha]+masks,[source_sha256,alpha_sha256,target_sha256,foreground_sha256,background_sha256,protected_sha256]):\n        if digest(v)!=h:raise ValueError(\'source/alpha/review binding mismatch\')\n    if np.any((alpha[target]==0)|(alpha[target]==255)):raise ValueError(\'fractional target only\')\n    if np.any(alpha[foreground_donor]<250)or np.any(alpha[background_donor]!=0):raise ValueError(\'foreground donor alpha>=250 and background alpha0 required\')\n    if not np.any(alpha[protected]>0):raise ValueError(\'protected product required\')\n    if mixing_space not in (\'srgb\',\'linear\'):raise ValueError(\'explicit mixing-space hypothesis required\')\n    if isinstance(strength,(bool,np.bool_))or not isinstance(strength,(int,float))or not math.isfinite(strength)or not 0<strength<=1:raise ValueError(\'strength in(0,1]\')\n    for v,lo,hi in [(feather,1,8),(edge_width,1,8),(max_distance,1,32),(max_alpha_delta,1,128),(max_rgb_delta,1,96),(max_residual,1,24),(minimum_contrast,16,128)]:\n        if type(v)is not int or not lo<=v<=hi:raise ValueError(\'bounded integer controls required\')\n    hard=alpha>=128\n    if hard.all()or not hard.any()or np.any(distance_transform_edt(hard)[target]>edge_width):raise ValueError(\'nontrivial support and narrow reviewed edge band required\')\n    df,iff=distance_transform_edt(~foreground_donor,return_indices=True);db,ib=distance_transform_edt(~background_donor,return_indices=True)\n    if np.any(df[target]>max_distance)or np.any(db[target]>max_distance):raise ValueError(\'every target needs nearby reviewed donors\')\n    def decode(x):\n        x=x.astype(float)/255\n        return np.where(x<=.04045,x/12.92,((x+.055)/1.055)**2.4)if mixing_space==\'linear\'else x\n    def encode(x):\n        x=np.clip(x,0,1)\n        return 255*(np.where(x<=.0031308,12.92*x,1.055*x**(1/2.4)-.055)if mixing_space==\'linear\'else x)\n    c=decode(rgb[target]);f=decode(rgb[iff[0][target],iff[1][target]]);b=decode(rgb[ib[0][target],ib[1][target]])\n    direction=f-b;denom=np.sum(direction**2,axis=1)\n    proposal=np.clip(np.sum((c-b)*direction,axis=1)/np.maximum(denom,1e-12),0,1)\n    residual=np.max(np.abs(encode(b+proposal[:,None]*direction)-rgb[target]),axis=1)\n    contrast=np.max(np.abs(encode(f)-encode(b)),axis=1)\n    foreground_delta=np.max(np.abs(encode(f)-rgb[target]),axis=1)\n    alpha_delta=np.abs(proposal*255-alpha[target])\n    eligible=(residual<=max_residual)&(contrast>=minimum_contrast)&(foreground_delta<=max_rgb_delta)&(alpha_delta<=max_alpha_delta)\n    eligible_map=np.zeros(alpha.shape,bool);eligible_map[target]=eligible\n    taper=np.minimum(1.,distance_transform_edt(np.pad(eligible_map,1))[1:-1,1:-1][target]/feather);weight=strength*taper\n    out=rgb.copy();aa=alpha.copy();newrgb=np.rint(rgb[target]*(1-weight[:,None])+encode(f)*weight[:,None]).astype(np.uint8);newalpha=np.clip(np.rint(alpha[target]*(1-weight)+proposal*255*weight),1,254).astype(np.uint8)\n    values=out[target];values[eligible]=newrgb[eligible];out[target]=values;av=aa[target];av[eligible]=newalpha[eligible];aa[target]=av\n    if not np.array_equal(out[~target],rgb[~target])or not np.array_equal(aa[~target],alpha[~target]):raise RuntimeError(\'outside-target invariant failed\')\n    report=dict(schema=\'reviewed-local-edge-fit.v1\',source_sha256=digest(rgb),initial_alpha_sha256=digest(alpha),target_sha256=digest(target),foreground_sha256=digest(foreground_donor),background_sha256=digest(background_donor),protected_sha256=digest(protected),output_rgb_sha256=digest(out),output_alpha_sha256=digest(aa),mixing_space=mixing_space,review_note=review_note,strength=strength,feather=feather,edge_width=edge_width,max_distance=max_distance,max_alpha_delta=max_alpha_delta,max_rgb_delta=max_rgb_delta,max_residual=max_residual,minimum_contrast=minimum_contrast,target_pixels=int(target.sum()),eligible_pixels=int(eligible.sum()),rejected_residual=int((residual>max_residual).sum()),rejected_contrast=int((contrast<minimum_contrast).sum()),rejected_rgb_budget=int((foreground_delta>max_rgb_delta).sum()),rejected_alpha_budget=int((alpha_delta>max_alpha_delta).sum()),rgb_changed_pixels=int(np.any(out!=rgb,2).sum()),alpha_changed_pixels=int((aa!=alpha).sum()),observed_max_rgb_delta=int(np.abs(out.astype(int)-rgb).max()),observed_max_alpha_delta=int(np.abs(aa.astype(int)-alpha).max()),support_exact=True,protected_rgb_alpha_exact=True,source_resampled=False,accepted=False,default_pipeline_changed=False,physical_recovery=False,limitations=[\'Donor correspondence, opacity and mixing space are reviewer hypotheses, not observations under the edge.\',\'RGB and alpha can change at the reviewed boundary, flattening real shading or changing apparent shape.\',\'Fit residual measures agreement with the assumed local model, not optical or photographic accuracy.\',\'Eligibility rejection counts can overlap. Ineligible pixels remain unchanged and may cause transitions.\',\'Lighting, pose, label semantics, contact and glass optics are not recovered.\'])\n    return out,aa,report\n'
# ---- embedded module: renderello/reviewed_local_edge_fit.py ----
path = base / 'renderello/reviewed_local_edge_fit.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Opt-in source-bound transfer of a solver\'s reviewed trimap-unknown patch.\n\nScope/protection are human hypotheses. No inference, geometry or quality claim.\n"""\nimport numpy as np\nfrom scipy.ndimage import distance_transform_edt\nfrom .candidate_diagnostics import native_rgb,native_alpha,digest\nfrom .matting import check_trimap\n\n\ndef apply_reviewed_matting_patch(rgb,alpha,estimated_rgb,estimated_alpha,trimap,region,protected,*,\n        source_sha256,alpha_sha256,estimated_rgb_sha256,estimated_alpha_sha256,\n        trimap_sha256,region_sha256,protected_sha256,reviewed=False,material=None,\n        review_note=\'\',color_mode=None,feather=8,max_rgb_delta=24,max_alpha_delta=255,\n        support_policy=None):\n    native_rgb(rgb);native_alpha(alpha,rgb.shape[:2]);native_rgb(estimated_rgb);native_alpha(estimated_alpha,alpha.shape)\n    if estimated_rgb.shape!=rgb.shape:raise ValueError(\'native candidate shape required; no resize\')\n    if alpha.size>1_000_000:raise ValueError(\'1MP allocation budget\')\n    check_trimap(trimap,alpha.shape)\n    if reviewed is not True or material not in (\'opaque\',\'opaque_fibres\'):raise ValueError(\'explicit reviewed opaque or opaque-fibres region required\')\n    if not isinstance(review_note,str)or not review_note.strip()or len(review_note)>2000:raise ValueError(\'bounded nonempty review note required\')\n    for mask in (region,protected):\n        if not isinstance(mask,np.ndarray)or mask.dtype!=np.bool_ or mask.shape!=alpha.shape or not mask.any():raise ValueError(\'nonempty native boolean region/protection required\')\n    if np.any(region&protected):raise ValueError(\'region and protection must be disjoint\')\n    if not np.any(protected&(alpha>0)):raise ValueError(\'protected source product required\')\n    for a,h in zip([rgb,alpha,estimated_rgb,estimated_alpha,trimap,region,protected],[source_sha256,alpha_sha256,estimated_rgb_sha256,estimated_alpha_sha256,trimap_sha256,region_sha256,protected_sha256]):\n        if digest(a)!=h:raise ValueError(\'source/solver/review binding mismatch\')\n    if color_mode not in (\'source\',\'bounded-estimate\'):raise ValueError(\'explicit source or bounded-estimate colour mode required\')\n    if support_policy not in (\'retain\',\'reviewed-change\'):raise ValueError(\'explicit retain or reviewed-change support policy required\')\n    for v,lo,hi in [(feather,1,16),(max_rgb_delta,1,64),(max_alpha_delta,1,255)]:\n        if type(v)is not int or not lo<=v<=hi:raise ValueError(\'bounded integer feather/edit budgets required\')\n    target=region&(trimap==128)\n    if not target.any():raise ValueError(\'reviewed region has no trimap-unknown pixels\')\n    weight=np.minimum(1.,distance_transform_edt(np.pad(region,1))[1:-1,1:-1]/feather)*target\n    aa=np.rint(alpha*(1-weight)+estimated_alpha*weight).astype(np.uint8)\n    if support_policy==\'retain\':\n        aa[alpha==0]=0;aa[alpha==255]=255\n        fractional=(alpha>0)&(alpha<255);aa[fractional]=np.clip(aa[fractional],1,254)\n    if int(np.abs(aa.astype(np.int16)-alpha).max())>max_alpha_delta:raise ValueError(\'requested alpha change exceeds budget; no partial silent alpha clamp\')\n    out=rgb.copy();color_target=target&(aa>0)&(aa<255)&(alpha<255)\n    if color_mode==\'bounded-estimate\':\n        delta=(estimated_rgb.astype(float)-rgb)*weight[...,None];peak=np.abs(delta).max(2);delta*=np.minimum(1.,max_rgb_delta/np.maximum(peak,1.))[...,None]\n        candidate=np.rint(np.clip(rgb+delta,0,255)).astype(np.uint8);out[color_target]=candidate[color_target]\n    changed_rgb=np.any(out!=rgb,2);changed_alpha=aa!=alpha\n    if np.any(changed_rgb&~target)or np.any(changed_alpha&~target)or not np.array_equal(out[protected],rgb[protected])or not np.array_equal(aa[protected],alpha[protected]):raise RuntimeError(\'patch protection invariant failed\')\n    if np.any(changed_rgb&(alpha==255)):raise RuntimeError(\'original opaque source RGB changed\')\n    support_added=(alpha==0)&(aa>0);support_removed=(alpha>0)&(aa==0)\n    if support_policy==\'retain\'and(support_added.any()or support_removed.any()):raise RuntimeError(\'retained support changed\')\n    return out,aa,dict(schema=\'reviewed-matting-patch.v1\',source_sha256=digest(rgb),initial_alpha_sha256=digest(alpha),estimated_rgb_sha256=digest(estimated_rgb),estimated_alpha_sha256=digest(estimated_alpha),trimap_sha256=digest(trimap),region_sha256=digest(region),protected_sha256=digest(protected),output_rgb_sha256=digest(out),output_alpha_sha256=digest(aa),review_note=review_note,material=material,color_mode=color_mode,feather=feather,max_rgb_delta=max_rgb_delta,max_alpha_delta=max_alpha_delta,support_policy=support_policy,target_pixels=int(target.sum()),changed_rgb_pixels=int(changed_rgb.sum()),changed_alpha_pixels=int(changed_alpha.sum()),support_added_pixels=int(support_added.sum()),support_removed_pixels=int(support_removed.sum()),observed_max_rgb_delta=int(np.abs(out.astype(int)-rgb).max()),observed_max_alpha_delta=int(np.abs(aa.astype(int)-alpha).max()),original_opaque_rgb_exact=True,protected_rgb_alpha_exact=True,outside_unknown_region_exact=True,no_inference=True,source_resampled=False,accepted=False,default_pipeline_changed=False,physical_recovery=False,solver_provenance=\'NOT_AUTHENTICATED_BY_HASH\',limitations=[\'Region, protection and trimap are reviewer hypotheses, not semantic or optical truth.\',\'SHA256 binds candidate bytes but does not authenticate the model or its inputs.\',\'Estimated alpha can remove detail or include background when reviewed-change is selected.\',\'Bounded colour changes can flatten fur texture or erase genuine shading.\',\'Region and eligibility boundaries can leave seams; native light/dark inspection remains required.\',\'No camera, illumination, physical contact or transparent-material recovery.\'])\n'
# ---- embedded module: renderello/reviewed_matting_patch.py ----
path = base / 'renderello/reviewed_matting_patch.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Opt-in bounded use of an external foreground estimate on reviewed edge strips.\n\nThe external estimate is not a source-RGB-locked result. Only the explicit target\ncan change; original alpha, protected RGB and all other RGB remain exact. No model\nis called. This is a colour hypothesis, not recovered foreground or relighting.\n"""\nimport math\nfrom numbers import Real\nimport numpy as np\nfrom scipy.ndimage import distance_transform_edt\nfrom .candidate_diagnostics import native_rgb,native_alpha,digest\n\n\ndef import_reviewed_model_edge(rgb,alpha,estimate,target,protected,*,\n        source_sha256,alpha_sha256,estimate_sha256,target_sha256,protected_sha256,\n        reviewed=False,material=None,review_note=\'\',edge_width=4.,strength=.75,\n        max_code_delta=24,feather=3):\n    native_rgb(rgb);native_rgb(estimate);native_alpha(alpha,rgb.shape[:2])\n    if estimate.shape!=rgb.shape:raise ValueError(\'native estimate shape must match; no resize\')\n    if alpha.size>1_000_000:raise ValueError(\'1MP native allocation budget\')\n    if reviewed is not True or material!=\'opaque\':raise ValueError(\'explicit opaque source/edge review required\')\n    if not isinstance(review_note,str)or not review_note.strip()or len(review_note)>2000:raise ValueError(\'bounded nonempty review note required\')\n    for mask in (target,protected):\n        if not isinstance(mask,np.ndarray)or mask.dtype!=np.bool_ or mask.shape!=alpha.shape or not mask.any():raise ValueError(\'nonempty native boolean target/protected masks required\')\n    for value,expected in ((rgb,source_sha256),(alpha,alpha_sha256),(estimate,estimate_sha256),(target,target_sha256),(protected,protected_sha256)):\n        if digest(value)!=expected:raise ValueError(\'source/alpha/estimate/review binding mismatch\')\n    if np.any(target&protected):raise ValueError(\'target overlaps protected source\')\n    if not np.any(protected&(alpha>0)):raise ValueError(\'protected foreground required\')\n    if np.any(alpha[target]==0):raise ValueError(\'no zero-alpha background RGB edits\')\n    for value,lo,hi in ((edge_width,0,8),(strength,0,1)):\n        if isinstance(value,(bool,np.bool_))or not isinstance(value,Real)or not math.isfinite(value)or not lo<value<=hi:raise ValueError(\'finite bounded controls required\')\n    if type(max_code_delta)is not int or not 1<=max_code_delta<=32:raise ValueError(\'integer code delta1..32 required\')\n    if type(feather)is not int or not 1<=feather<=8:raise ValueError(\'integer feather1..8 required\')\n    hard=alpha>=128\n    if not hard.any()or hard.all():raise ValueError(\'nontrivial foreground/background support required\')\n    if np.any(distance_transform_edt(hard)[target]>edge_width):raise ValueError(\'target outside declared edge band\')\n    # Preserve the estimate\'s signed RGB correction direction. Bound its largest\n    # channel before feathering; do not clip channels independently or silently\n    # adopt its alpha. Zero-pad to treat image-frame target borders consistently.\n    delta=estimate[target].astype(np.float64)-rgb[target].astype(np.float64)\n    peak=np.abs(delta).max(axis=1)\n    gain=np.minimum(1.,max_code_delta/np.maximum(peak,1.))*float(strength)\n    taper=np.minimum(1.,distance_transform_edt(np.pad(target,1))[1:-1,1:-1][target]/feather)\n    correction=delta*(gain*taper)[:,None]\n    out=rgb.copy();out[target]=np.clip(np.rint(rgb[target]+correction),0,255).astype(np.uint8)\n    observed=np.abs(out.astype(np.int16)-rgb.astype(np.int16))\n    changed=np.any(observed,axis=2)\n    if observed.max()>max_code_delta or np.any(changed&~target)or not np.array_equal(out[protected],rgb[protected]):raise RuntimeError(\'reviewed model edge invariant failed\')\n    return out,dict(schema=\'reviewed-model-edge.v1\',source_sha256=digest(rgb),alpha_sha256=digest(alpha),estimate_sha256=digest(estimate),target_sha256=digest(target),protected_sha256=digest(protected),output_sha256=digest(out),review_note=review_note,material=material,edge_width=float(edge_width),strength=float(strength),max_code_delta=max_code_delta,feather=feather,target_pixels=int(target.sum()),changed_rgb_pixels=int(changed.sum()),changed_opaque_rgb_pixels=int((changed&(alpha==255)).sum()),observed_max_code_delta=int(observed.max()),bounded_estimate_pixels=int((peak>max_code_delta).sum()),protected_rgb_exact=True,outside_target_rgb_exact=True,alpha_changed=False,source_resampled=False,model_called=False,accepted=False,default_pipeline_changed=False,physical_recovery=False,independent_ground_truth=False,limitations=[\'External estimate and target review are hypotheses, not foreground truth.\',\'Opaque edge RGB may change; this is not a source-RGB-locked result.\',\'All alpha is retained, including wrong coverage and missing details.\',\'Feathering and code bounds do not prove a seamless or better edge.\',\'Source light, label semantics, material, geometry and contact are not recovered.\'])\n'
# ---- embedded module: renderello/reviewed_model_edge.py ----
path = base / 'renderello/reviewed_model_edge.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Opt-in near-opaque interior repair, never automatic alpha saturation.\n\nOnly an explicit reviewed region on opaque material may approach alpha255. A\nbounded existing opacity deficit and interior clearance are refusal safeguards,\nnot material or segmentation evidence. No default pipeline integration.\n"""\nimport numpy as np\nfrom scipy.ndimage import distance_transform_edt\nfrom .candidate_diagnostics import native_rgb,native_alpha,digest\n\n\ndef restore_reviewed_opaque_core(rgb,alpha,region,*,source_sha256,alpha_sha256,\n        region_sha256,reviewed=False,material=None,review_note=\'\',max_deficit=8,\n        clearance=8,feather=4):\n    native_rgb(rgb);native_alpha(alpha,rgb.shape[:2])\n    if alpha.size>1_000_000:raise ValueError(\'1MP native budget; no resize\')\n    if reviewed is not True or material!=\'opaque\':raise ValueError(\'explicit opaque-core source review required\')\n    if not isinstance(review_note,str)or not review_note.strip()or len(review_note)>2000:raise ValueError(\'bounded nonempty review note required\')\n    if not isinstance(region,np.ndarray)or region.dtype!=np.bool_ or region.shape!=alpha.shape or not region.any():raise ValueError(\'nonempty native boolean reviewed region required\')\n    for a,h in ((rgb,source_sha256),(alpha,alpha_sha256),(region,region_sha256)):\n        if digest(a)!=h:raise ValueError(\'source/alpha/region binding mismatch\')\n    if type(max_deficit)is not int or not 1<=max_deficit<=16:raise ValueError(\'integer opacity deficit1..16 required\')\n    if type(clearance)is not int or not 4<=clearance<=64:raise ValueError(\'integer clearance4..64 required\')\n    if type(feather)is not int or not 2<=feather<=16:raise ValueError(\'integer feather2..16 required\')\n    if np.any(alpha[region]<255-max_deficit):raise ValueError(\'reviewed region exceeds near-opaque deficit budget\')\n    hard=alpha>=128\n    if hard.all()or not hard.any():raise ValueError(\'nontrivial original foreground/background required\')\n    # Padding prevents an image-frame crop from falsely looking interior.\n    interior=distance_transform_edt(np.pad(hard,1))[1:-1,1:-1]\n    if np.any(interior[region]<clearance):raise ValueError(\'reviewed core lacks boundary/frame clearance\')\n    ramp=np.minimum(1.,distance_transform_edt(np.pad(region,1))[1:-1,1:-1]/feather)\n    core=region&(ramp>=1.)\n    if not core.any():raise ValueError(\'reviewed region has no full-strength interior after feathering\')\n    out=alpha.copy();gain=np.rint((255-alpha[region].astype(np.float64))*ramp[region]).astype(np.uint8)\n    out[region]=alpha[region]+gain\n    changed=out!=alpha\n    if np.any(changed&~region)or np.any(out<alpha)or not np.all(out[core]==255):raise RuntimeError(\'opaque-core scope invariant failed\')\n    return rgb.copy(),out,dict(schema=\'reviewed-opaque-core.v1\',source_sha256=digest(rgb),alpha_sha256=digest(alpha),region_sha256=digest(region),output_alpha_sha256=digest(out),review_note=review_note,material=material,max_deficit=max_deficit,clearance=clearance,feather=feather,minimum_clearance=float(interior[region].min()),reviewed_pixels=int(region.sum()),full_strength_pixels=int(core.sum()),new_opaque_pixels=int(((alpha<255)&(out==255)).sum()),alpha_changed_pixels=int(changed.sum()),maximum_alpha_gain=int(gain.max()),all_source_rgb_exact=True,outside_region_alpha_exact=True,source_resampled=False,accepted=False,default_pipeline_changed=False,independent_ground_truth=False,limitations=[\'Opaque material and region membership are reviewer hypotheses, not inferred facts.\',\'Near-opaque alpha does not prove opacity; wrong review can erase real transmission.\',\'Feathering can leave a coverage seam. Inspect native light/dark composites.\',\'Boundary alpha, holes, source illumination, material reflectance and scene contact are unchanged.\'])\n'
# ---- embedded module: renderello/reviewed_opaque_core.py ----
path = base / 'renderello/reviewed_opaque_core.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""File-bound reviewed near-opaque interior repair. Never automatic alpha saturation."""\nfrom pathlib import Path\nimport argparse\nimport json\nimport os\nimport shutil\nimport tempfile\nimport numpy as np\nfrom PIL import Image\nfrom .image_intake import _read_bounded\nfrom .candidate_diagnostics import digest\nfrom .reviewed_edge_cli import read_image, sha, _json_pairs\nfrom .reviewed_opaque_core import restore_reviewed_opaque_core\n\nNAMES = (\'source\', \'alpha\', \'region\')\n\n\ndef load_spec(path, expected):\n    if Path(path).is_symlink():\n        raise ValueError(\'regular spec file required\')\n    raw = _read_bounded(Path(path), 65_536)\n    if sha(raw) != expected:\n        raise ValueError(\'spec file hash mismatch\')\n    def nonfinite(value):\n        raise ValueError(\'nonfinite JSON\')\n    s = json.loads(raw, object_pairs_hook=_json_pairs, parse_constant=nonfinite)\n    fields = {\'schema\', \'files\', \'reviewed\', \'material\', \'review_note\', \'controls\'}\n    if not isinstance(s, dict) or set(s) != fields or s[\'schema\'] != \'reviewed-opaque-core-files.v1\':\n        raise ValueError(\'exact opaque-core spec required\')\n    if not isinstance(s[\'files\'], dict) or set(s[\'files\']) != set(NAMES):\n        raise ValueError(\'all three file bindings required\')\n    for h in s[\'files\'].values():\n        if not isinstance(h, str) or len(h) != 64 or any(c not in \'0123456789abcdef\' for c in h):\n            raise ValueError(\'lowercase file SHA-256 required\')\n    if not isinstance(s[\'controls\'], dict) or set(s[\'controls\']) != {\'max_deficit\', \'clearance\', \'feather\'}:\n        raise ValueError(\'exact explicit opaque-core controls required\')\n    return s\n\n\ndef execute(*, source, alpha, region, spec, spec_sha256, output_dir):\n    dest = Path(output_dir)\n    if dest.exists() or dest.is_symlink():\n        raise ValueError(\'output exists; preserve earlier evidence\')\n    s = load_spec(spec, spec_sha256)\n    arrays, records = [], {}\n    for n, path in zip(NAMES, (source, alpha, region)):\n        if Path(path).is_symlink():\n            raise ValueError(\'regular image file required\')\n        a, record = read_image(path, s[\'files\'][n], n)\n        arrays.append(a); records[n] = record\n    rgb, a, region = arrays\n    out_rgb, out_alpha, metrics = restore_reviewed_opaque_core(rgb, a, region,\n        source_sha256=digest(rgb), alpha_sha256=digest(a), region_sha256=digest(region), reviewed=s[\'reviewed\'], material=s[\'material\'],\n        review_note=s[\'review_note\'], **s[\'controls\'])\n    if not np.array_equal(out_rgb, rgb) or not np.array_equal(out_alpha[~region], a[~region]):\n        raise RuntimeError(\'source or outside-region invariant failed\')\n    dest.parent.mkdir(parents=True, exist_ok=True)\n    temp = Path(tempfile.mkdtemp(prefix=\'.opaque-core-stage-\', dir=dest.parent))\n    try:\n        Image.fromarray(out_rgb).save(temp/\'source-rgb.png\')\n        Image.fromarray(out_alpha).save(temp/\'alpha.png\')\n        Image.fromarray(np.dstack([out_rgb, out_alpha])).save(temp/\'component-rgba.png\')\n        for label, value in [(\'dark\', 35), (\'light\', 242)]:\n            aa = out_alpha.astype(np.uint32)[:, :, None]\n            comp = ((out_rgb.astype(np.uint32)*aa + value*(255-aa)+127)//255).astype(np.uint8)\n            Image.fromarray(comp).save(temp/(label+\'.png\'))\n        manifest = dict(schema=\'reviewed-opaque-core-file-result.v1\', spec_sha256=spec_sha256,\n            inputs=records, controls=s[\'controls\'], metrics=metrics,\n            outputs={p.name: sha(p.read_bytes()) for p in temp.iterdir()},\n            review_is_caller_declaration=True, source_rgb_exact=True, accepted=False,\n            source_resampled=False, default_pipeline_changed=False)\n        (temp/\'manifest.json\').write_text(json.dumps(manifest, indent=2, allow_nan=False)+\'\\n\')\n        dest.mkdir()\n        try:\n            for p in temp.iterdir(): os.rename(p, dest/p.name)\n        except BaseException:\n            shutil.rmtree(dest)\n            raise\n        return manifest\n    finally:\n        shutil.rmtree(temp)\n\n\ndef main(argv=None):\n    p = argparse.ArgumentParser(description=__doc__)\n    for n in NAMES + (\'spec\', \'spec-sha256\', \'output-dir\'): p.add_argument(\'--\'+n, required=True)\n    a = p.parse_args(argv)\n    try: result = execute(**vars(a))\n    except (ValueError, OSError) as exc: p.exit(2, \'REFUSED: \'+str(exc)+\'\\n\')\n    print(json.dumps(result, indent=2, allow_nan=False))\n\n\nif __name__ == \'__main__\': main()\n'
# ---- embedded module: renderello/reviewed_opaque_core_cli.py ----
path = base / 'renderello/reviewed_opaque_core_cli.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Named native part preservation, not segmentation or independent label truth."""\nimport numpy as np\nfrom .candidate_diagnostics import native_rgb,native_alpha,digest\n\n\ndef audit_parts(rgb,prior_alpha,candidate_rgb,candidate_alpha,regions,*,source_sha256,\n                prior_alpha_sha256,reviewed=False,review_note=\'\'):\n    native_rgb(rgb);native_alpha(prior_alpha,rgb.shape[:2]);native_rgb(candidate_rgb)\n    native_alpha(candidate_alpha,prior_alpha.shape)\n    if candidate_rgb.shape!=rgb.shape:raise ValueError(\'same native RGB coordinates required\')\n    if prior_alpha.size>1_000_000:raise ValueError(\'1MP native budget; no resize\')\n    if reviewed is not True or not isinstance(review_note,str) or not review_note.strip() or len(review_note)>2000:\n        raise ValueError(\'explicit region review and bounded note required\')\n    if source_sha256!=digest(rgb) or prior_alpha_sha256!=digest(prior_alpha):\n        raise ValueError(\'source/prior binding mismatch\')\n    if not isinstance(regions,list) or not 1<=len(regions)<=32:raise ValueError(\'one to32 reviewed regions required\')\n    names=set();rows=[]\n    for region in regions:\n        if not isinstance(region,dict):raise ValueError(\'region records required\')\n        name=region.get(\'name\');mask=region.get(\'mask\')\n        if not isinstance(name,str) or not name.strip() or len(name)>100 or name in names:raise ValueError(\'unique bounded region names required\')\n        names.add(name)\n        if not isinstance(mask,np.ndarray) or mask.dtype!=np.bool_ or mask.shape!=prior_alpha.shape or not mask.any():\n            raise ValueError(\'nonempty native boolean region required\')\n        if region.get(\'mask_sha256\')!=digest(mask):raise ValueError(\'region binding mismatch\')\n        if not np.any(mask&(prior_alpha>0)):raise ValueError(\'region must include prior foreground\')\n        before=prior_alpha[mask].astype(np.int16);after=candidate_alpha[mask].astype(np.int16)\n        delta=after-before;rgb_changed=np.any(candidate_rgb[mask]!=rgb[mask],axis=1)\n        rows.append(dict(name=name,mask_sha256=digest(mask),pixels=int(mask.sum()),\n            rgb_changed_pixels=int(rgb_changed.sum()),alpha_changed_pixels=int(np.count_nonzero(delta)),\n            alpha_decreased_pixels=int((delta<0).sum()),alpha_increased_pixels=int((delta>0).sum()),\n            alpha_loss_units=int(np.maximum(-delta,0).sum()),alpha_gain_units=int(np.maximum(delta,0).sum()),\n            prior_covered_pixels_lost=int(((before>0)&(after==0)).sum()),\n            exact_prior_preservation=not bool(rgb_changed.any() or delta.any())))\n    return dict(schema=\'reviewed-native-parts.v1\',source_sha256=digest(rgb),prior_alpha_sha256=digest(prior_alpha),\n        candidate_rgb_sha256=digest(candidate_rgb),candidate_alpha_sha256=digest(candidate_alpha),\n        review_note=review_note,regions=rows,all_named_parts_preserved=all(x[\'exact_prior_preservation\'] for x in rows),\n        accepted=False,independent_ground_truth=False,semantic_completeness=\'NOT_VERIFIED\',\n        limitations=[\'Exact preservation of reviewed prior pixels only. Prior segmentation may be wrong.\',\n            \'Named parts are reviewer declarations, not model-detected semantics.\',\n            \'Unreviewed parts, visible scene appearance and label transcription are not verified.\',\n            \'A changed alpha can be an improvement; a failed preservation check requires review, not automatic repair.\'])\n'
# ---- embedded module: renderello/reviewed_parts.py ----
path = base / 'renderello/reviewed_parts.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Bounded height-band penumbra design on reviewed 2D casts, not light recovery."""\nfrom dataclasses import dataclass,asdict\nimport math\nimport cv2\nimport numpy as np\nfrom .reviewed_silhouette_cast import cast_source_silhouette,SilhouetteCast\nfrom .candidate_diagnostics import digest\nfrom .candidate_harmonization import linear_over\nfrom .checked_research.contact_camera import linear,srgb\n\n@dataclass(frozen=True)\nclass PenumbraProfile:\n    near_sigma:float=.75\n    far_sigma:float=9.\n    far_opacity_ratio:float=.6\n    levels:int=8\n\n\ndef cast_reviewed_penumbra(rgb,alpha,background,contact,review,profile):\n    if not isinstance(profile,PenumbraProfile):raise ValueError(\'typed penumbra profile required\')\n    for name,lo,hi in [(\'near_sigma\',.5,12),(\'far_sigma\',.5,12),(\'far_opacity_ratio\',.2,1.)]:\n        v=getattr(profile,name)\n        if isinstance(v,bool) or not isinstance(v,(int,float)) or not math.isfinite(v) or not lo<=v<=hi:\n            raise ValueError(name+\' outside bounded design controls\')\n    if profile.far_sigma<profile.near_sigma:raise ValueError(\'far sigma must not be smaller than near\')\n    if type(profile.levels) is not int or not 4<=profile.levels<=12:raise ValueError(\'4..12integer bands required\')\n    # Reuse all source/support/contact/light/boundary checks. Margin must cover\n    # the largest new kernel, never silently replace the old review controls.\n    if not isinstance(review,SilhouetteCast):raise ValueError(\'typed silhouette review required\')\n    if review.blur_sigma!=profile.far_sigma:raise ValueError(\'review blur margin must equal maximum profile sigma\')\n    _,_,_,meta=cast_source_silhouette(rgb,alpha,background,contact,review)\n    h,w=alpha.shape;matrix=np.array(meta[\'projection_matrix\'])\n    projected=cv2.warpAffine(alpha.astype(np.float64)/255,matrix,(w,h),flags=cv2.INTER_LINEAR,borderMode=cv2.BORDER_CONSTANT)\n    nonzero=np.where(alpha>0)[0];height=max(1,review.floor_row-int(nonzero.min()))\n    # With this affine projection, target depth is proportional to source\n    # height. That proxy is not recovered 3D distance or light size.\n    depth=np.clip((np.arange(h)-review.floor_row)/(review.cast_y_per_height*height),0,1)[:,None]\n    result=np.zeros((h,w),np.float64)\n    for i,t in enumerate(np.linspace(0,1,profile.levels)):\n        weights=np.maximum(1-np.abs(depth-t)*(profile.levels-1),0)\n        sigma=profile.near_sigma+(profile.far_sigma-profile.near_sigma)*t\n        margin=math.ceil(3*sigma);atten=1-(1-profile.far_opacity_ratio)*t\n        layer=projected*weights*atten\n        result+=cv2.GaussianBlur(layer,(2*margin+1,2*margin+1),sigma,borderType=cv2.BORDER_CONSTANT)\n    cast=np.clip(result,0,1)*review.opacity\n    cast[:review.floor_row]=0;hard=alpha>=128;cast[hard]=0\n    shade=1-(1-contact)*(1-cast);shade[hard]=0\n    out=linear_over(rgb,alpha,srgb(linear(background)*(1-shade[...,None])))\n    if not np.array_equal(out[alpha==255],rgb[alpha==255]):raise RuntimeError(\'opaque source RGB changed\')\n    meta.update(schema=\'reviewed-height-band-penumbra.v1\',profile=asdict(profile),\n        penumbra_proxy=\'affine projected height, not measured3Ddistance\',cast_sha256=digest(cast),\n        combined_shadow_sha256=digest(shade),result_sha256=digest(out),cast_pixels=int((cast>0).sum()))\n    meta[\'limitations\']=meta[\'limitations\'][:-1]+[\'Height-band blur and opacity are authored design approximations, not measured light size or physical penumbra.\']\n    return out,cast,shade,meta\n'
# ---- embedded module: renderello/reviewed_penumbra.py ----
path = base / 'renderello/reviewed_penumbra.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Local explicit reviewed-mask pipeline. No network or automatic checkpoint download."""\nfrom pathlib import Path\nimport io,json,zipfile,uuid\nfrom .prompted_mask import validate_prompts,guarded_file\nfrom .intent_mask import gate_file\n\n\ndef unpack_automatic(data,folder):\n    if not data or len(data)>20_000_000:raise ValueError(\'automatic bundle ZIP cap20MB\')\n    with zipfile.ZipFile(io.BytesIO(data)) as z:\n        infos=z.infolist()\n        if len(infos)>20 or sum(i.file_size for i in infos)>40_000_000:raise ValueError(\'bundle exceeds20files/40MB unpacked\')\n        required={\'source.png\',\'cutout.png\',\'estimated-alpha.png\'}\n        manifests={\'bilateral-mask-manifest.json\',\'pyramid-mask-manifest.json\',\'semantic-mask-manifest.json\'}\n        names=[i.filename for i in infos]\n        if any(\'/\' in n or \'\\\\\' in n or n.startswith(\'.\') for n in names) or len(set(names))!=len(names):raise ValueError(\'flat unique filenames required, no directories\')\n        found=manifests.intersection(names)\n        if len(found)!=1 or not required.issubset(names):raise ValueError(\'missing automatic manifest/alpha/source/cutout\')\n        folder.mkdir()\n        for name in required|found:\n            info=z.getinfo(name)\n            if info.flag_bits&1:raise ValueError(\'encrypted bundle unsupported\')\n            (folder/name).write_bytes(z.read(name))\n\n\ndef run_pipeline(out_root,job,prompts,bundle,checkpoint):\n    import re\n    if checkpoint is None or not Path(checkpoint).is_file():raise ValueError(\'server checkpoint not configured; see UI startup instructions\')\n    if not re.fullmatch(r\'prompt-[a-f0-9]{12}\',job):raise ValueError(\'invalid source job\')\n    prepared=out_root/job;source=prepared/\'original-source.bin\'\n    if not source.is_file():raise ValueError(\'prepared source not found\')\n    from PIL import Image\n    with Image.open(source) as im:validate_prompts(prompts,source,*im.size)\n    result=out_root/(\'intent-\'+uuid.uuid4().hex[:12]);result.mkdir()\n    try:\n        unpack_automatic(bundle,result/\'automatic\')\n        pp=result/\'reviewed-prompts.json\';pp.write_text(json.dumps(prompts,indent=2))\n        guarded_file(source,result/\'reviewed\',pp,Path(checkpoint))\n        report=gate_file(source,result/\'automatic\',result/\'reviewed\',result/\'combined\')\n        # Compact downloadable bundle preserves the inputs, prompts and reviews.\n        archive=result/\'reviewed-mask-result.zip\'\n        with zipfile.ZipFile(archive,\'w\',zipfile.ZIP_DEFLATED) as z:\n            for p in result.rglob(\'*\'):\n                if p.is_file() and p!=archive:z.write(p,str(p.relative_to(result)))\n        return dict(job=result.name,report=report,preview_urls={color:\'/file/\'+result.name+\'/combined/\'+color+\'-review.png\' for color in (\'white\',\'black\',\'magenta\')},download_url=\'/file/\'+result.name+\'/reviewed-mask-result.zip\')\n    except Exception as e:\n        (result/\'FAILED.txt\').write_text(type(e).__name__+\': \'+str(e))\n        raise\n\n\ndef _snapshot(folder):\n    from .intent_mask import sha\n    return {str(p.relative_to(folder)):sha(p) for p in sorted(folder.rglob(\'*\')) if p.is_file() and p.name not in {\'preview-state.json\',\'FAILED.txt\'}}\n\n\ndef preview_pipeline(out_root,job,prompts,bundle,checkpoint):\n    """Infer only. No intersection until a separate exact-preview confirmation."""\n    import re,hashlib\n    from .intent_mask import load_bundle,sha\n    from PIL import Image\n    if checkpoint is None or not Path(checkpoint).is_file():raise ValueError(\'server checkpoint not configured\')\n    if not re.fullmatch(r\'prompt-[a-f0-9]{12}\',job):raise ValueError(\'invalid source job\')\n    source=out_root/job/\'original-source.bin\'\n    if not source.is_file():raise ValueError(\'prepared source not found\')\n    with Image.open(source) as im:\n        objects=validate_prompts(prompts,source,*im.size);size=im.size\n    if any(1 not in o[\'labels\'] for o in objects):raise ValueError(\'at least one explicit positive product point per object is required for UI preview; points do not guarantee quality\')\n    result=out_root/(\'intent-\'+uuid.uuid4().hex[:12]);result.mkdir()\n    try:\n        unpack_automatic(bundle,result/\'automatic\')\n        manifest=next(p.name for p in (result/\'automatic\').glob(\'*manifest.json\'))\n        load_bundle(result/\'automatic\',manifest,sha(source),size)\n        (result/\'original-source.bin\').write_bytes(source.read_bytes())\n        pp=result/\'reviewed-prompts.json\';pp.write_text(json.dumps(prompts,indent=2))\n        guarded_file(result/\'original-source.bin\',result/\'reviewed\',pp,Path(checkpoint))\n        files=_snapshot(result);fingerprint=hashlib.sha256(json.dumps(files,sort_keys=True).encode()).hexdigest()\n        (result/\'preview-state.json\').write_text(json.dumps(dict(files=files,fingerprint=fingerprint,state=\'AWAITING_EXPLICIT_SILHOUETTE_REVIEW\'),indent=2))\n        return dict(job=result.name,fingerprint=fingerprint,stage=\'REVIEW_SILHOUETTE_NOT_COMBINED\',preview_urls={c:\'/file/\'+result.name+\'/reviewed/\'+c+\'-review.png\' for c in (\'white\',\'black\',\'magenta\')})\n    except Exception as e:\n        (result/\'FAILED.txt\').write_text(type(e).__name__+\': \'+str(e));raise\n\n\ndef finish_pipeline(out_root,job,fingerprint,acceptance):\n    import re\n    if acceptance!=\'I inspected this exact inferred silhouette\':raise ValueError(\'explicit silhouette acceptance required\')\n    if not re.fullmatch(r\'intent-[a-f0-9]{12}\',job):raise ValueError(\'invalid preview job\')\n    result=out_root/job;state=json.loads((result/\'preview-state.json\').read_text())\n    if state.get(\'state\')!=\'AWAITING_EXPLICIT_SILHOUETTE_REVIEW\':raise ValueError(\'preview is not awaiting review; preserve existing result\')\n    if fingerprint!=state.get(\'fingerprint\') or _snapshot(result)!=state.get(\'files\'):raise ValueError(\'preview inputs or images changed; infer and review again\')\n    report=gate_file(result/\'original-source.bin\',result/\'automatic\',result/\'reviewed\',result/\'combined\')\n    state[\'state\']=\'EXPLICIT_SILHOUETTE_REVIEW_CONFIRMED_NOT_QUALITY_ACCEPTED\';(result/\'preview-state.json\').write_text(json.dumps(state,indent=2))\n    archive=result/\'reviewed-mask-result.zip\'\n    with zipfile.ZipFile(archive,\'w\',zipfile.ZIP_DEFLATED) as z:\n        for p in result.rglob(\'*\'):\n            if p.is_file() and p!=archive:z.write(p,str(p.relative_to(result)))\n    return dict(job=result.name,report=report,preview_urls={c:\'/file/\'+result.name+\'/combined/\'+c+\'-review.png\' for c in (\'white\',\'black\',\'magenta\')},download_url=\'/file/\'+result.name+\'/reviewed-mask-result.zip\')\n'
# ---- embedded module: renderello/reviewed_pipeline.py ----
path = base / 'renderello/reviewed_pipeline.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Actual-target-bound native placement and designed 2D contact.\n\nOpt-in research. Support/exclusion masks and pose/light labels are caller review\nhypotheses, not inferred geometry. No source relighting, resampling, glass or\nphotographic acceptance. A matching category is necessary, never sufficient.\n"""\nfrom dataclasses import dataclass,asdict,replace\nimport numpy as np\nfrom .candidate_diagnostics import native_rgb,native_alpha,digest\nfrom .contact_spans import ContactPolicy,ContactSpan,contact_map,eligible\nfrom .candidate_harmonization import linear_over\nfrom .checked_research.contact_camera import linear,srgb\n\n\n@dataclass(frozen=True)\nclass ScenePlacementReview:\n    source_sha256:str\n    alpha_sha256:str\n    scene_sha256:str\n    surface_sha256:str\n    exclusion_sha256:str\n    source_pose:str\n    target_pose:str\n    source_light:str\n    target_light:str\n    source_note:str\n    target_note:str\n    source_reviewed:bool=False\n    target_reviewed:bool=False\n\n    def __post_init__(self):\n        for name in (\'source_sha256\',\'alpha_sha256\',\'scene_sha256\',\'surface_sha256\',\'exclusion_sha256\'):\n            h=getattr(self,name)\n            if not isinstance(h,str)or len(h)!=64 or any(c not in \'0123456789abcdef\'for c in h):\n                raise ValueError(\'exact source/alpha/scene/surface/exclusion pixel hashes required\')\n        for name in (\'source_pose\',\'target_pose\'):\n            if getattr(self,name)not in (\'unknown\',\'front\',\'elevated\',\'overhead\',\'suspended\'):\n                raise ValueError(\'explicit reviewed pose category required\')\n        for name in (\'source_light\',\'target_light\'):\n            if getattr(self,name)not in (\'unknown\',\'soft_left\',\'soft_right\',\'overhead_soft\',\'diffuse\',\'hard_left\',\'hard_right\',\'mixed\'):\n                raise ValueError(\'explicit reviewed light category required\')\n        for name in (\'source_note\',\'target_note\'):\n            v=getattr(self,name)\n            if not isinstance(v,str)or not v.strip()or len(v)>2000:raise ValueError(\'bounded source and actual-target review notes required\')\n        if type(self.source_reviewed)is not bool or type(self.target_reviewed)is not bool:\n            raise ValueError(\'boolean source/target review declarations required\')\n\n\ndef plan_reviewed_placement(source,alpha,scene,surface,exclusion,review,policy,spans,*,translation):\n    native_rgb(source);native_alpha(alpha,source.shape[:2]);native_rgb(scene)\n    h,w=alpha.shape;ch,cw=scene.shape[:2]\n    if alpha.size>1_000_000 or ch*cw>2_000_000:raise ValueError(\'native1MPsource/2MPscene research allocation budget\')\n    for value in (surface,exclusion):\n        if not isinstance(value,np.ndarray)or value.dtype!=np.bool_ or value.shape!=(ch,cw):\n            raise ValueError(\'native boolean actual-scene surface/exclusion masks required\')\n    if not isinstance(review,ScenePlacementReview)or not isinstance(policy,ContactPolicy):\n        raise ValueError(\'typed actual-target and contact reviews required\')\n    values=dict(source_sha256=source,alpha_sha256=alpha,scene_sha256=scene,surface_sha256=surface,exclusion_sha256=exclusion)\n    if any(getattr(review,k)!=digest(v)for k,v in values.items()):raise ValueError(\'actual-target/source review binding changed\')\n    if policy.source_sha256!=review.source_sha256 or policy.alpha_sha256!=review.alpha_sha256:\n        raise ValueError(\'contact policy binds different native input\')\n    if not isinstance(translation,(tuple,list))or len(translation)!=2 or any(type(v)is not int or v<0 for v in translation):\n        raise ValueError(\'explicit nonnegative integer native translation required\')\n    if not isinstance(spans,(list,tuple))or any(not isinstance(s,ContactSpan)for s in spans):raise ValueError(\'typed native contact spans required\')\n    if len(spans)>64 or len({s.name for s in spans})!=len(spans):raise ValueError(\'bounded unique native contact spans required\')\n    if any(s.x1>w or s.bottom_max>=h for s in spans):raise ValueError(\'contact spans outside original source\')\n    dx,dy=translation;blocks=[]\n    if not review.source_reviewed:blocks.append(\'SOURCE_REVIEW_NOT_CONFIRMED\')\n    if not review.target_reviewed:blocks.append(\'ACTUAL_TARGET_REVIEW_NOT_CONFIRMED\')\n    if not policy.reviewed:blocks.append(\'CONTACT_REVIEW_NOT_CONFIRMED\')\n    if policy.material!=\'opaque\':blocks.append(\'OPAQUE_ONLY_NO_GLASS_OR_REFLECTION_RECOVERY\')\n    if policy.framing!=\'complete\':blocks.append(\'COMPLETE_SOURCE_NOT_DECLARED\')\n    if review.source_pose in (\'unknown\',\'suspended\')or review.target_pose in (\'unknown\',\'suspended\'):\n        blocks.append(\'SUPPORTED_SOURCE_TARGET_POSE_NOT_ESTABLISHED\')\n    elif review.source_pose!=review.target_pose:blocks.append(\'ACTUAL_TARGET_CAMERA_MISMATCH_NO_VIEWPOINT_CHANGE\')\n    if \'unknown\'in(review.source_light,review.target_light):blocks.append(\'SOURCE_TARGET_LIGHT_NOT_REVIEWED\')\n    elif review.source_light!=review.target_light:blocks.append(\'ACTUAL_TARGET_LIGHT_MISMATCH_NO_PRODUCT_RELIGHTING\')\n    expected_camera=\'flatlay\'if review.target_pose==\'overhead\'else review.target_pose\n    if policy.camera!=expected_camera or policy.source_pose!=review.source_pose:\n        blocks.append(\'CONTACT_POLICY_CONTRADICTS_ACTUAL_SCENE_REVIEW\')\n    if not surface.any():blocks.append(\'NO_REVIEWED_VISIBLE_SUPPORT_SURFACE\')\n    if dx+w>cw or dy+h>ch:blocks.append(\'WHOLE_NATIVE_SOURCE_DOES_NOT_FIT_NO_RESIZE_OR_CROP\')\n    hard=alpha>=128\n    if not hard.any()or hard.all():blocks.append(\'NONTRIVIAL_FOREGROUND_SUPPORT_REQUIRED\')\n    if any(edge.any()for edge in(hard[0],hard[-1],hard[:,0],hard[:,-1])):\n        blocks.append(\'FRAME_TOUCHING_SUPPORT_REQUIRES_SOURCE_REVIEW\')\n    overlap=0;off_surface=None\n    if dx+w<=cw and dy+h<=ch:\n        overlap=int(((alpha>0)&exclusion[dy:dy+h,dx:dx+w]).sum())\n        if overlap:blocks.append(\'PRODUCT_OVERLAPS_REVIEWED_SCENE_EXCLUSION\')\n        if review.target_pose==\'overhead\':\n            if spans:blocks.append(\'FLATLAY_DOES_NOT_USE_UPRIGHT_SPANS\')\n            off_surface=int(((alpha>0)&~surface[dy:dy+h,dx:dx+w]).sum())\n            if off_surface:blocks.append(\'FLATLAY_PRODUCT_FOOTPRINT_OUTSIDE_REVIEWED_SURFACE\')\n        elif review.target_pose in (\'front\',\'elevated\'):\n            if policy.opacity==0:blocks.append(\'UPRIGHT_VISIBLE_CONTACT_REQUIRED\')\n            if not spans:blocks.append(\'UPRIGHT_NATIVE_CONTACT_SPANS_REQUIRED\')\n            else:\n                lower=np.where(hard.any(0),h-1-np.argmax(hard[::-1],axis=0),-1);off_surface=0\n                used=set()\n                for span in spans:\n                    cols=np.arange(span.x0,span.x1);rows=lower[cols]\n                    if any(int(c)in used for c in cols):raise ValueError(\'overlapping native contact spans\')\n                    used.update(map(int,cols))\n                    if np.any((rows<span.bottom_min)|(rows>span.bottom_max)):\n                        blocks.append(\'NATIVE_CONTACT_RANGE_CONTRADICTS_ALPHA\');continue\n                    cy=rows+dy;cx=cols+dx\n                    if np.any(cy+1>=ch):blocks.append(\'CONTACT_REACHES_SCENE_FRAME\');continue\n                    if np.any(cy+policy.length>=ch):blocks.append(\'INSUFFICIENT_FINITE_SHADOW_ROOM\')\n                    bad=~(surface[cy,cx]&surface[cy+1,cx])|exclusion[cy+1,cx]\n                    off_surface+=int(bad.sum())\n                if off_surface:blocks.append(\'CONTACT_FOOTPRINT_OUTSIDE_REVIEWED_SURFACE\')\n    return dict(schema=\'reviewed-actual-scene-placement.v1\',review=asdict(review),contact_policy=asdict(policy),\n        native_spans=[asdict(s)for s in spans],translation=list(translation),source_size=[w,h],scene_size=[cw,ch],\n        blockers=list(dict.fromkeys(blocks)),eligible_for_designed_preview=not blocks,\n        product_exclusion_overlap_pixels=overlap,off_surface_footprint_pixels=off_surface,\n        review_is_caller_declaration=True,actual_target_bytes_bound=True,recovered_geometry=False,\n        source_resampled=False,source_cropped=False,product_relighting=False,accepted=False,\n        limitations=[\'Surface, exclusions, pose and light are reviewer hypotheses, not inferred facts.\',\n            \'Matching pose/light categories do not prove calibrated camera or physically matched illumination.\',\n            \'Alpha completeness, edge contamination, source reflections and existing shadows remain unchanged.\',\n            \'Designed2Dcontact and pixel attachment do not imply photographic or benchmark acceptance.\'])\n\n\nclass PlacementRefused(ValueError):\n    def __init__(self,report):\n        self.report=report\n        super().__init__(\'Actual scene placement refused: \'+\', \'.join(report[\'blockers\']))\n\n\ndef compose_reviewed_placement(source,alpha,scene,surface,exclusion,review,policy,spans,*,translation):\n    report=plan_reviewed_placement(source,alpha,scene,surface,exclusion,review,policy,spans,translation=translation)\n    if report[\'blockers\']:raise PlacementRefused(report)\n    # Source-native eligibility before translating, so scene padding cannot excuse\n    # a source-bound contour/pose claim. No source RGB or alpha edit is made.\n    eligible(source,alpha,policy)\n    dx,dy=translation;h,w=alpha.shape;rgb=np.zeros_like(scene);a=np.zeros(scene.shape[:2],np.uint8)\n    rgb[dy:dy+h,dx:dx+w]=source;a[dy:dy+h,dx:dx+w]=alpha\n    cp=replace(policy,source_sha256=digest(rgb),alpha_sha256=digest(a))\n    shifted=[ContactSpan(s.name,s.x0+dx,s.x1+dx,s.bottom_min+dy,s.bottom_max+dy)for s in spans]\n    shade,contact=contact_map(rgb,a,shifted,cp);outside=(~surface)|exclusion;clipped=shade.copy();clipped[outside]=0\n    if policy.camera!=\'flatlay\':\n        hard=a>=128;lower=np.where(hard.any(0),a.shape[0]-1-np.argmax(hard[::-1],axis=0),-1)\n        for span in shifted:\n            cols=np.arange(span.x0,span.x1)\n            if not np.all(clipped[lower[cols]+1,cols]>0):raise ValueError(\'designed contact detached after target clipping\')\n    out=linear_over(rgb,a,srgb(linear(scene)*(1-clipped[...,None])))\n    if not np.array_equal(out[a==255],rgb[a==255])or np.any(clipped[a>=128]):raise RuntimeError(\'native source/shadow invariant failed\')\n    report.update(contact=contact,shadow_clipped_pixels=int(((shade>0)&outside).sum()),shadow_outside_reviewed_surface_pixels=0,\n        opaque_rgb_exact_pixels=int((a==255).sum()),result_sha256=digest(out),shadow_sha256=digest(clipped),\n        alpha_changed=False,product_rgb_repainted=False,photographic_acceptance=\'NOT_EVALUATED\')\n    return out,clipped,report\n'
# ---- embedded module: renderello/reviewed_scene_placement.py ----
path = base / 'renderello/reviewed_scene_placement.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Source-bound smooth shading-gain hypothesis, never intrinsic/physical truth.\nOnly explicitly reviewed opaque interior pixels change. No albedo replacement.\n"""\nfrom numbers import Real\nimport math\nimport numpy as np\nfrom scipy.ndimage import gaussian_filter,distance_transform_edt\nfrom .candidate_diagnostics import native_rgb,native_alpha,digest\nfrom .checked_research.detail_transfer import decode,encode\n\n\ndef apply_reviewed_shading_gain(rgb,alpha,shading,target,protected,*,source_sha256,\n        alpha_sha256,shading_sha256,target_sha256,protected_sha256,review_note,\n        reviewed=False,material=None,sigma=8.,feather=8.,strength=.5,\n        minimum_gain=.8,max_code_delta=24):\n    native_rgb(rgb);native_alpha(alpha,rgb.shape[:2])\n    if alpha.size>1_000_000:raise ValueError(\'1MP native allocation cap; no resize\')\n    if reviewed is not True or material!=\'opaque\':raise ValueError(\'explicit opaque interior/source alignment review required\')\n    if not isinstance(review_note,str) or not review_note.strip() or len(review_note)>2000:raise ValueError(\'bounded review note required\')\n    if not isinstance(shading,np.ndarray) or shading.dtype.kind!=\'f\' or shading.shape!=alpha.shape or not np.isfinite(shading).all() or np.any(shading<0):raise ValueError(\'native finite nonnegative floating scalar shading required\')\n    for m in (target,protected):\n        if not isinstance(m,np.ndarray) or m.dtype!=np.bool_ or m.shape!=alpha.shape or not m.any():raise ValueError(\'nonempty native boolean reviewed masks required\')\n    if np.any(target&protected) or np.any(alpha[target]!=255):raise ValueError(\'target must be opaque and disjoint from protected regions\')\n    if np.any(target[0]) or np.any(target[-1]) or np.any(target[:,0]) or np.any(target[:,-1]):raise ValueError(\'interior target cannot touch frame\')\n    for a,h in [(rgb,source_sha256),(alpha,alpha_sha256),(shading,shading_sha256),(target,target_sha256),(protected,protected_sha256)]:\n        if digest(a)!=h:raise ValueError(\'source/shading/mask binding mismatch\')\n    for v,lo,hi in [(sigma,1,32),(feather,1,32),(strength,0,1)]:\n        if isinstance(v,(bool,np.bool_)) or not isinstance(v,Real) or not math.isfinite(v) or not lo<v<=hi:raise ValueError(\'finite bounded smooth controls required\')\n    if isinstance(minimum_gain,(bool,np.bool_)) or not isinstance(minimum_gain,Real) or not math.isfinite(minimum_gain) or not .5<=minimum_gain<1:raise ValueError(\'minimum gain0.5..below1 required\')\n    if type(max_code_delta)is not int or not 1<=max_code_delta<=32:raise ValueError(\'integer delta1..32 required\')\n    # Normalized smoothing excludes other materials/labels from the reference.\n    weights=target.astype(float);den=gaussian_filter(weights,sigma,mode=\'constant\')\n    smoothed=gaussian_filter(shading.astype(float)*weights,sigma,mode=\'constant\')/np.maximum(den,1e-12)\n    values=smoothed[target]\n    if np.any(values<=1e-8):raise ValueError(\'positive reviewed shading required\')\n    reference=float(np.median(values));base=np.maximum(minimum_gain,np.minimum(1.,reference/values))\n    distance=distance_transform_edt(target)[target]\n    # First target row has gain1; smoothstep eases to the full bounded change.\n    t=np.clip((distance-1)/feather,0,1);ease=t*t*(3-2*t);gain=1-strength*ease*(1-base)\n    result=encode(decode(rgb[target])*gain[:,None]);delta=rgb[target].astype(np.int16)-result.astype(np.int16)\n    if np.any(delta<0):raise RuntimeError(\'darken-only invariant failed\')\n    if np.any(delta>max_code_delta):raise ValueError(\'shading hypothesis exceeds channel delta budget\')\n    out=rgb.copy();out[target]=result;changed=np.any(out!=rgb,axis=2)\n    if np.any(changed&~target) or not np.array_equal(out[protected],rgb[protected]):raise RuntimeError(\'protected source invariant failed\')\n    return out,dict(schema=\'reviewed-shading-gain.v1\',source_sha256=digest(rgb),alpha_sha256=digest(alpha),shading_sha256=digest(shading),target_sha256=digest(target),protected_sha256=digest(protected),result_sha256=digest(out),review_note=review_note,sigma=float(sigma),feather=float(feather),strength=float(strength),minimum_gain=float(minimum_gain),max_code_delta=max_code_delta,changed_pixels=int(changed.sum()),observed_maximum_code_delta=int(delta.max()),reference_shading=reference,actual_minimum_gain=float(gain.min()),source_rgb_exact=not bool(changed.any()),alpha_changed=False,protected_rgb_exact=True,outside_target_rgb_exact=True,source_resampled=False,albedo_replaced=False,physical_relighting=False,intrinsics_truth=False,accepted=False,default_changed=False,limitations=[\'Saved model shading is an aligned hypothesis, not recovered physical illumination.\',\'Achromatic darkening can damage real material color, highlights and texture.\',\'Outline, label, pose, contact and background mismatch are not corrected.\',\'Native visual comparison required; no numeric quality acceptance.\'])\n'
# ---- embedded module: renderello/reviewed_shading_gain.py ----
path = base / 'renderello/reviewed_shading_gain.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Opt-in source-reviewed affine silhouette cast, not recovered physical light.\n\nOrthographic-like 2D projection on a declared neutral floor. No source RGB,\nalpha, camera or material edit. Existing contact is retained separately.\n"""\nfrom dataclasses import dataclass,asdict\nimport math\nimport cv2\nimport numpy as np\nfrom .candidate_diagnostics import native_rgb,native_alpha,digest\nfrom .candidate_harmonization import linear_over\nfrom .checked_research.contact_camera import linear,srgb\n\n@dataclass(frozen=True)\nclass SilhouetteCast:\n    floor_row:int\n    support_x0:int\n    support_x1:int\n    cast_x_per_height:float\n    cast_y_per_height:float\n    blur_sigma:float\n    opacity:float\n    source_sha256:str\n    alpha_sha256:str\n    background_sha256:str\n    contact_sha256:str\n    review_note:str\n    reviewed:bool=False\n    source_key_side:str=\'left\'\n    floor_kind:str=\'continuous_neutral\'\n\n\ndef cast_source_silhouette(rgb,alpha,background,contact,review):\n    native_rgb(rgb);native_rgb(background);native_alpha(alpha,rgb.shape[:2])\n    h,w=alpha.shape\n    if rgb.shape!=background.shape or h*w>2_000_000:raise ValueError(\'same native canvas within2MP required\')\n    if not isinstance(review,SilhouetteCast) or review.reviewed is not True or review.floor_kind!=\'continuous_neutral\':\n        raise ValueError(\'typed reviewed continuous neutral floor required\')\n    if not isinstance(review.review_note,str) or not review.review_note.strip() or len(review.review_note)>2000:\n        raise ValueError(\'source light/support review note required\')\n    if not isinstance(contact,np.ndarray) or contact.dtype.kind!=\'f\' or contact.shape!=alpha.shape or not np.isfinite(contact).all() or np.any((contact<0)|(contact>.5)):\n        raise ValueError(\'finite native contact attenuation0..0.5required\')\n    bindings=[(rgb,review.source_sha256),(alpha,review.alpha_sha256),(background,review.background_sha256),(contact,review.contact_sha256)]\n    if any(digest(v)!=s for v,s in bindings):raise ValueError(\'source/alpha/background/contact binding mismatch\')\n    if any(type(getattr(review,k)) is not int for k in [\'floor_row\',\'support_x0\',\'support_x1\']):raise ValueError(\'integer native support required\')\n    if not 0<=review.support_x0<review.support_x1<=w or not 0<=review.floor_row<h-1:raise ValueError(\'support outside canvas\')\n    ranges=[(\'cast_x_per_height\',-.8,.8),(\'cast_y_per_height\',.03,.35),(\'blur_sigma\',.5,12),(\'opacity\',0,.4)]\n    for name,lo,hi in ranges:\n        v=getattr(review,name)\n        if isinstance(v,bool) or not isinstance(v,(int,float)) or not math.isfinite(v) or not lo<=v<=hi:raise ValueError(name+\' outside bounded controls\')\n    if review.source_key_side not in (\'left\',\'right\'):raise ValueError(\'reviewed source key side left/right required\')\n    if (review.source_key_side==\'left\' and review.cast_x_per_height<=0) or (review.source_key_side==\'right\' and review.cast_x_per_height>=0):\n        raise ValueError(\'cast must point away from reviewed side key\')\n    hard=alpha>=128\n    if not hard.any() or hard.all():raise ValueError(\'nontrivial native silhouette required\')\n    if any(e.any() for e in [hard[0],hard[-1],hard[:,0],hard[:,-1]]):raise ValueError(\'complete silhouette must not touch frame\')\n    if np.any(contact[hard]):raise ValueError(\'existing contact crosses hard foreground\')\n    lower=np.where(hard.any(0),h-1-np.argmax(hard[::-1],axis=0),-1)\n    cols=np.arange(review.support_x0,review.support_x1);rows=lower[cols]\n    if np.any(np.abs(rows-review.floor_row)>2) or not np.all(contact[rows+1,cols]>0):\n        raise ValueError(\'reviewed base/floor/contact not attached\')\n    yy,xx=np.where(alpha>0)\n    if yy.max()>review.floor_row+2:raise ValueError(\'nonzero source alpha below declared floor beyond2pixel tolerance\')\n    # y height above support maps to positive floor depth; no hidden clipping.\n    kx,ky=review.cast_x_per_height,review.cast_y_per_height\n    tx=xx+kx*(review.floor_row-yy);ty=review.floor_row+ky*(review.floor_row-yy)\n    margin=math.ceil(3*review.blur_sigma)\n    if tx.min()-margin<0 or tx.max()+margin>=w or ty.max()+margin>=h:\n        raise ValueError(\'projected silhouette plus3sigma blur leaves canvas; refuse rather than truncate\')\n    matrix=np.array([[1,-kx,kx*review.floor_row],[0,-ky,(1+ky)*review.floor_row]],np.float64)\n    projection=cv2.warpAffine(alpha.astype(np.float64)/255,matrix,(w,h),flags=cv2.INTER_LINEAR,borderMode=cv2.BORDER_CONSTANT)\n    blurred=cv2.GaussianBlur(projection,(2*margin+1,2*margin+1),review.blur_sigma,borderType=cv2.BORDER_CONSTANT)\n    cast=blurred*review.opacity\n    cast[:review.floor_row]=0\n    cast[hard]=0\n    # Multiplicative attenuation keeps contact readable under the cast.\n    shade=1-(1-contact)*(1-cast);shade[hard]=0\n    out=linear_over(rgb,alpha,srgb(linear(background)*(1-shade[...,None])))\n    if not np.array_equal(out[alpha==255],rgb[alpha==255]):raise RuntimeError(\'opaque source RGB changed\')\n    return out,cast,shade,dict(schema=\'reviewed-silhouette-cast.v1\',review=asdict(review),\n        projection_matrix=matrix.tolist(),cast_sha256=digest(cast),combined_shadow_sha256=digest(shade),\n        result_sha256=digest(out),cast_pixels=int((cast>0).sum()),source_resampled=False,\n        source_rgb_repainted=False,alpha_repainted=False,opaque_source_pixels_exact=True,\n        recovered_geometry=False,physical_relighting=False,accepted=False,default_pipeline_changed=False,\n        limitations=[\'2D affine silhouette projection, not recovered3Dshape or physical illumination.\',\n          \'Source light side, opaque material and floor support are reviewer declarations.\',\n          \'Source highlights/reflections, alpha errors and material mismatch remain unchanged.\',\n          \'Uniform floor blur/opacity is a design approximation, not a measured penumbra.\'])\n'
# ---- embedded module: renderello/reviewed_silhouette_cast.py ----
path = base / 'renderello/reviewed_silhouette_cast.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Explicit source/mask coordinate preparation. Not enabled by legacy routes.\n\nA prepared image is a derivative, not an invented/reconstructed product. Existing\nalpha and caller masks are not evidence of semantic segmentation correctness.\n"""\nfrom pathlib import Path\nimport hashlib,json,tempfile,shutil\nfrom PIL import Image\nfrom .image_intake import inspect_image,IntakePolicy,IntakeError,_read_bounded\n\n_TRANSPOSE={2:Image.Transpose.FLIP_LEFT_RIGHT,3:Image.Transpose.ROTATE_180,\n 4:Image.Transpose.FLIP_TOP_BOTTOM,5:Image.Transpose.TRANSPOSE,\n 6:Image.Transpose.ROTATE_270,7:Image.Transpose.TRANSVERSE,8:Image.Transpose.ROTATE_90}\n\ndef prepare_reviewed_source(source,destination,*,mask=None,mask_coordinates=None,policy=None):\n """Create a new normalized derivative and reviewed-coordinate mask.\n\n For an external mask, coordinates must explicitly be \'encoded\' (raw pixel\n grid before EXIF) or \'display\' (normalized display orientation). Width/height\n alone cannot disambiguate flips,180-degree turns,or square images. An external\n mask conflicting with source alpha is refused,never multiplied/replaced.\n """\n dest=Path(destination)\n if dest.exists():raise FileExistsError(\'Preserve previous prepared sources\')\n if mask is None and mask_coordinates is not None:raise ValueError(\'Mask coordinates without mask\')\n if mask is not None and mask_coordinates not in (\'encoded\',\'display\'):raise ValueError(\'Declare encoded or display mask coordinates\')\n p=policy or IntakePolicy();r=inspect_image(source,p)\n try:\n  image=r.image\n  alpha=image.getchannel(\'A\') if image.mode==\'RGBA\' else None\n  mask_hash=None\n  if mask is not None:\n   data=_read_bounded(Path(mask),p.max_bytes);mask_hash=hashlib.sha256(data).hexdigest()\n   # Pillow can expose a 16-bit grayscale PNG as I; reject before decode.\n   if not data.startswith(b\'\\x89PNG\\r\\n\\x1a\\n\') or len(data)<29 or data[12:16]!=b\'IHDR\' or data[24]!=8:raise IntakeError(\'Mask must be an 8-bit PNG, no silent depth conversion\')\n   from io import BytesIO\n   with Image.open(BytesIO(data)) as m:\n    # External mask carries no hidden orientation or colour interpretation.\n    if m.format!=\'PNG\' or m.mode!=\'L\' or getattr(m,\'n_frames\',1)!=1:raise IntakeError(\'Mask must be single-frame 8-bit grayscale PNG\')\n    if m.getexif().get(274,1)!=1 or m.info.get(\'icc_profile\') or \'transparency\' in m.info:raise IntakeError(\'Mask metadata is ambiguous; use plain grayscale coordinates\')\n    expected=tuple(r.record[\'input_size\'] if mask_coordinates==\'encoded\' else r.record[\'output_size\'])\n    if m.size!=expected:raise IntakeError(\'Mask dimensions disagree with declared coordinates\')\n    m.load();matte=m.copy()\n   o=r.record[\'orientation\']\n   if mask_coordinates==\'encoded\' and o in _TRANSPOSE:matte=matte.transpose(_TRANSPOSE[o])\n   if alpha is not None and matte.tobytes()!=alpha.tobytes():raise IntakeError(\'External mask conflicts with embedded source alpha\')\n   alpha=matte\n  if alpha is None:raise IntakeError(\'No reviewed alpha: prepare an explicit mask or provide embedded alpha\')\n  if alpha.getextrema()[1]==0:raise IntakeError(\'Empty alpha cannot represent a visible product\')\n  # Staging prevents a complete-looking output after decode/write failure.\n  dest.parent.mkdir(parents=True,exist_ok=True)\n  stage=Path(tempfile.mkdtemp(prefix=\'.\'+dest.name+\'-\',dir=dest.parent))\n  try:\n   rgb=image.convert(\'RGB\');rgb.info.clear();alpha.info.clear()\n   rgb.save(stage/\'normalized-source.png\');alpha.save(stage/\'reviewed-alpha.png\')\n   for path,ref in [(stage/\'normalized-source.png\',rgb),(stage/\'reviewed-alpha.png\',alpha)]:\n    with Image.open(path) as check:\n     check.load()\n     if check.size!=ref.size or check.mode!=ref.mode or check.tobytes()!=ref.tobytes():raise IntakeError(\'Prepared PNG round-trip mismatch\')\n   record=dict(intake=r.record,mask_coordinates=mask_coordinates,external_mask_sha256=mask_hash,\n    source_png_sha256=hashlib.sha256((stage/\'normalized-source.png\').read_bytes()).hexdigest(),\n    alpha_png_sha256=hashlib.sha256((stage/\'reviewed-alpha.png\').read_bytes()).hexdigest(),\n    semantic_mask_accuracy=\'NOT_VERIFIED\',publication_status=\'NOT_EVALUATED\',legacy_routes_changed=False)\n   (stage/\'prepared-source.json\').write_text(json.dumps(record,indent=2))\n   dest.mkdir()\n   try:\n    for child in stage.iterdir():child.rename(dest/child.name)\n   except BaseException:\n    (dest/\'PREPARATION_INCOMPLETE\').touch();raise\n   return record\n  finally:\n   # Only the agent-created staging directory, never a caller\'s folder.\n   if stage.exists():shutil.rmtree(stage)\n finally:r.close()\n\n\ndef main(argv=None):\n import argparse\n p=argparse.ArgumentParser(description=\'Prepare a separate normalized source and explicitly aligned mask; legacy generation defaults stay unchanged\')\n p.add_argument(\'--source\',required=True,type=Path);p.add_argument(\'--out\',required=True,type=Path)\n p.add_argument(\'--mask\',type=Path);p.add_argument(\'--mask-coordinates\',choices=[\'encoded\',\'display\'])\n p.add_argument(\'--allow-colour-conversion\',action=\'store_true\')\n a=p.parse_args(argv)\n result=prepare_reviewed_source(a.source,a.out,mask=a.mask,mask_coordinates=a.mask_coordinates,policy=IntakePolicy(allow_colour_conversion=a.allow_colour_conversion))\n print(json.dumps(result,indent=2))\n\nif __name__==\'__main__\':main()\n'
# ---- embedded module: renderello/reviewed_source.py ----
path = base / 'renderello/reviewed_source.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Reviewed source-white floor tint illustration, not shadow recovery or relighting.\n\nOnly for a manually reviewed neutral-white source floor and a light, flat target.\nRetains measured source-floor darkening instead of inventing an ellipse. Fractional\nalpha and source-floor classification remain estimates. No default integration.\n"""\nimport numpy as np\nfrom .candidate_diagnostics import native_rgb, native_alpha, digest\n\n\ndef tint_reviewed_source_floor(rgb, alpha, background, selection, *,\n        source_sha256, alpha_sha256, background_sha256, selection_sha256,\n        reviewed=False, source_floor=None, review_note=\'\'):\n    native_rgb(rgb); native_alpha(alpha, rgb.shape[:2]); native_rgb(background)\n    if rgb.shape != background.shape or alpha.size > 1_000_000:\n        raise ValueError(\'native matching target and 1MP budget required\')\n    if reviewed is not True or source_floor != \'neutral_white_opaque\':\n        raise ValueError(\'explicit neutral-white opaque source-floor review required\')\n    if not isinstance(review_note, str) or not review_note.strip() or len(review_note)>2000:\n        raise ValueError(\'bounded source-floor review note required\')\n    native_alpha(selection, alpha.shape)\n    if not selection.any() or not np.any((selection>0)&(alpha==0)):\n        raise ValueError(\'nonempty reviewed source-floor selection required\')\n    if not np.any(alpha==255):\n        raise ValueError(\'opaque source anchor required\')\n    for arr,h in [(rgb,source_sha256),(alpha,alpha_sha256),\n                  (background,background_sha256),(selection,selection_sha256)]:\n        if digest(arr)!=h: raise ValueError(\'source/alpha/background/selection binding mismatch\')\n    # A light flat target is an explicit limit, not evidence the physical light matches.\n    if background.min()<160 or not np.all(background==background[0,0]):\n        raise ValueError(\'light flat target only; no dark/material scene extrapolation\')\n    observed=(selection>0)&(alpha==0)\n    spread=rgb.max(axis=2).astype(int)-rgb.min(axis=2).astype(int)\n    if np.any(spread[observed]>12):\n        raise ValueError(\'selected observed source floor is not neutral\')\n    # sRGB-domain tint is deliberately labelled, not a radiometric shadow estimate.\n    a=alpha[:,:,None].astype(np.float64)/255\n    m=selection[:,:,None].astype(np.float64)/255\n    base=rgb*a+background*(1-a)\n    floor=background*((1-m)+m*(rgb.astype(np.float64)/255))\n    out=np.rint(rgb*a+floor*(1-a)).clip(0,255).astype(np.uint8)\n    baseline=np.rint(base).astype(np.uint8)\n    opaque=alpha==255\n    if not np.array_equal(out[opaque],rgb[opaque]) or not np.array_equal(out[selection==0],baseline[selection==0]):\n        raise RuntimeError(\'source/outside-selection invariants failed\')\n    return out,dict(schema=\'reviewed-source-floor-tint.v1\',\n        source_sha256=digest(rgb),alpha_sha256=digest(alpha),\n        background_sha256=digest(background),selection_sha256=digest(selection),\n        output_sha256=digest(out),review_note=review_note,\n        opaque_source_rgb_exact=True,outside_selection_baseline_exact=True,\n        source_resampled=False,source_lighting_unchanged=True,\n        physical_relighting=False,recovered_shadow=False,accepted=False,\n        default_pipeline_changed=False,changed_pixels=int(np.any(out!=baseline,axis=2).sum()),\n        limitations=[\'Reviewer source-floor selection is not semantic truth.\',\n          \'Source background must actually be neutral white; a near-white source is not calibrated.\',\n          \'sRGB tint approximation, not a physical shadow or material model.\',\n          \'Fractional mask errors may double-darken source boundaries.\',\n          \'Wrong source-floor annotation can carry source objects or marks into the new background.\',\n          \'Flat light backgrounds only. No photographic acceptance or production default.\'])\n'
# ---- embedded module: renderello/reviewed_source_floor.py ----
path = base / 'renderello/reviewed_source_floor.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Native, explicit opt-in reviewed source-floor illustration. No generation/API.\n\nLight flat targets only. Annotations and uncalibrated source-floor intensity are\nnot physical geometry, relighting or photographic acceptance. Defaults untouched.\n"""\nfrom pathlib import Path\nimport argparse\nimport hashlib\nimport json\nimport shutil\nimport tempfile\nfrom io import BytesIO\n\nMAX_INPUT_BYTES = 16_000_000\n\n\ndef _snapshot(path):\n    with Path(path).open(\'rb\') as f:\n        data = f.read(MAX_INPUT_BYTES + 1)\n    if len(data) > MAX_INPUT_BYTES:\n        raise ValueError(\'input exceeds16MB bounded snapshot\')\n    return data\nimport numpy as np\nfrom PIL import Image\nfrom .candidate_diagnostics import digest\nfrom .reviewed_source_floor import tint_reviewed_source_floor\n\n\ndef sha256_file(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for block in iter(lambda: f.read(1024 * 1024), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef run_files(source, alpha, selection, output, *, source_sha256,\n              alpha_sha256, selection_sha256, background_rgb,\n              reviewed=False, review_note=\'\'):\n    paths = [Path(source), Path(alpha), Path(selection)]\n    output = Path(output)\n    if output.exists():\n        raise ValueError(\'preserve previous output\')\n    snapshots = []\n    for path, expected in zip(paths, [source_sha256, alpha_sha256, selection_sha256]):\n        if not isinstance(expected, str) or len(expected) != 64 or any(c not in \'0123456789abcdef\' for c in expected):\n            raise ValueError(\'explicit lowercase file SHA256 required\')\n        data = _snapshot(path)\n        if hashlib.sha256(data).hexdigest() != expected:\n            raise ValueError(\'source/alpha/selection file binding mismatch\')\n        snapshots.append(data)\n    if not isinstance(background_rgb, (list, tuple)) or len(background_rgb) != 3 or any(type(v) is not int or not 160 <= v <= 255 for v in background_rgb):\n        raise ValueError(\'three integer light-background channels160..255 required\')\n    arrays = []\n    icc = None\n    for index, data in enumerate(snapshots):\n        with Image.open(BytesIO(data)) as im:\n            if im.mode != (\'RGB\' if index == 0 else \'L\'):\n                raise ValueError(\'native RGB source and L alpha/selection, no conversion\')\n            if im.width * im.height > 1_000_000 or min(im.size) < 2:\n                raise ValueError(\'native2px..1MP required, no resize\')\n            if index == 0:\n                icc = im.info.get(\'icc_profile\')\n            arrays.append(np.array(im))\n    rgb, a, selected = arrays\n    if a.shape != rgb.shape[:2] or selected.shape != a.shape:\n        raise ValueError(\'native dimensions must match\')\n    bg = np.full_like(rgb, background_rgb)\n    out, record = tint_reviewed_source_floor(rgb, a, bg, selected,\n        source_sha256=digest(rgb), alpha_sha256=digest(a),\n        background_sha256=digest(bg), selection_sha256=digest(selected),\n        reviewed=reviewed, source_floor=\'neutral_white_opaque\', review_note=review_note)\n    base = np.rint(rgb * (a[:, :, None] / 255) + bg * (1 - a[:, :, None] / 255)).astype(np.uint8)\n    record.update(file_bindings=dict(zip([\'source\', \'alpha\', \'selection\'],\n        [source_sha256, alpha_sha256, selection_sha256])),\n        background_rgb=list(background_rgb), source_size=[rgb.shape[1], rgb.shape[0]],\n        original_alpha_unchanged=True, paid_calls=0, local_review_only=True,\n        input_read_semantics=\'bounded_immutable_byte_snapshots_hashed_before_decode\',\n        max_input_file_bytes=MAX_INPUT_BYTES)\n    output.parent.mkdir(parents=True, exist_ok=True)\n    temp = Path(tempfile.mkdtemp(prefix=\'reviewed-source-floor-\', dir=output.parent))\n    try:\n        kw = {\'icc_profile\': icc} if icc else {}\n        for name, value in [(\'no-floor.png\', base), (\'reviewed-floor.png\', out)]:\n            Image.fromarray(value).save(temp / name, **kw)\n            saved = Image.open(temp / name)\n            if not np.array_equal(np.array(saved), value) or saved.info.get(\'icc_profile\') != icc:\n                raise RuntimeError(\'export RGB/profile verification failed\')\n        (temp / \'review.json\').write_text(json.dumps(record, indent=2))\n        if output.exists():\n            raise ValueError(\'output appeared during run; preserve it\')\n        temp.rename(output)\n    finally:\n        if temp.exists():\n            shutil.rmtree(temp)\n    return record\n\n\ndef main(argv=None):\n    p = argparse.ArgumentParser(description=__doc__)\n    for name in [\'source\', \'alpha\', \'selection\', \'output\']:\n        p.add_argument(\'--\' + name, type=Path, required=True)\n    for name in [\'source-sha256\', \'alpha-sha256\', \'selection-sha256\', \'review-note\']:\n        p.add_argument(\'--\' + name, required=True)\n    p.add_argument(\'--background-rgb\', type=int, nargs=3, required=True)\n    p.add_argument(\'--reviewed-neutral-white-floor\', action=\'store_true\', required=True)\n    a = p.parse_args(argv)\n    result = run_files(a.source, a.alpha, a.selection, a.output,\n        source_sha256=a.source_sha256, alpha_sha256=a.alpha_sha256,\n        selection_sha256=a.selection_sha256, background_rgb=a.background_rgb,\n        reviewed=a.reviewed_neutral_white_floor, review_note=a.review_note)\n    print(json.dumps(result, indent=2))\n    return result\n\n\nif __name__ == \'__main__\':\n    main()\n'
# ---- embedded module: renderello/reviewed_source_floor_cli.py ----
path = base / 'renderello/reviewed_source_floor_cli.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Reviewed finite surface-shadow supplement. Designed2D light cue, not relighting.\nThis never estimates geometry, camera or source illumination. An explicit source-\nbound review is required. It only changes the background below the product.\n"""\nfrom dataclasses import dataclass,asdict\nimport math\nimport numpy as np\nfrom .candidate_diagnostics import native_rgb,native_alpha,digest\nfrom .candidate_harmonization import linear_over\nfrom .checked_research.contact_camera import linear,srgb\n\n\n@dataclass(frozen=True)\nclass SurfaceShadow:\n    center_x:int\n    floor_row:int\n    sigma_x:float\n    sigma_y:float\n    opacity:float\n    source_sha256:str\n    alpha_sha256:str\n    background_sha256:str\n    contact_sha256:str\n    review_note:str\n    reviewed:bool=False\n    surface_kind:str=\'continuous_neutral_floor\'\n\n    def __post_init__(self):\n        if self.reviewed is not True or self.surface_kind!=\'continuous_neutral_floor\':\n            raise ValueError(\'reviewed continuous neutral floor required; no graphic wall/plinth assumption\')\n        if any(type(v) is not int or v<0 for v in [self.center_x,self.floor_row]):raise ValueError(\'native integer anchor required\')\n        for key,low,high in [(\'sigma_x\',2,256),(\'sigma_y\',1,32),(\'opacity\',.01,.35)]:\n            v=getattr(self,key)\n            if isinstance(v,bool) or not isinstance(v,(int,float)) or not math.isfinite(v) or not low<=v<=high:\n                raise ValueError(\'bounded finite \'+key+\' required\')\n        for key in [\'source_sha256\',\'alpha_sha256\',\'background_sha256\',\'contact_sha256\']:\n            v=getattr(self,key)\n            if not isinstance(v,str) or len(v)!=64 or any(c not in \'0123456789abcdef\' for c in v):raise ValueError(\'pixel hashes required\')\n        if not isinstance(self.review_note,str) or not self.review_note.strip() or len(self.review_note)>2000:raise ValueError(\'source light and surface review note required\')\n\n\ndef supplement_surface_shadow(rgb,alpha,background,contact,policy,*,fractional_underlay=False):\n    if type(fractional_underlay) is not bool:raise ValueError(\'explicit boolean fractional underlay mode required\')\n    native_rgb(rgb);native_alpha(alpha,rgb.shape[:2]);native_rgb(background)\n    if background.shape!=rgb.shape:raise ValueError(\'same native canvas required\')\n    if not isinstance(policy,SurfaceShadow):raise ValueError(\'typed source-bound surface review required\')\n    if not isinstance(contact,np.ndarray) or contact.dtype.kind!=\'f\' or contact.shape!=alpha.shape or not np.isfinite(contact).all() or np.any((contact<0)|(contact>.5)):\n        raise ValueError(\'finite floating contact shade0..0.5 required\')\n    for a,key in [(rgb,\'source_sha256\'),(alpha,\'alpha_sha256\'),(background,\'background_sha256\'),(contact,\'contact_sha256\')]:\n        if digest(a)!=getattr(policy,key):raise ValueError(\'review belongs to different \'+key)\n    h,w=alpha.shape\n    if policy.center_x>=w or policy.floor_row>=h-2:raise ValueError(\'floor anchor outside canvas\')\n    hard=alpha>=128\n    if not hard.any() or hard.all() or np.any(contact[hard]):raise ValueError(\'nontrivial hard support with shadow excluded required\')\n    rows=np.flatnonzero(hard[:,policy.center_x])\n    if not len(rows) or abs(int(rows[-1])-policy.floor_row)>2:\n        raise ValueError(\'floor center must bind to the actual native base within2pixels\')\n    if contact[int(rows[-1])+1,policy.center_x]<=0:raise ValueError(\'existing contact missing at reviewed base\')\n    # Symmetric floor footprint avoids a hard horizontal onset at the floor row.\n    # Hard foreground excludes the shadow; background beside/behind the base can shade.\n    # Finite3sigma support is an illustration bound, not inferred surface geometry.\n    y,x=np.indices(alpha.shape);xx=(x-policy.center_x)/policy.sigma_x;yy=(y-policy.floor_row-1)/policy.sigma_y\n    broad=policy.opacity*np.exp(-.5*(xx*xx+yy*yy))\n    # The opt-in mode shades the background under fractional coverage, rather\n    # than treating alpha128..254 as fully opaque. Original default retained.\n    exclusion=(alpha==255) if fractional_underlay else hard\n    broad[(abs(xx)>3)|(abs(yy)>3)|exclusion]=0\n    shade=1-(1-contact)*(1-broad);shade[exclusion]=0\n    result=linear_over(rgb,alpha,srgb(linear(background)*(1-shade[...,None])))\n    if not np.array_equal(result[alpha==255],rgb[alpha==255]):raise RuntimeError(\'opaque core changed\')\n    return result,shade,dict(schema=\'reviewed-surface-shadow.v1\',policy=asdict(policy),\n        output_sha256=digest(result),shadow_sha256=digest(shade),\n        broad_pixels=int((broad>0).sum()),fractional_underlay=fractional_underlay,\n        shadow_exclusion=\'alpha255_only\' if fractional_underlay else \'alpha128_and_above_legacy\',\n        fractional_underlay_pixels=int(((shade>0)&(alpha>0)&(alpha<255)).sum()),\n        source_rgb_repainted=False,source_resampled=False,\n        source_lighting_unchanged=True,physical_relighting=False,recovered_3d_geometry=False,\n        opaque_input_rgb_exact=True,accepted=False,independent_review=False,\n        limitation=\'Manually designed finite floor attenuation. Review cannot establish actual source lighting, geometry or photographic acceptance.\')\n'
# ---- embedded module: renderello/reviewed_surface_shadow.py ----
path = base / 'renderello/reviewed_surface_shadow.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Verified saved native matting to reviewed 2D contact comparison.\nNo new inference, pixel scaling, pose change, scene-light inference or acceptance.\n"""\nfrom dataclasses import asdict, replace\nimport numpy as np\nfrom .candidate_diagnostics import digest, native_rgb\nfrom .native_result_import import load_verified_result\nfrom .contact_spans import ContactPolicy, ContactSpan, render_contact\nfrom .candidate_harmonization import linear_over\n\n\ndef compare_saved_result_contact(directory, source, initial_alpha, background, spans, policy,\n        *, support_floor_row=None, support_tolerance=2, translation=(0,0)):\n    if not isinstance(policy, ContactPolicy):\n        raise ValueError(\'typed source-bound contact policy required\')\n    expected_source_pose = \'overhead\' if policy.camera == \'flatlay\' else policy.camera\n    if expected_source_pose != policy.source_pose:\n        raise ValueError(\'this comparison requires compatible declared source and target camera\')\n    flatlay = policy.camera == \'flatlay\'\n    if flatlay and (support_floor_row is not None or spans):\n        raise ValueError(\'flatlay uses the full silhouette; no upright spans or floor row\')\n    alpha, replay = load_verified_result(directory, source, initial_alpha)\n    if digest(source) != policy.source_sha256 or digest(alpha) != policy.alpha_sha256:\n        raise ValueError(\'scene declarations do not bind to this verified native result\')\n    native_rgb(background)\n    if not isinstance(translation,(tuple,list)) or len(translation)!=2 or any(type(v) is not int or v<0 for v in translation):\n        raise ValueError(\'nonnegative integer source-to-canvas translation required\')\n    dx,dy=translation\n    h,w=source.shape[:2];ch,cw=background.shape[:2]\n    if dx+w>cw or dy+h>ch:\n        raise ValueError(\'entire native source must fit canvas; no crop or resize\')\n    original_policy=asdict(policy)\n    if not flatlay and (type(support_floor_row) is not int or not 0 <= support_floor_row < alpha.shape[0]-1):\n        raise ValueError(\'explicit native support floor row required\')\n    if source.shape != background.shape or dx or dy:\n        translated=np.zeros(background.shape,np.uint8);translated_alpha=np.zeros((ch,cw),np.uint8)\n        translated[dy:dy+h,dx:dx+w]=source;translated_alpha[dy:dy+h,dx:dx+w]=alpha\n        source,alpha=translated,translated_alpha\n        if any(not isinstance(s,ContactSpan) for s in spans):raise ValueError(\'typed native source spans required\')\n        spans=[ContactSpan(s.name,s.x0+dx,s.x1+dx,s.bottom_min+dy,s.bottom_max+dy) for s in spans]\n        policy=replace(policy,source_sha256=digest(source),alpha_sha256=digest(alpha))\n    native_support_floor_row=support_floor_row\n    if not flatlay:\n        support_floor_row+=dy\n    final, shade, contact = render_contact(source, alpha, background, spans, policy)\n    no_contact = linear_over(source, alpha, background)\n    hard = alpha >= 128\n    lower = np.where(hard.any(0), alpha.shape[0]-1-np.argmax(hard[::-1],0), -1)\n    if not flatlay and (type(support_floor_row) is not int or not 0 <= support_floor_row < alpha.shape[0]-1):\n        raise ValueError(\'explicit native support floor row required\')\n    if type(support_tolerance) is not int or not 0 <= support_tolerance <= 4:\n        raise ValueError(\'support tolerance must be integer 0..4 pixels\')\n    attachments = []\n    for span in spans:\n        cols = np.arange(span.x0,span.x1)\n        if np.any(np.abs(lower[cols]-support_floor_row) > support_tolerance):\n            raise ValueError(\'reviewed support contradicts declared floor row; no shadow under elevated contour\')\n        attached = shade[lower[cols]+1,cols] > 0\n        attachments.append(dict(name=span.name, columns=int(len(cols)),\n            first_row_shaded_columns=int(attached.sum()), all_attached=bool(attached.all())))\n    if attachments and not all(a[\'all_attached\'] for a in attachments):\n        raise ValueError(\'designed contact has a detached first row\')\n    if np.any(shade[hard]) or not np.array_equal(final[alpha==255],source[alpha==255]):\n        raise RuntimeError(\'hard-support shadow exclusion/source opaque invariant failed\')\n    record=dict(schema=\'saved-native-contact-comparison.v1\', import_replay=replay,\n        contact=contact, policy=asdict(policy), attachments=attachments,\n        support_floor_row=support_floor_row, support_tolerance=support_tolerance,\n        source_policy=original_policy, native_support_floor_row=native_support_floor_row,\n        source_to_canvas_translation=[dx,dy], original_source_size=[w,h],\n        source_cropped=False,\n        support_plane=\'REVIEWED_FLATLAY_SILHOUETTE\' if flatlay else \'EXPLICIT_REVIEW_NOT_RECOVERED_GEOMETRY\',\n        source_target_camera_compatible=True,\n        no_contact_sha256=digest(no_contact), result_sha256=digest(final),\n        source_rgb_repainted=False, alpha_repainted=False, source_resampled=False,\n        opaque_source_pixels_exact=True, shadow_on_hard_support_pixels=int(np.count_nonzero(shade[hard])),\n        assistant_visual_review_required=True, accepted=False, independent_review=False,\n        limitations=[\'Material, completeness, camera and contact spans are reviewed declarations, not inferred facts.\',\n          \'Same source lighting/reflections and alpha errors survive.\',\n          \'Pixel attachment does not establish plausible 3D support, scene light or publication quality.\',\n          \'Saved model execution provenance remains a manifest claim; numeric replay proves consistency only.\'])\n    return no_contact, final, shade, record\n'
# ---- embedded module: renderello/saved_native_contact.py ----
path = base / 'renderello/saved_native_contact.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Acceptance screen for generated background scenes, before compositing.\n\nCatches frames where the generator draws set equipment or frame-like straight\nedges intruding into the scene (observed: a black softbox frame crossing the\nright side of a studio sweep). The scene handed here is the EMPTY generated\nbackground (no product yet), so long straight high-contrast lines are almost\nalways equipment or frame artifacts, not scene content; the wall/floor sweep\ncurve is not straight and does not trigger the detector.\n\nThe screen is a heuristic gate for regeneration or fallback, never a quality\ncertificate: an accepted scene can still be wrong in other ways, and the\ncomposite still goes through the same-coordinate QA and eye review.\n"""\nfrom __future__ import annotations\n\nimport cv2\nimport numpy as np\n\n# A straight line segment at least this fraction of the short canvas side is\n# treated as an equipment/frame intrusion.\nMIN_LINE_FRAC = 0.30\n# Canny thresholds tuned for soft studio gradients: below these, sweep shading\n# does not edge-detect; hard equipment edges do.\nCANNY_LO = 50\nCANNY_HI = 150\n\n\ndef _luma(rgb: np.ndarray) -> np.ndarray:\n    f = rgb.astype(np.float32)\n    return 0.299 * f[..., 0] + 0.587 * f[..., 1] + 0.114 * f[..., 2]\n\n\ndef long_straight_edges(scene_rgb: np.ndarray) -> dict:\n    """Detect long straight high-contrast edges (equipment frames, set walls\'\n    hard corners) in a generated empty background. Returns diagnostics with\n    \'intrusion\' True when any straight line exceeds MIN_LINE_FRAC of the short\n    side."""\n    h, w = scene_rgb.shape[:2]\n    short = min(h, w)\n    lu = _luma(scene_rgb).astype(np.uint8)\n    edges = cv2.Canny(lu, CANNY_LO, CANNY_HI)\n    lines = cv2.HoughLinesP(\n        edges, 1, np.pi / 180,\n        threshold=max(40, int(0.12 * short)),\n        minLineLength=max(60, int(MIN_LINE_FRAC * short)),\n        maxLineGap=max(8, int(0.012 * short)),\n    )\n    found = []\n    if lines is not None:\n        for l in lines.reshape(-1, 4):\n            x1, y1, x2, y2 = (int(v) for v in l)\n            length = round(float(((x2 - x1) ** 2 + (y2 - y1) ** 2) ** 0.5), 1)\n            found.append({"p1": [x1, y1], "p2": [x2, y2], "length": length})\n    diag = {\n        "lines": found,\n        "min_line_px": int(MIN_LINE_FRAC * short),\n        "intrusion": bool(found),\n    }\n    return diag\n'
# ---- embedded module: renderello/scene_acceptance.py ----
path = base / 'renderello/scene_acceptance.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Pre-paste emptiness screen for generated background scenes.\n\nFlags a scene that appears to contain an object where the product will be\npasted - the 20 Nov 2026 duplicate-product failure mode. This is a SCREEN:\nevery verdict feeds the caller\'s scene-regeneration budget and sets\nneeds_eye_review=True unconditionally. It never certifies a scene clean and\nnever substitutes for human eye review.\n\nHeuristics (all conservative, flag-only):\n- object-like contrast blob inside the future placement rectangle\n  (an empty guide scene is low-contrast there);\n- long straight edges inside the placement rectangle (scene_acceptance\n  covers the whole frame; this focuses where a duplicate would stand).\n"""\nimport numpy as np\n\n\ndef _placement_rect(placement):\n    px, py, pw, ph = placement\n    return int(px), int(py), int(pw), int(ph)\n\n\ndef screen_scene_emptiness(scene_rgb, placement) -> dict:\n    """scene_rgb: HxWx3 uint8 generated scene BEFORE the product paste.\n    placement: (px, py, pw, ph) where the product cutout will land.\n    Returns a flags dict; needs_eye_review is always True."""\n    import cv2\n    if not isinstance(scene_rgb, np.ndarray) or scene_rgb.ndim != 3 \\\n            or scene_rgb.shape[2] != 3 or scene_rgb.dtype != np.uint8:\n        raise ValueError("scene must be a nonempty uint8 RGB array")\n    h, w = scene_rgb.shape[:2]\n    px, py, pw, ph = _placement_rect(placement)\n    if min(pw, ph) < 1 or px < 0 or py < 0 or px + pw > w or py + ph > h:\n        raise ValueError("placement outside the scene")\n    roi = scene_rgb[py:py + ph, px:px + pw]\n    gray = cv2.cvtColor(roi, cv2.COLOR_RGB2GRAY).astype(np.float32)\n    full = cv2.cvtColor(scene_rgb, cv2.COLOR_RGB2GRAY).astype(np.float32)\n\n    # Contrast blob: an empty two-tone guide renders smoothly; a rendered\n    # object raises local contrast well above the scene\'s median texture.\n    roi_std = float(gray.std())\n    scene_std = float(full.std())\n    blob_suspect = roi_std > max(18.0, 1.6 * scene_std)\n\n    # Straight-edge intrusion restricted to the placement area.\n    edges = cv2.Canny(roi.astype(np.uint8), 60, 160)\n    lines = cv2.HoughLinesP(edges, 1, np.pi / 180,\n                            threshold=max(20, min(pw, ph) // 4),\n                            minLineLength=max(16, min(pw, ph) // 3),\n                            maxLineGap=4)\n    long_lines = 0 if lines is None else int(len(lines))\n    edge_suspect = long_lines >= 3\n\n    suspect_object = bool(blob_suspect or edge_suspect)\n    return {\n        "scope": "PLACEMENT_RECTANGLE_ONLY",\n        "suspect_object_in_scene": suspect_object,\n        "contrast_blob_suspect": blob_suspect,\n        "roi_luma_std": round(roi_std, 2),\n        "scene_luma_std": round(scene_std, 2),\n        "placement_long_straight_edges": long_lines,\n        "straight_edge_suspect": edge_suspect,\n        "screen_only": True,\n        "needs_eye_review": True,\n    }\n'
# ---- embedded module: renderello/scene_emptiness.py ----
path = base / 'renderello/scene_emptiness.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Declared scene-only exposure/color matching, not physical relighting.\n\nTransforms only background pixels before native product composition. No light,\nmaterial or camera estimation is claimed. Every setting is explicit and saved.\n"""\nimport numpy as np\n\n\ndef finish_background(rgb, *, exposure_ev=0., white_balance=(1.,1.,1.)):\n    if not isinstance(rgb,np.ndarray) or rgb.dtype!=np.uint8 or rgb.ndim!=3 or rgb.shape[2]!=3:\n        raise ValueError(\'background must be RGB uint8\')\n    if not np.isfinite(exposure_ev) or abs(exposure_ev)>.5:\n        raise ValueError(\'scene exposure must be finite and within half a stop\')\n    wb=np.asarray(white_balance,dtype=np.float64)\n    if wb.shape!=(3,) or not np.isfinite(wb).all() or (wb<.85).any() or (wb>1.15).any():\n        raise ValueError(\'declared white balance gains must be finite RGB0.85to1.15\')\n    if exposure_ev==0 and (wb==1).all():return rgb.copy()\n    srgb=rgb.astype(np.float64)/255\n    linear=np.where(srgb<=.04045,srgb/12.92,((srgb+.055)/1.055)**2.4)\n    linear=np.clip(linear*(2**exposure_ev)*wb,0,1)\n    result=np.where(linear<=.0031308,12.92*linear,1.055*linear**(1/2.4)-.055)\n    return np.rint(np.clip(result,0,1)*255).astype(np.uint8)\n\n\ndef validate_support_plane(mask, alpha, placement):\n    """Require all lower opaque contour samples on an explicitly reviewed plane.\n\n    The mask is geometry supplied after inspecting the actual background,\n    never inferred from its filename, generated prompt or product pixel audit.\n    """\n    if not isinstance(mask,np.ndarray) or mask.dtype!=np.uint8 or mask.ndim!=2 or min(mask.shape)<1:\n        raise ValueError(\'support plane must be uint8 L\')\n    if not isinstance(alpha,np.ndarray) or alpha.dtype!=np.uint8 or alpha.ndim!=2:\n        raise ValueError(\'product alpha must be uint8 L\')\n    x,y=placement;h,w=mask.shape;ph,pw=alpha.shape\n    if any(type(v) is not int for v in [x,y]) or x<0 or y<0 or x+pw>w or y+ph>h:\n        raise ValueError(\'native placement outside support canvas\')\n    yy,xx=np.where(alpha>127)\n    if not len(yy):raise ValueError(\'opaque support contour required\')\n    bottom=int(yy.max());band=max(2,round(ph*.025));samples=[]\n    for col in range(pw):\n        rows=np.flatnonzero(alpha[:,col]>127)\n        if len(rows) and rows[-1]>=bottom-band:samples.append((y+int(rows[-1])+1,x+col))\n    if not samples or any(row>=h or mask[row,col]!=255 for row,col in samples):\n        raise ValueError(\'product support misses the reviewed support plane; reframe background or placement\')\n    return len(samples)\n\n\ndef inspect_support_components(alpha):\n    """Refuse detached lower islands that silently set the contact baseline.\n\n    Component area is not product identity. This detects ambiguity; it never\n    deletes a watermark, removes a component or declares a semantic mask good.\n    Intentionally disconnected products need a separately reviewed support path.\n    """\n    import cv2\n    if not isinstance(alpha,np.ndarray) or alpha.dtype!=np.uint8 or alpha.ndim!=2:\n        raise ValueError(\'product alpha must be uint8 L\')\n    n,labels,stats,centers=cv2.connectedComponentsWithStats((alpha>127).astype(np.uint8),connectivity=8)\n    if n<2:raise ValueError(\'opaque support contour required\')\n    main=1+int(np.argmax(stats[1:,cv2.CC_STAT_AREA]))\n    main_bottom=int(stats[main,cv2.CC_STAT_TOP]+stats[main,cv2.CC_STAT_HEIGHT]-1)\n    ambiguous=[]\n    for idx in range(1,n):\n        if idx==main:continue\n        area=int(stats[idx,cv2.CC_STAT_AREA]);bottom=int(stats[idx,cv2.CC_STAT_TOP]+stats[idx,cv2.CC_STAT_HEIGHT]-1)\n        if bottom>main_bottom+3:\n            ambiguous.append(dict(area=area,bottom=bottom,box=[int(v) for v in stats[idx,:4]]))\n    report=dict(component_count=n-1,largest_component_area=int(stats[main,cv2.CC_STAT_AREA]),largest_component_bottom=main_bottom,lower_detached_components=ambiguous,mask_semantics=\'NOT_EVALUATED\')\n    if ambiguous:raise ValueError(\'detached lower mask component sets contact baseline; review source mask and intended support without silently deleting components\')\n    return report\n'
# ---- embedded module: renderello/scene_finish.py ----
path = base / 'renderello/scene_finish.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Explicit actual-scene native placement command. No model or fallback.\n\nCaller declarations are not owner authority, calibrated geometry or photo approval.\n"""\nfrom pathlib import Path\nfrom io import BytesIO\nimport argparse, hashlib, json\nimport numpy as np\nfrom PIL import Image\nfrom .reviewed_scene_placement import ScenePlacementReview, plan_reviewed_placement, compose_reviewed_placement\nfrom .contact_spans import ContactPolicy, ContactSpan\n\nNAMES=(\'source\',\'alpha\',\'scene\',\'surface\',\'exclusion\')\n\n\ndef _snapshot(path,limit):\n    with Path(path).open(\'rb\') as stream:\n        value=stream.read(limit+1)\n    if len(value)>limit:raise ValueError(\'bounded input snapshot required\')\n    return value\n\n\ndef _decode(data,mode,budget):\n    with Image.open(BytesIO(data)) as im:\n        if im.format!=\'PNG\' or im.mode!=mode or getattr(im,\'n_frames\',1)!=1:\n            raise ValueError(\'single-frame native \'+mode+\' PNG required; no conversion\')\n        if im.getexif().get(274,1)!=1:raise ValueError(\'normalize orientation before reviewing\')\n        if im.width*im.height>budget:raise ValueError(\'native allocation budget exceeded; no resize\')\n        im.load()\n        return np.array(im)\n\n\ndef run(source,alpha,scene,surface,exclusion,review,out):\n    out=Path(out)\n    if out.exists():raise FileExistsError(\'preserve earlier output; choose a new directory\')\n    blobs={n:_snapshot(p,32_000_000) for n,p in zip(NAMES,(source,alpha,scene,surface,exclusion))}\n    blobs[\'review\']=_snapshot(review,64_000)\n    spec=json.loads(blobs[\'review\'])\n    if not isinstance(spec,dict) or set(spec)!={\'schema\',\'files\',\'review\',\'contact_policy\',\'spans\',\'translation\'} or spec[\'schema\']!=\'renderello-actual-scene-placement-review.v1\':\n        raise ValueError(\'exact scene placement command review schema required\')\n    hashes={n:hashlib.sha256(blobs[n]).hexdigest() for n in NAMES}\n    if spec[\'files\']!=hashes:raise ValueError(\'encoded input snapshot changed\')\n    rgb=_decode(blobs[\'source\'],\'RGB\',1_000_000);a=_decode(blobs[\'alpha\'],\'L\',1_000_000)\n    bg=_decode(blobs[\'scene\'],\'RGB\',2_000_000)\n    plane=_decode(blobs[\'surface\'],\'L\',2_000_000);ex=_decode(blobs[\'exclusion\'],\'L\',2_000_000)\n    if any(not np.isin(m,[0,255]).all() for m in (plane,ex)):raise ValueError(\'binary0/255 scene masks required\')\n    if not isinstance(spec[\'spans\'],list) or len(spec[\'spans\'])>64:raise ValueError(\'bounded native span list required\')\n    r=ScenePlacementReview(**spec[\'review\']);p=ContactPolicy(**spec[\'contact_policy\']);spans=[ContactSpan(**s) for s in spec[\'spans\']]\n    plane=plane>0;ex=ex>0\n    plan=plan_reviewed_placement(rgb,a,bg,plane,ex,r,p,spans,translation=spec[\'translation\'])\n    # All parsing, binding and geometry validation precedes creating the result.\n    # Unexpected errors after this point leave the marker, not a success manifest.\n    out.mkdir(parents=True);(out/\'INCOMPLETE\').touch()\n    for n in NAMES:(out/(n+\'.png\')).write_bytes(blobs[n])\n    (out/\'review.json\').write_bytes(blobs[\'review\'])\n    manifest=dict(schema=\'renderello-actual-scene-placement-result.v1\',status=\'REFUSED\',\n        input_file_sha256={n:hashlib.sha256(b).hexdigest() for n,b in blobs.items()},\n        report=plan,photo_accepted=False,polished_templates=0,models_called=0,\n        review_is_caller_declaration_not_owner_or_quality_approval=True,outputs=[])\n    if not plan[\'blockers\']:\n        result,shadow,report=compose_reviewed_placement(rgb,a,bg,plane,ex,r,p,spans,translation=spec[\'translation\'])\n        Image.fromarray(result).save(out/\'designed-preview.png\')\n        # PNG is a rounded visualization only; retain actual float attenuation too.\n        Image.fromarray(np.rint(shadow*255).astype(np.uint8)).save(out/\'shadow-visualization.png\')\n        np.save(out/\'shadow.npy\',shadow,allow_pickle=False)\n        manifest.update(status=\'EXECUTED_REVIEW_REQUIRED\',report=report,outputs=[dict(path=n,sha256=hashlib.sha256((out/n).read_bytes()).hexdigest()) for n in (\'designed-preview.png\',\'shadow-visualization.png\',\'shadow.npy\')])\n    (out/\'manifest.json\').write_text(json.dumps(manifest,indent=2))\n    (out/\'INCOMPLETE\').unlink()\n    return manifest\n\n\ndef main(argv=None):\n    parser=argparse.ArgumentParser(description=__doc__)\n    for name in NAMES+(\'review\',\'out\'):parser.add_argument(\'--\'+name,type=Path,required=True)\n    args=parser.parse_args(argv)\n    result=run(**vars(args));print(json.dumps(result,indent=2));return 0\n\n\nif __name__==\'__main__\':raise SystemExit(main())\n'
# ---- embedded module: renderello/scene_placement_cli.py ----
path = base / 'renderello/scene_placement_cli.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Opt-in real background plate composition with declared support geometry.\n\nThe image generator makes empty scenery only. No automatic camera/lighting\ninference, no product relighting, no glass optics or photographic acceptance.\n"""\nfrom pathlib import Path\nfrom io import BytesIO\nimport hashlib,json\nimport cv2\nimport numpy as np\nfrom PIL import Image\nfrom .prepared_compose import load_prepared\nfrom .native_pixel_audit import audit_native_pixels\n\n\ndef support_shadow(alpha, canvas, placement, *, cast_offset=(0,0), opacity=.16):\n    """Two-scale contact cue plus optional designer-directed 2D cast projection.\n\n    cast_offset is the chosen floor displacement of the product\'s highest point.\n    It is not recovered light direction or physical geometry.\n    """\n    if alpha.dtype!=np.uint8 or alpha.ndim!=2 or min(alpha.shape)<1:\n        raise ValueError(\'alpha must be nonempty uint8 L\')\n    w,h=canvas;x,y=placement;ph,pw=alpha.shape\n    if any(type(v) is not int for v in [w,h,x,y]) or not (0<=x<=w-pw and 0<=y<=h-ph):\n        raise ValueError(\'native product must fit inside canvas\')\n    if not 0<=opacity<=.35 or len(cast_offset)!=2 or not np.isfinite(cast_offset).all():\n        raise ValueError(\'invalid declared shadow parameters\')\n    visible=alpha>127;yy,xx=np.where(visible)\n    if not len(yy):raise ValueError(\'opaque support contour required\')\n    bottom=int(yy.max());support=np.zeros((h,w),np.float32)\n    band=max(2,round(ph*.025))\n    for col in range(pw):\n        rows=np.flatnonzero(visible[:,col])\n        if len(rows) and rows[-1]>=bottom-band:\n            fy=y+int(rows[-1])+1\n            if fy<h:support[fy,x+col]=alpha[rows[-1],col]/255.\n    # Contact follows actual bottom contours, rather than a full-width ellipse.\n    tight=cv2.GaussianBlur(support,(0,0),sigmaX=1.5,sigmaY=1.2)\n    broad=cv2.GaussianBlur(support,(0,0),sigmaX=max(3,pw*.025),sigmaY=max(2,ph*.008))\n    for layer,peak in [(tight,.24),(broad,.12)]:\n        if layer.max()>0:layer*=peak/layer.max()\n    contact=1-(1-tight)*(1-broad)\n    dx,dy=cast_offset\n    if dx or dy:\n        # The lowest support row maps to itself. Higher silhouette pixels\n        # project onto the explicitly selected support-plane direction.\n        denom=max(1,bottom)\n        matrix=np.float32([[1,-dx/denom,x+dx*bottom/denom],\n                           [0,-dy/denom,y+bottom+dy*bottom/denom]])\n        cast=cv2.warpAffine(alpha.astype(np.float32)/255,matrix,(w,h),flags=cv2.INTER_LINEAR,borderMode=cv2.BORDER_CONSTANT)\n        cast=cv2.GaussianBlur(cast,(0,0),sigmaX=max(3,pw*.025),sigmaY=max(2,ph*.012))*opacity\n        contact=1-(1-contact)*(1-cast)\n    return np.clip(contact,0,.45)\n\n\ndef compose_plate(prepared,plate_path,out,*,baseline=None,center_x=512,cast_offset=(0,0),contact=True,\n                  support_plane_path=None,contact_strength=1.,scene_exposure_ev=0.,scene_white_balance=(1.,1.,1.),background_kind=\'unspecified_inspected_background\',shadow_model=\'legacy\',shadow_softness=1.,reviewed_contact_span=None,source_origin=None):\n    """Native-source placement onto an inspected plate. Refuse any resizing."""\n    if background_kind not in [\'photograph\',\'generated\',\'procedural\',\'unspecified_inspected_background\']:\n        raise ValueError(\'declare photograph/generated/procedural/unspecified background kind\')\n    if shadow_model not in [\'legacy\',\'height_soft_linear\']:raise ValueError(\'unknown shadow model\')\n    if not np.isfinite(shadow_softness) or not .25<=shadow_softness<=3:raise ValueError(\'softness must be finite0.25to3\')\n    out=Path(out)\n    if out.exists():raise ValueError(\'output exists; preserve older candidate\')\n    if source_origin is None:\n        if type(baseline) is not int or type(center_x) is not int:raise ValueError(\'integer placement required\')\n    else:\n        if baseline is not None or center_x!=512:raise ValueError(\'source origin cannot combine with baseline/centre placement\')\n        if not isinstance(source_origin,(list,tuple)) or len(source_origin)!=2 or any(type(v) is not int for v in source_origin):raise ValueError(\'source origin must be two integers\')\n    record,files=load_prepared(prepared)\n    source=np.array(Image.open(BytesIO(files[\'normalized-source.png\'])))\n    alpha=np.array(Image.open(BytesIO(files[\'reviewed-alpha.png\'])))\n    raw=Path(plate_path).read_bytes()\n    with Image.open(BytesIO(raw)) as im:\n        if im.mode!=\'RGB\' or im.size!=(1024,1024):raise ValueError(\'reviewed plate must be1024x1024RGB\')\n        im.load();scene=np.array(im)\n    from .scene_finish import inspect_support_components\n    support_components=inspect_support_components(alpha)\n    yy,xx=np.where(alpha>0)\n    if not len(yy):raise ValueError(\'empty alpha\')\n    sx,sy=int(xx.min()),int(yy.min());sw,sh=int(xx.max())+1-sx,int(yy.max())+1-sy\n    a=alpha[sy:sy+sh,sx:sx+sw];rgb=source[sy:sy+sh,sx:sx+sw]\n    support=np.where((a>127).any(axis=1))[0]\n    if not len(support):raise ValueError(\'opaque support contour required\')\n    if source_origin is None:\n        x=center_x-sw//2;y=baseline-int(support[-1])-1\n    else:\n        x=source_origin[0]+sx;y=source_origin[1]+sy\n        baseline=y+int(support[-1])+1\n    if x<0 or y<0 or x+sw>1024 or y+sh>1024:raise ValueError(\'native product does not fit declared placement\')\n    if not np.isfinite(contact_strength) or not 0<=contact_strength<=2:\n        raise ValueError("contact strength must be finite0to2")\n    from .scene_finish import finish_background,validate_support_plane\n    plane=None;plane_hash=None;support_samples=None\n    if support_plane_path is not None:\n        plane_bytes=Path(support_plane_path).read_bytes()\n        with Image.open(BytesIO(plane_bytes)) as im:\n            if im.mode!="L" or im.size!=(1024,1024):raise ValueError("support plane must be1024x1024L")\n            plane=np.array(im)\n        support_samples=validate_support_plane(plane,a,(x,y))\n        plane_hash=hashlib.sha256(plane_bytes).hexdigest()\n    scene=finish_background(scene,exposure_ev=scene_exposure_ev,white_balance=scene_white_balance)\n    from .scene_shadow import projected_soft_shadow,attenuate_linear,reviewed_span_contact\n    crop_span=None\n    if reviewed_contact_span is not None:\n        if not contact or shadow_model!=\'height_soft_linear\':raise ValueError(\'reviewed contact span requires enabled height-soft contact\')\n        if not isinstance(reviewed_contact_span,(list,tuple)) or len(reviewed_contact_span)!=2 or any(type(v) is not int for v in reviewed_contact_span):raise ValueError(\'reviewed contact span must be integer source columns\')\n        crop_span=(reviewed_contact_span[0]-sx,reviewed_contact_span[1]-sx)\n        reviewed_span_contact(a,(1024,1024),(x,y),crop_span)\n        if plane is not None:\n            for col in range(*crop_span):\n                row=int(np.flatnonzero(a[:,col]>127)[-1])+y+1\n                if plane[row,x+col]!=255:raise ValueError(\'reviewed contact span misses support plane\')\n    shadow_fn=support_shadow if shadow_model==\'legacy\' else projected_soft_shadow\n    extra={} if shadow_model==\'legacy\' else {\'softness\':shadow_softness}\n    if crop_span is not None:extra[\'contact_span\']=crop_span\n    shadow=(shadow_fn(a,(1024,1024),(x,y),cast_offset=cast_offset,**extra) if contact else np.zeros((1024,1024),np.float32))\n    shadow=np.clip(shadow*contact_strength,0,.65)\n    if plane is not None:shadow*=plane.astype(np.float32)/255\n    # Neutral multiplicative attenuation changes only the scene, never RGB\n    # inside the source product. The source\'s existing highlights stay as-is.\n    if shadow_model==\'legacy\':\n        shaded=scene.astype(np.float32)*(1-shadow[...,None])\n    else:\n        # Keep the reviewed near-contact cue while softening only the cast.\n        near=(support_shadow(a,(1024,1024),(x,y),cast_offset=(0,0)) if crop_span is None else reviewed_span_contact(a,(1024,1024),(x,y),crop_span)) if contact else np.zeros((1024,1024),np.float32)\n        near=np.clip(near*contact_strength,0,.65)\n        if plane is not None:near*=plane.astype(np.float32)/255\n        cast_only=np.clip((shadow-near)/np.maximum(1-near,1e-6),0,1)\n        shaded=attenuate_linear(scene,cast_only).astype(np.float32)*(1-near[...,None])\n    opacity=a[...,None].astype(np.float32)/255\n    region=shaded[y:y+sh,x:x+sw]\n    region[:]=region*(1-opacity)+rgb.astype(np.float32)*opacity\n    final=np.clip(shaded,0,255).astype(np.uint8)\n    audit=audit_native_pixels(source,alpha,final,(sx,sy,sw,sh),(x,y,sw,sh))\n    if audit[\'status\']!=\'MATCH\':raise ValueError(\'native product opaque RGB failed\')\n    out.mkdir(parents=True)\n    Image.fromarray(final).save(out/\'candidate.png\');Image.fromarray(np.rint(shadow*255).astype(np.uint8)).save(out/\'shadow-alpha.png\')\n    meta=dict(placement_mode=\'original_source_origin\' if source_origin is not None else \'alpha_bounds_center_baseline\',original_source_origin=[x-sx,y-sy],reviewed_contact_span_source_columns=reviewed_contact_span,contact_span_geometry=\'EXPLICIT_REVIEW_NOT_INFERRED\' if crop_span is not None else \'lowest_row_band\',support_component_review=support_components,shadow_model=shadow_model,shadow_softness=shadow_softness,shadow_light_geometry=\'DESIGNER_DECLARED_NOT_INFERRED\',shadow_alpha_meaning=\'geometric_strength_field_NOT_sRGB_transmittance\' if shadow_model!=\'legacy\' else \'sRGB_multiplicative_attenuation\',support_plane_sha256=plane_hash,support_plane_samples=support_samples,contact_strength=contact_strength,scene_exposure_ev=scene_exposure_ev,scene_white_balance=list(scene_white_balance),scene_finish=\'declared_scene_only_tone_matching_NOT_physical_relighting\',source_record=record,background_sha256=hashlib.sha256(raw).hexdigest(),source_box=[sx,sy,sw,sh],placement=[x,y],baseline=baseline,cast_offset=list(cast_offset),shadow_kind=\'declared_2D_support_and_optional_projection\',background_kind=background_kind,source_scale=1,product_relighting=False,glass_optics=False,native_pixel_audit=audit,photographic_acceptance=\'NOT_EVALUATED\',source_rights=\'NOT_UPGRADED\')\n    (out/\'scene-plate.json\').write_text(json.dumps(meta,indent=2,allow_nan=False))\n    return meta\n'
# ---- embedded module: renderello/scene_plate.py ----
path = base / 'renderello/scene_plate.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Generative scene providers - the pluggable backends that create the\nenvironment a verified product cutout is composited into.\n\nDesign contract (per the problem statement and owner rules):\n\n- A provider generates the SCENE ONLY. The product is composited afterward\n  by compose.composite_with_scene / compose.composite, so product pixels are\n  never at the mercy of a generative model on the deterministic routes.\n- Every provider declares its cost and its limits honestly:\n  procedural   = free, CPU, always available, NOT photoreal-AI quality\n  sdxl_lightning = free, needs a GPU (Colab/Kaggle), AI-quality empty scenes\n  hf_inference = free-tier hosted API, needs HF token, rate-limited\n  magichour    = paid credits, live only with the API key, RENDERELLO_MH_LIVE=1\n                 and an owner-authorized credit budget (server-side guards)\n- A provider that is not available says so; nothing silently falls back to a\n  weaker provider. Callers pick explicitly and the manifest records which\n  provider produced each output.\n"""\nfrom __future__ import annotations\n\nimport io\nimport os\nfrom dataclasses import dataclass\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image\n\nfrom . import longprompt\nfrom .compose import procedural_background\nfrom .schemas import TemplateSpec\n\n\n@dataclass\nclass ProviderInfo:\n    name: str\n    label: str\n    cost: str\n    requires: str\n    available: bool\n    unavailable_reason: str = ""\n    quality_note: str = ""\n\n\nclass SceneProvider:\n    name = "base"\n    label = "base"\n    cost = "unknown"\n    quality_note = ""\n\n    def available(self) -> tuple[bool, str]:\n        return False, "not implemented"\n\n    def info(self) -> ProviderInfo:\n        ok, why = self.available()\n        return ProviderInfo(self.name, self.label, self.cost, "",\n                            ok, "" if ok else why, self.quality_note)\n\n    def generate_scene(self, size: tuple[int, int], prompt: str, seed: int,\n                       template: TemplateSpec,\n                       negative: str = "") -> np.ndarray:\n        raise NotImplementedError\n\n\nclass ProceduralProvider(SceneProvider):\n    """Deterministic procedural backdrop (gradient + texture + bokeh).\n\n    Always available on CPU. Honest labeling: these are graphic backdrops,\n    not AI-photoreal scenes. The product itself remains pixel-exact.\n    """\n    name = "procedural"\n    label = "Procedural studio backdrop (CPU, free)"\n    cost = "free"\n    quality_note = ("Graphic backdrop, not an AI-photoreal scene; product "\n                    "pixels stay source-exact.")\n\n    def available(self):\n        return True, ""\n\n    def generate_scene(self, size, prompt, seed, template, negative=""):\n        return procedural_background(size, template, seed)\n\n\nclass SDXLLightningProvider(SceneProvider):\n    """SDXL-Lightning empty-scene generation on a local GPU.\n\n    Free, runs in the Colab notebook (interactive GPU use - the allowed\n    Colab usage). 4-step Lightning UNet on SDXL base; long prompts are fed\n    through longprompt.encode_long_prompt so no clause is dropped. The scene\n    is generated EMPTY (the prompt must not name the product); the verified\n    product cutout is composited on top afterward.\n    """\n    name = "sdxl_lightning"\n    label = "SDXL-Lightning AI scene (free, needs GPU)"\n    cost = "free (your Colab/Kaggle GPU)"\n    quality_note = ("AI-photoreal empty scene; fully opaque product pixels "\n                    "composited bit-exact on top (verified scope; edges and "\n                    "mask semantics excluded). Lighting/shadow harmony "\n                    "reviewed by QA + eye.")\n\n    def __init__(self, model_dir: str | None = None, device: str = "cuda"):\n        self.model_dir = model_dir\n        self.device = device\n        self._pipe = None\n\n    def available(self):\n        try:\n            import torch  # noqa\n            import diffusers  # noqa\n        except ImportError:\n            return False, "torch/diffusers not installed (install on the GPU host)"\n        if self.device == "cuda" and not torch.cuda.is_available():\n            return False, "no CUDA GPU visible"\n        return True, ""\n\n    def _load(self):\n        if self._pipe is not None:\n            return self._pipe\n        import torch\n        from diffusers import (StableDiffusionXLPipeline, EulerDiscreteScheduler,\n                               UNet2DConditionModel)\n        from huggingface_hub import hf_hub_download\n        from safetensors.torch import load_file\n        base = "stabilityai/stable-diffusion-xl-base-1.0"\n        base_revision = \'462165984030d82259a11f4367a4eed129e94a7b\'\n        lightning_revision = \'c9a24f48e1c025556787b0c58dd67a091ece2e44\'\n        checkpoint = hf_hub_download(\n            repo_id="ByteDance/SDXL-Lightning",\n            filename="sdxl_lightning_4step_unet.safetensors",\n            revision=lightning_revision)\n        unet = UNet2DConditionModel.from_config(base, subfolder="unet",\n                                                  revision=base_revision)\n        unet = unet.to(self.device, torch.float16)\n        weights = load_file(checkpoint, device=self.device)\n        unet.load_state_dict(weights)\n        del weights\n        pipe = StableDiffusionXLPipeline.from_pretrained(\n            base, unet=unet, torch_dtype=torch.float16, variant="fp16",\n            revision=base_revision)\n        pipe.scheduler = EulerDiscreteScheduler.from_config(\n            pipe.scheduler.config, timestep_spacing="trailing")\n        pipe.to(self.device)\n        self._pipe = pipe\n        return pipe\n\n    def generate_scene(self, size, prompt, seed, template, negative=""):\n        import torch\n        pipe = self._load()\n        w, h = size\n        # SDXL native multiples of 8; keep longest side <= 1024 for speed.\n        scale = 1024 / max(w, h)\n        gw, gh = max(256, int(w * scale) // 8 * 8), max(256, int(h * scale) // 8 * 8)\n        pe, ne, pooled, neg_pooled = longprompt.encode_long_prompt(\n            pipe, prompt, self.device, negative)\n        gen = torch.Generator(device=self.device).manual_seed(seed)\n        kwargs = dict(num_inference_steps=4, guidance_scale=0.0,\n                      width=gw, height=gh, generator=gen,\n                      prompt_embeds=pe, negative_prompt_embeds=ne)\n        if pooled is not None:\n            kwargs["pooled_prompt_embeds"] = pooled\n            kwargs["negative_pooled_prompt_embeds"] = neg_pooled\n        img = pipe(**kwargs).images[0]\n        return np.asarray(img.resize((w, h), Image.LANCZOS).convert("RGB"))\n\n\nclass HFInferenceProvider(SceneProvider):\n    """Hugging Face hosted inference (free tier, rate-limited).\n\n    Uses the account\'s token from the HF_TOKEN environment variable - never\n    stored in code, Drive or the repo. Scene-only text-to-image; the product\n    is composited afterward. Free tier is rate-limited: callers surface\n    HTTP 429 honestly instead of retrying in a loop.\n    """\n    name = "hf_inference"\n    label = "Hugging Face hosted model (free tier, rate-limited)"\n    cost = "free tier (rate-limited)"\n    quality_note = "AI-photoreal empty scene via hosted API; product composited on top."\n    DEFAULT_MODEL = "stabilityai/stable-diffusion-xl-base-1.0"\n\n    def __init__(self, model: str | None = None, token_env: str = "HF_TOKEN"):\n        self.model = model or os.environ.get("HF_MODEL", self.DEFAULT_MODEL)\n        self.token_env = token_env\n\n    def available(self):\n        if not os.environ.get(self.token_env):\n            return False, f"{self.token_env} not set"\n        try:\n            import requests  # noqa\n        except ImportError:\n            return False, "requests not installed"\n        return True, ""\n\n    def generate_scene(self, size, prompt, seed, template, negative=""):\n        import requests\n        token = os.environ[self.token_env]\n        url = f"https://router.huggingface.co/hf-inference/models/{self.model}"\n        resp = requests.post(\n            url, headers={"Authorization": f"Bearer {token}"},\n            json={"inputs": prompt, "parameters": {\n                "negative_prompt": negative, "seed": seed,\n                "width": min(size[0], 1024), "height": min(size[1], 1024),\n                "num_inference_steps": 28}},\n            timeout=180)\n        if resp.status_code == 429:\n            raise RuntimeError("HF free-tier rate limit reached (429); "\n                               "wait or run the Colab notebook GPU instead")\n        if resp.status_code == 503:\n            raise RuntimeError("HF model is loading (503); retry shortly")\n        resp.raise_for_status()\n        img = Image.open(io.BytesIO(resp.content)).convert("RGB")\n        return np.asarray(img.resize(size, Image.LANCZOS))\n\n\nclass MagicHourProvider(SceneProvider):\n    """Magic Hour AI - PAID credits, product-preserving scene-compose.\n\n    Live calls stay disabled unless the API key exists AND RENDERELLO_MH_LIVE=1\n    is set for the deployment. Every live run also passes the server-side\n    aggregate budget guard in studio.py (owner-authorized cap, per-batch\n    limit, optional live-balance floor); without a budget the run refuses\n    with zero bill.\n    """\n    name = "magichour"\n    label = "Magic Hour AI (paid credits, budget-capped)"\n    cost = "paid credits"\n    quality_note = ("Background-only generation with a preserved product "\n                    "cutout; eye review remains the acceptance gate.")\n\n    def available(self):\n        if not os.environ.get("MAGIC_HOUR_API_KEY"):\n            return False, "MAGIC_HOUR_API_KEY not set"\n        if os.environ.get("RENDERELLO_MH_LIVE") != "1":\n            return False, ("not enabled on this deployment "\n                           "(RENDERELLO_MH_LIVE unset)")\n        return True, ""\n\n    def generate_scene(self, size, prompt, seed, template, negative=""):\n        raise RuntimeError(\n            "Magic Hour runs through studio.run_studio(provider=\'magichour\') "\n            "- the scene-compose route drives scene generation internally")\n\n\nPROVIDERS: dict[str, type[SceneProvider]] = {\n    p.name: p for p in (ProceduralProvider, SDXLLightningProvider,\n                        HFInferenceProvider, MagicHourProvider)}\n\n\ndef get_provider(name: str, **kwargs) -> SceneProvider:\n    try:\n        cls = PROVIDERS[name]\n    except KeyError:\n        raise KeyError(f"unknown scene provider \'{name}\'. available: "\n                       f"{\', \'.join(sorted(PROVIDERS))}") from None\n    return cls(**kwargs) if name == "sdxl_lightning" else (\n        cls(**kwargs) if name == "hf_inference" else cls())\n\n\ndef provider_report() -> list[dict]:\n    """Availability of every provider, for UIs and manifests."""\n    out = []\n    for cls in PROVIDERS.values():\n        try:\n            info = cls().info()\n        except Exception as e:\n            info = ProviderInfo(cls.name, cls.label, cls.cost, "", False,\n                                f"{type(e).__name__}: {e}", "")\n        row = info.__dict__\n        if cls.name == "magichour":\n            try:\n                from .backends.magichour import MODEL_TABLE\n                row["models"] = {m: t["credits"] for m, t in MODEL_TABLE.items()}\n            except Exception:\n                pass\n        out.append(row)\n    return out\n'
# ---- embedded module: renderello/scene_providers.py ----
path = base / 'renderello/scene_providers.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Explicit soft-light shadow approximation. Not recovered scene geometry.\n\nA flat silhouette projection is sliced by normalized source height. More distant\nslices have wider penumbrae and weaker opacity. This remains a 2D approximation,\nnot a physically reconstructed product or measured light.\n"""\nimport cv2\nimport numpy as np\n\n\ndef projected_soft_shadow(alpha, canvas, placement, *, cast_offset=(0,0), opacity=.16, softness=1., contact_span=None):\n    from .scene_plate import support_shadow\n    if not np.isfinite(opacity) or not 0<=opacity<=.35:raise ValueError(\'invalid declared opacity\')\n    # Reuse established shape/placement/opacity checks before allocating layers.\n    contact=(support_shadow(alpha,canvas,placement,cast_offset=(0,0),opacity=opacity) if contact_span is None else reviewed_span_contact(alpha,canvas,placement,contact_span))\n    if not np.isfinite(softness) or not .25<=softness<=3:\n        raise ValueError(\'softness must be finite0.25to3\')\n    if len(cast_offset)!=2 or not np.isfinite(cast_offset).all():\n        raise ValueError(\'finite cast offset required\')\n    dx,dy=cast_offset\n    if abs(dx)>canvas[0] or abs(dy)>canvas[1]:\n        raise ValueError(\'cast offset exceeds canvas\')\n    if not dx and not dy:return contact\n    w,h=canvas;x,y=placement;ph,pw=alpha.shape\n    bottom=int(np.where(alpha>127)[0].max());denom=max(1,bottom)\n    height=np.clip((bottom-np.arange(ph,dtype=np.float32))/denom,0,1)\n    matrix=np.float32([[1,-dx/denom,x+dx],[0,-dy/denom,y+bottom+dy]])\n    cast=np.zeros((h,w),np.float32)\n    centers=np.linspace(0,1,9)\n    for level in centers:\n        # Linear partition of unity gives smooth height weighting without bands.\n        weight=np.maximum(0,1-np.abs(height-level)*8)\n        layer=alpha.astype(np.float32)/255*weight[:,None]\n        warped=cv2.warpAffine(layer,matrix,(w,h),flags=cv2.INTER_LINEAR,borderMode=cv2.BORDER_CONSTANT)\n        sigma=softness*(1.2+level*max(5,pw*.075))\n        layer=cv2.GaussianBlur(warped,(0,0),sigmaX=sigma,sigmaY=max(1,sigma*.6))\n        cast+=layer*opacity*(1-.45*level)\n    return np.clip(1-(1-contact)*(1-np.clip(cast,0,.35)),0,.45)\n\n\ndef attenuate_linear(rgb, shadow):\n    """Neutral linear-light transmittance, with exact no-shadow identity."""\n    if not isinstance(rgb,np.ndarray) or rgb.dtype!=np.uint8 or rgb.ndim!=3 or rgb.shape[2]!=3:\n        raise ValueError(\'uint8RGB required\')\n    if not isinstance(shadow,np.ndarray) or shadow.shape!=rgb.shape[:2] or not np.isfinite(shadow).all() or (shadow<0).any() or (shadow>1).any():\n        raise ValueError(\'finite aligned shadow0to1 required\')\n    s=rgb.astype(np.float64)/255\n    lin=np.where(s<=.04045,s/12.92,((s+.055)/1.055)**2.4)*(1-shadow[...,None])\n    out=np.where(lin<=.0031308,lin*12.92,1.055*lin**(1/2.4)-.055)\n    result=np.rint(np.clip(out,0,1)*255).astype(np.uint8)\n    result[shadow==0]=rgb[shadow==0]\n    return result\n\n\ndef reviewed_span_contact(alpha,canvas,placement,span):\n    """Explicit half-open native crop-column contact span, never inferred geometry.\n\n    Choose only after image inspection. Follows the lower opaque contour within\n    the declared span, including sloped bases. Does not alter alpha or RGB.\n    """\n    from .scene_plate import support_shadow\n    support_shadow(alpha,canvas,placement) # established alpha/canvas checks\n    if not isinstance(span,(list,tuple)) or len(span)!=2 or any(type(v) is not int for v in span):\n        raise ValueError(\'contact span must be two integer crop columns\')\n    left,right=span;ph,pw=alpha.shape;w,h=canvas;x,y=placement\n    if not 0<=left<right<=pw or right-left<2:raise ValueError(\'contact span outside native crop or too narrow\')\n    support=np.zeros((h,w),np.float32)\n    for col in range(left,right):\n        rows=np.flatnonzero(alpha[:,col]>127)\n        if not len(rows) or y+int(rows[-1])+1>=h:raise ValueError(\'declared contact column has no supported lower contour\')\n        support[y+int(rows[-1])+1,x+col]=alpha[rows[-1],col]/255.\n    tight=cv2.GaussianBlur(support,(0,0),sigmaX=1.5,sigmaY=1.2)\n    broad=cv2.GaussianBlur(support,(0,0),sigmaX=max(3,pw*.025),sigmaY=max(2,ph*.008))\n    for layer,peak in [(tight,.24),(broad,.12)]:\n        if layer.max()>0:layer*=peak/layer.max()\n    return np.clip(1-(1-tight)*(1-broad),0,.45)\n'
# ---- embedded module: renderello/scene_shadow.py ----
path = base / 'renderello/scene_shadow.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Shared data structures. Everything crossing module boundaries is one of these,\nso the benchmark, QA and gallery all speak the same record shapes."""\nfrom __future__ import annotations\n\nimport json\nimport math\nfrom dataclasses import dataclass, field, asdict\nfrom enum import Enum\nfrom pathlib import Path\n\n\nclass Route(str, Enum):\n    PURE_PROMPT = "pure_prompt"          # Route 1: whole-image generative edit\n    CUTOUT_COMPOSE = "cutout_compose"    # Route 2: deterministic cutout + composite\n    HYBRID = "hybrid"                    # Route 3: composite + constrained harmonize\n    SCENE_COMPOSE = "scene_compose"      # Route 2b: relit/generated scene + source-exact product paste (needs scene_fn, GPU in practice)\n\n\nclass Aspect(str, Enum):\n    SQUARE = "1:1"\n    PORTRAIT = "4:5"\n    LANDSCAPE = "3:2"\n\n    @property\n    def size(self) -> tuple[int, int]:\n        # Working canvases are multiples of 32 with the EXACT labeled ratio.\n        # Keep the longest side <=1120; model adapters handle their own padding.\n        return {Aspect.SQUARE: (1024, 1024),\n                Aspect.PORTRAIT: (896, 1120),\n                Aspect.LANDSCAPE: (1056, 704)}[self]\n\n\n@dataclass(frozen=True)\nclass TemplateSpec:\n    """One reusable shoot template. Prompt fragments stay composable so custom\n    instructions append without rewriting the fidelity guards."""\n    key: str\n    name: str\n    category: str\n    scene: str            # environment / surface / props\n    lighting: str         # light direction, quality, color temperature\n    camera: str           # lens / angle / framing language\n    mood: str\n    aspect: Aspect\n    background_hint: str  # guides procedural background + cutout-route grading\n    shadow: str           # shadow behavior the composite must honor\n\n\n@dataclass(frozen=True)\nclass EditRequest:\n    """One generation: source + template + optional owner instruction + seed."""\n    source_path: Path\n    template: TemplateSpec\n    custom_instruction: str = ""\n    seed: int = 0\n    aspect: Aspect | None = None        # override; default = template aspect\n    reference_path: Path | None = None  # optional inspiration/backdrop photo (PDF section 4, optional)\n    mask_path: Path | None = None       # optional explicit alpha, source-sized L-mode PNG\n\n    @property\n    def effective_aspect(self) -> Aspect:\n        return self.aspect or self.template.aspect\n\n\n@dataclass\nclass CutoutResult:\n    rgba_path: Path                     # product with alpha matte\n    matte_path: Path                    # 8-bit alpha matte (255 = product)\n    bbox: tuple[int, int, int, int]     # x, y, w, h of the product in source coords\n    backend: str                        # birefnet / rmbg2 / grabcut\n    coverage: float                     # matte area fraction, sanity signal\n\n\n@dataclass\nclass QAReport:\n    """Fidelity of one output vs its source. Product-region metrics are computed\n    only inside the (aligned) product matte; \'pixel-exactness\' claims are valid\n    only for untransformed interior pixels (Route 2)."""\n    source: str\n    output: str\n    route: str\n    template: str\n    seed: int\n    ssim_product: float | None = None\n    l1_product: float | None = None\n    color_drift: float | None = None    # Bhattacharyya distance, product region\n    edge_retained: float | None = None  # fraction of source edges present in output\n    critical_errors: list[str] = field(default_factory=list)\n    score_fidelity: int | None = None   # 1-5 rubric (plan Part A); None = unscored\n    notes: str = ""\n\n    def to_json(self) -> str:\n        # JSON has no NaN/Infinity. Missing metric evidence is null, not zero\n        # or an invented pass, and the original report remains unchanged.\n        data = asdict(self)\n        for field in (\'ssim_product\', \'l1_product\', \'color_drift\', \'edge_retained\'):\n            value = data[field]\n            if value is not None and not math.isfinite(value):\n                data[field] = None\n        return json.dumps(data, indent=2, allow_nan=False)\n\n\n@dataclass\nclass VariantResult:\n    """One produced alternative with everything needed to audit it."""\n    route: Route\n    seed: int\n    output_path: Path\n    qa: QAReport\n    intermediates: dict[str, Path] = field(default_factory=dict)\n    backend: str = ""\n    credits_estimated: int = 0          # Magic Hour routes only; 0 for local routes\n    seconds: float = 0.0                # per-variant wall time (latency metric)\n    success: bool = True                 # executor/QA completed, not photo acceptance\n    provider_evidence: dict | None = None # actual recorded per-attempt facts; unknown is not zero\n\n\n@dataclass\nclass RunResult:\n    request: EditRequest\n    variants: list[VariantResult]\n    run_dir: Path\n    manifest_path: Path\n\n    def best_by_qa(self) -> VariantResult | None:\n        scored = [v for v in self.variants if v.success and v.output_path.is_file() and v.qa.score_fidelity is not None]\n        return max(scored, key=lambda v: v.qa.score_fidelity) if scored else None\n'
# ---- embedded module: renderello/schemas.py ----
path = base / 'renderello/schemas.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Explicit CPU IS-Net trial. Not automatic backend selection or semantic acceptance."""\nfrom pathlib import Path\nimport hashlib,json,os,time\nimport numpy as np\nfrom PIL import Image\n\nMODEL=\'isnet-general-use\'\nMODEL_URL=\'https://github.com/danielgatis/rembg/releases/download/v0.0.0/isnet-general-use.onnx\'\nMODEL_MD5=\'fc16ebd8b0c10d971d3513d564d01e29\'\n\n\ndef cache_dir():\n    # Models stay inside Renderello. Do not inherit a global U2NET_HOME.\n    return Path(__file__).resolve().parent.parent/\'.semantic-models\'\n\n\ndef infer_file(source:Path,out:Path,allow_download=False):\n    if out.exists():raise ValueError(\'output exists; preserve earlier attempts\')\n    with Image.open(source) as im:\n        if im.width*im.height>4_000_000:raise ValueError(\'source exceeds4million pixels; no silent resize\')\n        rgb=im.convert(\'RGB\');pixels=np.array(rgb)\n    cache=cache_dir();model=cache/(MODEL+\'.onnx\')\n    if model.exists():\n        if hashlib.md5(model.read_bytes()).hexdigest()!=MODEL_MD5:raise ValueError(\'cached model checksum mismatch; no automatic overwrite\')\n    elif not allow_download:raise ValueError(\'model not cached; --allow-download is required for the official free179MB model\')\n    try:\n        from rembg import new_session\n    except ImportError as e:raise RuntimeError(\'install optional requirements-semantic.txt in an isolated Renderello environment\') from e\n    os.environ[\'U2NET_HOME\']=str(cache);os.environ.pop(\'MODEL_CHECKSUM_DISABLED\',None)\n    started=time.monotonic();session=new_session(MODEL,providers=[\'CPUExecutionProvider\'])\n    if hashlib.md5(model.read_bytes()).hexdigest()!=MODEL_MD5:raise ValueError(\'downloaded model checksum mismatch\')\n    matte=np.array(session.predict(rgb)[0])\n    if matte.dtype!=np.uint8 or matte.shape!=pixels.shape[:2] or not np.any(matte):raise ValueError(\'invalid or empty inferred alpha\')\n    out.mkdir(parents=True)\n    for name,data in [(\'source.png\',pixels),(\'estimated-alpha.png\',matte),(\'cutout.png\',np.dstack([pixels,matte]))]:Image.fromarray(data).save(out/name)\n    for label,color in [(\'black\',(20,20,20)),(\'white\',(250,250,250)),(\'magenta\',(190,0,150))]:\n        bg=Image.new(\'RGB\',rgb.size,color);bg.paste(rgb,mask=Image.fromarray(matte));bg.save(out/(label+\'-review.png\'))\n    import importlib.metadata\n    report=dict(model=MODEL,model_url=MODEL_URL,model_md5=MODEL_MD5,model_sha256=hashlib.sha256(model.read_bytes()).hexdigest(),\n                package=\'rembg\',package_version=importlib.metadata.version(\'rembg\'),provider=\'CPUExecutionProvider\',source_sha256=hashlib.sha256(source.read_bytes()).hexdigest(),\n                source_rgb_unchanged=True,semantic_completeness=\'NOT_VERIFIED\',source_framing=\'NOT_VERIFIED\',quality=\'NOT_EVALUATED\',seconds=time.monotonic()-started,\n                warnings=[\'Inspect every part and hole. This mask failed the yellow notebook test.\',\'A cropped source cannot become complete. Source RGB may still contain reflected background colors.\',\'Optional model weights are the rembg-maintained ONNX conversion, not a claim of original author conversion.\'],\n                sources=[\'https://github.com/danielgatis/rembg\',\'https://github.com/xuebinqin/DIS\'])\n    report[\'output_sha256\']={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in out.iterdir() if p.is_file()}\n    (out/\'semantic-mask-manifest.json\').write_text(json.dumps(report,indent=2));return report\n\n\ndef guarded_file(source,out,allow_download=False):\n    """Per-image subprocess;180s wall-time. RSS check Linux-only,2.5GiB."""\n    import subprocess,sys,tempfile\n    cmd=[sys.executable,\'-m\',\'renderello.semantic_mask\',\'--source\',str(source),\'--out\',str(out)]\n    if allow_download:cmd+=[\'--allow-download\']\n    env=os.environ.copy();env[\'OMP_NUM_THREADS\']=\'2\'\n    with tempfile.TemporaryFile(mode=\'w+\') as stdout,tempfile.TemporaryFile(mode=\'w+\') as stderr:\n        p=subprocess.Popen(cmd,stdout=stdout,stderr=stderr,text=True,env=env);start=time.monotonic()\n        try:\n            while p.poll() is None:\n                if time.monotonic()-start>180:raise RuntimeError(\'semantic-mask180second timeout; attempt not accepted\')\n                try:rss=int(next(l.split()[1] for l in Path(f\'/proc/{p.pid}/status\').read_text().splitlines() if l.startswith(\'VmRSS:\')))\n                except (FileNotFoundError,StopIteration):rss=0\n                if rss>2560*1024:raise RuntimeError(\'semantic-mask2.5GiB RSS limit; attempt not accepted\')\n                time.sleep(.08)\n        finally:\n            if p.poll() is None:\n                p.terminate()\n                try:p.wait(timeout=5)\n                except subprocess.TimeoutExpired:p.kill();p.wait()\n        stdout.seek(0);stderr.seek(0)\n        if p.returncode:raise RuntimeError(\'semantic-mask worker failed: \'+stderr.read()[-2000:])\n        return json.load(stdout)\n\n\ndef main():\n    import argparse\n    p=argparse.ArgumentParser(description=__doc__);p.add_argument(\'--source\',type=Path,required=True);p.add_argument(\'--out\',type=Path,required=True);p.add_argument(\'--allow-download\',action=\'store_true\');a=p.parse_args();print(json.dumps(infer_file(a.source,a.out,a.allow_download)))\n\nif __name__==\'__main__\':main()\n'
# ---- embedded module: renderello/semantic_mask.py ----
path = base / 'renderello/semantic_mask.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Explicit geometry-prior rasterization, not a shadow estimator.\n\nA caller supplies a model\'s regression values and foreground rotated box.\nThis module does not load a model, infer light, or modify a product image.\nIt avoids writing into a temporary ndarray dtype-conversion copy.\n"""\nimport math\nimport numpy as np\nfrom PIL import Image,ImageDraw\n\n\ndef rasterize_shadow_prior(regression, foreground, size=(512,512)):\n    """Return a real uint8 box mask and diagnostics, from GPS-style offsets.\n\n    foreground=(center_x,center_y,width,height,angle_degrees).\n    regression=(normalized_dx,normalized_dy,log_width,log_height,angle_radians).\n    Mask samples are 0 or 1, as expected by the published control input.\n    """\n    if not isinstance(size,(tuple,list)) or len(size)!=2 or any(type(n)!=int or not 1<=n<=4096 for n in size):raise ValueError(\'size must be two integer dimensions 1-4096\')\n    try:r=np.asarray(regression,dtype=float);f=np.asarray(foreground,dtype=float)\n    except (TypeError,ValueError):raise ValueError(\'regression and foreground must be five finite numbers\')\n    if r.shape!=(5,) or f.shape!=(5,) or not np.isfinite(r).all() or not np.isfinite(f).all():raise ValueError(\'regression and foreground must be five finite numbers\')\n    if np.any(np.abs(r[:2])>16) or np.any(np.abs(r[2:4])>8) or abs(r[4])>math.pi*8:raise ValueError(\'regression outside bounded geometry range\')\n    w,h=size\n    if f[2]<=0 or f[3]<=0 or f[2]>w*4 or f[3]>h*4:raise ValueError(\'foreground dimensions must be positive and bounded\')\n    cx=r[0]*f[2]+f[0];cy=r[1]*f[3]+f[1];bw=f[2]*math.exp(r[2]);bh=f[3]*math.exp(r[3]);angle=math.degrees(r[4])+f[4]\n    if angle>0:bw,bh=bh,bw;angle-=90\n    a=math.radians(angle);rotation=np.array([[math.cos(a),-math.sin(a)],[math.sin(a),math.cos(a)]])\n    points=np.array([[-bw/2,-bh/2],[bw/2,-bh/2],[bw/2,bh/2],[-bw/2,bh/2]])@rotation.T+np.array([cx,cy])\n    points=np.clip(points,[0,0],[w-1,h-1]);integer_points=points.astype(\'int32\')\n    im=Image.new(\'L\',(w,h));ImageDraw.Draw(im).polygon([tuple(p) for p in integer_points.tolist()],fill=1)\n    mask=np.array(im,dtype=np.uint8)\n    return mask,dict(rotated_box=[float(cx),float(cy),float(bw),float(bh),float(angle)],polygon=integer_points.tolist(),nonzero_pixels=int(np.count_nonzero(mask)),geometry_inferred_here=False,quality=\'NOT_EVALUATED\')\n\n\ndef add_shape_prior(base_embeddings, labels, centroids):\n    """Return one centroid addition to a copied embedding array, never in place.\n\n    Reuse the ORIGINAL base for every diffusion step. Passing a previously\n    conditioned array intentionally adds again and is not idempotent.\n    This helper does not classify the mask or load pickled model content.\n    """\n    base=np.asarray(base_embeddings)\n    ids=np.asarray(labels)\n    if base.ndim!=3 or not np.issubdtype(base.dtype,np.floating) or not np.isfinite(base).all():raise ValueError(\'base must be finite floating batch x slots x dimensions\')\n    if ids.shape!=base.shape[:2] or not np.issubdtype(ids.dtype,np.integer):raise ValueError(\'labels must be an integer batch x slots array\')\n    if not isinstance(centroids,dict):raise ValueError(\'centroids must be an already-loaded dictionary\')\n    result=base.copy()\n    for i in range(base.shape[0]):\n        for j in range(base.shape[1]):\n            label=int(ids[i,j])\n            if label not in centroids:continue\n            value=np.asarray(centroids[label],dtype=base.dtype)\n            if value.shape!=(base.shape[2],) or not np.isfinite(value).all():raise ValueError(\'centroid dimensions or values invalid\')\n            with np.errstate(over=\'ignore\',invalid=\'ignore\'):result[i,j]+=value\n    if not np.isfinite(result).all():raise ValueError(\'conditioned embeddings overflowed\')\n    return result\n'
# ---- embedded module: renderello/shadow_geometry.py ----
path = base / 'renderello/shadow_geometry.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Prepare an auditable, opt-in libcom shadow-module experiment.\n\nNo source file is edited by this module. This is not an upstream or validated\nmodel release, and is not imported by Renderello\'s production path.\n"""\nimport hashlib\n\nPINNED_LIBCOM_REVISION=\'32afffddf985e3c2832262d41960c8702b8c0fcb\'\nPINNED_SHADOW_SHA256=\'59e636774f0a1b7eb3c9912e453cf6ed01c92dbde1ff831ff8b874e1f03a870d\'\n\n\ndef prepare_experimental_shadow_source(original):\n    """Return new bytes plus a change ledger; reject unknown original bytes.\n\n    Does not install, download, import dependencies, or load model weights.\n    Removing the unused ControlLDM import is a SEPARATE preflight change and\n    is deliberately not applied here, because it belongs to another file.\n    """\n    if not isinstance(original,bytes):raise ValueError(\'source must be original bytes\')\n    digest=hashlib.sha256(original).hexdigest()\n    if digest!=PINNED_SHADOW_SHA256:raise ValueError(\'publisher source hash does not match pinned revision\')\n    text=original.decode(\'utf-8\')\n    replacements=[\n        (\'temp = pred_bbx[i,2]\',\'temp = pred_bbx[i,2].clone()\',\'clone scalar before overlapping tensor assignments\'),\n        (\'cv2.fillPoly(bbx_mask[i].cpu().numpy().astype(np.uint8), [np.int32(box_points)], 1)\',\n         \'box_array = bbx_mask[i].cpu().numpy().astype(np.uint8)\\n                        cv2.fillPoly(box_array, [np.int32(box_points)], 1)\\n                        bbx_mask[i] = torch.from_numpy(box_array).to(device=self.device, dtype=self.weight_dtype)\',\n         \'retain filled uint8 box and explicitly copy back to model tensor\'),\n        ("mask_embeddings = batch[\'embeddings\'].to(self.device, dtype=self.weight_dtype)",\n         "base_mask_embeddings = batch[\'embeddings\'].to(self.device, dtype=self.weight_dtype).clone()",\'retain unconditioned base once per image\'),\n        (\'# Mask embeddings adjustment\\n                        mask_label\',\n         \'# Mask embeddings adjustment: fresh base each denoising step\\n                        mask_embeddings = base_mask_embeddings.clone()\\n                        mask_label\',\n         \'condition a fresh base once per denoising step\'),\n    ]\n    ledger=[]\n    for old,new,reason in replacements:\n        if text.count(old)!=1:raise ValueError(\'expected one exact patch target\')\n        text=text.replace(old,new,1);ledger.append(reason)\n    compile(text,\'shadow_generation_v485.py\',\'exec\')\n    result=text.encode(\'utf-8\')\n    return result,dict(revision=PINNED_LIBCOM_REVISION,original_sha256=digest,experimental_sha256=hashlib.sha256(result).hexdigest(),changes=ledger,model_inference_tested=False,default_enabled=False)\n'
# ---- embedded module: renderello/shadow_source_patch.py ----
path = base / 'renderello/shadow_source_patch.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Opt-in native scene-context extension. Not product staging or outpainting.\n\nRetains the complete input photo including its actual lighting and contact.\nOnly new outer canvas pixels are synthesized from reviewed scene border colours.\nThis deliberately cannot remove the old backdrop, change pose or relight glass.\n"""\nimport math\nimport numpy as np\nfrom scipy.ndimage import gaussian_filter1d\nfrom .candidate_diagnostics import native_rgb,native_alpha,digest\n\n\ndef extend_source_context(rgb,alpha,*,material,reviewed_source_sha256,\n                          reviewed_alpha_sha256,canvas,offset,band=8,decay=48,smooth_outer=False):\n    native_rgb(rgb);native_alpha(alpha,rgb.shape[:2])\n    if rgb.shape[0]*rgb.shape[1]>1_500_000:raise ValueError(\'bounded1.5MP native source\')\n    if material!=\'opaque\':raise ValueError(\'opaque scene-context route only; no glass or material recovery\')\n    if digest(rgb)!=reviewed_source_sha256 or digest(alpha)!=reviewed_alpha_sha256:\n        raise ValueError(\'exact source/alpha review binding required\')\n    if not isinstance(canvas,(tuple,list))or len(canvas)!=2 or any(type(v)is not int or v<3 for v in canvas)or math.prod(canvas)>1_500_000:\n        raise ValueError(\'bounded integer canvas<=1.5MP required\')\n    if not isinstance(offset,(tuple,list))or len(offset)!=2 or any(type(v)is not int for v in offset):raise ValueError(\'integer source translation required\')\n    if type(smooth_outer)is not bool:raise ValueError(\'boolean smooth-outer switch required\')\n    h,w=alpha.shape;cw,ch=canvas;dx,dy=offset\n    if min(dx,dy)<0 or dx+w>cw or dy+h>ch:raise ValueError(\'entire source frame must fit; no crop or resize\')\n    if type(band)is not int or not 2<=band<=min(64,min(w,h)//4):raise ValueError(\'bounded scene border band required\')\n    if isinstance(decay,bool)or not isinstance(decay,(int,float))or not math.isfinite(decay)or not 4<=decay<=256:raise ValueError(\'bounded finite extension decay required\')\n    if not(alpha>=128).any():raise ValueError(\'nonempty hard foreground required\')\n    border=np.ones((h,w),bool);border[band:-band,band:-band]=False\n    if (alpha[border]>0).any():raise ValueError(\'source alpha intersects border band; scene-only extension unverified\')\n    samples=rgb[border].astype(float)\n    med=np.median(samples,axis=0);spread=float(np.percentile(np.linalg.norm(samples-med,axis=1),90))\n    if spread>32:raise ValueError(\'nonuniform scene border; no photographic texture or geometry extension\')\n    sy,sx=np.nonzero(border);design=np.column_stack([np.ones(len(sx)),sx/max(1,w-1),sy/max(1,h-1)])\n    coefficients=np.linalg.lstsq(design,samples,rcond=None)[0]\n    yy,xx=np.mgrid[:ch,:cw];local_x=(xx-dx)/max(1,w-1);local_y=(yy-dy)/max(1,h-1)\n    # Clamp fit coordinates to observed source bounds. Extrapolated slopes can\n    # invent impossible light ramps; fade only to the observed affine edge tone.\n    plane=coefficients[0]+np.clip(local_x,0,1)[...,None]*coefficients[1]+np.clip(local_y,0,1)[...,None]*coefficients[2]\n    sx=np.clip(xx-dx,0,w-1);sy=np.clip(yy-dy,0,h-1)\n    edge=rgb[sy,sx].astype(float);edgeplane=coefficients[0]+sx[...,None]/max(1,w-1)*coefficients[1]+sy[...,None]/max(1,h-1)*coefficients[2]\n    distance=np.hypot(xx-(sx+dx),yy-(sy+dy))\n    residual=edge-edgeplane\n    if smooth_outer:\n        # Smooth ONLY scene-border sequences tangentially. Never blur across\n        # the product or use a mirrored/replicated product as extension data.\n        tx=np.arange(w)/max(1,w-1);ty=np.arange(h)/max(1,h-1)\n        top=rgb[0].astype(float)-(coefficients[0]+tx[:,None]*coefficients[1])\n        bottom=rgb[-1].astype(float)-(coefficients[0]+tx[:,None]*coefficients[1]+coefficients[2])\n        left=rgb[:,0].astype(float)-(coefficients[0]+ty[:,None]*coefficients[2])\n        right=rgb[:,-1].astype(float)-(coefficients[0]+coefficients[1]+ty[:,None]*coefficients[2])\n        vx=np.maximum.reduce([dx-xx,xx-(dx+w-1),np.zeros_like(xx)]).astype(float)\n        vy=np.maximum.reduce([dy-yy,yy-(dy+h-1),np.zeros_like(yy)]).astype(float)\n        weight=np.divide(vx,vx+vy,out=np.zeros_like(vx),where=vx+vy>0)\n        target=np.minimum(distance/2,128);levels=[0,1,2,4,8,16,32,64,128];filtered=np.zeros_like(residual)\n        for i,level in enumerate(levels):\n            arrays=[v if level==0 else gaussian_filter1d(v,level,axis=0,mode=\'nearest\')for v in [top,bottom,left,right]]\n            vertical=np.where((xx<dx)[...,None],arrays[2][sy],arrays[3][sy])\n            horizontal=np.where((yy<dy)[...,None],arrays[0][sx],arrays[1][sx])\n            field=vertical*weight[...,None]+horizontal*(1-weight[...,None])\n            if i==0:blend=np.clip(1-target,0,1)\n            elif i==len(levels)-1:blend=np.clip((target-levels[i-1])/(level-levels[i-1]),0,1)\n            else:blend=np.minimum(np.clip((target-levels[i-1])/(level-levels[i-1]),0,1),np.clip((levels[i+1]-target)/(levels[i+1]-level),0,1))\n            filtered+=field*blend[...,None]\n        residual=filtered\n    result=np.rint(np.clip(plane+residual*np.exp(-distance[...,None]/decay),0,255)).astype(np.uint8)\n    result[dy:dy+h,dx:dx+w]=rgb\n    changed=np.any(result[dy:dy+h,dx:dx+w]!=rgb,axis=2)\n    if changed.any():raise RuntimeError(\'source context changed\')\n    report=dict(schema=\'source-context-extension.v1\',source_rgb_sha256=digest(rgb),alpha_sha256=digest(alpha),\n        canvas=list(canvas),integer_translation=list(offset),source_size=[w,h],source_changed_pixels=0,\n        source_pixels_preserved=h*w,new_outer_pixels=cw*ch-h*w,border_band=band,border90_rgb_distance=spread,\n        decay=decay,smooth_outer=smooth_outer,fit_coefficients=coefficients.tolist(),source_resampled=False,source_contact_retained=True,\n        source_lighting_retained=True,new_contact_synthesized=False,background_replaced=False,\n        new_template_executed=False,optics_recovered=False,photo_accepted=False,\n        limitations=[\'The complete old scene remains. This is added copy space, not a product staging template.\',\n            \'Smooth outer colour continuation has no recovered physical geometry or photographic texture.\',\n            \'A caller mask is not ground truth; undetected products at the border can contaminate extension.\',\n            \'Joins and repeated edge-colour streaks require native visual review.\'])\n    return result,report\n'
# ---- embedded module: renderello/source_context_extension.py ----
path = base / 'renderello/source_context_extension.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Explicit source-camera contract and non-semantic mask preflight.\n\nNever infer pose, product identity, transparency or label readability from a filename.\nDeclarations are reviewer inputs, not automatic vision or authenticated approval.\n"""\nfrom dataclasses import asdict, dataclass\nimport hashlib\nimport cv2\nimport numpy as np\nfrom .boundary import frame_edge_contact\n\nPOSES=(\'unknown\',\'front\',\'elevated\',\'overhead\',\'suspended\')\n\n\n@dataclass(frozen=True)\nclass SourceContract:\n    pose: str = \'unknown\'\n    material: str = \'unknown\'\n    framing: str = \'unknown\'\n    genuine_tiny_text: bool = False\n\n    def __post_init__(self):\n        if self.pose not in POSES:raise ValueError(\'unsupported source pose\')\n        if self.material not in (\'unknown\',\'opaque\',\'transparent\'):raise ValueError(\'unsupported source material\')\n        if self.framing not in (\'unknown\',\'complete\',\'cropped\'):raise ValueError(\'unsupported source framing\')\n\n\ndef source_mask_preflight(rgb, alpha, bbox, contract=None, *, overhead_scene=False,\n                          glass_mode=False, require_complete=False):\n    c=contract or SourceContract()\n    if rgb.ndim!=3 or rgb.shape[2]!=3 or alpha.ndim!=2 or rgb.shape[:2]!=alpha.shape:\n        raise ValueError(\'source and alpha dimensions must agree\')\n    if rgb.dtype!=np.uint8 or alpha.dtype!=np.uint8:raise ValueError(\'source and alpha must be uint8\')\n    x,y,w,h=bbox;H,W=alpha.shape\n    if min(x,y)<0 or min(w,h)<=0 or x+w>W or y+h>H:raise ValueError(\'invalid matte bbox\')\n    warnings=[];blockers=[]\n    edge=frame_edge_contact(alpha)\n    n,labels,stats,_=cv2.connectedComponentsWithStats((alpha>127).astype(np.uint8),8)\n    areas=stats[1:,cv2.CC_STAT_AREA].tolist()\n    visible=int((alpha>127).sum());soft=int(((alpha>0)&(alpha<255)).sum())\n    if visible==0:blockers.append(\'Empty product mask.\')\n    if c.pose==\'unknown\':warnings.append(\'Source camera/pose unverified; no camera compatibility claim.\')\n    if overhead_scene and c.pose==\'unknown\':\n        warnings.append(\'Requested overhead template requires camera review; unknown source pose does not establish compatibility.\')\n    if overhead_scene and c.pose not in (\'unknown\',\'overhead\'):\n        blockers.append(\'Declared source pose \'+c.pose+\' is incompatible with an overhead scene; source-exact paste cannot change camera angle.\')\n    if c.pose==\'suspended\':warnings.append(\'Suspended source pose needs an explicit floating concept, not a floor-contact claim.\')\n    if c.material==\'transparent\' and not glass_mode:\n        blockers.append(\'Transparent source needs an explicit transmission workflow; opaque RGB paste retains the old backdrop.\')\n    if c.material==\'unknown\':warnings.append(\'Material unverified; opaque paste may fail on transparent products.\')\n    if c.framing==\'cropped\' or edge[\'needs_review\']:\n        warnings.append(\'Source or alpha touches a frame cut; missing product parts cannot be recovered faithfully.\')\n        if require_complete:blockers.append(\'Complete-product requirement conflicts with a cropped source/mask.\')\n    if not c.genuine_tiny_text:warnings.append(\'No verified genuine tiny-text test; cannot claim fine-print preservation.\')\n    if visible and max(areas,default=0)/visible<.98:warnings.append(\'Multiple mask components: review every product, strap, handle and loose part; do not remove automatically.\')\n    if visible and soft/max(1,visible)>.3:warnings.append(\'Large soft-alpha fraction; inspect halos/transparency rather than certifying RGB fidelity.\')\n    # Cross-check bbox without changing alpha; data errors must not silently crop it.\n    yy,xx=np.where(alpha>0)\n    if len(xx) and (xx.min()<x or yy.min()<y or xx.max()>=x+w or yy.max()>=y+h):\n        warnings.append(\'Nonzero alpha outside declared bbox; inspect soft-edge loss before use.\')\n    return dict(declaration=asdict(c),declaration_provenance=\'reviewer_input_not_automatic_vision\',\n        passed=not blockers,blockers=blockers,warnings=warnings,frame_contact=edge,\n        mask_components=areas,foreground_px=visible,soft_alpha_px=soft,\n        source_rgb_sha256=hashlib.sha256(rgb.tobytes()).hexdigest(),\n        alpha_sha256=hashlib.sha256(alpha.tobytes()).hexdigest(),\n        semantic_mask_accuracy=\'NOT_VERIFIED\',publication_status=\'NOT_EVALUATED\')\n'
# ---- embedded module: renderello/source_contract.py ----
path = base / 'renderello/source_contract.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Native-pixel scene control, not generative relighting or automatic mask approval.\n\nA reviewed mask and explicitly reviewed opaque regions are required. The source\nis translated by whole pixels only. No resampling, warp or RGB repaint occurs.\nTransparent packaging retains old-scene reflections/transmission and is not\nclaimed optically correct. Output is a preview until separately reviewed.\n"""\nfrom pathlib import Path\nimport hashlib,json\nimport numpy as np\nfrom PIL import Image,ImageFilter\n\n\ndef compose_native(source, alpha_path, out, *, protected_regions, canvas=(1024,1024), offset=None):\n    source,alpha_path,out=map(Path,(source,alpha_path,out))\n    if out.exists():raise ValueError(\'output exists; preserve earlier versions\')\n    im=Image.open(source).convert(\'RGB\');alpha=Image.open(alpha_path).convert(\'L\')\n    if alpha.size!=im.size:raise ValueError(\'mask must match native source size\')\n    if im.width*im.height>4_000_000:raise ValueError(\'source too large for bounded preview\')\n    if not protected_regions:raise ValueError(\'reviewed opaque regions required\')\n    if len(canvas)!=2 or any(type(x)!=int or x<1 for x in canvas) or canvas[0]*canvas[1]>4_000_000:raise ValueError(\'invalid or oversized canvas\')\n    bbox=alpha.getbbox()\n    if bbox is None:raise ValueError(\'empty mask\')\n    if offset is None:offset=((canvas[0]-(bbox[2]-bbox[0]))//2-bbox[0],(canvas[1]-(bbox[3]-bbox[1]))//2-bbox[1])\n    if len(offset)!=2 or any(type(x)!=int for x in offset):raise ValueError(\'integer translation only\')\n    dx,dy=offset\n    if bbox[0]+dx<0 or bbox[1]+dy<0 or bbox[2]+dx>canvas[0] or bbox[3]+dy>canvas[1]:raise ValueError(\'mask would be clipped\')\n    aa=np.array(alpha);rgb=np.array(im);checks=[]\n    for name,box in protected_regions.items():\n        if len(box)!=4 or any(type(x)!=int for x in box):raise ValueError(\'integer boxes required\')\n        x1,y1,x2,y2=box\n        if not(0<=x1<x2<=im.width and 0<=y1<y2<=im.height):raise ValueError(\'box outside source\')\n        if np.any(aa[y1:y2,x1:x2]!=255):raise ValueError(\'protected region must already be opaque in reviewed mask\')\n        checks.append({\'name\':name,\'source_box\':box,\'output_box\':[x1+dx,y1+dy,x2+dx,y2+dy]})\n    # Flat neutral scene, low contrast falloff. Product RGB never relit.\n    yy,xx=np.mgrid[:canvas[1],:canvas[0]];grad=np.clip((yy/canvas[1])*8+((xx-canvas[0]*.4)/canvas[0])**2*8,0,15)\n    bg=np.stack([242-grad,239-grad,234-grad],axis=-1).astype(\'uint8\');base=Image.fromarray(bg)\n    # Explicitly synthetic contact ellipse, no claim of geometry correctness.\n    shadow=np.zeros((canvas[1],canvas[0]),dtype=\'uint8\');cx=(bbox[0]+bbox[2])/2+dx;cy=bbox[3]+dy-2\n    ex=max(20,(bbox[2]-bbox[0])*.38);ey=12\n    shadow=((np.exp(-((xx-cx)/ex)**2-((yy-cy)/ey)**2))*45).astype(\'uint8\')\n    shade=Image.fromarray(shadow).filter(ImageFilter.GaussianBlur(6));base.paste(Image.new(\'RGB\',canvas,(85,80,75)),(0,0),shade)\n    base.paste(im,(dx,dy),alpha)\n    seen=np.array(base)\n    for c in checks:\n        x1,y1,x2,y2=c[\'source_box\'];u1,v1,u2,v2=c[\'output_box\'];diff=np.abs(seen[v1:v2,u1:u2].astype(\'int16\')-rgb[y1:y2,x1:x2].astype(\'int16\'))\n        c.update(changed_pixels=int(np.any(diff,axis=2).sum()),max_channel_error=int(diff.max()),pixels=(x2-x1)*(y2-y1))\n        if c[\'changed_pixels\']:raise RuntimeError(\'protected source region changed in actual output\')\n    ys,xs=np.nonzero(aa==255)\n    opaque_error=np.abs(seen[ys+dy,xs+dx].astype(\'int16\')-rgb[ys,xs].astype(\'int16\'))\n    all_opaque={\'pixels\':len(xs),\'changed_pixels\':int(np.any(opaque_error,axis=1).sum()),\'max_channel_error\':int(opaque_error.max())}\n    if all_opaque[\'changed_pixels\']:raise RuntimeError(\'opaque source pixels changed\')\n    out.mkdir(parents=True);base.save(out/\'output.png\');alpha.save(out/\'reviewed-alpha.png\')\n    rgba=im.copy();rgba.putalpha(alpha);rgba.save(out/\'source-native-cutout.png\')\n    manifest={\'kind\':\'SOURCE_LOCKED_NATIVE_PREVIEW_NOT_QUALITY_APPROVAL\',\'source_sha256\':hashlib.sha256(source.read_bytes()).hexdigest(),\'source_size\':list(im.size),\'canvas\':list(canvas),\'integer_translation\':list(offset),\'source_resampled\':False,\'rgb_relit\':False,\'protected_regions\':checks,\'all_opaque_audit\':all_opaque,\'mask_source\':\'caller-supplied reviewed mask, not automatic approval\',\'limitations\':[\'Source blur and viewpoint remain.\',\'Transparent material retains old-scene transmission/reflections.\',\'Contact ellipse is procedural, not physically estimated.\',\'Mask edges need visual inspection.\'],\'output_sha256\':hashlib.sha256((out/\'output.png\').read_bytes()).hexdigest()}\n    (out/\'manifest.json\').write_text(json.dumps(manifest,indent=2));return manifest\n'
# ---- embedded module: renderello/source_locked_preview.py ----
path = base / 'renderello/source_locked_preview.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Import a pinned external RGBA without accepting its foreground or scene quality.\n\nThe file hash binds supplied evidence; it does not authenticate the model or its\npublisher. Exact source RGB is required even under zero alpha. No resize,\nrepainting, mask enforcement, or automatic candidate selection happens here.\n"""\nfrom io import BytesIO\nfrom pathlib import Path\nimport hashlib\nimport re\nimport os\nimport stat\nimport zlib\nimport numpy as np\nfrom PIL import Image\nfrom .candidate_diagnostics import native_rgb, digest\n\n\ndef _strict_png(data):\n    if not data.startswith(b\'\\x89PNG\\r\\n\\x1a\\n\'):\n        raise ValueError(\'Plain 8-bit RGBA PNG required\')\n    pos=8;count=0;idat=False\n    interpretation={b\'iCCP\',b\'gAMA\',b\'sRGB\',b\'cHRM\',b\'cICP\',b\'mDCV\',b\'cLLI\',b\'eXIf\',b\'tRNS\',b\'acTL\',b\'fcTL\',b\'fdAT\'}\n    while pos<len(data):\n        count+=1\n        if count>4096 or len(data)-pos<12:raise ValueError(\'Bounded complete PNG chunks required\')\n        length=int.from_bytes(data[pos:pos+4],\'big\');kind=data[pos+4:pos+8];end=pos+12+length\n        if end>len(data):raise ValueError(\'Truncated PNG chunk\')\n        payload=data[pos+8:pos+8+length];crc=int.from_bytes(data[pos+8+length:end],\'big\')\n        if zlib.crc32(kind+payload)&0xffffffff!=crc:raise ValueError(\'PNG chunk checksum mismatch\')\n        if count==1:\n            if kind!=b\'IHDR\' or length!=13 or payload[8]!=8 or payload[9]!=6:\n                raise ValueError(\'Explicit 8-bit RGBA PNG header required; no depth conversion\')\n        elif kind==b\'IHDR\':raise ValueError(\'Duplicate PNG header\')\n        if kind in interpretation:raise ValueError(\'PNG colour/orientation/transparency/frame interpretation metadata refused\')\n        if kind==b\'IDAT\':idat=True\n        if kind==b\'IEND\':\n            if length or end!=len(data)or not idat:raise ValueError(\'Exact complete PNG stream required\')\n            return\n        pos=end\n    raise ValueError(\'PNG end marker required\')\n\n\ndef load_source_locked_rgba(path, source_rgb, expected_sha256):\n    native_rgb(source_rgb)\n    if source_rgb.shape[0] * source_rgb.shape[1] > 1_000_000:\n        raise ValueError(\'External RGBA exceeds one-megapixel replay budget\')\n    if not isinstance(expected_sha256, str) or re.fullmatch(\'[0-9a-f]{64}\', expected_sha256) is None:\n        raise ValueError(\'Lowercase SHA-256 of the encoded result is required\')\n    path = Path(path)\n    if path.is_symlink():raise ValueError(\'Regular bounded RGBA result required\')\n    flags=os.O_RDONLY|getattr(os,\'O_NONBLOCK\',0)|getattr(os,\'O_NOFOLLOW\',0)\n    with os.fdopen(os.open(path,flags),\'rb\') as stream:\n        info=os.fstat(stream.fileno())\n        if not stat.S_ISREG(info.st_mode)or info.st_size>20_000_000:\n            raise ValueError(\'Regular bounded RGBA result required\')\n        data=stream.read(20_000_001)\n    if len(data) > 20_000_000 or hashlib.sha256(data).hexdigest() != expected_sha256:\n        raise ValueError(\'External RGBA size or SHA-256 mismatch\')\n    _strict_png(data)\n    with Image.open(BytesIO(data)) as im:\n        if im.format != \'PNG\' or im.mode != \'RGBA\' or im.size != (source_rgb.shape[1], source_rgb.shape[0]):\n            raise ValueError(\'Native-size RGBA PNG required; no conversion or resize\')\n        if getattr(im, \'n_frames\', 1) != 1 or im.info.get(\'icc_profile\') or im.getexif().get(274, 1) != 1:\n            raise ValueError(\'Single normalized unprofiled RGBA frame required\')\n        rgba = np.array(im)\n    changed = int(np.count_nonzero(np.any(rgba[:, :, :3] != source_rgb, axis=2)))\n    if changed:\n        raise ValueError(f\'External RGBA repaints {changed} source RGB pixels\')\n    alpha = rgba[:, :, 3].copy()\n    return alpha, dict(schema=\'source-locked-rgba-import.v1\', encoded_sha256=expected_sha256,\n        source_pixels_sha256=digest(source_rgb), imported_alpha_sha256=digest(alpha),\n        source_rgb_exact=True, source_rgb_changed_pixels=0, no_new_inference=True,\n        model_provenance=\'NOT_AUTHENTICATED_BY_PIXEL_REPLAY\', automatic_acceptance=False,\n        physical_relighting=False, glass_optics_verified=False)\n'
# ---- embedded module: renderello/source_locked_rgba_import.py ----
path = base / 'renderello/source_locked_rgba_import.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Explicit constrained template prompts, not the original shoot specification.\n\nNo model execution, mask recovery, viewpoint/light inference or acceptance.\nThe caller\'s source declaration is a hypothesis and never authenticates a photo.\n"""\nfrom dataclasses import asdict\nfrom .schemas import EditRequest\nfrom .source_contract import SourceContract\nfrom .prompts import FIDELITY_GUARD, validate_instruction\n\n# Environment-only alternatives. Original camera/light/rotation/reflection\n# demands are intentionally not carried into this constrained experiment.\nCONTEXTS = {\n \'clean_studio\':\'a plain neutral-white studio background with uncluttered negative space\',\n \'luxury_hero\':\'a dark charcoal premium studio background, without a generated product reflection\',\n \'editorial_beauty\':\'a quiet pastel beige editorial background with distant soft botanical accents\',\n \'lifestyle_kitchen\':\'a softly defocused warm kitchen environment behind the existing product\',\n \'natural_outdoor\':\'a defocused natural green outdoor environment behind the existing product\',\n \'minimal_tabletop\':\'a simple light-gray background without a newly invented podium under the product\',\n \'seasonal_holiday\':\'a warm seasonal background with restrained distant bokeh and no product occlusion\',\n \'social_ad\':\'a bold single-colour graphic background with copy space, no added text and no product rotation or floating\',\n \'reflective_surface\':\'a dark premium background without inventing a new reflection of the product\',\n \'flat_lay\':\'an uncluttered neutral background, retaining the existing source viewpoint rather than demanding overhead\',\n}\n\n\ndef build_source_preserving_prompt(request: EditRequest, declaration: SourceContract):\n    if not isinstance(request,EditRequest) or not isinstance(declaration,SourceContract):\n        raise ValueError(\'typed request and explicit source declaration required\')\n    if declaration.pose==\'unknown\' or declaration.material==\'unknown\' or declaration.framing==\'unknown\':\n        raise ValueError(\'explicit pose/material/framing declarations required; no inference\')\n    if declaration.material!=\'opaque\':\n        raise ValueError(\'opaque-only constrained experiment; no glass/transmission recovery\')\n    if declaration.framing!=\'complete\':\n        raise ValueError(\'complete source required; no uncropping or missing-part invention\')\n    key=request.template.key\n    if key not in CONTEXTS:raise ValueError(\'registered constrained context required\')\n    instruction=validate_instruction(request.custom_instruction).strip()\n    parts=[FIDELITY_GUARD,\n       \'This is a source-preserving background-context experiment, not a new camera or product-lighting setup.\',\n       f\'Retain the photographed product viewpoint and pose (caller declaration: {declaration.pose}); do not rotate, tilt, reshape, uncrop or reconstruct it.\',\n       \'Retain the existing product illumination, highlights, colour and material. Do not relight or repaint the product to fit the background.\',\n       f\'Change only the surrounding environment to {CONTEXTS[key]}.\',\n       \'Do not add or remove products, source-visible parts, product reflections or cast shadows that would require guessing hidden geometry. Target support and contact still need review.\']\n    if instruction:parts.append(\'Additional environment-only instruction: \'+instruction+\'.\')\n    return dict(schema=\'renderello.constrained-context-prompt.v1\',prompt=\' \'.join(parts),\n        base_template=asdict(request.template),variant=\'EXPLICIT_SOURCE_PRESERVING_CONTEXT_NOT_ORIGINAL_SHOOT\',\n        source_declaration=asdict(declaration),declaration_provenance=\'CALLER_HYPOTHESIS_NOT_CAPTURE_TRUTH\',\n        original_requirements_not_fulfilled=[\'original_scene_specification\',\'original_camera_specification\',\'original_lighting_specification\',\'requested_product_rotation_or_new_reflection\'],\n        source_pixel_lock_guaranteed=False,model_obedience_verified=False,alpha_verified=False,\n        target_contact_verified=False,polished_template=False,photo_accepted=False,\n        executed=False,paid_call_authorized=False,default_pipeline_changed=False)\n\n\ndef main(argv=None):\n    """Write an explicit executable prompt-preparation artifact, never generate."""\n    import argparse, json, hashlib\n    from pathlib import Path\n    from .templates import get_template\n    from .pipeline_intake import load_generation_image\n    parser=argparse.ArgumentParser(description=\'Prepare a constrained source-preserving context prompt. No image generation or spending.\')\n    parser.add_argument(\'--source\',type=Path,required=True)\n    parser.add_argument(\'--template\',required=True,choices=tuple(CONTEXTS))\n    parser.add_argument(\'--source-pose\',required=True,choices=(\'front\',\'elevated\',\'overhead\',\'suspended\'))\n    parser.add_argument(\'--material\',required=True,choices=(\'opaque\',))\n    parser.add_argument(\'--framing\',required=True,choices=(\'complete\',))\n    parser.add_argument(\'--instruction\',default=\'\')\n    parser.add_argument(\'--allow-colour-conversion\',action=\'store_true\')\n    parser.add_argument(\'--out\',type=Path,required=True)\n    args=parser.parse_args(argv)\n    if args.out.exists():raise FileExistsError(\'new prompt artifact required; never replace earlier work\')\n    request=EditRequest(args.source,get_template(args.template),custom_instruction=args.instruction)\n    record=build_source_preserving_prompt(request,SourceContract(args.source_pose,args.material,args.framing))\n    image,intake=load_generation_image(args.source,allow_colour_conversion=args.allow_colour_conversion)\n    image.close()\n    record[\'image_intake\']=intake\n    record[\'source_file_sha256\']=intake[\'source_sha256\']\n    record[\'benchmark_role\']=\'SEPARATE_CONSTRAINED_EXPERIMENT_CANNOT_REPLACE_ORIGINAL_TEMPLATE_CASE\'\n    record[\'generation_authority\']=\'NONE_PROMPT_PREPARATION_ONLY\'\n    args.out.parent.mkdir(parents=True,exist_ok=True)\n    with args.out.open(\'x\')as f:json.dump(record,f,indent=2,default=str)\n    print(str(args.out))\n    return 0\n\n\nif __name__==\'__main__\':\n    raise SystemExit(main())\n'
# ---- embedded module: renderello/source_preserving_prompt.py ----
path = base / 'renderello/source_preserving_prompt.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Content-bound source records. Rights metadata is not permission or clearance."""\nfrom __future__ import annotations\nimport hashlib\nimport json\nfrom pathlib import Path\n\n\ndef identify_source(path: Path) -> dict:\n    digest = hashlib.sha256(path.read_bytes()).hexdigest()\n    registry = Path(__file__).with_name(\'source_registry.json\')\n    records = json.loads(registry.read_text()) if registry.is_file() else []\n    match = next((r for r in records if r[\'sha256\'] == digest), None)\n    if match:\n        return dict(match, match_basis=\'exact_file_sha256\',\n                    capture_origin=match.get(\'capture_origin\', \'UNVERIFIED\'),\n                    physical_photo_eligible=match.get(\'physical_photo_eligible\'),\n                    rights_status=\'Recorded attribution only; no current legal clearance. Private test use; verify rights before republication.\')\n    return {\'sha256\': digest, \'classification\': \'UNKNOWN / unclassified source\',\n            \'match_basis\': \'no_registered_content_match\',\n            \'capture_origin\': \'UNVERIFIED\', \'physical_photo_eligible\': None,\n            \'rights_status\': \'No source or rights record. Do not assume synthetic, licensed, or publishable.\'}\n'
# ---- embedded module: renderello/source_provenance.py ----
path = base / 'renderello/source_provenance.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Studio: the single high-level generate API shared by the website and the\nColab notebook. One call = the brief\'s working flow:\n\n    product photo(s) -> preset -> optional long instruction + references\n    -> N variants -> QA -> compare/download bundle\n\nTwo generation modes, chosen explicitly per run (no silent fallback):\n- compose: deterministic cutout-compose. Scene providers create the\n  environment only (procedural / sdxl_lightning / hf_inference); product\n  pixels stay source-exact.\n- magichour: whole-image edit through the Magic Hour editor API - the\n  brief\'s intended backend, owner-directed 2026-11-20. Live calls need the\n  key plus RENDERELLO_MH_LIVE=1, set only with a fresh owner spend\n  approval; without configuration the run refuses with zero bill.\n  RENDERELLO_MH_DEV_MOCK=1 is an explicit developer placeholder mode,\n  labeled as such everywhere.\n\nEvery run writes a manifest recording provider, prompt plan, QA verdicts\nand receipts. Nothing is hidden: failures are listed, not dropped.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport os\nimport sys\nimport time\nfrom dataclasses import dataclass, field, asdict\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image\n\nfrom . import compose, prompts, qa\nfrom .longprompt import plan_prompt\nfrom .multinput import (GroupMember, compose_group, cutout_products,\n                        merged_reference_style)\nfrom .presets import Preset, build_detailed_prompt, get_preset\nfrom .scene_providers import SceneProvider, get_provider\nfrom .schemas import Aspect\n\n\ndef runtime_versions() -> dict:\n    """Installed package versions only; never environment variables or secrets."""\n    from importlib.metadata import version, PackageNotFoundError\n    versions = {}\n    for name in ("torch", "diffusers", "transformers", "accelerate", "safetensors",\n                 "huggingface_hub", "Pillow", "numpy", "onnxruntime", "gradio"):\n        try:\n            versions[name] = version(name)\n        except PackageNotFoundError:\n            versions[name] = None\n    return versions\n\n\n@dataclass\nclass StudioRequest:\n    product_paths: list[Path]\n    preset_key: str\n    instruction: str = ""\n    reference_paths: list[Path] = field(default_factory=list)\n    variants: int = 4\n    aspect: str | None = None          # "1:1" | "4:5" | "3:2" | None\n    provider: str = "procedural"       # procedural | sdxl_lightning | hf_inference | magichour\n    mh_route: str = "scene"            # scene = background-only + preserved cutout; whole = whole-image edit; hybrid = preserved cutout + harmonize pass\n    material: str | None = None\n    product_description: str = ""\n    knob_values: dict[str, str] = field(default_factory=dict)\n    device: str = "cpu"\n\n\n@dataclass\nclass VariantOut:\n    index: int\n    seed: int\n    image: str                    # path relative to run dir\n    provider: str\n    prompt_report: dict\n    qa: dict\n    warnings: list[str] = field(default_factory=list)\n    error: str | None = None\n\n\n@dataclass\nclass StudioResult:\n    run_id: str\n    run_dir: Path\n    preset: str\n    provider_requested: str\n    variants: list[VariantOut]\n    group_errors: list[str]\n    reference_style: dict | None\n    manifest_path: Path\n    elapsed_s: float\n\n\n_ASPECTS = {a.value: a for a in Aspect}\n\n\ndef _product_entries(product_paths, members: list[GroupMember]) -> list[dict]:\n    """Manifest product rows: file identity plus which cutout backend actually\n    produced the matte, so a silent fallback can never hide in a run."""\n    by_name = {Path(m.path).name: m for m in members}\n    entries: list[dict] = []\n    for p in product_paths:\n        name = Path(p).name\n        entry = {"file": name, "sha16": _hash_file(Path(p))}\n        m = by_name.get(name)\n        if m is not None and m.cutout is not None:\n            entry["cutout_backend"] = m.cutout.backend\n            entry["cutout_coverage"] = round(m.cutout.coverage, 4)\n        entries.append(entry)\n    return entries\n\n\ndef _cutout_warnings(members: list[GroupMember]) -> list[str]:\n    """User-visible honesty lines when the learned cutout did not run."""\n    warns: list[str] = []\n    for m in members:\n        if m.cutout is not None and m.cutout.backend == "grabcut":\n            warns.append(\n                f"{Path(m.path).name}: cutout fell back to OpenCV grabcut "\n                "(learned BiRefNet failed - see the RuntimeWarning in the "\n                "run output for the reason). Matte edges are crude, not "\n                "publish-quality.")\n    return warns\n\n\ndef _hash_file(p: Path) -> str:\n    h = hashlib.sha256()\n    with open(p, "rb") as f:\n        for chunk in iter(lambda: f.read(1 << 20), b""):\n            h.update(chunk)\n    return h.hexdigest()[:16]\n\n\nimport threading\n_BUDGET_LOCK = threading.Lock()\n\n\nclass _ledger_file_lock():\n    """Cross-process exclusive lock on the ledger (fcntl flock). The thread\n    lock above serializes this process; the file lock serializes concurrent\n    workers on the same host (e.g. multiple uvicorn processes sharing the\n    runs volume)."""\n    def __enter__(self):\n        import fcntl\n        lock = _ledger_path().parent / (self._name() + ".lock")\n        lock.parent.mkdir(parents=True, exist_ok=True)\n        self._f = open(lock, "a")\n        fcntl.flock(self._f, fcntl.LOCK_EX)\n        return self\n\n    def __exit__(self, *exc):\n        import fcntl\n        fcntl.flock(self._f, fcntl.LOCK_UN)\n        self._f.close()\n\n    @staticmethod\n    def _name() -> str:\n        return _ledger_path().name\n\n\ndef _ledger_path() -> Path:\n    return Path(os.environ.get("RENDERELLO_MH_LEDGER",\n                               "runs/mh-spend-ledger.jsonl"))\n\n\ndef _ledger_append(row: dict) -> None:\n    ledger = _ledger_path()\n    ledger.parent.mkdir(parents=True, exist_ok=True)\n    row = dict(row)\n    row["ts"] = time.strftime("%Y-%m-%dT%H:%M:%S")\n    with ledger.open("a") as f:\n        f.write(json.dumps(row) + "\\n")\n\n\ndef _ledger_spent() -> int:\n    # Reservations count positive; reconciliations are negative adjustments,\n    # so the sum is the current committed spend. A crashed batch keeps its\n    # worst-case reservation - conservative, never under-counts.\n    ledger = _ledger_path()\n    spent = 0\n    if ledger.exists():\n        for line in ledger.read_text().splitlines():\n            try:\n                row = json.loads(line)\n            except json.JSONDecodeError:\n                continue\n            amt = row.get("credits_spent")\n            if isinstance(amt, (int, float)):\n                spent += int(amt)\n    return spent\n\n\ndef _budget_checks(model: str, need: int, require_floor: bool) -> None:\n    """All refusal checks for a batch costing `need` credits. Raises\n    RuntimeError with a zero-bill message on any failure."""\n    from .backends.magichour import MODEL_TABLE\n    raw = os.environ.get("RENDERELLO_MH_BUDGET_CREDITS", "").strip()\n    if not raw:\n        raise RuntimeError(\n            "Magic Hour live generation is OFF: no credit budget allocated "\n            "(RENDERELLO_MH_BUDGET_CREDITS unset). Zero credits billed.")\n    try:\n        budget = int(raw)\n    except ValueError:\n        raise RuntimeError("RENDERELLO_MH_BUDGET_CREDITS is not an integer; "\n                           "live generation refused, zero credits billed.")\n    listed = MODEL_TABLE.get(model, {}).get("credits")\n    if listed is None:\n        raise RuntimeError(f"unknown model {model!r}; live generation refused, "\n                           "zero credits billed.")\n    max_batch = int(os.environ.get("RENDERELLO_MH_MAX_BATCH_CREDITS", "120") or 120)\n    if need > max_batch:\n        raise RuntimeError(\n            f"batch too large: this run could cost up to {need} credits "\n            f"(limit {max_batch} per batch). Reduce variants/templates or "\n            "the owner raises RENDERELLO_MH_MAX_BATCH_CREDITS. Refused - "\n            "zero credits billed.")\n    spent = _ledger_spent()\n    if spent + need > budget:\n        raise RuntimeError(\n            f"Magic Hour budget exhausted: {spent}/{budget} credits committed, "\n            f"this batch could cost up to {need}. Refused - zero credits "\n            "billed. The owner must authorize more budget.")\n    # Deploy-proof hard cap: the ledger lives on ephemeral storage and resets\n    # on every redeploy, so the ledger check alone could re-allow spend\n    # already consumed. RENDERELLO_MH_CAP_ANCHOR = "<anchor_balance>:<total>"\n    # anchors the cap in the LIVE account balance, which cannot reset:\n    #   spent_so_far = anchor_balance - current_balance\n    # A balance ABOVE the anchor (top-up, credit adjustment) fails CLOSED:\n    # spend freezes until the owner re-anchors - top-ups can never widen the\n    # authorized total. Combined with the web layer\'s single-live-job rule,\n    # every posted charge is reflected before the next reservation is read.\n    # Durable ledger (GitLab CAS project): survives redeploys, unlike the\n    # on-host JSONL. The effective spend is the MAX of every counter - no\n    # counter is ever trusted to be complete on its own.\n    from . import mh_ledger\n    durable_spent = None\n    if mh_ledger.enabled():\n        durable_spent = mh_ledger.spent()  # raises (fail closed) on failure\n        if durable_spent > spent:\n            spent = durable_spent\n            if spent + need > budget:\n                raise RuntimeError(\n                    f"Magic Hour budget exhausted: {spent}/{budget} credits "\n                    f"committed (durable ledger), this batch could cost up "\n                    f"to {need}. Refused - zero credits billed. The owner "\n                    "must authorize more budget.")\n    anchor_raw = os.environ.get("RENDERELLO_MH_CAP_ANCHOR", "").strip()\n    floor_raw = os.environ.get("RENDERELLO_MH_MIN_BALANCE", "").strip()\n    if not anchor_raw and not floor_raw:\n        if require_floor:\n            raise RuntimeError(\n                "RENDERELLO_MH_CAP_ANCHOR unset: the balance-anchored hard "\n                "cap is required for live spend (the ledger resets on "\n                "redeploy). Refused - zero credits billed.")\n        return\n    try:\n        from .backends.magichour import MagicHourBackend\n        bal = MagicHourBackend(model=model, live=True).balance()\n        balance = int(bal.get("credits", 0))\n    except Exception as e:\n        raise RuntimeError(f"could not verify live account balance "\n                           f"({type(e).__name__}); live generation "\n                           "refused, zero credits billed.")\n    if anchor_raw:\n        try:\n            a_bal_s, a_total_s = anchor_raw.split(":")\n            a_bal, a_total = int(a_bal_s), int(a_total_s)\n        except ValueError:\n            raise RuntimeError("RENDERELLO_MH_CAP_ANCHOR must be "\n                               "\'<anchor_balance>:<authorized_total>\'; "\n                               "refused, zero credits billed.")\n        spent_live = a_bal - balance\n        _LAST_BALANCE["value"] = balance  # prime the per-call drop check\n        if spent_live < 0:\n            raise RuntimeError(\n                "live balance is ABOVE the authorized anchor (top-up or "\n                "adjustment?): spend frozen until the owner re-anchors. "\n                "Refused - zero credits billed.")\n        # Both counters in the SAME units: the durable ledger is seeded\n        # with RENDERELLO_MH_SPENT_BASELINE (spend before the anchor was\n        # set), so durable_total == baseline + spent_since_anchor when\n        # nothing is lost. durable < baseline + spent_live means the\n        # durable ledger lost history (wrong project, reseeded file,\n        # manual edit) - freeze rather than guess. (durable > baseline +\n        # spent_live is normal: reservations land in the durable ledger\n        # before the charge posts to the account balance.)\n        baseline_raw = os.environ.get("RENDERELLO_MH_SPENT_BASELINE", "0").strip()\n        try:\n            baseline = int(baseline_raw) if baseline_raw else 0\n        except ValueError:\n            raise RuntimeError("RENDERELLO_MH_SPENT_BASELINE is not an "\n                               "integer; live generation refused, zero "\n                               "credits billed.")\n        if durable_spent is not None and durable_spent < baseline + spent_live:\n            raise RuntimeError(\n                "durable ledger disagrees with the balance anchor "\n                f"(durable {durable_spent} < baseline {baseline} + "\n                f"anchor-derived {spent_live}): spend frozen until the "\n                "owner reconciles the counters. Refused - zero credits "\n                "billed.")\n        if spent_live + need > a_total:\n            raise RuntimeError(\n                f"Magic Hour hard cap reached: {spent_live}/{a_total} "\n                f"credits used since anchor, this batch needs up to {need}. "\n                "Refused - zero credits billed. The owner must authorize "\n                "more budget.")\n    elif floor_raw:\n        try:\n            floor = int(floor_raw)\n        except ValueError:\n            raise RuntimeError("RENDERELLO_MH_MIN_BALANCE is not an integer; "\n                               "live generation refused, zero credits billed.")\n        if balance - need < floor:\n            raise RuntimeError(\n                f"live balance floor reached: balance {balance}, batch needs "\n                f"up to {need}, floor {floor}. Refused - zero credits billed. "\n                "The owner must authorize more budget.")\n\n\n_LAST_BALANCE = {"value": None}\n\n\ndef _live_model_allowlist() -> set:\n    raw = os.environ.get("RENDERELLO_MH_LIVE_MODELS",\n                         "seedream-v5-flash").strip()\n    return {m.strip() for m in raw.split(",") if m.strip()}\n\n\ndef _post_call_charge(backend, listed: int, model: str) -> tuple:\n    """Authoritative charge for ONE paid call: the LARGER of the provider\'s\n    reported credits_charged (submission accept or terminal detail) and the\n    actual account balance drop since the previous check. Returns\n    (credits, abort_reason). abort_reason is set when the batch must stop:\n    the balance dropped more than the provider reported, the observed\n    charge exceeds the listed rate, or the balance is unreadable after a\n    paid call. A smaller drop (refund/adjustment) NEVER reduces the\n    counted spend."""\n    obs = _observed_charge(backend)\n    prev = _LAST_BALANCE["value"]\n    try:\n        from .backends.magichour import MagicHourBackend\n        bal = MagicHourBackend(model=model, live=True).balance()\n        now = int(bal.get("credits", 0))\n    except Exception as e:\n        return (obs if obs is not None else listed), (\n            f"post-call balance unreadable ({type(e).__name__}) after a "\n            "paid call: batch aborted; the counted charge above IS "\n            "reconciled to both ledgers (not dropped); further spend "\n            "fails closed until the balance reads again")\n    _LAST_BALANCE["value"] = now\n    if prev is not None:\n        drop = prev - now\n        if drop > (obs if obs is not None else 0):\n            return drop, (\n                f"account balance dropped {drop} credits but the provider "\n                f"reported {obs}: the larger figure is counted and the "\n                "batch aborted")\n    if obs is not None and obs > listed:\n        return obs, (\n            f"provider charged {obs} credits for this call, above the "\n            f"listed {listed}: batch aborted, remaining variants not "\n            "attempted")\n    return (obs if obs is not None else listed), None\n\n\ndef reserve_budget(model: str, count: int) -> int:\n    """Atomic check + reservation of the worst-case batch cost. The lock\n    closes the check-then-spend race between concurrent jobs; the durable\n    ledger row stands even if the process dies mid-batch (conservative).\n    Returns the reserved credit amount."""\n    from .backends.magichour import MODEL_TABLE\n    if model not in _live_model_allowlist():\n        raise RuntimeError(\n            f"model {model!r} is not on the live allowlist "\n            "(RENDERELLO_MH_LIVE_MODELS, default seedream-v5-flash: the "\n            "only model whose exact request cost is source-verified at "\n            "the pinned 640px / single-image shape). Refused - zero "\n            "credits billed.")\n    listed = MODEL_TABLE.get(model, {}).get("credits")\n    if listed is None:\n        raise RuntimeError(f"unknown model {model!r}; live generation refused, "\n                           "zero credits billed.")\n    need = listed * max(1, count)\n    with _BUDGET_LOCK, _ledger_file_lock():\n        _budget_checks(model, need, require_floor=True)\n        # Durable ledger FIRST: the reservation must survive a redeploy\n        # before any paid call can happen. A failed durable reserve fails\n        # closed - the paid call never happens (LedgerError propagates).\n        from . import mh_ledger\n        if mh_ledger.enabled():\n            mh_ledger.reserve(need)\n        _ledger_append({"model": model, "credits_spent": need,\n                        "kind": "reserve", "charged_observed": False})\n    return need\n\n\ndef reconcile_budget(model: str, reserved: int, actual: int) -> None:\n    """After a batch: adjust the reservation to the observed (or\n    conservatively listed) actual spend."""\n    # Unknown observed charge: keep the full reservation (conservative).\n    if actual is None:\n        return\n    delta = int(actual) - int(reserved)\n    if not delta:\n        return\n    with _BUDGET_LOCK, _ledger_file_lock():\n        from . import mh_ledger\n        if mh_ledger.enabled():\n            try:\n                # Move the reservation TO the observed charge - DOWN when\n                # lower, UP when the provider overcharged: an overcharge\n                # is real spend the durable ledger must record, or a\n                # restart would permanently under-count it.\n                mh_ledger.reconcile(reserved, actual)\n            except mh_ledger.LedgerError:\n                # Failed reconcile KEEPS the reservation in both ledgers.\n                # If the real charge was higher, the balance anchor\n                # exceeds the durable ledger and spend freezes until the\n                # owner reconciles the counters - fail closed either way.\n                return\n        _ledger_append({"model": model, "credits_spent": delta,\n                        "kind": "reconcile",\n                        "charged_observed": True,\n                        "overcharge": bool(delta > 0)})\n\n\ndef _enforce_budget(model: str, count: int = 1) -> None:\n    """Check-only pre-flight (no reservation): used by the web layer to fail\n    fast before upload work. The authoritative control is reserve_budget."""\n    from .backends.magichour import MODEL_TABLE\n    listed = MODEL_TABLE.get(model, {}).get("credits")\n    if listed is None:\n        raise RuntimeError(f"unknown model {model!r}; live generation refused, "\n                           "zero credits billed.")\n    _budget_checks(model, listed * max(1, count), require_floor=False)\n\n\ndef _observed_charge(backend) -> int | None:\n    """Highest charge the provider reported for the last call, from the\n    submission accept (authoritative at POST time per the API reference:\n    the 200 response carries credits_charged) or the terminal detail.\n    None when neither was observed (unsettled)."""\n    state = backend.last_result or {}\n    obs = None\n    for row in (state.get("submission"), state.get("final")):\n        if isinstance(row, dict):\n            c = row.get("credits_charged")\n            if isinstance(c, (int, float)) and c >= 0:\n                obs = max(obs or 0, int(c))\n    return obs\n\n\nclass _OverchargeError(RuntimeError):\n    """The provider charged above the listed reservation for one call.\n    Raised to abort the rest of a batch immediately."""\n\n\nSCENE_ATTEMPTS_DEFAULT = 1\nSCENE_ATTEMPTS_MAX = 3\n\n\ndef scene_attempts_from(knobs: dict) -> int:\n    """Single parser for the scene_max_attempts knob, shared by the route and\n    the web layer so reservations can never undercount: clamp to 1..3, any\n    junk/negative/oversize value falls back to the default."""\n    try:\n        n = int(str(knobs.get("scene_max_attempts", SCENE_ATTEMPTS_DEFAULT)))\n    except (TypeError, ValueError):\n        return SCENE_ATTEMPTS_DEFAULT\n    if n < 1 or n > SCENE_ATTEMPTS_MAX:\n        return SCENE_ATTEMPTS_DEFAULT\n    return n\n\n\ndef budget_status(model: str = "seedream-v5-flash") -> dict:\n    """Current budget view for UIs: allocated, spent, remaining, per-image\n    listed cost. Never raises; unconfigured budget reports zeros."""\n    from .backends.magichour import MODEL_TABLE\n    raw = os.environ.get("RENDERELLO_MH_BUDGET_CREDITS", "").strip()\n    try:\n        budget = int(raw) if raw else 0\n    except ValueError:\n        budget = 0\n    spent = _ledger_spent()\n    durable = None\n    try:\n        from . import mh_ledger\n        if mh_ledger.enabled():\n            durable = mh_ledger.spent()\n            spent = max(spent, durable)\n    except Exception:\n        pass  # status is informational; the reserve path fails closed\n    out = {"allocated": budget, "spent": spent,\n           "remaining": max(0, budget - spent),\n           "credits_per_image": MODEL_TABLE.get(model, {}).get("credits"),\n           "max_batch": int(os.environ.get("RENDERELLO_MH_MAX_BATCH_CREDITS",\n                                           "120") or 120)}\n    if durable is not None:\n        out["durable_spent"] = durable\n    return out\n\n\ndef record_spend(model: str, charged, listed: int) -> None:\n    """Append one line to the aggregate spend ledger after a live call.\n    Unknown charges are recorded at the listed rate (conservative)."""\n    ledger = Path(os.environ.get("RENDERELLO_MH_LEDGER",\n                                 "runs/mh-spend-ledger.jsonl"))\n    ledger.parent.mkdir(parents=True, exist_ok=True)\n    amt = charged if isinstance(charged, (int, float)) and charged >= 0 else listed\n    with ledger.open("a") as f:\n        f.write(json.dumps({"ts": time.strftime("%Y-%m-%dT%H:%M:%S"),\n                            "model": model, "credits_spent": int(amt),\n                            "charged_observed": isinstance(charged, (int, float))}) + "\\n")\n\n\ndef _qa_dict(report) -> dict:\n    return {\n        "fidelity_score": report.score_fidelity,\n        "critical_errors": list(report.critical_errors),\n        "notes": report.notes,\n        "metrics": {k: (round(float(x), 4) if x is not None else None)\n                    for k, x in (("ssim_product", report.ssim_product),\n                                 ("l1_product", report.l1_product),\n                                 ("color_drift", report.color_drift),\n                                 ("edge_retained", report.edge_retained))},\n    }\n\n\ndef _save_contact_sheet(run_dir: Path, variants: list[VariantOut],\n                        members: list[GroupMember]) -> None:\n    """Side-by-side source(s) + variants for quick eye review/download."""\n    tiles: list[Image.Image] = []\n    for m in members:\n        if m.cutout is not None:\n            tiles.append(Image.open(m.path).convert("RGB"))\n    for v in variants:\n        if v.image and (run_dir / v.image).exists():\n            tiles.append(Image.open(run_dir / v.image).convert("RGB"))\n    if not tiles:\n        return\n    h = 420\n    scaled = []\n    for t in tiles:\n        w = max(1, int(t.width * h / t.height))\n        scaled.append(t.resize((w, h), Image.LANCZOS))\n    sheet = Image.new("RGB", (sum(t.width for t in scaled) + 8 * len(scaled),\n                              h + 8), (24, 24, 24))\n    x = 8\n    for t in scaled:\n        sheet.paste(t, (x, 4))\n        x += t.width + 8\n    sheet.save(run_dir / "contact_sheet.png")\n\n\ndef run_studio(req: StudioRequest, out_root: Path,\n               provider: SceneProvider | None = None) -> StudioResult:\n    """Run the full flow once. CPU-safe; GPU/live providers raise honestly."""\n    t0 = time.time()\n    preset: Preset = get_preset(req.preset_key)\n    template = preset.template\n    prompts.validate_instruction(req.instruction)\n    aspect = _ASPECTS[req.aspect] if req.aspect else template.aspect\n\n    run_id = time.strftime("%Y%m%d-%H%M%S") + "-" + hashlib.sha1(\n        (str(req.product_paths) + str(time.time())).encode()).hexdigest()[:6]\n    run_dir = Path(out_root) / run_id\n    run_dir.mkdir(parents=True, exist_ok=False)\n\n    # Reference style (never composited; steers scene/prompt only).\n    ref_style = merged_reference_style([Path(p) for p in req.reference_paths])\n    ref_fragment = ref_style.prompt_fragment() if ref_style else ""\n\n    # Product cutouts (one per uploaded product photo). Editor mode does not\n    # need them, but they are cheap and power the QA matte + honest errors.\n    inter = run_dir / "intermediates"\n    members = cutout_products([Path(p) for p in req.product_paths], inter,\n                              device=req.device)\n    group_errors = [f"{Path(m.path).name}: {m.error}" for m in members if m.error]\n    ok_members = [m for m in members if m.cutout is not None]\n\n    if req.provider == "magichour":\n        if req.mh_route not in ("scene", "whole", "hybrid"):\n            raise ValueError(f"unknown Magic Hour route {req.mh_route!r}; "\n                             "expected scene, whole or hybrid")\n        if req.mh_route == "whole":\n            return _run_magichour(req, preset, aspect, run_id, run_dir,\n                                  ref_style, ref_fragment, members, group_errors, t0)\n        return _run_magichour_scene(req, preset, aspect, run_id, run_dir,\n                                    ref_style, ref_fragment, members,\n                                    group_errors, t0,\n                                    harmonize=(req.mh_route == "hybrid"))\n\n    provider = provider or get_provider(req.provider)\n    p_ok, p_why = provider.available()\n\n    variants: list[VariantOut] = []\n    n = max(1, min(req.variants, 4))\n    for i in range(n):\n        seed = int(req.knob_values.get("seed", 0) or 0) + i * 101\n        full_prompt = build_detailed_prompt(\n            preset, product_description=req.product_description,\n            material=req.material, instruction=req.instruction,\n            reference_style=ref_fragment, knob_values=req.knob_values)\n        plan = plan_prompt(full_prompt, prompts.build_negative_prompt(),\n                           provider.name)\n        v = VariantOut(index=i, seed=seed, image="", provider=provider.name,\n                       prompt_report=plan.report(), qa={})\n        out_path = run_dir / f"variant_{i}_{provider.name}.png"\n        try:\n            if not ok_members:\n                raise RuntimeError("no product cutout succeeded: " +\n                                   "; ".join(group_errors))\n            if not p_ok:\n                raise RuntimeError(f"provider unavailable: {p_why}")\n            if len(ok_members) > 1:\n                canvas = compose_group(members, template, aspect, seed,\n                                       reference_style=ref_style)\n                v.warnings.append("group layout on procedural backdrop")\n            else:\n                co = ok_members[0].cutout\n                scene = provider.generate_scene(\n                    aspect.size, plan.full_text, seed, template,\n                    negative=plan.negative)\n                # Empty scene replaces the procedural backdrop; the verified\n                # product cutout is pasted source-exact on top, with the\n                # template\'s contact-shadow language (composite reference path).\n                canvas_img = compose.composite(\n                    co, template, aspect, seed,\n                    reflection=(template.key == "reflective_surface"),\n                    reference=np.asarray(scene))\n                canvas = np.asarray(canvas_img)\n            Image.fromarray(canvas).save(out_path)\n            v.image = out_path.name\n            src_img = Image.open(ok_members[0].path).convert("RGB")\n            report = qa.evaluate(src_img, out_path, route="cutout_compose",\n                                 template=preset.key, seed=seed)\n            v.qa = _qa_dict(report)\n        except Exception as e:\n            v.error = f"{type(e).__name__}: {e}"\n        variants.append(v)\n\n    elapsed = time.time() - t0\n    manifest = {\n        "run_id": run_id,\n        "created": time.strftime("%Y-%m-%d %H:%M:%S"),\n        "preset": preset.key,\n        "provider_requested": req.provider,\n        "provider_used": provider.name,\n        "provider_available": p_ok,\n        "provider_note": provider.quality_note,\n        "mode": "cutout-compose",\n        "products": _product_entries(req.product_paths, members),\n        "warnings": _cutout_warnings(members),\n        "references": [Path(p).name for p in req.reference_paths],\n        "reference_style": (asdict(ref_style) if ref_style else None),\n        "instruction": req.instruction,\n        "material": req.material,\n        "knobs": req.knob_values,\n        "aspect": aspect.value,\n        "group_errors": group_errors,\n        "variants": [asdict(v) for v in variants],\n        "elapsed_s": round(elapsed, 2),\n        "contact_sheet": "contact_sheet.png",\n        "honesty": ("Deterministic routes keep product pixels source-exact; "\n                    "generative scenes change only the environment. QA scores "\n                    "are automated proxies - human eye review remains the "\n                    "acceptance gate per the brief."),\n    }\n    manifest_path = run_dir / "manifest.json"\n    manifest["runtime_versions"] = runtime_versions()\n    manifest_path.write_text(json.dumps(manifest, indent=2))\n    _save_contact_sheet(run_dir, variants, members)\n\n    return StudioResult(\n        run_id=run_id, run_dir=run_dir, preset=preset.key,\n        provider_requested=req.provider, variants=variants,\n        group_errors=group_errors,\n        reference_style=(asdict(ref_style) if ref_style else None),\n        manifest_path=manifest_path, elapsed_s=elapsed)\n\n\ndef _run_magichour(req: StudioRequest, preset: Preset, aspect: Aspect,\n                   run_id: str, run_dir: Path, ref_style, ref_fragment: str,\n                   members: list[GroupMember], group_errors: list[str],\n                   t0: float) -> StudioResult:\n    """Whole-image edit route through the Magic Hour editor API.\n\n    Live calls require the key in the environment AND RENDERELLO_MH_LIVE=1,\n    which is set only with a fresh owner spend approval. Without that, the\n    run REFUSES with a configuration-required message and zero bill - fake\n    outputs never appear in the user flow. A synthetic placeholder exists\n    only for explicit developer testing with RENDERELLO_MH_DEV_MOCK=1, and\n    is labeled as such in every manifest/variant. Products AND references\n    are passed as editor input images (the API supports up to 9 on\n    nano-banana/seedream families), so multi-product shots and style\n    references work natively.\n    """\n    from .backends.magichour import MagicHourBackend\n    model = req.knob_values.get("mh_model", "nano-banana-2.1")\n    live = os.environ.get("RENDERELLO_MH_LIVE") == "1"\n    dev_mock = os.environ.get("RENDERELLO_MH_DEV_MOCK") == "1"\n    if not live and not dev_mock:\n        raise RuntimeError(\n            "Magic Hour is not configured for live generation on this "\n            "server (no API key / live flag). Configuration required - "\n            "zero credits were billed and no image was generated. "\n            "Developers: RENDERELLO_MH_DEV_MOCK=1 runs the explicit "\n            "placeholder test mode.")\n    reserved = 0\n    actual = {"n": 0}\n    n = max(1, min(req.variants, 4))\n    if live:\n        # Only a SINGLE-INPUT request has source-verified exact pricing\n        # (30 credits, seedream-v5-flash @ 640px, image_count=1). Extra\n        # product images and reference images make the request multi-input\n        # with unverified cost - refuse BEFORE any paid call.\n        if len(req.product_paths) > 1 or req.reference_paths:\n            raise RuntimeError(\n                "live Magic Hour accepts exactly ONE input image: extra "\n                "product images and reference images are refused before "\n                "any paid call until multi-input pricing is verified. "\n                "Zero credits billed.")\n        reserved = reserve_budget(model, n)\n    backend = MagicHourBackend(model=model, live=live,\n                               receipt_path=run_dir / "mh-receipts.jsonl")\n    variants: list[VariantOut] = []\n    try:\n      for i in range(n):\n        seed = int(req.knob_values.get("seed", 0) or 0) + i * 101\n        full_prompt = build_detailed_prompt(\n            preset, product_description=req.product_description,\n            material=req.material, instruction=req.instruction,\n            reference_style=ref_fragment, knob_values=req.knob_values)\n        plan = plan_prompt(full_prompt, prompts.build_negative_prompt(),\n                           "magichour")\n        v = VariantOut(index=i, seed=seed, image="", provider="magichour",\n                       prompt_report=plan.report(), qa={})\n        out_path = run_dir / f"variant_{i}_magichour.png"\n        spend_recorded = False\n        try:\n            if not req.product_paths:\n                raise RuntimeError("no product images supplied")\n            src = Image.open(req.product_paths[0]).convert("RGB")\n            extras = [Image.open(p).convert("RGB")\n                      for p in list(req.product_paths[1:]) + list(req.reference_paths)]\n            edited = backend.edit(src, plan.full_text, plan.negative, seed,\n                                  aspect, extra_images=extras)\n            if live:\n                from .backends.magichour import MODEL_TABLE\n                listed = MODEL_TABLE[model]["credits"]\n                charge, abort_reason = _post_call_charge(backend, listed,\n                                                         model)\n                actual["n"] += charge\n                spend_recorded = True\n                if abort_reason:\n                    v.warnings.append(abort_reason)\n                    edited.save(out_path)\n                    v.image = out_path.name\n                    variants.append(v)\n                    break\n            edited.save(out_path)\n            v.image = out_path.name\n            v.warnings.append(\n                "whole-image AI edit: product pixels are regenerated, not "\n                "source-exact - verify label/logo fidelity before publishing")\n            report = qa.evaluate(src, out_path, route="pure_prompt",\n                                 template=preset.key, seed=seed)\n            v.qa = _qa_dict(report)\n            if backend.last_result and backend.last_result.get("submission"):\n                sub = backend.last_result["submission"]\n                v.warnings.append(\n                    f"project {sub.get(\'id\')} "\n                    f"credits_charged={sub.get(\'credits_charged\')}")\n        except Exception as e:\n            if live and not spend_recorded:\n                # A failed live call may still have been billed (transport\n                # error after POST, failed poll/download). Use the\n                # submission-accept observed charge when the provider\n                # reported one; the listed rate only as the last fallback.\n                from .backends.magichour import MODEL_TABLE\n                obs = _observed_charge(backend)\n                actual["n"] += (obs if obs is not None\n                                else MODEL_TABLE[model]["credits"])\n            v.error = f"{type(e).__name__}: {e}"\n        variants.append(v)\n    finally:\n        if live:\n            # An exception escaping the loop (anything not handled per\n            # variant, incl. BaseException) may have interrupted a call that\n            # was already accepted: keep the full reservation (unknown\n            # charge) instead of refunding it.\n            reconcile_budget(model, reserved,\n                             actual["n"] if sys.exc_info()[0] is None else None)\n    elapsed = time.time() - t0\n    manifest = {\n        "run_id": run_id, "created": time.strftime("%Y-%m-%d %H:%M:%S"),\n        "preset": preset.key, "provider_requested": req.provider,\n        "provider_used": "magichour", "mode": "whole-image edit",\n        "model": model, "live": backend.live,\n        "credits_note": ("DEVELOPER PLACEHOLDER MODE - synthetic gradient, not a real generation, zero credits" if not backend.live else\n                         "see mh-receipts.jsonl for charged amounts"),\n        "products": _product_entries(req.product_paths, members),\n        "warnings": _cutout_warnings(members),\n        "references": [Path(p).name for p in req.reference_paths],\n        "reference_style": (asdict(ref_style) if ref_style else None),\n        "instruction": req.instruction, "material": req.material,\n        "knobs": req.knob_values, "aspect": aspect.value,\n        "group_errors": group_errors,\n        "variants": [asdict(v) for v in variants],\n        "elapsed_s": round(elapsed, 2),\n        "contact_sheet": "contact_sheet.png",\n        "honesty": ("Magic Hour edits regenerate the whole image including "\n                    "the product; QA scores are automated proxies and human "\n                    "eye review is the acceptance gate per the brief."),\n    }\n    manifest_path = run_dir / "manifest.json"\n    manifest["runtime_versions"] = runtime_versions()\n    manifest_path.write_text(json.dumps(manifest, indent=2))\n    _save_contact_sheet(run_dir, variants, members)\n    return StudioResult(\n        run_id=run_id, run_dir=run_dir, preset=preset.key,\n        provider_requested=req.provider, variants=variants,\n        group_errors=group_errors,\n        reference_style=(asdict(ref_style) if ref_style else None),\n        manifest_path=manifest_path, elapsed_s=elapsed)\n\n\ndef _run_magichour_scene(req: StudioRequest, preset: Preset, aspect: Aspect,\n                         run_id: str, run_dir: Path, ref_style, ref_fragment: str,\n                         members: list[GroupMember], group_errors: list[str],\n                         t0: float, harmonize: bool = False) -> StudioResult:\n    """Scene-compose Magic Hour route: the paid model generates ONLY the\n    background scene; the verified product cutout is pasted back source-exact\n    with a grounded contact shadow. Product pixels are never the edit target.\n\n    harmonize=True restores the 13 Nov 2026 hybrid route: after the\n    source-exact paste, ONE extra paid edit pass blends lighting, shadow and\n    grading so the product sits naturally (product-fidelity prompt). The\n    harmonize pass CAN repaint product pixels, so its output is flagged for\n    mandatory eye review and region-drift QA.\n\n    Live calls require the key, RENDERELLO_MH_LIVE=1, and the whole batch\n    (variants x scene attempts, plus one harmonize call per variant when\n    enabled) must fit the aggregate budget - otherwise the run refuses with\n    zero bill. Generated scenes with long straight-edge intrusions (softbox\n    frames) or a suspected object in the placement area are rejected and\n    regenerated, capped by SCENE_MAX_ATTEMPTS; each attempt is a paid\n    generation and is recorded in the ledger whether it succeeds or fails.\n    """\n    from .backends.magichour import MagicHourBackend, MODEL_TABLE\n    from .scene_acceptance import long_straight_edges\n    template = preset.template\n    model = req.knob_values.get("mh_model", "seedream-v5-flash")\n    live = os.environ.get("RENDERELLO_MH_LIVE") == "1"\n    dev_mock = os.environ.get("RENDERELLO_MH_DEV_MOCK") == "1"\n    if not live and not dev_mock:\n        raise RuntimeError(\n            "Magic Hour is not configured for live generation on this "\n            "server (no API key / live flag). Configuration required - "\n            "zero credits were billed and no image was generated. "\n            "Developers: RENDERELLO_MH_DEV_MOCK=1 runs the explicit "\n            "placeholder test mode.")\n    max_attempts = scene_attempts_from(req.knob_values)\n    n = max(1, min(req.variants, 4))\n    reserved = 0\n    actual = {"n": 0}\n    if live:\n        # Worst case: every variant needs every scene attempt, plus one\n        # harmonize edit per variant on the hybrid route.\n        reserved = reserve_budget(model, n * (max_attempts + int(harmonize)))\n    backend = MagicHourBackend(model=model, live=live,\n                               receipt_path=run_dir / "mh-receipts.jsonl")\n    ok_members = [m for m in members if m.cutout is not None]\n    variants: list[VariantOut] = []\n    try:\n      for i in range(n):\n        seed = int(req.knob_values.get("seed", 0) or 0) + i * 101\n        full_prompt = build_detailed_prompt(\n            preset, product_description=req.product_description,\n            material=req.material, instruction=req.instruction,\n            reference_style=ref_fragment, knob_values=req.knob_values)\n        plan = plan_prompt(full_prompt, prompts.build_negative_prompt(),\n                           "magichour")\n        # The model receives ONLY the environment: a long, detailed,\n        # non-contradictory empty-set prompt. Product detail stays in\n        # plan.full_text for the report; it never reaches the scene model\n        # (the 20 Nov 2026 duplicate-product failure came from sending the\n        # product-bearing prompt with a contradictory \'no product\' tail).\n        from .empty_scene_prompt import build_empty_scene_prompt\n        try:\n            scene_prompt = build_empty_scene_prompt(\n                template, aspect_key=aspect.value,\n                instruction=req.instruction)\n        except ValueError:\n            # The instruction named the product: it cannot reach the scene\n            # model without re-creating the contradictory prompt. Drop it\n            # from the scene prompt and say so on the variant.\n            scene_prompt = build_empty_scene_prompt(\n                template, aspect_key=aspect.value)\n            _scene_note_dropped = True\n        else:\n            _scene_note_dropped = False\n        v = VariantOut(index=i, seed=seed, image="", provider="magichour",\n                       prompt_report=plan.report(), qa={})\n        out_path = run_dir / f"variant_{i}_magichour_scene.png"\n        attempt = {"n": 0}\n\n        def scene_fn(canvas, _prompt, _seed=seed, _attempt=attempt):\n            _attempt["n"] += 1\n            if dev_mock and not live:\n                return np.asarray(canvas)  # placeholder: gray canvas unchanged\n            try:\n                out = backend.edit(Image.fromarray(canvas),\n                                   prompt=scene_prompt,\n                                   negative_prompt="", seed=_seed + _attempt["n"],\n                                   aspect=aspect)\n            except Exception:\n                # Failed after a possible accept: count the submission-accept\n                # observed charge when reported, listed only as fallback.\n                obs = _observed_charge(backend)\n                actual["n"] += (obs if obs is not None\n                                else MODEL_TABLE[model]["credits"])\n                raise\n            charge, abort_reason = _post_call_charge(\n                backend, MODEL_TABLE[model]["credits"], model)\n            actual["n"] += charge\n            if abort_reason:\n                # Overcharge / balance mismatch / unreadable balance: the\n                # charge is counted above; stop this scene and the batch.\n                raise _OverchargeError(abort_reason)\n            return np.asarray(out.convert("RGB"))\n\n        try:\n            if not ok_members:\n                raise RuntimeError("no product cutout succeeded: " +\n                                   "; ".join(group_errors))\n            if len(ok_members) > 1:\n                raise RuntimeError(\n                    "scene-compose composites one product per run; "\n                    "multi-product groups are not supported on the Magic "\n                    "Hour route - run products separately")\n            co = ok_members[0].cutout\n            img, diag = compose.composite_with_scene(\n                co, preset.template, aspect, seed, scene_fn,\n                prompt=scene_prompt,\n                shadow_style=preset.template.shadow,\n                scene_auto_lift=True,\n                cache_dir=run_dir / "intermediates" / f"scene_layers_seed{seed}",\n                scene_accept=long_straight_edges,\n                scene_max_attempts=max_attempts,\n                conditioning="empty_guide")\n            if _scene_note_dropped:\n                v.warnings.append(\n                    "custom instruction named the product: excluded from the "\n                    "empty-scene prompt to keep it non-contradictory; "\n                    "environment-only wording was used")\n            if isinstance(img, np.ndarray):\n                Image.fromarray(img).save(out_path)\n            else:\n                img.save(out_path)\n            # Per-run native pixel audit: the placed opaque product pixels\n            # must equal the source RGB at the cutout bbox - measured, not\n            # asserted. Size change or mismatch lands in the manifest with\n            # its exact status; MATCH alone is not a quality acceptance.\n            from .native_pixel_audit import audit_native_pixels\n            _rgba = np.array(Image.open(co.rgba_path).convert("RGBA"))\n            _bx, _by, _bw, _bh = co.bbox\n            _px, _py, _pw, _ph = diag["placement"]\n            native_audit = audit_native_pixels(\n                np.array(Image.open(ok_members[0].path).convert("RGB")),\n                _rgba[..., 3], np.array(Image.open(out_path).convert("RGB")),\n                (int(_bx), int(_by), int(_bw), int(_bh)),\n                (int(_px), int(_py), int(_pw), int(_ph)))\n            if harmonize:\n                # Hybrid route: ONE extra paid pass over the correct\n                # composite to blend lighting/shadow/grading. Fidelity guard\n                # first; product repaint risk is disclosed and eye review is\n                # mandatory - this output is NOT source-exact by design.\n                harm_prompt = (\n                    prompts.FIDELITY_GUARD +\n                    " This is already the correct product cutout on a new "\n                    "background. Only harmonize lighting, shadow and color "\n                    "grading so the product sits naturally in the scene "\n                    f"({template.lighting}; {template.scene}). Do not move, "\n                    "resize, rotate or repaint the product.")\n                pre_harm_path = out_path\n                out_path = run_dir / f"variant_{i}_magichour_hybrid.png"\n                with Image.open(pre_harm_path) as _pre:\n                    _pre.convert("RGB").save(out_path)\n                try:\n                    edited = backend.edit(Image.open(out_path).convert("RGB"),\n                                          prompt=harm_prompt,\n                                          negative_prompt="", seed=seed + 500,\n                                          aspect=aspect)\n                except Exception:\n                    if live:\n                        obs = _observed_charge(backend)\n                        actual["n"] += (obs if obs is not None\n                                        else MODEL_TABLE[model]["credits"])\n                    raise\n                if live:\n                    charge, abort_reason = _post_call_charge(\n                        backend, MODEL_TABLE[model]["credits"], model)\n                    actual["n"] += charge\n                    if abort_reason:\n                        edited.convert("RGB").save(out_path)\n                        v.image = out_path.name\n                        v.warnings.append(abort_reason)\n                        variants.append(v)\n                        break\n                edited.convert("RGB").save(out_path)\n                # Post-harmonize audit: the harmonize pass may repaint the\n                # product, so the FINAL image gets its own comparison,\n                # recorded as drift - never inherited from the pre-pass MATCH.\n                post_harm_audit = audit_native_pixels(\n                    np.array(Image.open(ok_members[0].path).convert("RGB")),\n                    _rgba[..., 3],\n                    np.array(Image.open(out_path).convert("RGB")),\n                    (int(_bx), int(_by), int(_bw), int(_bh)),\n                    (int(_px), int(_py), int(_pw), int(_ph)))\n                post_harm_audit["interpretation"] = (\n                    "POST_HARMONIZE_DRIFT: mismatch expected - the model "\n                    "blends lighting/grading over the composite; MATCH on "\n                    "the final image is not claimed and not required. Eye "\n                    "review against the pre-harmonize composite is the gate.")\n                v.warnings.append(\n                    "hybrid harmonize pass: lighting/shadow/grading blended "\n                    "by the model over the source-exact composite; product "\n                    "pixels may have been repainted - EYE REVIEW REQUIRED, "\n                    "compare against the pre-harmonize composite "\n                    f"{pre_harm_path.name}")\n            v.image = out_path.name\n            v.warnings.append(\n                "product-preserving route: only the background is generated; "\n                "product pixels come from the source cutout (scaled to the "\n                "template placement, not resampled by the model)")\n            if diag.get("scene_acceptance", {}).get("intrusion"):\n                v.warnings.append(\n                    "scene screen still flags an intrusion after the configured call limit; "\n                    "no further paid scene call was made; DO NOT treat this as a quality pass")\n            if attempt["n"] > 1:\n                v.warnings.append(\n                    f"scene regenerated {attempt[\'n\']}x: straight-edge "\n                    "artifact rejection fired (softbox/frame intrusion)")\n            v.warnings.append(\n                "eye review required - QA heuristics are screens, not proof")\n            src_img = Image.open(ok_members[0].path).convert("RGB")\n            report = qa.evaluate(src_img, out_path, route="cutout_compose",\n                                 template=preset.key, seed=seed)\n            v.qa = _qa_dict(report)\n            v.qa["native_pixel_audit"] = native_audit\n            if harmonize:\n                v.qa["native_pixel_audit_purpose"] = "PRE_HARMONIZE_COMPOSITE"\n                v.qa["post_harmonize_audit"] = post_harm_audit\n            v.qa["scene_diag"] = {k: diag[k] for k in (\n                "ring_vs_far_luma_diff", "needs_eye_review",\n                "publication_status", "scene_acceptance", "conditioning",\n                "emptiness_screen") if k in diag}\n        except Exception as e:\n            if live and attempt["n"] == 0 and not isinstance(e, _OverchargeError):\n                # The call may have been billed despite the error.\n                actual["n"] += MODEL_TABLE[model]["credits"]\n            v.error = f"{type(e).__name__}: {e}"\n            if isinstance(e, _OverchargeError):\n                # Overcharge already counted at the call; stop the batch.\n                variants.append(v)\n                break\n        variants.append(v)\n    finally:\n        if live:\n            # An exception escaping the loop (anything not handled per\n            # variant, incl. BaseException) may have interrupted a call that\n            # was already accepted: keep the full reservation (unknown\n            # charge) instead of refunding it.\n            reconcile_budget(model, reserved,\n                             actual["n"] if sys.exc_info()[0] is None else None)\n    elapsed = time.time() - t0\n    manifest = {\n        "run_id": run_id, "created": time.strftime("%Y-%m-%d %H:%M:%S"),\n        "preset": preset.key, "provider_requested": req.provider,\n        "provider_used": "magichour",\n        "mode": ("hybrid (background-only generation + preserved cutout + "\n                 "harmonize pass; product repaint possible - eye review)"\n                 if harmonize else\n                 "scene-compose (background-only generation + preserved cutout)"),\n        "model": model, "live": backend.live,\n        "credits_note": ("DEVELOPER PLACEHOLDER MODE - no live calls, zero credits"\n                         if not backend.live else\n                         "see mh-receipts.jsonl and the spend ledger for charged amounts"),\n        "budget": budget_status(model),\n        "products": _product_entries(req.product_paths, members),\n        "warnings": _cutout_warnings(members),\n        "references": [Path(p).name for p in req.reference_paths],\n        "reference_style": (asdict(ref_style) if ref_style else None),\n        "instruction": req.instruction, "material": req.material,\n        "knobs": req.knob_values, "aspect": aspect.value,\n        "group_errors": group_errors,\n        "variants": [asdict(v) for v in variants],\n        "elapsed_s": round(elapsed, 2),\n        "contact_sheet": "contact_sheet.png",\n        "honesty": ("Scene-compose generates only the background and pastes "\n                    "the source product cutout back; placement rescales the "\n                    "cutout but the model never regenerates product pixels. "\n                    "QA scores are automated proxies - human eye review "\n                    "remains the acceptance gate per the brief."),\n    }\n    manifest_path = run_dir / "manifest.json"\n    manifest["runtime_versions"] = runtime_versions()\n    manifest_path.write_text(json.dumps(manifest, indent=2))\n    _save_contact_sheet(run_dir, variants, members)\n    return StudioResult(\n        run_id=run_id, run_dir=run_dir, preset=preset.key,\n        provider_requested=req.provider, variants=variants,\n        group_errors=group_errors,\n        reference_style=(asdict(ref_style) if ref_style else None),\n        manifest_path=manifest_path, elapsed_s=elapsed)\n'
# ---- embedded module: renderello/studio.py ----
path = base / 'renderello/studio.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Small supervised Magic Hour editor pilot. Never retries a POST.\n\nA reservation is durable before sending. An interrupted/ambiguous request blocks\nsubsequent spending until a human reviews provider state. This module does not\nverify owner authority: caller must have independently established that scope.\n"""\nfrom __future__ import annotations\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nfrom .credit_ledger import CreditLedger\nfrom .matched_trial import _canonical, _identifier, _text, read_json\n\nBASE = \'https://api.magichour.ai\'\nQUOTES_1K = {\'gpt-image-2.5-flare\':200,\'nano-banana-pro\':150,\'nano-banana-2.1\':50}\nPRICE_SOURCE = \'https://static.magichour.ai/_next/static/chunks/056hftg9bmzl8.js\'\n\ndef build_request(case_id, model, prompt, asset_path):\n    _identifier(case_id);_text(prompt,\'prompt\',15000)\n    if model not in QUOTES_1K:\n        raise ValueError(\'model outside verified pilot prices\')\n    if not isinstance(asset_path,str) or not asset_path.startswith(\'api-assets/\') or \'..\' in asset_path:\n        raise ValueError(\'verified uploaded source asset required\')\n    return dict(name=\'Renderello matched pilot \'+case_id+\' \'+model,\n                image_count=1,model=model,aspect_ratio=\'1:1\',resolution=\'1k\',\n                style=dict(prompt=prompt),assets=dict(image_file_paths=[asset_path]))\n\ndef submit_once(session, key, request, ledger, job, evidence_root, expected_account):\n    _identifier(job)\n    # No permissive defaults: compare exact current request with supported shape.\n    model=request.get(\'model\')\n    if set(request) != {\'name\',\'image_count\',\'model\',\'aspect_ratio\',\'resolution\',\'style\',\'assets\'}:\n        raise ValueError(\'unexpected request fields\')\n    if model not in QUOTES_1K or request[\'image_count\'] != 1 or type(request[\'image_count\']) is not int or request[\'resolution\']!=\'1k\' or request[\'aspect_ratio\']!=\'1:1\':\n        raise ValueError(\'unquoted configuration refused\')\n    _text(request[\'name\'],\'name\',200)\n    if set(request[\'style\'])!={\'prompt\'} or set(request[\'assets\'])!={\'image_file_paths\'} or len(request[\'assets\'][\'image_file_paths\'])!=1:\n        raise ValueError(\'unsupported request shape\')\n    build_request(job,model,request[\'style\'][\'prompt\'],request[\'assets\'][\'image_file_paths\'][0])\n    root=Path(evidence_root);root.mkdir(parents=True,exist_ok=True)\n    folder=root/job\n    if folder.exists():\n        raise FileExistsError(\'existing job: inspect evidence, never repeat submission\')\n    headers={\'Authorization\':\'Bearer \'+key}\n    account=session.get(BASE+\'/v1/account\',headers=headers,timeout=30)\n    account.raise_for_status();before=account.json()\n    if before.get(\'id\')!=expected_account:\n        raise ValueError(\'current account differs from authorized project account\')\n    quote=QUOTES_1K[model]\n    if type(before.get(\'credits\')) is not int or before[\'credits\']<quote:\n        raise ValueError(\'insufficient or unverified available credits\')\n    folder.mkdir()\n    (folder/\'account-before.json\').write_bytes(_canonical(before))\n    request_bytes=_canonical(request)\n    (folder/\'request.json\').write_bytes(request_bytes)\n    ledger.reserve(job,quote,cost_source=PRICE_SOURCE,account_id=expected_account,\n                   request_sha256=hashlib.sha256(request_bytes).hexdigest())\n    (folder/\'SUBMISSION_MAY_HAVE_OCCURRED\').write_text(\'Reservation recorded before exactly one POST. Never retry blindly.\\n\')\n    # Intentionally no exception recovery that could retry or erase a reservation.\n    response=session.post(BASE+\'/v1/ai-image-editor\',headers=headers,json=request,timeout=90)\n    (folder/\'response-status.txt\').write_text(str(response.status_code))\n    (folder/\'response.json\').write_text(response.text)\n    response.raise_for_status()\n    body=response.json()\n    if not isinstance(body.get(\'id\'),str) or not body[\'id\']:\n        raise ValueError(\'missing project ID; reservation remains unresolved\')\n    ledger.submitted(job,body[\'id\'])\n    charged=body.get(\'credits_charged\')\n    if type(charged) is not int or charged<0:\n        raise ValueError(\'missing actual charge; reservation remains unresolved\')\n    snapshot=ledger.settle(job,charged,receipt_reference=str(folder/\'response.json\'))\n    (folder/\'ledger-after.json\').write_bytes(_canonical(snapshot))\n    if charged!=quote or snapshot[\'over_cap\']:\n        raise ValueError(\'actual charge differs from verified quote; stop and review\')\n    return body\n'
# ---- embedded module: renderello/supervised_mh_pilot.py ----
path = base / 'renderello/supervised_mh_pilot.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Source-bound draft-template feasibility, not scene or photographic acceptance.\n\nProfiles are explicit readings of this draft library\'s requests, not vision.\nNo camera, source lighting, pose, geometry or transparent optics is invented.\n"""\nfrom dataclasses import dataclass, asdict\nfrom types import MappingProxyType\nimport numpy as np\nfrom .candidate_diagnostics import native_rgb, native_alpha, digest\nfrom .schemas import Aspect\nfrom .templates import TEMPLATES\n\n\n@dataclass(frozen=True)\nclass TemplateNeeds:\n    camera: str\n    support: str\n    light: str\n    reflection: bool = False\n    rotated_product: bool = False\n\n\nNEEDS = MappingProxyType({\n \'clean_studio\': TemplateNeeds(\'front\',\'floor\',\'front_left_soft\'),\n \'luxury_hero\': TemplateNeeds(\'low\',\'floor\',\'top_rim\',True),\n # Three-quarter is an azimuth convention here, not evidence of elevation.\n # The text also says product height, so do not silently choose an elevated\n # camera. Require disambiguation before source-locked preview.\n \'editorial_beauty\': TemplateNeeds(\'unknown\',\'platform\',\'left_soft\'),\n \'lifestyle_kitchen\': TemplateNeeds(\'front\',\'floor\',\'rear_left_window\'),\n \'natural_outdoor\': TemplateNeeds(\'low\',\'floor\',\'open_shade\'),\n \'minimal_tabletop\': TemplateNeeds(\'front\',\'platform\',\'overhead_soft\'),\n \'seasonal_holiday\': TemplateNeeds(\'elevated\',\'floor\',\'warm_mixed\'),\n \'social_ad\': TemplateNeeds(\'suspended\',\'none\',\'graphic_hard\',False,True),\n \'reflective_surface\': TemplateNeeds(\'low\',\'floor\',\'symmetric_side\',True),\n \'flat_lay\': TemplateNeeds(\'overhead\',\'flatlay\',\'overhead_soft\'),\n})\nassert set(NEEDS) == set(TEMPLATES)\n\n\n@dataclass(frozen=True)\nclass SourceSceneReview:\n    source_sha256: str\n    alpha_sha256: str\n    pose: str\n    material: str\n    framing: str\n    support: str\n    light: str\n    note: str\n    reviewed: bool = False\n\n    def __post_init__(self):\n        for h in (self.source_sha256,self.alpha_sha256):\n            if not isinstance(h,str) or len(h)!=64 or any(c not in \'0123456789abcdef\'for c in h):\n                raise ValueError(\'native source/alpha pixel SHA256 required\')\n        choices=dict(pose=(\'unknown\',\'front\',\'low\',\'elevated\',\'overhead\',\'suspended\'),\n            material=(\'unknown\',\'opaque\',\'transparent\',\'reflective\'),\n            framing=(\'unknown\',\'complete\',\'cropped\'),\n            support=(\'unknown\',\'floor\',\'platform\',\'flatlay\',\'none\'),\n            light=(\'unknown\',\'front_left_soft\',\'top_rim\',\'left_soft\',\'rear_left_window\',\n                   \'open_shade\',\'overhead_soft\',\'warm_mixed\',\'graphic_hard\',\'symmetric_side\'))\n        for n,allowed in choices.items():\n            if getattr(self,n) not in allowed:raise ValueError(\'unsupported source \'+n)\n        if type(self.reviewed)is not bool:raise ValueError(\'boolean review declaration required\')\n        if not isinstance(self.note,str)or not self.note.strip()or len(self.note)>2000:\n            raise ValueError(\'bounded source/lighting review note required\')\n\n\ndef plan_template(rgb,alpha,template,review,*,aspect=None):\n    native_rgb(rgb);native_alpha(alpha,rgb.shape[:2])\n    if alpha.size>1_000_000:raise ValueError(\'native1MP review budget; no downscale\')\n    if template not in NEEDS:raise ValueError(\'registered draft template required\')\n    if not isinstance(review,SourceSceneReview):raise ValueError(\'typed source/scene review required\')\n    if review.source_sha256!=digest(rgb)or review.alpha_sha256!=digest(alpha):\n        raise ValueError(\'review belongs to different source/alpha\')\n    selected=TEMPLATES[template].aspect if aspect is None else Aspect(aspect)\n    need=NEEDS[template];blockers=[];warnings=[]\n    if not review.reviewed:blockers.append(\'SOURCE_REVIEW_NOT_CONFIRMED\')\n    if review.material!=\'opaque\':blockers.append(\'OPAQUE_ONLY_NO_GLASS_OR_REFLECTIVE_RECOVERY\')\n    if review.framing!=\'complete\':blockers.append(\'COMPLETE_SOURCE_NOT_ESTABLISHED\')\n    if need.camera==\'unknown\':blockers.append(\'TEMPLATE_CAMERA_ELEVATION_NOT_DISAMBIGUATED\')\n    elif review.pose==\'unknown\':blockers.append(\'SOURCE_CAMERA_ELEVATION_UNREVIEWED\')\n    elif review.pose!=need.camera:blockers.append(\'CAMERA_MISMATCH_NO_VIEWPOINT_CHANGE\')\n    # Floor/platform contact types differ: a prior floor cannot certify support\n    # on a newly authored plinth. Its new footprint needs its own surface review.\n    if review.support!=need.support:blockers.append(\'SUPPORT_MISMATCH_REQUIRES_NEW_SURFACE_REVIEW\')\n    if review.light==\'unknown\':blockers.append(\'SOURCE_LIGHT_UNREVIEWED\')\n    elif review.light!=need.light:blockers.append(\'LIGHT_MISMATCH_NO_PRODUCT_RELIGHTING\')\n    if need.reflection:blockers.append(\'REQUESTED_REFLECTION_NOT_RECOVERED\')\n    if need.rotated_product:blockers.append(\'REQUESTED_ROTATION_NOT_SOURCE_LOCKED\')\n    support=alpha>0;hard=alpha>=128\n    if not hard.any():blockers.append(\'NO_HARD_FOREGROUND\')\n    if any(v.any()for v in(support[0],support[-1],support[:,0],support[:,-1])):\n        blockers.append(\'FRAME_TOUCHING_ALPHA_REQUIRES_REVIEW\')\n    w,h=selected.size;sh,sw=alpha.shape\n    # Entire native source is retained, including soft support. No hidden crop.\n    translation=((w-sw)//2,(h-sh)//2)if sw<=w and sh<=h else None\n    if translation is None:blockers.append(\'NATIVE_SOURCE_DOES_NOT_FIT_ASPECT_NO_RESIZE\')\n    if need.support!=\'none\':warnings.append(\'Actual target surface and every product contact footprint still need raster/visual review.\')\n    warnings.extend([\'Matching reviewed categories does not prove matched physical lighting or perspective.\',\n        \'Template is a draft, not an evaluated polished cover or publish-usable output.\',\n        \'No source alpha correction, edge RGB recovery, optical glass, product relighting or reflection is performed.\'])\n    return dict(schema=\'source-template-plan.v1\',template=template,template_name=TEMPLATES[template].name,\n        template_state=\'DRAFT_NOT_PHOTOGRAPHICALLY_ACCEPTED\',needs=asdict(need),source_review=asdict(review),\n        camera_requirement_text=TEMPLATES[template].camera,\n        camera_semantics=\'COARSE_ELEVATION_ONLY_NOT_AZIMUTH_FOCAL_LENGTH_OR_PERSPECTIVE_MATCH\',\n        review_is_caller_declaration=True,source_size=[sw,sh],aspect=selected.value,canvas=[w,h],\n        whole_source_translation=list(translation)if translation is not None else None,\n        blockers=blockers,warnings=warnings,eligible_for_reviewed_preview=not blockers,\n        source_rgb_locked=True,product_light_changed=False,source_resampled=False,\n        glass_recovered=False,reflection_recovered=False,photo_accepted=False,\n        actual_target_scene_verified=False,template_polish_verified=False,default_changed=False)\n'
# ---- embedded module: renderello/template_compatibility.py ----
path = base / 'renderello/template_compatibility.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Free boundary/contact control. Native source RGB and reviewed label cores stay locked.\nNot a physical renderer, automatic mask reviewer, or transparent-material recovery.\n"""\nfrom pathlib import Path\nimport hashlib,json\nimport numpy as np\nfrom PIL import Image,ImageDraw\nfrom scipy.ndimage import distance_transform_edt,gaussian_filter\n\n\ndef refine_alpha(alpha, protected_regions, *, trim=0.7, feather=1.4, contour=None):\n    a=np.asarray(alpha.convert(\'L\'))\n    if trim<0 or feather<=0:raise ValueError(\'invalid boundary width\')\n    mask=a>0\n    if contour is not None:\n        c=Image.new(\'L\',alpha.size);ImageDraw.Draw(c).polygon(contour,fill=255)\n        mask &= np.array(c)>0\n    dist=distance_transform_edt(mask)\n    result=np.minimum(a,np.clip((dist-trim)/feather,0,1)*255).astype(\'uint8\')\n    for box in protected_regions.values():\n        x1,y1,x2,y2=box\n        if not (0<=x1<x2<=a.shape[1] and 0<=y1<y2<=a.shape[0]):raise ValueError(\'invalid protected box\')\n        if np.any(a[y1:y2,x1:x2]!=255):raise ValueError(\'protected region not opaque\')\n        if np.any(~mask[y1:y2,x1:x2]):raise ValueError(\'contour crosses protected region\')\n        result[y1:y2,x1:x2]=255\n    return Image.fromarray(result)\n\n\ndef compose_template_contact(source,alpha,output,*,protected_regions,offset,canvas=(1024,1024),background=None):\n    output=Path(output)\n    if output.exists():raise ValueError(\'preserve previous output\')\n    rgb=np.array(source.convert(\'RGB\'));a=np.array(alpha.convert(\'L\'))\n    if rgb.shape[:2]!=a.shape or rgb.shape[0]*rgb.shape[1]>4000000:raise ValueError(\'invalid source/mask\')\n    if len(offset)!=2 or any(type(v)!=int for v in offset):raise ValueError(\'integer translation required\')\n    if not protected_regions:raise ValueError(\'reviewed label/texture regions required\')\n    dx,dy=offset;h,w=canvas[1],canvas[0];ys,xs=np.nonzero(a)\n    if not len(xs) or min(xs)+dx<0 or min(ys)+dy<0 or max(xs)+dx>=w or max(ys)+dy>=h:raise ValueError(\'empty or clipped source\')\n    yy,xx=np.mgrid[:h,:w];grad=np.clip((yy/h)*8+((xx-w*.4)/w)**2*8,0,15)\n    bg=np.stack([242-grad,239-grad,234-grad],axis=-1)\n    if background is not None:\n        if background.size!=canvas or background.mode!=\'RGB\':raise ValueError(\'background must match RGB canvas\')\n        bg=np.asarray(background).astype(float)\n    # Support comes from the lowest reviewed silhouette, not its box centre.\n    opaque=a>=128;bottom=np.full(a.shape[1],-1)\n    for x in np.where(opaque.any(axis=0))[0]:bottom[x]=np.where(opaque[:,x])[0][-1]\n    low=int(bottom.max());support_x=np.where(bottom>=low-5)[0]\n    cx=float(support_x.mean()+dx);cy=low+dy\n    support_width=max(float(np.ptp(support_x)),12)\n    # A compact contact term plus a soft directional tabletop shadow.\n    near=np.exp(-((xx-cx)/(support_width*.60+5))**2-((yy-cy-1)/3.8)**2)*.28\n    broad=np.exp(-((xx-cx-15)/(max(40,support_width*.85)))**2-((yy-cy-5)/12)**2)*.12\n    shade=np.clip(gaussian_filter(near,1)+gaussian_filter(broad,4),0,.40)\n    bg=bg*(1-shade[...,None])+np.array([70,65,60])*shade[...,None]\n    base=Image.fromarray(np.round(bg).astype(\'uint8\'));base.paste(source.convert(\'RGB\'),(dx,dy),alpha)\n    seen=np.array(base);checks=[]\n    for name,box in protected_regions.items():\n        x1,y1,x2,y2=box\n        if np.any(a[y1:y2,x1:x2]!=255):raise ValueError(\'protected region must be opaque\')\n        err=np.abs(seen[y1+dy:y2+dy,x1+dx:x2+dx].astype(int)-rgb[y1:y2,x1:x2].astype(int))\n        if np.any(err):raise RuntimeError(\'source label changed\')\n        checks.append({\'name\':name,\'source_box\':box,\'pixels\':int((x2-x1)*(y2-y1)),\'changed_pixels\':0,\'max_channel_error\':0})\n    oy,ox=np.nonzero(a==255);diff=seen[oy+dy,ox+dx].astype(int)-rgb[oy,ox].astype(int)\n    assert not np.any(diff)\n    output.mkdir(parents=True);base.save(output/\'output.png\');alpha.save(output/\'reviewed-alpha.png\');source.save(output/\'source-native.png\')\n    manifest={\'kind\':\'BOUNDARY_CONTACT_CONTROL_NOT_PRODUCTION_APPROVAL\',\'integer_translation\':[dx,dy],\'source_resampled\':False,\'source_rgb_repainted\':False,\'transparent_lid_recovered\':False,\'support_centre\':[cx,int(cy)],\'support_width\':support_width,\'protected_regions\':checks,\'opaque_pixels\':len(ox),\'opaque_changed_pixels\':0,\'output_sha256\':hashlib.sha256((output/\'output.png\').read_bytes()).hexdigest(),\'limitations\':[\'Reviewed alpha boundary may differ from original alpha.\',\'Contact shadow is a 2D support heuristic, not recovered scene geometry.\',\'Transparent lid RGB retains old-scene transmission and reflections. No measured clean plate exists.\',\'Unchanged source perspective may remain physically unsupported.\']}\n    (output/\'manifest.json\').write_text(json.dumps(manifest,indent=2));return manifest\n\n\ndef preview_reviewed(source,alpha_path,regions,root,*,reviewed=False,template=\'minimal_tabletop\',instruction=\'\',alternatives=2,material=\'unknown\',offset=None,aspect=None,scene_review=None):\n    from .colab_demo import validate_choices\n    from .compose import procedural_background\n    import shutil\n    spec=validate_choices(template,instruction,alternatives)\n    from .schemas import Aspect\n    # Preserve the legacy square default; template framing is explicit opt-in.\n    chosen_aspect=Aspect.SQUARE if aspect is None else (spec.aspect if aspect==\'template\' else Aspect(aspect))\n    canvas=chosen_aspect.size\n    if reviewed is not True:raise ValueError(\'Explicit alpha review required\')\n    if material not in (\'unknown\',\'opaque\',\'transparent\'):raise ValueError(\'Unknown material\')\n    if material==\'transparent\':raise ValueError(\'Reviewed RGB paste-back is not optical transparency recovery\')\n    source,alpha_path,root=Path(source),Path(alpha_path),Path(root)\n    if root.exists():raise ValueError(\'Preserve earlier results\')\n    im=Image.open(source).convert(\'RGB\');alpha=Image.open(alpha_path).convert(\'L\')\n    if alpha.size!=im.size:raise ValueError(\'Native source/alpha dimensions differ\')\n    if not isinstance(regions,dict) or not regions:raise ValueError(\'Protected regions required\')\n    for name,box in regions.items():\n        if not isinstance(name,str) or not name or len(box)!=4 or any(type(v)!=int for v in box):raise ValueError(\'Invalid protected region\')\n        x1,y1,x2,y2=box\n        if not (0<=x1<x2<=im.width and 0<=y1<y2<=im.height):raise ValueError(\'Protected region outside source\')\n        if np.any(np.asarray(alpha)[y1:y2,x1:x2]!=255):raise ValueError(\'Protected region must be opaque\')\n    compatibility=None\n    if scene_review is not None:\n        from .template_compatibility import plan_template\n        compatibility=plan_template(np.array(im),np.array(alpha),template,scene_review,aspect=chosen_aspect)\n        if not compatibility[\'eligible_for_reviewed_preview\']:\n            raise ValueError(\'Source/template compatibility: \'+\', \'.join(compatibility[\'blockers\']))\n    bb=alpha.getbbox()\n    if not bb:raise ValueError(\'Empty alpha\')\n    if offset is None:\n        offset=compatibility[\'whole_source_translation\'] if compatibility is not None else [(canvas[0]-(bb[2]-bb[0]))//2-bb[0],(canvas[1]-(bb[3]-bb[1]))//2-bb[1]]\n    if len(offset)!=2 or any(type(v)!=int for v in offset):raise ValueError(\'integer translation required\')\n    if compatibility is not None:\n        if min(offset)<0 or offset[0]+im.width>canvas[0] or offset[1]+im.height>canvas[1]:\n            raise ValueError(\'Compatibility plan retains whole native source; requested placement clips source frame\')\n        compatibility[\'executed_source_translation\']=list(offset)\n    if bb[0]+offset[0]<0 or bb[1]+offset[1]<0 or bb[2]+offset[0]>canvas[0] or bb[3]+offset[1]>canvas[1]:\n        raise ValueError(\'native reviewed source does not fit requested aspect; no silent resize or crop\')\n    root.mkdir();shutil.copyfile(source,root/(\'source-original\'+source.suffix));alpha.save(root/\'reviewed-alpha.png\');outputs=[];audits=[]\n    for i in range(alternatives):\n        bg=Image.fromarray(procedural_background(canvas,spec,i)).convert(\'RGB\')\n        out=root/(\'alternative-\'+str(i+1));audit=compose_template_contact(im,alpha,out,protected_regions=regions,offset=list(offset),canvas=canvas,background=bg);outputs.append(str(out/\'output.png\'));audits.append(audit)\n    manifest={\'kind\':\'REVIEWED_SOURCE_LOCK_TEMPLATE_CPU_CONTROL_NOT_PRODUCTION_ACCEPTANCE\',\'source_sha256\':hashlib.sha256(source.read_bytes()).hexdigest(),\'template\':template,\'aspect\':chosen_aspect.value,\'canvas\':list(canvas),\'template_aspect\':spec.aspect.value,\'aspect_selection\':\'legacy_square_default\' if aspect is None else \'explicit\',\'custom_instruction_recorded_not_generatively_executed\':instruction,\'manual_alpha_review_required\':True,\'transparent_optics_recovered\':False,\'audits\':audits,\'outputs\':outputs,\'paid_credits\':0,\'source_template_compatibility\':compatibility}\n    (root/\'manifest.json\').write_text(json.dumps(manifest,indent=2))\n    return {\'kind\':manifest[\'kind\'],\'outputs\':outputs,\'manifest\':str(root/\'manifest.json\'),\'root\':str(root)}\n'
# ---- embedded module: renderello/template_locked_preview.py ----
path = base / 'renderello/template_locked_preview.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Shoot template library. Draft set per the research plan Part D; the final set\nis chosen from evaluated quality (Phase 0/1 gate), so templates are data, not code\npaths - add or drop entries here without touching the pipeline.\n\nFragment discipline: scene/lighting/camera/mood describe ONLY the environment.\nNothing here may restyle the product itself; prompts.py adds the hard fidelity\nguard around these fragments.\n"""\nfrom __future__ import annotations\n\nfrom .schemas import Aspect, TemplateSpec\n\nTEMPLATES: dict[str, TemplateSpec] = {t.key: t for t in [\n    TemplateSpec(\n        key="clean_studio", name="Clean Studio / White Background",\n        category="catalog",\n        scene="seamless white studio backdrop, product centered on an invisible surface",\n        lighting="large soft key light front-left, fill card right, even shadowless wrap, 5500K neutral",\n        camera="85mm product lens, eye-level, product occupying 70% of frame height",\n        mood="clinical, honest, marketplace-ready",\n        aspect=Aspect.SQUARE,\n        background_hint="neutral white to very light gray vertical gradient",\n        shadow="soft faint contact shadow directly beneath product",\n    ),\n    TemplateSpec(\n        key="luxury_hero", name="Luxury Hero",\n        category="hero",\n        scene="polished black stone slab, dark charcoal void background, subtle rim glow",\n        lighting="single dramatic strip softbox from above-behind, warm 4300K accent kicker, deep controlled falloff",\n        camera="100mm macro product lens, slightly low hero angle, tight premium crop",\n        mood="premium, aspirational, high-contrast",\n        aspect=Aspect.PORTRAIT,\n        background_hint="near-black gradient with faint warm rim halo behind product",\n        shadow="crisp short contact shadow, slight reflection on slab",\n    ),\n    TemplateSpec(\n        key="editorial_beauty", name="Editorial Beauty / Skincare",\n        category="beauty",\n        scene="matte pastel stone pedestal, soft beige studio, one out-of-focus botanical sprig far behind",\n        lighting="diffused north-window softbox left, gentle top fill, 5000K, airy highlights",\n        camera="70mm, three-quarter angle at product height, generous negative space above",\n        mood="clean, editorial, calm skincare aesthetic",\n        aspect=Aspect.PORTRAIT,\n        background_hint="soft beige-pink seamless with gentle vignette",\n        shadow="long soft shadow falling right, low contrast",\n    ),\n    TemplateSpec(\n        key="lifestyle_kitchen", name="Lifestyle Kitchen",\n        category="lifestyle",\n        scene="light oak countertop, blurred bright kitchen interior behind, morning atmosphere",\n        lighting="warm window light from rear-left, white bounce card front, 4800K",\n        camera="50mm at counter height, shallow depth of field, background bokeh",\n        mood="warm, lived-in, appetizing",\n        aspect=Aspect.LANDSCAPE,\n        background_hint="warm cream tones, soft horizontal counter line, bokeh circles top third",\n        shadow="natural directional shadow to the right, diffused edge",\n    ),\n    TemplateSpec(\n        key="natural_outdoor", name="Natural Outdoor",\n        category="lifestyle",\n        scene="weathered stone or wood surface outdoors, softly blurred green foliage background",\n        lighting="open-shade daylight with dappled sun accents, 5600K, gentle sky fill",\n        camera="60mm at surface level, low horizon, background defocused",\n        mood="fresh, organic, eco",\n        aspect=Aspect.LANDSCAPE,\n        background_hint="muted green wash with soft dappled light patches",\n        shadow="soft dappled shadow consistent with foliage light",\n    ),\n    TemplateSpec(\n        key="minimal_tabletop", name="Minimalist Tabletop",\n        category="catalog",\n        scene="single matte neutral podium on a plain light-gray sweep, no props",\n        lighting="one large overhead softbox, minimal fill, 5200K, sculpted but simple",\n        camera="85mm, straight-on at product center, symmetrical framing",\n        mood="minimal, architectural, quiet",\n        aspect=Aspect.SQUARE,\n        background_hint="uniform light gray, very subtle floor-wall seam low in frame",\n        shadow="single soft ellipse under podium edge",\n    ),\n    TemplateSpec(\n        key="seasonal_holiday", name="Seasonal / Holiday",\n        category="campaign",\n        scene="festive tabletop with restrained seasonal accents (pine sprig, warm bokeh lights far behind)",\n        lighting="warm tungsten key 3800K with cool rim separation, gentle sparkle highlights",\n        camera="70mm, slightly elevated 15 degrees, product hero with accents at frame edges only",\n        mood="festive but premium, never cluttered",\n        aspect=Aspect.SQUARE,\n        background_hint="deep warm amber gradient with distant gold bokeh",\n        shadow="warm soft shadow, subtle festive glow at base",\n    ),\n    TemplateSpec(\n        key="social_ad", name="Social Ad",\n        category="campaign",\n        scene="bold single-color backdrop with graphic diagonal light band, product floating slightly above surface",\n        lighting="hard directional key for a defined graphic shadow, colored gel accent from rear, high contrast",\n        camera="50mm, dynamic slight tilt, product upper-center leaving copy space bottom-left",\n        mood="energetic, thumb-stopping, modern ad",\n        aspect=Aspect.PORTRAIT,\n        background_hint="saturated solid color with diagonal lighter band",\n        shadow="hard-edged graphic drop shadow offset down-right",\n    ),\n    TemplateSpec(\n        key="reflective_surface", name="Reflective Surface / Mirror",\n        category="hero",\n        scene="glossy reflective black acrylic surface, dark gradient studio behind",\n        lighting="two strip softboxes left/right for symmetric edge highlights, 5000K",\n        camera="90mm at surface level, product with clear mirror reflection below",\n        mood="sleek, tech, polished",\n        aspect=Aspect.SQUARE,\n        background_hint="dark neutral gradient, no visible horizon",\n        shadow="reflection fades with distance, minimal ambient shadow",\n    ),\n    TemplateSpec(\n        key="flat_lay", name="Flat Lay / Top-Down",\n        category="catalog",\n        scene="textured linen or paper surface viewed directly from above, minimal side props at corners",\n        lighting="large overhead diffused source, perfectly even, no directional bias, 5500K",\n        camera="top-down 90 degrees, product centered with even margins",\n        mood="tidy, graphic, editorial catalog",\n        aspect=Aspect.SQUARE,\n        background_hint="uniform light textile texture",\n        shadow="very soft ambient occlusion ring only",\n    ),\n]}\n\nassert len(TEMPLATES) >= 8, "problem statement requires at least 8 templates"\n\n\ndef get_template(key: str) -> TemplateSpec:\n    try:\n        return TEMPLATES[key]\n    except KeyError:\n        raise KeyError(f"unknown template \'{key}\'. available: {\', \'.join(sorted(TEMPLATES))}") from None\n\n\ndef all_templates() -> list[TemplateSpec]:\n    return list(TEMPLATES.values())\n'
# ---- embedded module: renderello/templates.py ----
path = base / 'renderello/templates.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Opt-in guarded native-mask research, not a production compositor or quality gate.\n\nOriginal RGB is copied without resizing. Alpha is estimated separately on bounded\nnative tiles. Seam agreement is a numerical consistency limit, not semantic truth,\nphotographic acceptance, or equivalence to a global matting solution.\n"""\nfrom pathlib import Path\nfrom io import BytesIO\nimport hashlib\nimport json\nimport shutil\nimport tempfile\n\nimport numpy as np\nfrom PIL import Image\nfrom .matting import guarded_file\n\nMAX_SOURCE_PIXELS = 4_000_000\nMAX_TILE_PIXELS = 1_000_000\nMAX_FILE_BYTES = 32_000_000\n\n\ndef tile_starts(length, tile, overlap):\n    if any(type(x) is not int for x in (length, tile, overlap)) or length < 1 or tile < 3 or not 1 <= overlap < tile:\n        raise ValueError(\'integer length, tile >=3 and overlap1..tile-1 required\')\n    if length <= tile:\n        return [0]\n    return sorted(set([*range(0, length-tile, tile-overlap), length-tile]))\n\n\ndef _read_image(path, mode):\n    p = Path(path)\n    if p.is_symlink() or not p.is_file() or p.stat().st_size > MAX_FILE_BYTES:\n        raise ValueError(\'bounded regular image file required\')\n    raw = p.read_bytes()\n    if len(raw) > MAX_FILE_BYTES:\n        raise ValueError(\'image bytes exceed bound\')\n    with Image.open(BytesIO(raw)) as im:\n        if im.mode != mode or getattr(im, \'n_frames\', 1) != 1:\n            raise ValueError(\'single-frame native \'+mode+\' image required; no conversion\')\n        if im.getexif().get(274, 1) != 1 or im.info.get(\'icc_profile\') or \'transparency\' in im.info:\n            raise ValueError(\'normalize orientation and color metadata explicitly first\')\n        if im.width*im.height > MAX_SOURCE_PIXELS:\n            raise ValueError(\'research source exceeds four million pixels; no resize\')\n        im.load()\n        return np.array(im), hashlib.sha256(raw).hexdigest()\n\n\ndef _weight(height, width, overlap):\n    def ramp(n):\n        return np.minimum(1., np.minimum((np.arange(n)+1)/overlap, (n-np.arange(n))/overlap))\n    return ramp(height)[:, None]*ramp(width)[None, :]\n\n\ndef stitch_tiles(shape, tiles, overlap, seam_limit):\n    """Compare EVERY intersecting tile before deciding whether alpha may be saved."""\n    if type(seam_limit) is not int or not 0 <= seam_limit <= 255:\n        raise ValueError(\'integer alpha seam limit0to255 required\')\n    if not isinstance(shape,(tuple,list)) or len(shape)!=2 or any(type(x) is not int or x<1 for x in shape) or shape[0]*shape[1]>MAX_SOURCE_PIXELS:\n        raise ValueError(\'bounded native shape required\')\n    if type(overlap) is not int or not 1<=overlap<=1000 or not 1<=len(tiles)<=64:\n        raise ValueError(\'bounded overlap and1to64 tiles required\')\n    h, w = shape\n    sums = np.zeros(shape, np.float64)\n    weights = np.zeros(shape, np.float64)\n    seams = []\n    for i, (x, y, a) in enumerate(tiles):\n        if not isinstance(a, np.ndarray) or a.dtype != np.uint8 or a.ndim != 2 or a.size > MAX_TILE_PIXELS:\n            raise ValueError(\'bounded uint8 tile required\')\n        if type(x) is not int or type(y) is not int:raise ValueError(\'integer tile origin required\')\n        th, tw = a.shape\n        if min(th, tw) < 1 or x < 0 or y < 0 or x+tw > w or y+th > h:\n            raise ValueError(\'tile outside native source\')\n        for j, (px, py, pa) in enumerate(tiles[:i]):\n            left, top = max(x, px), max(y, py)\n            right, bottom = min(x+tw, px+pa.shape[1]), min(y+th, py+pa.shape[0])\n            if left >= right or top >= bottom:\n                continue\n            diff = np.abs(a[top-y:bottom-y, left-x:right-x].astype(np.int16)-pa[top-py:bottom-py, left-px:right-px].astype(np.int16))\n            seams.append(dict(tiles=[j,i], box=[left,top,right,bottom], max_difference=int(diff.max()), differing_pixels=int(np.count_nonzero(diff)), over_limit=int(np.count_nonzero(diff>seam_limit)), mean_difference=float(diff.mean())))\n        weight = _weight(th, tw, overlap)\n        sums[y:y+th,x:x+tw] += a*weight\n        weights[y:y+th,x:x+tw] += weight\n    if not np.all(weights > 0):\n        raise ValueError(\'native source has uncovered pixels\')\n    accepted = not any(s[\'over_limit\'] for s in seams)\n    return (np.rint(sums/weights).astype(np.uint8) if accepted else None), seams\n\n\ndef run(source, initial_alpha, out, *, tile_size=850, overlap=100, seam_limit=16):\n    """Retain failed tile evidence but NEVER save a stitched mask on seam refusal."""\n    out = Path(out)\n    if out.exists() or out.is_symlink():\n        raise FileExistsError(\'preserve older research output\')\n    if type(tile_size) is not int or not 3 <= tile_size <= 1000 or tile_size*tile_size > MAX_TILE_PIXELS:\n        raise ValueError(\'tile size3to1000 required; million-pixel guard unchanged\')\n    if type(overlap) is not int or not 1 <= overlap < tile_size:\n        raise ValueError(\'overlap1..tile-1 required\')\n    if type(seam_limit) is not int or not 0 <= seam_limit <= 255:\n        raise ValueError(\'integer alpha seam limit0to255 required\')\n    rgb, source_hash = _read_image(source, \'RGB\')\n    alpha, alpha_hash = _read_image(initial_alpha, \'L\')\n    if rgb.shape[:2] != alpha.shape or alpha.size < 9:\n        raise ValueError(\'native source and alpha must align\')\n    h, w = alpha.shape\n    xs, ys = tile_starts(w,tile_size,overlap), tile_starts(h,tile_size,overlap)\n    if len(xs)*len(ys) > 64:\n        raise ValueError(\'research capped at64 tiles\')\n    out.parent.mkdir(parents=True,exist_ok=True)\n    stage = Path(tempfile.mkdtemp(prefix=\'.\'+out.name+\'-\',dir=out.parent))\n    report = dict(status=\'INCOMPLETE\', source_sha256=source_hash, initial_alpha_sha256=alpha_hash, source_shape=list(rgb.shape), source_rgb_resampled=False, tile_size=tile_size, overlap=overlap, seam_limit=seam_limit, quality=\'NOT_EVALUATED\', semantics=\'NOT_VERIFIED\', global_solver_equivalence=False, defaults_changed=False, tiles=[])\n    try:\n        Image.fromarray(rgb).save(stage/\'source.png\')\n        Image.fromarray(alpha).save(stage/\'initial-alpha.png\')\n        tiles=[]\n        for y in ys:\n            for x in xs:\n                i=len(tiles); a=alpha[y:y+tile_size,x:x+tile_size]; c=rgb[y:y+tile_size,x:x+tile_size]\n                sp,ap=stage/f\'tile{i}-source.png\',stage/f\'tile{i}-alpha.png\'\n                Image.fromarray(c).save(sp);Image.fromarray(a).save(ap)\n                guarded_file(sp,ap,stage/f\'tile{i}-cf\',method=\'cf\',radius=3,edge_colors=False)\n                with Image.open(stage/f\'tile{i}-cf/source.png\') as im:\n                    if not np.array_equal(np.array(im),c):raise ValueError(\'tile source RGB changed\')\n                with Image.open(stage/f\'tile{i}-cf/refined-alpha.png\') as im:\n                    if im.mode!=\'L\' or im.size!=(a.shape[1],a.shape[0]):raise ValueError(\'tile alpha shape/mode changed\')\n                    refined=np.array(im)\n                with Image.open(stage/f\'tile{i}-cf/trimap.png\') as im:tri=np.array(im)\n                if not np.all(refined[tri==0]==0) or not np.all(refined[tri==255]==255):raise ValueError(\'protected trimap core changed\')\n                tiles.append((x,y,refined));report[\'tiles\'].append(dict(index=i,origin=[x,y],shape=list(a.shape),pixels=a.size,source_rgb_exact=True,trimap_cores_exact=True))\n        stitched,seams=stitch_tiles(alpha.shape,tiles,overlap,seam_limit);report[\'seams\']=seams\n        report[\'status\']=\'SEAM_LIMIT_PASSED_NOT_QUALITY_ACCEPTED\' if stitched is not None else \'REJECTED_SEAM_LIMIT\'\n        if stitched is not None:\n            Image.fromarray(stitched).save(stage/\'stitched-alpha.png\')\n            report[\'stitched_alpha_sha256\']=hashlib.sha256((stage/\'stitched-alpha.png\').read_bytes()).hexdigest()\n        (stage/\'research-manifest.json\').write_text(json.dumps(report,indent=2,allow_nan=False))\n        # mkdir, not replace: never overwrite output created during processing.\n        out.mkdir()\n        for p in stage.iterdir():p.rename(out/p.name)\n        return report\n    finally:\n        shutil.rmtree(stage,ignore_errors=True)\n\n\ndef main(argv=None):\n    import argparse\n    p=argparse.ArgumentParser(description=__doc__)\n    for flag in (\'source\',\'alpha\',\'out\'):p.add_argument(\'--\'+flag,required=True,type=Path)\n    p.add_argument(\'--tile-size\',type=int,default=850);p.add_argument(\'--overlap\',type=int,default=100);p.add_argument(\'--seam-limit\',type=int,default=16)\n    a=p.parse_args(argv);r=run(a.source,a.alpha,a.out,tile_size=a.tile_size,overlap=a.overlap,seam_limit=a.seam_limit);print(json.dumps(r,indent=2));return 0 if r[\'status\'].startswith(\'SEAM_LIMIT_PASSED\') else 2\n\nif __name__==\'__main__\':raise SystemExit(main())\n'
# ---- embedded module: renderello/tiled_matting_research.py ----
path = base / 'renderello/tiled_matting_research.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Minimal local web UI for the problem statement\'s working flow (section 4):\nupload real product photo -> select template WITH VISUAL PREVIEW -> optional\nmultiline custom instruction -> generate 2-4 alternatives -> compare/download or\nedit/regenerate, with square/portrait/landscape output where the pipeline\npermits. Stdlib only (no new deps), binds 127.0.0.1, serves run artifacts\nfrom the out dir. This is a prototype UI for the owner\'s machine, not a\nhosted product.\n\nHonesty labels: result scores are AUTOMATED STRUCTURAL metrics (pixel-level\nfidelity checks), not human ratings and not publish-ready judgments. Template\npreviews are demo renders on a synthetic placeholder product until the real\n20+ source-image benchmark supplies genuine cover examples.\n\nRun:  python -m renderello.webui --port 8000 --out runs/webui\n"""\nfrom __future__ import annotations\n\nimport zipfile\nimport argparse\nimport html\nimport io\nimport json\nimport re\nimport secrets\nimport threading\nimport uuid\nfrom http.server import BaseHTTPRequestHandler, ThreadingHTTPServer\nfrom pathlib import Path\nfrom urllib.parse import parse_qs, urlparse\n\nfrom .backends.base import StubEditor\nfrom .config import RunConfig\nfrom .pipeline import run_pipeline\nfrom .prompts import validate_instruction\nfrom .gallery import _source_caption\nfrom .schemas import Aspect, EditRequest, Route\nfrom .templates import all_templates, get_template\nfrom .source_contract import SourceContract\n\n_PAGE = """<!doctype html><html><head><meta charset="utf-8">\n<meta name="viewport" content="width=device-width,initial-scale=1">\n<title>Renderello</title>\n<style>\n*{{box-sizing:border-box}}\nbody{{font-family:system-ui,sans-serif;max-width:1100px;margin:24px auto;\npadding:0 14px;color:#1c1c1e;overflow-x:hidden}}\nlabel{{display:block;margin-top:12px;font-weight:600}}\nselect,input[type=text],textarea{{width:100%;padding:8px;margin-top:4px}}\ntextarea{{min-height:130px;resize:vertical;font:inherit;line-height:1.45}}\n.tplgrid{{display:grid;grid-template-columns:repeat(auto-fill,minmax(150px,1fr));\ngap:10px;margin-top:6px}}\n.tpl{{border:2px solid #ccc;border-radius:8px;padding:8px;cursor:pointer;\nfont-weight:400;font-size:13px;background:#fff}}\n.tpl input{{margin-right:6px}}\n.tpl img{{width:100%;border-radius:5px;display:block;margin-bottom:6px;\nbackground:#f2f2f4;min-height:60px}}\n.tpl.sel{{border-color:#0a6cff}}\n.card{{border:1px solid #ccc;border-radius:8px;padding:14px;margin-top:20px}}\n.grid{{display:grid;grid-template-columns:repeat(auto-fill,minmax(300px,1fr));\ngap:14px}}\n.shot img{{width:100%;border-radius:6px;display:block;border:1px solid #e2e2e4}}\n.meta{{overflow-wrap:anywhere;font-size:12px;color:#555;white-space:pre-wrap;margin-top:6px}}\n.bad{{color:#a00}}\na.dl{{display:inline-block;margin-top:8px;padding:6px 12px;background:#0a6cff;\ncolor:#fff;border-radius:6px;text-decoration:none;font-size:13px}}\n.note{{font-size:12px;color:#777;margin-top:4px}}\nbutton{{padding:10px 22px;font-size:15px;background:#0a6cff;color:#fff;\nborder:0;border-radius:8px;cursor:pointer}}\n</style>\n<script>\nfunction pick(id){{document.querySelectorAll(\'.tpl\').forEach(e=>e.classList.remove(\'sel\'));\ndocument.getElementById(\'l\'+id).classList.add(\'sel\');}}\n</script></head><body>\n<h1>Renderello / Shotelier - product photography prototype</h1>\n<p class="meta">{backend_status}</p>\n<p><a href="/mask-prompts">Prepare reviewed mask boxes / points (optional)</a></p>\n<form method="post" action="/run" enctype="multipart/form-data">\n<label>Product photo <input type="file" name="photo" accept="image/*" required></label>\n<label><input type="checkbox" name="allow_colour_conversion" value="yes">Allow embedded ICC profile conversion to sRGB (source and reference). Original files are retained. Normalized RGB may differ; no colour-accuracy guarantee.</label>\n<label>Template (demo previews on a synthetic placeholder product; real cover\nexamples come from the benchmark)</label>\n<div class="tplgrid">{options}</div>\n<label>Custom instruction (scene/lighting only, optional)\n<textarea name="instruction" maxlength="16000" rows="6"\n placeholder="Describe the surface, scene, light, camera and placement in detail"></textarea></label>\n<p class="note">Up to 16,000 characters. Full text is retained without silent cuts. Image models may still ignore details. Procedural and mock routes do not interpret every clause.</p>\n<label><input type="checkbox" name="use_embedded_alpha" value="yes">Use the uploaded source\'s existing alpha for cutout/scene composition. No segmentation model; alpha quality and material semantics remain unverified. Cannot combine with a separate alpha or RGBA result.</label>\n<label>Optional alpha mask (8-bit grayscale PNG, same dimensions as source; white product, black background)<input type="file" name="mask" accept="image/png"></label>\n<label>Optional source-RGB-locked RGBA result (PNG)\n<input type="file" name="rgba_result" accept="image/png"></label>\n<p class="note">Use with a normalized RGB PNG source and cutout/scene compose only. Every imported RGB pixel must match the source. Alpha is retained for review, not certified. Do not supply a separate alpha mask as well. No model is called for this import.</p>\n<label>Reference/backdrop photo (optional; used as the scene backdrop,\n cover-fitted to the chosen aspect)\n<input type="file" name="reference" accept="image/*"></label>\n<label>Output aspect\n<select name="aspect">\n<option value="">Template default</option>\n<option value="1:1">Square 1:1</option>\n<option value="4:5">Portrait 4:5</option>\n<option value="3:2">Landscape 3:2</option></select></label>\n<label>Alternatives <select name="variants">\n<option>2</option><option>3</option><option>4</option></select></label>\n<fieldset><legend>Source review (your declarations, not automatic image analysis)</legend>\n<label>Camera / pose <select name="source_pose"><option value="unknown">Not reviewed</option><option value="front">Front / low view</option><option value="elevated">Elevated view</option><option value="overhead">Overhead</option><option value="suspended">Suspended / floating</option></select></label>\n<label>Material <select name="source_material"><option value="unknown">Not reviewed</option><option value="opaque">Opaque</option><option value="transparent">Transparent / translucent</option></select></label>\n<label>Product framing <select name="source_framing"><option value="unknown">Not reviewed</option><option value="complete">Whole product present</option><option value="cropped">Cropped / missing parts</option></select></label>\n<label><input type="checkbox" name="require_complete" value="yes"> Require a complete product (block detected frame cuts)</label>\n<p class="note">Source-exact paste cannot change the camera angle or restore missing parts. Transparent products need a separate transmission workflow. Genuine fine-print preservation is not verified.</p></fieldset>\n<label>Route <select name="route">\n<option value="cutout_compose">cutout + compose (source pixels; mask review required)</option>\n<option value="pure_prompt">pure prompt (stub editor unless GPU build)</option>\n<option value="hybrid">hybrid</option>\n<option value="scene_compose">scene compose (MOCK scene unless GPU relight is plugged in)</option></select></label>\n<p><button type="submit">Generate</button></p>\n</form>{result}</body></html>"""\n\n_RESULT_HEAD = (\'<div class="card"><h2>Results</h2>\'\n                \'<p class="note">Prototype outputs. Scores below are automated \'\n                \'structural fidelity metrics only - not human ratings and not \'\n                \'publish-ready judgments.</p><div class="grid">\')\n\n_RERUN = """</div></div>\n<div class="card"><h2>Edit / regenerate</h2>\n<form method="post" action="/rerun">\n<input type="hidden" name="job" value="{job}">\n<label><input type="checkbox" name="use_embedded_alpha" value="yes">Use source existing alpha (composition only; mask/material unverified).</label>\n<label><input type="checkbox" name="allow_colour_conversion" value="yes">Allow embedded ICC profile conversion to sRGB (source/reference). Original files retained; normalized RGB may change.</label>\n<label>Template <select name="template">{options}</select></label>\n<label>Custom instruction\n<textarea name="instruction" maxlength="16000" rows="6">{instruction}</textarea></label>\n<p class="note">Up to 16,000 characters, retained without silent cuts. Model obedience is not guaranteed.</p>\n<label>Output aspect\n<select name="aspect">\n<option value="">Template default</option>\n<option value="1:1"{a11}>Square 1:1</option>\n<option value="4:5"{a45}>Portrait 4:5</option>\n<option value="3:2"{a32}>Landscape 3:2</option></select></label>\n<label>Alternatives <select name="variants">\n<option>2</option><option>3</option><option>4</option></select></label>\n<fieldset><legend>Source review (your declarations, not automatic image analysis)</legend>\n<label>Camera / pose <select name="source_pose"><option value="unknown">Not reviewed</option><option value="front">Front / low view</option><option value="elevated">Elevated view</option><option value="overhead">Overhead</option><option value="suspended">Suspended / floating</option></select></label>\n<label>Material <select name="source_material"><option value="unknown">Not reviewed</option><option value="opaque">Opaque</option><option value="transparent">Transparent / translucent</option></select></label>\n<label>Product framing <select name="source_framing"><option value="unknown">Not reviewed</option><option value="complete">Whole product present</option><option value="cropped">Cropped / missing parts</option></select></label>\n<label><input type="checkbox" name="require_complete" value="yes"> Require a complete product (block detected frame cuts)</label>\n<p class="note">Source-exact paste cannot change the camera angle or restore missing parts. Transparent products need a separate transmission workflow. Genuine fine-print preservation is not verified.</p></fieldset>\n<label>Route <select name="route">\n<option value="cutout_compose">cutout + compose (source pixels; mask review required)</option>\n<option value="pure_prompt">pure prompt (stub editor unless GPU build)</option>\n<option value="hybrid">hybrid</option>\n<option value="scene_compose">scene compose (MOCK scene unless GPU relight is plugged in)</option></select></label>\n<p><button type="submit">Regenerate with new seeds</button></p>\n</form></div>"""\n\n\ndef _parse_multipart(body: bytes, content_type: str) -> dict[str, tuple[str, bytes]]:\n    """Small strict multipart parser (stdlib cgi is deprecated). Returns\n    field -> (filename or \'\', data)."""\n    from email.parser import BytesParser\n    from email.policy import default\n    if not content_type.lower().startswith("multipart/form-data"):\n        raise ValueError("multipart/form-data required")\n    message = BytesParser(policy=default).parsebytes(\n        ("Content-Type: " + content_type + "\\r\\nMIME-Version: 1.0\\r\\n\\r\\n").encode() + body)\n    if not message.is_multipart():\n        raise ValueError("multipart boundary missing or invalid")\n    fields = {}\n    for part in message.iter_parts():\n        name = part.get_param("name", header="content-disposition")\n        if name:\n            fields[name] = (part.get_filename() or "", part.get_payload(decode=True) or b"")\n    return fields\n\n\ndef _template_cards(selected: str = "") -> str:\n    cards = []\n    for i, t in enumerate(all_templates()):\n        sel = " sel" if (t.key == selected or (not selected and i == 0)) else ""\n        chk = " checked" if (t.key == selected or (not selected and i == 0)) else ""\n        cards.append(\n            f\'<label class="tpl{sel}" id="l{t.key}">\'\n            f\'<img src="/preview/{t.key}.png" loading="lazy" alt="" \'\n            f\'onerror="this.style.display=\\\'none\\\'">\'\n            f\'<input type="radio" name="template" value="{t.key}"{chk} \'\n            f\'onclick="pick(\\\'{t.key}\\\')">{html.escape(t.name)}\'\n            f\'<div class="note">{html.escape(t.category)} - {t.aspect.value}</div>\'\n            f\'</label>\')\n    return "".join(cards)\n\n\ndef _aspect_from(value: str) -> Aspect | None:\n    return Aspect(value) if value in (a.value for a in Aspect) else None\n\n\ndef _render_previews(out_root: Path) -> None:\n    """Render one demo preview per template on a synthetic placeholder product.\n    Prepared before server start with known alpha, no segmentation/model call.\n    These are NOT the benchmark cover examples."""\n    from PIL import Image, ImageDraw\n    from .cutout import _save_cutout\n    import numpy as np\n    from .compose import composite\n    prev = out_root / "_previews"\n    prev.mkdir(parents=True, exist_ok=True)\n    if all((prev / f"{t.key}.png").exists() for t in all_templates()):\n        return\n    img = Image.new("RGB", (480, 480), (170, 172, 174))\n    d = ImageDraw.Draw(img)\n    d.rectangle([0, 384, 480, 480], fill=(150, 152, 154))\n    d.rectangle([190, 130, 290, 380], fill=(36, 130, 140))\n    d.rectangle([210, 96, 270, 130], fill=(28, 28, 30))\n    d.rectangle([200, 210, 280, 330], fill=(246, 246, 244), outline=(60, 60, 60))\n    src = prev / "_demo_source.png"\n    img.save(src)\n    # This is our synthetic placeholder, so its exact alpha is known. Do not\n    # launch a model or GrabCut for preview generation or race the first page.\n    alpha = Image.new("L", img.size, 0)\n    ad = ImageDraw.Draw(alpha)\n    ad.rectangle([190,130,290,380],fill=255)\n    ad.rectangle([210,96,270,130],fill=255)\n    co = _save_cutout(np.array(img), np.array(alpha), prev / "_cutout", "demo", "synthetic_known_alpha")\n    for t in all_templates():\n        out = prev / f"{t.key}.png"\n        if out.exists():\n            continue\n        try:\n            composite(co, t, t.aspect, seed=7,\n                      reflection="reflect" in t.scene.lower()).save(out)\n        except Exception:\n            pass  # missing preview is cosmetic only\n\n\ndef _mock_scene_fn(gray, prompt):\n    """MOCK scene for the demo UI (labeled): vertical warm gradient. A GPU build\n    replaces this with IC-Light (iclight_gpu.ICLight.relight). Never real relighting."""\n    import numpy as np\n    h, w = gray.shape[:2]\n    y = np.linspace(0, 1, h, dtype=np.float32)[:, None, None]\n    top, bot = np.array([205, 200, 192], np.float32), np.array([120, 105, 92], np.float32)\n    return (top * (1 - y) + bot * y).repeat(w, axis=1).astype(np.uint8)\n\n\n_mock_scene_fn.mock_scene = True\n\nclass _Handler(BaseHTTPRequestHandler):\n    out_root: Path = Path("runs/webui")\n\n    def _send(self, code: int, body: bytes, ctype: str = "text/html"):\n        self.send_response(code)\n        self.send_header("Content-Type", ctype)\n        self.send_header("Content-Length", str(len(body)))\n        self.end_headers()\n        self.wfile.write(body)\n\n    def _serve_file(self, rel: str):\n        root = self.out_root.resolve()\n        target = (root / rel).resolve()\n        if root not in target.parents and target != root:\n            self._send(403, b"forbidden", "text/plain")\n            return\n        if target.is_file():\n            ctype = {".png": "image/png", ".jpg": "image/jpeg",\n                     ".html": "text/html"}.get(target.suffix, "text/plain")\n            self._send(200, target.read_bytes(), ctype)\n            return\n        self._send(404, b"not found", "text/plain")\n\n    def do_GET(self):\n        path = urlparse(self.path).path\n        if path == "/reviewed-status":\n            checkpoint=getattr(self,"reviewed_checkpoint",None)\n            self._send(200,json.dumps({"configured": bool(checkpoint and checkpoint.is_file())}).encode(),"application/json")\n            return\n        if path == "/mask-prompts":\n            from .prompt_editor import PAGE\n            self._send(200,PAGE.encode())\n            return\n        if path == "/":\n            self._send(200, _PAGE.format(options=_template_cards(),\n                                         result="", backend_status=html.escape(getattr(self,"backend_status","Offline stub editor; no live generation"))).encode())\n            return\n        if path.startswith("/preview/"):\n            key = re.sub(r"[^a-z0-9_]", "", path[len("/preview/"):].removesuffix(".png"))\n            self._serve_file(f"_previews/{key}.png")\n            return\n        if path.startswith("/file/"):\n            self._serve_file(path[len("/file/"):])\n            return\n        self._send(404, b"not found", "text/plain")\n\n    def _run_job(self, src: Path, template_key: str, instruction: str,\n                 aspect: Aspect | None, variants: int, route: Route,\n                 ref_upload: tuple[str, bytes] | None = None,\n                 mask_upload: tuple[str, bytes] | None = None,\n                 source_contract: SourceContract | None = None, require_complete: bool = False,\n                 rgba_upload: tuple[str, bytes] | None = None, allow_colour_conversion: bool = False, use_embedded_alpha: bool = False):\n        if use_embedded_alpha and (mask_upload or rgba_upload):\n            raise ValueError("choose embedded alpha, separate mask or RGBA result, not a combination")\n        if variants not in (2,3,4):\n            raise ValueError("alternatives must be 2, 3 or 4")\n        job = self.out_root / uuid.uuid4().hex[:12]\n        job.mkdir(parents=True, exist_ok=True)\n        dst = job / f"source{src.suffix}"\n        dst.write_bytes(src.read_bytes())\n        contract = source_contract or SourceContract()\n        (job / "source-review.json").write_text(json.dumps({"pose":contract.pose,"material":contract.material,\n            "framing":contract.framing,"require_complete":require_complete}))\n        tpl = get_template(template_key)\n        ref_dst = None\n        if ref_upload and ref_upload[1]:\n            ref_dst = job / f"reference{Path(ref_upload[0] or \'ref.png\').suffix or \'.png\'}"\n            ref_dst.write_bytes(ref_upload[1])\n        mask_dst = None\n        if mask_upload and mask_upload[1]:\n            mask_dst = job / "mask.png"\n            mask_dst.write_bytes(mask_upload[1])\n        if rgba_upload and rgba_upload[1]:\n            if mask_upload and mask_upload[1]:raise ValueError("Choose RGBA result or separate alpha mask, not both")\n            if route not in (Route.CUTOUT_COMPOSE,Route.SCENE_COMPOSE):raise ValueError("RGBA import is only for cutout/scene compose, not product regeneration")\n            from .reviewed_edge_cli import read_image,sha\n            from .source_locked_rgba_import import load_source_locked_rgba\n            from PIL import Image\n            # This opt-in path requires already normalized native RGB PNG,\n            # avoiding a second normalization convention in the legacy pipeline.\n            rgb,source_record=read_image(dst,sha(dst.read_bytes()),\'source\')\n            imported=job/\'external-rgba.png\';imported.write_bytes(rgba_upload[1])\n            alpha,evidence=load_source_locked_rgba(imported,rgb,sha(rgba_upload[1]))\n            mask_dst=job/\'mask.png\';Image.fromarray(alpha).save(mask_dst)\n            evidence.update(source_file_sha256=source_record[\'file_sha256\'],source_normalized_by_import=False,alpha_semantics=\'NOT_VERIFIED\',publication_status=\'NOT_EVALUATED\')\n            (job/\'rgba-import.json\').write_text(json.dumps(evidence,indent=2))\n        cfg = RunConfig(seed=secrets.randbelow(10_000), variants=variants,\n                        out_root=job)\n        if getattr(self,\'live_editor\',False) and route is Route.HYBRID:\n            raise ValueError(\'Magic Hour mask-constrained hybrid is unsupported; use pure_prompt or an explicit local compose route\')\n        editor = self.editor_factory(job) if route in (Route.PURE_PROMPT, Route.HYBRID) else None\n        # An uploaded reference is the user\'s chosen backdrop: prefer it over the\n        # mock scene so scene_compose composites onto THEIR photo (GPU-free path).\n        scene_fn = _mock_scene_fn if (route is Route.SCENE_COMPOSE and ref_dst is None) else None\n        result = run_pipeline(EditRequest(source_path=dst, template=tpl,\n                                          custom_instruction=instruction,\n                                          aspect=aspect, reference_path=ref_dst, mask_path=mask_dst),\n                              [route], cfg, editor=editor, scene_fn=scene_fn,\n                              source_contract=contract,require_complete=require_complete,allow_colour_conversion=allow_colour_conversion,use_embedded_alpha=use_embedded_alpha)\n        return job, dst, result\n\n    def _results_page(self, job: Path, src: Path, result, template_key: str,\n                      instruction: str, aspect: Aspect | None) -> bytes:\n        cards = [_RESULT_HEAD]\n        import_record=job/\'rgba-import.json\'\n        if import_record.is_file():\n            cards.append(\'<p class="meta" style="grid-column:1/-1">VERIFIED SOURCE-RGB RGBA IMPORT: native PNG RGB matches every source pixel. Alpha semantics and publication quality are NOT VERIFIED. The same saved RGBA is rechecked on regenerate. No model is called by import.</p>\')\n        manifest = json.loads(result.manifest_path.read_text())\n        intake = manifest.get("image_intake", {})\n        if intake:\n            import html as _html\n            if intake.get("embedded_alpha_used"):\n                cards.append(\'<p class="meta" style="grid-column:1/-1">EMBEDDED SOURCE ALPHA: retained from the upload after orientation normalization. No segmentation model was called. Mask quality and material semantics are NOT VERIFIED.</p>\')\n            text = "IMAGE INTAKE: " + ", ".join(intake.get("transformations", [])) if intake.get("transformations") else "IMAGE INTAKE: unprofiled decoded RGB assumed sRGB, not colour-accuracy verified."\n            cards.append(\'<p class="meta" style="grid-column:1/-1">\' + _html.escape(text + " Product comparisons use normalized RGB, not original encoded colour values.") + "</p>")\n        check = manifest.get("source_mask_preflight", {})\n        if check:\n            review_text = "SOURCE REVIEW: " + json.dumps(check.get("declaration", {})) + "\\n"\n            review_text += "\\n".join(check.get("blockers", []) + check.get("warnings", []))\n            review_text += "\\nMask semantic accuracy: NOT VERIFIED. Publication readiness: NOT EVALUATED."\n            cards.append(\'<div class="meta bad" style="grid-column:1/-1">\' + html.escape(review_text) + \'</div>\')\n        if any(v.get(\'product_transform\',{}).get(\'product_resampled\') for v in manifest.get(\'variants\',[]) if isinstance(v.get(\'product_transform\'),dict)):\n            cards.append(\'<p class="meta" style="grid-column:1/-1">PRODUCT RESAMPLING: one or more alternatives resize the product to template framing. Resampling and alpha blending can change visible RGB. Import verification, when supplied, does not certify native pixel-exact final outputs. Inspect fine print in downloads.</p>\')\n        cards.append(f\'<div class="shot"><img src="/file/{job.name}/{src.name}">\'\n                     f\'<div class="meta">{html.escape(_source_caption(manifest))}</div></div>\')\n        for v in result.variants:\n            if not v.success or not v.output_path.is_file():\n                flags = "; ".join(v.qa.critical_errors) or "Output file is missing"\n                partial = v.output_path.is_file()\n                text = (("FAILED / PARTIAL ARTIFACT - NOT A COMPLETED OUTPUT\\n" if partial else "FAILED / NOT GENERATED\\n") + f"{v.route.value} seed {v.seed}\\n" +\n                        ("Artifact retained for debugging. No quality rating.\\n" if partial else "No image output and no quality rating.\\n") + flags)\n                cards.append(f\'<div class="shot"><div class="meta bad">{html.escape(text)}</div></div>\')\n                continue\n            from .execution_kind import describe_execution\n            recorded = next((row.get("execution_kind") for row in manifest.get("variants",[]) if row.get("route")==v.route.value and row.get("seed")==v.seed),None)\n            execution = recorded or describe_execution(v.route.value,v.backend)\n            rel = v.output_path.relative_to(self.out_root)\n            q = v.qa\n            cls = "bad" if q.critical_errors else ""\n            flags = ("FLAGS: " + "; ".join(q.critical_errors)) if q.critical_errors else ""\n            from .fidelity_review import review_requirement, review_caption\n            fidelity_requirement = review_requirement(v.route.value, q.critical_errors)\n            flags = execution[\'caption\'] + \'\\n\' + review_caption(fidelity_requirement) + ("\\n" + flags if flags else "")\n            if fidelity_requirement[\'required\']:\n                cls = "bad"\n            review = (\'<p class="meta bad">REVIEW REQUIRED: GrabCut fallback may retain source background or cut product edges. \'\n                      \'A high pixel-fidelity score does not validate the mask.</p>\'\n                      if v.backend == "grabcut" else \'\')\n            href = html.escape(str(rel))\n            cards.append(\n                f\'<div class="shot">{review}<img src="/file/{href}">\'\n                f\'<div class="meta {cls}">{v.route.value} seed {v.seed} - \'\n                f\'{result.request.effective_aspect.value}\\n\'\n                f\'auto fidelity {"NOT_EVALUATED (review needed)" if q.score_fidelity is None else str(q.score_fidelity)+"/5"} (structural metric - not a \'\n                f\'human or publishability rating)\\n{html.escape(flags)}</div>\'\n                f\'<a class="dl" href="/file/{href}" \'\n                f\'download="{v.route.value}_seed{v.seed}.png">Download PNG</a></div>\')\n        if result.request.mask_path is not None:\n            cards.insert(1, \'<p class="meta">SUPPLIED MASK: assisted alpha retained on regenerate. Not automatic segmentation; inspect edges.</p>\')\n        a = aspect.value if aspect else ""\n        route = result.variants[0].route.value if result.variants else "cutout_compose"\n        rerun_template = _RERUN\n        if result.request.reference_path is not None:\n            rerun_template = rerun_template.replace(\'<h2>Edit / regenerate</h2>\',\n                \'<h2>Edit / regenerate</h2><p class="note">Saved reference/backdrop is retained on regenerate.</p>\')\n        # Preserve the actual route and alternatives rather than silently\n        # switching to first-option defaults on edit/regenerate.\n        for value in (r.value for r in Route):\n            rerun_template = rerun_template.replace(f\'value="{value}">\', f\'value="{value}"\' +\n                                                   (\' selected\' if value == route else \'\') + \'>\')\n        saved_review = json.loads((job / "source-review.json").read_text())\n        for name in ("pose", "material", "framing"):\n            pattern = r\'(<select name="source_\' + name + r\'">)(.*?)(</select>)\'\n            def selected(m):\n                return m[1] + m[2].replace(\'value="\' + saved_review[name] + \'">\',\n                    \'value="\' + saved_review[name] + \'" selected>\') + m[3]\n            rerun_template = re.sub(pattern, selected, rerun_template, flags=re.S)\n        if saved_review["require_complete"]:\n            rerun_template = rerun_template.replace(\'name="require_complete" value="yes"\', \'name="require_complete" value="yes" checked\')\n        if intake.get("policy", {}).get("allow_colour_conversion"):\n            rerun_template = rerun_template.replace(\'name="allow_colour_conversion" value="yes"\', \'name="allow_colour_conversion" value="yes" checked\')\n        if intake.get("embedded_alpha_used"):\n            rerun_template = rerun_template.replace(\'name="use_embedded_alpha" value="yes"\', \'name="use_embedded_alpha" value="yes" checked\')\n        n_variants = len(result.variants)\n        for value in (2,3,4):\n            rerun_template = rerun_template.replace(f\'<option>{value}</option>\',\n                f\'<option\' + (\' selected\' if value == n_variants else \'\') + f\'>{value}</option>\')\n        rerun = rerun_template.format(\n            job=job.name,\n            options="".join(\n                f\'<option value="{t.key}"{" selected" if t.key == template_key else ""}>\'\n                f\'{html.escape(t.name)}</option>\' for t in all_templates()),\n            instruction=html.escape(instruction),\n            a11=" selected" if a == "1:1" else "",\n            a45=" selected" if a == "4:5" else "",\n            a32=" selected" if a == "3:2" else "")\n        return _PAGE.format(backend_status=html.escape(getattr(self,"backend_status","Offline stub editor")), options=_template_cards(template_key),\n                            result="".join(cards) + rerun).encode()\n\n    def do_POST(self):\n        path = urlparse(self.path).path\n        length = int(self.headers.get("Content-Length", 0))\n        if length > 25_000_000:\n            self._send(413, b"photo too large (25MB cap)", "text/plain")\n            return\n        try:\n            if path in ("/reviewed-run","/reviewed-confirm"):\n                from .reviewed_pipeline import preview_pipeline,finish_pipeline\n                origin=self.headers.get("Origin")\n                if origin and origin!="http://"+self.headers.get("Host",""):\n                    raise ValueError("same-origin local request required")\n                lock=getattr(self,"reviewed_lock",None)\n                if lock is None or not lock.acquire(blocking=False):raise ValueError("reviewed pipeline busy or not configured")\n                try:\n                    fields=_parse_multipart(self.rfile.read(length),self.headers.get("Content-Type",""))\n                    if path=="/reviewed-run":\n                        d=preview_pipeline(self.out_root,fields["job"][1].decode(),json.loads(fields["prompts"][1]),fields["automatic"][1],getattr(self,"reviewed_checkpoint",None))\n                    else:\n                        d=finish_pipeline(self.out_root,fields["job"][1].decode(),fields["fingerprint"][1].decode(),fields["acceptance"][1].decode())\n                    self._send(200,json.dumps(d).encode(),"application/json")\n                finally:lock.release()\n                return\n            if path == "/prompt-prepare":\n                from .prompt_editor import prepare_source\n                fields = _parse_multipart(self.rfile.read(length),self.headers.get("Content-Type", ""))\n                prepared = prepare_source(fields["photo"][1],self.out_root)\n                self._send(200,json.dumps(prepared).encode(),"application/json")\n                return\n            if path == "/run":\n                fields = _parse_multipart(self.rfile.read(length),\n                                          self.headers.get("Content-Type", ""))\n                fname, data = fields["photo"]\n                if not data:\n                    raise ValueError("empty photo upload")\n                ext = Path(fname or "photo.png").suffix or ".png"\n                tmp_src = self.out_root / f"_upload{uuid.uuid4().hex[:6]}{ext}"\n                self.out_root.mkdir(parents=True, exist_ok=True)\n                tmp_src.write_bytes(data)\n                template_key = fields.get("template", ("", b"clean_studio"))[1].decode()\n                instruction = validate_instruction(fields.get("instruction", ("", b""))[1].decode())\n                aspect = _aspect_from(fields.get("aspect", ("", b""))[1].decode())\n                variants = int(fields.get("variants", ("", b"2"))[1].decode())\n                route = Route(fields.get("route", ("", b"cutout_compose"))[1].decode())\n                job, dst, result = self._run_job(tmp_src, template_key,\n                                                 instruction, aspect, variants, route,\n                                                 ref_upload=fields.get("reference"), mask_upload=fields.get("mask"),\n                                                 rgba_upload=fields.get("rgba_result"),\n                                                 allow_colour_conversion=fields.get("allow_colour_conversion",("",b""))[1]==b"yes",\n                                                 use_embedded_alpha=fields.get("use_embedded_alpha",("",b""))[1]==b"yes",\n                                                 source_contract=SourceContract(\n                                                     fields.get("source_pose",("",b"unknown"))[1].decode(),\n                                                     fields.get("source_material",("",b"unknown"))[1].decode(),\n                                                     fields.get("source_framing",("",b"unknown"))[1].decode()),\n                                                 require_complete=fields.get("require_complete",("",b""))[1]==b"yes")\n                tmp_src.unlink(missing_ok=True)\n            elif path == "/rerun":\n                form = parse_qs(self.rfile.read(length).decode())\n                job_name = form.get("job", [""])[0]\n                if not re.fullmatch(r"[a-f0-9]{12}", job_name):\n                    raise ValueError("invalid job ID")\n                src = next((self.out_root / job_name).glob("source.*"), None)\n                if src is None:\n                    raise ValueError("job not found or source missing")\n                template_key = form.get("template", ["clean_studio"])[0]\n                instruction = validate_instruction(form.get("instruction", [""])[0])\n                aspect = _aspect_from(form.get("aspect", [""])[0])\n                variants = int(form.get("variants", ["2"])[0])\n                route = Route(form.get("route", ["cutout_compose"])[0])\n                old_reference = next((self.out_root / job_name).glob("reference.*"), None)\n                ref_upload = ((old_reference.name, old_reference.read_bytes())\n                              if old_reference is not None else None)\n                old_mask = self.out_root / job_name / "mask.png"\n                mask_upload = ("mask.png", old_mask.read_bytes()) if old_mask.is_file() else None\n                old_rgba=self.out_root/job_name/\'external-rgba.png\'\n                rgba_upload=(\'external-rgba.png\',old_rgba.read_bytes())if old_rgba.is_file()else None\n                if rgba_upload is not None:mask_upload=None\n                prior_path = self.out_root / job_name / "source-review.json"\n                prior = json.loads(prior_path.read_text()) if prior_path.is_file() else {}\n                contract = SourceContract(*(form.get("source_"+k,[prior.get(k,"unknown")])[0]\n                                           for k in ("pose","material","framing")))\n                # New browser form always supplies source_pose; absence means an older API caller.\n                complete = (form.get("require_complete",[""])[0]=="yes" if "source_pose" in form\n                            else bool(prior.get("require_complete",False)))\n                job, dst, result = self._run_job(src, template_key, instruction,\n                                                 aspect, variants, route, ref_upload=ref_upload, mask_upload=mask_upload,\n                                                 rgba_upload=rgba_upload,source_contract=contract,require_complete=complete,\n                                                 allow_colour_conversion=form.get("allow_colour_conversion",[""])[0]=="yes",\n                                                 use_embedded_alpha=form.get("use_embedded_alpha",[""])[0]=="yes")\n            else:\n                self._send(404, b"not found", "text/plain")\n                return\n        except (ValueError, KeyError, StopIteration, RuntimeError, OSError, zipfile.BadZipFile) as e:\n            self._send(400, f"bad request: {html.escape(str(e))}".encode(),\n                       "text/plain")\n            return\n        self._send(200, self._results_page(job, dst, result, template_key,\n                                           instruction, aspect))\n\n    def log_message(self, *a):  # keep console clean during tests\n        pass\n\n\ndef serve(port: int, out_root: Path, reviewed_checkpoint: Path | None = None, *, editor_kind="stub", mh_model="qwen-edit", live=False) -> ThreadingHTTPServer:\n    if editor_kind not in (\'stub\',\'magichour\') or type(live)is not bool:\n        raise ValueError(\'explicit supported server editor mode required\')\n    if live and editor_kind!=\'magichour\':raise ValueError(\'live requires the Magic Hour editor\')\n    if editor_kind==\'magichour\':\n        from .backends.magichour import MagicHourBackend, MODEL_TABLE\n        if mh_model not in MODEL_TABLE:raise ValueError(\'unknown Magic Hour model\')\n        def factory(job):return MagicHourBackend(model=mh_model,live=live,cost_log=job/\'mh_cost_log.jsonl\',receipt_path=job/\'mh-projects.jsonl\')\n        status=(\'LIVE Magic Hour \'+mh_model+\'; pure_prompt generates chargeable outputs. Each generate/regenerate can spend credits. Owner approval is required outside this software.\' if live else \'MOCK Magic Hour \'+mh_model+\'; zero-spend offline demonstration, not model quality.\')\n    else:\n        factory=lambda job:StubEditor()\n        status=\'Offline stub editor; zero-spend demonstration, not model quality.\'\n    out_root = out_root.resolve()\n    out_root.mkdir(parents=True, exist_ok=True)\n    handler = type("BoundHandler", (_Handler,), {"out_root": out_root,"reviewed_checkpoint":reviewed_checkpoint.resolve() if reviewed_checkpoint else None,"reviewed_lock":threading.Lock(),"editor_factory":staticmethod(factory),"backend_status":status,"live_editor":live})\n    _render_previews(out_root)  # ready before the first page can request images\n    server = ThreadingHTTPServer(("127.0.0.1", port), handler)\n    threading.Thread(target=server.serve_forever, daemon=True).start()\n    return server\n\n\ndef main() -> int:\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--port", type=int, default=8000)\n    ap.add_argument("--out", type=Path, default=Path("runs/webui"))\n    ap.add_argument("--reviewed-checkpoint",type=Path,help="explicit existing pinned HQ-SAM checkpoint, no download")\n    ap.add_argument(\'--editor\',choices=[\'stub\',\'magichour\'],default=\'stub\',help=\'server-owned editor mode; browser requests cannot change it\')\n    ap.add_argument(\'--mh-model\',default=\'qwen-edit\')\n    ap.add_argument(\'--live\',action=\'store_true\',help=\'enable chargeable Magic Hour pure_prompt generation; separate owner approval required; never use while paid calls are paused\')\n    args = ap.parse_args()\n    if args.live:print(\'WARNING: live generation spends credits per request, including regenerate. A flag is not owner authorization.\')\n    server = serve(args.port, args.out,args.reviewed_checkpoint,editor_kind=args.editor,mh_model=args.mh_model,live=args.live)\n    print(f"Renderello UI on http://127.0.0.1:{args.port} (Ctrl+C to stop)")\n    try:\n        threading.Event().wait()\n    except KeyboardInterrupt:\n        server.shutdown()\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n'
# ---- embedded module: renderello/webui.py ----
path = base / 'renderello/webui.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
SRC = '"""Research-only transmission hypotheses, never recovered glass optics.\n\nFor assumed white illumination/plate P=1 in linear RGB, C=R+T admits\n0<=T<=min(C) for achromatic transmission and nonnegative additive R.\nEvery T in that interval reconstructs C, so reconstruction cannot select T.\nThe input alpha is estimated spatial coverage, not measured transparency.\n"""\nimport math\nimport numpy as np\nfrom .candidate_diagnostics import native_rgb, native_alpha, digest\nfrom .candidate_harmonization import linear_over\nfrom .refractive_transfer import decode, encode\n\n\ndef white_reference_hypothesis(source, alpha, target, white_samples, *,\n                               bindings, transmission_fraction,\n                               assume_white_reference=False, reviewed=False,\n                               review_note=\'\'):\n    native_rgb(source)\n    native_alpha(alpha, source.shape[:2])\n    native_rgb(target)\n    if target.shape != source.shape:\n        raise ValueError(\'native matching target required; no resize\')\n    if (not isinstance(white_samples, np.ndarray)\n            or white_samples.dtype != np.bool_\n            or white_samples.shape != alpha.shape\n            or int(white_samples.sum()) < 64):\n        raise ValueError(\'at least 64 explicitly reviewed native white samples required\')\n    if np.any(alpha[white_samples] != 0):\n        raise ValueError(\'white-reference samples must be outside estimated object coverage\')\n    expected = dict(source=digest(source), alpha=digest(alpha),\n                    target=digest(target), white_samples=digest(white_samples))\n    if not isinstance(bindings, dict) or bindings != expected:\n        raise ValueError(\'exact source/alpha/target/white-sample bindings required\')\n    if (reviewed is not True or assume_white_reference is not True\n            or not isinstance(review_note, str) or not review_note.strip()\n            or len(review_note) > 2000):\n        raise ValueError(\'explicit research white-reference assumption and review required\')\n    k = transmission_fraction\n    if type(k) not in (float, int) or not math.isfinite(k) or not 0 <= k <= 1:\n        raise ValueError(\'finite transmission fraction in 0..1 required\')\n    white_fraction = float(np.mean(np.all(source[white_samples] >= 250, axis=1)))\n    if white_fraction < .95:\n        raise ValueError(\'reviewed exterior samples do not support white-reference shortcut\')\n    if not np.any(alpha):\n        raise ValueError(\'nonempty estimated object coverage required\')\n    observed = decode(source)\n    upper = np.min(observed, axis=2)\n    transmission = k * upper\n    residual = observed - transmission[:, :, None]\n    prediction = residual + transmission[:, :, None] * decode(target)\n    if np.any(prediction < 0) or np.any(prediction > 1):\n        raise ValueError(\'hypothesis exceeds linear gamut; no clipping\')\n    transformed = encode(prediction)\n    candidate = linear_over(transformed, alpha, target)\n    reconstructed = encode(residual + transmission[:, :, None])\n    report = dict(\n        schema=\'white-reference-transmission-hypothesis.v1\', bindings=expected,\n        transmission_fraction=k, white_sample_pixels=int(white_samples.sum()),\n        white_sample_near_white_fraction=white_fraction,\n        layer_reconstruction_changed_pixels=int(np.any(reconstructed != source, axis=2).sum()),\n        candidate_changed_source_pixels=int(np.any(candidate != source, axis=2).sum()),\n        alpha_unchanged=True, source_resampled=False,\n        reference_origin=\'ASSUMED_UNIFORM_WHITE_NOT_CAPTURED_CLEAN_PLATE\',\n        transmission_origin=\'FEASIBLE_PHOTOMETRIC_HYPOTHESIS_NOT_MEASURED\',\n        flow_origin=\'IDENTITY_HYPOTHESIS_NOT_REFRACTION\',\n        real_glass_validated=False, photo_accepted=False, candidate_selected=False,\n        source_rgb_locked=False, default_pipeline_changed=False, review_note=review_note,\n        limitations=[\n            \'White exterior samples cannot prove the unseen plate behind the object.\',\n            \'Exact layer reconstruction is an algebraic identity, not optical validation.\',\n            \'Transmission and additive residual cannot be identified from this single source.\',\n            \'Highlights, reflections and dark edges can be removed or retained incorrectly.\',\n            \'No refraction, caustics, geometry, new illumination or missing framing is recovered.\',\n            \'Estimated alpha adds spatial-coverage uncertainty, especially at boundaries.\',\n            \'Changed product RGB is expected; this is not the source-RGB-locked route.\'\n        ])\n    return candidate, transmission, report\n'
# ---- embedded module: renderello/white_reference_transmission.py ----
path = base / 'renderello/white_reference_transmission.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(SRC)
print('wrote', path)


In [ ]:
import importlib, sys
sys.path.insert(0, '/content/renderello-src')
import renderello
importlib.reload(renderello)
print('renderello package ready:', renderello.__file__)

## 2. Choose an engine
- **procedural**: free, CPU, deterministic backdrop; product pixels stay source-exact.
- **sdxl_lightning**: free AI scene on this Colab GPU; the product is composited on top.
- **magichour**: Magic Hour AI editor. In this notebook the local engine runs as a dry-run mock: it shows the request that would be sent and spends nothing. LIVE calls never run from the notebook's local engine by design: a notebook-side ledger could not see website spend and the two together could exceed the authorized total. Live runs go through the hosted website bridge in section 7, so one server-side ledger, balance anchor and budget guard protect every live surface. The Magic Hour integration itself is unchanged and remains in the pipeline code below; the procedural and SDXL engines are fully functional. Live calls use one input image per call, seedream-v5-flash at 640px, documented normal price 30 credits per image (the provider's documented list price, not a billing guarantee).

Spend guard (server-side, applies to every live surface): a per-call reservation against the authorized credit budget, an account-balance anchor, and a durable ledger. If the budget, anchor, or ledger configuration is missing or unreadable, the guard refuses BEFORE any paid call and zero credits are billed. The deployment's ledger credentials and API key are private to its owner, are never shared, and must never be pasted into this notebook.

Cutouts use the best learned backend available: BiRefNet when torch and kornia are importable, otherwise the compact U2Net-p ONNX model, otherwise an OpenCV grabcut fallback (cruder edges). The backend actually used is printed in section 6 and recorded in manifest.json.

In [ ]:
provider = 'sdxl_lightning'  # 'procedural' (free) | 'sdxl_lightning' (free, GPU) | 'magichour' (dry-run mock)
mhModel = 'seedream-v5-flash'  # live allowlist default: one input image, 640px, 30 credits
liveMode = False          # dry-run mock; live calls are blocked in this notebook (see above)
creditBudget = 30         # maximum credits a live run may spend; above this it is refused
import os
os.environ['RENDERELLO_MH_BUDGET_CREDITS'] = str(creditBudget)
if provider == 'magichour' and liveMode:
    raise RuntimeError(
        'Live Magic Hour calls never run from the notebook local engine: they go through '
        'the hosted website bridge in section 7, so one server-side ledger guards every '
        'live surface. Use liveMode = False here (dry-run mock) or section 7 for live runs.')
print('engine:', provider, '| magichour model:', mhModel, '| live:', liveMode)

## 3. Upload product photo(s) and optional reference image(s)
Every product photo is kept pixel-exact in compose mode; reference images steer style only, never composited.

In [ ]:
from google.colab import files
import pathlib
up = pathlib.Path('/content/uploads'); up.mkdir(exist_ok=True)
def _save(uploaded):
    saved = []
    for name, data in uploaded.items():
        p = up / pathlib.Path(name).name
        p.write_bytes(data)
        saved.append(p)
    return saved
print('Upload PRODUCT photo(s):')
products = _save(files.upload())
print('Upload REFERENCE image(s) (optional - cancel to skip):')
try:
    refs = _save(files.upload())
except Exception:
    refs = []
print('products:', [p.name for p in products], '| references:', [p.name for p in refs])

## 4. Pick a preset and describe the shot

In [ ]:
from renderello.presets import PRESETS
for key, preset in PRESETS.items():
    print(f'{key:18s} {preset.name:22s} {preset.tagline}')
presetKey = 'clean_studio'
productDescription = ''   # e.g. '100 ml amber glass fragrance bottle with black cap'
materialChoice = ''       # matte_plastic|glossy_plastic|glass|metal|fabric|ceramic|wood|leather (blank = auto)
instructionText = ''      # scene, mood, props, palette, camera; environment only, never the product
variantCount = 2

## 5. Generate

In [ ]:
from renderello.studio import StudioRequest, run_studio
runResult = run_studio(StudioRequest(
    product_paths=products, preset_key=presetKey, instruction=instructionText,
    reference_paths=refs, variants=variantCount, provider=provider,
    material=materialChoice or None, product_description=productDescription,
    knob_values={'mh_model': mhModel}), pathlib.Path('/content/runs'))
print('run dir:', runResult.run_dir)
for variant in runResult.variants:
    print('variant:', variant.image or f'FAILED: {variant.error}')

## 6. Review and download
Contact sheet + manifest (prompt, QA verdicts, credits) included. QA honesty: whole-image AI edits regenerate the product - verify label/logo fidelity before publishing.

In [ ]:
from PIL import Image
import json
display(Image.open(runResult.run_dir / 'contact_sheet.png'))
manifest = json.loads((runResult.run_dir / 'manifest.json').read_text())
print('mode:', manifest['mode'], '| provider:', manifest['provider_used'])
for p in manifest.get('products', []):
    print('product:', p['file'], '| cutout backend:', p.get('cutout_backend', 'n/a'),
          '| coverage:', p.get('cutout_coverage', 'n/a'))
for w in manifest.get('warnings', []):
    print('WARNING:', w)
for v in manifest['variants']:
    print(v['image'], '| qa:', (v.get('qa') or {}).get('publish_ready'), '|', '; '.join(v.get('warnings') or []))
import shutil
shutil.make_archive('/content/renderello_run', 'zip', runResult.run_dir)
files.download('/content/renderello_run.zip')

## 7. (Optional) One live Magic Hour run through the hosted website
Live spend goes through the hosted website's guarded endpoints, so the SAME server-side budget guard, balance anchor and durable spend ledger protect this notebook and the website - one ledger, no double counting.

- Sign in with YOUR OWN website account; it is entered at runtime and is never stored in this notebook.
- No Magic Hour API key and no ledger token appear in this notebook; they stay server-side.
- Nothing is spent automatically: step 7.2 prints the server's own estimate, step 7.3 needs your typed confirmation, and the server reserves and checks the credits before any paid call - a refusal bills zero credits.
- Exactly one paid image per confirmed run: the first uploaded product photo, one template, one variant (documented normal price 30 credits; the provider's list price, not a billing guarantee).
- Live runs work only while the deployment owner has Magic Hour enabled on the server; otherwise the server refuses before any paid call.
- The session lives only in this runtime's memory; step 7.5 signs out.

In [ ]:
# 7.1 Sign in to the hosted website (your own account, runtime only)
bridgeEnabled = False  # explicit opt-in only; this is NOT payment approval
typedConfirmation = None
spendConfirmationToken = None
bridgeQuoteToken = None
bridgeSessionToken = None
site = None  # discard any stale runtime session before the default-off guard
if not bridgeEnabled:
    print('7.1 skipped: paid bridge disabled by design; not part of unattended Run-all.')
    print('For the optional single live run set bridgeEnabled = True above and run 7.1-7.5 manually.')
else:
    import secrets
    bridgeSessionToken = secrets.token_hex(16)
    siteUrl = 'https://renderello-studio.onrender.com'
    import json, requests, getpass
    site = requests.Session()
    signIn = site.post(siteUrl + '/api/login', allow_redirects=False, json={
        'email': getpass.getpass('Website sign-in email: '),
        'password': getpass.getpass('Website sign-in password: ')}, timeout=30)
    print('sign-in status:', signIn.status_code)
    signIn.raise_for_status()
    del signIn  # drop the auth response object; only the session cookie persists

In [ ]:
if globals().get('bridgeEnabled') is not True:
    print('7.2 skipped: paid bridge disabled by design; not part of unattended Run-all.')
else:
    # 7.2 Server-side estimate (never spends)
    assert globals().get('bridgeEnabled') is True and globals().get('bridgeSessionToken'), 'Bridge disabled: run enabled 7.1 first'
    assert globals().get('site') is not None, 'No current bridge session'
    bridgeQuoteToken = None
    bridgePreset = 'clean_studio'
    bridgeSceneAttempts = 1  # no automatic second paid scene attempt
    estimate = site.get(siteUrl + '/api/estimate', timeout=30, params={
        'provider': 'magichour', 'mh_model': mhModel, 'variants': 1,
        'presets': bridgePreset, 'mh_route': 'scene',
        'knob_scene_max_attempts': str(bridgeSceneAttempts)}).json()
    print(json.dumps(estimate, indent=2))
    assert estimate.get('credits_per_image'), 'server estimate unavailable - stop here'
    assert estimate.get('within_budget') is True, \
        'server says this batch exceeds the approved budget - stop, zero credits billed'
    assert estimate.get('within_batch_limit') is True, \
        'server says this batch exceeds the per-batch limit - stop, zero credits billed'
    bridgeQuoteToken = bridgeSessionToken

In [ ]:
if globals().get('bridgeEnabled') is not True:
    print('7.3 skipped: paid bridge disabled by design; not part of unattended Run-all.')
else:
    # 7.3 Explicit confirmation - nothing is sent to Magic Hour before this cell
    assert globals().get('bridgeEnabled') is True, 'Bridge disabled'
    assert globals().get('bridgeSessionToken') and globals().get('bridgeQuoteToken') == bridgeSessionToken, 'Get a fresh estimate in 7.2'
    typedConfirmation = None
    spendConfirmationToken = None
    print('Documented normal price for this request:',
          estimate['credits_per_image'], 'credits (list price, not a billing guarantee).')
    print('Budget state reported by the server:', estimate['budget'])
    typedConfirmation = input('Type exactly SPEND to authorize ONE live image: ')
    assert typedConfirmation == 'SPEND', 'not confirmed - nothing was sent, zero credits billed'
    spendConfirmationToken = bridgeSessionToken

In [ ]:
if globals().get('bridgeEnabled') is not True:
    print('7.4 skipped: paid bridge disabled by design; not part of unattended Run-all.')
else:
    # 7.4 The single live call + download (everything paid-adjacent happens HERE,
    # while the session is alive; the finally-block signs out only after downloads)
    assert globals().get('bridgeEnabled') is True, 'Bridge disabled'
    assert globals().get('bridgeSessionToken') and globals().get('spendConfirmationToken') == bridgeSessionToken, 'No fresh spending confirmation for this session'
    spendConfirmationToken = None  # consume before any paid POST
    import time, re
    assert typedConfirmation == 'SPEND', \
        'stale cell: rerun 7.3 and type SPEND again - a consumed confirmation cannot spend twice'
    typedConfirmation = None  # consumed BEFORE the POST: a rerun of this cell cannot double-charge
    spentBefore = estimate['budget']['spent']
    print('pre-POST durable spent baseline:', spentBefore, \
          '- write this down; recovery below compares against it')
    job = {}
    savedFiles = []
    try:
        with open(products[0], 'rb') as fh:
            try:
                sent = site.post(siteUrl + '/api/generate', timeout=120,
                    allow_redirects=False,
                    files={'products': (products[0].name, fh)},
                    data={'preset': bridgePreset, 'provider': 'magichour', 'variants': '1',
                          'instruction': instructionText,
                          'product_description': productDescription, 'mh_model': mhModel,
                          'mh_route': 'scene', 'knob_scene_max_attempts': str(bridgeSceneAttempts)})
            except requests.Timeout:
                raise SystemExit(
                    'POST timed out: the server MAY have accepted the job. '
                    'DO NOT RERUN this cell and DO NOT start a new run - a rerun '
                    'would submit a second paid job. Recovery: sign in again (7.1), '
                    're-run 7.2 and compare budget.spent with the baseline printed '
                    'above. A rise by credits_per_image means the job was accepted; '
                    'an unchanged value is NOT proof it was not submitted (another '
                    'website session can move the same budget). Check the website '
                    'Results tab and NEVER resubmit until the deployment owner '
                    'confirms the job state.')
        print('generate status:', sent.status_code)
        if sent.status_code >= 500:
            raise SystemExit(
                f'server error {sent.status_code}: paid state UNKNOWN - the request '
                'may have been accepted before the failure. DO NOT RERUN this cell. '
                'Recover exactly as for a timeout: re-login (7.1), re-run 7.2, '
                'compare budget.spent with the baseline, check the Results tab, and '
                'never resubmit until the deployment owner confirms.')
        if sent.status_code != 200:
            print('server refusal detail:', sent.text[:400])
        sent.raise_for_status()  # 4xx refusals bill zero credits
        jobId = sent.json()['job_id']
        del sent
        deadline = time.monotonic() + 900
        while time.monotonic() < deadline:
            time.sleep(5)
            job = site.get(siteUrl + '/api/jobs/' + jobId, timeout=30).json()
            if job.get('status') != 'running':
                break
        if job.get('status') == 'running':
            raise SystemExit(
                f'job {jobId} still running after 15 minutes. It may still spend '
                'or finish. DO NOT RERUN this cell. This cell signed out in its '
                'finally block, so sign in again with 7.1 first, then poll '
                'GET /api/jobs/' + jobId + ' with the new session, or check the '
                'Results tab. Never start a new paid job until this one resolves.')
        print('job:', job.get('status'), job.get('error') or '')
        assert job.get('status') == 'done', (\
            'live run did not complete (job id ' + jobId + '). A failed or '
            'interrupted job MAY STILL HAVE BILLED credits. DO NOT RERUN this '
            'cell: sign in again (7.1), re-run 7.2 and check budget.spent and '
            'the Results tab before any new attempt.')
        runDir = job['runs'][0]['run_dir']
        # Result paths must be server-relative run ids on the pinned host - never
        # follow an absolute URL or unexpected shape returned inside job data.
        assert re.fullmatch(r'[0-9]{8}-[0-9]{6}-[0-9a-f]+', runDir), \
            'unexpected run id shape - refusing to fetch an unverified result path'
        for name in ['contact_sheet.png', 'manifest.json']:
            blob = site.get(siteUrl + '/api/runs/' + runDir + '/' + name, timeout=60)
            if blob.status_code != 200:
                raise SystemExit(
                    f'result download failed ({blob.status_code}) for {name}. The '
                    'paid job COMPLETED - do not rerun 7.4. Sign in again (7.1) and '
                    'retry only this download block, or use the Results tab.')
            out = pathlib.Path('/content/live_' + name)
            out.write_bytes(blob.content)
            savedFiles.append(str(out))
            print('saved', out, len(blob.content), 'bytes')
            del blob
    finally:
        # Sign out on every path - after downloads, refusal or failure. The server
        # drops only this runtime's own session; other browser sessions persist.
        site.post(siteUrl + '/api/logout', allow_redirects=False, timeout=30)
        print('signed out; live session closed')

In [ ]:
if globals().get('bridgeEnabled') is not True:
    print('7.5 skipped: paid bridge disabled by design; not part of unattended Run-all.')
else:
    # 7.5 Review the downloaded result (files already saved in 7.4; no session needed)
    assert globals().get('bridgeEnabled') is True, 'Bridge disabled'
    from PIL import Image
    display(Image.open('/content/live_contact_sheet.png'))
    manifestLive = json.loads(pathlib.Path('/content/live_manifest.json').read_text())
    print(json.dumps({k: manifestLive.get(k) for k in
          ('mode', 'provider_used', 'budget', 'knobs') if k in manifestLive}, indent=2))
    print('eye review remains the acceptance gate - QA scores are screens, not proof')